In [4]:

from pathlib import Path
import json

# Find the dataset files mounted by Kaggle
matches = list(Path("/kaggle/input").rglob("label_vocab.json"))

if not matches:
    print("Dataset not found. Check the Kaggle Input panel.")
else:
    phase3_root = matches[0].parent
    print("Dataset directory:", phase3_root)

    required_files = [
        "CL_train.json", "CL_dev.json", "CL_test.json",
        "IT_train.json", "IT_dev.json", "IT_test.json",
        "label_vocab.json"
    ]

    for name in required_files:
        path = phase3_root / name
        print(("OK" if path.is_file() else "MISSING"), name)

    # Inspect the label vocabulary
    vocab_path = phase3_root / "label_vocab.json"
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab = json.load(f)

    print("\nLabel vocabulary:")
    print(vocab)


Dataset directory: /kaggle/input/datasets/aq1de3/phase3
OK CL_train.json
OK CL_dev.json
OK CL_test.json
OK IT_train.json
OK IT_dev.json
OK IT_test.json
OK label_vocab.json

Label vocabulary:
{'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}


In [5]:

import json
from pathlib import Path
from collections import Counter

DATA_DIR = Path("/kaggle/input/datasets/aq1de3/phase3")

with open(DATA_DIR / "label_vocab.json", encoding="utf-8") as f:
    label_vocab = json.load(f)

valid_labels = set(label_vocab.keys())

def load_json(filename):
    with open(DATA_DIR / filename, encoding="utf-8") as f:
        return json.load(f)

def get_documents(data):
    # Support common JSON layouts without assuming a specific document ID key.
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ("documents", "data", "examples", "records"):
            if key in data and isinstance(data[key], list):
                return data[key]
        return list(data.values())
    raise TypeError(f"Unexpected JSON type: {type(data)}")

def inspect_split(filename):
    data = load_json(filename)
    docs = get_documents(data)

    print(f"\n{'='*65}\n{filename}")
    print("Top-level type:", type(data).__name__)
    print("Number of documents:", len(docs))

    if not docs:
        print("WARNING: Empty dataset!")
        return

    first = docs[0]
    print("First document type:", type(first).__name__)
    if isinstance(first, dict):
        print("First document keys:", list(first.keys()))
        print("Text field type:", type(first.get("text")).__name__)
        print("Labels field type:", type(first.get("labels")).__name__)

    counts = Counter()
    total_sentences = 0
    alignment_errors = []
    unknown_labels = Counter()
    missing_fields = 0
    lengths = []

    for i, doc in enumerate(docs):
        if not isinstance(doc, dict):
            alignment_errors.append((i, "document is not a dictionary"))
            continue

        text = doc.get("text")
        labels = doc.get("labels")

        if text is None or labels is None:
            missing_fields += 1
            continue

        # Expected format: text and labels are ordered lists.
        if not isinstance(text, list) or not isinstance(labels, list):
            alignment_errors.append(
                (i, f"text={type(text).__name__}, labels={type(labels).__name__}")
            )
            continue

        total_sentences += len(text)
        lengths.append(len(text))

        if len(text) != len(labels):
            alignment_errors.append(
                (i, f"{len(text)} sentences vs {len(labels)} labels")
            )

        for label in labels:
            if label not in valid_labels:
                unknown_labels[str(label)] += 1
            counts[label] += 1

    print("Total sentences:", total_sentences)
    if lengths:
        print("Sentences/document — min:", min(lengths),
              "| median:", sorted(lengths)[len(lengths)//2],
              "| max:", max(lengths))
    print("Documents missing text/labels:", missing_fields)
    print("Sentence-label alignment errors:", len(alignment_errors))
    if alignment_errors:
        print("First alignment issues:", alignment_errors[:5])
    print("Unexpected labels:", dict(unknown_labels) if unknown_labels else "None")

    print("\nLabel distribution:")
    for label in label_vocab:
        n = counts[label]
        print(f"{label:28s} {n:8d}  ({100*n/max(total_sentences,1):6.2f}%)")

    return {
        "documents": len(docs),
        "sentences": total_sentences,
        "counts": counts,
        "alignment_errors": alignment_errors,
        "unknown_labels": unknown_labels,
    }

results = {}
for filename in ["CL_train.json", "CL_dev.json",
                 "IT_train.json", "IT_dev.json"]:
    results[filename] = inspect_split(filename)



CL_train.json
Top-level type: list
Number of documents: 40
First document type: dict
First document keys: ['id', 'text', 'labels']
Text field type: list
Labels field type: list
Total sentences: 10593
Sentences/document — min: 19 | median: 123 | max: 1816
Documents missing text/labels: 0
Sentence-label alignment errors: 0
Unexpected labels: None

Label distribution:
Fact                             2979  ( 28.12%)
Issue                              17  (  0.16%)
ArgumentPetitioner                976  (  9.21%)
ArgumentRespondent                996  (  9.40%)
Statute                           367  (  3.46%)
Dissent                           284  (  2.68%)
PrecedentReliedUpon              1305  ( 12.32%)
PrecedentNotReliedUpon            209  (  1.97%)
PrecedentOverruled                  0  (  0.00%)
RulingByLowerCourt                438  (  4.13%)
RatioOfTheDecision               2856  ( 26.96%)
RulingByPresentCourt              113  (  1.07%)
None                               53  (  0

In [8]:

from pathlib import Path

roots = [Path("/kaggle/input"), Path("/kaggle/working")]
patterns = {
    "config.json",
    "model.safetensors",
    "pytorch_model.bin",
    "adapter_config.json",
    "adapter_model.safetensors",
}

found = set()

for root in roots:
    for path in root.rglob("*"):
        if path.is_file() and (
            path.name in patterns
            or path.suffix in {".pt", ".pth", ".ckpt"}
        ):
            found.add(str(path))

for path in sorted(found):
    print(path)

if not found:
    print("No standard checkpoint files found.")
    print("\nTop-level Kaggle inputs:")
    for path in Path("/kaggle/input").iterdir():
        print(path)


/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt
/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt
/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt


In [12]:

from pathlib import Path
import torch

CHECKPOINTS = {
    "A1_GPT": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2_GQA": "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "A3_PrefixLM": "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "A4_EncoderDecoder": "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
}

def inspect_value(value, prefix="", depth=0):
    if isinstance(value, dict):
        print(f"{prefix}dict with {len(value)} keys")
        if depth < 2:
            for i, (key, item) in enumerate(value.items()):
                if i >= 30:
                    print(prefix, "... additional keys omitted")
                    break
                inspect_value(item, f"{prefix}{key}: ", depth + 1)
    elif isinstance(value, torch.Tensor):
        print(f"{prefix}Tensor shape={tuple(value.shape)}, dtype={value.dtype}")
    elif isinstance(value, (list, tuple)):
        print(f"{prefix}{type(value).__name__}, length={len(value)}")
    elif isinstance(value, (int, float, str, bool)) or value is None:
        print(f"{prefix}{value!r}"[:200])
    else:
        print(f"{prefix}type={type(value).__name__}")

for name, filename in CHECKPOINTS.items():
    path = Path(filename)
    print("\n" + "=" * 70)
    print(name, "| exists:", path.is_file())

    if not path.is_file():
        continue

    print("File size:", round(path.stat().st_size / (1024**2), 2), "MB")

    try:
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=True
        )
        inspect_value(checkpoint)
        del checkpoint
    except Exception as e:
        print("Safe inspection could not load this checkpoint:")
        print(type(e).__name__, str(e)[:500])
        print("Do not retry with weights_only=False yet.")



A1_GPT | exists: True
File size: 593.71 MB
dict with 6 keys
step: 500
epoch: 0
train_loss: 5.366012096405029
val_loss: None
model_state_dict: dict with 101 keys
model_state_dict: token_embedding.weight: Tensor shape=(50000, 512), dtype=torch.float32
model_state_dict: position_embedding.weight: Tensor shape=(2048, 512), dtype=torch.float32
model_state_dict: blocks.0.norm1.weight: Tensor shape=(512,), dtype=torch.float32
model_state_dict: blocks.0.norm1.bias: Tensor shape=(512,), dtype=torch.float32
model_state_dict: blocks.0.attn.in_proj_weight: Tensor shape=(1536, 512), dtype=torch.float32
model_state_dict: blocks.0.attn.in_proj_bias: Tensor shape=(1536,), dtype=torch.float32
model_state_dict: blocks.0.attn.out_proj.weight: Tensor shape=(512, 512), dtype=torch.float32
model_state_dict: blocks.0.attn.out_proj.bias: Tensor shape=(512,), dtype=torch.float32
model_state_dict: blocks.0.norm2.weight: Tensor shape=(512,), dtype=torch.float32
model_state_dict: blocks.0.norm2.bias: Tensor shap

In [11]:

from pathlib import Path
import json

tokenizer_path = Path(
    "/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json"
)

with open(tokenizer_path, encoding="utf-8") as f:
    tokenizer_data = json.load(f)

print("Tokenizer file:", tokenizer_path)
print("Top-level keys:", list(tokenizer_data.keys()))
print("Model type:", tokenizer_data.get("model", {}).get("type"))
print("Vocabulary size:",
      len(tokenizer_data.get("model", {}).get("vocab", {})))
print("Special tokens:", tokenizer_data.get("added_tokens", [])[:10])


Tokenizer file: /kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
Top-level keys: ['version', 'truncation', 'padding', 'added_tokens', 'normalizer', 'pre_tokenizer', 'post_processor', 'decoder', 'model']
Model type: BPE
Vocabulary size: 50000
Special tokens: [{'id': 0, 'content': '[PAD]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 1, 'content': '[UNK]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 2, 'content': '[BOS]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 3, 'content': '[EOS]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 4, 'content': '[MASK]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}]


In [13]:
import torch
from collections import Counter

# A3 checkpoint
CHECKPOINT_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "step-0001000-pt/step_0001000.pt"
)

print("PyTorch:", torch.__version__)
print("Loading checkpoint...")

# Load checkpoint on CPU first; no GPU required for this inspection
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"\nNested dictionary: {key}")
            print("Number of entries:", len(value))

            if value and all(
                isinstance(v, torch.Tensor) for v in value.values()
            ):
                state_dict = value
                print("Using this dictionary as the model state dict.")
                break
    else:
        raise RuntimeError(
            "Could not find a nested state dictionary containing tensors."
        )
else:
    raise TypeError("Unexpected checkpoint format.")

print("\nFirst 30 parameter names and shapes:")
for i, (name, tensor) in enumerate(state_dict.items()):
    if i >= 30:
        break
    print(f"{name:55s} {tuple(tensor.shape)}")

print("\nParameter-name prefixes:")
prefixes = Counter(
    name.split(".")[0] if "." in name else name
    for name in state_dict
)
for prefix, count in prefixes.items():
    print(f"{prefix:25s}: {count}")

print("\nCheckpoint metadata:")
for key, value in checkpoint.items():
    if not isinstance(value, (dict, torch.Tensor)):
        print(f"{key}: {value}")

print("\nA3 checkpoint inspection complete.")
print("No model training was performed.")

PyTorch: 2.11.0+cu128
Loading checkpoint...

Checkpoint type: dict
Top-level keys: ['model', 'step', 'tokens_seen', 'architecture', 'parameters', 'context_length', 'hidden_size', 'num_layers', 'num_heads', 'ffn_size', 'prefix_ratio', 'dropout', 'learning_rate', 'weight_decay']

Nested dictionary: model
Number of entries: 100
Using this dictionary as the model state dict.

First 30 parameter names and shapes:
token_embedding.weight                                  (50000, 512)
position_embedding.weight                               (2048, 512)
layers.0.ln1.weight                                     (512,)
layers.0.ln1.bias                                       (512,)
layers.0.attn.in_proj_weight                            (1536, 512)
layers.0.attn.in_proj_bias                              (1536,)
layers.0.attn.out_proj.weight                           (512, 512)
layers.0.attn.out_proj.bias                             (512,)
layers.0.ln2.weight                                     (512,)


In [14]:
import torch
import torch.nn as nn

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "step-0001000-pt/step_0001000.pt"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)
state_dict = checkpoint["model"]

print("Checkpoint architecture:", checkpoint["architecture"])
print("Checkpoint parameters:", f'{checkpoint["parameters"]:,}')

# Look for model classes already defined in this notebook.
candidates = {
    name: obj
    for name, obj in globals().items()
    if isinstance(obj, type)
    and issubclass(obj, nn.Module)
    and obj is not nn.Module
}

print("\nPyTorch model classes currently defined:")
if candidates:
    for name, cls in candidates.items():
        print(f" - {name}: {cls.__module__}.{cls.__qualname__}")
else:
    print("No custom model classes found.")

# Show the exact checkpoint structure needed for compatibility.
print("\nA3 state-dict structure:")
for key in state_dict:
    if key.startswith(("token_embedding", "position_embedding", "layers.0", "ln_f")):
        print(key, tuple(state_dict[key].shape))

print("\nNext: identify the original A3 model class before strict loading.")

Checkpoint architecture: A3_prefix_lm_hybrid
Checkpoint parameters: 51,868,672

PyTorch model classes currently defined:
No custom model classes found.

A3 state-dict structure:
token_embedding.weight (50000, 512)
position_embedding.weight (2048, 512)
layers.0.ln1.weight (512,)
layers.0.ln1.bias (512,)
layers.0.attn.in_proj_weight (1536, 512)
layers.0.attn.in_proj_bias (1536,)
layers.0.attn.out_proj.weight (512, 512)
layers.0.attn.out_proj.bias (512,)
layers.0.ln2.weight (512,)
layers.0.ln2.bias (512,)
layers.0.ffn.0.weight (2048, 512)
layers.0.ffn.0.bias (2048,)
layers.0.ffn.3.weight (512, 2048)
layers.0.ffn.3.bias (512,)
ln_f.weight (512,)
ln_f.bias (512,)

Next: identify the original A3 model class before strict loading.


In [18]:
import os

print("Searching common Kaggle input and working directories...")

roots = ["/kaggle/input", "/kaggle/working"]

matches = []

for root in roots:
    if not os.path.exists(root):
        continue

    for current_dir, dirs, files in os.walk(root):
        # Avoid descending into irrelevant hidden directories
        dirs[:] = [d for d in dirs if not d.startswith(".")]

        for filename in files:
            if filename.lower().endswith(".ipynb"):
                path = os.path.join(current_dir, filename)
                matches.append(path)

                if len(matches) >= 100:
                    break

        if len(matches) >= 100:
            break

if matches:
    print("\nNotebook files found:")
    for path in matches:
        print(path)
else:
    print("\nNo notebook files found in these directories.")
    print("The original Phase 1 notebook may not be attached to this Kaggle session.")

Searching common Kaggle input and working directories...

Notebook files found:
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb


In [19]:
import json

NOTEBOOK_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "phase1-model/phase1_model_code_extract.ipynb"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "A3_prefix_lm_hybrid",
    "PrefixLMModel",
    "PrefixLMBlock",
    "create_prefix_lm_mask",
    "torch.save",
    "state_dict",
]

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))

    if cell.get("cell_type") != "code":
        continue

    matched = [k for k in keywords if k.lower() in source.lower()]

    if matched:
        print(f"\n{'=' * 20} CELL {i} {'=' * 20}")
        print("Matched:", ", ".join(matched))
        print(source)


==================== CELL 6 ====================
Matched: PrefixLMModel, PrefixLMBlock, create_prefix_lm_mask
# ============================================================
# ARCHITECTURE 3 — PREFIX-LM HYBRID
# STEP 3: MODEL DEFINITION + PARAMETER CHECK
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Prefix-LM Transformer Block
# ------------------------------------------------------------

class PrefixLMBlock(nn.Module):

    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.0):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_

In [20]:
import os

ROOT = "/kaggle/input/datasets/aq1de3"

extensions = (".py", ".ipynb", ".txt", ".md")

print("Searching for source files...\n")

found = []

for current_dir, dirs, files in os.walk(ROOT):
    for filename in files:
        if filename.lower().endswith(extensions):
            path = os.path.join(current_dir, filename)
            found.append(path)
            print(path)

print(f"\nTotal source files found: {len(found)}")

Searching for source files...

/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb

Total source files found: 1


In [21]:
import torch

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "step-0001000-pt/step_0001000.pt"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

state_dict = checkpoint["model"]

print("A3 checkpoint metadata:")
for key in [
    "architecture",
    "parameters",
    "context_length",
    "hidden_size",
    "num_layers",
    "num_heads",
    "ffn_size",
    "prefix_ratio",
    "dropout",
]:
    print(f"{key}: {checkpoint.get(key)}")

print("\nAll checkpoint parameter names:")
for name, tensor in state_dict.items():
    print(f"{name:48s} {tuple(tensor.shape)}")

print(f"\nTotal state-dict entries: {len(state_dict)}")
print(f"Total parameters: {sum(t.numel() for t in state_dict.values()):,}")

A3 checkpoint metadata:
architecture: A3_prefix_lm_hybrid
parameters: 51868672
context_length: 2048
hidden_size: 512
num_layers: 8
num_heads: 8
ffn_size: 2048
prefix_ratio: 0.5
dropout: 0.1

All checkpoint parameter names:
token_embedding.weight                           (50000, 512)
position_embedding.weight                        (2048, 512)
layers.0.ln1.weight                              (512,)
layers.0.ln1.bias                                (512,)
layers.0.attn.in_proj_weight                     (1536, 512)
layers.0.attn.in_proj_bias                       (1536,)
layers.0.attn.out_proj.weight                    (512, 512)
layers.0.attn.out_proj.bias                      (512,)
layers.0.ln2.weight                              (512,)
layers.0.ln2.bias                                (512,)
layers.0.ffn.0.weight                            (2048, 512)
layers.0.ffn.0.bias                              (2048,)
layers.0.ffn.3.weight                            (512, 2048)
layers.0.ffn.3.bi

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "step-0001000-pt/step_0001000.pt"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
state_dict = checkpoint["model"]


class A3Block(nn.Module):
    def __init__(self, d_model=512, n_heads=8, ffn_dim=2048, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )

    def forward(self, x, mask):
        h = self.ln1(x)
        a, _ = self.attn(
            h, h, h, attn_mask=mask, need_weights=False
        )
        x = x + a
        return x + self.ffn(self.ln2(x))


class A3Diagnostic(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(50000, 512)
        self.position_embedding = nn.Embedding(2048, 512)
        self.layers = nn.ModuleList([
            A3Block() for _ in range(8)
        ])
        self.ln_f = nn.LayerNorm(512)

    def forward(self, input_ids):
        _, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device)
        x = self.token_embedding(input_ids)
        x = x + self.position_embedding(positions).unsqueeze(0)

        # Prefix-LM mask, using the checkpoint's prefix ratio.
        prefix_len = min(
            int(seq_len * checkpoint["prefix_ratio"]), seq_len
        )
        mask = torch.full(
            (seq_len, seq_len), float("-inf"), device=input_ids.device
        )
        mask[:prefix_len, :prefix_len] = 0.0
        mask[prefix_len:, :prefix_len] = 0.0
        continuation_len = seq_len - prefix_len
        if continuation_len > 0:
            causal = torch.tril(torch.ones(
                continuation_len, continuation_len, device=input_ids.device
            ))
            mask[prefix_len:, prefix_len:] = torch.where(
                causal.bool(), 0.0, float("-inf")
            )

        for layer in self.layers:
            x = layer(x, mask)
        return self.ln_f(x)


model = A3Diagnostic()

# Strict loading: fails if any key or shape differs.
model.load_state_dict(state_dict, strict=True)
model = model.to(device).eval()

print("Strict checkpoint loading: PASS")

test_ids = torch.randint(0, 50000, (1, 32), device=device)
with torch.no_grad():
    output = model(test_ids)

print("Input shape:", tuple(test_ids.shape))
print("Output shape:", tuple(output.shape))
print("Output finite:", bool(torch.isfinite(output).all()))
print("Diagnostic forward pass: PASS")

Strict checkpoint loading: PASS
Input shape: (1, 32)
Output shape: (1, 32, 512)
Output finite: True
Diagnostic forward pass: PASS


In [24]:

import os
import json
from tokenizers import Tokenizer

DATA_ROOT = "/kaggle/input/datasets/aq1de3/phase3"
TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json"

# Files are directly inside phase3, not inside phase3/Train
required_files = [
    TOKENIZER_PATH,
    os.path.join(DATA_ROOT, "CL_train.json"),
    os.path.join(DATA_ROOT, "IT_train.json"),
    os.path.join(DATA_ROOT, "CL_dev.json"),
    os.path.join(DATA_ROOT, "IT_dev.json"),
    os.path.join(DATA_ROOT, "label_vocab.json"),
]

for path in required_files:
    print(("OK      " if os.path.exists(path) else "MISSING "), path)

assert all(os.path.exists(p) for p in required_files), (
    "A required file is missing. Check the Kaggle Input paths."
)

with open(os.path.join(DATA_ROOT, "label_vocab.json"), "r") as f:
    label_vocab = json.load(f)

print("\nNumber of labels:", len(label_vocab))
print("Labels:", label_vocab)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
print("\nTokenizer vocabulary size:", tokenizer.get_vocab_size())

for token in ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]:
    print(f"{token} ID:", tokenizer.token_to_id(token))

# Inspect the dataset structure and verify text-label alignment
for domain in ["CL", "IT"]:
    path = os.path.join(DATA_ROOT, f"{domain}_train.json")

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"\n--- {domain} TRAIN ---")
    print("Container type:", type(data).__name__)

    if isinstance(data, dict):
        print("Top-level keys:", list(data.keys())[:10])
        first_key = next(iter(data))
        record = data[first_key]
        print("First document ID:", first_key)
    elif isinstance(data, list):
        record = data[0]
        print("Number of records:", len(data))
    else:
        raise TypeError(f"Unexpected dataset structure: {type(data)}")

    print("Record type:", type(record).__name__)

    if isinstance(record, dict):
        print("Record keys:", list(record.keys()))
        texts = record.get("text")
        labels = record.get("labels")

        print("Number of sentences:", len(texts) if texts is not None else None)
        print("Number of labels:", len(labels) if labels is not None else None)

        if texts and labels:
            print("Example sentence:", texts[0][:300])
            print("Example label:", labels[0])

            encoded = tokenizer.encode(texts[0])
            print("Token IDs (first 40):", encoded.ids[:40])
            print("Token count:", len(encoded.ids))
            print("Decoded text:", tokenizer.decode(encoded.ids)[:300])

            assert len(texts) == len(labels), (
                f"{domain}: sentence-label length mismatch"
            )
    else:
        print("First record:", str(record)[:500])

print("\nSmoke test completed. No training performed.")


OK       /kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
OK       /kaggle/input/datasets/aq1de3/phase3/CL_train.json
OK       /kaggle/input/datasets/aq1de3/phase3/IT_train.json
OK       /kaggle/input/datasets/aq1de3/phase3/CL_dev.json
OK       /kaggle/input/datasets/aq1de3/phase3/IT_dev.json
OK       /kaggle/input/datasets/aq1de3/phase3/label_vocab.json

Number of labels: 13
Labels: {'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}

Tokenizer vocabulary size: 50000
[PAD] ID: 0
[UNK] ID: 1
[BOS] ID: 2
[EOS] ID: 3
[MASK] ID: 4

--- CL TRAIN ---
Container type: list
Number of records: 40
Record type: dict
Record keys: ['id', 'text', 'labels']
Number of sentences: 43
Number of labels: 43
Example sentence: Consequent upon the repeal of the MRTP Act this compla

In [25]:

import os
import json
import numpy as np
from collections import Counter
from tokenizers import Tokenizer

DATA_ROOT = "/kaggle/input/datasets/aq1de3/phase3"
TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

with open(os.path.join(DATA_ROOT, "label_vocab.json"), encoding="utf-8") as f:
    label_vocab = json.load(f)

valid_labels = set(label_vocab.keys())
overall_lengths = []

for domain in ["CL", "IT"]:
    for split in ["train", "dev"]:
        path = os.path.join(DATA_ROOT, f"{domain}_{split}.json")
        with open(path, encoding="utf-8") as f:
            documents = json.load(f)

        label_counts = Counter()
        sentence_count = 0
        empty_sentences = 0
        alignment_errors = []
        invalid_labels = []
        document_ids = []
        token_lengths = []

        for doc_idx, doc in enumerate(documents):
            doc_id = doc.get("id", f"doc_{doc_idx}")
            document_ids.append(doc_id)
            texts = doc.get("text")
            labels = doc.get("labels")

            if not isinstance(texts, list) or not isinstance(labels, list):
                alignment_errors.append(
                    f"{doc_id}: text/labels are not lists"
                )
                continue

            if len(texts) != len(labels):
                alignment_errors.append(
                    f"{doc_id}: {len(texts)} sentences vs {len(labels)} labels"
                )
                continue

            for sent_idx, (sentence, label) in enumerate(zip(texts, labels)):
                sentence_count += 1

                if not isinstance(sentence, str) or not sentence.strip():
                    empty_sentences += 1

                if label not in valid_labels:
                    invalid_labels.append((doc_id, sent_idx, label))
                else:
                    label_counts[label] += 1

                if isinstance(sentence, str) and sentence.strip():
                    length = len(tokenizer.encode(sentence).ids)
                    token_lengths.append(length)
                    overall_lengths.append(length)

        duplicates = len(document_ids) - len(set(document_ids))

        print(f"\n{'=' * 55}")
        print(f"{domain} {split.upper()}")
        print(f"Documents: {len(documents)}")
        print(f"Sentences: {sentence_count}")
        print(f"Alignment errors: {len(alignment_errors)}")
        print(f"Invalid labels: {len(invalid_labels)}")
        print(f"Empty sentences: {empty_sentences}")
        print(f"Duplicate document IDs: {duplicates}")

        if token_lengths:
            print(
                "Token lengths (min / median / p95 / p99 / max):",
                np.percentile(token_lengths, [0, 50, 95, 99, 100]).round(1).tolist()
            )
            print("Sentences over 256 tokens:", sum(n > 256 for n in token_lengths))
            print("Sentences over 512 tokens:", sum(n > 512 for n in token_lengths))
            print("Sentences over 1024 tokens:", sum(n > 1024 for n in token_lengths))

        print("Label counts:", dict(label_counts))
        if alignment_errors:
            print("Alignment examples:", alignment_errors[:3])
        if invalid_labels:
            print("Invalid-label examples:", invalid_labels[:3])

        assert not alignment_errors, f"Alignment errors in {domain} {split}"
        assert not invalid_labels, f"Invalid labels in {domain} {split}"

print("\nFULL DATA VALIDATION FINISHED")
if overall_lengths:
    print("Total non-empty sentences tokenized:", len(overall_lengths))
    print("Overall maximum token length:", max(overall_lengths))



CL TRAIN
Documents: 40
Sentences: 10593
Alignment errors: 0
Invalid labels: 0
Empty sentences: 0
Duplicate document IDs: 0
Token lengths (min / median / p95 / p99 / max): [2.0, 33.0, 86.0, 140.0, 863.0]
Sentences over 256 tokens: 18
Sentences over 512 tokens: 3
Sentences over 1024 tokens: 0
Label counts: {'Fact': 2979, 'ArgumentPetitioner': 976, 'ArgumentRespondent': 996, 'RatioOfTheDecision': 2856, 'RulingByPresentCourt': 113, 'RulingByLowerCourt': 438, 'None': 53, 'PrecedentNotReliedUpon': 209, 'PrecedentReliedUpon': 1305, 'Statute': 367, 'Issue': 17, 'Dissent': 284}

CL DEV
Documents: 5
Sentences: 1396
Alignment errors: 0
Invalid labels: 0
Empty sentences: 0
Duplicate document IDs: 0
Token lengths (min / median / p95 / p99 / max): [2.0, 34.0, 88.0, 154.3, 416.0]
Sentences over 256 tokens: 4
Sentences over 512 tokens: 0
Sentences over 1024 tokens: 0
Label counts: {'Fact': 249, 'ArgumentPetitioner': 344, 'None': 2, 'ArgumentRespondent': 236, 'RatioOfTheDecision': 288, 'RulingByPresen

In [26]:

import os
import json

NOTEBOOK_PATH = (
    "/kaggle/input/datasets/aq1de3/phase1-model/"
    "phase1_model_code_extract.ipynb"
)

print("Notebook exists:", os.path.exists(NOTEBOOK_PATH))

if not os.path.exists(NOTEBOOK_PATH):
    print("\nAvailable files in the Phase 1 model input:")
    for root, dirs, files in os.walk("/kaggle/input/datasets/aq1de3/phase1-model"):
        for filename in files:
            print(os.path.join(root, filename))
else:
    with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    cells = notebook.get("cells", [])
    print("Total notebook cells:", len(cells))

    keywords = [
        "class ", "def create_", "architecture", "state_dict",
        "D_MODEL", "N_LAYERS", "N_HEADS", "PREFIX_LEN",
        "GQA", "LORMA", "encoder_layers", "decoder_layers",
        "model.load_state_dict", "torch.save"
    ]

    for i, cell in enumerate(cells):
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)

        hits = [word for word in keywords if word.lower() in source.lower()]
        if hits:
            print(f"\n{'=' * 65}")
            print(f"CELL {i} | matches: {', '.join(hits)}")
            print(source[:3500])
            if len(source) > 3500:
                print("... [cell truncated for display]")


Notebook exists: True
Total notebook cells: 13

CELL 0 | matches: architecture
# Phase 1 model-code extracts (from the uploaded notebook)

These are copied source cells from the supplied notebook, not a verified runnable loader. Do not Run All. The notebook contains multiple architecture variants and benchmark/training cells.

**Compatibility warning:** checkpoint inspection showed A1 keys such as `blocks.0.norm1` (not standard Hugging Face GPT-2 keys); A2 checkpoint tensors are width 512, while the later A2 cell 122 configures width 528. Therefore these cells must be matched against the exact saved checkpoint before use. A3 and A4 also need strict state-dict verification.

This extract intentionally omits pretraining loops and corpus download/training cells.

CELL 1 | matches: architecture
## Original notebook cell 9: A1 architecture/configuration cell (GPT2-based candidate; checkpoint compatibility must be checked)

Copied verbatim from the supplied notebook. Review paths/configurati

In [27]:

import torch

CHECKPOINTS = {
    "A1": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
}

def inspect_checkpoint(name, path):
    print(f"\n{'=' * 60}\n{name}: {path}")
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    if not isinstance(ckpt, dict):
        print("Unexpected checkpoint type:", type(ckpt))
        return

    print("Top-level keys:", list(ckpt.keys())[:20])

    state = None
    for key in ["model", "model_state_dict", "state_dict"]:
        if key in ckpt and isinstance(ckpt[key], dict):
            state = ckpt[key]
            print("State-dict key:", key)
            break

    if state is None:
        print("No recognized state dictionary found.")
        return

    print("Number of tensors:", len(state))
    print("Metadata:")
    for key in ["step", "epoch", "architecture", "hidden_size",
                "num_layers", "num_heads", "d_model", "train_loss"]:
        if key in ckpt:
            print(f"  {key}: {ckpt[key]}")

    print("Representative tensor shapes:")
    for key, tensor in list(state.items())[:12]:
        if hasattr(tensor, "shape"):
            print(f"  {key}: {tuple(tensor.shape)}")

for name, path in CHECKPOINTS.items():
    inspect_checkpoint(name, path)



A1: /kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
Top-level keys: ['step', 'epoch', 'train_loss', 'val_loss', 'model_state_dict', 'optimizer_state_dict']
State-dict key: model_state_dict
Number of tensors: 101
Metadata:
  step: 500
  epoch: 0
  train_loss: 5.366012096405029
Representative tensor shapes:
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512)
  blocks.0.norm1.weight: (512,)
  blocks.0.norm1.bias: (512,)
  blocks.0.attn.in_proj_weight: (1536, 512)
  blocks.0.attn.in_proj_bias: (1536,)
  blocks.0.attn.out_proj.weight: (512, 512)
  blocks.0.attn.out_proj.bias: (512,)
  blocks.0.norm2.weight: (512,)
  blocks.0.norm2.bias: (512,)
  blocks.0.ffn.0.weight: (2048, 512)
  blocks.0.ffn.0.bias: (2048,)

A2: /kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt
Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
State-dict key: model_state_dict
N

In [28]:

import json

NOTEBOOK_PATH = (
    "/kaggle/input/datasets/aq1de3/phase1-model/"
    "phase1_model_code_extract.ipynb"
)

with open(NOTEBOOK_PATH, encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    # Show architecture implementation cells, not training loops.
    markers = [
        "class PrefixLMBlock",
        "class PrefixLMModel",
        "class GQAAttention",
        "class A2Block",
        "class A2GQALM",
        "class EncoderBlock",
        "class DecoderBlock",
        "class EncoderDecoderLM",
        "class GPT2LMHeadModel",
    ]

    found = [m for m in markers if m in source]
    if found:
        print(f"\n{'=' * 70}")
        print(f"CELL {i} | {', '.join(found)}")
        print(source)



CELL 4 | class GQAAttention
# ============================================================
# PHASE 1 — ARCHITECTURE 2
# GQA — UPDATED MODEL WIDTH
# PARAMETER + GPU CHECK
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

print("=" * 70)
print("ARCHITECTURE 2 — GQA UPDATED RESOURCE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

VOCAB_SIZE = Tokenizer.from_file(
    TOKENIZER_PATH
).get_vocab_size()

SEQ_LEN = 2048

D_MODEL = 528
N_LAYERS = 8
N_Q_HEADS = 8
N_KV_HEADS = 2
FFN_DIM = 2048

HEAD_DIM = D_MODEL // N_Q_HEADS
GROUP_SIZE = N_Q_HEADS // N_KV_HEADS

print(f"\nVocabulary       : {VOCAB_SIZE:,}")
print(f"Context length   : {SEQ_LEN:,}")
print(f"Model dimension  : 

In [29]:

import torch

paths = {
    "A1": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
}

for name, path in paths.items():
    print(f"\n{'=' * 24} {name} {'=' * 24}")
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    state_key = next(
        (k for k in ("model", "model_state_dict", "state_dict")
         if k in ckpt and isinstance(ckpt[k], dict)),
        None
    )

    if state_key is None:
        print("No recognized state dictionary. Top-level keys:", list(ckpt))
        continue

    state = ckpt[state_key]
    print("State key:", state_key)
    print("Tensor count:", len(state))

    for key, tensor in state.items():
        if hasattr(tensor, "shape"):
            print(f"{key}: {tuple(tensor.shape)}")
        else:
            print(f"{key}: {type(tensor).__name__}")



======================== A1 ========================
State key: model_state_dict
Tensor count: 101
token_embedding.weight: (50000, 512)
position_embedding.weight: (2048, 512)
blocks.0.norm1.weight: (512,)
blocks.0.norm1.bias: (512,)
blocks.0.attn.in_proj_weight: (1536, 512)
blocks.0.attn.in_proj_bias: (1536,)
blocks.0.attn.out_proj.weight: (512, 512)
blocks.0.attn.out_proj.bias: (512,)
blocks.0.norm2.weight: (512,)
blocks.0.norm2.bias: (512,)
blocks.0.ffn.0.weight: (2048, 512)
blocks.0.ffn.0.bias: (2048,)
blocks.0.ffn.2.weight: (512, 2048)
blocks.0.ffn.2.bias: (512,)
blocks.1.norm1.weight: (512,)
blocks.1.norm1.bias: (512,)
blocks.1.attn.in_proj_weight: (1536, 512)
blocks.1.attn.in_proj_bias: (1536,)
blocks.1.attn.out_proj.weight: (512, 512)
blocks.1.attn.out_proj.bias: (512,)
blocks.1.norm2.weight: (512,)
blocks.1.norm2.bias: (512,)
blocks.1.ffn.0.weight: (2048, 512)
blocks.1.ffn.0.bias: (2048,)
blocks.1.ffn.2.weight: (512, 2048)
blocks.1.ffn.2.bias: (512,)
blocks.2.norm1.weight: (51

In [30]:

import torch

paths = {
    "A1": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
}

for name, path in paths.items():
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print(f"\n{'=' * 18} {name} {'=' * 18}")

    for key, value in ckpt.items():
        if key not in ("model", "model_state_dict", "state_dict",
                       "optimizer_state_dict"):
            if isinstance(value, (str, int, float, bool, type(None))):
                print(f"{key}: {value}")
            elif isinstance(value, dict):
                print(f"{key}: dict with {len(value)} entries")
            else:
                print(f"{key}: {type(value).__name__}")



================== A1 ==================
step: 500
epoch: 0
train_loss: 5.366012096405029
val_loss: None

================== A2 ==================
step: 9000
tokens_processed: 73728000
train_loss: 6.801188290119171
reason: regular_checkpoint
saved_at: 2026-10-05T19:17:55.016904

================== A3 ==================
step: 1000
tokens_seen: 512000
architecture: A3_prefix_lm_hybrid
parameters: 51868672
context_length: 2048
hidden_size: 512
num_layers: 8
num_heads: 8
ffn_size: 2048
prefix_ratio: 0.5
dropout: 0.1
learning_rate: 0.0001
weight_decay: 0.1

================== A4 ==================
step: 22000
tokens_processed: 180096704
train_loss: 0.1944424146786332
reason: regular_checkpoint
saved_at: 2026-10-05T15:18:59.981007


In [31]:

import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith((".ipynb", ".py")):
            print(os.path.join(root, file))


/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb


In [32]:

import json

path = "/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("Total cells:", len(nb["cells"]))

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    print(f"\n{'=' * 18} CELL {i} {'=' * 18}")
    print(source[:700].replace("\n", " "))


Total cells: 13

================== CELL 0 ==================
# Phase 1 model-code extracts (from the uploaded notebook)  These are copied source cells from the supplied notebook, not a verified runnable loader. Do not Run All. The notebook contains multiple architecture variants and benchmark/training cells.  **Compatibility warning:** checkpoint inspection showed A1 keys such as `blocks.0.norm1` (not standard Hugging Face GPT-2 keys); A2 checkpoint tensors are width 512, while the later A2 cell 122 configures width 528. Therefore these cells must be matched against the exact saved checkpoint before use. A3 and A4 also need strict state-dict verification.  This extract intentionally omits pretraining loops and corpus download/training cells.

================== CELL 1 ==================
## Original notebook cell 9: A1 architecture/configuration cell (GPT2-based candidate; checkpoint compatibility must be checked)  Copied verbatim from the supplied notebook. Review paths/configuration 

In [33]:

import json

path = "/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i in [9, 10]:
    cell = nb["cells"][i]
    source = "".join(cell.get("source", []))
    print(f"\n{'=' * 20} CELL {i} {'=' * 20}")
    print(source)



==================== CELL 9 ====================
## Original notebook cell 27: Original checkpoint save/load helpers

Copied verbatim from the supplied notebook. Review paths/configuration before execution.


==================== CELL 10 ====================
# ============================================================
# PHASE 1 — TRAINING INFRASTRUCTURE
# STEP 4: CHECKPOINT SYSTEM
# ============================================================

import os
import torch

CHECKPOINT_ROOT = "/kaggle/working/phase1_checkpoints"

os.makedirs(
    CHECKPOINT_ROOT,
    exist_ok=True
)


def save_checkpoint(
    model,
    optimizer,
    step,
    epoch,
    train_loss,
    val_loss,
    model_name,
    extra_state=None
):
    """
    Save a complete training checkpoint.

    Includes:
    - model weights
    - optimizer state
    - training step
    - epoch
    - train loss
    - validation loss
    - optional extra state
    """

    checkpoint = {
        "step": step,
        "epoch": epoc

In [34]:

import torch

paths = {
    "A1": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
}

for name, path in paths.items():
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    state_key = next(
        k for k in ("model_state_dict", "model", "state_dict")
        if k in ckpt and isinstance(ckpt[k], dict)
    )
    state = ckpt[state_key]

    print(f"\n{name}")
    print("Metadata:", {
        k: v for k, v in ckpt.items()
        if k not in ("model_state_dict", "model", "state_dict",
                     "optimizer_state_dict")
        and isinstance(v, (str, int, float, bool, type(None)))
    })

    for prefix in ("blocks.", "layers.", "encoder_layers.", "decoder_layers."):
        indices = sorted({
            int(k.split(".")[1])
            for k in state
            if k.startswith(prefix)
            and len(k.split(".")) > 2
            and k.split(".")[1].isdigit()
        })
        if indices:
            print(prefix, "indices:", indices)



A1
Metadata: {'step': 500, 'epoch': 0, 'train_loss': 5.366012096405029, 'val_loss': None}
blocks. indices: [0, 1, 2, 3, 4, 5, 6, 7]

A2
Metadata: {'step': 9000, 'tokens_processed': 73728000, 'train_loss': 6.801188290119171, 'reason': 'regular_checkpoint', 'saved_at': '2026-10-05T19:17:55.016904'}
layers. indices: [0, 1, 2, 3, 4, 5, 6, 7, 8]

A3
Metadata: {'step': 1000, 'tokens_seen': 512000, 'architecture': 'A3_prefix_lm_hybrid', 'parameters': 51868672, 'context_length': 2048, 'hidden_size': 512, 'num_layers': 8, 'num_heads': 8, 'ffn_size': 2048, 'prefix_ratio': 0.5, 'dropout': 0.1, 'learning_rate': 0.0001, 'weight_decay': 0.1}
layers. indices: [0, 1, 2, 3, 4, 5, 6, 7]

A4
Metadata: {'step': 22000, 'tokens_processed': 180096704, 'train_loss': 0.1944424146786332, 'reason': 'regular_checkpoint', 'saved_at': '2026-10-05T15:18:59.981007'}
encoder_layers. indices: [0, 1, 2, 3, 4, 5]
decoder_layers. indices: [0, 1, 2, 3, 4, 5]


In [36]:

import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if "phase1_model" in file.lower():
            print(os.path.join(root, file))


/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb


In [37]:

import json

path = "/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("This is the extracted notebook with", len(nb["cells"]), "cells.")
print("The original notebook has been uploaded to this chat,")
print("but it must also be attached in Kaggle to inspect it there.")


This is the extracted notebook with 13 cells.
The original notebook has been uploaded to this chat,
but it must also be attached in Kaggle to inspect it there.


In [38]:

import os

roots = [
    "/kaggle/input/datasets/aq1de3",
    "/kaggle/working",
]

for root in roots:
    print(f"\n=== {root} ===")
    if not os.path.exists(root):
        print("Not found")
        continue

    for current, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in {".git", "__pycache__"}
        ]

        for name in files:
            low = name.lower()
            if (
                low.endswith(".ipynb")
                or low.endswith(".pt")
                or "phase1" in low and low.endswith(".py")
            ):
                print(os.path.join(current, name))



=== /kaggle/input/datasets/aq1de3 ===
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt
/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt
/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt

=== /kaggle/working ===
/kaggle/working/.virtual_documents/__notebook_source__.ipynb


In [39]:

import os
import torch

ROOT = "/kaggle/input/datasets/aq1de3"

checkpoints = {
    "A1": os.path.join(
        ROOT, "architecture1-sustained-step-500-pt",
        "architecture1_sustained_step_500.pt"
    ),
    "A2": os.path.join(
        ROOT, "a2-step-9000-pt", "a2_step_9000.pt"
    ),
    "A3": os.path.join(
        ROOT, "step-0001000-pt", "step_0001000.pt"
    ),
    "A4": os.path.join(
        ROOT, "a4-step-22000-pt", "a4_step_22000.pt"
    ),
}

for name, path in checkpoints.items():
    print(f"\n{'=' * 16} {name} {'=' * 16}")
    print("Exists:", os.path.isfile(path))
    if not os.path.isfile(path):
        continue

    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print("Top-level keys:", list(ckpt.keys()))

    for key, value in ckpt.items():
        if isinstance(value, dict):
            tensors = [
                (k, tuple(v.shape))
                for k, v in value.items()
                if torch.is_tensor(v)
            ]
            print(f"{key}: {len(tensors)} direct tensors")
            print("First 8:", tensors[:8])
        elif isinstance(value, (str, int, float, bool)):
            print(f"{key}: {value}")



================ A1 ================
Exists: True
Top-level keys: ['step', 'epoch', 'train_loss', 'val_loss', 'model_state_dict', 'optimizer_state_dict']
step: 500
epoch: 0
train_loss: 5.366012096405029
model_state_dict: 101 direct tensors
First 8: [('token_embedding.weight', (50000, 512)), ('position_embedding.weight', (2048, 512)), ('blocks.0.norm1.weight', (512,)), ('blocks.0.norm1.bias', (512,)), ('blocks.0.attn.in_proj_weight', (1536, 512)), ('blocks.0.attn.in_proj_bias', (1536,)), ('blocks.0.attn.out_proj.weight', (512, 512)), ('blocks.0.attn.out_proj.bias', (512,))]
optimizer_state_dict: 0 direct tensors
First 8: []

================ A2 ================
Exists: True
Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
model_state_dict: 113 direct tensors
First 8: [('output_bias', (50000,)), ('token_embedding.weight', (50000, 512)), ('position_embedding.weight', (2048, 512)), ('layers.0.norm1.weight', (512,)), ('layers.0.norm1.bias

In [40]:
import os

for root in [
    "/kaggle/input/datasets/aq1de3/phase1-model",
    "/kaggle/working"
]:
    print(f"\n{root}")
    if os.path.exists(root):
        for name in os.listdir(root):
            print(" ", name)
    else:
        print(" Directory not found")


/kaggle/input/datasets/aq1de3/phase1-model
  phase1_model_code_extract.ipynb

/kaggle/working
  CS781_Phase3
  .virtual_documents


In [41]:

import os
import torch

ROOT = "/kaggle/input/datasets/aq1de3"

CHECKPOINTS = {
    "A1": os.path.join(
        ROOT, "architecture1-sustained-step-500-pt",
        "architecture1_sustained_step_500.pt"
    ),
    "A2": os.path.join(
        ROOT, "a2-step-9000-pt", "a2_step_9000.pt"
    ),
    "A3": os.path.join(
        ROOT, "step-0001000-pt", "step_0001000.pt"
    ),
    "A4": os.path.join(
        ROOT, "a4-step-22000-pt", "a4_step_22000.pt"
    ),
}

for name, path in CHECKPOINTS.items():
    print(f"\n{'=' * 65}\n{name}: {path}")
    assert os.path.isfile(path), f"Missing checkpoint: {path}"

    checkpoint = torch.load(
        path, map_location="cpu", weights_only=True
    )

    state = checkpoint.get(
        "model_state_dict",
        checkpoint.get("model", checkpoint)
    )

    print("Checkpoint step:", checkpoint.get("step", "not recorded"))
    print("Tensor count:", len(state))

    for key, tensor in list(state.items())[:12]:
        if torch.is_tensor(tensor):
            print(f"  {key}: {tuple(tensor.shape)}")

    print("First and last parameter names:")
    keys = list(state.keys())
    print("  first:", keys[:3])
    print("  last: ", keys[-3:])



A1: /kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
Checkpoint step: 500
Tensor count: 101
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512)
  blocks.0.norm1.weight: (512,)
  blocks.0.norm1.bias: (512,)
  blocks.0.attn.in_proj_weight: (1536, 512)
  blocks.0.attn.in_proj_bias: (1536,)
  blocks.0.attn.out_proj.weight: (512, 512)
  blocks.0.attn.out_proj.bias: (512,)
  blocks.0.norm2.weight: (512,)
  blocks.0.norm2.bias: (512,)
  blocks.0.ffn.0.weight: (2048, 512)
  blocks.0.ffn.0.bias: (2048,)
First and last parameter names:
  first: ['token_embedding.weight', 'position_embedding.weight', 'blocks.0.norm1.weight']
  last:  ['final_norm.weight', 'final_norm.bias', 'lm_head.weight']

A2: /kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt
Checkpoint step: 9000
Tensor count: 113
  output_bias: (50000,)
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512)
  layers.0.norm1.

In [45]:
import torch

candidate_names = [
    "GQALanguageModel",
    "A2GQALM",
    "A2Model",
    "PrefixLMModel",
    "EncoderDecoderLM",
]

for name in candidate_names:
    obj = globals().get(name)
    print(
        f"{name}:",
        "defined" if obj is not None else "not defined",
        f"| type={type(obj).__name__}" if obj is not None else ""
    )

GQALanguageModel: not defined 
A2GQALM: not defined 
A2Model: not defined 
PrefixLMModel: not defined 
EncoderDecoderLM: not defined 


In [47]:

from pathlib import Path
import json

matches = list(
    Path("/kaggle/input").rglob("notebook1d0fcbb0df (3).ipynb")
)

print("Matching notebooks:", len(matches))
for i, path in enumerate(matches):
    print(i, path)

if matches:
    notebook_path = matches[0]
    nb = json.loads(notebook_path.read_text(encoding="utf-8"))
    print("\nSelected:", notebook_path)
    print("Total cells:", len(nb["cells"]))

    terms = (
        "class ", "GPT2LMHeadModel", "GQALanguageModel",
        "PrefixLMModel", "EncoderDecoderLM",
        "A2GQALM", "model_state_dict"
    )

    for i, cell in enumerate(nb["cells"]):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        found = [term for term in terms if term in source]

        if found:
            print(f"\n--- Cell {i} | matches: {found} ---")
            lines = source.splitlines()
            for j, line in enumerate(lines):
                if any(term in line for term in terms):
                    start = max(0, j - 1)
                    end = min(len(lines), j + 3)
                    print("\n".join(
                        f"{k + 1}: {lines[k]}"
                        for k in range(start, end)
                    ))
else:
    print("Notebook not found. Check the exact filename in the Input panel.")


Matching notebooks: 1
0 /kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb

Selected: /kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb
Total cells: 160

--- Cell 5 | matches: ['GPT2LMHeadModel'] ---
6: import time
7: from transformers import GPT2Config, GPT2LMHeadModel
8: 
9: print("=" * 70)
38: 
39: model = GPT2LMHeadModel(config).to(device)
40: 
41: parameters = sum(p.numel() for p in model.parameters())

--- Cell 8 | matches: ['GPT2LMHeadModel'] ---
12: from tokenizers import Tokenizer
13: from transformers import GPT2Config, GPT2LMHeadModel
14: 
15: print("=" * 70)
71: 
72: model = GPT2LMHeadModel(config).to(device)
73: 
74: parameter_count = sum(

--- Cell 9 | matches: ['GPT2LMHeadModel'] ---
8: from tokenizers import Tokenizer
9: from transformers import GPT2Config, GPT2LMHeadModel
10: 
11: print("=" * 70)
58: # ------------------------------------------------------------
59: model = GPT2LMHeadModel(config)
60: 
61: # ------------------------

In [48]:

import json
from pathlib import Path

p = Path(
    "/kaggle/input/datasets/aq1de3/notebook1/"
    "notebook1d0fcbb0df (3).ipynb"
)
nb = json.loads(p.read_text(encoding="utf-8"))

# Original notebook cells use zero-based indices.
inspect_cells = [13, 14, 15, 18, 22, 139, 141, 142, 143, 147, 150, 152]

for i in inspect_cells:
    if i >= len(nb["cells"]):
        continue

    cell = nb["cells"][i]
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    print(f"\n{'=' * 25} CELL {i} {'=' * 25}")
    print("Total source lines:", len(lines))

    # Print class declarations and configuration assignments first.
    for j, line in enumerate(lines):
        if (
            line.lstrip().startswith("class ")
            or any(key in line for key in [
                "D_MODEL =", "D_MODEL=", "N_LAYERS =",
                "N_LAYERS=", "self.layers =", "self.blocks =",
                "self.final_norm", "self.ln_f",
                "self.output_bias", "self.lm_head",
                "self.encoder_layers", "self.decoder_layers",
                "self.encoder_pos", "self.decoder_pos"
            ])
        ):
            print(f"{j + 1}: {line[:180]}")



========================= CELL 13 =========================
Total source lines: 371
30: D_MODEL = 528
31: N_LAYERS = 8
53: class GQAAttention(nn.Module):
184: class GQABlock(nn.Module):
224: class GQALanguageModel(nn.Module):
240:         self.blocks = nn.ModuleList(
247:         self.final_norm = nn.LayerNorm(
252:         self.lm_head = nn.Linear(
258:         self.lm_head.weight = (
280:         x = self.final_norm(x)
282:         return self.lm_head(x)

========================= CELL 14 =========================
Total source lines: 477
30: D_MODEL = 528
31: N_LAYERS = 8
51: class GQAAttention(nn.Module):
182: class GQABlock(nn.Module):
222: class GQALanguageModel(nn.Module):
238:         self.blocks = nn.ModuleList(
245:         self.final_norm = nn.LayerNorm(
249:         self.lm_head = nn.Linear(
256:         self.lm_head.weight = (
321:         x = self.final_norm(x)
323:         logits = self.lm_head(x)

========================= CELL 15 =========================
Total source 

In [49]:

import json
from pathlib import Path

p = Path(
    "/kaggle/input/datasets/aq1de3/notebook1/"
    "notebook1d0fcbb0df (3).ipynb"
)
nb = json.loads(p.read_text(encoding="utf-8"))

terms = [
    "D_MODEL = 512",
    "D_MODEL=512",
    "d_model=512",
    "d_model = 512",
    "self.layers = nn.ModuleList",
    "self.encoder_layers = nn.ModuleList",
    "self.decoder_layers = nn.ModuleList",
    "self.output_bias",
]

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    lines = "".join(cell.get("source", [])).splitlines()
    hits = [
        (j + 1, line.strip())
        for j, line in enumerate(lines)
        if any(term in line for term in terms)
    ]

    if hits:
        print(f"\n--- Cell {i} ---")
        for line_no, line in hits:
            print(f"{line_no}: {line}")



--- Cell 16 ---
14: D_MODEL = 512

--- Cell 21 ---
17: D_MODEL = 512

--- Cell 42 ---
16: D_MODEL = 512

--- Cell 44 ---
25: D_MODEL = 512

--- Cell 70 ---
6: d_model=512,

--- Cell 71 ---
7: D_MODEL = 512

--- Cell 77 ---
11: D_MODEL = 512


In [50]:

import json
from pathlib import Path

p = Path(
    "/kaggle/input/datasets/aq1de3/notebook1/"
    "notebook1d0fcbb0df (3).ipynb"
)
nb = json.loads(p.read_text(encoding="utf-8"))

inspect_cells = [16, 21, 42, 44, 70, 71, 77]

terms = [
    "class ", "D_MODEL", "N_LAYERS", "N_HEADS",
    "N_Q_HEADS", "N_KV_HEADS", "nn.ModuleList",
    "nn.MultiheadAttention", "q_proj", "k_proj", "v_proj",
    "encoder_layers", "decoder_layers", "model =",
    "model1 =", "model2 =", "model3 =", "model4 =",
    "load_state_dict", "final_norm", "ln_f",
    "output_bias", "encoder_pos", "decoder_pos"
]

for i in inspect_cells:
    cell = nb["cells"][i]
    if cell.get("cell_type") != "code":
        continue

    lines = "".join(cell.get("source", [])).splitlines()
    print(f"\n{'=' * 20} CELL {i} {'=' * 20}")

    for j, line in enumerate(lines):
        if any(term in line for term in terms):
            print(f"{j + 1}: {line[:200]}")



==================== CELL 16 ====================
14: D_MODEL = 512
15: N_LAYERS = 8
16: N_HEADS = 8
38: print(f"Layers          : {N_LAYERS}")
39: print(f"Model dimension : {D_MODEL}")
40: print(f"Attention heads : {N_HEADS}")
42: print(f"Head dimension  : {D_MODEL // N_HEADS}")

==================== CELL 21 ====================
17: D_MODEL = 512
20: N_HEADS = 8
39: print(f"Model dimension       : {D_MODEL}")
40: print(f"Attention heads       : {N_HEADS}")
41: print(f"Head dimension        : {D_MODEL // N_HEADS}")

==================== CELL 42 ====================
16: D_MODEL = 512
17: N_LAYERS = 8
18: N_HEADS = 8
38:     n_embd=D_MODEL,
39:     n_layer=N_LAYERS,
40:     n_head=N_HEADS,
45: model = GPT2LMHeadModel(config).to(device)
147: resumed_model = GPT2LMHeadModel(config).to(device)
164: resumed_model.load_state_dict(
168: resumed_optimizer.load_state_dict(

==================== CELL 44 ====================
25: D_MODEL = 512
26: N_LAYERS = 8
27: N_HEADS = 8
79:     n_embd=D_MODE

In [51]:

import json
from pathlib import Path

p = Path(
    "/kaggle/input/datasets/aq1de3/notebook1/"
    "notebook1d0fcbb0df (3).ipynb"
)
nb = json.loads(p.read_text(encoding="utf-8"))

for i in [16, 21, 70]:
    cell = nb["cells"][i]
    source = "".join(cell.get("source", []))

    print(f"\n{'=' * 25} CELL {i} {'=' * 25}")
    print(source)



========================= CELL 16 =========================
# ============================================================
# ARCHITECTURE 3 — PREFIX-LM HYBRID
# STEP 1: CONFIGURATION
# ============================================================

import torch

# -----------------------------
# Model configuration
# -----------------------------
VOCAB_SIZE = 50_000
MAX_SEQ_LEN = 2_048

D_MODEL = 512
N_LAYERS = 8
N_HEADS = 8
FFN_DIM = 2_048

DROPOUT = 0.0

# Prefix-LM setting
# The first part of the sequence is treated as the prefix.
PREFIX_RATIO = 0.5
PREFIX_LEN = int(MAX_SEQ_LEN * PREFIX_RATIO)

# Special tokens from our tokenizer
PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3
MASK_ID = 4

print("Architecture 3: Prefix-LM Hybrid")
print("=" * 50)
print(f"Vocabulary size : {VOCAB_SIZE:,}")
print(f"Context length  : {MAX_SEQ_LEN:,}")
print(f"Prefix length   : {PREFIX_LEN:,}")
print(f"Layers          : {N_LAYERS}")
print(f"Model dimension : {D_MODEL}")
print(f"Attention heads : {N_HEADS}")


In [52]:

import json
from pathlib import Path

p = Path(
    "/kaggle/input/datasets/aq1de3/notebook1/"
    "notebook1d0fcbb0df (3).ipynb"
)
nb = json.loads(p.read_text(encoding="utf-8"))

terms = [
    "def build_architecture_1",
    "build_architecture_1 =",
    "architecture1_sustained_step_500.pt",
    "architecture1_sustained",
    "step_0001000.pt",
    "a2_step_9000.pt",
    "a4_step_22000.pt",
]

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))
    hits = [t for t in terms if t in src]

    if hits:
        print(f"\n--- CELL {i} | {hits} ---")
        lines = src.splitlines()
        for j, line in enumerate(lines):
            if any(t in line for t in terms):
                start = max(0, j - 5)
                end = min(len(lines), j + 9)
                print("\n".join(
                    f"{k + 1}: {lines[k]}"
                    for k in range(start, end)
                ))



--- CELL 29 | ['architecture1_sustained'] ---
14: 
15: SUSTAINED_STEPS = 500
16: TRAIN_SEQ_LEN = 2048
17: CHECKPOINT_EVERY = 250
18: 
19: MODEL_NAME = "architecture1_sustained"
20: 
21: # ------------------------------------------------------------
22: # Fresh optimizer
23: # ------------------------------------------------------------
24: 
25: model.train()
26: 
27: optimizer_sustained = torch.optim.AdamW(


In [53]:

import json
from pathlib import Path

p = Path(
    "/kaggle/input/datasets/aq1de3/notebook1/"
    "notebook1d0fcbb0df (3).ipynb"
)
nb = json.loads(p.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))
    lines = src.splitlines()

    matches = [
        (j, line.strip())
        for j, line in enumerate(lines)
        if (
            "build_architecture" in line
            or "architecture1_sustained_step_500" in line
            or "torch.save(" in line
        )
    ]

    if matches:
        print(f"\n{'=' * 20} CELL {i} {'=' * 20}")
        for j, line in matches:
            print(f"{j + 1}: {line}")
            if "build_architecture" in line:
                start = max(0, j - 8)
                end = min(len(lines), j + 15)
                print("\n".join(
                    f"{k + 1}: {lines[k]}"
                    for k in range(start, end)
                ))



==================== CELL 27 ====================
57: torch.save(

==================== CELL 35 ====================
186: torch.save(

==================== CELL 42 ====================
126: torch.save(

==================== CELL 44 ====================
303: torch.save(

==================== CELL 45 ====================
380: torch.save(

==================== CELL 70 ====================
3: model = build_architecture_1(
1: # Architecture 1: Decoder-only GPT-style model
2: 
3: model = build_architecture_1(
4:     vocab_size=50_000,
5:     seq_len=2048,
6:     d_model=512,
7:     n_layers=8,
8:     n_heads=8,
9:     ffn_dim=2048,
10:     dropout=0.0,
11: ).cuda()
12: 
13: total_params = sum(p.numel() for p in model.parameters())
14: 
15: print(f"Parameters: {total_params:,}")
16: print(f"Parameters (M): {total_params / 1e6:.2f}M")
17: print(f"Device: {next(model.parameters()).device}")

==================== CELL 94 ====================
330: torch.save(

==================== CELL 96 ======

In [54]:

terms = (
    "def build_architecture_1",
    "build_architecture_1 =",
    "def architecture1",
    "architecture1_sustained",
)

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    lines = "".join(cell.get("source", [])).splitlines()
    hits = [
        j for j, line in enumerate(lines)
        if any(term in line for term in terms)
    ]

    if hits:
        print(f"\n{'=' * 18} CELL {i}: DEFINITIONS / REFERENCES {'=' * 18}")
        shown = set()
        for j in hits:
            for k in range(max(0, j - 10), min(len(lines), j + 22)):
                if k not in shown:
                    print(f"{k + 1}: {lines[k]}")
                    shown.add(k)
            print("-" * 50)

# Inspect checkpoint-save code and its surrounding metadata.
for i in [27, 29, 70]:
    if i >= len(nb["cells"]) or nb["cells"][i].get("cell_type") != "code":
        continue
    lines = "".join(nb["cells"][i].get("source", [])).splitlines()
    if any("torch.save(" in line for line in lines):
        print(f"\n{'=' * 18} CELL {i}: SAVE CONTEXT {'=' * 18}")
        for j, line in enumerate(lines):
            if "torch.save(" in line:
                start = max(0, j - 12)
                end = min(len(lines), j + 22)
                print("\n".join(
                    f"{k + 1}: {lines[k]}" for k in range(start, end)
                ))



================== CELL 29: DEFINITIONS / REFERENCES ==================
9: import torch
10: 
11: # ------------------------------------------------------------
12: # Settings
13: # ------------------------------------------------------------
14: 
15: SUSTAINED_STEPS = 500
16: TRAIN_SEQ_LEN = 2048
17: CHECKPOINT_EVERY = 250
18: 
19: MODEL_NAME = "architecture1_sustained"
20: 
21: # ------------------------------------------------------------
22: # Fresh optimizer
23: # ------------------------------------------------------------
24: 
25: model.train()
26: 
27: optimizer_sustained = torch.optim.AdamW(
28:     model.parameters(),
29:     lr=3e-4,
30:     betas=(0.9, 0.95),
31:     weight_decay=0.1
32: )
33: 
34: # ------------------------------------------------------------
35: # GPU cleanup
36: # ------------------------------------------------------------
37: 
38: if torch.cuda.is_available():
39:     torch.cuda.empty_cache()
40:     torch.cuda.reset_peak_memory_stats()
---------------

In [55]:

import torch

path = (
    "/kaggle/input/datasets/aq1de3/"
    "architecture1-sustained-step-500-pt/"
    "architecture1_sustained_step_500.pt"
)

ckpt = torch.load(path, map_location="cpu", weights_only=False)

print("Checkpoint type:", type(ckpt).__name__)
if isinstance(ckpt, dict):
    print("Top-level keys:", list(ckpt.keys()))

    for key, value in ckpt.items():
        if key != "model_state_dict" and isinstance(value, (str, int, float, bool, type(None))):
            print(f"{key}: {value}")

    state = ckpt.get("model_state_dict")
    if state is None:
        state = ckpt.get("state_dict")
    if state is None and all(isinstance(v, torch.Tensor) for v in ckpt.values()):
        state = ckpt

    if isinstance(state, dict):
        print("\nTensor count:", sum(torch.is_tensor(v) for v in state.values()))
        print("First 30 parameter names and shapes:")
        for name, value in list(state.items())[:30]:
            if torch.is_tensor(value):
                print(name, tuple(value.shape))


Checkpoint type: dict
Top-level keys: ['step', 'epoch', 'train_loss', 'val_loss', 'model_state_dict', 'optimizer_state_dict']
step: 500
epoch: 0
train_loss: 5.366012096405029
val_loss: None

Tensor count: 101
First 30 parameter names and shapes:
token_embedding.weight (50000, 512)
position_embedding.weight (2048, 512)
blocks.0.norm1.weight (512,)
blocks.0.norm1.bias (512,)
blocks.0.attn.in_proj_weight (1536, 512)
blocks.0.attn.in_proj_bias (1536,)
blocks.0.attn.out_proj.weight (512, 512)
blocks.0.attn.out_proj.bias (512,)
blocks.0.norm2.weight (512,)
blocks.0.norm2.bias (512,)
blocks.0.ffn.0.weight (2048, 512)
blocks.0.ffn.0.bias (2048,)
blocks.0.ffn.2.weight (512, 2048)
blocks.0.ffn.2.bias (512,)
blocks.1.norm1.weight (512,)
blocks.1.norm1.bias (512,)
blocks.1.attn.in_proj_weight (1536, 512)
blocks.1.attn.in_proj_bias (1536,)
blocks.1.attn.out_proj.weight (512, 512)
blocks.1.attn.out_proj.bias (512,)
blocks.1.norm2.weight (512,)
blocks.1.norm2.bias (512,)
blocks.1.ffn.0.weight (2048, 

In [56]:

import ast

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))
    try:
        tree = ast.parse(src)
    except SyntaxError:
        continue

    definitions = [
        node for node in ast.walk(tree)
        if isinstance(node, (ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef))
        and (
            isinstance(node, ast.ClassDef)
            or any(term in node.name.lower() for term in (
                "build", "architecture", "model"
            ))
        )
    ]

    if definitions:
        print(f"\nCELL {i}")
        for node in definitions:
            print(f"  {type(node).__name__}: {node.name}, line {node.lineno}")

    relevant = [
        (j + 1, line.strip())
        for j, line in enumerate(src.splitlines())
        if any(term in line for term in (
            "self.blocks", "self.final_norm", "self.lm_head",
            "in_proj_weight", "ffn.0", "D_MODEL = 512"
        ))
    ]
    if relevant:
        print("  Relevant lines:")
        for line_no, line in relevant[:20]:
            print(f"    {line_no}: {line}")



CELL 13
  ClassDef: GQAAttention, line 53
  ClassDef: GQABlock, line 184
  ClassDef: GQALanguageModel, line 224
  Relevant lines:
    240: self.blocks = nn.ModuleList(
    247: self.final_norm = nn.LayerNorm(
    252: self.lm_head = nn.Linear(
    258: self.lm_head.weight = (
    277: for block in self.blocks:
    280: x = self.final_norm(x)
    282: return self.lm_head(x)

CELL 14
  ClassDef: GQAAttention, line 51
  ClassDef: GQABlock, line 182
  ClassDef: GQALanguageModel, line 222
  Relevant lines:
    238: self.blocks = nn.ModuleList(
    245: self.final_norm = nn.LayerNorm(
    249: self.lm_head = nn.Linear(
    256: self.lm_head.weight = (
    318: for block in self.blocks:
    321: x = self.final_norm(x)
    323: logits = self.lm_head(x)

CELL 15
  ClassDef: GQAAttention, line 65
  ClassDef: GQABlock, line 196
  ClassDef: GQALanguageModel, line 236
  Relevant lines:
    252: self.blocks = nn.ModuleList(
    259: self.final_norm = nn.LayerNorm(
    263: self.lm_head = nn.Linear(

In [57]:

state = ckpt["model_state_dict"]

print("Embedding/head:")
for name in ("token_embedding.weight", "lm_head.weight"):
    print(name, tuple(state[name].shape))

print(
    "Same tensor storage in checkpoint:",
    state["token_embedding.weight"].data_ptr()
    == state["lm_head.weight"].data_ptr()
)

print("\nFinal parameter names:")
for name in state:
    if (
        name.startswith("blocks.7.")
        or name.startswith("final_norm.")
        or name.startswith("lm_head.")
    ):
        print(name, tuple(state[name].shape))


Embedding/head:
token_embedding.weight (50000, 512)
lm_head.weight (50000, 512)
Same tensor storage in checkpoint: True

Final parameter names:
blocks.7.norm1.weight (512,)
blocks.7.norm1.bias (512,)
blocks.7.attn.in_proj_weight (1536, 512)
blocks.7.attn.in_proj_bias (1536,)
blocks.7.attn.out_proj.weight (512, 512)
blocks.7.attn.out_proj.bias (512,)
blocks.7.norm2.weight (512,)
blocks.7.norm2.bias (512,)
blocks.7.ffn.0.weight (2048, 512)
blocks.7.ffn.0.bias (2048,)
blocks.7.ffn.2.weight (512, 2048)
blocks.7.ffn.2.bias (512,)
final_norm.weight (512,)
final_norm.bias (512,)
lm_head.weight (50000, 512)


In [58]:

for i in [13, 14, 15]:
    print(f"\n{'=' * 24} CELL {i} {'=' * 24}")
    lines = "".join(nb["cells"][i]["source"]).splitlines()

    # Show attention, block, and model definitions.
    for start, end in [(45, 145), (175, 290)]:
        if start < len(lines):
            print(f"\n--- lines {start + 1} to {min(end, len(lines))} ---")
            for j in range(start, min(end, len(lines))):
                print(f"{j + 1}: {lines[j]}")



======================== CELL 13 ========================

--- lines 46 to 145 ---
46: print(f"Q heads per KV   : {GROUP_SIZE}")
47: print(f"FFN dimension    : {FFN_DIM}")
48: 
49: # ============================================================
50: # GQA ATTENTION
51: # ============================================================
52: 
53: class GQAAttention(nn.Module):
54: 
55:     def __init__(
56:         self,
57:         d_model,
58:         n_q_heads,
59:         n_kv_heads,
60:         max_seq_len,
61:     ):
62:         super().__init__()
63: 
64:         self.d_model = d_model
65:         self.n_q_heads = n_q_heads
66:         self.n_kv_heads = n_kv_heads
67:         self.head_dim = d_model // n_q_heads
68:         self.group_size = n_q_heads // n_kv_heads
69: 
70:         self.q_proj = nn.Linear(
71:             d_model,
72:             d_model,
73:         )
74: 
75:         self.k_proj = nn.Linear(
76:             d_model,
77:             n_kv_heads * self.head_dim,
78:     

In [59]:

i = 18
src = "".join(nb["cells"][i]["source"]).splitlines()

print(f"Cell {i}: Prefix-LM implementation")
for j, line in enumerate(src):
    print(f"{j + 1}: {line}")


Cell 18: Prefix-LM implementation
1: # ============================================================
2: # ARCHITECTURE 3 — PREFIX-LM HYBRID
3: # STEP 3: MODEL DEFINITION + PARAMETER CHECK
4: # ============================================================
5: 
6: import math
7: import torch
8: import torch.nn as nn
9: import torch.nn.functional as F
10: 
11: # ------------------------------------------------------------
12: # Prefix-LM Transformer Block
13: # ------------------------------------------------------------
14: 
15: class PrefixLMBlock(nn.Module):
16: 
17:     def __init__(self, d_model, n_heads, ffn_dim, dropout=0.0):
18:         super().__init__()
19: 
20:         self.norm1 = nn.LayerNorm(d_model)
21:         self.attn = nn.MultiheadAttention(
22:             embed_dim=d_model,
23:             num_heads=n_heads,
24:             dropout=dropout,
25:             batch_first=True
26:         )
27: 
28:         self.norm2 = nn.LayerNorm(d_model)
29: 
30:         self.ffn = nn.Se

In [60]:

import ast

terms = [
    "in_proj_weight",
    "nn.MultiheadAttention",
    "class GPTBlock",
    "class TransformerBlock",
    "class CausalSelfAttention",
    "build_architecture_1",
]

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))
    found = [term for term in terms if term in src]

    if found:
        print(f"\nCELL {i} | Matches: {found}")
        lines = src.splitlines()
        for j, line in enumerate(lines):
            if any(term in line for term in found):
                start = max(0, j - 5)
                end = min(len(lines), j + 9)
                print("\n".join(
                    f"{k + 1}: {lines[k]}"
                    for k in range(start, end)
                ))
                print("-" * 40)



CELL 18 | Matches: ['nn.MultiheadAttention']
16: 
17:     def __init__(self, d_model, n_heads, ffn_dim, dropout=0.0):
18:         super().__init__()
19: 
20:         self.norm1 = nn.LayerNorm(d_model)
21:         self.attn = nn.MultiheadAttention(
22:             embed_dim=d_model,
23:             num_heads=n_heads,
24:             dropout=dropout,
25:             batch_first=True
26:         )
27: 
28:         self.norm2 = nn.LayerNorm(d_model)
29: 
----------------------------------------

CELL 22 | Matches: ['nn.MultiheadAttention']
17:     def __init__(self, d_model, n_heads, ffn_dim):
18:         super().__init__()
19: 
20:         self.norm1 = nn.LayerNorm(d_model)
21: 
22:         self.self_attn = nn.MultiheadAttention(
23:             embed_dim=d_model,
24:             num_heads=n_heads,
25:             batch_first=True
26:         )
27: 
28:         self.norm2 = nn.LayerNorm(d_model)
29: 
30:         self.ffn = nn.Sequential(
----------------------------------------
62:     d

In [61]:

import ast

terms = [
    "in_proj_weight",
    "nn.MultiheadAttention",
    "class GPTBlock",
    "class TransformerBlock",
    "class CausalSelfAttention",
    "build_architecture_1",
]

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))
    found = [term for term in terms if term in src]

    if found:
        print(f"\nCELL {i} | Matches: {found}")
        lines = src.splitlines()
        for j, line in enumerate(lines):
            if any(term in line for term in found):
                start = max(0, j - 5)
                end = min(len(lines), j + 9)
                print("\n".join(
                    f"{k + 1}: {lines[k]}"
                    for k in range(start, end)
                ))
                print("-" * 40)



CELL 18 | Matches: ['nn.MultiheadAttention']
16: 
17:     def __init__(self, d_model, n_heads, ffn_dim, dropout=0.0):
18:         super().__init__()
19: 
20:         self.norm1 = nn.LayerNorm(d_model)
21:         self.attn = nn.MultiheadAttention(
22:             embed_dim=d_model,
23:             num_heads=n_heads,
24:             dropout=dropout,
25:             batch_first=True
26:         )
27: 
28:         self.norm2 = nn.LayerNorm(d_model)
29: 
----------------------------------------

CELL 22 | Matches: ['nn.MultiheadAttention']
17:     def __init__(self, d_model, n_heads, ffn_dim):
18:         super().__init__()
19: 
20:         self.norm1 = nn.LayerNorm(d_model)
21: 
22:         self.self_attn = nn.MultiheadAttention(
23:             embed_dim=d_model,
24:             num_heads=n_heads,
25:             batch_first=True
26:         )
27: 
28:         self.norm2 = nn.LayerNorm(d_model)
29: 
30:         self.ffn = nn.Sequential(
----------------------------------------
62:     d

In [62]:

for i, cell in enumerate(nb["cells"]):
    src = "".join(cell.get("source", []))

    if "build_architecture_1" not in src:
        continue

    print(f"\n{'=' * 16} CELL {i} {'=' * 16}")
    print(src[:3000])

    outputs = cell.get("outputs", [])
    print("\nSaved output count:", len(outputs))

    for out in outputs:
        data = out.get("data", {})
        text_data = data.get("text/plain", [])
        if text_data:
            print("".join(text_data)[:1500])

        error = out.get("evalue")
        if error:
            print("Saved error:", error)



================ CELL 70 ================
# Architecture 1: Decoder-only GPT-style model

model = build_architecture_1(
    vocab_size=50_000,
    seq_len=2048,
    d_model=512,
    n_layers=8,
    n_heads=8,
    ffn_dim=2048,
    dropout=0.0,
).cuda()

total_params = sum(p.numel() for p in model.parameters())

print(f"Parameters: {total_params:,}")
print(f"Parameters (M): {total_params / 1e6:.2f}M")
print(f"Device: {next(model.parameters()).device}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Saved output count: 0


In [63]:

import json
from pathlib import Path

extract_path = Path(
    "/kaggle/input/datasets/aq1de3/notebook1/"
    "phase1_model_code_extract.ipynb"
)

print("Extracted notebook exists:", extract_path.exists())

if extract_path.exists():
    extracted = json.loads(
        extract_path.read_text(encoding="utf-8")
    )
    print("Extracted notebook cells:", len(extracted["cells"]))

    for i, cell in enumerate(extracted["cells"]):
        src = "".join(cell.get("source", []))
        if any(term in src for term in (
            "build_architecture_1",
            "class GPT",
            "class Transformer",
            "in_proj_weight",
        )):
            print(f"\nCELL {i}")
            print(src[:1800])


Extracted notebook exists: False


In [64]:
from pathlib import Path

for root in [
    Path("/kaggle/input/datasets/aq1de3"),
    Path("/kaggle/working"),
]:
    print(f"\nSearching: {root}")
    if root.exists():
        for p in root.rglob("phase1_model_code_extract.ipynb"):
            print("Extracted notebook:", p)
        for p in root.rglob("notebook1d0fcbb0df (3).ipynb"):
            print("Original notebook:", p)


Searching: /kaggle/input/datasets/aq1de3
Extracted notebook: /kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
Original notebook: /kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb

Searching: /kaggle/working


In [65]:
import json
from pathlib import Path

extract_path = Path(
    "/kaggle/input/datasets/aq1de3/phase1-model/"
    "phase1_model_code_extract.ipynb"
)

extracted = json.loads(extract_path.read_text(encoding="utf-8"))
print("Extracted notebook cells:", len(extracted["cells"]))

terms = (
    "build_architecture_1",
    "class GQAAttention",
    "class GQABlock",
    "class GQALanguageModel",
    "class PrefixLMModel",
    "class EncoderDecoderLM",
    "class GPT",
    "class Transformer",
)

for i, cell in enumerate(extracted["cells"]):
    src = "".join(cell.get("source", []))
    if any(term in src for term in terms):
        print(f"\n{'=' * 25} CELL {i} {'=' * 25}")
        print(src[:5000])

Extracted notebook cells: 13

========================= CELL 4 =========================
# ============================================================
# PHASE 1 — ARCHITECTURE 2
# GQA — UPDATED MODEL WIDTH
# PARAMETER + GPU CHECK
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

print("=" * 70)
print("ARCHITECTURE 2 — GQA UPDATED RESOURCE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

VOCAB_SIZE = Tokenizer.from_file(
    TOKENIZER_PATH
).get_vocab_size()

SEQ_LEN = 2048

D_MODEL = 528
N_LAYERS = 8
N_Q_HEADS = 8
N_KV_HEADS = 2
FFN_DIM = 2048

HEAD_DIM = D_MODEL // N_Q_HEADS
GROUP_SIZE = N_Q_HEADS // N_KV_HEADS

print(f"\nVocabulary       : {VOCAB_SIZE:,}")
print(f"

In [66]:
import json
from pathlib import Path

original_path = Path(
    "/kaggle/input/datasets/aq1de3/notebook1/"
    "notebook1d0fcbb0df (3).ipynb"
)

nb = json.loads(original_path.read_text(encoding="utf-8"))
print("Original notebook cells:", len(nb["cells"]))

terms = [
    "def build_architecture_1",
    "def build_architecture_2",
    "class A2GQALM",
    "class GQALanguageModel",
    "class PrefixLMModel",
    "class EncoderDecoderLM",
    "class A3",
    "class A4",
]

for term in terms:
    matches = []
    for i, cell in enumerate(nb["cells"]):
        src = "".join(cell.get("source", []))
        if term in src:
            matches.append(i)
    print(f"{term}: cells {matches}")

print("\n--- Model-definition candidates ---")
for i, cell in enumerate(nb["cells"]):
    src = "".join(cell.get("source", []))
    if any(term in src for term in [
        "def build_architecture_1",
        "def build_architecture_2",
        "class A2GQALM",
        "class PrefixLMModel",
        "class EncoderDecoderLM",
    ]):
        print(f"\n===== CELL {i} =====")
        print(src[:7000])

Original notebook cells: 160
def build_architecture_1: cells []
def build_architecture_2: cells []
class A2GQALM: cells [122, 127, 139]
class GQALanguageModel: cells [13, 14, 15]
class PrefixLMModel: cells [18]
class EncoderDecoderLM: cells [22]
class A3: cells []
class A4: cells []

--- Model-definition candidates ---

===== CELL 18 =====
# ============================================================
# ARCHITECTURE 3 — PREFIX-LM HYBRID
# STEP 3: MODEL DEFINITION + PARAMETER CHECK
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Prefix-LM Transformer Block
# ------------------------------------------------------------

class PrefixLMBlock(nn.Module):

    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.0):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
  

In [67]:
import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/notebook1/"
    "notebook1d0fcbb0df (3).ipynb"
)

nb = json.loads(path.read_text(encoding="utf-8"))

terms = [
    "torch.save",
    "model_state_dict",
    "architecture1_sustained_step_500",
    "a2_step_9000",
    "step_0001000",
    "a4_step_22000",
    "load_state_dict",
    "state_dict()",
]

for term in terms:
    print(f"\n{'=' * 18} {term} {'=' * 18}")
    found = False

    for i, cell in enumerate(nb["cells"]):
        src = "".join(cell.get("source", []))
        if term.lower() in src.lower():
            found = True
            print(f"\nCELL {i}")
            lines = src.splitlines()
            for j, line in enumerate(lines):
                if term.lower() in line.lower():
                    start = max(0, j - 8)
                    end = min(len(lines), j + 12)
                    print("\n".join(
                        f"{k + 1}: {lines[k]}"
                        for k in range(start, end)
                    ))
                    print("---")

    if not found:
        print("No match found.")


================== torch.save ==================

CELL 27
49:     if extra_state is not None:
50:         checkpoint["extra_state"] = extra_state
51: 
52:     path = os.path.join(
53:         CHECKPOINT_ROOT,
54:         f"{model_name}_step_{step}.pt"
55:     )
56: 
57:     torch.save(
58:         checkpoint,
59:         path
60:     )
61: 
62:     return path
63: 
64: 
65: def load_checkpoint(
66:     path,
67:     model,
68:     optimizer=None,
---

CELL 35
178: # SAVE DRY-RUN CHECKPOINT
179: # ------------------------------------------------------------
180: 
181: checkpoint_path = os.path.join(
182:     CHECKPOINT_DIR,
183:     "arch1_dryrun_100k.pt"
184: )
185: 
186: torch.save(
187:     {
188:         "model_state_dict": model.state_dict(),
189:         "optimizer_state_dict": optimizer.state_dict(),
190:         "step": step,
191:         "tokens_seen": tokens_seen,
192:         "last_loss": loss_history[-1] if loss_history else None,
193:         "architecture": "decoder_only_

In [68]:
import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/notebook1/"
    "notebook1d0fcbb0df (3).ipynb"
)
nb = json.loads(path.read_text(encoding="utf-8"))

# Print architecture-1 model definitions and training setup.
for i in [13, 14, 15, 16, 17, 27, 35, 44, 45, 70, 71, 77]:
    if i < len(nb["cells"]):
        src = "".join(nb["cells"][i].get("source", []))
        print(f"\n{'=' * 20} CELL {i} {'=' * 20}")
        print(src[:6500])


==================== CELL 13 ====================
# ============================================================
# PHASE 1 — ARCHITECTURE 2
# GQA — UPDATED MODEL WIDTH
# PARAMETER + GPU CHECK
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

print("=" * 70)
print("ARCHITECTURE 2 — GQA UPDATED RESOURCE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

VOCAB_SIZE = Tokenizer.from_file(
    TOKENIZER_PATH
).get_vocab_size()

SEQ_LEN = 2048

D_MODEL = 528
N_LAYERS = 8
N_Q_HEADS = 8
N_KV_HEADS = 2
FFN_DIM = 2048

HEAD_DIM = D_MODEL // N_Q_HEADS
GROUP_SIZE = N_Q_HEADS // N_KV_HEADS

print(f"\nVocabulary       : {VOCAB_SIZE:,}")
print(f"Context length   : {SEQ_LEN:,}")
print

In [69]:
import json
from pathlib import Path
from tokenizers import Tokenizer

# Existing Kaggle inputs
DATA_ROOT = Path("/kaggle/input/datasets/aq1de3/phase3")
TOKENIZER_PATH = Path("/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json")
OUT_DIR = Path("/kaggle/working/phase3_prepared")
OUT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

with open(DATA_ROOT / "label_vocab.json", "r", encoding="utf-8") as f:
    label_vocab = json.load(f)

splits = {
    "CL_train": DATA_ROOT / "CL_train.json",
    "CL_dev":   DATA_ROOT / "CL_dev.json",
    "IT_train": DATA_ROOT / "IT_train.json",
    "IT_dev":   DATA_ROOT / "IT_dev.json",
}

prepared = {}
total_sentences = 0

for split_name, path in splits.items():
    with open(path, "r", encoding="utf-8") as f:
        documents = json.load(f)

    rows = []
    for doc in documents:
        doc_id = doc["id"]
        sentences = doc["text"]
        labels = doc["labels"]

        # Stop rather than silently corrupt sentence/label alignment.
        assert isinstance(sentences, list), (
            f"{split_name}: expected doc['text'] to be a sentence list; "
            f"got {type(sentences).__name__} for document {doc_id}"
        )
        assert len(sentences) == len(labels), (
            f"{split_name}: alignment error in document {doc_id}: "
            f"{len(sentences)} sentences vs {len(labels)} labels"
        )

        for sentence_idx, (sentence, label) in enumerate(zip(sentences, labels)):
            assert isinstance(sentence, str) and sentence.strip(), (
                f"{split_name}: empty/invalid sentence in document {doc_id}, "
                f"index {sentence_idx}"
            )
            assert label in label_vocab, (
                f"{split_name}: unknown label {label!r} in document {doc_id}"
            )

            # Keep the complete token sequence for now; truncation policy
            # will be applied deliberately when the model input is defined.
            input_ids = tokenizer.encode(sentence).ids

            rows.append({
                "doc_id": str(doc_id),
                "sentence_idx": sentence_idx,
                "text": sentence,
                "input_ids": input_ids,
                "label": label,
                "label_id": label_vocab[label],
            })

    output_path = OUT_DIR / f"{split_name}.jsonl"
    with open(output_path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    prepared[split_name] = rows
    total_sentences += len(rows)

    print(
        f"{split_name}: {len(documents)} documents, "
        f"{len(rows)} sentences, saved to {output_path}"
    )

print(f"\nTokenizer vocabulary: {tokenizer.get_vocab_size():,}")
print(f"Total prepared sentences: {total_sentences:,}")
print(f"Prepared files: {OUT_DIR}")

CL_train: 40 documents, 10593 sentences, saved to /kaggle/working/phase3_prepared/CL_train.jsonl
CL_dev: 5 documents, 1396 sentences, saved to /kaggle/working/phase3_prepared/CL_dev.jsonl
IT_train: 40 documents, 6252 sentences, saved to /kaggle/working/phase3_prepared/IT_train.jsonl
IT_dev: 5 documents, 746 sentences, saved to /kaggle/working/phase3_prepared/IT_dev.jsonl

Tokenizer vocabulary: 50,000
Total prepared sentences: 18,987
Prepared files: /kaggle/working/phase3_prepared


In [70]:
import json
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader

PREP_DIR = Path("/kaggle/working/phase3_prepared")
PAD_ID = 0  # Verified tokenizer padding ID

class RhetoricalRoleDataset(Dataset):
    def __init__(self, jsonl_path):
        self.rows = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                self.rows.append(json.loads(line))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]

def collate_rr(batch):
    max_len = max(len(row["input_ids"]) for row in batch)

    input_ids = torch.full(
        (len(batch), max_len), PAD_ID, dtype=torch.long
    )
    attention_mask = torch.zeros(
        (len(batch), max_len), dtype=torch.long
    )
    labels = torch.tensor(
        [row["label_id"] for row in batch], dtype=torch.long
    )

    for i, row in enumerate(batch):
        ids = torch.tensor(row["input_ids"], dtype=torch.long)
        input_ids[i, :len(ids)] = ids
        attention_mask[i, :len(ids)] = 1

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "doc_ids": [row["doc_id"] for row in batch],
        "sentence_indices": [row["sentence_idx"] for row in batch],
    }

# Small smoke test only; this does not train anything.
train_dataset = RhetoricalRoleDataset(PREP_DIR / "CL_train.jsonl")
loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_rr,
)

batch = next(iter(loader))

print("Batch tensor shapes:")
for key in ("input_ids", "attention_mask", "labels"):
    print(f"{key}: {tuple(batch[key].shape)}, dtype={batch[key].dtype}")

print("Padding mask values:", torch.unique(batch["attention_mask"]).tolist())
print("Labels in first batch:", batch["labels"].tolist())
print("Document IDs:", batch["doc_ids"])
print("Sentence indices:", batch["sentence_indices"])

Batch tensor shapes:
input_ids: (4, 84), dtype=torch.int64
attention_mask: (4, 84), dtype=torch.int64
labels: (4,), dtype=torch.int64
Padding mask values: [0, 1]
Labels in first batch: [0, 0, 0, 0]
Document IDs: ['CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733', 'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733', 'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733', 'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733']
Sentence indices: [0, 1, 2, 3]


In [71]:
import json
from collections import Counter
from pathlib import Path

PREP_DIR = Path("/kaggle/working/phase3_prepared")

for split in ["CL_train", "CL_dev", "IT_train", "IT_dev"]:
    counts = Counter()

    with open(PREP_DIR / f"{split}.jsonl", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            counts[row["label"]] += 1

    print(f"\n{'=' * 45}\n{split} — {sum(counts.values())} sentences")
    for label, count in sorted(counts.items(), key=lambda item: item[1]):
        print(f"{label:<28} {count:>6} ({100 * count / sum(counts.values()):5.2f}%)")


CL_train — 10593 sentences
Issue                            17 ( 0.16%)
None                             53 ( 0.50%)
RulingByPresentCourt            113 ( 1.07%)
PrecedentNotReliedUpon          209 ( 1.97%)
Dissent                         284 ( 2.68%)
Statute                         367 ( 3.46%)
RulingByLowerCourt              438 ( 4.13%)
ArgumentPetitioner              976 ( 9.21%)
ArgumentRespondent              996 ( 9.40%)
PrecedentReliedUpon            1305 (12.32%)
RatioOfTheDecision             2856 (26.96%)
Fact                           2979 (28.12%)

CL_dev — 1396 sentences
None                              2 ( 0.14%)
Issue                             3 ( 0.21%)
RulingByPresentCourt             18 ( 1.29%)
RulingByLowerCourt               56 ( 4.01%)
Statute                          58 ( 4.15%)
PrecedentReliedUpon             142 (10.17%)
ArgumentRespondent              236 (16.91%)
Fact                            249 (17.84%)
RatioOfTheDecision              288 (20.63%)
Ar

In [72]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)
import json

with open(
    "/kaggle/input/datasets/aq1de3/phase3/label_vocab.json",
    "r",
    encoding="utf-8",
) as f:
    label_vocab = json.load(f)

# Preserve the official label-ID order.
labels_by_id = [
    label for label, label_id in
    sorted(label_vocab.items(), key=lambda item: item[1])
]
label_ids = list(range(len(labels_by_id)))

def evaluate_rr(y_true, y_pred, split_name="evaluation"):
    assert len(y_true) == len(y_pred), (
        f"Length mismatch: {len(y_true)} gold labels vs "
        f"{len(y_pred)} predictions"
    )

    print(f"\n{split_name}")
    print(f"Sentences evaluated: {len(y_true)}")
    print(
        "Macro-F1:",
        round(f1_score(
            y_true, y_pred,
            labels=label_ids,
            average="macro",
            zero_division=0,
        ), 4),
    )
    print(
        "Macro Precision:",
        round(precision_score(
            y_true, y_pred,
            labels=label_ids,
            average="macro",
            zero_division=0,
        ), 4),
    )
    print(
        "Macro Recall:",
        round(recall_score(
            y_true, y_pred,
            labels=label_ids,
            average="macro",
            zero_division=0,
        ), 4),
    )
    print("\nPer-class report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=label_ids,
        target_names=labels_by_id,
        zero_division=0,
        digits=4,
    ))

print("Evaluation function ready.")
print("Number of official labels:", len(labels_by_id))
print("Label order:", labels_by_id)

Evaluation function ready.
Number of official labels: 13
Label order: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']


In [73]:
import json
from collections import Counter
from pathlib import Path

PREP_DIR = Path("/kaggle/working/phase3_prepared")

def load_rows(split):
    with open(PREP_DIR / f"{split}.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

for domain in ["CL", "IT"]:
    train_rows = load_rows(f"{domain}_train")
    dev_rows = load_rows(f"{domain}_dev")

    majority_label = Counter(
        row["label_id"] for row in train_rows
    ).most_common(1)[0][0]

    y_true = [row["label_id"] for row in dev_rows]
    y_pred = [majority_label] * len(y_true)

    print(f"\n{domain}: majority baseline predicts label ID {majority_label}")
    evaluate_rr(y_true, y_pred, split_name=f"{domain} majority baseline")


CL: majority baseline predicts label ID 0

CL majority baseline
Sentences evaluated: 1396
Macro-F1: 0.0233
Macro Precision: 0.0137
Macro Recall: 0.0769

Per-class report:
                        precision    recall  f1-score   support

                  Fact     0.1784    1.0000    0.3027       249
                 Issue     0.0000    0.0000    0.0000         3
    ArgumentPetitioner     0.0000    0.0000    0.0000       344
    ArgumentRespondent     0.0000    0.0000    0.0000       236
               Statute     0.0000    0.0000    0.0000        58
               Dissent     0.0000    0.0000    0.0000         0
   PrecedentReliedUpon     0.0000    0.0000    0.0000       142
PrecedentNotReliedUpon     0.0000    0.0000    0.0000         0
    PrecedentOverruled     0.0000    0.0000    0.0000         0
    RulingByLowerCourt     0.0000    0.0000    0.0000        56
    RatioOfTheDecision     0.0000    0.0000    0.0000       288
  RulingByPresentCourt     0.0000    0.0000    0.0000      

In [74]:
import os

roots = [
    "/kaggle/working",
    "/kaggle/input/datasets/aq1de3",
]

extensions = (".py", ".ipynb")

for root in roots:
    print(f"\n=== Searching {root} ===")

    if not os.path.exists(root):
        print("Path not found.")
        continue

    found = 0
    for current, dirs, files in os.walk(root):
        # Avoid unnecessary traversal into hidden metadata folders.
        dirs[:] = [d for d in dirs if not d.startswith(".")]

        for filename in files:
            if filename.lower().endswith(extensions):
                path = os.path.join(current, filename)
                print(path)
                found += 1

    print(f"Source files found: {found}")


=== Searching /kaggle/working ===
Source files found: 0

=== Searching /kaggle/input/datasets/aq1de3 ===
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
/kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb
Source files found: 2


In [75]:
import torch
import gc

CKPT_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "a2-step-9000-pt/a2_step_9000.pt"
)

ckpt = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt.get("model_state_dict", ckpt)
print("Checkpoint step:", ckpt.get("step", "unknown"))
print("Checkpoint tensors:", len(state))

# Inspect A2 variants already defined in the notebook.
# This assumes the notebook's A2 classes/configuration cells
# have already been executed.
candidates = {
    name: obj for name, obj in globals().items()
    if isinstance(obj, type)
    and issubclass(obj, torch.nn.Module)
    and name in {
        "A2GQALM",
        "GQALanguageModel",
    }
}

if not candidates:
    print(
        "\nNo candidate A2 classes are currently defined "
        "in this notebook kernel."
    )
    print("Do not instantiate a replacement model yet.")
else:
    for name, cls in candidates.items():
        try:
            model = cls()
            model_state = model.state_dict()

            expected = set(state)
            actual = set(model_state)

            missing = sorted(expected - actual)
            unexpected = sorted(actual - expected)

            shape_mismatches = [
                (key, tuple(state[key].shape),
                 tuple(model_state[key].shape))
                for key in sorted(expected & actual)
                if state[key].shape != model_state[key].shape
            ]

            print(f"\n=== {name} ===")
            print("Checkpoint keys:", len(expected))
            print("Model keys:", len(actual))
            print("Missing model keys:", len(missing))
            print("Unexpected model keys:", len(unexpected))
            print("Shape mismatches:", len(shape_mismatches))

            if missing:
                print("Missing examples:", missing[:8])
            if unexpected:
                print("Unexpected examples:", unexpected[:8])
            if shape_mismatches:
                print("Shape mismatch examples:", shape_mismatches[:8])

            if not missing and not unexpected and not shape_mismatches:
                print("EXACT KEY/SHAPE MATCH")
            else:
                print("NOT COMPATIBLE")

            del model
            gc.collect()

        except Exception as e:
            print(f"\n{name} could not be instantiated: {e}")

del ckpt, state
gc.collect()
print("\nRead-only compatibility check complete.")

Checkpoint step: 9000
Checkpoint tensors: 113

No candidate A2 classes are currently defined in this notebook kernel.
Do not instantiate a replacement model yet.

Read-only compatibility check complete.


In [76]:
import json

NOTEBOOK_PATH = (
    "/kaggle/input/datasets/aq1de3/notebook1/"
    "notebook1d0fcbb0df (3).ipynb"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "class A2GQALM",
    "class A2Block",
    "class A2GQAAttention",
    "N_KV_HEADS",
]

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    hits = [k for k in keywords if k in source]

    if hits:
        print(f"\n{'=' * 60}")
        print(f"CELL {i} | Matches: {hits}")
        print(source[:12000])


CELL 12 | Matches: ['N_KV_HEADS']
# ============================================================
# PHASE 1 — ARCHITECTURE 2
# EFFICIENT TRANSFORMER — GROUPED-QUERY ATTENTION (GQA)
# MODEL CONSTRUCTION & RESOURCE CHECK
# ============================================================

import torch
from tokenizers import Tokenizer
from transformers import GPT2Config, GPT2LMHeadModel

print("=" * 70)
print("PHASE 1 — ARCHITECTURE 2: EFFICIENT TRANSFORMER (GQA)")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

SEQ_LEN = 2048
VOCAB_SIZE = Tokenizer.from_file(
    TOKENIZER_PATH
).get_vocab_size()

# ------------------------------------------------------------
# Architecture 2 configuration
#
# Query heads       : 8
# Key/Value heads   : 2
#
# 4 Query heads share each K/V head.
# ---------------------------

In [77]:
import torch
from collections import Counter

path = "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)
state = ckpt.get("model_state_dict", ckpt)

print("Step:", ckpt.get("step"))
print("Tensor count:", len(state))

print("\nEmbedding / output tensors:")
for k, v in state.items():
    if any(s in k.lower() for s in ("embedding", "output_bias", "final_norm", "norm")):
        print(f"{k:55s} {tuple(v.shape)}")

print("\nLayer indices found:")
indices = set()
for k in state:
    parts = k.split(".")
    for i, part in enumerate(parts[:-1]):
        if part == "layers" and parts[i + 1].isdigit():
            indices.add(int(parts[i + 1]))
print(sorted(indices))
print("Layer count:", len(indices))

print("\nAttention projection shapes:")
for k, v in state.items():
    if any(s in k for s in ("q_proj", "k_proj", "v_proj", "out_proj")):
        print(f"{k:55s} {tuple(v.shape)}")

print("\nCheckpoint inspection complete.")
del ckpt, state

Step: 9000
Tensor count: 113

Embedding / output tensors:
output_bias                                             (50000,)
token_embedding.weight                                  (50000, 512)
position_embedding.weight                               (2048, 512)
layers.0.norm1.weight                                   (512,)
layers.0.norm1.bias                                     (512,)
layers.0.norm2.weight                                   (512,)
layers.0.norm2.bias                                     (512,)
layers.1.norm1.weight                                   (512,)
layers.1.norm1.bias                                     (512,)
layers.1.norm2.weight                                   (512,)
layers.1.norm2.bias                                     (512,)
layers.2.norm1.weight                                   (512,)
layers.2.norm1.bias                                     (512,)
layers.2.norm2.weight                                   (512,)
layers.2.norm2.bias                            

In [78]:
import torch

path = "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)
state = ckpt.get("model_state_dict", ckpt)

print("All unique parameter-name patterns:")
patterns = set()

for key, tensor in state.items():
    parts = key.split(".")
    normalized = ".".join(
        "<layer>" if p.isdigit() and i > 0 and parts[i-1] == "layers"
        else p
        for i, p in enumerate(parts)
    )
    patterns.add((normalized, tuple(tensor.shape)))

for name, shape in sorted(patterns):
    print(f"{name:55s} {shape}")

print("\nBias tensors:")
for key, tensor in state.items():
    if key.endswith(".bias") or key == "output_bias":
        print(f"{key:55s} {tuple(tensor.shape)}")

del ckpt, state

All unique parameter-name patterns:
final_norm.bias                                         (512,)
final_norm.weight                                       (512,)
layers.<layer>.attention.k_proj.weight                  (128, 512)
layers.<layer>.attention.out_proj.weight                (512, 512)
layers.<layer>.attention.q_proj.weight                  (512, 512)
layers.<layer>.attention.v_proj.weight                  (128, 512)
layers.<layer>.ffn.fc1.bias                             (2048,)
layers.<layer>.ffn.fc1.weight                           (2048, 512)
layers.<layer>.ffn.fc2.bias                             (512,)
layers.<layer>.ffn.fc2.weight                           (512, 2048)
layers.<layer>.norm1.bias                               (512,)
layers.<layer>.norm1.weight                             (512,)
layers.<layer>.norm2.bias                               (512,)
layers.<layer>.norm2.weight                             (512,)
output_bias                                            

In [79]:
import torch

CKPT_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "a2-step-9000-pt/a2_step_9000.pt"
)

checkpoint = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

state = checkpoint.get("model_state_dict", checkpoint)

# Verify dimensions directly from checkpoint tensors.
assert state["token_embedding.weight"].shape == (50000, 512)
assert state["position_embedding.weight"].shape == (2048, 512)
assert state["final_norm.weight"].shape == (512,)
assert state["output_bias"].shape == (50000,)

layer_ids = sorted({
    int(key.split(".")[1])
    for key in state
    if key.startswith("layers.")
})

assert layer_ids == list(range(9)), layer_ids

for i in layer_ids:
    prefix = f"layers.{i}."
    assert state[prefix + "attention.q_proj.weight"].shape == (512, 512)
    assert state[prefix + "attention.k_proj.weight"].shape == (128, 512)
    assert state[prefix + "attention.v_proj.weight"].shape == (128, 512)
    assert state[prefix + "attention.out_proj.weight"].shape == (512, 512)
    assert state[prefix + "ffn.fc1.weight"].shape == (2048, 512)
    assert state[prefix + "ffn.fc2.weight"].shape == (512, 2048)

print("A2 checkpoint dimensions verified.")
print("Layers:", len(layer_ids))
print("Hidden size:", state["token_embedding.weight"].shape[1])
print("Vocabulary size:", state["token_embedding.weight"].shape[0])
print("Context length:", state["position_embedding.weight"].shape[0])
print("Attention: grouped-query (Q width 512; K/V width 128)")
print("Feed-forward width:", state["layers.0.ffn.fc1.weight"].shape[0])
print("Checkpoint step:", checkpoint.get("step", "not recorded"))

A2 checkpoint dimensions verified.
Layers: 9
Hidden size: 512
Vocabulary size: 50000
Context length: 2048
Attention: grouped-query (Q width 512; K/V width 128)
Feed-forward width: 2048
Checkpoint step: 9000


In [85]:
import json

NOTEBOOK_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "notebook1/notebook1d0fcbb0df (3).ipynb"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

terms = [
    "output_bias",
    "class GQALanguageModel",
    "GroupedQueryAttention",
    "a2_step_9000",
    "D_MODEL = 512",
    "D_MODEL = 528",
]

matches = 0

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    found = [term for term in terms if term.lower() in source.lower()]

    if found:
        matches += 1
        print(f"\n{'=' * 70}")
        print(f"CELL {i} | Matches: {found}")
        print(f"{'=' * 70}")
        print(source[:7000])

print(f"\nTotal matching cells: {matches}")


CELL 13 | Matches: ['class GQALanguageModel', 'D_MODEL = 528']
# ============================================================
# PHASE 1 — ARCHITECTURE 2
# GQA — UPDATED MODEL WIDTH
# PARAMETER + GPU CHECK
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

print("=" * 70)
print("ARCHITECTURE 2 — GQA UPDATED RESOURCE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

VOCAB_SIZE = Tokenizer.from_file(
    TOKENIZER_PATH
).get_vocab_size()

SEQ_LEN = 2048

D_MODEL = 528
N_LAYERS = 8
N_Q_HEADS = 8
N_KV_HEADS = 2
FFN_DIM = 2048

HEAD_DIM = D_MODEL // N_Q_HEADS
GROUP_SIZE = N_Q_HEADS // N_KV_HEADS

print(f"\nVocabulary       : {VOCAB_SIZE:,}")
print(f"Context length   : {SEQ_L

In [86]:
import os

ROOT = "/kaggle/input/datasets/aq1de3"
terms = [
    "output_bias",
    "final_norm",
    "layers.8.attention",
    "N_LAYERS = 9",
]

matches = 0

for root, dirs, files in os.walk(ROOT):
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for filename in files:
        if not filename.endswith((".py", ".ipynb")):
            continue

        path = os.path.join(root, filename)

        try:
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
        except (UnicodeDecodeError, OSError):
            continue

        found = [term for term in terms if term in content]

        if found:
            matches += 1
            print(f"\nFILE: {path}")
            print(f"MATCHES: {found}")

            for term in found:
                pos = content.find(term)
                print(f"\n--- {term} ---")
                print(content[max(0, pos-400):pos+1000])

print(f"\nFiles containing matching terms: {matches}")


FILE: /kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
MATCHES: ['final_norm']

--- final_norm ---
    "        )\n",
    "\n",
    "        self.position_embedding = nn.Embedding(\n",
    "            SEQ_LEN,\n",
    "            D_MODEL,\n",
    "        )\n",
    "\n",
    "        self.blocks = nn.ModuleList(\n",
    "            [\n",
    "                GQABlock()\n",
    "                for _ in range(N_LAYERS)\n",
    "            ]\n",
    "        )\n",
    "\n",
    "        self.final_norm = nn.LayerNorm(\n",
    "            D_MODEL\n",
    "        )\n",
    "\n",
    "        # Weight tying\n",
    "        self.lm_head = nn.Linear(\n",
    "            D_MODEL,\n",
    "            VOCAB_SIZE,\n",
    "            bias=False,\n",
    "        )\n",
    "\n",
    "        self.lm_head.weight = (\n",
    "            self.token_embedding.weight\n",
    "        )\n",
    "\n",
    "    def forward(self, input_ids):\n",
    "\n",
    "        

In [87]:
import torch

path = (
    "/kaggle/input/datasets/aq1de3/"
    "a2-step-9000-pt/a2_step_9000.pt"
)

ckpt = torch.load(path, map_location="cpu", weights_only=False)

print("Checkpoint metadata:")
for key, value in ckpt.items():
    if key != "model_state_dict" and not isinstance(value, dict):
        print(f"{key}: {value}")
    elif key not in ("model_state_dict", "optimizer_state_dict"):
        print(f"{key}: {value}")

print("\nTop-level keys:", list(ckpt.keys()))

Checkpoint metadata:
step: 9000
tokens_processed: 73728000
train_loss: 6.801188290119171
reason: regular_checkpoint
saved_at: 2026-10-05T19:17:55.016904

Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']


In [88]:
import json

path = (
    "/kaggle/input/datasets/aq1de3/"
    "notebook1/notebook1d0fcbb0df (3).ipynb"
)

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

cell_index = 152
cell = nb["cells"][cell_index]
source = "".join(cell.get("source", []))

print("Cell type:", cell.get("cell_type"))
print("Cell index:", cell_index)
print("Cell length:", len(source))
print("=" * 80)
print(source[:16000])

Cell type: code
Cell index: 152
Cell length: 15330
# ============================================================
# A2 — FINAL STABLE HIGH-SPEED PRODUCTION
# Batch 8 × 512 tokens
# ============================================================

import os
import re
import gc
import math
import time
import shutil

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from tokenizers import Tokenizer
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# ============================================================
# 1. CONFIGURATION
# ============================================================

DEVICE = torch.device("cuda")

REPO = "Exploration-Lab/CS-781-Capstone"

VOCAB_SIZE = 50_000
MAX_CONTEXT = 2_048

TRAIN_BLOCK = 512
BATCH_SIZE = 8

TOKENS_PER_UPDATE = (
    BATCH_SIZE * (TRAIN_BLOCK - 1)
)

TARGET_TOKENS = 200_000_000

LOG_EVERY = 50
SAVE_EVERY = 2000

CHECKPOINT_DIR = (
    "/kaggle/worki

In [89]:
import os

ROOT = "/kaggle/input/datasets/aq1de3"
term = "A2GQADecoder"
matches = 0

for root, dirs, files in os.walk(ROOT):
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for filename in files:
        if not filename.endswith((".py", ".ipynb")):
            continue

        path = os.path.join(root, filename)

        try:
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
        except (OSError, UnicodeDecodeError):
            continue

        start = 0
        while True:
            pos = content.find(term, start)
            if pos == -1:
                break

            matches += 1
            print(f"\nFILE: {path}")
            print(f"POSITION: {pos}")
            print(content[max(0, pos - 1200):pos + 3500])
            start = pos + len(term)

print(f"\nTotal occurrences: {matches}")


Total occurrences: 0


In [90]:
import os

ROOT = "/kaggle/working"
term = "A2GQADecoder"
found = False

for root, dirs, files in os.walk(ROOT):
    for filename in files:
        if filename.endswith((".py", ".ipynb", ".txt")):
            path = os.path.join(root, filename)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()
                if term in content:
                    print("FOUND:", path)
                    found = True
            except (OSError, UnicodeDecodeError):
                pass

if not found:
    print("Not found in /kaggle/working.")

FOUND: /kaggle/working/.virtual_documents/__notebook_source__.ipynb


In [92]:
import os

path = "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"

print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path) if os.path.exists(path) else "missing")

if os.path.exists(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for i in range(8):
            line = f.readline()
            if not line:
                break
            print(f"Line {i+1}: {line[:250]!r}")

Exists: True
Size: 72733
Line 1: '\n'
Line 2: 'from pathlib import Path\n'
Line 3: 'import json\n'
Line 4: '\n'
Line 5: '# Find the dataset files mounted by Kaggle\n'
Line 6: 'matches = list(Path("/kaggle/input").rglob("label_vocab.json"))\n'
Line 7: '\n'
Line 8: 'if not matches:\n'


In [93]:
import torch

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "a2-step-9000-pt/a2_step_9000.pt"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

state = checkpoint.get("model_state_dict", checkpoint)

for name, tensor in state.items():
    if isinstance(tensor, torch.Tensor):
        print(f"{name:60s} {tuple(tensor.shape)}")

output_bias                                                  (50000,)
token_embedding.weight                                       (50000, 512)
position_embedding.weight                                    (2048, 512)
layers.0.norm1.weight                                        (512,)
layers.0.norm1.bias                                          (512,)
layers.0.attention.q_proj.weight                             (512, 512)
layers.0.attention.k_proj.weight                             (128, 512)
layers.0.attention.v_proj.weight                             (128, 512)
layers.0.attention.out_proj.weight                           (512, 512)
layers.0.norm2.weight                                        (512,)
layers.0.norm2.bias                                          (512,)
layers.0.ffn.fc1.weight                                      (2048, 512)
layers.0.ffn.fc1.bias                                        (2048,)
layers.0.ffn.fc2.weight                                      (512, 2048)
layers.0

In [94]:
import torch

state = checkpoint.get("model_state_dict", checkpoint)

# Verify architecture from checkpoint tensors
assert state["token_embedding.weight"].shape == (50000, 512)
assert state["position_embedding.weight"].shape == (2048, 512)
assert state["output_bias"].shape == (50000,)
assert state["final_norm.weight"].shape == (512,)

layer_ids = sorted({
    int(key.split(".")[1])
    for key in state
    if key.startswith("layers.")
})

assert layer_ids == list(range(9)), layer_ids

for i in layer_ids:
    prefix = f"layers.{i}."
    assert state[prefix + "attention.q_proj.weight"].shape == (512, 512)
    assert state[prefix + "attention.k_proj.weight"].shape == (128, 512)
    assert state[prefix + "attention.v_proj.weight"].shape == (128, 512)
    assert state[prefix + "attention.out_proj.weight"].shape == (512, 512)
    assert state[prefix + "ffn.fc1.weight"].shape == (2048, 512)
    assert state[prefix + "ffn.fc2.weight"].shape == (512, 2048)

print("Checkpoint structure validated.")
print("Layers:", len(layer_ids))
print("Hidden size: 512")
print("Query heads: 8")
print("KV heads: 2")
print("Head dimension:", 512 // 8)
print("Attention projection biases:", any(
    ".attention." in k and k.endswith(".bias") for k in state
))

Checkpoint structure validated.
Layers: 9
Hidden size: 512
Query heads: 8
KV heads: 2
Head dimension: 64
Attention projection biases: False


In [1]:

from pathlib import Path

root = Path("/kaggle/input")

for path in sorted(root.rglob("*")):
    if path.is_file():
        print(f"{path}  |  {path.stat().st_size / (1024**2):.2f} MB")


/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt  |  196.55 MB
/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt  |  367.79 MB
/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt  |  593.71 MB
/kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb  |  0.64 MB
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb  |  0.05 MB
/kaggle/input/datasets/aq1de3/phase3/CL_dev.json  |  0.32 MB
/kaggle/input/datasets/aq1de3/phase3/CL_test.json  |  0.27 MB
/kaggle/input/datasets/aq1de3/phase3/CL_train.json  |  2.35 MB
/kaggle/input/datasets/aq1de3/phase3/IT_dev.json  |  0.16 MB
/kaggle/input/datasets/aq1de3/phase3/IT_test.json  |  0.19 MB
/kaggle/input/datasets/aq1de3/phase3/IT_train.json  |  1.44 MB
/kaggle/input/datasets/aq1de3/phase3/label_vocab.json  |  0.00 MB
/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt  |  197.90 MB
/kaggle/input/datasets/aq1de3/tokenizer-json/tok

In [2]:

import torch
from pathlib import Path

root = Path("/kaggle/input")
checkpoints = list(root.rglob("*.pt")) + list(root.rglob("*.pth"))

for path in checkpoints:
    print(f"\n--- {path.name} ---")
    try:
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=True
        )
        if isinstance(checkpoint, dict):
            print("Keys:", list(checkpoint.keys())[:30])
            for key, value in checkpoint.items():
                if isinstance(value, dict):
                    print(f"  {key}: {len(value)} entries")
        else:
            print("Object type:", type(checkpoint))
    except Exception as e:
        print("Could not inspect:", str(e)[:250])



--- architecture1_sustained_step_500.pt ---
Keys: ['step', 'epoch', 'train_loss', 'val_loss', 'model_state_dict', 'optimizer_state_dict']
  model_state_dict: 101 entries
  optimizer_state_dict: 2 entries

--- a2_step_9000.pt ---
Keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
  model_state_dict: 113 entries

--- step_0001000.pt ---
Keys: ['model', 'step', 'tokens_seen', 'architecture', 'parameters', 'context_length', 'hidden_size', 'num_layers', 'num_heads', 'ffn_size', 'prefix_ratio', 'dropout', 'learning_rate', 'weight_decay']
  model: 100 entries

--- a4_step_22000.pt ---
Keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
  model_state_dict: 260 entries


In [3]:

import torch
from pathlib import Path

root = Path("/kaggle/input")
names = [
    "architecture1_sustained_step_500.pt",
    "a2_step_9000.pt",
    "step_0001000.pt",
    "a4_step_22000.pt",
]

for name in names:
    matches = list(root.rglob(name))
    if not matches:
        print(f"\nNOT FOUND: {name}")
        continue

    path = matches[0]
    ckpt = torch.load(path, map_location="cpu", weights_only=True)

    print(f"\n{'='*60}\n{name}")
    if isinstance(ckpt, dict):
        for key in [
            "architecture", "step", "epoch", "train_loss",
            "val_loss", "parameters", "context_length",
            "hidden_size", "num_layers", "num_heads",
            "ffn_size", "prefix_ratio"
        ]:
            if key in ckpt:
                print(f"{key}: {ckpt[key]}")

        state = next(
            (ckpt[k] for k in ["model_state_dict", "model"]
             if isinstance(ckpt.get(k), dict)),
            None
        )

        if state is not None:
            print("Parameter entries:", len(state))
            for key, value in list(state.items())[:12]:
                if hasattr(value, "shape"):
                    print(f"  {key}: {tuple(value.shape)}")
                else:
                    print(f"  {key}: {type(value).__name__}")



architecture1_sustained_step_500.pt
step: 500
epoch: 0
train_loss: 5.366012096405029
val_loss: None
Parameter entries: 101
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512)
  blocks.0.norm1.weight: (512,)
  blocks.0.norm1.bias: (512,)
  blocks.0.attn.in_proj_weight: (1536, 512)
  blocks.0.attn.in_proj_bias: (1536,)
  blocks.0.attn.out_proj.weight: (512, 512)
  blocks.0.attn.out_proj.bias: (512,)
  blocks.0.norm2.weight: (512,)
  blocks.0.norm2.bias: (512,)
  blocks.0.ffn.0.weight: (2048, 512)
  blocks.0.ffn.0.bias: (2048,)

a2_step_9000.pt
step: 9000
train_loss: 6.801188290119171
Parameter entries: 113
  output_bias: (50000,)
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512)
  layers.0.norm1.weight: (512,)
  layers.0.norm1.bias: (512,)
  layers.0.attention.q_proj.weight: (512, 512)
  layers.0.attention.k_proj.weight: (128, 512)
  layers.0.attention.v_proj.weight: (128, 512)
  layers.0.attention.out_proj.weight: (512, 512)
  l

In [4]:

from pathlib import Path

root = Path("/kaggle/input")

for pattern in ["*.ipynb", "tokenizer.json", "*.json"]:
    print(f"\n--- {pattern} ---")
    for path in root.rglob(pattern):
        print(path)



--- *.ipynb ---
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
/kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb

--- tokenizer.json ---
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json

--- *.json ---
/kaggle/input/datasets/aq1de3/phase3/CL_dev.json
/kaggle/input/datasets/aq1de3/phase3/IT_dev.json
/kaggle/input/datasets/aq1de3/phase3/IT_test.json
/kaggle/input/datasets/aq1de3/phase3/CL_train.json
/kaggle/input/datasets/aq1de3/phase3/CL_test.json
/kaggle/input/datasets/aq1de3/phase3/label_vocab.json
/kaggle/input/datasets/aq1de3/phase3/IT_train.json
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json


In [ ]:

from pathlib import Path

root = Path("/kaggle/input")

patterns = [
    "architecture1_sustained_step_500.pt",
    "a2_step_9000.pt",
    "step_0001000.pt",
    "a4_step_22000.pt",
    "tokenizer.json",
    "train.json",
    "dev.json",
    "test.json",
]

for name in patterns:
    matches = list(root.rglob(name))
    print(f"\n{name}:")
    if matches:
        for p in matches:
            print(f"  {p} ({p.stat().st_size / 1024:.1f} KB)")
    else:
        print("  NOT FOUND")


In [2]:
from pathlib import Path

root = Path("/kaggle/input")

print("=== Checkpoint files ===")
for p in root.rglob("*.pt"):
    print(f"{p} | {p.stat().st_size / (1024**2):.1f} MB")

=== Checkpoint files ===
/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt | 197.9 MB
/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt | 196.5 MB
/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt | 367.8 MB
/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt | 593.7 MB


In [3]:
from pathlib import Path
import torch

checkpoint_paths = [
    Path("/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt"),
    Path("/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt"),
    Path("/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt"),
    Path("/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt"),
]

for path in checkpoint_paths:
    print("\n" + "=" * 70)
    print("FILE:", path.name)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print("Top-level keys:", list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, (int, float, str, bool)):
            print(f"{key}: {value}")
        elif isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} entries")
            for name, tensor in list(value.items())[:5]:
                if isinstance(tensor, torch.Tensor):
                    print(f"  {name}: {tuple(tensor.shape)}")
                else:
                    print(f"  {name}: {type(tensor).__name__}")


FILE: architecture1_sustained_step_500.pt
Top-level keys: ['step', 'epoch', 'train_loss', 'val_loss', 'model_state_dict', 'optimizer_state_dict']
step: 500
epoch: 0
train_loss: 5.366012096405029
model_state_dict: dictionary with 101 entries
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512)
  blocks.0.norm1.weight: (512,)
  blocks.0.norm1.bias: (512,)
  blocks.0.attn.in_proj_weight: (1536, 512)
optimizer_state_dict: dictionary with 2 entries
  state: dict
  param_groups: list

FILE: a2_step_9000.pt
Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
model_state_dict: dictionary with 113 entries
  output_bias: (50000,)
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512)
  layers.0.norm1.weight: (512,)
  layers.0.norm1.bias: (512,)
step: 9000
tokens_processed: 73728000
train_loss: 6.801188290119171
reason: regular_checkpoint
saved_at: 2026-10-05T19:17:55.016904

FILE: step_0001000.p

In [4]:
import json
from pathlib import Path
from collections import Counter

DATA = Path("/kaggle/input/datasets/aq1de3")

files = {
    "train": DATA / "trainjson/train.json",
    "dev": DATA / "devjson/dev.json",
    "test": DATA / "testjson/test.json",
}

for split, path in files.items():
    print(f"\n{'=' * 60}\n{split.upper()}")
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Record count:", len(data))
    if not data:
        print("Dataset is empty.")
        continue

    print("First record keys:", list(data[0].keys()))
    print("First record ID:", data[0].get("id"))
    print("First record text preview:",
          str(data[0].get("text", ""))[:300])

    if split in ("train", "dev"):
        labels = [
            row.get("label")
            for row in data
            if row.get("label") is not None
        ]
        print("Label counts:", dict(Counter(labels)))
        print("Records with labels:", len(labels))

    lengths = [
        len(str(row.get("text", "")).split())
        for row in data
    ]
    if lengths:
        lengths.sort()
        print("Word counts (approx.):")
        print("  Min:", lengths[0])
        print("  Median:", lengths[len(lengths) // 2])
        print("  90th percentile:", lengths[int(0.9 * (len(lengths) - 1))])
        print("  Max:", lengths[-1])


TRAIN
Path: /kaggle/input/datasets/aq1de3/trainjson/train.json
Exists: True
Record count: 37387
First record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
First record ID: 2020_1
First record text preview:  Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09.12.2015 passed by the Division Bench of the High Court 1 dismissing S.T. No.192 of 2014 and other companynected matters and thereby affirming the decision of the Single Judge of the High Court passed on 12.03.2014 in Wr
Label counts: {0: 22067, 1: 15320}
Records with labels: 37387
Word counts (approx.):
  Min: 35
  Median: 2192
  90th percentile: 6575
  Max: 412606

DEV
Path: /kaggle/input/datasets/aq1de3/devjson/dev.json
Exists: True
Record count: 3505
First record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
First record ID: 1989_75
First record text preview: civil appellate jurisdiction civil appeal number 1709
o

In [5]:
import json
from pathlib import Path

base = Path("/kaggle/input/datasets/aq1de3")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train = load_json(base / "trainjson/train.json")
dev   = load_json(base / "devjson/dev.json")
test  = load_json(base / "testjson/test.json")

tokenizer_path = base / "tokenizer-json/tokenizer.json"

def inspect_split(name, rows):
    ids = [r.get("id") for r in rows]
    texts = [str(r.get("text", "")).strip() for r in rows]

    print(f"\n{name}:")
    print("Rows:", len(rows))
    print("Unique IDs:", len(set(ids)))
    print("Missing IDs:", sum(x is None for x in ids))
    print("Empty texts:", sum(not x for x in texts))
    print("Duplicate IDs:", len(ids) - len(set(ids)))
    print("Duplicate non-empty texts:", 
          len(texts) - len(set(t for t in texts if t)))

inspect_split("TRAIN", train)
inspect_split("DEV", dev)
inspect_split("TEST", test)

train_ids = {r["id"] for r in train if r.get("id") is not None}
dev_ids   = {r["id"] for r in dev if r.get("id") is not None}
test_ids  = {r["id"] for r in test if r.get("id") is not None}

print("\n=== Cross-split checks ===")
print("Train/dev ID overlap:", len(train_ids & dev_ids))
print("Train/test ID overlap:", len(train_ids & test_ids))
print("Dev/test ID overlap:", len(dev_ids & test_ids))

train_texts = {str(r.get("text", "")).strip() for r in train}
dev_texts = {str(r.get("text", "")).strip() for r in dev}
print("Exact train/dev text overlap:", len(train_texts & dev_texts))

print("\n=== Tokenizer ===")
print("Tokenizer exists:", tokenizer_path.exists())
if tokenizer_path.exists():
    print("Tokenizer size (MB):", round(tokenizer_path.stat().st_size / 1024**2, 2))


TRAIN:
Rows: 37387
Unique IDs: 32305
Missing IDs: 0
Empty texts: 0
Duplicate IDs: 5082
Duplicate non-empty texts: 5729

DEV:
Rows: 3505
Unique IDs: 2511
Missing IDs: 0
Empty texts: 0
Duplicate IDs: 994
Duplicate non-empty texts: 994

TEST:
Rows: 1517
Unique IDs: 1517
Missing IDs: 0
Empty texts: 0
Duplicate IDs: 0
Duplicate non-empty texts: 0

=== Cross-split checks ===
Train/dev ID overlap: 0
Train/test ID overlap: 0
Dev/test ID overlap: 1517
Exact train/dev text overlap: 0

=== Tokenizer ===
Tokenizer exists: True
Tokenizer size (MB): 3.37


In [9]:
from collections import defaultdict, Counter

def diagnose_split(name, rows):
    groups = defaultdict(list)
    for r in rows:
        groups[r["id"]].append(r)

    duplicate_groups = {
        k: v for k, v in groups.items() if len(v) > 1
    }

    conflicting_labels = 0
    different_texts = 0

    for records in duplicate_groups.values():
        labels = {
            r.get("label") for r in records
            if r.get("label") is not None
        }
        texts = {
            str(r.get("text", "")).strip()
            for r in records
        }

        if len(labels) > 1:
            conflicting_labels += 1
        if len(texts) > 1:
            different_texts += 1

    print(f"\n{name}")
    print("Unique IDs:", len(groups))
    print("IDs occurring multiple times:", len(duplicate_groups))
    print("Duplicate-ID groups with conflicting labels:", conflicting_labels)
    print("Duplicate-ID groups with different texts:", different_texts)

    print("Examples of duplicate IDs:")
    for case_id, records in list(duplicate_groups.items())[:3]:
        print({
            "id": case_id,
            "rows": len(records),
            "labels": [r.get("label") for r in records],
            "text_lengths": [len(str(r.get("text", ""))) for r in records],
        })

diagnose_split("TRAIN", train)
diagnose_split("DEV", dev)

# Compare IDs only; do not inspect test labels.
dev_ids = {r["id"] for r in dev}
test_ids = {r["id"] for r in test}

print("\nDEV / TEST ID overlap")
print("Overlapping IDs:", len(dev_ids & test_ids))
print("Test IDs absent from dev:", len(test_ids - dev_ids))
print("Sample overlapping IDs:", list(sorted(dev_ids & test_ids))[:10])

# Compare text for overlapping IDs without using test labels.
dev_by_id = {r["id"]: r for r in dev}
test_by_id = {r["id"]: r for r in test}

same_text = sum(
    str(dev_by_id[i].get("text", "")).strip()
    == str(test_by_id[i].get("text", "")).strip()
    for i in dev_ids & test_ids
)
print("Overlapping IDs with identical dev/test text:", same_text)


TRAIN
Unique IDs: 32305
IDs occurring multiple times: 5082
Duplicate-ID groups with conflicting labels: 0
Duplicate-ID groups with different texts: 0
Examples of duplicate IDs:
{'id': '2019_890', 'rows': 2, 'labels': [1, 1], 'text_lengths': [28354, 28354]}
{'id': '2014_170', 'rows': 2, 'labels': [0, 0], 'text_lengths': [31637, 31637]}
{'id': '2010_721', 'rows': 2, 'labels': [1, 1], 'text_lengths': [18402, 18402]}

DEV
Unique IDs: 2511
IDs occurring multiple times: 994
Duplicate-ID groups with conflicting labels: 0
Duplicate-ID groups with different texts: 0
Examples of duplicate IDs:
{'id': '1989_75', 'rows': 2, 'labels': [0, 0], 'text_lengths': [34774, 34774]}
{'id': '1959_102', 'rows': 2, 'labels': [0, 0], 'text_lengths': [12428, 12428]}
{'id': '1979_76', 'rows': 2, 'labels': [1, 1], 'text_lengths': [43689, 43689]}

DEV / TEST ID overlap
Overlapping IDs: 1517
Test IDs absent from dev: 0
Sample overlapping IDs: ['1950_14', '1950_23', '1950_38', '1951_10', '1951_30', '1951_33', '1951_

In [10]:
from pathlib import Path
import json

tokenizer_path = Path(
    "/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json"
)

with open(tokenizer_path, "r", encoding="utf-8") as f:
    tok_data = json.load(f)

print("Tokenizer top-level keys:", list(tok_data.keys()))
print("Model type:", tok_data.get("model", {}).get("type"))

vocab = tok_data.get("model", {}).get("vocab", {})
print("Vocabulary entries:", len(vocab) if isinstance(vocab, dict) else "not a dict")

print("Added tokens:", len(tok_data.get("added_tokens", [])))
print("Tokenizer normalizer:", tok_data.get("normalizer"))
print("Tokenizer pre-tokenizer:", tok_data.get("pre_tokenizer"))
print("Tokenizer post-processor:", tok_data.get("post_processor"))

Tokenizer top-level keys: ['version', 'truncation', 'padding', 'added_tokens', 'normalizer', 'pre_tokenizer', 'post_processor', 'decoder', 'model']
Model type: BPE
Vocabulary entries: 50000
Added tokens: 5
Tokenizer normalizer: None
Tokenizer pre-tokenizer: {'type': 'ByteLevel', 'add_prefix_space': False, 'trim_offsets': True, 'use_regex': True}
Tokenizer post-processor: None


In [12]:

from tokenizers import Tokenizer
from pathlib import Path

tokenizer_path = Path(
    "/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json"
)

tokenizer = Tokenizer.from_file(str(tokenizer_path))

print("Tokenizer loaded successfully")
print("Vocabulary size:", tokenizer.get_vocab_size())

print("\nSpecial tokens:")
for token in tokenizer.get_added_tokens_decoder().values():
    print(token)

sample_text = train[0]["text"][:2000]
encoded = tokenizer.encode(sample_text)

print("\nFirst 30 token IDs:", encoded.ids[:30])
print("Token count in sample:", len(encoded.ids))
print("Decoded preview:", tokenizer.decode(encoded.ids[:30]))
print("Configured truncation:", tokenizer.truncation)
print("Configured padding:", tokenizer.padding)


Tokenizer loaded successfully
Vocabulary size: 50000

Special tokens:
[PAD]
[UNK]
[BOS]
[EOS]
[MASK]

First 30 token IDs: [11053, 12503, 11483, 16, 374, 18, 4228, 2238, 4814, 741, 247, 238, 4422, 278, 1227, 476, 2783, 18, 698, 18, 1368, 757, 309, 238, 1481, 1374, 247, 238, 940, 430]
Token count in sample: 414
Decoded preview: ĠUday ĠUmesh ĠLalit , ĠJ . ĠThese Ġappeals Ġarise Ġout Ġof Ġthe ĠJudgment Ġand ĠOrder Ġdated Ġ09 . 12 . 2015 Ġpassed Ġby Ġthe ĠDivision ĠBench Ġof Ġthe ĠHigh ĠCourt
Configured truncation: None
Configured padding: None


In [13]:
from pathlib import Path

roots = [
    Path("/kaggle/input/datasets/aq1de3"),
    Path("/kaggle/working"),
]

for root in roots:
    print(f"\n=== Python files in {root} ===")
    if root.exists():
        for p in root.rglob("*.py"):
            print(p)


=== Python files in /kaggle/input/datasets/aq1de3 ===

=== Python files in /kaggle/working ===


In [18]:
from pathlib import Path

root = Path("/kaggle/input/datasets/aq1de3")

for folder_name in ["phase1_model", "phase3", "notebook1"]:
    folder = root / folder_name
    print(f"\n=== {folder} ===")

    if folder.exists():
        for p in folder.rglob("*"):
            if p.is_file():
                print(p, f"({p.stat().st_size / 1024**2:.2f} MB)")
    else:
        print("Folder not found at this path")
        


=== /kaggle/input/datasets/aq1de3/phase1_model ===
Folder not found at this path

=== /kaggle/input/datasets/aq1de3/phase3 ===
/kaggle/input/datasets/aq1de3/phase3/CL_dev.json (0.32 MB)
/kaggle/input/datasets/aq1de3/phase3/IT_dev.json (0.16 MB)
/kaggle/input/datasets/aq1de3/phase3/IT_test.json (0.19 MB)
/kaggle/input/datasets/aq1de3/phase3/CL_train.json (2.35 MB)
/kaggle/input/datasets/aq1de3/phase3/CL_test.json (0.27 MB)
/kaggle/input/datasets/aq1de3/phase3/label_vocab.json (0.00 MB)
/kaggle/input/datasets/aq1de3/phase3/IT_train.json (1.44 MB)

=== /kaggle/input/datasets/aq1de3/notebook1 ===
/kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb (0.64 MB)


In [19]:
from pathlib import Path

root = Path("/kaggle/input")

for p in root.rglob("*.ipynb"):
    print(p)

/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a4a4a4.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a2a2a2.ipynb
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
/kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb
/kaggle/input/datasets/aq1de3/p2p2p2/phase-2 (1).ipynb


In [20]:

import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/phase1-model/"
    "phase1_model_code_extract.ipynb"
)

with open(path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Notebook:", path.name)
print("Total cells:", len(notebook.get("cells", [])))

keywords = [
    "class ", "def forward", "nn.Module",
    "token_embedding", "encoder_layers",
    "decoder_layers", "load_state_dict",
    "state_dict", "Architecture"
]

for i, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if any(k.lower() in source.lower() for k in keywords):
        print(f"\n{'=' * 60}")
        print(f"CELL {i}")
        print(source[:6000])


Notebook: phase1_model_code_extract.ipynb
Total cells: 13

CELL 2
# ============================================================
# PHASE 1 — ARCHITECTURE 1
# DECODER-ONLY TRANSFORMER
# MODEL CONSTRUCTION & RESOURCE CHECK
# ============================================================

import torch
from tokenizers import Tokenizer
from transformers import GPT2Config, GPT2LMHeadModel

print("=" * 70)
print("PHASE 1 — ARCHITECTURE 1: DECODER-ONLY TRANSFORMER")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

VOCAB_SIZE = tokenizer.get_vocab_size()

print(f"\nVocabulary size : {VOCAB_SIZE:,}")

# ----------------------------

In [21]:

import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/p1p1p1/a4a4a4.ipynb"
)

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "ENCODER_LEN", "DECODER_LEN", "D_MODEL",
    "N_ENCODER_LAYERS", "N_DECODER_LAYERS",
    "N_HEADS", "FFN_DIM", "class EncoderDecoderLM",
    "def forward", "model = EncoderDecoderLM",
    "load_state_dict"
]

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if any(k.lower() in source.lower() for k in keywords):
        print(f"\n{'=' * 60}\nCELL {i}")
        print(source[:7000])



CELL 3
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = 50_000
D_MODEL = 512
NUM_HEADS = 8
FFN_DIM = 2048
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
ENCODER_CONTEXT = 1024
DECODER_CONTEXT = 1024
DROPOUT = 0.1


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, q, k, v, causal=False):
        B, Tq, C = q.shape
        Tk = k.shape[1]

        q = self.q_proj(q)
        k = self.k_proj(k)
        v = self.v_proj(v)

        q = q.view(B, Tq, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view

In [22]:
from pathlib import Path
import torch

checkpoint_path = Path(
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt"
)

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

state = ckpt["model_state_dict"]

print("Checkpoint step:", ckpt["step"])
print("Number of tensors:", len(state))
print("\nFirst 25 parameter names and shapes:")

for name, tensor in list(state.items())[:25]:
    print(f"{name:55s} {tuple(tensor.shape)}")

print("\nArchitecture signatures:")
for prefix in [
    "encoder_pos.",
    "encoder_position.",
    "encoder_layers.",
    "encoder_blocks.",
    "encoder_norm.",
    "encoder_ln.",
]:
    print(f"{prefix:20s}", any(k.startswith(prefix) for k in state))

Checkpoint step: 22000
Number of tensors: 260

First 25 parameter names and shapes:
token_embedding.weight                                  (50000, 512)
encoder_pos.weight                                      (1024, 512)
decoder_pos.weight                                      (1024, 512)
encoder_layers.0.norm1.weight                           (512,)
encoder_layers.0.norm1.bias                             (512,)
encoder_layers.0.attn.q_proj.weight                     (512, 512)
encoder_layers.0.attn.q_proj.bias                       (512,)
encoder_layers.0.attn.k_proj.weight                     (512, 512)
encoder_layers.0.attn.k_proj.bias                       (512,)
encoder_layers.0.attn.v_proj.weight                     (512, 512)
encoder_layers.0.attn.v_proj.bias                       (512,)
encoder_layers.0.attn.out_proj.weight                   (512, 512)
encoder_layers.0.attn.out_proj.bias                     (512,)
encoder_layers.0.norm2.weight                           (512,)
en

In [23]:
from collections import Counter

keys = list(state.keys())

print("Encoder layers:", len({
    k.split(".")[1] for k in keys
    if k.startswith("encoder_layers.")
}))
print("Decoder layers:", len({
    k.split(".")[1] for k in keys
    if k.startswith("decoder_layers.")
}))

print("\nKey prefixes:")
print(Counter(k.split(".")[0] for k in keys))

print("\nRequired parameter groups present:")
for prefix in [
    "token_embedding.",
    "encoder_pos.",
    "decoder_pos.",
    "encoder_layers.",
    "decoder_layers.",
    "encoder_norm.",
    "decoder_norm.",
    "lm_head.",
]:
    matching = [k for k in keys if k.startswith(prefix)]
    print(f"{prefix:22s} {len(matching)} tensors")

Encoder layers: 6
Decoder layers: 6

Key prefixes:
Counter({'decoder_layers': 156, 'encoder_layers': 96, 'encoder_norm': 2, 'decoder_norm': 2, 'token_embedding': 1, 'encoder_pos': 1, 'decoder_pos': 1, 'lm_head': 1})

Required parameter groups present:
token_embedding.       1 tensors
encoder_pos.           1 tensors
decoder_pos.           1 tensors
encoder_layers.        96 tensors
decoder_layers.        156 tensors
encoder_norm.          2 tensors
decoder_norm.          2 tensors
lm_head.               1 tensors


In [29]:

print("A4EncoderDecoder defined:", "A4EncoderDecoder" in globals())
print("model defined:", "model" in globals())


A4EncoderDecoder defined: False
model defined: False


In [31]:

import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = 50_000
D_MODEL = 512
NUM_HEADS = 8
FFN_DIM = 2048
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
ENCODER_CONTEXT = 1024
DECODER_CONTEXT = 1024
DROPOUT = 0.1


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, q, k, v, causal=False):
        B, Tq, C = q.shape
        Tk = k.shape[1]

        q = self.q_proj(q).view(
            B, Tq, self.num_heads, self.head_dim
        ).transpose(1, 2)
        k = self.k_proj(k).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)
        v = self.v_proj(v).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q, k, v,
            is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0
        )
        out = out.transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, ffn_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(D_MODEL)
        self.attn = MultiHeadAttention(D_MODEL, NUM_HEADS, DROPOUT)
        self.norm2 = nn.LayerNorm(D_MODEL)
        self.ffn = FeedForward(D_MODEL, FFN_DIM, DROPOUT)

    def forward(self, x):
        n = self.norm1(x)
        x = x + self.attn(n, n, n)
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(D_MODEL)
        self.self_attn = MultiHeadAttention(D_MODEL, NUM_HEADS, DROPOUT)
        self.norm2 = nn.LayerNorm(D_MODEL)
        self.cross_attn = MultiHeadAttention(D_MODEL, NUM_HEADS, DROPOUT)
        self.norm3 = nn.LayerNorm(D_MODEL)
        self.ffn = FeedForward(D_MODEL, FFN_DIM, DROPOUT)

    def forward(self, x, memory):
        n = self.norm1(x)
        x = x + self.self_attn(n, n, n, causal=True)
        x = x + self.cross_attn(
            self.norm2(x), memory, memory, causal=False
        )
        x = x + self.ffn(self.norm3(x))
        return x


class A4EncoderDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.encoder_pos = nn.Embedding(ENCODER_CONTEXT, D_MODEL)
        self.decoder_pos = nn.Embedding(DECODER_CONTEXT, D_MODEL)

        self.encoder_layers = nn.ModuleList([
            EncoderBlock() for _ in range(NUM_ENCODER_LAYERS)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderBlock() for _ in range(NUM_DECODER_LAYERS)
        ])

        self.encoder_norm = nn.LayerNorm(D_MODEL)
        self.decoder_norm = nn.LayerNorm(D_MODEL)
        self.lm_head = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, encoder_ids, decoder_ids):
        Te = encoder_ids.shape[1]
        Td = decoder_ids.shape[1]

        enc_pos = torch.arange(Te, device=encoder_ids.device)
        dec_pos = torch.arange(Td, device=decoder_ids.device)

        encoder = (
            self.token_embedding(encoder_ids)
            + self.encoder_pos(enc_pos)[None, :, :]
        )
        decoder = (
            self.token_embedding(decoder_ids)
            + self.decoder_pos(dec_pos)[None, :, :]
        )

        encoder = self.dropout(encoder)
        decoder = self.dropout(decoder)

        for layer in self.encoder_layers:
            encoder = layer(encoder)
        encoder = self.encoder_norm(encoder)

        for layer in self.decoder_layers:
            decoder = layer(decoder, encoder)
        decoder = self.decoder_norm(decoder)

        return self.lm_head(decoder)


print("A4EncoderDecoder class defined successfully.")


A4EncoderDecoder class defined successfully.


In [32]:

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = A4EncoderDecoder().to(device)

print("Model created successfully on:", device)


Model created successfully on: cuda


In [33]:

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = A4EncoderDecoder().to(device)

print("Model created successfully")
print("Device:", device)
print("Parameters (M):", sum(
    p.numel() for p in model.parameters()
) / 1e6)


Model created successfully
Device: cuda
Parameters (M): 70.78912


In [34]:

import torch

checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-22000-pt/a4_step_22000.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model = model.to(device).eval()

print("Checkpoint step:", checkpoint["step"])
print("Strict loading:", result)
print("Device:", device)
print("Parameter count:", sum(p.numel() for p in model.parameters()))
print("Checkpoint loading succeeded.")


Checkpoint step: 22000
Strict loading: <All keys matched successfully>
Device: cuda
Parameter count: 70789120
Checkpoint loading succeeded.


In [35]:
import torch

checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-22000-pt/a4_step_22000.pt"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

model = A4EncoderDecoder()

# Require an exact match: no missing or unexpected keys.
result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(device).eval()

print("Checkpoint step:", checkpoint["step"])
print("Strict loading:", result)
print("Device:", device)
print("Parameter count:", sum(p.numel() for p in model.parameters()))
print("Checkpoint loading succeeded.")

Checkpoint step: 22000
Strict loading: <All keys matched successfully>
Device: cuda
Parameter count: 70789120
Checkpoint loading succeeded.


In [36]:

with torch.no_grad():
    encoder_ids = torch.randint(
        0, VOCAB_SIZE, (1, 16), device=device
    )
    decoder_ids = torch.randint(
        0, VOCAB_SIZE, (1, 8), device=device
    )

    output = model(encoder_ids, decoder_ids)

print("Output shape:", tuple(output.shape))
print(
    "Forward pass successful:",
    output.shape == (1, 8, VOCAB_SIZE)
)


Output shape: (1, 8, 50000)
Forward pass successful: True


In [37]:

from tokenizers import Tokenizer

tokenizer_path = (
    "/kaggle/input/datasets/aq1de3/"
    "tokenizer-json/tokenizer.json"
)

tokenizer = Tokenizer.from_file(tokenizer_path)

print("Vocabulary size:", tokenizer.get_vocab_size())

for token in ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]:
    print(f"{token}:", tokenizer.token_to_id(token))

sample_text = "This is a test legal judgment."
encoded = tokenizer.encode(sample_text)

print("Sample token IDs:", encoded.ids)
print("Decoded text:", tokenizer.decode(encoded.ids))


Vocabulary size: 50000
[PAD]: 0
[UNK]: 1
[BOS]: 2
[EOS]: 3
[MASK]: 4
Sample token IDs: [1909, 291, 239, 2949, 1999, 942, 18]
Decoded text: This Ġis Ġa Ġtest Ġlegal Ġjudgment .


In [38]:

import json
import numpy as np
from pathlib import Path

train_path = "/kaggle/input/datasets/aq1de3/trainjson/train.json"
dev_path = "/kaggle/input/datasets/aq1de3/devjson/dev.json"

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(dev_path, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

def inspect_lengths(data, name, sample_size=500):
    # Deterministic sample to keep the initial inspection manageable.
    sample = data[:sample_size]
    lengths = [
        len(tokenizer.encode(row["text"]).ids)
        for row in sample
    ]
    print(f"\n{name}: {len(sample)} documents sampled")
    print("Median tokens:", int(np.median(lengths)))
    print("90th percentile:", int(np.percentile(lengths, 90)))
    print("Maximum tokens in sample:", max(lengths))
    print("Over 1024 tokens:", sum(n > 1024 for n in lengths))

inspect_lengths(train_data, "Train")
inspect_lengths(dev_data, "Development")



Train: 500 documents sampled
Median tokens: 3709
90th percentile: 10979
Maximum tokens in sample: 124262
Over 1024 tokens: 458

Development: 500 documents sampled
Median tokens: 3833
90th percentile: 8875
Maximum tokens in sample: 59887
Over 1024 tokens: 479


In [40]:

def chunk_count(token_count, chunk_size=1024, overlap=128):
    if token_count <= chunk_size:
        return 1
    stride = chunk_size - overlap
    return 1 + (token_count - chunk_size + stride - 1) // stride

for name, data in [("Train", train_data), ("Development", dev_data)]:
    sample = data[:500]
    lengths = [
        len(tokenizer.encode(row["text"]).ids)
        for row in sample
    ]
    counts = [chunk_count(n) for n in lengths]

    print(f"\n{name} — first 500 documents")
    print("Average chunks/document:", round(sum(counts) / len(counts), 2))
    print("Maximum chunks/document:", max(counts))
    print("Documents needing multiple chunks:", sum(c > 1 for c in counts))



Train — first 500 documents
Average chunks/document: 7.07
Maximum chunks/document: 139
Documents needing multiple chunks: 458

Development — first 500 documents
Average chunks/document: 6.03
Maximum chunks/document: 67
Documents needing multiple chunks: 479


In [41]:

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory allocated (GB):",
        round(torch.cuda.memory_allocated() / 1024**3, 2)
    )
    print(
        "GPU memory reserved (GB):",
        round(torch.cuda.memory_reserved() / 1024**3, 2)
    )
else:
    print("CUDA unavailable; currently using CPU.")


GPU: Tesla T4
GPU memory allocated (GB): 0.27
GPU memory reserved (GB): 0.53


In [42]:

print("Training rows:", len(train_data))
print("First record keys:", list(train_data[0].keys()))
print("First record ID:", train_data[0]["id"])
print("First record label:", train_data[0]["label"])
print("Label counts:")

from collections import Counter
print(Counter(row["label"] for row in train_data))


Training rows: 37387
First record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
First record ID: 2020_1
First record label: 0
Label counts:
Counter({0: 22067, 1: 15320})


In [43]:

print("Model class:", type(model).__name__)
print("Encoder layers:", len(model.encoder_layers))
print("Encoder normalization:", type(model.encoder_norm).__name__)
print("Token embedding shape:", tuple(model.token_embedding.weight.shape))
print("Encoder position limit:", model.encoder_pos.num_embeddings)


Model class: A4EncoderDecoder
Encoder layers: 6
Encoder normalization: LayerNorm
Token embedding shape: (50000, 512)
Encoder position limit: 1024


In [45]:

import torch

model.eval()

with torch.no_grad():
    sample_ids = torch.tensor(
        [[2, 1909, 291, 239, 2949, 3]],
        dtype=torch.long,
        device=device
    )

    positions = torch.arange(
        sample_ids.shape[1],
        device=device
    )

    x = (
        model.token_embedding(sample_ids)
        + model.encoder_pos(positions)[None, :, :]
    )

    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    encoder_output = model.encoder_norm(x)

print("Input shape:", tuple(sample_ids.shape))
print("Encoder output shape:", tuple(encoder_output.shape))
print("Output finite:", torch.isfinite(encoder_output).all().item())


Input shape: (1, 6)
Encoder output shape: (1, 6, 512)
Output finite: True


In [46]:

import torch

# Inspect a small batch of real training records
batch_size = 2
records = train_data[:batch_size]

texts = [row["text"] for row in records]
labels = torch.tensor(
    [int(row["label"]) for row in records],
    dtype=torch.long,
    device=device
)

max_length = 1024
pad_id = tokenizer.token_to_id("[PAD]")
bos_id = tokenizer.token_to_id("[BOS]")
eos_id = tokenizer.token_to_id("[EOS]")

encoded_batch = []

for text in texts:
    ids = tokenizer.encode(text).ids
    ids = ids[:max_length]

    # Reserve space for BOS/EOS
    ids = ids[:max_length - 2]
    ids = [bos_id] + ids + [eos_id]

    encoded_batch.append(ids)

# Pad sequences to the longest sequence in this batch
batch_length = max(len(ids) for ids in encoded_batch)
input_ids = torch.full(
    (batch_size, batch_length),
    pad_id,
    dtype=torch.long,
    device=device
)
attention_mask = torch.zeros_like(input_ids)

for i, ids in enumerate(encoded_batch):
    input_ids[i, :len(ids)] = torch.tensor(ids, device=device)
    attention_mask[i, :len(ids)] = 1

print("Input shape:", tuple(input_ids.shape))
print("Attention mask shape:", tuple(attention_mask.shape))
print("Labels:", labels.tolist())
print("Sequence lengths:", attention_mask.sum(dim=1).tolist())
print("Padding ID:", pad_id)


Input shape: (2, 1024)
Attention mask shape: (2, 1024)
Labels: [0, 0]
Sequence lengths: [1024, 1024]
Padding ID: 0


In [47]:

model.eval()

with torch.no_grad():
    x = model.token_embedding(input_ids)

    positions = torch.arange(
        input_ids.shape[1], device=device
    )
    x = x + model.encoder_pos(positions)[None, :, :]
    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    token_representations = model.encoder_norm(x)

    # Exclude padding positions from the pooled representation
    mask = attention_mask.unsqueeze(-1).to(
        token_representations.dtype
    )
    pooled = (
        (token_representations * mask).sum(dim=1)
        / mask.sum(dim=1).clamp(min=1)
    )

print("Token representations:", tuple(token_representations.shape))
print("Pooled representations:", tuple(pooled.shape))
print("Finite values:", torch.isfinite(pooled).all().item())


Token representations: (2, 1024, 512)
Pooled representations: (2, 512)
Finite values: True


In [48]:

import torch
import torch.nn as nn

classification_head = nn.Linear(512, 2).to(device)

# Test with the pooled representations from the previous cell
logits = classification_head(pooled)

print("Pooled representation shape:", tuple(pooled.shape))
print("Classification logits shape:", tuple(logits.shape))
print("Predicted classes:", logits.argmax(dim=1).tolist())


Pooled representation shape: (2, 512)
Classification logits shape: (2, 2)
Predicted classes: [0, 0]


In [49]:

import torch.nn.functional as F

loss = F.cross_entropy(logits, labels)

print("Logits shape:", tuple(logits.shape))
print("Labels shape:", tuple(labels.shape))
print("Cross-entropy loss:", loss.item())
print("Loss is finite:", torch.isfinite(loss).item())


Logits shape: (2, 2)
Labels shape: (2,)
Cross-entropy loss: 0.6493417620658875
Loss is finite: True


In [50]:

def count_parameters(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(
        p.numel() for p in module.parameters()
        if p.requires_grad
    )
    return total, trainable

model_total, model_trainable = count_parameters(model)
head_total, head_trainable = count_parameters(classification_head)

print("A4 total parameters:", f"{model_total:,}")
print("A4 trainable parameters:", f"{model_trainable:,}")
print("Classification-head parameters:", f"{head_total:,}")
print("Classification-head trainable:", f"{head_trainable:,}")


A4 total parameters: 70,789,120
A4 trainable parameters: 70,789,120
Classification-head parameters: 1,026
Classification-head trainable: 1,026


In [51]:

# Freeze pretrained A4 parameters
for param in model.parameters():
    param.requires_grad = False

# Keep the classification head trainable
for param in classification_head.parameters():
    param.requires_grad = True

print(
    "A4 trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)
print(
    "Head trainable parameters:",
    sum(p.numel() for p in classification_head.parameters()
        if p.requires_grad)
)


A4 trainable parameters: 0
Head trainable parameters: 1026


In [52]:

import torch
import torch.nn as nn

# Set encoder to evaluation mode; keep the head ready for training
model.eval()
classification_head.train()

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    classification_head.parameters(),
    lr=1e-3
)

print("Encoder frozen:", all(
    not p.requires_grad for p in model.parameters()
))
print("Head in training mode:", classification_head.training)
print("Optimizer:", type(optimizer).__name__)
print("Loss:", type(criterion).__name__)


Encoder frozen: True
Head in training mode: True
Optimizer: AdamW
Loss: CrossEntropyLoss


In [53]:

# Reuse the pooled representations and labels from the previous test batch.
# This checks the optimizer independently of the encoder.

classification_head.train()
optimizer.zero_grad(set_to_none=True)

logits = classification_head(pooled)
loss = criterion(logits, labels)

loss.backward()

print("Loss:", loss.item())
print(
    "Head gradient exists:",
    classification_head.weight.grad is not None
)
print(
    "Gradient finite:",
    torch.isfinite(classification_head.weight.grad).all().item()
)

optimizer.step()

print("One optimizer update completed.")


Loss: 0.6493417620658875
Head gradient exists: True
Gradient finite: True
One optimizer update completed.


In [54]:

from collections import Counter

print("Training records:", len(train_data))
print("Development records:", len(dev_data))

print("Training labels:", Counter(
    int(row["label"]) for row in train_data
))
print("Development labels:", Counter(
    int(row["label"]) for row in dev_data
))

assert all(int(row["label"]) in (0, 1) for row in train_data)
assert all(int(row["label"]) in (0, 1) for row in dev_data)

print("Dataset and binary-label checks passed.")


Training records: 37387
Development records: 3505
Training labels: Counter({0: 22067, 1: 15320})
Development labels: Counter({1: 1756, 0: 1749})
Dataset and binary-label checks passed.


In [55]:

import torch

def prepare_batch(records, max_length=1024):
    pad_id = tokenizer.token_to_id("[PAD]")
    bos_id = tokenizer.token_to_id("[BOS]")
    eos_id = tokenizer.token_to_id("[EOS]")

    all_ids = []
    all_labels = []

    for row in records:
        ids = tokenizer.encode(row["text"]).ids
        ids = ids[:max_length - 2]
        ids = [bos_id] + ids + [eos_id]

        # Fixed-length padding
        ids += [pad_id] * (max_length - len(ids))

        all_ids.append(ids)
        all_labels.append(int(row["label"]))

    input_ids = torch.tensor(all_ids, dtype=torch.long, device=device)
    labels = torch.tensor(all_labels, dtype=torch.long, device=device)
    attention_mask = (input_ids != pad_id).long()

    return input_ids, attention_mask, labels

# Test with two records
test_ids, test_mask, test_labels = prepare_batch(train_data[:2])

print("Input IDs:", tuple(test_ids.shape))
print("Mask:", tuple(test_mask.shape))
print("Labels:", tuple(test_labels.shape))
print("Valid label values:", sorted(set(test_labels.tolist())))
print("All shapes and labels valid:",
      test_ids.shape == (2, 1024)
      and test_mask.shape == (2, 1024)
      and test_labels.shape == (2,))


Input IDs: (2, 1024)
Mask: (2, 1024)
Labels: (2,)
Valid label values: [0]
All shapes and labels valid: True


In [57]:

model.eval()
classification_head.eval()

with torch.no_grad():
    positions = torch.arange(1024, device=device)

    x = (
        model.token_embedding(test_ids)
        + model.encoder_pos(positions)[None, :, :]
    )
    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    token_representations = model.encoder_norm(x)

    mask = test_mask.unsqueeze(-1).to(token_representations.dtype)
    pooled_test = (
        (token_representations * mask).sum(dim=1)
        / mask.sum(dim=1).clamp(min=1)
    )

    test_logits = classification_head(pooled_test)
    test_loss = criterion(test_logits, test_labels)

print("Pooled shape:", tuple(pooled_test.shape))
print("Logits shape:", tuple(test_logits.shape))
print("Loss finite:", torch.isfinite(test_loss).item())
print("Forward pass valid:", test_logits.shape == (2, 2))


Pooled shape: (2, 512)
Logits shape: (2, 2)
Loss finite: True
Forward pass valid: True


In [58]:

# Select one training example from each class
record_0 = next(r for r in train_data if int(r["label"]) == 0)
record_1 = next(r for r in train_data if int(r["label"]) == 1)
records = [record_0, record_1]

input_ids, attention_mask, labels = prepare_batch(records)

model.eval()
classification_head.train()
optimizer.zero_grad(set_to_none=True)

# Frozen encoder: no gradient calculation needed
with torch.no_grad():
    positions = torch.arange(input_ids.shape[1], device=device)

    x = (
        model.token_embedding(input_ids)
        + model.encoder_pos(positions)[None, :, :]
    )

    for layer in model.encoder_layers:
        x = layer(x)

    token_representations = model.encoder_norm(x)

    mask = attention_mask.unsqueeze(-1).to(token_representations.dtype)
    pooled = (
        (token_representations * mask).sum(dim=1)
        / mask.sum(dim=1).clamp(min=1)
    )

# Train the classification head
logits = classification_head(pooled)
loss = criterion(logits, labels)

loss.backward()
optimizer.step()

print("Labels:", labels.tolist())
print("Logits shape:", tuple(logits.shape))
print("Training loss:", round(loss.item(), 4))
print("Gradient finite:", torch.isfinite(
    classification_head.weight.grad
).all().item())
print("One training update completed.")


Labels: [0, 1]
Logits shape: (2, 2)
Training loss: 0.6902
Gradient finite: True
One training update completed.


In [59]:

import random
from collections import Counter

random.seed(42)

# Select a balanced subset for the initial smoke test
class_0 = [r for r in train_data if int(r["label"]) == 0]
class_1 = [r for r in train_data if int(r["label"]) == 1]

n_per_class = 32
trial_records = (
    random.sample(class_0, n_per_class)
    + random.sample(class_1, n_per_class)
)
random.shuffle(trial_records)

print("Trial records:", len(trial_records))
print("Trial label counts:", Counter(
    int(r["label"]) for r in trial_records
))
print("Unique IDs:", len({r["id"] for r in trial_records}))


Trial records: 64
Trial label counts: Counter({0: 32, 1: 32})
Unique IDs: 64


In [60]:

import math

batch_size = 2

num_batches = math.ceil(len(trial_records) / batch_size)

print("Trial records:", len(trial_records))
print("Batch size:", batch_size)
print("Batches per pass:", num_batches)
print("Last batch size:", len(trial_records) % batch_size or batch_size)
print("Optimizer parameters:", sum(
    p.numel()
    for group in optimizer.param_groups
    for p in group["params"]
))


Trial records: 64
Batch size: 2
Batches per pass: 32
Last batch size: 2
Optimizer parameters: 1026


In [61]:

import random
import torch

random.seed(42)

batch_size = 2
random.shuffle(trial_records)

model.eval()                    # Frozen A4 encoder
classification_head.train()    # Train classification head

epoch_losses = []
correct = 0
total = 0

for start in range(0, len(trial_records), batch_size):
    batch_records = trial_records[start:start + batch_size]

    input_ids, attention_mask, labels = prepare_batch(batch_records)

    optimizer.zero_grad(set_to_none=True)

    # Extract frozen encoder representations
    with torch.no_grad():
        positions = torch.arange(
            input_ids.shape[1], device=device
        )

        x = (
            model.token_embedding(input_ids)
            + model.encoder_pos(positions)[None, :, :]
        )

        # Match the encoder's training-time computation.
        # Padding is masked during pooling, not during attention.
        for layer in model.encoder_layers:
            x = layer(x)

        token_representations = model.encoder_norm(x)

        mask = attention_mask.unsqueeze(-1).to(
            token_representations.dtype
        )

        pooled = (
            (token_representations * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1)
        )

    # Train the classification head
    logits = classification_head(pooled)
    loss = criterion(logits, labels)

    if not torch.isfinite(loss):
        raise RuntimeError("Non-finite loss encountered.")

    loss.backward()

    if classification_head.weight.grad is None:
        raise RuntimeError("Classification-head gradient is missing.")

    optimizer.step()

    epoch_losses.append(loss.item())
    predictions = logits.argmax(dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

print("Training records:", len(trial_records))
print("Batches completed:", len(epoch_losses))
print("Average training loss:", round(sum(epoch_losses) / len(epoch_losses), 4))
print("Training accuracy:", round(correct / total, 4))
print("Encoder frozen:", all(
    not p.requires_grad for p in model.parameters()
))
print("Head trainable:", all(
    p.requires_grad for p in classification_head.parameters()
))


Training records: 64
Batches completed: 32
Average training loss: 0.7449
Training accuracy: 0.3438
Encoder frozen: True
Head trainable: True


In [62]:

model.eval()
classification_head.eval()

all_true = []
all_pred = []

with torch.no_grad():
    for start in range(0, len(trial_records), 2):
        records = trial_records[start:start + 2]
        input_ids, attention_mask, labels = prepare_batch(records)

        positions = torch.arange(input_ids.shape[1], device=device)
        x = (
            model.token_embedding(input_ids)
            + model.encoder_pos(positions)[None, :, :]
        )

        for layer in model.encoder_layers:
            x = layer(x)

        token_representations = model.encoder_norm(x)

        mask = attention_mask.unsqueeze(-1).to(token_representations.dtype)
        pooled = (
            (token_representations * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1)
        )

        logits = classification_head(pooled)
        preds = logits.argmax(dim=1)

        all_true.extend(labels.cpu().tolist())
        all_pred.extend(preds.cpu().tolist())

from collections import Counter

print("True label counts:", Counter(all_true))
print("Predicted label counts:", Counter(all_pred))
print("Correct:", sum(t == p for t, p in zip(all_true, all_pred)), "/", len(all_true))


True label counts: Counter({0: 32, 1: 32})
Predicted label counts: Counter({1: 44, 0: 20})
Correct: 36 / 64


In [63]:

import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

model.eval()
classification_head.eval()

y_true = []
y_pred = []
batch_size = 2

with torch.no_grad():
    for start in range(0, len(dev_data), batch_size):
        records = dev_data[start:start + batch_size]

        input_ids, attention_mask, labels = prepare_batch(records)

        positions = torch.arange(
            input_ids.shape[1], device=device
        )

        x = (
            model.token_embedding(input_ids)
            + model.encoder_pos(positions)[None, :, :]
        )

        for layer in model.encoder_layers:
            x = layer(x)

        token_representations = model.encoder_norm(x)

        mask = attention_mask.unsqueeze(-1).to(
            token_representations.dtype
        )

        pooled = (
            (token_representations * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1)
        )

        logits = classification_head(pooled)
        predictions = logits.argmax(dim=1)

        y_true.extend(labels.cpu().tolist())
        y_pred.extend(predictions.cpu().tolist())

print("Development records:", len(y_true))
print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Macro-F1:", round(f1_score(y_true, y_pred, average="macro"), 4))
print("Macro precision:", round(precision_score(
    y_true, y_pred, average="macro", zero_division=0
), 4))
print("Macro recall:", round(recall_score(
    y_true, y_pred, average="macro", zero_division=0
), 4))

print("\nConfusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

print("\nClass-wise report:")
print(classification_report(
    y_true, y_pred, labels=[0, 1], zero_division=0
))


Development records: 3505
Accuracy: 0.5067
Macro-F1: 0.5057
Macro precision: 0.5069
Macro recall: 0.5068

Confusion matrix (rows=true, columns=predicted):
[[967 782]
 [947 809]]

Class-wise report:
              precision    recall  f1-score   support

           0       0.51      0.55      0.53      1749
           1       0.51      0.46      0.48      1756

    accuracy                           0.51      3505
   macro avg       0.51      0.51      0.51      3505
weighted avg       0.51      0.51      0.51      3505



In [64]:

import math

batch_size = 2
num_train_records = len(train_data)
num_batches = math.ceil(num_train_records / batch_size)

print("Training records:", num_train_records)
print("Batch size:", batch_size)
print("Batches per epoch:", num_batches)
print("Encoder frozen:", all(
    not p.requires_grad for p in model.parameters()
))
print("Head trainable:", all(
    p.requires_grad for p in classification_head.parameters()
))
print("Optimizer parameters:", sum(
    p.numel()
    for group in optimizer.param_groups
    for p in group["params"]
))


Training records: 37387
Batch size: 2
Batches per epoch: 18694
Encoder frozen: True
Head trainable: True
Optimizer parameters: 1026


In [65]:

import time
import numpy as np
import torch
from torch.utils.data import DataLoader

# Verify prerequisites
assert "model" in globals(), "Load the A4 checkpoint first."
assert "tokenizer" in globals(), "Load the tokenizer first."
assert "train_data" in globals(), "Load train.json first."
assert "dev_data" in globals(), "Load dev.json first."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 1024
BATCH_SIZE = 4

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("Device:", device)
print("Training records:", len(train_data))
print("Development records:", len(dev_data))
print("Batch size:", BATCH_SIZE)
print("Maximum sequence length:", MAX_LENGTH)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


Device: cuda
Training records: 37387
Development records: 3505
Batch size: 4
Maximum sequence length: 1024
GPU: Tesla T4


In [66]:

import time
import numpy as np

MAX_LENGTH = 1024
CONTENT_LENGTH = MAX_LENGTH - 2

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

def tokenize_records(records, has_labels=True):
    sequences = []
    labels = []

    # Truncate during tokenization to avoid processing entire long documents.
    tokenizer.enable_truncation(max_length=CONTENT_LENGTH)

    try:
        for i, row in enumerate(records):
            ids = tokenizer.encode(row["text"]).ids
            ids = [BOS_ID] + ids + [EOS_ID]
            sequences.append(ids)

            if has_labels:
                labels.append(int(row["label"]))

            if (i + 1) % 5000 == 0:
                print(f"Tokenized {i + 1:,}/{len(records):,}")
    finally:
        tokenizer.no_truncation()

    # Store token IDs compactly; all IDs fit in uint16.
    token_array = np.zeros(
        (len(sequences), MAX_LENGTH), dtype=np.uint16
    )
    mask_array = np.zeros(
        (len(sequences), MAX_LENGTH), dtype=np.bool_
    )

    for i, ids in enumerate(sequences):
        length = min(len(ids), MAX_LENGTH)
        token_array[i, :length] = ids[:length]
        mask_array[i, :length] = True

    result = {
        "input_ids": token_array,
        "attention_mask": mask_array,
    }

    if has_labels:
        result["labels"] = np.asarray(labels, dtype=np.int64)

    return result

start = time.time()

train_tokens = tokenize_records(train_data, has_labels=True)
dev_tokens = tokenize_records(dev_data, has_labels=True)

print("\nTokenization complete.")
print("Train shape:", train_tokens["input_ids"].shape)
print("Dev shape:", dev_tokens["input_ids"].shape)
print("Elapsed minutes:", round((time.time() - start) / 60, 2))
print("Train labels:", np.bincount(train_tokens["labels"]))
print("Dev labels:", np.bincount(dev_tokens["labels"]))


Tokenized 5,000/37,387
Tokenized 10,000/37,387
Tokenized 15,000/37,387
Tokenized 20,000/37,387
Tokenized 25,000/37,387
Tokenized 30,000/37,387
Tokenized 35,000/37,387

Tokenization complete.
Train shape: (37387, 1024)
Dev shape: (3505, 1024)
Elapsed minutes: 5.0
Train labels: [22067 15320]
Dev labels: [1749 1756]


In [67]:

import time
import numpy as np
import torch

from torch.utils.data import DataLoader, TensorDataset

# Freeze the pretrained encoder
model.eval()
for p in model.parameters():
    p.requires_grad = False

BATCH_SIZE = 4

def cache_representations(tokenized_data, name):
    ids = torch.from_numpy(tokenized_data["input_ids"].astype(np.int32))
    masks = torch.from_numpy(tokenized_data["attention_mask"].astype(np.int32))

    loader = DataLoader(
        TensorDataset(ids, masks),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    cached = []
    start = time.time()

    with torch.inference_mode():
        for step, (input_ids, attention_mask) in enumerate(loader):
            input_ids = input_ids.long().to(device)
            attention_mask = attention_mask.bool().to(device)

            batch_size, seq_len = input_ids.shape
            positions = torch.arange(seq_len, device=device)

            x = model.token_embedding(input_ids)
            x = x + model.encoder_pos(positions)[None, :, :]
            x = model.dropout(x)

            for layer in model.encoder_layers:
                x = layer(x)

            x = model.encoder_norm(x)

            # Masked mean pooling
            mask = attention_mask.unsqueeze(-1).to(x.dtype)
            pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

            cached.append(pooled.cpu().to(torch.float32))

            if (step + 1) % 500 == 0:
                print(
                    f"{name}: {min((step + 1) * BATCH_SIZE, len(ids)):,}"
                    f"/{len(ids):,} documents processed"
                )

    result = torch.cat(cached, dim=0)
    print(
        f"{name} cached: {tuple(result.shape)} | "
        f"time: {(time.time() - start) / 60:.2f} minutes"
    )
    return result

train_features = cache_representations(train_tokens, "Train")
dev_features = cache_representations(dev_tokens, "Dev")

train_labels_tensor = torch.tensor(
    train_tokens["labels"], dtype=torch.long
)
dev_labels_tensor = torch.tensor(
    dev_tokens["labels"], dtype=torch.long
)

print("\nRepresentation caching complete.")
print("Train features:", train_features.shape)
print("Dev features:", dev_features.shape)
print("Encoder frozen:", all(not p.requires_grad for p in model.parameters()))
print("Finite train features:", torch.isfinite(train_features).all().item())


Train: 2,000/37,387 documents processed
Train: 4,000/37,387 documents processed
Train: 6,000/37,387 documents processed
Train: 8,000/37,387 documents processed
Train: 10,000/37,387 documents processed
Train: 12,000/37,387 documents processed
Train: 14,000/37,387 documents processed
Train: 16,000/37,387 documents processed
Train: 18,000/37,387 documents processed
Train: 20,000/37,387 documents processed
Train: 22,000/37,387 documents processed
Train: 24,000/37,387 documents processed
Train: 26,000/37,387 documents processed
Train: 28,000/37,387 documents processed
Train: 30,000/37,387 documents processed
Train: 32,000/37,387 documents processed
Train: 34,000/37,387 documents processed
Train: 36,000/37,387 documents processed
Train cached: (37387, 512) | time: 12.21 minutes
Dev: 2,000/3,505 documents processed
Dev cached: (3505, 512) | time: 1.15 minutes

Representation caching complete.
Train features: torch.Size([37387, 512])
Dev features: torch.Size([3505, 512])
Encoder frozen: True
F

In [68]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import time

# Reset the classifier so this is a fresh full-data experiment
torch.manual_seed(42)
classification_head = nn.Linear(512, 2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    classification_head.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

train_dataset = TensorDataset(
    train_features,
    train_labels_tensor
)
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

EPOCHS = 10

print("Training records:", len(train_dataset))
print("Batches per epoch:", len(train_loader))
print("Epochs:", EPOCHS)
print("Trainable classifier parameters:",
      sum(p.numel() for p in classification_head.parameters() if p.requires_grad))

history = []
start_time = time.time()

for epoch in range(EPOCHS):
    classification_head.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = classification_head(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    epoch_loss = total_loss / total
    epoch_accuracy = correct / total

    history.append({
        "epoch": epoch + 1,
        "loss": epoch_loss,
        "accuracy": epoch_accuracy
    })

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_accuracy:.4f} | "
        f"Elapsed: {(time.time() - start_time) / 60:.2f} min"
    )

print("\nClassifier training complete.")


Training records: 37387
Batches per epoch: 147
Epochs: 10
Trainable classifier parameters: 1026
Epoch 1/10 | Loss: 0.6752 | Accuracy: 0.5898 | Elapsed: 0.01 min
Epoch 2/10 | Loss: 0.6725 | Accuracy: 0.5906 | Elapsed: 0.02 min
Epoch 3/10 | Loss: 0.6700 | Accuracy: 0.5930 | Elapsed: 0.02 min
Epoch 4/10 | Loss: 0.6700 | Accuracy: 0.5925 | Elapsed: 0.03 min
Epoch 5/10 | Loss: 0.6690 | Accuracy: 0.5953 | Elapsed: 0.04 min
Epoch 6/10 | Loss: 0.6686 | Accuracy: 0.5953 | Elapsed: 0.04 min
Epoch 7/10 | Loss: 0.6687 | Accuracy: 0.5959 | Elapsed: 0.05 min
Epoch 8/10 | Loss: 0.6672 | Accuracy: 0.5963 | Elapsed: 0.06 min
Epoch 9/10 | Loss: 0.6670 | Accuracy: 0.5983 | Elapsed: 0.07 min
Epoch 10/10 | Loss: 0.6671 | Accuracy: 0.5992 | Elapsed: 0.07 min

Classifier training complete.


In [69]:

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

classification_head.eval()

dev_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(
        dev_features,
        dev_labels_tensor
    ),
    batch_size=512,
    shuffle=False
)

all_predictions = []
all_labels = []

with torch.inference_mode():
    for features, labels in dev_loader:
        features = features.to(device)
        logits = classification_head(features)
        predictions = logits.argmax(dim=1).cpu().numpy()

        all_predictions.extend(predictions.tolist())
        all_labels.extend(labels.numpy().tolist())

y_true = np.asarray(all_labels)
y_pred = np.asarray(all_predictions)

print("Development records:", len(y_true))
print(f"Accuracy:   {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro-F1:   {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Macro Precision: {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"Macro Recall:    {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")

print("\nConfusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y_true, y_pred))

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    labels=[0, 1],
    zero_division=0
))

print("Predicted label counts:", np.bincount(y_pred, minlength=2))


Development records: 3505
Accuracy:   0.5258
Macro-F1:   0.4487
Macro Precision: 0.5607
Macro Recall:    0.5266

Confusion matrix (rows=true, columns=predicted):
[[1577  172]
 [1490  266]]

Classification report:
              precision    recall  f1-score   support

           0       0.51      0.90      0.65      1749
           1       0.61      0.15      0.24      1756

    accuracy                           0.53      3505
   macro avg       0.56      0.53      0.45      3505
weighted avg       0.56      0.53      0.45      3505

Predicted label counts: [3067  438]


In [70]:

import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

torch.manual_seed(43)

# Inverse-frequency weights from the training labels
counts = np.bincount(train_tokens["labels"], minlength=2)
weights = len(train_tokens["labels"]) / (2.0 * counts)
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

print("Training label counts:", counts)
print("Class weights:", class_weights.tolist())

weighted_head = nn.Linear(512, 2).to(device)
weighted_criterion = nn.CrossEntropyLoss(weight=class_weights)
weighted_optimizer = torch.optim.AdamW(
    weighted_head.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

weighted_loader = DataLoader(
    TensorDataset(train_features, train_labels_tensor),
    batch_size=256,
    shuffle=True
)

for epoch in range(10):
    weighted_head.train()
    total_loss = 0.0

    for features, labels in weighted_loader:
        features = features.to(device)
        labels = labels.to(device)

        weighted_optimizer.zero_grad()
        logits = weighted_head(features)
        loss = weighted_criterion(logits, labels)
        loss.backward()
        weighted_optimizer.step()
        total_loss += loss.item() * labels.size(0)

    print(
        f"Epoch {epoch + 1}/10 | "
        f"Weighted loss: {total_loss / len(train_labels_tensor):.4f}"
    )

weighted_head.eval()
predictions = []

with torch.inference_mode():
    for features, _ in torch.utils.data.DataLoader(
        TensorDataset(dev_features, dev_labels_tensor),
        batch_size=512,
        shuffle=False
    ):
        logits = weighted_head(features.to(device))
        predictions.extend(logits.argmax(dim=1).cpu().tolist())

y_true = dev_labels_tensor.numpy()
y_pred_weighted = np.asarray(predictions)

print("\nWeighted classifier results")
print("Accuracy:", round(accuracy_score(y_true, y_pred_weighted), 4))
print("Macro-F1:", round(f1_score(y_true, y_pred_weighted, average="macro"), 4))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred_weighted))
print("Predicted counts:", np.bincount(y_pred_weighted, minlength=2))


Training label counts: [22067 15320]
Class weights: [0.8471246361732483, 1.2202023267745972]
Epoch 1/10 | Weighted loss: 0.6917
Epoch 2/10 | Weighted loss: 0.6887
Epoch 3/10 | Weighted loss: 0.6862
Epoch 4/10 | Weighted loss: 0.6860
Epoch 5/10 | Weighted loss: 0.6863
Epoch 6/10 | Weighted loss: 0.6855
Epoch 7/10 | Weighted loss: 0.6844
Epoch 8/10 | Weighted loss: 0.6841
Epoch 9/10 | Weighted loss: 0.6842
Epoch 10/10 | Weighted loss: 0.6836

Weighted classifier results
Accuracy: 0.5327
Macro-F1: 0.5235
Confusion matrix:
[[1177  572]
 [1066  690]]
Predicted counts: [2243 1262]


In [71]:

import json

experiment_results = [
    {
        "experiment": "A4 frozen encoder - standard loss",
        "accuracy": 0.5258,
        "macro_f1": 0.4487,
        "macro_precision": 0.5607,
        "macro_recall": 0.5266,
        "trainable_parameters": 1026
    },
    {
        "experiment": "A4 frozen encoder - class-weighted loss",
        "accuracy": 0.5327,
        "macro_f1": 0.5235,
        "trainable_parameters": 1026
    }
]

with open("/kaggle/working/phase4_experiment_results.json", "w") as f:
    json.dump(experiment_results, f, indent=2)

print("Saved experiment log.")
for result in experiment_results:
    print(result["experiment"], "| Macro-F1:", result["macro_f1"])


Saved experiment log.
A4 frozen encoder - standard loss | Macro-F1: 0.4487
A4 frozen encoder - class-weighted loss | Macro-F1: 0.5235


In [72]:

import torch
import torch.nn as nn

# Restore the original pretrained A4 weights before this experiment.
# This prevents changes from another experiment carrying over.
checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-22000-pt/a4_step_22000.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(device)
model.eval()

# Freeze every pretrained parameter first
for parameter in model.parameters():
    parameter.requires_grad = False

# BitFit: train bias parameters only
for name, parameter in model.named_parameters():
    if name.endswith(".bias"):
        parameter.requires_grad = True

# Fresh classification head
torch.manual_seed(44)
bitfit_head = nn.Linear(512, 2).to(device)

# Trainable parameters include encoder biases and the classification head
bitfit_parameters = [
    parameter for parameter in model.parameters()
    if parameter.requires_grad
] + list(bitfit_head.parameters())

trainable_count = sum(
    parameter.numel() for parameter in bitfit_parameters
)
total_count = sum(parameter.numel() for parameter in model.parameters())
head_count = sum(parameter.numel() for parameter in bitfit_head.parameters())

print("Experiment: A4 BitFit")
print("Checkpoint step:", checkpoint.get("step", "unknown"))
print("Total encoder parameters:", f"{total_count:,}")
print("Trainable encoder bias parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Trainable head parameters:", f"{head_count:,}")
print("Total trainable parameters:", f"{trainable_count:,}")
print("Encoder restored and BitFit configured.")


Experiment: A4 BitFit
Checkpoint step: 22000
Total encoder parameters: 70,789,120
Trainable encoder bias parameters: 83,968
Trainable head parameters: 1,026
Total trainable parameters: 84,994
Encoder restored and BitFit configured.


In [73]:

import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Weighted loss, using the class weights calculated from training data
counts = torch.bincount(train_labels_tensor)
class_weights = (
    len(train_labels_tensor) / (2.0 * counts.float())
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Include only trainable BitFit parameters and the new head
optimizer = torch.optim.AdamW(
    bitfit_parameters,
    lr=1e-4,
    weight_decay=1e-4
)

# Cached features are fixed representations from the frozen encoder.
# To genuinely train BitFit biases, we must run the encoder forward
# during training rather than train only on the cached features.
print("Important: cached features cannot train encoder biases.")
print("This cell checks whether raw token batches are available.")

assert "train_tokens" in globals(), "Training tokens are missing."
assert "train_data" in globals(), "Training records are missing."

print("BitFit parameter setup verified.")
print("Next: run the raw-token BitFit training loop.")


Important: cached features cannot train encoder biases.
This cell checks whether raw token batches are available.
BitFit parameter setup verified.
Next: run the raw-token BitFit training loop.


In [74]:

import time
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Ensure only BitFit biases and the classifier head are trainable
model.eval()
for name, parameter in model.named_parameters():
    parameter.requires_grad = name.endswith(".bias")

bitfit_head.train()

# Verify the model and head use the same device
model.to(device)
bitfit_head.to(device)

class RawTokenDataset(Dataset):
    def __init__(self, tokenized):
        self.ids = tokenized["input_ids"]
        self.masks = tokenized["attention_mask"]
        self.labels = tokenized["labels"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return (
            torch.from_numpy(self.ids[index].astype(np.int64)),
            torch.from_numpy(self.masks[index].astype(np.bool_)),
            int(self.labels[index])
        )

train_loader_bitfit = DataLoader(
    RawTokenDataset(train_tokens),
    batch_size=2,
    shuffle=True,
    num_workers=0
)

counts = np.bincount(train_tokens["labels"], minlength=2)
weights = len(train_tokens["labels"]) / (2.0 * counts)
class_weights = torch.tensor(
    weights, dtype=torch.float32, device=device
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

bitfit_parameters = [
    p for p in model.parameters() if p.requires_grad
] + list(bitfit_head.parameters())

optimizer = torch.optim.AdamW(
    bitfit_parameters,
    lr=1e-5,
    weight_decay=1e-4
)

print("Trainable parameters:", sum(p.numel() for p in bitfit_parameters))
print("Training batches:", len(train_loader_bitfit))
print("Batch size: 2")
print("Sequence length: 1024")
print("Starting one full BitFit epoch...")

start = time.time()
model.train()

total_loss = 0.0
total_correct = 0
total_examples = 0

for step, (input_ids, attention_mask, labels) in enumerate(
    train_loader_bitfit, start=1
):
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    positions = torch.arange(input_ids.shape[1], device=device)
    x = model.token_embedding(input_ids)
    x = x + model.encoder_pos(positions)[None, :, :]
    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    x = model.encoder_norm(x)

    mask = attention_mask.unsqueeze(-1).to(x.dtype)
    pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    logits = bitfit_head(pooled)
    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()

    total_loss += loss.item() * labels.size(0)
    total_correct += (logits.argmax(dim=1) == labels).sum().item()
    total_examples += labels.size(0)

    if step % 500 == 0:
        print(
            f"Batch {step}/{len(train_loader_bitfit)} | "
            f"Loss: {total_loss / total_examples:.4f} | "
            f"Elapsed: {(time.time() - start) / 60:.1f} min"
        )

model.eval()
bitfit_head.eval()

print("\nBitFit epoch complete.")
print("Mean training loss:", round(total_loss / total_examples, 4))
print("Training accuracy:", round(total_correct / total_examples, 4))
print("Elapsed minutes:", round((time.time() - start) / 60, 2))


Trainable parameters: 84994
Training batches: 18694
Batch size: 2
Sequence length: 1024
Starting one full BitFit epoch...
Batch 500/18694 | Loss: 0.6961 | Elapsed: 1.0 min
Batch 1000/18694 | Loss: 0.6932 | Elapsed: 2.0 min
Batch 1500/18694 | Loss: 0.6913 | Elapsed: 3.0 min
Batch 2000/18694 | Loss: 0.6924 | Elapsed: 4.0 min
Batch 2500/18694 | Loss: 0.6915 | Elapsed: 5.0 min
Batch 3000/18694 | Loss: 0.6915 | Elapsed: 6.0 min
Batch 3500/18694 | Loss: 0.6905 | Elapsed: 7.0 min
Batch 4000/18694 | Loss: 0.6902 | Elapsed: 7.9 min
Batch 4500/18694 | Loss: 0.6905 | Elapsed: 8.9 min
Batch 5000/18694 | Loss: 0.6909 | Elapsed: 9.9 min
Batch 5500/18694 | Loss: 0.6909 | Elapsed: 10.9 min
Batch 6000/18694 | Loss: 0.6908 | Elapsed: 11.9 min
Batch 6500/18694 | Loss: 0.6911 | Elapsed: 12.9 min
Batch 7000/18694 | Loss: 0.6911 | Elapsed: 13.9 min
Batch 7500/18694 | Loss: 0.6908 | Elapsed: 14.9 min
Batch 8000/18694 | Loss: 0.6905 | Elapsed: 15.9 min
Batch 8500/18694 | Loss: 0.6904 | Elapsed: 16.9 min
Batch

In [75]:

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)
from torch.utils.data import DataLoader, TensorDataset

model.eval()
bitfit_head.eval()

dev_loader_bitfit = DataLoader(
    TensorDataset(
        torch.from_numpy(dev_tokens["input_ids"].astype(np.int64)),
        torch.from_numpy(dev_tokens["attention_mask"].astype(np.bool_)),
        torch.tensor(dev_tokens["labels"], dtype=torch.long)
    ),
    batch_size=2,
    shuffle=False
)

predictions = []
true_labels = []

with torch.inference_mode():
    for input_ids, attention_mask, labels in dev_loader_bitfit:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        positions = torch.arange(input_ids.shape[1], device=device)
        x = model.token_embedding(input_ids)
        x = x + model.encoder_pos(positions)[None, :, :]

        for layer in model.encoder_layers:
            x = layer(x)

        x = model.encoder_norm(x)

        mask = attention_mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        logits = bitfit_head(pooled)
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
        true_labels.extend(labels.tolist())

y_true_bitfit = np.asarray(true_labels)
y_pred_bitfit = np.asarray(predictions)

print("Development records:", len(y_true_bitfit))
print("Accuracy:", round(accuracy_score(y_true_bitfit, y_pred_bitfit), 4))
print("Macro-F1:", round(f1_score(y_true_bitfit, y_pred_bitfit, average="macro"), 4))
print("Macro precision:", round(precision_score(y_true_bitfit, y_pred_bitfit, average="macro", zero_division=0), 4))
print("Macro recall:", round(recall_score(y_true_bitfit, y_pred_bitfit, average="macro", zero_division=0), 4))
print("\nConfusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y_true_bitfit, y_pred_bitfit))
print("\nClassification report:")
print(classification_report(y_true_bitfit, y_pred_bitfit, labels=[0, 1], zero_division=0))
print("Predicted label counts:", np.bincount(y_pred_bitfit, minlength=2))


Development records: 3505
Accuracy: 0.499
Macro-F1: 0.353
Macro precision: 0.4995
Macro recall: 0.5

Confusion matrix (rows=true, columns=predicted):
[[1707   42]
 [1714   42]]

Classification report:
              precision    recall  f1-score   support

           0       0.50      0.98      0.66      1749
           1       0.50      0.02      0.05      1756

    accuracy                           0.50      3505
   macro avg       0.50      0.50      0.35      3505
weighted avg       0.50      0.50      0.35      3505

Predicted label counts: [3421   84]


In [76]:

import json
import os

results_path = "/kaggle/working/phase4_experiment_results.json"

with open(results_path, "r") as f:
    results = json.load(f)

results.append({
    "experiment": "A4 BitFit",
    "accuracy": 0.4990,
    "macro_f1": 0.3530,
    "macro_precision": 0.4995,
    "macro_recall": 0.5000,
    "trainable_parameters": 84994,
    "training_loss": 0.6891,
    "training_accuracy": 0.5849
})

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print("Updated experiment log:", results_path)
print("Total logged experiments:", len(results))
for item in results:
    print(item["experiment"], "Macro-F1:", item["macro_f1"])


Updated experiment log: /kaggle/working/phase4_experiment_results.json
Total logged experiments: 3
A4 frozen encoder - standard loss Macro-F1: 0.4487
A4 frozen encoder - class-weighted loss Macro-F1: 0.5235
A4 BitFit Macro-F1: 0.353


In [77]:

import torch
import torch.nn as nn

checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-22000-pt/a4_step_22000.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(device)
model.eval()

# Freeze the original A4 parameters
for parameter in model.parameters():
    parameter.requires_grad = False

# Inspect square linear layers suitable for multiplicative adaptation
print("Restored A4 checkpoint step:", checkpoint.get("step"))
print("\nSquare linear layers in A4 encoder:")

for name, module in model.named_modules():
    if (
        isinstance(module, nn.Linear)
        and module.in_features == module.out_features
        and name.startswith("encoder_layers.")
    ):
        print(
            f"{name}: "
            f"in={module.in_features}, "
            f"out={module.out_features}"
        )

print("\nAll original encoder parameters frozen:",
      all(not p.requires_grad for p in model.parameters()))


Restored A4 checkpoint step: 22000

Square linear layers in A4 encoder:
encoder_layers.0.attn.q_proj: in=512, out=512
encoder_layers.0.attn.k_proj: in=512, out=512
encoder_layers.0.attn.v_proj: in=512, out=512
encoder_layers.0.attn.out_proj: in=512, out=512
encoder_layers.1.attn.q_proj: in=512, out=512
encoder_layers.1.attn.k_proj: in=512, out=512
encoder_layers.1.attn.v_proj: in=512, out=512
encoder_layers.1.attn.out_proj: in=512, out=512
encoder_layers.2.attn.q_proj: in=512, out=512
encoder_layers.2.attn.k_proj: in=512, out=512
encoder_layers.2.attn.v_proj: in=512, out=512
encoder_layers.2.attn.out_proj: in=512, out=512
encoder_layers.3.attn.q_proj: in=512, out=512
encoder_layers.3.attn.k_proj: in=512, out=512
encoder_layers.3.attn.v_proj: in=512, out=512
encoder_layers.3.attn.out_proj: in=512, out=512
encoder_layers.4.attn.q_proj: in=512, out=512
encoder_layers.4.attn.k_proj: in=512, out=512
encoder_layers.4.attn.v_proj: in=512, out=512
encoder_layers.4.attn.out_proj: in=512, out=51

In [78]:

import math
import torch
import torch.nn as nn

# LoRMA+-style multiplicative adapter:
# y = xW + xW * (xA)B * scale
# A starts at zero, so the initial adapted output matches the original layer.
class LoRMAPlusLinear(nn.Module):
    def __init__(self, base_layer, rank=8, alpha=16):
        super().__init__()
        self.base = base_layer
        self.rank = rank
        self.scale = alpha / rank

        in_features = base_layer.in_features
        out_features = base_layer.out_features

        self.A = nn.Parameter(
            torch.zeros(in_features, rank,
                        device=base_layer.weight.device,
                        dtype=base_layer.weight.dtype)
        )
        self.B = nn.Parameter(
            torch.zeros(rank, out_features,
                        device=base_layer.weight.device,
                        dtype=base_layer.weight.dtype)
        )

        # Freeze the original linear layer
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        base_output = self.base(x)
        low_rank = (x @ self.A) @ self.B
        return base_output + base_output * low_rank * self.scale


# Attach adapters to q_proj and v_proj in every encoder block.
adapter_names = []

for layer_index, layer in enumerate(model.encoder_layers):
    for projection_name in ["q_proj", "v_proj"]:
        original = getattr(layer.attn, projection_name)
        adapted = LoRMAPlusLinear(original, rank=8, alpha=16)
        setattr(layer.attn, projection_name, adapted)
        adapter_names.append(
            f"encoder_layers.{layer_index}.attn.{projection_name}"
        )

# Count parameters
trainable = [
    (name, p)
    for name, p in model.named_parameters()
    if p.requires_grad
]

print("Adapters attached:", len(adapter_names))
print("Trainable adapter parameters:", sum(p.numel() for _, p in trainable))
print("Adapted layers:")
for name in adapter_names:
    print(name)

# Confirm the adapter's initial output matches the original projection.
test_layer = model.encoder_layers[0].attn.q_proj
test_input = torch.randn(1, 4, 512, device=device)
with torch.no_grad():
    base_output = test_layer.base(test_input)
    adapted_output = test_layer(test_input)

print("Initial output matches base:",
      torch.allclose(base_output, adapted_output, atol=1e-6))


Adapters attached: 12
Trainable adapter parameters: 98304
Adapted layers:
encoder_layers.0.attn.q_proj
encoder_layers.0.attn.v_proj
encoder_layers.1.attn.q_proj
encoder_layers.1.attn.v_proj
encoder_layers.2.attn.q_proj
encoder_layers.2.attn.v_proj
encoder_layers.3.attn.q_proj
encoder_layers.3.attn.v_proj
encoder_layers.4.attn.q_proj
encoder_layers.4.attn.v_proj
encoder_layers.5.attn.q_proj
encoder_layers.5.attn.v_proj
Initial output matches base: True


In [79]:
# Fix adapter initialization so gradients can flow
import torch
import torch.nn as nn

adapter_count = 0

for module in model.modules():
    if hasattr(module, "A") and hasattr(module, "B"):
        nn.init.normal_(module.A, mean=0.0, std=0.02)
        nn.init.zeros_(module.B)
        adapter_count += 1

print("Adapters reinitialized:", adapter_count)
print("A matrices are nonzero:", all(
    torch.count_nonzero(m.A).item() > 0
    for m in model.modules()
    if hasattr(m, "A") and hasattr(m, "B")
))
print("B matrices are initially zero:", all(
    torch.count_nonzero(m.B).item() == 0
    for m in model.modules()
    if hasattr(m, "A") and hasattr(m, "B")
))

Adapters reinitialized: 12
A matrices are nonzero: True
B matrices are initially zero: True


In [80]:
# Gradient-flow sanity check — no optimizer step is performed
model.train()
model.zero_grad(set_to_none=True)

device = next(model.parameters()).device
dummy_ids = torch.tensor(
    [[2, 10, 25, 40, 3]],
    dtype=torch.long,
    device=device
)

positions = torch.arange(dummy_ids.shape[1], device=device)
x = model.token_embedding(dummy_ids) + model.encoder_pos(positions)[None, :, :]
x = model.dropout(x)

for layer in model.encoder_layers:
    x = layer(x)

encoded = model.encoder_norm(x)
test_loss = encoded.float().square().mean()
test_loss.backward()

adapter_grads = [
    p.grad for module in model.modules()
    if hasattr(module, "A") and hasattr(module, "B")
    for p in (module.A, module.B)
]

print("Adapters checked:", len(adapter_grads) // 2)
print("A gradients present:", all(
    module.A.grad is not None
    for module in model.modules()
    if hasattr(module, "A") and hasattr(module, "B")
))
print("B gradients present:", all(
    module.B.grad is not None
    for module in model.modules()
    if hasattr(module, "A") and hasattr(module, "B")
))
print("At least one nonzero adapter gradient:", any(
    grad is not None and torch.count_nonzero(grad).item() > 0
    for grad in adapter_grads
))

model.zero_grad(set_to_none=True)

Adapters checked: 12
A gradients present: True
B gradients present: True
At least one nonzero adapter gradient: True


In [81]:
# Inspect trainable parameters before training
trainable_params = [
    (name, param)
    for name, param in model.named_parameters()
    if param.requires_grad
]

print("Trainable parameter tensors:", len(trainable_params))
print("Trainable parameter count:",
      sum(param.numel() for _, param in trainable_params))

print("\nFirst 30 trainable parameter names:")
for name, param in trainable_params[:30]:
    print(name, tuple(param.shape))

print("\nFrozen original encoder parameters:",
      all(
          not param.requires_grad
          for name, param in model.named_parameters()
          if "encoder_layers" in name
          and ".A" not in name
          and ".B" not in name
      ))

Trainable parameter tensors: 24
Trainable parameter count: 98304

First 30 trainable parameter names:
encoder_layers.0.attn.q_proj.A (512, 8)
encoder_layers.0.attn.q_proj.B (8, 512)
encoder_layers.0.attn.v_proj.A (512, 8)
encoder_layers.0.attn.v_proj.B (8, 512)
encoder_layers.1.attn.q_proj.A (512, 8)
encoder_layers.1.attn.q_proj.B (8, 512)
encoder_layers.1.attn.v_proj.A (512, 8)
encoder_layers.1.attn.v_proj.B (8, 512)
encoder_layers.2.attn.q_proj.A (512, 8)
encoder_layers.2.attn.q_proj.B (8, 512)
encoder_layers.2.attn.v_proj.A (512, 8)
encoder_layers.2.attn.v_proj.B (8, 512)
encoder_layers.3.attn.q_proj.A (512, 8)
encoder_layers.3.attn.q_proj.B (8, 512)
encoder_layers.3.attn.v_proj.A (512, 8)
encoder_layers.3.attn.v_proj.B (8, 512)
encoder_layers.4.attn.q_proj.A (512, 8)
encoder_layers.4.attn.q_proj.B (8, 512)
encoder_layers.4.attn.v_proj.A (512, 8)
encoder_layers.4.attn.v_proj.B (8, 512)
encoder_layers.5.attn.q_proj.A (512, 8)
encoder_layers.5.attn.q_proj.B (8, 512)
encoder_layers.5.a

In [82]:
# Inspect model components that may contain the classification head
print("Model attributes that may identify the classifier:")

for name in dir(model):
    if any(key in name.lower() for key in ("class", "head", "pool", "linear")):
        try:
            value = getattr(model, name)
            if isinstance(value, nn.Module):
                print(f"\n{name}: {value}")
        except Exception:
            pass

print("\nOther notebook variables that may be classifiers:")
for name, value in list(globals().items()):
    if isinstance(value, nn.Module) and value is not model:
        if any(key in name.lower() for key in ("class", "head", "clf")):
            print(name, type(value).__name__)

Model attributes that may identify the classifier:

lm_head: Linear(in_features=512, out_features=50000, bias=False)

Other notebook variables that may be classifiers:
classification_head Linear
weighted_head Linear
bitfit_head Linear


In [83]:
for name in ["classification_head", "weighted_head", "bitfit_head"]:
    head = globals()[name]
    print(f"\n{name}:")
    print("  Architecture:", head)
    print("  Input features:", head.in_features)
    print("  Output classes:", head.out_features)
    print("  Trainable:", any(p.requires_grad for p in head.parameters()))
    print("  Device:", next(head.parameters()).device)


classification_head:
  Architecture: Linear(in_features=512, out_features=2, bias=True)
  Input features: 512
  Output classes: 2
  Trainable: True
  Device: cuda:0

weighted_head:
  Architecture: Linear(in_features=512, out_features=2, bias=True)
  Input features: 512
  Output classes: 2
  Trainable: True
  Device: cuda:0

bitfit_head:
  Architecture: Linear(in_features=512, out_features=2, bias=True)
  Input features: 512
  Output classes: 2
  Trainable: True
  Device: cuda:0


In [84]:
# Configure adapter experiment
import torch

# Freeze all model parameters first
for param in model.parameters():
    param.requires_grad = False

# Re-enable adapter parameters only
for module in model.modules():
    if hasattr(module, "A") and hasattr(module, "B"):
        module.A.requires_grad = True
        module.B.requires_grad = True

# Enable the selected classification head
classification_head = classification_head.to(device)
for param in classification_head.parameters():
    param.requires_grad = True

# Optimizer updates adapters and classifier
trainable_params = [
    p for p in model.parameters() if p.requires_grad
] + [
    p for p in classification_head.parameters() if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=2e-4,
    weight_decay=0.01
)

print("Trainable parameter count:",
      sum(p.numel() for p in trainable_params))
print("Adapter parameters:",
      sum(p.numel() for module in model.modules()
          if hasattr(module, "A") and hasattr(module, "B")
          for p in (module.A, module.B)))
print("Classifier parameters:",
      sum(p.numel() for p in classification_head.parameters()
          if p.requires_grad))
print("Optimizer created:", isinstance(optimizer, torch.optim.Optimizer))

Trainable parameter count: 99330
Adapter parameters: 98304
Classifier parameters: 1026
Optimizer created: True


In [85]:
print("Training data variable exists:", "train_data" in globals())
print("Development data variable exists:", "dev_data" in globals())
print("Batch function exists:", "prepare_batch" in globals())

if "train_data" in globals():
    print("Training records:", len(train_data))
    print("First record type:", type(train_data[0]).__name__)
    print("First record keys:",
          list(train_data[0].keys()) if isinstance(train_data[0], dict)
          else "Not a dictionary")

if "dev_data" in globals():
    print("Development records:", len(dev_data))
    print("First development record keys:",
          list(dev_data[0].keys()) if isinstance(dev_data[0], dict)
          else "Not a dictionary")

Training data variable exists: True
Development data variable exists: True
Batch function exists: True
Training records: 37387
First record type: dict
First record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
Development records: 3505
First development record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']


In [86]:
# Inspect the existing batch preparation function
sample_batch = prepare_batch(train_data[:2], max_length=1024)

print("Batch type:", type(sample_batch).__name__)

if isinstance(sample_batch, dict):
    for key, value in sample_batch.items():
        if torch.is_tensor(value):
            print(
                f"{key}: shape={tuple(value.shape)}, "
                f"dtype={value.dtype}, device={value.device}"
            )
        else:
            print(f"{key}: type={type(value).__name__}, value={value}")
elif isinstance(sample_batch, (tuple, list)):
    for i, value in enumerate(sample_batch):
        if torch.is_tensor(value):
            print(
                f"Item {i}: shape={tuple(value.shape)}, "
                f"dtype={value.dtype}, device={value.device}"
            )
        else:
            print(f"Item {i}: {type(value).__name__}")
else:
    print(sample_batch)

Batch type: tuple
Item 0: shape=(2, 1024), dtype=torch.int64, device=cuda:0
Item 1: shape=(2, 1024), dtype=torch.int64, device=cuda:0
Item 2: shape=(2,), dtype=torch.int64, device=cuda:0


In [87]:
import inspect

print("Encoder layer forward signature:")
print(inspect.signature(model.encoder_layers[0].forward))

print("\nEncoder layer class:")
print(type(model.encoder_layers[0]).__name__)

print("\nPadding token ID:", tokenizer.token_to_id("[PAD]"))
print("Classifier output classes:", classification_head.out_features)

# Check case-level labels from the sample batch
input_ids, token_labels, case_labels = sample_batch
print("\nCase labels:", case_labels.tolist())
print("Unique case labels:", torch.unique(case_labels).tolist())

Encoder layer forward signature:
(x)

Encoder layer class:
EncoderBlock

Padding token ID: 0
Classifier output classes: 2

Case labels: [0, 0]
Unique case labels: [0]


In [88]:
model.eval()
classification_head.eval()

input_ids, token_labels, case_labels = prepare_batch(
    train_data[:4], max_length=1024
)

with torch.no_grad():
    positions = torch.arange(input_ids.shape[1], device=device)
    x = model.token_embedding(input_ids) + model.encoder_pos(positions)[None, :, :]
    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    encoded = model.encoder_norm(x)

    # Exclude padding tokens from mean pooling
    attention_mask = (input_ids != 0).unsqueeze(-1)
    encoded = encoded * attention_mask
    pooled = encoded.sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)

    logits = classification_head(pooled)

print("Input shape:", tuple(input_ids.shape))
print("Pooled representation shape:", tuple(pooled.shape))
print("Logits shape:", tuple(logits.shape))
print("Case labels:", case_labels.tolist())
print("Predictions:", logits.argmax(dim=1).tolist())
print("Finite logits:", torch.isfinite(logits).all().item())

Input shape: (4, 1024)
Pooled representation shape: (4, 512)
Logits shape: (4, 2)
Case labels: [0, 0, 0, 0]
Predictions: [0, 0, 0, 0]
Finite logits: True


In [89]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

model.train()
classification_head.train()

# Small debugging subset
debug_records = train_data[:64]
debug_batch_size = 8

debug_losses = []
adapter_params = [
    p for module in model.modules()
    if hasattr(module, "A") and hasattr(module, "B")
    for p in (module.A, module.B)
]
head_params = list(classification_head.parameters())

for start in range(0, len(debug_records), debug_batch_size):
    batch = prepare_batch(
        debug_records[start:start + debug_batch_size],
        max_length=1024
    )
    input_ids, _, labels = batch

    optimizer.zero_grad(set_to_none=True)

    positions = torch.arange(input_ids.shape[1], device=device)
    x = model.token_embedding(input_ids) + model.encoder_pos(positions)[None, :, :]
    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    encoded = model.encoder_norm(x)
    mask = (input_ids != 0).unsqueeze(-1)
    pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    logits = classification_head(pooled)
    loss = F.cross_entropy(logits, labels)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(adapter_params + head_params, 1.0)
    optimizer.step()

    debug_losses.append(loss.item())

print("Debugging batches:", len(debug_losses))
print("First batch loss:", round(debug_losses[0], 4))
print("Last batch loss:", round(debug_losses[-1], 4))
print("All losses finite:", all(torch.isfinite(torch.tensor(v)).item() for v in debug_losses))

Debugging batches: 8
First batch loss: 0.4726
Last batch loss: 0.6628
All losses finite: True


In [90]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

model.eval()
classification_head.eval()

debug_true = []
debug_pred = []

with torch.no_grad():
    for start in range(0, len(debug_records), debug_batch_size):
        batch = prepare_batch(
            debug_records[start:start + debug_batch_size],
            max_length=1024
        )
        input_ids, _, labels = batch

        positions = torch.arange(input_ids.shape[1], device=device)
        x = model.token_embedding(input_ids) + model.encoder_pos(positions)[None, :, :]
        x = model.dropout(x)

        for layer in model.encoder_layers:
            x = layer(x)

        encoded = model.encoder_norm(x)
        mask = (input_ids != 0).unsqueeze(-1)
        pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        logits = classification_head(pooled)

        debug_true.extend(labels.cpu().tolist())
        debug_pred.extend(logits.argmax(dim=1).cpu().tolist())

print("Label distribution:", dict(zip(*np.unique(debug_true, return_counts=True))))
print("Prediction distribution:", dict(zip(*np.unique(debug_pred, return_counts=True))))
print("Training-subset accuracy:", round(accuracy_score(debug_true, debug_pred), 4))
print("Training-subset Macro-F1:", round(
    f1_score(debug_true, debug_pred, average="macro", zero_division=0), 4
))

Label distribution: {np.int64(0): np.int64(41), np.int64(1): np.int64(23)}
Prediction distribution: {np.int64(0): np.int64(63), np.int64(1): np.int64(1)}
Training-subset accuracy: 0.625
Training-subset Macro-F1: 0.3846


In [91]:
import random
import torch
import torch.nn as nn

# Reproducibility
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Shuffle the full training records, then take a pilot subset
shuffled_indices = list(range(len(train_data)))
random.shuffle(shuffled_indices)
pilot_records = [train_data[i] for i in shuffled_indices[:2048]]

# Compute class weights from the FULL training set
label_counts = torch.bincount(
    torch.tensor([int(record["label"]) for record in train_data]),
    minlength=2
).float()

class_weights = (label_counts.sum() / (2 * label_counts)).to(device)

# Reset optimizer with a conservative learning rate
trainable_params = [
    p for p in model.parameters() if p.requires_grad
] + [
    p for p in classification_head.parameters() if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=5e-5,
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

print("Pilot training examples:", len(pilot_records))
print("Full training label counts:", label_counts.tolist())
print("Class weights:", class_weights.tolist())
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Trainable parameters:", sum(p.numel() for p in trainable_params))
print("Pilot label counts:", {
    label: sum(int(r["label"]) == label for r in pilot_records)
    for label in (0, 1)
})

Pilot training examples: 2048
Full training label counts: [22067.0, 15320.0]
Class weights: [0.8471246361732483, 1.2202023267745972]
Learning rate: 5e-05
Trainable parameters: 99330
Pilot label counts: {0: 1222, 1: 826}


In [92]:
import random
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

batch_size = 8
random.shuffle(pilot_records)

model.train()
classification_head.train()

epoch_losses = []
train_true = []
train_pred = []

for start in tqdm(range(0, len(pilot_records), batch_size)):
    records = pilot_records[start:start + batch_size]
    input_ids, _, labels = prepare_batch(records, max_length=1024)

    optimizer.zero_grad(set_to_none=True)

    positions = torch.arange(input_ids.shape[1], device=device)
    x = model.token_embedding(input_ids) + model.encoder_pos(positions)[None, :, :]
    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    encoded = model.encoder_norm(x)
    mask = (input_ids != 0).unsqueeze(-1)
    pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    logits = classification_head(pooled)
    loss = F.cross_entropy(logits, labels, weight=class_weights)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()

    epoch_losses.append(loss.item())
    train_true.extend(labels.detach().cpu().tolist())
    train_pred.extend(logits.detach().argmax(dim=1).cpu().tolist())

print("Pilot epoch completed")
print("Batches:", len(epoch_losses))
print("Average training loss:", round(sum(epoch_losses) / len(epoch_losses), 4))
print("Training Macro-F1:", round(
    f1_score(train_true, train_pred, average="macro", zero_division=0), 4
))
print("Prediction counts:", {
    label: train_pred.count(label) for label in (0, 1)
})

  0%|          | 0/256 [00:00<?, ?it/s]

Pilot epoch completed
Batches: 256
Average training loss: 0.6939
Training Macro-F1: 0.505
Prediction counts: {0: 1654, 1: 394}


In [93]:
import torch
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)
from tqdm.auto import tqdm

model.eval()
classification_head.eval()

dev_true = []
dev_pred = []

with torch.no_grad():
    for start in tqdm(range(0, len(dev_data), 8)):
        records = dev_data[start:start + 8]
        input_ids, _, labels = prepare_batch(
            records, max_length=1024
        )

        positions = torch.arange(input_ids.shape[1], device=device)
        x = model.token_embedding(input_ids) + model.encoder_pos(positions)[None, :, :]
        x = model.dropout(x)

        for layer in model.encoder_layers:
            x = layer(x)

        encoded = model.encoder_norm(x)
        mask = (input_ids != 0).unsqueeze(-1)
        pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        logits = classification_head(pooled)
        dev_true.extend(labels.cpu().tolist())
        dev_pred.extend(logits.argmax(dim=1).cpu().tolist())

print("Development examples:", len(dev_true))
print("Accuracy:", round(accuracy_score(dev_true, dev_pred), 4))
print("Macro-F1:", round(f1_score(
    dev_true, dev_pred, average="macro", zero_division=0
), 4))
print("Macro precision:", round(precision_score(
    dev_true, dev_pred, average="macro", zero_division=0
), 4))
print("Macro recall:", round(recall_score(
    dev_true, dev_pred, average="macro", zero_division=0
), 4))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(dev_true, dev_pred))
print("\nClassification report:")
print(classification_report(
    dev_true, dev_pred, labels=[0, 1], zero_division=0
))

  0%|          | 0/439 [00:00<?, ?it/s]

Development examples: 3505
Accuracy: 0.5201
Macro-F1: 0.4806
Macro precision: 0.528
Macro recall: 0.5196
Confusion matrix (rows=true, columns=predicted):
[[ 428 1321]
 [ 361 1395]]

Classification report:
              precision    recall  f1-score   support

           0       0.54      0.24      0.34      1749
           1       0.51      0.79      0.62      1756

    accuracy                           0.52      3505
   macro avg       0.53      0.52      0.48      3505
weighted avg       0.53      0.52      0.48      3505



In [94]:
import numpy as np
import torch
from sklearn.metrics import f1_score, confusion_matrix

model.eval()
classification_head.eval()

dev_true = []
dev_prob1 = []

with torch.no_grad():
    for start in range(0, len(dev_data), 8):
        records = dev_data[start:start + 8]
        input_ids, _, labels = prepare_batch(records, max_length=1024)

        positions = torch.arange(input_ids.shape[1], device=device)
        x = model.token_embedding(input_ids) + model.encoder_pos(positions)[None, :, :]
        x = model.dropout(x)

        for layer in model.encoder_layers:
            x = layer(x)

        encoded = model.encoder_norm(x)
        mask = (input_ids != 0).unsqueeze(-1)
        pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        logits = classification_head(pooled)
        probabilities = torch.softmax(logits, dim=1)[:, 1]

        dev_true.extend(labels.cpu().tolist())
        dev_prob1.extend(probabilities.cpu().tolist())

dev_true = np.asarray(dev_true)
dev_prob1 = np.asarray(dev_prob1)

print("Threshold | Macro-F1 | Class 0 recall | Class 1 recall")
print("-" * 55)

for threshold in np.arange(0.35, 0.651, 0.05):
    predictions = (dev_prob1 >= threshold).astype(int)
    cm = confusion_matrix(dev_true, predictions, labels=[0, 1])
    recall0 = cm[0, 0] / max(cm[0].sum(), 1)
    recall1 = cm[1, 1] / max(cm[1].sum(), 1)
    score = f1_score(dev_true, predictions, average="macro", zero_division=0)

    print(f"{threshold:.2f}      | {score:.4f}   | {recall0:.4f}         | {recall1:.4f}")

Threshold | Macro-F1 | Class 0 recall | Class 1 recall
-------------------------------------------------------
0.35      | 0.3354   | 0.0017         | 0.9989
0.40      | 0.3461   | 0.0132         | 0.9909
0.45      | 0.3879   | 0.0663         | 0.9453
0.50      | 0.4806   | 0.2447         | 0.7944
0.55      | 0.5314   | 0.5723         | 0.4920
0.60      | 0.4610   | 0.8879         | 0.1726
0.65      | 0.3707   | 0.9897         | 0.0376


In [95]:
import json
from pathlib import Path

experiment_result = {
    "experiment": "A4 multiplicative low-rank adapters pilot",
    "checkpoint": "A4 step 22000",
    "training_examples": 2048,
    "epochs": 1,
    "batch_size": 8,
    "learning_rate": 5e-5,
    "trainable_parameters": 99330,
    "development_examples": 3505,
    "development_macro_f1_threshold_0.50": 0.4806,
    "development_macro_f1_threshold_0.55": 0.5314,
    "selected_threshold": 0.55,
    "threshold_selection_note": "Chosen on the same development set; potentially optimistic."
}

results_path = Path("/kaggle/working/a4_adapter_pilot_results.json")
results_path.write_text(json.dumps(experiment_result, indent=2))

print("Saved results to:", results_path)
print(json.dumps(experiment_result, indent=2))

Saved results to: /kaggle/working/a4_adapter_pilot_results.json
{
  "experiment": "A4 multiplicative low-rank adapters pilot",
  "checkpoint": "A4 step 22000",
  "training_examples": 2048,
  "epochs": 1,
  "batch_size": 8,
  "learning_rate": 5e-05,
  "trainable_parameters": 99330,
  "development_examples": 3505,
  "development_macro_f1_threshold_0.50": 0.4806,
  "development_macro_f1_threshold_0.55": 0.5314,
  "selected_threshold": 0.55,
  "threshold_selection_note": "Chosen on the same development set; potentially optimistic."
}


In [96]:
import inspect

print("Adapter class:")
print(type(model.encoder_layers[0].attn.q_proj).__name__)

print("\nAdapter class source, if available:")
try:
    print(inspect.getsource(type(model.encoder_layers[0].attn.q_proj)))
except (OSError, TypeError):
    print("Source unavailable through inspect.")

print("\nWrapped base layer:")
adapter = model.encoder_layers[0].attn.q_proj
print(type(adapter.base).__name__)
print("Base weight shape:", tuple(adapter.base.weight.shape))
print("Rank:", adapter.rank)
print("Scale:", adapter.scale)
print("A shape:", tuple(adapter.A.shape))
print("B shape:", tuple(adapter.B.shape))

Adapter class:
LoRMAPlusLinear

Adapter class source, if available:
Source unavailable through inspect.

Wrapped base layer:
Linear
Base weight shape: (512, 512)
Rank: 8
Scale: 2.0
A shape: (512, 8)
B shape: (8, 512)


In [97]:
import torch

adapter = model.encoder_layers[0].attn.q_proj
base = adapter.base

print("Base layer type:", type(base).__name__)
print("Weight shape:", tuple(base.weight.shape))
print("Bias exists:", base.bias is not None)
print("Adapter rank:", adapter.rank)
print("Adapter scale:", adapter.scale)

# Verify the wrapper's current output formula on a tiny input
x = torch.randn(2, 3, base.in_features, device=base.weight.device)
with torch.no_grad():
    base_out = base(x)
    wrapped_out = adapter(x)

print("Base output shape:", tuple(base_out.shape))
print("Wrapped output shape:", tuple(wrapped_out.shape))
print("Current outputs initially match base:",
      torch.allclose(base_out, wrapped_out, atol=1e-6))

Base layer type: Linear
Weight shape: (512, 512)
Bias exists: True
Adapter rank: 8
Adapter scale: 2.0
Base output shape: (2, 3, 512)
Wrapped output shape: (2, 3, 512)
Current outputs initially match base: False


In [98]:
import dis

adapter = model.encoder_layers[0].attn.q_proj

print("Forward method:")
try:
    print(inspect.getsource(adapter.forward))
except (OSError, TypeError):
    dis.dis(adapter.forward)

print("\nParameter statistics:")
print("A mean/std:", adapter.A.detach().mean().item(),
      adapter.A.detach().std().item())
print("B max abs:", adapter.B.detach().abs().max().item())

with torch.no_grad():
    x = torch.randn(2, 3, adapter.base.in_features, device=adapter.base.weight.device)
    base_out = adapter.base(x)
    wrapped_out = adapter(x)
    print("Max absolute output difference:",
          (base_out - wrapped_out).abs().max().item())

Forward method:
    def forward(self, x):
        base_output = self.base(x)
        low_rank = (x @ self.A) @ self.B
        return base_output + base_output * low_rank * self.scale


Parameter statistics:
A mean/std: 0.00014776625903323293 0.020237378776073456
B max abs: 0.004501421004533768
Max absolute output difference: 0.04420316219329834


In [99]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PaperLoRMAPlusLinear(nn.Module):
    """
    LoRMA+ weight transformation:
        W_eff = (I + (alpha/r) * B @ A) @ W0

    B: (out_features, rank)
    A: (rank, out_features)
    """
    def __init__(self, base_layer, rank=8, alpha=16):
        super().__init__()
        self.base = base_layer
        self.rank = rank
        self.scale = alpha / rank

        out_features, in_features = base_layer.weight.shape
        if out_features != in_features:
            raise ValueError(
                "This identity-inflated implementation requires a square weight matrix."
            )

        self.A = nn.Parameter(
            torch.empty(rank, out_features,
                        device=base_layer.weight.device,
                        dtype=base_layer.weight.dtype)
        )
        self.B = nn.Parameter(
            torch.zeros(out_features, rank,
                        device=base_layer.weight.device,
                        dtype=base_layer.weight.dtype)
        )

        nn.init.normal_(self.A, mean=0.0, std=0.02)

        for param in self.base.parameters():
            param.requires_grad = False

    def forward(self, x):
        # Effective W = (I + scale * B @ A) @ W0
        delta = (self.B @ self.A) @ self.base.weight
        effective_weight = self.base.weight + self.scale * delta
        return F.linear(x, effective_weight, self.base.bias)


# Test the wrapper independently on a copied linear layer
import copy

test_base = copy.deepcopy(
    model.encoder_layers[0].attn.q_proj.base
)
test_wrapper = PaperLoRMAPlusLinear(test_base, rank=8, alpha=16)

test_x = torch.randn(
    2, 3, test_base.in_features,
    device=test_base.weight.device,
    dtype=test_base.weight.dtype
)

with torch.no_grad():
    base_y = test_base(test_x)
    wrapped_y = test_wrapper(test_x)

print("Base output shape:", tuple(base_y.shape))
print("LoRMA+ output shape:", tuple(wrapped_y.shape))
print("Initially matches base:",
      torch.allclose(base_y, wrapped_y, atol=1e-6))
print("Maximum initial difference:",
      (base_y - wrapped_y).abs().max().item())
print("Adapter parameters:",
      test_wrapper.A.numel() + test_wrapper.B.numel())

Base output shape: (2, 3, 512)
LoRMA+ output shape: (2, 3, 512)
Initially matches base: True
Maximum initial difference: 0.0
Adapter parameters: 8192


In [100]:
import copy
import torch

# Save the original base linear layers before replacing wrappers
target_names = []
for layer_idx, layer in enumerate(model.encoder_layers):
    for proj_name in ("q_proj", "v_proj"):
        target_names.append((layer_idx, proj_name))

# Replace wrappers with paper-aligned LoRMA+ wrappers
for layer_idx, proj_name in target_names:
    attention = model.encoder_layers[layer_idx].attn
    old_wrapper = getattr(attention, proj_name)

    # Keep the underlying pretrained/base linear layer
    base_layer = old_wrapper.base

    new_wrapper = PaperLoRMAPlusLinear(
        base_layer,
        rank=8,
        alpha=16
    )
    setattr(attention, proj_name, new_wrapper)

print("Replaced wrappers:", len(target_names))
print("First adapted layer:",
      type(model.encoder_layers[0].attn.q_proj).__name__)
print("Trainable adapter parameters:",
      sum(
          p.numel()
          for module in model.modules()
          if isinstance(module, PaperLoRMAPlusLinear)
          for p in (module.A, module.B)
      ))
print("All adapter B matrices initially zero:",
      all(
          torch.count_nonzero(module.B).item() == 0
          for module in model.modules()
          if isinstance(module, PaperLoRMAPlusLinear)
      ))

Replaced wrappers: 12
First adapted layer: PaperLoRMAPlusLinear
Trainable adapter parameters: 98304
All adapter B matrices initially zero: True


In [101]:
model.eval()

adapter = model.encoder_layers[0].attn.q_proj
test_x = torch.randn(
    2, 3, adapter.base.in_features,
    device=adapter.base.weight.device,
    dtype=adapter.base.weight.dtype
)

with torch.no_grad():
    base_output = adapter.base(test_x)
    adapted_output = adapter(test_x)

print("Output shapes match:",
      base_output.shape == adapted_output.shape)
print("Initially identical:",
      torch.allclose(base_output, adapted_output, atol=1e-6))
print("Maximum absolute difference:",
      (base_output - adapted_output).abs().max().item())
print("Base weights frozen:",
      all(not p.requires_grad for p in adapter.base.parameters()))

Output shapes match: True
Initially identical: True
Maximum absolute difference: 0.0
Base weights frozen: True


In [102]:
import torch
import torch.nn as nn

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Enable only the new LoRMA+ factors
lorma_modules = [
    module for module in model.modules()
    if isinstance(module, PaperLoRMAPlusLinear)
]

for module in lorma_modules:
    module.A.requires_grad = True
    module.B.requires_grad = True

# Enable classification head
classification_head = classification_head.to(device)
for param in classification_head.parameters():
    param.requires_grad = True

trainable_params = [
    p for module in lorma_modules for p in (module.A, module.B)
] + list(classification_head.parameters())

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=5e-5,
    weight_decay=0.01
)

print("LoRMA+ modules:", len(lorma_modules))
print("Adapter parameters:", sum(
    p.numel() for module in lorma_modules
    for p in (module.A, module.B)
))
print("Classifier parameters:", sum(
    p.numel() for p in classification_head.parameters()
))
print("Total trainable parameters:", sum(
    p.numel() for p in trainable_params
))
print("Optimizer rebuilt:", isinstance(optimizer, torch.optim.Optimizer))
print("All base linear weights frozen:", all(
    not p.requires_grad
    for module in lorma_modules
    for p in module.base.parameters()
))

LoRMA+ modules: 12
Adapter parameters: 98304
Classifier parameters: 1026
Total trainable parameters: 99330
Optimizer rebuilt: True
All base linear weights frozen: True


In [103]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

batch_size = 8
model.train()
classification_head.train()

epoch_losses = []
train_true = []
train_pred = []

for start in tqdm(range(0, len(pilot_records), batch_size)):
    records = pilot_records[start:start + batch_size]
    input_ids, _, labels = prepare_batch(records, max_length=1024)

    optimizer.zero_grad(set_to_none=True)

    positions = torch.arange(input_ids.shape[1], device=device)
    x = model.token_embedding(input_ids) + model.encoder_pos(positions)[None, :, :]
    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    encoded = model.encoder_norm(x)
    mask = (input_ids != 0).unsqueeze(-1)
    pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    logits = classification_head(pooled)
    loss = F.cross_entropy(logits, labels, weight=class_weights)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()

    epoch_losses.append(loss.item())
    train_true.extend(labels.detach().cpu().tolist())
    train_pred.extend(logits.detach().argmax(dim=1).cpu().tolist())

print("Corrected LoRMA+ pilot completed")
print("Batches:", len(epoch_losses))
print("Average training loss:", round(sum(epoch_losses) / len(epoch_losses), 4))
print("Training Macro-F1:", round(
    f1_score(train_true, train_pred, average="macro", zero_division=0), 4
))
print("Prediction counts:", {
    label: train_pred.count(label) for label in (0, 1)
})

  0%|          | 0/256 [00:00<?, ?it/s]

Corrected LoRMA+ pilot completed
Batches: 256
Average training loss: 0.6865
Training Macro-F1: 0.5506
Prediction counts: {0: 1342, 1: 706}


In [104]:
import torch
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)
from tqdm.auto import tqdm

model.eval()
classification_head.eval()

lorma_dev_true = []
lorma_dev_probs = []

with torch.no_grad():
    for start in tqdm(range(0, len(dev_data), 8)):
        records = dev_data[start:start + 8]
        input_ids, _, labels = prepare_batch(
            records, max_length=1024
        )

        positions = torch.arange(input_ids.shape[1], device=device)
        x = model.token_embedding(input_ids) + model.encoder_pos(positions)[None, :, :]
        x = model.dropout(x)

        for layer in model.encoder_layers:
            x = layer(x)

        encoded = model.encoder_norm(x)
        mask = (input_ids != 0).unsqueeze(-1)
        pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        logits = classification_head(pooled)
        probabilities = torch.softmax(logits, dim=1)

        lorma_dev_true.extend(labels.cpu().tolist())
        lorma_dev_probs.extend(probabilities.cpu().tolist())

lorma_dev_true = torch.tensor(lorma_dev_true).numpy()
lorma_dev_probs = torch.tensor(lorma_dev_probs).numpy()

for threshold in [0.50, 0.55]:
    predictions = (lorma_dev_probs[:, 1] >= threshold).astype(int)
    print(f"\nThreshold: {threshold:.2f}")
    print("Accuracy:", round(accuracy_score(lorma_dev_true, predictions), 4))
    print("Macro-F1:", round(f1_score(
        lorma_dev_true, predictions, average="macro", zero_division=0
    ), 4))
    print("Macro precision:", round(precision_score(
        lorma_dev_true, predictions, average="macro", zero_division=0
    ), 4))
    print("Macro recall:", round(recall_score(
        lorma_dev_true, predictions, average="macro", zero_division=0
    ), 4))
    print("Confusion matrix (rows=true, columns=predicted):")
    print(confusion_matrix(lorma_dev_true, predictions, labels=[0, 1]))

  0%|          | 0/439 [00:00<?, ?it/s]


Threshold: 0.50
Accuracy: 0.5158
Macro-F1: 0.4724
Macro precision: 0.5227
Macro recall: 0.5153
Confusion matrix (rows=true, columns=predicted):
[[ 401 1348]
 [ 349 1407]]

Threshold: 0.55
Accuracy: 0.5332
Macro-F1: 0.533
Macro precision: 0.5334
Macro recall: 0.5333
Confusion matrix (rows=true, columns=predicted):
[[978 771]
 [865 891]]


In [105]:
import json
import torch
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score

save_dir = Path("/kaggle/working/lorma_plus_a4")
save_dir.mkdir(parents=True, exist_ok=True)

# Save model state and classifier separately
torch.save({
    "model_state_dict": model.state_dict(),
    "classification_head_state_dict": classification_head.state_dict(),
    "adapter": "LoRMA+",
    "base_checkpoint": "A4 step 22000",
    "rank": 8,
    "alpha": 16,
    "trainable_parameters": 99330,
}, save_dir / "lorma_plus_a4_checkpoint.pt")

# Save measured results
results = {
    "experiment": "Corrected LoRMA+ on A4",
    "training_subset_size": 2048,
    "epochs": 1,
    "batch_size": 8,
    "learning_rate": 5e-5,
    "trainable_parameters": 99330,
    "development_examples": 3505,
    "threshold_0.50": {
        "accuracy": 0.5158,
        "macro_f1": 0.4724,
        "macro_precision": 0.5227,
        "macro_recall": 0.5153,
        "confusion_matrix": [[401, 1348], [349, 1407]]
    },
    "threshold_0.55": {
        "accuracy": 0.5332,
        "macro_f1": 0.5330,
        "macro_precision": 0.5334,
        "macro_recall": 0.5333,
        "confusion_matrix": [[978, 771], [865, 891]]
    },
    "selected_threshold": 0.55,
    "note": "Threshold selected using the same development set; result may be optimistic."
}

with open(save_dir / "results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Checkpoint saved:", save_dir / "lorma_plus_a4_checkpoint.pt")
print("Results saved:", save_dir / "results.json")
print("Development Macro-F1 at threshold 0.55:", results["threshold_0.55"]["macro_f1"])

Checkpoint saved: /kaggle/working/lorma_plus_a4/lorma_plus_a4_checkpoint.pt
Results saved: /kaggle/working/lorma_plus_a4/results.json
Development Macro-F1 at threshold 0.55: 0.533


In [106]:
from pathlib import Path

checkpoint_paths = [
    "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
]

for path in checkpoint_paths:
    p = Path(path)
    print(("FOUND " if p.is_file() else "MISSING "), path)
    if p.is_file():
        print(f"       Size: {p.stat().st_size / (1024**2):.1f} MB")

FOUND  /kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
       Size: 593.7 MB
FOUND  /kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt
       Size: 196.5 MB
FOUND  /kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt
       Size: 197.9 MB
FOUND  /kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt
       Size: 367.8 MB


In [107]:
import torch

checkpoint_paths = {
    "A1": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
}

for name, path in checkpoint_paths.items():
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)

    if isinstance(checkpoint, dict):
        print("Top-level keys:", list(checkpoint.keys())[:20])

        state = checkpoint.get("model_state_dict", checkpoint.get("model", checkpoint))
        if isinstance(state, dict):
            print("Tensor count:", sum(torch.is_tensor(v) for v in state.values()))
            print("First 15 state keys and shapes:")
            shown = 0
            for key, value in state.items():
                if torch.is_tensor(value):
                    print(f"  {key}: {tuple(value.shape)}")
                    shown += 1
                    if shown >= 15:
                        break
    else:
        print("Checkpoint type:", type(checkpoint).__name__)


==================== A1 ====================
Top-level keys: ['step', 'epoch', 'train_loss', 'val_loss', 'model_state_dict', 'optimizer_state_dict']
Tensor count: 101
First 15 state keys and shapes:
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512)
  blocks.0.norm1.weight: (512,)
  blocks.0.norm1.bias: (512,)
  blocks.0.attn.in_proj_weight: (1536, 512)
  blocks.0.attn.in_proj_bias: (1536,)
  blocks.0.attn.out_proj.weight: (512, 512)
  blocks.0.attn.out_proj.bias: (512,)
  blocks.0.norm2.weight: (512,)
  blocks.0.norm2.bias: (512,)
  blocks.0.ffn.0.weight: (2048, 512)
  blocks.0.ffn.0.bias: (2048,)
  blocks.0.ffn.2.weight: (512, 2048)
  blocks.0.ffn.2.bias: (512,)
  blocks.1.norm1.weight: (512,)

==================== A2 ====================
Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
Tensor count: 113
First 15 state keys and shapes:
  output_bias: (50000,)
  token_embedding.weight: (50000, 512)
  pos

In [108]:
import torch

checkpoint_paths = {
    "A1": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
}

for name, path in checkpoint_paths.items():
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    state = ckpt.get(
        "model_state_dict",
        ckpt.get("model", ckpt)
    )

    print(f"\n{name}")
    for key in (
        "architecture", "parameters", "context_length",
        "hidden_size", "num_layers", "num_heads",
        "ffn_size", "step", "tokens_processed", "tokens_seen"
    ):
        if key in ckpt:
            print(f"  {key}: {ckpt[key]}")

    layer_prefixes = ("blocks.", "layers.", "encoder_layers.")
    layer_ids = set()
    for key in state:
        for prefix in layer_prefixes:
            if key.startswith(prefix):
                try:
                    layer_ids.add(int(key[len(prefix):].split(".")[0]))
                except ValueError:
                    pass
    print("  Detected layer indices:", sorted(layer_ids))


A1
  step: 500
  Detected layer indices: [0, 1, 2, 3, 4, 5, 6, 7]

A2
  step: 9000
  tokens_processed: 73728000
  Detected layer indices: [0, 1, 2, 3, 4, 5, 6, 7, 8]

A3
  architecture: A3_prefix_lm_hybrid
  parameters: 51868672
  context_length: 2048
  hidden_size: 512
  num_layers: 8
  num_heads: 8
  ffn_size: 2048
  step: 1000
  tokens_seen: 512000
  Detected layer indices: [0, 1, 2, 3, 4, 5, 6, 7]

A4
  step: 22000
  tokens_processed: 180096704
  Detected layer indices: [0, 1, 2, 3, 4, 5]


In [109]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory total (GB):", round(
    torch.cuda.get_device_properties(0).total_memory / 1e9, 2
))
print("GPU memory allocated (GB):", round(
    torch.cuda.memory_allocated(0) / 1e9, 2
))
print("GPU memory reserved (GB):", round(
    torch.cuda.memory_reserved(0) / 1e9, 2
))

a2_path = "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt"
a2_ckpt = torch.load(a2_path, map_location="cpu", weights_only=False)

print("\nA2 metadata:")
for key, value in a2_ckpt.items():
    if key != "model_state_dict":
        print(f"{key}: {value}")

GPU: Tesla T4
GPU memory total (GB): 15.64
GPU memory allocated (GB): 0.31
GPU memory reserved (GB): 2.13

A2 metadata:
step: 9000
tokens_processed: 73728000
train_loss: 6.801188290119171
reason: regular_checkpoint
saved_at: 2026-10-05T19:17:55.016904


In [110]:
import inspect
import torch

print("A4 model class:", type(model).__name__)
print("A4 forward signature:", inspect.signature(model.forward))
print("A4 encoder layer count:", len(model.encoder_layers))
print("A4 decoder layer count:", len(model.decoder_layers))
print("Token embedding shape:", tuple(model.token_embedding.weight.shape))
print("LM head shape:", tuple(model.lm_head.weight.shape))
print("Classification head:", classification_head)

print("\nCurrent GPU memory:")
print("Allocated (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))
print("Reserved (GB):", round(torch.cuda.memory_reserved() / 1e9, 2))

A4 model class: A4EncoderDecoder
A4 forward signature: (encoder_ids, decoder_ids)
A4 encoder layer count: 6
A4 decoder layer count: 6
Token embedding shape: (50000, 512)
LM head shape: (50000, 512)
Classification head: Linear(in_features=512, out_features=2, bias=True)

Current GPU memory:
Allocated (GB): 0.31
Reserved (GB): 2.13


In [111]:
from pathlib import Path
import torch

checkpoint_path = Path(
    "/kaggle/working/lorma_plus_a4/lorma_plus_a4_checkpoint.pt"
)

print("Checkpoint exists:", checkpoint_path.is_file())

if checkpoint_path.is_file():
    saved = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    print("Saved keys:", list(saved.keys()))
    print("Adapter method:", saved.get("adapter"))
    print("Base checkpoint:", saved.get("base_checkpoint"))
    print("Trainable parameters:", saved.get("trainable_parameters"))
    print("Model state entries:", len(saved["model_state_dict"]))
    print("Classifier state entries:",
          len(saved["classification_head_state_dict"]))

Checkpoint exists: True
Saved keys: ['model_state_dict', 'classification_head_state_dict', 'adapter', 'base_checkpoint', 'rank', 'alpha', 'trainable_parameters']
Adapter method: LoRMA+
Base checkpoint: A4 step 22000
Trainable parameters: 99330
Model state entries: 284
Classifier state entries: 2


In [112]:
import torch
import torch.nn as nn

print("Available model-like objects:")

for name, value in list(globals().items()):
    if isinstance(value, nn.Module):
        print(f"{name}: {type(value).__name__}")

print("\nAvailable dataset and tokenizer objects:")
for name in ["train_data", "dev_data", "test_data", "tokenizer", "prepare_batch"]:
    print(f"{name}: {'available' if name in globals() else 'missing'}")

Available model-like objects:
base: Linear
model: A4EncoderDecoder
layer: EncoderBlock
classification_head: Linear
criterion: CrossEntropyLoss
weighted_head: Linear
weighted_criterion: CrossEntropyLoss
bitfit_head: Linear
module: PaperLoRMAPlusLinear
original: Linear
adapted: LoRMAPlusLinear
test_layer: LoRMAPlusLinear
head: Linear
adapter: PaperLoRMAPlusLinear
test_base: Linear
test_wrapper: PaperLoRMAPlusLinear
attention: MultiHeadAttention
old_wrapper: LoRMAPlusLinear
base_layer: Linear
new_wrapper: PaperLoRMAPlusLinear

Available dataset and tokenizer objects:
train_data: available
dev_data: available
test_data: missing
tokenizer: available
prepare_batch: available


In [113]:
import torch.nn as nn

print("=== A3-related notebook objects ===")

for name, value in list(globals().items()):
    if "A3" in name.upper() or "PREFIX" in name.upper():
        try:
            print(name, "->", type(value).__name__)
        except Exception:
            pass

print("\n=== Model class definitions ===")

for name, value in list(globals().items()):
    if isinstance(value, type) and issubclass(value, nn.Module):
        if any(term in name.lower() for term in ["a1", "a2", "a3", "prefix", "hybrid"]):
            print(name)

print("\n=== A3 checkpoint ===")
from pathlib import Path

a3_path = Path(
    "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt"
)
print("Checkpoint exists:", a3_path.is_file())

if a3_path.is_file():
    checkpoint = torch.load(
        a3_path, map_location="cpu", weights_only=False
    )
    print("Checkpoint type:", type(checkpoint).__name__)
    if isinstance(checkpoint, dict):
        print("Checkpoint keys:", list(checkpoint.keys())[:30])

=== A3-related notebook objects ===
prefix -> str
layer_prefixes -> tuple

=== Model class definitions ===

=== A3 checkpoint ===
Checkpoint exists: True
Checkpoint type: dict
Checkpoint keys: ['model', 'step', 'tokens_seen', 'architecture', 'parameters', 'context_length', 'hidden_size', 'num_layers', 'num_heads', 'ffn_size', 'prefix_ratio', 'dropout', 'learning_rate', 'weight_decay']


In [114]:
import torch

a3_path = "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt"
ckpt = torch.load(a3_path, map_location="cpu", weights_only=False)

print("=== A3 configuration ===")
for key in [
    "architecture",
    "parameters",
    "context_length",
    "hidden_size",
    "num_layers",
    "num_heads",
    "ffn_size",
    "prefix_ratio",
    "dropout",
]:
    print(f"{key}: {ckpt.get(key)}")

print("\n=== State-dictionary keys and shapes ===")
state = ckpt["model"]

print("State type:", type(state).__name__)
print("Number of entries:", len(state))

for i, (key, value) in enumerate(state.items()):
    if i == 35:
        break
    print(key, tuple(value.shape) if hasattr(value, "shape") else type(value).__name__)

=== A3 configuration ===
architecture: A3_prefix_lm_hybrid
parameters: 51868672
context_length: 2048
hidden_size: 512
num_layers: 8
num_heads: 8
ffn_size: 2048
prefix_ratio: 0.5
dropout: 0.1

=== State-dictionary keys and shapes ===
State type: OrderedDict
Number of entries: 100
token_embedding.weight (50000, 512)
position_embedding.weight (2048, 512)
layers.0.ln1.weight (512,)
layers.0.ln1.bias (512,)
layers.0.attn.in_proj_weight (1536, 512)
layers.0.attn.in_proj_bias (1536,)
layers.0.attn.out_proj.weight (512, 512)
layers.0.attn.out_proj.bias (512,)
layers.0.ln2.weight (512,)
layers.0.ln2.bias (512,)
layers.0.ffn.0.weight (2048, 512)
layers.0.ffn.0.bias (2048,)
layers.0.ffn.3.weight (512, 2048)
layers.0.ffn.3.bias (512,)
layers.1.ln1.weight (512,)
layers.1.ln1.bias (512,)
layers.1.attn.in_proj_weight (1536, 512)
layers.1.attn.in_proj_bias (1536,)
layers.1.attn.out_proj.weight (512, 512)
layers.1.attn.out_proj.bias (512,)
layers.1.ln2.weight (512,)
layers.1.ln2.bias (512,)
layers.1.ff

In [115]:
from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input/datasets/aq1de3"),
]

terms = [
    "A3_prefix_lm_hybrid",
    "prefix_ratio",
    "class A3",
    "class Prefix",
]

extensions = {".py", ".ipynb", ".txt"}

for root in roots:
    print(f"\nSearching: {root}")
    if not root.exists():
        print("Directory not found")
        continue

    matches = 0
    for path in root.rglob("*"):
        if not path.is_file() or path.suffix not in extensions:
            continue

        try:
            if path.stat().st_size > 10_000_000:
                continue
            content = path.read_text(errors="ignore")
        except Exception:
            continue

        found = [term for term in terms if term.lower() in content.lower()]
        if found:
            print(path, "->", found)
            matches += 1
            if matches >= 20:
                print("Search output capped at 20 files.")
                break


Searching: /kaggle/working
/kaggle/working/.virtual_documents/__notebook_source__.ipynb -> ['A3_prefix_lm_hybrid', 'prefix_ratio', 'class A3', 'class Prefix']

Searching: /kaggle/input/datasets/aq1de3
/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb -> ['A3_prefix_lm_hybrid', 'prefix_ratio', 'class A3', 'class Prefix']
/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb -> ['A3_prefix_lm_hybrid', 'prefix_ratio', 'class A3', 'class Prefix']
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb -> ['class Prefix']
/kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb -> ['prefix_ratio', 'class Prefix']
/kaggle/input/datasets/aq1de3/p2p2p2/phase-2 (1).ipynb -> ['A3_prefix_lm_hybrid', 'prefix_ratio', 'class A3']


In [116]:
import json
from pathlib import Path

notebook_path = Path(
    "/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if (
        "A3_prefix_lm_hybrid" in source
        or "class A3" in source
        or "prefix_ratio" in source
    ):
        print(f"\n{'='*25} CELL {i} {'='*25}")
        print(source[:12000])


========================= CELL 17 =========================
# ============================================================
# A3 PREFIX-LM HYBRID — MODEL
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

DEVICE = "cuda"

VOCAB_SIZE = 50_000
CONTEXT_LEN = 2_048
HIDDEN_SIZE = 512
NUM_LAYERS = 8
NUM_HEADS = 8
FFN_SIZE = 2_048
PREFIX_RATIO = 0.5

class PrefixLMBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(HIDDEN_SIZE)
        self.attn = nn.MultiheadAttention(
            embed_dim=HIDDEN_SIZE,
            num_heads=NUM_HEADS,
            batch_first=True
        )

        self.ln2 = nn.LayerNorm(HIDDEN_SIZE)

        self.ffn = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, FFN_SIZE),
            nn.GELU(),
            nn.Linear(FFN_SIZE, HIDDEN_SIZE)
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)

        attn_o

In [117]:
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = 50_000
CONTEXT_LEN = 2_048
HIDDEN_SIZE = 512
NUM_LAYERS = 8
NUM_HEADS = 8
FFN_SIZE = 2_048
PREFIX_RATIO = 0.5
DROPOUT = 0.1

class PrefixLMBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(HIDDEN_SIZE)
        self.attn = nn.MultiheadAttention(
            HIDDEN_SIZE, NUM_HEADS,
            dropout=DROPOUT, batch_first=True
        )
        self.attn_dropout = nn.Dropout(DROPOUT)
        self.ln2 = nn.LayerNorm(HIDDEN_SIZE)
        self.ffn = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, FFN_SIZE),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(FFN_SIZE, HIDDEN_SIZE),
            nn.Dropout(DROPOUT)
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)
        attn_out, _ = self.attn(
            h, h, h, attn_mask=attn_mask,
            need_weights=False
        )
        x = x + self.attn_dropout(attn_out)
        x = x + self.ffn(self.ln2(x))
        return x

class A3PrefixLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, HIDDEN_SIZE)
        self.position_embedding = nn.Embedding(CONTEXT_LEN, HIDDEN_SIZE)
        self.embedding_dropout = nn.Dropout(DROPOUT)
        self.layers = nn.ModuleList(
            [PrefixLMBlock() for _ in range(NUM_LAYERS)]
        )
        self.ln_f = nn.LayerNorm(HIDDEN_SIZE)
        self.register_buffer(
            "position_ids",
            torch.arange(CONTEXT_LEN).unsqueeze(0),
            persistent=False
        )

    def build_mask(self, seq_len, device):
        prefix_len = int(seq_len * PREFIX_RATIO)
        mask = torch.full(
            (seq_len, seq_len), float("-inf"), device=device
        )
        mask[:prefix_len, :prefix_len] = 0
        for i in range(prefix_len, seq_len):
            mask[i, :prefix_len] = 0
            mask[i, prefix_len:i + 1] = 0
        return mask

    def forward(self, input_ids):
        _, seq_len = input_ids.shape
        positions = self.position_ids[:, :seq_len]
        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )
        x = self.embedding_dropout(x)
        mask = self.build_mask(seq_len, input_ids.device)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.ln_f(x)
        return F.linear(x, self.token_embedding.weight)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
a3_path = "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt"

a3_checkpoint = torch.load(
    a3_path, map_location="cpu", weights_only=False
)

a3_student = A3PrefixLM().to(device)
load_result = a3_student.load_state_dict(
    a3_checkpoint["model"], strict=True
)
a3_student.eval()

print("Checkpoint loaded:", load_result)
print("Architecture:", a3_checkpoint["architecture"])
print("Checkpoint step:", a3_checkpoint["step"])
print("Parameters:", f"{sum(p.numel() for p in a3_student.parameters()):,}")
print("Expected:", f"{a3_checkpoint['parameters']:,}")
print("Device:", device)

Checkpoint loaded: <All keys matched successfully>
Architecture: A3_prefix_lm_hybrid
Checkpoint step: 1000
Parameters: 51,868,672
Expected: 51,868,672
Device: cuda


In [118]:
print("=== Teacher ===")
print("A4 model available:", "model" in globals())
print("A4 classifier available:", "classification_head" in globals())
print("LoRMA+ wrappers:", sum(
    isinstance(m, PaperLoRMAPlusLinear)
    for m in model.modules()
) if "model" in globals() else "A4 model missing")

print("\n=== Data ===")
print("Training rows:", len(train_data))
print("Development rows:", len(dev_data))

if len(train_data):
    print("Training record keys:", list(train_data[0].keys()))
if len(dev_data):
    print("Development record keys:", list(dev_data[0].keys()))

print("\n=== Batch preparation check ===")
sample_ids, sample_token_labels, sample_labels = prepare_batch(
    train_data[:2], max_length=1024
)
print("Input shape:", tuple(sample_ids.shape))
print("Labels shape:", tuple(sample_labels.shape))
print("Labels:", sample_labels.tolist())
print("Input device:", sample_ids.device)

=== Teacher ===
A4 model available: True
A4 classifier available: True
LoRMA+ wrappers: 12

=== Data ===
Training rows: 37387
Development rows: 3505
Training record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
Development record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']

=== Batch preparation check ===
Input shape: (2, 1024)
Labels shape: (2,)
Labels: [0, 0]
Input device: cuda:0


In [119]:
import torch
import torch.nn as nn

# Keep both pretrained backbones frozen.
model.eval()
a3_student.eval()

for p in model.parameters():
    p.requires_grad = False

for p in a3_student.parameters():
    p.requires_grad = False

# Separate classification heads for this distillation experiment.
distill_student_head = nn.Linear(512, 2).to(device)

# Teacher head comes from the saved A4 LoRMA+ experiment.
classification_head.eval()
for p in classification_head.parameters():
    p.requires_grad = False

# Only the new student classification head will be trained.
distill_optimizer = torch.optim.AdamW(
    distill_student_head.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

print("Teacher frozen:", all(not p.requires_grad for p in model.parameters()))
print("Student backbone frozen:", all(not p.requires_grad for p in a3_student.parameters()))
print("Student head trainable:", all(p.requires_grad for p in distill_student_head.parameters()))
print("Student-head parameters:", sum(p.numel() for p in distill_student_head.parameters()))
print("Device:", device)

Teacher frozen: True
Student backbone frozen: True
Student head trainable: True
Student-head parameters: 1026
Device: cuda


In [120]:
import torch
import torch.nn.functional as F

model.eval()
a3_student.eval()
classification_head.eval()
distill_student_head.eval()

input_ids, _, labels = prepare_batch(
    train_data[:2], max_length=1024
)
input_ids = input_ids.to(device)

with torch.no_grad():
    # A4 teacher: encoder representation + masked mean pooling
    positions = torch.arange(input_ids.shape[1], device=device)
    x = model.token_embedding(input_ids) + model.encoder_pos(positions)[None, :, :]
    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    a4_encoded = model.encoder_norm(x)
    mask = (input_ids != 0).unsqueeze(-1)
    a4_pooled = (a4_encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    teacher_logits = classification_head(a4_pooled)

    # A3 student: hidden representations from the original A3 model
    seq_len = input_ids.shape[1]
    positions_a3 = a3_student.position_ids[:, :seq_len]
    a3_x = (
        a3_student.token_embedding(input_ids)
        + a3_student.position_embedding(positions_a3)
    )
    a3_x = a3_student.embedding_dropout(a3_x)
    attention_mask = a3_student.build_mask(seq_len, input_ids.device)

    for layer in a3_student.layers:
        a3_x = layer(a3_x, attention_mask)

    a3_encoded = a3_student.ln_f(a3_x)
    a3_mask = (input_ids != 0).unsqueeze(-1)
    a3_pooled = (a3_encoded * a3_mask).sum(dim=1) / a3_mask.sum(dim=1).clamp(min=1)
    student_logits = distill_student_head(a3_pooled)

print("Teacher representation:", tuple(a4_pooled.shape))
print("Teacher logits:", tuple(teacher_logits.shape))
print("Student representation:", tuple(a3_pooled.shape))
print("Student logits:", tuple(student_logits.shape))
print("Teacher logits finite:", torch.isfinite(teacher_logits).all().item())
print("Student logits finite:", torch.isfinite(student_logits).all().item())
print("Labels:", labels.tolist())

Teacher representation: (2, 512)
Teacher logits: (2, 2)
Student representation: (2, 512)
Student logits: (2, 2)
Teacher logits finite: True
Student logits finite: True
Labels: [0, 0]


In [121]:
import random
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DISTILL_N = 2048
DISTILL_BATCH_SIZE = 8
TEMPERATURE = 2.0
DISTILL_WEIGHT = 0.5
DISTILL_EPOCHS = 1

# Reproducible training subset.
indices = list(range(len(train_data)))
random.shuffle(indices)
distill_train_records = [train_data[i] for i in indices[:DISTILL_N]]

distill_student_head.train()
model.eval()
a3_student.eval()
classification_head.eval()

loss_total = 0.0
preds_all, labels_all = [], []
num_batches = (len(distill_train_records) + DISTILL_BATCH_SIZE - 1) // DISTILL_BATCH_SIZE

for start in range(0, len(distill_train_records), DISTILL_BATCH_SIZE):
    records = distill_train_records[start:start + DISTILL_BATCH_SIZE]
    input_ids, _, labels = prepare_batch(records, max_length=1024)
    input_ids = input_ids.to(device)
    labels = labels.to(device)

    # Teacher soft targets: no gradients.
    with torch.no_grad():
        positions = torch.arange(input_ids.shape[1], device=device)
        tx = model.token_embedding(input_ids) + model.encoder_pos(positions)[None, :, :]
        tx = model.dropout(tx)
        for layer in model.encoder_layers:
            tx = layer(tx)
        tx = model.encoder_norm(tx)

        tmask = (input_ids != 0).unsqueeze(-1)
        tpooled = (tx * tmask).sum(dim=1) / tmask.sum(dim=1).clamp(min=1)
        teacher_logits = classification_head(tpooled)

        teacher_soft = F.softmax(teacher_logits / TEMPERATURE, dim=-1)

    # Student representation.
    positions_a3 = a3_student.position_ids[:, :input_ids.shape[1]]
    sx = (
        a3_student.token_embedding(input_ids)
        + a3_student.position_embedding(positions_a3)
    )
    sx = a3_student.embedding_dropout(sx)
    amask = a3_student.build_mask(input_ids.shape[1], input_ids.device)

    for layer in a3_student.layers:
        sx = layer(sx, amask)
    sx = a3_student.ln_f(sx)

    smask = (input_ids != 0).unsqueeze(-1)
    spooled = (sx * smask).sum(dim=1) / smask.sum(dim=1).clamp(min=1)
    student_logits = distill_student_head(spooled)

    hard_loss = F.cross_entropy(student_logits, labels)
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / TEMPERATURE, dim=-1),
        teacher_soft,
        reduction="batchmean"
    ) * (TEMPERATURE ** 2)

    loss = (
        (1.0 - DISTILL_WEIGHT) * hard_loss
        + DISTILL_WEIGHT * soft_loss
    )

    distill_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        distill_student_head.parameters(), 1.0
    )
    distill_optimizer.step()

    loss_total += loss.item()
    preds_all.extend(student_logits.argmax(dim=-1).detach().cpu().tolist())
    labels_all.extend(labels.detach().cpu().tolist())

    if (start // DISTILL_BATCH_SIZE + 1) % 64 == 0:
        print(
            f"Batch {start // DISTILL_BATCH_SIZE + 1}/{num_batches} "
            f"| Loss {loss.item():.4f}"
        )

print("\n=== Distillation training results ===")
print("Training records:", len(distill_train_records))
print("Batches:", num_batches)
print("Mean combined loss:", round(loss_total / num_batches, 4))
print("Training Macro-F1:", round(
    f1_score(labels_all, preds_all, average="macro", zero_division=0), 4
))
print("Prediction counts:", np.bincount(preds_all, minlength=2).tolist())

Batch 64/256 | Loss 0.3162
Batch 128/256 | Loss 0.3562
Batch 192/256 | Loss 0.3562
Batch 256/256 | Loss 0.3443

=== Distillation training results ===
Training records: 2048
Batches: 256
Mean combined loss: 0.3695
Training Macro-F1: 0.4963
Prediction counts: [1433, 615]


In [122]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix
)

@torch.no_grad()
def evaluate_a4_teacher(records, batch_size=8):
    model.eval()
    classification_head.eval()
    all_probs, all_labels = [], []

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        ids, _, labels = prepare_batch(batch, max_length=1024)
        ids = ids.to(device)

        positions = torch.arange(ids.shape[1], device=device)
        x = model.token_embedding(ids) + model.encoder_pos(positions)[None, :, :]
        x = model.dropout(x)
        for layer in model.encoder_layers:
            x = layer(x)
        encoded = model.encoder_norm(x)

        mask = (ids != 0).unsqueeze(-1)
        pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        logits = classification_head(pooled)

        all_probs.extend(F.softmax(logits, dim=-1)[:, 1].cpu().tolist())
        all_labels.extend(labels.tolist())

    return np.array(all_labels), np.array(all_probs)

@torch.no_grad()
def evaluate_a3_student(records, batch_size=8):
    a3_student.eval()
    distill_student_head.eval()
    all_probs, all_labels = [], []

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        ids, _, labels = prepare_batch(batch, max_length=1024)
        ids = ids.to(device)

        seq_len = ids.shape[1]
        positions = a3_student.position_ids[:, :seq_len]
        x = (
            a3_student.token_embedding(ids)
            + a3_student.position_embedding(positions)
        )
        x = a3_student.embedding_dropout(x)
        attention_mask = a3_student.build_mask(seq_len, ids.device)

        for layer in a3_student.layers:
            x = layer(x, attention_mask)
        encoded = a3_student.ln_f(x)

        mask = (ids != 0).unsqueeze(-1)
        pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        logits = distill_student_head(pooled)

        all_probs.extend(F.softmax(logits, dim=-1)[:, 1].cpu().tolist())
        all_labels.extend(labels.tolist())

    return np.array(all_labels), np.array(all_probs)

def report_metrics(name, y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    print(f"\n=== {name} | threshold={threshold:.2f} ===")
    print("Accuracy:", round(accuracy_score(y_true, predictions), 4))
    print("Macro-F1:", round(f1_score(y_true, predictions, average="macro", zero_division=0), 4))
    print("Macro precision:", round(precision_score(y_true, predictions, average="macro", zero_division=0), 4))
    print("Macro recall:", round(recall_score(y_true, predictions, average="macro", zero_division=0), 4))
    print("Confusion matrix:\n", confusion_matrix(y_true, predictions, labels=[0, 1]))

dev_y_teacher, dev_p_teacher = evaluate_a4_teacher(dev_data)
dev_y_student, dev_p_student = evaluate_a3_student(dev_data)

assert np.array_equal(dev_y_teacher, dev_y_student)

report_metrics("A4 LoRMA+ teacher", dev_y_teacher, dev_p_teacher)
report_metrics("A3 distilled student", dev_y_student, dev_p_student)

print("\nDevelopment records evaluated:", len(dev_y_student))


=== A4 LoRMA+ teacher | threshold=0.50 ===
Accuracy: 0.5158
Macro-F1: 0.4724
Macro precision: 0.5227
Macro recall: 0.5153
Confusion matrix:
 [[ 401 1348]
 [ 349 1407]]

=== A3 distilled student | threshold=0.50 ===
Accuracy: 0.501
Macro-F1: 0.3338
Macro precision: 0.2505
Macro recall: 0.5
Confusion matrix:
 [[   0 1749]
 [   0 1756]]

Development records evaluated: 3505


In [123]:
import numpy as np
from sklearn.metrics import f1_score

print("Student positive-probability distribution:")
print("Min:", round(float(dev_p_student.min()), 4))
print("25th percentile:", round(float(np.quantile(dev_p_student, 0.25)), 4))
print("Median:", round(float(np.median(dev_p_student)), 4))
print("75th percentile:", round(float(np.quantile(dev_p_student, 0.75)), 4))
print("Max:", round(float(dev_p_student.max()), 4))
print("Actual class counts:", np.bincount(dev_y_student, minlength=2).tolist())

print("\nStudent threshold diagnostic:")
for threshold in [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]:
    preds = (dev_p_student >= threshold).astype(int)
    score = f1_score(
        dev_y_student, preds,
        average="macro", zero_division=0
    )
    counts = np.bincount(preds, minlength=2)
    print(
        f"Threshold={threshold:.2f} | "
        f"Macro-F1={score:.4f} | "
        f"Class 0={counts[0]} | Class 1={counts[1]}"
    )

Student positive-probability distribution:
Min: 0.552
25th percentile: 0.5545
Median: 0.5549
75th percentile: 0.5552
Max: 0.5588
Actual class counts: [1749, 1756]

Student threshold diagnostic:
Threshold=0.30 | Macro-F1=0.3338 | Class 0=0 | Class 1=3505
Threshold=0.40 | Macro-F1=0.3338 | Class 0=0 | Class 1=3505
Threshold=0.45 | Macro-F1=0.3338 | Class 0=0 | Class 1=3505
Threshold=0.50 | Macro-F1=0.3338 | Class 0=0 | Class 1=3505
Threshold=0.55 | Macro-F1=0.3338 | Class 0=0 | Class 1=3505
Threshold=0.60 | Macro-F1=0.3329 | Class 0=3505 | Class 1=0
Threshold=0.70 | Macro-F1=0.3329 | Class 0=3505 | Class 1=0


In [124]:
import torch

print("Student head weights:")
print(distill_student_head.weight.detach().cpu())
print("\nStudent head bias:")
print(distill_student_head.bias.detach().cpu())

print("\nHead weight standard deviation:",
      distill_student_head.weight.detach().std().item())
print("Head bias difference:",
      (distill_student_head.bias[1] -
       distill_student_head.bias[0]).item())

print("\nStudent pooled representation variation:")
print("Dev probability standard deviation:",
      float(dev_p_student.std()))

Student head weights:
tensor([[-0.0425, -0.0032, -0.0425,  ...,  0.0064,  0.0245,  0.0167],
        [ 0.0172,  0.0384, -0.0250,  ..., -0.0423, -0.0358,  0.0035]])

Student head bias:
tensor([-0.0232,  0.0347])

Head weight standard deviation: 0.026049289852380753
Head bias difference: 0.057935670018196106

Student pooled representation variation:
Dev probability standard deviation: 0.0006721618693181843


In [125]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score

random.seed(43)
np.random.seed(43)
torch.manual_seed(43)
torch.cuda.manual_seed_all(43)

# Fresh head: preserve the original distillation head and its results.
weighted_student_head = nn.Linear(512, 2).to(device)
weighted_optimizer = torch.optim.AdamW(
    weighted_student_head.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

# Class weights calculated from the full training set.
train_label_counts = np.bincount(
    [int(r["label"]) for r in train_data], minlength=2
)
class_weights_student = torch.tensor(
    train_label_counts.sum() / (2.0 * train_label_counts),
    dtype=torch.float32,
    device=device
)

# Reuse the same 2,048-record subset for a controlled comparison.
weighted_student_head.train()
a3_student.eval()
batch_size = 8
loss_sum = 0.0
train_preds, train_labels = [], []

for start in range(0, len(distill_train_records), batch_size):
    records = distill_train_records[start:start + batch_size]
    input_ids, _, labels = prepare_batch(records, max_length=1024)
    input_ids = input_ids.to(device)
    labels = labels.to(device)

    # Frozen A3 representation.
    with torch.no_grad():
        seq_len = input_ids.shape[1]
        positions = a3_student.position_ids[:, :seq_len]
        x = (
            a3_student.token_embedding(input_ids)
            + a3_student.position_embedding(positions)
        )
        x = a3_student.embedding_dropout(x)
        attn_mask = a3_student.build_mask(seq_len, input_ids.device)

        for layer in a3_student.layers:
            x = layer(x, attn_mask)

        x = a3_student.ln_f(x)
        mask = (input_ids != 0).unsqueeze(-1)
        pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    logits = weighted_student_head(pooled)
    loss = F.cross_entropy(
        logits, labels, weight=class_weights_student
    )

    weighted_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    weighted_optimizer.step()

    loss_sum += loss.item()
    train_preds.extend(logits.argmax(-1).detach().cpu().tolist())
    train_labels.extend(labels.cpu().tolist())

print("Training records:", len(distill_train_records))
print("Class counts:", train_label_counts.tolist())
print("Class weights:", class_weights_student.tolist())
print("Mean training loss:", round(loss_sum / (
    (len(distill_train_records) + batch_size - 1) // batch_size
), 4))
print("Training Macro-F1:", round(
    f1_score(train_labels, train_preds, average="macro", zero_division=0), 4
))
print("Training predictions:", np.bincount(train_preds, minlength=2).tolist())

Training records: 2048
Class counts: [22067, 15320]
Class weights: [0.8471246361732483, 1.2202023267745972]
Mean training loss: 0.7165
Training Macro-F1: 0.4903
Training predictions: [1172, 876]


In [126]:
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix
)

a3_student.eval()
weighted_student_head.eval()

weighted_dev_probs = []
weighted_dev_labels = []

with torch.no_grad():
    for start in range(0, len(dev_data), 8):
        records = dev_data[start:start + 8]
        input_ids, _, labels = prepare_batch(records, max_length=1024)
        input_ids = input_ids.to(device)

        seq_len = input_ids.shape[1]
        positions = a3_student.position_ids[:, :seq_len]
        x = (
            a3_student.token_embedding(input_ids)
            + a3_student.position_embedding(positions)
        )
        x = a3_student.embedding_dropout(x)
        attn_mask = a3_student.build_mask(seq_len, input_ids.device)

        for layer in a3_student.layers:
            x = layer(x, attn_mask)

        x = a3_student.ln_f(x)
        mask = (input_ids != 0).unsqueeze(-1)
        pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        logits = weighted_student_head(pooled)
        weighted_dev_probs.extend(
            torch.softmax(logits, dim=-1)[:, 1].cpu().tolist()
        )
        weighted_dev_labels.extend(labels.tolist())

weighted_dev_probs = np.array(weighted_dev_probs)
weighted_dev_labels = np.array(weighted_dev_labels)
weighted_dev_preds = (weighted_dev_probs >= 0.5).astype(int)

print("Development records:", len(weighted_dev_labels))
print("Accuracy:", round(accuracy_score(weighted_dev_labels, weighted_dev_preds), 4))
print("Macro-F1:", round(f1_score(weighted_dev_labels, weighted_dev_preds, average="macro", zero_division=0), 4))
print("Macro precision:", round(precision_score(weighted_dev_labels, weighted_dev_preds, average="macro", zero_division=0), 4))
print("Macro recall:", round(recall_score(weighted_dev_labels, weighted_dev_preds, average="macro", zero_division=0), 4))
print("Confusion matrix:\n", confusion_matrix(weighted_dev_labels, weighted_dev_preds, labels=[0, 1]))
print("Prediction counts:", np.bincount(weighted_dev_preds, minlength=2).tolist())
print("Probability std:", round(float(weighted_dev_probs.std()), 6))

Development records: 3505
Accuracy: 0.501
Macro-F1: 0.3338
Macro precision: 0.2505
Macro recall: 0.5
Confusion matrix:
 [[   0 1749]
 [   0 1756]]
Prediction counts: [0, 3505]
Probability std: 0.000553


In [127]:
import os
import json
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

save_dir = "/kaggle/working/phase4_results"
os.makedirs(save_dir, exist_ok=True)

# Save the frozen-A3 student with its trained distillation head.
torch.save({
    "student_state_dict": a3_student.state_dict(),
    "distillation_head_state_dict": distill_student_head.state_dict(),
    "weighted_ce_head_state_dict": weighted_student_head.state_dict(),
    "base_checkpoint": "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "backbone_frozen": True,
    "training_records": len(distill_train_records),
    "batch_size": 8,
    "temperature": 2.0,
    "distillation_weight": 0.5,
}, os.path.join(save_dir, "a3_distillation_experiment.pt"))

# Record the already-measured distillation evaluation.
distillation_results = {
    "experiment": "A3 frozen-backbone knowledge distillation from A4 teacher",
    "development_records": 3505,
    "accuracy": 0.5010,
    "macro_f1": 0.3338,
    "macro_precision": 0.2505,
    "macro_recall": 0.5000,
    "confusion_matrix": [[0, 1749], [0, 1756]],
    "prediction_counts": {"class_0": 0, "class_1": 3505},
    "probability_std": 0.000553,
    "status": "unsuccessful: predicts class 1 for every development record",
    "caveat": "Frozen student representations did not yield useful class separation."
}

with open(os.path.join(save_dir, "a3_distillation_results.json"), "w") as f:
    json.dump(distillation_results, f, indent=2)

print("Saved files:")
print(os.path.join(save_dir, "a3_distillation_experiment.pt"))
print(os.path.join(save_dir, "a3_distillation_results.json"))

Saved files:
/kaggle/working/phase4_results/a3_distillation_experiment.pt
/kaggle/working/phase4_results/a3_distillation_results.json


In [128]:
import os

print("Phase 4 saved results:")
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith((".json", ".pt", ".pth")):
            path = os.path.join(root, file)
            print(f"{path} — {os.path.getsize(path) / (1024**2):.2f} MB")

Phase 4 saved results:
/kaggle/working/phase4_experiment_results.json — 0.00 MB
/kaggle/working/a4_adapter_pilot_results.json — 0.00 MB
/kaggle/working/lorma_plus_a4/lorma_plus_a4_checkpoint.pt — 270.52 MB
/kaggle/working/lorma_plus_a4/results.json — 0.00 MB
/kaggle/working/phase4_results/a3_distillation_experiment.pt — 197.91 MB
/kaggle/working/phase4_results/a3_distillation_results.json — 0.00 MB


In [136]:
import json
import os

paths = [
    "/kaggle/working/phase4_experiment_results.json",
    "/kaggle/working/a4_adapter_pilot_results.json",
    "/kaggle/working/lorma_plus_a4/results.json",
]

for path in paths:
    print("\n" + "=" * 70)
    print(os.path.basename(path))
    print("=" * 70)

    if os.path.exists(path):
        with open(path, "r") as f:
            data = json.load(f)
        print(json.dumps(data, indent=2))
    else:
        print("File not found")


phase4_experiment_results.json
[
  {
    "experiment": "A4 frozen encoder - standard loss",
    "accuracy": 0.5258,
    "macro_f1": 0.4487,
    "macro_precision": 0.5607,
    "macro_recall": 0.5266,
    "trainable_parameters": 1026
  },
  {
    "experiment": "A4 frozen encoder - class-weighted loss",
    "accuracy": 0.5327,
    "macro_f1": 0.5235,
    "trainable_parameters": 1026
  },
  {
    "experiment": "A4 BitFit",
    "accuracy": 0.499,
    "macro_f1": 0.353,
    "macro_precision": 0.4995,
    "macro_recall": 0.5,
    "trainable_parameters": 84994,
    "training_loss": 0.6891,
    "training_accuracy": 0.5849
  }
]

a4_adapter_pilot_results.json
{
  "experiment": "A4 multiplicative low-rank adapters pilot",
  "checkpoint": "A4 step 22000",
  "training_examples": 2048,
  "epochs": 1,
  "batch_size": 8,
  "learning_rate": 5e-05,
  "trainable_parameters": 99330,
  "development_examples": 3505,
  "development_macro_f1_threshold_0.50": 0.4806,
  "development_macro_f1_threshold_0.55": 0

In [138]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    dirs[:] = [d for d in dirs if ".ipynb_checkpoints" not in d]
    for name in files:
        if name.endswith(".ipynb"):
            print(os.path.join(root, name))

/kaggle/working/.virtual_documents/__notebook_source__.ipynb


In [140]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")

print("File exists:", path.exists())
print("File size:", path.stat().st_size if path.exists() else "N/A")

if path.exists():
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for i in range(25):
            line = f.readline()
            if not line:
                break
            print(f"{i + 1}: {line.rstrip()[:250]}")

File exists: True
File size: 185025
1: 
2: from pathlib import Path
3: import json
4: 
5: # Find the dataset files mounted by Kaggle
6: matches = list(Path("/kaggle/input").rglob("label_vocab.json"))
7: 
8: if not matches:
9:     print("Dataset not found. Check the Kaggle Input panel.")
10: else:
11:     phase3_root = matches[0].parent
12:     print("Dataset directory:", phase3_root)
13: 
14:     required_files = [
15:         "CL_train.json", "CL_dev.json", "CL_test.json",
16:         "IT_train.json", "IT_dev.json", "IT_test.json",
17:         "label_vocab.json"
18:     ]
19: 
20:     for name in required_files:
21:         path = phase3_root / name
22:         print(("OK" if path.is_file() else "MISSING"), name)
23: 
24:     # Inspect the label vocabulary
25:     vocab_path = phase3_root / "label_vocab.json"


In [141]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
source = path.read_text(encoding="utf-8", errors="replace")
lines = source.splitlines()

terms = [
    "A1", "A2", "A3", "A4",
    "LORMA", "LoRMA", "AdaLoRA",
    "BitFit", "adapter", "freeze",
    "distill", "EXPERIMENT",
    "trainable_parameters", "Macro-F1",
    "classification_head", "lora"
]

print("Source lines:", len(lines))

for term in terms:
    hits = [
        i for i, line in enumerate(lines)
        if term.lower() in line.lower()
    ]
    print(f"{term}: {len(hits)} matches")

print("\n--- Experiment headings and method definitions ---")
shown = set()

for i, line in enumerate(lines):
    low = line.lower()
    if (
        "experiment" in low
        or "class lorma" in low
        or "class loram" in low
        or "class adalora" in low
        or "class bitfit" in low
        or "knowledge distillation" in low
    ):
        start = max(0, i - 1)
        end = min(len(lines), i + 3)
        key = (start, end)
        if key not in shown:
            shown.add(key)
            print(f"\nLines {start + 1}-{end}:")
            for j in range(start, end):
                print(f"{j + 1}: {lines[j][:220]}")

Source lines: 6680
A1: 10 matches
A2: 46 matches
A3: 66 matches
A4: 66 matches
LORMA: 42 matches
LoRMA: 42 matches
AdaLoRA: 0 matches
BitFit: 44 matches
adapter: 62 matches
freeze: 7 matches
distill: 8 matches
EXPERIMENT: 25 matches
trainable_parameters: 7 matches
Macro-F1: 14 matches
classification_head: 69 matches
lora: 0 matches

--- Experiment headings and method definitions ---

Lines 4497-4500:
4497: 
4498: # Reset the classifier so this is a fresh full-data experiment
4499: torch.manual_seed(42)
4500: classification_head = nn.Linear(512, 2).to(device)

Lines 4705-4708:
4705: 
4706: experiment_results = [
4707:     {
4708:         "experiment": "A4 frozen encoder - standard loss",

Lines 4707-4710:
4707:     {
4708:         "experiment": "A4 frozen encoder - standard loss",
4709:         "accuracy": 0.5258,
4710:         "macro_f1": 0.4487,

Lines 4715-4718:
4715:     {
4716:         "experiment": "A4 frozen encoder - class-weighted loss",
4717:         "accuracy": 0.5327,
4718: 

In [142]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

# Display the original LORMA implementation and training setup.
for start, end in [(15080, 15220), (13880, 14010)]:
    print(f"\n{'=' * 18} Lines {start}-{end} {'=' * 18}")
    if start > len(lines):
        print("This line range is outside the source file.")
        continue

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1}: {lines[i][:220]}")


================== Lines 15080-15220 ==================
This line range is outside the source file.

================== Lines 13880-14010 ==================
This line range is outside the source file.


In [143]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

terms = [
    "class LORMALinear",
    "class LoRMALinear",
    "class LORMA",
    "LORMA into Q and V",
    "lorma_encoder =",
    "EXPERIMENT 4",
    "A4_LORMA",
]

for term in terms:
    hits = [
        i for i, line in enumerate(lines)
        if term.lower() in line.lower()
    ]

    print(f"\n{'=' * 60}\n{term}: {len(hits)} matches")

    for i in hits[:3]:
        start = max(0, i - 8)
        end = min(len(lines), i + 26)

        for j in range(start, end):
            print(f"{j + 1}: {lines[j][:220]}")
        print("---")

print("\nInspection only: no model weights changed.")


class LORMALinear: 0 matches

class LoRMALinear: 0 matches

class LORMA: 1 matches
5082: 
5083: import math
5084: import torch
5085: import torch.nn as nn
5086: 
5087: # LoRMA+-style multiplicative adapter:
5088: # y = xW + xW * (xA)B * scale
5089: # A starts at zero, so the initial adapted output matches the original layer.
5090: class LoRMAPlusLinear(nn.Module):
5091:     def __init__(self, base_layer, rank=8, alpha=16):
5092:         super().__init__()
5093:         self.base = base_layer
5094:         self.rank = rank
5095:         self.scale = alpha / rank
5096: 
5097:         in_features = base_layer.in_features
5098:         out_features = base_layer.out_features
5099: 
5100:         self.A = nn.Parameter(
5101:             torch.zeros(in_features, rank,
5102:                         device=base_layer.weight.device,
5103:                         dtype=base_layer.weight.dtype)
5104:         )
5105:         self.B = nn.Parameter(
5106:             torch.zeros(rank, out_features,


In [144]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

start = next(
    i for i, line in enumerate(lines)
    if "class LoRMAPlusLinear" in line
)

end = min(start + 65, len(lines))

print(f"Adapter implementation: lines {start + 1}-{end}\n")

for i in range(start, end):
    print(f"{i + 1}: {lines[i]}")

Adapter implementation: lines 5090-5154

5090: class LoRMAPlusLinear(nn.Module):
5091:     def __init__(self, base_layer, rank=8, alpha=16):
5092:         super().__init__()
5093:         self.base = base_layer
5094:         self.rank = rank
5095:         self.scale = alpha / rank
5096: 
5097:         in_features = base_layer.in_features
5098:         out_features = base_layer.out_features
5099: 
5100:         self.A = nn.Parameter(
5101:             torch.zeros(in_features, rank,
5102:                         device=base_layer.weight.device,
5103:                         dtype=base_layer.weight.dtype)
5104:         )
5105:         self.B = nn.Parameter(
5106:             torch.zeros(rank, out_features,
5107:                         device=base_layer.weight.device,
5108:                         dtype=base_layer.weight.dtype)
5109:         )
5110: 
5111:         # Freeze the original linear layer
5112:         for p in self.base.parameters():
5113:             p.requires_grad = False
51

In [145]:
import json
from pathlib import Path

path = Path("/kaggle/working/phase4_experiment_results.json")

if path.exists():
    results = json.loads(path.read_text())
    for item in results:
        print(item.get("experiment"), "| Macro-F1:", item.get("macro_f1"))
else:
    print("Experiment results file not found.")

print("\nSaved Phase 4 checkpoints:")
for p in Path("/kaggle/working").rglob("*.pt"):
    print(p, f"({p.stat().st_size / 1024**2:.1f} MB)")

A4 frozen encoder - standard loss | Macro-F1: 0.4487
A4 frozen encoder - class-weighted loss | Macro-F1: 0.5235
A4 BitFit | Macro-F1: 0.353

Saved Phase 4 checkpoints:
/kaggle/working/lorma_plus_a4/lorma_plus_a4_checkpoint.pt (270.5 MB)
/kaggle/working/phase4_results/a3_distillation_experiment.pt (197.9 MB)


In [146]:
import torch

for name in ["model", "a3_student", "classification_head",
             "weighted_student_head", "distill_student_head"]:
    obj = globals().get(name)
    print(f"{name}: {'available' if obj is not None else 'not available'}")

print("\nPhase 1 checkpoint files:")
from pathlib import Path

checkpoint_paths = [
    "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
]

for p in checkpoint_paths:
    path = Path(p)
    print(("FOUND" if path.exists() else "MISSING"), path.name)

model: available
a3_student: available
classification_head: available
weighted_student_head: available
distill_student_head: available

Phase 1 checkpoint files:
FOUND architecture1_sustained_step_500.pt
FOUND a2_step_9000.pt
FOUND step_0001000.pt
FOUND a4_step_22000.pt


In [147]:
import torch

checkpoints = {
    "A1": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
}

for name, path in checkpoints.items():
    print("\n" + "=" * 55)
    print(name, "checkpoint")
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    if isinstance(ckpt, dict):
        print("Top-level keys:", list(ckpt.keys())[:20])

        for key in ["architecture", "step", "context_length",
                    "hidden_size", "n_layers", "num_layers",
                    "tokens_processed", "train_loss"]:
            if key in ckpt:
                print(f"{key}: {ckpt[key]}")

        state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
        if isinstance(state, dict):
            tensors = [
                (k, tuple(v.shape))
                for k, v in state.items()
                if torch.is_tensor(v)
            ]
            print("Tensor entries:", len(tensors))
            for key, shape in tensors[:12]:
                print(f"  {key}: {shape}")
    else:
        print("Checkpoint type:", type(ckpt).__name__)


A1 checkpoint
Top-level keys: ['step', 'epoch', 'train_loss', 'val_loss', 'model_state_dict', 'optimizer_state_dict']
step: 500
train_loss: 5.366012096405029
Tensor entries: 101
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512)
  blocks.0.norm1.weight: (512,)
  blocks.0.norm1.bias: (512,)
  blocks.0.attn.in_proj_weight: (1536, 512)
  blocks.0.attn.in_proj_bias: (1536,)
  blocks.0.attn.out_proj.weight: (512, 512)
  blocks.0.attn.out_proj.bias: (512,)
  blocks.0.norm2.weight: (512,)
  blocks.0.norm2.bias: (512,)
  blocks.0.ffn.0.weight: (2048, 512)
  blocks.0.ffn.0.bias: (2048,)

A2 checkpoint
Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
step: 9000
tokens_processed: 73728000
train_loss: 6.801188290119171
Tensor entries: 113
  output_bias: (50000,)
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512)
  layers.0.norm1.weight: (512,)
  layers.0.norm1.bias: (512,)
  layers.0.atte

In [148]:
from pathlib import Path
import json

terms = [
    "class A1",
    "class A2",
    "class TransformerBlock",
    "class Transformer",
    "class Attention",
]

found = 0

for path in Path("/kaggle/input").rglob("*.ipynb"):
    try:
        nb = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue

    for idx, cell in enumerate(nb.get("cells", [])):
        src = "".join(cell.get("source", []))
        matches = [t for t in terms if t.lower() in src.lower()]
        if matches:
            found += 1
            print(f"\nFILE: {path.name} | CELL: {idx} | MATCHES: {matches}")
            print(src[:2200])

print("\nMatching cells:", found)


FILE: phase3-capstone.ipynb | CELL: 16 | MATCHES: ['class A2']

import json

NOTEBOOK_PATH = (
    "/kaggle/input/datasets/aq1de3/phase1-model/"
    "phase1_model_code_extract.ipynb"
)

with open(NOTEBOOK_PATH, encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    # Show architecture implementation cells, not training loops.
    markers = [
        "class PrefixLMBlock",
        "class PrefixLMModel",
        "class GQAAttention",
        "class A2Block",
        "class A2GQALM",
        "class EncoderBlock",
        "class DecoderBlock",
        "class EncoderDecoderLM",
        "class GPT2LMHeadModel",
    ]

    found = [m for m in markers if m in source]
    if found:
        print(f"\n{'=' * 70}")
        print(f"CELL {i} | {', '.join(found)}")
        print(source)


FILE: phase3-capstone.ipynb | CELL: 43 | MATCHES: ['class TransformerBlock',

In [149]:
import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/phase1-model/"
    "phase1_model_code_extract.ipynb"
)

print("File exists:", path.exists())

if path.exists():
    nb = json.loads(path.read_text(encoding="utf-8"))

    terms = [
        "build_architecture_1",
        "class TransformerBlock",
        "class Transformer",
        "class GPT",
        "class GQAAttention",
        "class A2GQAAttention",
    ]

    for term in terms:
        matches = []
        for i, cell in enumerate(nb.get("cells", [])):
            src = "".join(cell.get("source", []))
            if term.lower() in src.lower():
                matches.append((i, src))

        print(f"\n{term}: {[i for i, _ in matches]}")

        for i, src in matches[:2]:
            print(f"\n--- Cell {i} ---")
            print(src[:3500])

File exists: True

build_architecture_1: []

class TransformerBlock: []

class Transformer: []

class GPT: []

class GQAAttention: [4, 12]

--- Cell 4 ---
# ============================================================
# PHASE 1 — ARCHITECTURE 2
# GQA — UPDATED MODEL WIDTH
# PARAMETER + GPU CHECK
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

print("=" * 70)
print("ARCHITECTURE 2 — GQA UPDATED RESOURCE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

VOCAB_SIZE = Tokenizer.from_file(
    TOKENIZER_PATH
).get_vocab_size()

SEQ_LEN = 2048

D_MODEL = 528
N_LAYERS = 8
N_Q_HEADS = 8
N_KV_HEADS = 2
FFN_DIM = 2048

HEAD_DIM = D_MODEL // N_Q_HEADS
GROUP_SIZE = N_Q_HEADS // 

In [151]:
from pathlib import Path

roots = [
    Path("/kaggle/input/datasets/aq1de1"),
    Path("/kaggle/input/datasets/aq1de"),
    Path("/kaggle/working"),
]

matches = []

for root in roots:
    if root.exists():
        for p in root.rglob("a2_step_9000.pt"):
            matches.append(p)

print("A2 checkpoint matches:")
for p in matches:
    print(p)

if not matches:
    print("\nExact filename not found. Searching for other A2 checkpoint files...")
    for root in roots:
        if root.exists():
            for p in root.rglob("*a2*"):
                if p.is_file() and p.suffix in [".pt", ".pth"]:
                    print(p)

A2 checkpoint matches:

Exact filename not found. Searching for other A2 checkpoint files...


In [152]:
from pathlib import Path

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\nSearching {root} ...")
    if root.exists():
        count = 0
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() in (".pt", ".pth", ".ckpt"):
                print(f"{p}  ({p.stat().st_size / 1024**2:.1f} MB)")
                count += 1
        print("Checkpoint files found:", count)


Searching /kaggle/input ...
/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt  (197.9 MB)
/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt  (196.5 MB)
/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt  (367.8 MB)
/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt  (593.7 MB)
Checkpoint files found: 4

Searching /kaggle/working ...
/kaggle/working/lorma_plus_a4/lorma_plus_a4_checkpoint.pt  (270.5 MB)
/kaggle/working/phase4_results/a3_distillation_experiment.pt  (197.9 MB)
Checkpoint files found: 2


In [153]:
import torch

checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a2-step-9000-pt/a2_step_9000.pt"
)

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(ckpt))

if isinstance(ckpt, dict):
    print("\nTop-level keys:", list(ckpt.keys())[:30])

    state = ckpt.get(
        "model_state_dict",
        ckpt.get("state_dict", ckpt)
    )

    print("\nFirst 45 tensor names and shapes:")
    count = 0
    for name, value in state.items():
        if torch.is_tensor(value):
            print(name, tuple(value.shape))
            count += 1
            if count >= 45:
                break

Checkpoint type: <class 'dict'>

Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']

First 45 tensor names and shapes:
output_bias (50000,)
token_embedding.weight (50000, 512)
position_embedding.weight (2048, 512)
layers.0.norm1.weight (512,)
layers.0.norm1.bias (512,)
layers.0.attention.q_proj.weight (512, 512)
layers.0.attention.k_proj.weight (128, 512)
layers.0.attention.v_proj.weight (128, 512)
layers.0.attention.out_proj.weight (512, 512)
layers.0.norm2.weight (512,)
layers.0.norm2.bias (512,)
layers.0.ffn.fc1.weight (2048, 512)
layers.0.ffn.fc1.bias (2048,)
layers.0.ffn.fc2.weight (512, 2048)
layers.0.ffn.fc2.bias (512,)
layers.1.norm1.weight (512,)
layers.1.norm1.bias (512,)
layers.1.attention.q_proj.weight (512, 512)
layers.1.attention.k_proj.weight (128, 512)
layers.1.attention.v_proj.weight (128, 512)
layers.1.attention.out_proj.weight (512, 512)
layers.1.norm2.weight (512,)
layers.1.norm2.bias (512,)
layers.1.ffn.fc1.weight (

In [154]:
import json
from pathlib import Path

root = Path("/kaggle/input")
terms = [
    "a2_step_9000.pt",
    "model_state_dict",
    "class A2",
    "class GQAAttention",
]

for path in root.rglob("*.ipynb"):
    try:
        nb = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue

    hits = []
    for i, cell in enumerate(nb.get("cells", [])):
        src = "".join(cell.get("source", []))
        if any(term.lower() in src.lower() for term in terms):
            hits.append((i, src))

    if hits:
        print(f"\nNOTEBOOK: {path}")
        for i, src in hits[:4]:
            print(f"\n--- CELL {i} ---")
            print(src[:1800])


NOTEBOOK: /kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb

--- CELL 3 ---

from pathlib import Path
import torch

CHECKPOINTS = {
    "A1_GPT": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2_GQA": "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "A3_PrefixLM": "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "A4_EncoderDecoder": "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
}

def inspect_value(value, prefix="", depth=0):
    if isinstance(value, dict):
        print(f"{prefix}dict with {len(value)} keys")
        if depth < 2:
            for i, (key, item) in enumerate(value.items()):
                if i >= 30:
                    print(prefix, "... additional keys omitted")
                    break
                inspect_value(item, f"{prefix}{key}: ", depth + 1)
    elif isinstance(value, torch.Tensor):
        print(f"{prefix}Tensor shape=

In [155]:
import torch.nn as nn

print("Existing A2-related objects:")
for name in [
    "A2GQALM",
    "A2Block",
    "GQAAttention",
    "a2_model",
    "model_a2",
    "a2_classifier",
]:
    obj = globals().get(name)
    if obj is not None:
        print(f"{name}: {type(obj)}")

print("\nExisting model objects:")
for name in ["model", "a3_student", "classification_head"]:
    obj = globals().get(name)
    if isinstance(obj, nn.Module):
        print(f"\n{name}: {type(obj).__name__}")
        print("Parameter count:", sum(p.numel() for p in obj.parameters()))

Existing A2-related objects:

Existing model objects:

model: A4EncoderDecoder
Parameter count: 70887424

a3_student: A3PrefixLM
Parameter count: 51868672

classification_head: Linear
Parameter count: 1026


In [156]:
import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/p1p1p1/"
    "a3-phase1 (1).ipynb"
)

nb = json.loads(path.read_text(encoding="utf-8"))

for i in [3, 5]:
    cell = nb["cells"][i]
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    print(f"\n{'=' * 25} CELL {i} {'=' * 25}")
    print(source[:12000])


========================= CELL 3 =========================
# ============================================================
# A2 — VERIFIED GQA DECODER-ONLY MODEL
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class GQAAttention(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_q_heads,
        num_kv_heads,
        dropout=0.0
    ):
        super().__init__()

        assert hidden_size % num_q_heads == 0
        assert num_q_heads % num_kv_heads == 0

        self.hidden_size = hidden_size
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = hidden_size // num_q_heads

        kv_dim = num_kv_heads * self.head_dim

        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=True)
        self.k_proj = nn.Linear(hidden_size, kv_dim, bias=True)
        self.v_proj = nn.Linear(hidden_size, kv_dim, bias=True)
  

In [157]:
import torch

path = (
    "/kaggle/input/datasets/aq1de3/"
    "a2-step-9000-pt/a2_step_9000.pt"
)

ckpt = torch.load(path, map_location="cpu", weights_only=True)
state = ckpt["model_state_dict"]

print("Checkpoint metadata:")
for key in ["step", "tokens_processed", "train_loss"]:
    print(f"{key}: {ckpt.get(key)}")

print("\nAll top-level parameter prefixes:")
print(sorted(set(k.split(".")[0] for k in state)))

print("\nFinal-layer and output parameters:")
for key, value in state.items():
    if key.startswith("layers.7") or any(
        term in key for term in [
            "final", "norm", "output_bias", "lm_bias"
        ]
    ):
        print(key, tuple(value.shape))

Checkpoint metadata:
step: 9000
tokens_processed: 73728000
train_loss: 6.801188290119171

All top-level parameter prefixes:
['final_norm', 'layers', 'output_bias', 'position_embedding', 'token_embedding']

Final-layer and output parameters:
output_bias (50000,)
layers.0.norm1.weight (512,)
layers.0.norm1.bias (512,)
layers.0.norm2.weight (512,)
layers.0.norm2.bias (512,)
layers.1.norm1.weight (512,)
layers.1.norm1.bias (512,)
layers.1.norm2.weight (512,)
layers.1.norm2.bias (512,)
layers.2.norm1.weight (512,)
layers.2.norm1.bias (512,)
layers.2.norm2.weight (512,)
layers.2.norm2.bias (512,)
layers.3.norm1.weight (512,)
layers.3.norm1.bias (512,)
layers.3.norm2.weight (512,)
layers.3.norm2.bias (512,)
layers.4.norm1.weight (512,)
layers.4.norm1.bias (512,)
layers.4.norm2.weight (512,)
layers.4.norm2.bias (512,)
layers.5.norm1.weight (512,)
layers.5.norm1.bias (512,)
layers.5.norm2.weight (512,)
layers.5.norm2.bias (512,)
layers.6.norm1.weight (512,)
layers.6.norm1.bias (512,)
layers.6.n

In [158]:
import torch

path = (
    "/kaggle/input/datasets/aq1de3/"
    "a2-step-9000-pt/a2_step_9000.pt"
)

ckpt = torch.load(path, map_location="cpu", weights_only=True)
state = ckpt["model_state_dict"]

for layer_id in [0, 8]:
    prefix = f"layers.{layer_id}."
    print(f"\n--- {prefix} ---")
    for key, value in state.items():
        if key.startswith(prefix):
            print(f"{key}: {tuple(value.shape)}")

print("\nTotal layers:", len({
    key.split(".")[1]
    for key in state
    if key.startswith("layers.")
}))


--- layers.0. ---
layers.0.norm1.weight: (512,)
layers.0.norm1.bias: (512,)
layers.0.attention.q_proj.weight: (512, 512)
layers.0.attention.k_proj.weight: (128, 512)
layers.0.attention.v_proj.weight: (128, 512)
layers.0.attention.out_proj.weight: (512, 512)
layers.0.norm2.weight: (512,)
layers.0.norm2.bias: (512,)
layers.0.ffn.fc1.weight: (2048, 512)
layers.0.ffn.fc1.bias: (2048,)
layers.0.ffn.fc2.weight: (512, 2048)
layers.0.ffn.fc2.bias: (512,)

--- layers.8. ---
layers.8.norm1.weight: (512,)
layers.8.norm1.bias: (512,)
layers.8.attention.q_proj.weight: (512, 512)
layers.8.attention.k_proj.weight: (128, 512)
layers.8.attention.v_proj.weight: (128, 512)
layers.8.attention.out_proj.weight: (512, 512)
layers.8.norm2.weight: (512,)
layers.8.norm2.bias: (512,)
layers.8.ffn.fc1.weight: (2048, 512)
layers.8.ffn.fc1.bias: (2048,)
layers.8.ffn.fc2.weight: (512, 2048)
layers.8.ffn.fc2.bias: (512,)

Total layers: 9


In [159]:
import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/phase1-model/"
    "phase1_model_code_extract.ipynb"
)

nb = json.loads(path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"]):
    src = cell.get("source", "")
    if isinstance(src, list):
        src = "".join(src)

    if (
        "class GQAAttention" in src
        and "class A2Block" not in src
    ):
        print(f"\n--- Cell {i}: GQAAttention ---")
        print(src[:7000])


--- Cell 4: GQAAttention ---
# ============================================================
# PHASE 1 — ARCHITECTURE 2
# GQA — UPDATED MODEL WIDTH
# PARAMETER + GPU CHECK
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

print("=" * 70)
print("ARCHITECTURE 2 — GQA UPDATED RESOURCE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

VOCAB_SIZE = Tokenizer.from_file(
    TOKENIZER_PATH
).get_vocab_size()

SEQ_LEN = 2048

D_MODEL = 528
N_LAYERS = 8
N_Q_HEADS = 8
N_KV_HEADS = 2
FFN_DIM = 2048

HEAD_DIM = D_MODEL // N_Q_HEADS
GROUP_SIZE = N_Q_HEADS // N_KV_HEADS

print(f"\nVocabulary       : {VOCAB_SIZE:,}")
print(f"Context length   : {SEQ_LEN:,}")
print(f"Model dimension  :

In [160]:
import torch

path = (
    "/kaggle/input/datasets/aq1de3/"
    "a2-step-9000-pt/a2_step_9000.pt"
)

ckpt = torch.load(path, map_location="cpu", weights_only=True)
state = ckpt["model_state_dict"]

for prefix in [
    "layers.0.attention.",
    "layers.0.ffn.",
    "final_norm.",
]:
    print(f"\n{prefix}")
    for key, value in state.items():
        if key.startswith(prefix):
            print(f"{key}: {tuple(value.shape)}")

print("\nOutput bias exists:", "output_bias" in state)
print("Output bias shape:", tuple(state["output_bias"].shape))


layers.0.attention.
layers.0.attention.q_proj.weight: (512, 512)
layers.0.attention.k_proj.weight: (128, 512)
layers.0.attention.v_proj.weight: (128, 512)
layers.0.attention.out_proj.weight: (512, 512)

layers.0.ffn.
layers.0.ffn.fc1.weight: (2048, 512)
layers.0.ffn.fc1.bias: (2048,)
layers.0.ffn.fc2.weight: (512, 2048)
layers.0.ffn.fc2.bias: (512,)

final_norm.
final_norm.weight: (512,)
final_norm.bias: (512,)

Output bias exists: True
Output bias shape: (50000,)


In [161]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Phase4A2Attention(nn.Module):
    def __init__(self, hidden=512, q_heads=8, kv_heads=2):
        super().__init__()
        head_dim = hidden // q_heads
        kv_dim = kv_heads * head_dim

        self.q_proj = nn.Linear(hidden, hidden, bias=False)
        self.k_proj = nn.Linear(hidden, kv_dim, bias=False)
        self.v_proj = nn.Linear(hidden, kv_dim, bias=False)
        self.out_proj = nn.Linear(hidden, hidden, bias=False)
        self.q_heads = q_heads
        self.kv_heads = kv_heads
        self.head_dim = head_dim

    def forward(self, x):
        b, t, c = x.shape
        q = self.q_proj(x).view(
            b, t, self.q_heads, self.head_dim
        ).transpose(1, 2)
        k = self.k_proj(x).view(
            b, t, self.kv_heads, self.head_dim
        ).transpose(1, 2)
        v = self.v_proj(x).view(
            b, t, self.kv_heads, self.head_dim
        ).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, enable_gqa=True
        )
        y = y.transpose(1, 2).contiguous().view(b, t, c)
        return self.out_proj(y)


class Phase4A2FFN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(512, 2048)
        self.fc2 = nn.Linear(2048, 512)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x), approximate="tanh"))


class Phase4A2Layer(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(512)
        self.attention = Phase4A2Attention()
        self.norm2 = nn.LayerNorm(512)
        self.ffn = Phase4A2FFN()

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class Phase4A2Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(50000, 512)
        self.position_embedding = nn.Embedding(2048, 512)
        self.layers = nn.ModuleList([Phase4A2Layer() for _ in range(9)])
        self.final_norm = nn.LayerNorm(512)
        self.output_bias = nn.Parameter(torch.zeros(50000))

    def encode(self, input_ids):
        b, t = input_ids.shape
        positions = torch.arange(t, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        for layer in self.layers:
            x = layer(x)
        return self.final_norm(x)

    def forward(self, input_ids):
        x = self.encode(input_ids)
        return F.linear(x, self.token_embedding.weight, self.output_bias)


a2_checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a2-step-9000-pt/a2_step_9000.pt"
)
a2_checkpoint = torch.load(
    a2_checkpoint_path, map_location="cpu", weights_only=True
)

a2_model = Phase4A2Model()
result = a2_model.load_state_dict(
    a2_checkpoint["model_state_dict"], strict=True
)

print("Strict checkpoint load:", result)
print("A2 parameters:", sum(p.numel() for p in a2_model.parameters()))
print("Checkpoint step:", a2_checkpoint["step"])

Strict checkpoint load: <All keys matched successfully>
A2 parameters: 51513680
Checkpoint step: 9000


In [162]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
a2_model = a2_model.to(device).eval()

test_ids = torch.tensor(
    [[2, 100, 250, 300, 3, 0, 0, 0]],
    dtype=torch.long,
    device=device,
)

with torch.inference_mode():
    logits = a2_model(test_ids)
    features = a2_model.encode(test_ids)

print("Device:", device)
print("Logits shape:", tuple(logits.shape))
print("Features shape:", tuple(features.shape))
print("All logits finite:", torch.isfinite(logits).all().item())
print("All features finite:", torch.isfinite(features).all().item())

Device: cuda
Logits shape: (1, 8, 50000)
Features shape: (1, 8, 512)
All logits finite: True
All features finite: True


In [163]:
import inspect

names = [
    "prepare_batch",
    "evaluate_model",
    "evaluate_classifier",
    "get_case_features",
    "train_classifier",
    "classification_head",
    "weighted_student_head",
    "dev",
    "train",
]

for name in names:
    obj = globals().get(name)

    if obj is None:
        print(f"{name}: NOT FOUND")
    elif callable(obj) and not isinstance(obj, torch.nn.Module):
        try:
            print(f"\n{name}{inspect.signature(obj)}")
            if inspect.isfunction(obj):
                print(inspect.getsource(obj)[:1800])
        except Exception:
            print(f"{name}: callable ({type(obj).__name__})")
    else:
        print(f"{name}: {type(obj).__name__}")


prepare_batch(records, max_length=1024)
def prepare_batch(records, max_length=1024):
    pad_id = tokenizer.token_to_id("[PAD]")
    bos_id = tokenizer.token_to_id("[BOS]")
    eos_id = tokenizer.token_to_id("[EOS]")

    all_ids = []
    all_labels = []

    for row in records:
        ids = tokenizer.encode(row["text"]).ids
        ids = ids[:max_length - 2]
        ids = [bos_id] + ids + [eos_id]

        # Fixed-length padding
        ids += [pad_id] * (max_length - len(ids))

        all_ids.append(ids)
        all_labels.append(int(row["label"]))

    input_ids = torch.tensor(all_ids, dtype=torch.long, device=device)
    labels = torch.tensor(all_labels, dtype=torch.long, device=device)
    attention_mask = (input_ids != pad_id).long()

    return input_ids, attention_mask, labels

evaluate_model: NOT FOUND
evaluate_classifier: NOT FOUND
get_case_features: NOT FOUND
train_classifier: NOT FOUND
classification_head: Linear
weighted_student_head: Linear
dev: list
train: list


In [164]:
import torch

# Use a small batch from the existing development set.
sample_records = dev[:4]

input_ids, attention_mask, labels = prepare_batch(
    sample_records,
    max_length=1024
)

a2_model.eval()

with torch.inference_mode():
    features = a2_model.encode(input_ids)

    # Mean-pool only non-padding positions.
    mask = attention_mask.unsqueeze(-1).to(features.dtype)
    pooled_features = (
        (features * mask).sum(dim=1)
        / mask.sum(dim=1).clamp(min=1)
    )

print("Input IDs:", tuple(input_ids.shape))
print("Attention mask:", tuple(attention_mask.shape))
print("Labels:", tuple(labels.shape))
print("Token features:", tuple(features.shape))
print("Pooled features:", tuple(pooled_features.shape))
print("All features finite:", torch.isfinite(pooled_features).all().item())
print("Labels:", labels.tolist())

Input IDs: (4, 1024)
Attention mask: (4, 1024)
Labels: (4,)
Token features: (4, 1024, 512)
Pooled features: (4, 512)
All features finite: True
Labels: [0, 0, 1, 0]


In [165]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score

# Freeze the pretrained A2 backbone
a2_model.eval()
for param in a2_model.parameters():
    param.requires_grad = False

def extract_a2_features(records, batch_size=4):
    all_features, all_labels = [], []

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        input_ids, attention_mask, labels = prepare_batch(
            batch, max_length=1024
        )

        with torch.inference_mode():
            token_features = a2_model.encode(input_ids)
            mask = attention_mask.unsqueeze(-1).to(token_features.dtype)
            pooled = (
                (token_features * mask).sum(dim=1)
                / mask.sum(dim=1).clamp(min=1)
            )

        all_features.append(pooled.cpu())
        all_labels.append(labels.cpu())

    return torch.cat(all_features), torch.cat(all_labels)

# Small pilot subset
pilot_train = train[:256]
pilot_dev = dev[:256]

X_train_a2, y_train_a2 = extract_a2_features(pilot_train)
X_dev_a2, y_dev_a2 = extract_a2_features(pilot_dev)

a2_head = nn.Linear(512, 2).to(device)
optimizer = torch.optim.AdamW(
    a2_head.parameters(), lr=1e-3, weight_decay=0.01
)
criterion = nn.CrossEntropyLoss()

dataset = TensorDataset(X_train_a2, y_train_a2)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

a2_head.train()
for epoch in range(3):
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = a2_head(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}: loss={total_loss / len(loader):.4f}")

a2_head.eval()
with torch.inference_mode():
    dev_logits = a2_head(X_dev_a2.to(device))
    dev_preds = dev_logits.argmax(dim=1).cpu().numpy()

y_true = y_dev_a2.numpy()

print("Pilot dev accuracy:", round(accuracy_score(y_true, dev_preds), 4))
print("Pilot dev macro-F1:", round(f1_score(y_true, dev_preds, average="macro"), 4))
print("Trainable head parameters:", sum(p.numel() for p in a2_head.parameters() if p.requires_grad))

Epoch 1: loss=0.7284
Epoch 2: loss=0.7215
Epoch 3: loss=0.7047
Pilot dev accuracy: 0.5
Pilot dev macro-F1: 0.3333
Trainable head parameters: 1026


In [166]:
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report

print("True label counts:", Counter(y_true))
print("Predicted label counts:", Counter(dev_preds))
print("\nConfusion matrix:")
print(confusion_matrix(y_true, dev_preds, labels=[0, 1]))
print("\nClassification report:")
print(classification_report(
    y_true, dev_preds,
    labels=[0, 1],
    target_names=["Class 0", "Class 1"],
    zero_division=0
))

with torch.inference_mode():
    probabilities = torch.softmax(dev_logits, dim=1).cpu()

print("Mean predicted probabilities:", probabilities.mean(dim=0).tolist())
print("Probability standard deviation:", probabilities.std(dim=0).tolist())

True label counts: Counter({np.int64(0): 128, np.int64(1): 128})
Predicted label counts: Counter({np.int64(0): 256})

Confusion matrix:
[[128   0]
 [128   0]]

Classification report:
              precision    recall  f1-score   support

     Class 0       0.50      1.00      0.67       128
     Class 1       0.00      0.00      0.00       128

    accuracy                           0.50       256
   macro avg       0.25      0.50      0.33       256
weighted avg       0.25      0.50      0.33       256

Mean predicted probabilities: [0.5832642912864685, 0.4167357087135315]
Probability standard deviation: [0.0010002563940361142, 0.0010002583730965853]


In [167]:
print("Training feature shape:", tuple(X_train_a2.shape))
print("Feature mean:", X_train_a2.mean().item())
print("Feature standard deviation:", X_train_a2.std().item())
print("Mean feature norm:", X_train_a2.norm(dim=1).mean().item())

for label in [0, 1]:
    subset = X_train_a2[y_train_a2 == label]
    print(
        f"Class {label}: n={len(subset)}, "
        f"mean norm={subset.norm(dim=1).mean().item():.4f}, "
        f"feature std={subset.std().item():.4f}"
    )

Training feature shape: (256, 512)
Feature mean: 0.00042246514931321144
Feature standard deviation: 0.6577618718147278
Mean feature norm: 14.883398056030273
Class 0: n=140, mean norm=14.8836, feature std=0.6578
Class 1: n=116, mean norm=14.8832, feature std=0.6578


In [168]:
import torch.nn.functional as F

X = X_train_a2.float()
y = y_train_a2

mean0 = X[y == 0].mean(dim=0)
mean1 = X[y == 1].mean(dim=0)

cosine_similarity = F.cosine_similarity(
    mean0.unsqueeze(0),
    mean1.unsqueeze(0)
).item()

mean_difference = (mean0 - mean1).norm().item()

print("Class 0 centroid norm:", round(mean0.norm().item(), 4))
print("Class 1 centroid norm:", round(mean1.norm().item(), 4))
print("Centroid cosine similarity:", round(cosine_similarity, 6))
print("Distance between class centroids:", round(mean_difference, 6))

Class 0 centroid norm: 14.8831
Class 1 centroid norm: 14.8827
Centroid cosine similarity: 0.999998
Distance between class centroids: 0.028004


In [169]:
def extract_a2_eos_features(records, batch_size=4):
    all_features, all_labels = [], []

    eos_id = tokenizer.token_to_id("[EOS]")

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        input_ids, attention_mask, labels = prepare_batch(
            batch, max_length=1024
        )

        with torch.inference_mode():
            token_features = a2_model.encode(input_ids)

            # EOS immediately follows the final retained content token.
            eos_positions = (input_ids == eos_id).int().argmax(dim=1)
            rows = torch.arange(input_ids.size(0), device=input_ids.device)
            pooled = token_features[rows, eos_positions]

        all_features.append(pooled.cpu())
        all_labels.append(labels.cpu())

    return torch.cat(all_features), torch.cat(all_labels)

X_train_eos, y_train_eos = extract_a2_eos_features(train[:256])
X_dev_eos, y_dev_eos = extract_a2_eos_features(dev[:256])

print("EOS feature shape:", tuple(X_train_eos.shape))
print("Feature standard deviation:", X_train_eos.std().item())

for label in [0, 1]:
    centroid = X_train_eos[y_train_eos == label].mean(dim=0)
    print(f"Class {label} centroid norm: {centroid.norm().item():.4f}")

m0 = X_train_eos[y_train_eos == 0].mean(dim=0)
m1 = X_train_eos[y_train_eos == 1].mean(dim=0)

print(
    "EOS centroid cosine similarity:",
    torch.nn.functional.cosine_similarity(m0[None], m1[None]).item()
)
print("EOS centroid distance:", (m0 - m1).norm().item())

EOS feature shape: (256, 512)
Feature standard deviation: 0.6592493653297424
Class 0 centroid norm: 14.9155
Class 1 centroid norm: 14.9137
EOS centroid cosine similarity: 0.9999961853027344
EOS centroid distance: 0.04124454781413078


In [170]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

a2_eos_head = nn.Linear(512, 2).to(device)

counts = torch.bincount(y_train_eos, minlength=2).float()
class_weights = (counts.sum() / (2 * counts)).to(device)

optimizer = torch.optim.AdamW(
    a2_eos_head.parameters(), lr=1e-3, weight_decay=0.01
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

loader = DataLoader(
    TensorDataset(X_train_eos, y_train_eos),
    batch_size=32,
    shuffle=True
)

for epoch in range(5):
    a2_eos_head.train()
    losses = []

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(a2_eos_head(xb), yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    print(f"Epoch {epoch + 1}: loss={sum(losses)/len(losses):.4f}")

a2_eos_head.eval()
with torch.inference_mode():
    logits = a2_eos_head(X_dev_eos.to(device))
    predictions = logits.argmax(dim=1).cpu().numpy()

truth = y_dev_eos.numpy()

print("Accuracy:", round(accuracy_score(truth, predictions), 4))
print("Macro-F1:", round(f1_score(truth, predictions, average="macro"), 4))
print("Predicted classes:", torch.bincount(torch.tensor(predictions), minlength=2).tolist())
print("Confusion matrix:\n", confusion_matrix(truth, predictions, labels=[0, 1]))

Epoch 1: loss=0.7403
Epoch 2: loss=0.7087
Epoch 3: loss=0.6987
Epoch 4: loss=0.7061
Epoch 5: loss=0.6907
Accuracy: 0.5
Macro-F1: 0.3333
Predicted classes: [256, 0]
Confusion matrix:
 [[128   0]
 [128   0]]


In [171]:
import json
from pathlib import Path

paths = [
    Path("/kaggle/working/phase4_experiment_results.json"),
    Path("/kaggle/working/a4_adapter_pilot_results.json"),
    Path("/kaggle/working/lorma_plus_a4/results.json"),
    Path("/kaggle/working/phase4_results/a3_distillation_results.json"),
]

for path in paths:
    print(f"\n--- {path} ---")
    if path.exists():
        with open(path, "r") as f:
            data = json.load(f)
        print(json.dumps(data, indent=2)[:5000])
    else:
        print("NOT FOUND")


--- /kaggle/working/phase4_experiment_results.json ---
[
  {
    "experiment": "A4 frozen encoder - standard loss",
    "accuracy": 0.5258,
    "macro_f1": 0.4487,
    "macro_precision": 0.5607,
    "macro_recall": 0.5266,
    "trainable_parameters": 1026
  },
  {
    "experiment": "A4 frozen encoder - class-weighted loss",
    "accuracy": 0.5327,
    "macro_f1": 0.5235,
    "trainable_parameters": 1026
  },
  {
    "experiment": "A4 BitFit",
    "accuracy": 0.499,
    "macro_f1": 0.353,
    "macro_precision": 0.4995,
    "macro_recall": 0.5,
    "trainable_parameters": 84994,
    "training_loss": 0.6891,
    "training_accuracy": 0.5849
  }
]

--- /kaggle/working/a4_adapter_pilot_results.json ---
{
  "experiment": "A4 multiplicative low-rank adapters pilot",
  "checkpoint": "A4 step 22000",
  "training_examples": 2048,
  "epochs": 1,
  "batch_size": 8,
  "learning_rate": 5e-05,
  "trainable_parameters": 99330,
  "development_examples": 3505,
  "development_macro_f1_threshold_0.50": 0.

In [173]:
from pathlib import Path

p = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")

print("Exists:", p.exists())

if p.exists():
    print("File size:", p.stat().st_size, "bytes")
    print("First 300 characters:")
    print(repr(p.read_text(errors="replace")[:300]))

Exists: True
File size: 185025 bytes
First 300 characters:
'\nfrom pathlib import Path\nimport json\n\n# Find the dataset files mounted by Kaggle\nmatches = list(Path("/kaggle/input").rglob("label_vocab.json"))\n\nif not matches:\n    print("Dataset not found. Check the Kaggle Input panel.")\nelse:\n    phase3_root = matches[0].parent\n    print("Dataset directory:", p'


In [174]:
from pathlib import Path

p = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
source = p.read_text(errors="replace")

keywords = [
    "LORMA",
    "LoRMA",
    "AdaLoRA",
    "Adapter",
    "BitFit",
    "freeze",
    "unfreeze",
    "distill",
    "partial fine-tuning",
    "layer-wise",
]

for keyword in keywords:
    lines = [
        (i, line.strip()[:180])
        for i, line in enumerate(source.splitlines(), start=1)
        if keyword.lower() in line.lower()
    ]
    print(f"\n{keyword}: {len(lines)} matches")
    for line_no, text in lines[:5]:
        print(f"  Line {line_no}: {text}")


LORMA: 42 matches
  Line 801: "GQA", "LORMA", "encoder_layers", "decoder_layers",
  Line 5087: # LoRMA+-style multiplicative adapter:
  Line 5090: class LoRMAPlusLinear(nn.Module):
  Line 5127: adapted = LoRMAPlusLinear(original, rank=8, alpha=16)
  Line 5819: class PaperLoRMAPlusLinear(nn.Module):

LoRMA: 42 matches
  Line 801: "GQA", "LORMA", "encoder_layers", "decoder_layers",
  Line 5087: # LoRMA+-style multiplicative adapter:
  Line 5090: class LoRMAPlusLinear(nn.Module):
  Line 5127: adapted = LoRMAPlusLinear(original, rank=8, alpha=16)
  Line 5819: class PaperLoRMAPlusLinear(nn.Module):

AdaLoRA: 0 matches

Adapter: 62 matches
  Line 158: "adapter_config.json",
  Line 159: "adapter_model.safetensors",
  Line 5087: # LoRMA+-style multiplicative adapter:
  Line 5121: # Attach adapters to q_proj and v_proj in every encoder block.
  Line 5122: adapter_names = []

BitFit: 44 matches
  Line 4757: # BitFit: train bias parameters only
  Line 4764: bitfit_head = nn.Linear(512, 2).to(dev

In [175]:
from pathlib import Path

p = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = p.read_text(errors="replace").splitlines()

# Show the PaperLoRMAPlusLinear definition and nearby code.
start, end = 5810, 5885

for i in range(start, min(end, len(lines)) + 1):
    print(f"{i}: {lines[i - 1]}")

5810:     wrapped_out = adapter(x)
5811:     print("Max absolute output difference:",
5812:           (base_out - wrapped_out).abs().max().item())
5813: 
5814: 
5815: import torch
5816: import torch.nn as nn
5817: import torch.nn.functional as F
5818: 
5819: class PaperLoRMAPlusLinear(nn.Module):
5820:     """
5821:     LoRMA+ weight transformation:
5822:         W_eff = (I + (alpha/r) * B @ A) @ W0
5823: 
5824:     B: (out_features, rank)
5825:     A: (rank, out_features)
5826:     """
5827:     def __init__(self, base_layer, rank=8, alpha=16):
5828:         super().__init__()
5829:         self.base = base_layer
5830:         self.rank = rank
5831:         self.scale = alpha / rank
5832: 
5833:         out_features, in_features = base_layer.weight.shape
5834:         if out_features != in_features:
5835:             raise ValueError(
5836:                 "This identity-inflated implementation requires a square weight matrix."
5837:             )
5838: 
5839:         self.A = nn.Para

In [176]:
from pathlib import Path

roots = [Path("/kaggle/input"), Path("/kaggle/working")]

for root in roots:
    print(f"\n--- Searching {root} ---")
    count = 0
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".pdf", ".docx", ".txt"}:
            name = p.name.lower()
            if any(word in name for word in ["phase4", "assignment", "capstone", "instruction", "cs781"]):
                print(p)
                count += 1
                if count >= 30:
                    break


--- Searching /kaggle/input ---

--- Searching /kaggle/working ---


In [178]:

import inspect
import torch

names = [
    "prepare_batch",
    "evaluate_model",
    "evaluate_classifier",
    "get_case_features",
    "train_classifier",
    "classification_head",
    "weighted_student_head",
    "distill_student_head",
    "dev",
    "train",
]

for name in names:
    obj = globals().get(name)

    if obj is None:
        print(f"\n{name}: NOT FOUND")
    elif isinstance(obj, torch.nn.Module):
        print(f"\n{name}: {type(obj).__name__}")
    elif callable(obj):
        try:
            print(f"\n{name}{inspect.signature(obj)}")
            if inspect.isfunction(obj):
                print(inspect.getsource(obj)[:1800])
        except Exception as e:
            print(f"Inspection unavailable: {e}")
    else:
        print(f"\n{name}: {type(obj).__name__}")

import inspect
import torch

names = [
    "prepare_batch",
    "evaluate_model",
    "evaluate_classifier",
    "get_case_features",
    "train_classifier",
    "classification_head",
    "weighted_student_head",
    "distill_student_head",
    "dev",
    "train",
]

for name in names:
    obj = globals().get(name)

    if obj is None:
        print(f"\n{name}: NOT FOUND")
    elif isinstance(obj, torch.nn.Module):
        print(f"\n{name}: {type(obj).__name__}")
    elif callable(obj):
        try:
            print(f"\n{name}{inspect.signature(obj)}")
            if inspect.isfunction(obj):
                print(inspect.getsource(obj)[:1800])
        except Exception as e:
            print(f"Inspection unavailable: {e}")
    else:
        print(f"\n{name}: {type(obj).__name__}")



prepare_batch(records, max_length=1024)
def prepare_batch(records, max_length=1024):
    pad_id = tokenizer.token_to_id("[PAD]")
    bos_id = tokenizer.token_to_id("[BOS]")
    eos_id = tokenizer.token_to_id("[EOS]")

    all_ids = []
    all_labels = []

    for row in records:
        ids = tokenizer.encode(row["text"]).ids
        ids = ids[:max_length - 2]
        ids = [bos_id] + ids + [eos_id]

        # Fixed-length padding
        ids += [pad_id] * (max_length - len(ids))

        all_ids.append(ids)
        all_labels.append(int(row["label"]))

    input_ids = torch.tensor(all_ids, dtype=torch.long, device=device)
    labels = torch.tensor(all_labels, dtype=torch.long, device=device)
    attention_mask = (input_ids != pad_id).long()

    return input_ids, attention_mask, labels


evaluate_model: NOT FOUND

evaluate_classifier: NOT FOUND

get_case_features: NOT FOUND

train_classifier: NOT FOUND

classification_head: Linear

weighted_student_head: Linear

distill_student_head:

In [179]:
import torch

model.eval()

sample_records = dev[:2]
input_ids, attention_mask, labels = prepare_batch(sample_records)

with torch.no_grad():
    positions = torch.arange(input_ids.size(1), device=device)

    x = model.token_embedding(input_ids)
    x = x + model.encoder_pos(positions)[None, :, :]
    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    encoder_output = model.encoder_norm(x)

    # Index of the final non-padding token for each case
    last_indices = attention_mask.sum(dim=1) - 1

    batch_indices = torch.arange(input_ids.size(0), device=device)
    case_features = encoder_output[batch_indices, last_indices]

print("Input shape:", tuple(input_ids.shape))
print("Encoder output shape:", tuple(encoder_output.shape))
print("Case features shape:", tuple(case_features.shape))
print("Labels:", labels.tolist())
print("Features finite:", torch.isfinite(case_features).all().item())
print("Feature standard deviation:", case_features.std().item())

Input shape: (2, 1024)
Encoder output shape: (2, 1024, 512)
Case features shape: (2, 512)
Labels: [0, 0]
Features finite: True
Feature standard deviation: 0.9604300856590271


In [180]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
import numpy as np
import torch

def evaluate_predictions(labels, probabilities, threshold=0.5):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= threshold).astype(int)

    results = {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(
            f1_score(labels, predictions, average="macro", zero_division=0)
        ),
        "macro_precision": float(
            precision_score(labels, predictions, average="macro", zero_division=0)
        ),
        "macro_recall": float(
            recall_score(labels, predictions, average="macro", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(
            labels, predictions, labels=[0, 1]
        ).tolist(),
        "prediction_counts": {
            "0": int((predictions == 0).sum()),
            "1": int((predictions == 1).sum()),
        },
    }
    return results

print("Evaluation function created.")
print("Test:", evaluate_predictions(
    [0, 0, 1, 1],
    [0.2, 0.7, 0.4, 0.8],
))

Evaluation function created.
Test: {'threshold': 0.5, 'accuracy': 0.5, 'macro_f1': 0.5, 'macro_precision': 0.5, 'macro_recall': 0.5, 'confusion_matrix': [[1, 1], [1, 1]], 'prediction_counts': {'0': 2, '1': 2}}


In [181]:
import torch
import numpy as np
from tqdm.auto import tqdm

def extract_a4_features(records, batch_size=4, max_length=1024):
    model.eval()
    feature_batches = []
    label_batches = []

    for start in tqdm(range(0, len(records), batch_size)):
        batch = records[start:start + batch_size]
        input_ids, attention_mask, labels = prepare_batch(
            batch, max_length=max_length
        )

        with torch.no_grad():
            positions = torch.arange(
                input_ids.size(1), device=input_ids.device
            )
            x = model.token_embedding(input_ids)
            x = x + model.encoder_pos(positions)[None, :, :]
            x = model.dropout(x)

            for layer in model.encoder_layers:
                x = layer(x)

            x = model.encoder_norm(x)

            # Final non-padding token (normally EOS)
            last_indices = attention_mask.sum(dim=1) - 1
            batch_indices = torch.arange(
                input_ids.size(0), device=input_ids.device
            )
            features = x[batch_indices, last_indices]

        feature_batches.append(features.cpu())
        label_batches.append(labels.cpu())

    return (
        torch.cat(feature_batches),
        torch.cat(label_batches).numpy()
    )

# Extract development features
dev_features, dev_labels = extract_a4_features(dev)

# Evaluate the existing head without training it
classification_head.eval()
with torch.no_grad():
    dev_logits = classification_head(dev_features.to(device))
    dev_probabilities = torch.softmax(dev_logits, dim=1)[:, 1].cpu().numpy()

baseline_results = evaluate_predictions(
    dev_labels, dev_probabilities, threshold=0.5
)

print("A4 frozen-encoder baseline:")
print(baseline_results)
print("Feature matrix:", tuple(dev_features.shape))

  0%|          | 0/877 [00:00<?, ?it/s]

A4 frozen-encoder baseline:
{'threshold': 0.5, 'accuracy': 0.4972895863052782, 'macro_f1': 0.3356189347543952, 'macro_precision': 0.4240674318507891, 'macro_recall': 0.4982767569102292, 'confusion_matrix': [[1736, 13], [1749, 7]], 'prediction_counts': {'0': 3485, '1': 20}}
Feature matrix: (3505, 512)


In [182]:
import json
import os
import torch

print("=== Current classification head ===")
print("Head:", classification_head)
print("Weight shape:", tuple(classification_head.weight.shape))
print("Bias:", classification_head.bias.detach().cpu().tolist())
print("Weight std:", classification_head.weight.detach().float().std().item())

result_paths = [
    "/kaggle/working/phase4_experiment_results.json",
    "/kaggle/working/a4_adapter_pilot_results.json",
    "/kaggle/working/lorma_plus_a4/results.json",
    "/kaggle/working/phase4_results/a3_distillation_results.json",
]

for path in result_paths:
    print(f"\n=== {path} ===")
    if os.path.exists(path):
        try:
            with open(path, "r") as f:
                print(json.dumps(json.load(f), indent=2)[:6000])
        except Exception as e:
            print("Could not read JSON:", repr(e))
    else:
        print("File not found")

=== Current classification head ===
Head: Linear(in_features=512, out_features=2, bias=True)
Weight shape: (2, 512)
Bias: [0.035468004643917084, -0.017459765076637268]
Weight std: 0.05105826258659363

=== /kaggle/working/phase4_experiment_results.json ===
[
  {
    "experiment": "A4 frozen encoder - standard loss",
    "accuracy": 0.5258,
    "macro_f1": 0.4487,
    "macro_precision": 0.5607,
    "macro_recall": 0.5266,
    "trainable_parameters": 1026
  },
  {
    "experiment": "A4 frozen encoder - class-weighted loss",
    "accuracy": 0.5327,
    "macro_f1": 0.5235,
    "trainable_parameters": 1026
  },
  {
    "experiment": "A4 BitFit",
    "accuracy": 0.499,
    "macro_f1": 0.353,
    "macro_precision": 0.4995,
    "macro_recall": 0.5,
    "trainable_parameters": 84994,
    "training_loss": 0.6891,
    "training_accuracy": 0.5849
  }
]

=== /kaggle/working/a4_adapter_pilot_results.json ===
{
  "experiment": "A4 multiplicative low-rank adapters pilot",
  "checkpoint": "A4 step 22000

In [184]:
import os

notebook_path = "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"

print("File exists:", os.path.exists(notebook_path))

if os.path.exists(notebook_path):
    print("File size:", os.path.getsize(notebook_path), "bytes")

    with open(notebook_path, "rb") as f:
        beginning = f.read(500)

    print("\nFirst 500 bytes:")
    print(repr(beginning))
else:
    print("Notebook source file not found.")

File exists: True
File size: 185025 bytes

First 500 bytes:
b'\nfrom pathlib import Path\nimport json\n\n# Find the dataset files mounted by Kaggle\nmatches = list(Path("/kaggle/input").rglob("label_vocab.json"))\n\nif not matches:\n    print("Dataset not found. Check the Kaggle Input panel.")\nelse:\n    phase3_root = matches[0].parent\n    print("Dataset directory:", phase3_root)\n\n    required_files = [\n        "CL_train.json", "CL_dev.json", "CL_test.json",\n        "IT_train.json", "IT_dev.json", "IT_test.json",\n        "label_vocab.json"\n    ]\n\n    for name in re'


In [185]:
from pathlib import Path

notebook_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

source = notebook_path.read_text(encoding="utf-8", errors="replace")
lines = source.splitlines()

terms = [
    "LORMA",
    "LoRMA",
    "LORMALinear",
    "A1",
    "A2",
    "A3",
    "A4",
    "knowledge distillation",
    "trainable_parameters",
]

for term in terms:
    matches = [
        i for i, line in enumerate(lines)
        if term.lower() in line.lower()
    ]
    print(f"\n{term}: {len(matches)} matches")
    for i in matches[:5]:
        print(f"  Line {i + 1}: {lines[i].strip()[:180]}")


LORMA: 42 matches
  Line 801: "GQA", "LORMA", "encoder_layers", "decoder_layers",
  Line 5087: # LoRMA+-style multiplicative adapter:
  Line 5090: class LoRMAPlusLinear(nn.Module):
  Line 5127: adapted = LoRMAPlusLinear(original, rank=8, alpha=16)
  Line 5819: class PaperLoRMAPlusLinear(nn.Module):

LoRMA: 42 matches
  Line 801: "GQA", "LORMA", "encoder_layers", "decoder_layers",
  Line 5087: # LoRMA+-style multiplicative adapter:
  Line 5090: class LoRMAPlusLinear(nn.Module):
  Line 5127: adapted = LoRMAPlusLinear(original, rank=8, alpha=16)
  Line 5819: class PaperLoRMAPlusLinear(nn.Module):

LORMALinear: 0 matches

A1: 10 matches
  Line 188: "A1_GPT": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
  Line 824: "A1": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
  Line 909: "A1": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained

In [186]:
from pathlib import Path

path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

for start, end in [(5810, 5905), (6080, 6160)]:
    print(f"\n{'=' * 20} Lines {start}-{end} {'=' * 20}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1}: {lines[i]}")


==================== Lines 5810-5905 ====================
5810:     wrapped_out = adapter(x)
5811:     print("Max absolute output difference:",
5812:           (base_out - wrapped_out).abs().max().item())
5813: 
5814: 
5815: import torch
5816: import torch.nn as nn
5817: import torch.nn.functional as F
5818: 
5819: class PaperLoRMAPlusLinear(nn.Module):
5820:     """
5821:     LoRMA+ weight transformation:
5822:         W_eff = (I + (alpha/r) * B @ A) @ W0
5823: 
5824:     B: (out_features, rank)
5825:     A: (rank, out_features)
5826:     """
5827:     def __init__(self, base_layer, rank=8, alpha=16):
5828:         super().__init__()
5829:         self.base = base_layer
5830:         self.rank = rank
5831:         self.scale = alpha / rank
5832: 
5833:         out_features, in_features = base_layer.weight.shape
5834:         if out_features != in_features:
5835:             raise ValueError(
5836:                 "This identity-inflated implementation requires a square weight matrix.

In [187]:
from pathlib import Path

roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

extensions = {".pdf", ".txt", ".md", ".html"}

matches = []

for root in roots:
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in extensions:
                name = path.name.lower()
                if "lorma" in name or "bihany" in name:
                    matches.append(path)

print("Possible LORMA reference files:", len(matches))

for path in matches[:30]:
    print(path)

Possible LORMA reference files: 0


In [188]:
import torch

print("Model:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("\nEncoder attention projections:")

for i, layer in enumerate(model.encoder_layers):
    attn = layer.attn
    for name in ("q_proj", "k_proj", "v_proj", "out_proj"):
        module = getattr(attn, name, None)
        if module is not None:
            base = getattr(module, "base", module)
            weight = getattr(base, "weight", None)
            print(
                f"Layer {i} | {name} | "
                f"wrapper={type(module).__name__} | "
                f"weight_shape={tuple(weight.shape) if weight is not None else 'N/A'}"
            )

Model: A4EncoderDecoder
Device: cuda:0

Encoder attention projections:
Layer 0 | q_proj | wrapper=PaperLoRMAPlusLinear | weight_shape=(512, 512)
Layer 0 | k_proj | wrapper=Linear | weight_shape=(512, 512)
Layer 0 | v_proj | wrapper=PaperLoRMAPlusLinear | weight_shape=(512, 512)
Layer 0 | out_proj | wrapper=Linear | weight_shape=(512, 512)
Layer 1 | q_proj | wrapper=PaperLoRMAPlusLinear | weight_shape=(512, 512)
Layer 1 | k_proj | wrapper=Linear | weight_shape=(512, 512)
Layer 1 | v_proj | wrapper=PaperLoRMAPlusLinear | weight_shape=(512, 512)
Layer 1 | out_proj | wrapper=Linear | weight_shape=(512, 512)
Layer 2 | q_proj | wrapper=PaperLoRMAPlusLinear | weight_shape=(512, 512)
Layer 2 | k_proj | wrapper=Linear | weight_shape=(512, 512)
Layer 2 | v_proj | wrapper=PaperLoRMAPlusLinear | weight_shape=(512, 512)
Layer 2 | out_proj | wrapper=Linear | weight_shape=(512, 512)
Layer 3 | q_proj | wrapper=PaperLoRMAPlusLinear | weight_shape=(512, 512)
Layer 3 | k_proj | wrapper=Linear | weight_sh

In [189]:
for i, layer in enumerate(model.encoder_layers):
    for name in ("q_proj", "v_proj"):
        module = getattr(layer.attn, name)

        print(
            f"Layer {i} | {name} | "
            f"rank={module.rank} | "
            f"scale={module.scale} | "
            f"A={tuple(module.A.shape)} | "
            f"B={tuple(module.B.shape)} | "
            f"B_zero={bool(torch.count_nonzero(module.B).item() == 0)} | "
            f"base_frozen={not any(p.requires_grad for p in module.base.parameters())}"
        )

    if i == 1:
        break

Layer 0 | q_proj | rank=8 | scale=2.0 | A=(8, 512) | B=(512, 8) | B_zero=False | base_frozen=True
Layer 0 | v_proj | rank=8 | scale=2.0 | A=(8, 512) | B=(512, 8) | B_zero=False | base_frozen=True
Layer 1 | q_proj | rank=8 | scale=2.0 | A=(8, 512) | B=(512, 8) | B_zero=False | base_frozen=True
Layer 1 | v_proj | rank=8 | scale=2.0 | A=(8, 512) | B=(512, 8) | B_zero=False | base_frozen=True


In [191]:
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

if source_path.exists():
    source = source_path.read_text(errors="replace")
    lines = source.splitlines()

    matches = [
        i for i, line in enumerate(lines)
        if "class PaperLoRMAPlusLinear" in line
    ]

    if matches:
        for start in matches:
            print(f"\nClass definition at source line {start + 1}:\n")
            print("\n".join(lines[start:start + 45]))
    else:
        print("Class not found in this source file.")
else:
    print("Notebook source file not found.")


Class definition at source line 5819:

class PaperLoRMAPlusLinear(nn.Module):
    """
    LoRMA+ weight transformation:
        W_eff = (I + (alpha/r) * B @ A) @ W0

    B: (out_features, rank)
    A: (rank, out_features)
    """
    def __init__(self, base_layer, rank=8, alpha=16):
        super().__init__()
        self.base = base_layer
        self.rank = rank
        self.scale = alpha / rank

        out_features, in_features = base_layer.weight.shape
        if out_features != in_features:
            raise ValueError(
                "This identity-inflated implementation requires a square weight matrix."
            )

        self.A = nn.Parameter(
            torch.empty(rank, out_features,
                        device=base_layer.weight.device,
                        dtype=base_layer.weight.dtype)
        )
        self.B = nn.Parameter(
            torch.zeros(out_features, rank,
                        device=base_layer.weight.device,
                        dtype=base

In [192]:
import json
from pathlib import Path

files = [
    Path("/kaggle/working/phase4_experiment_results.json"),
    Path("/kaggle/working/a4_adapter_pilot_results.json"),
    Path("/kaggle/working/lorma_plus_a4/results.json"),
    Path("/kaggle/working/phase4_results/a3_distillation_results.json"),
]

for path in files:
    print(f"\n{'=' * 60}\n{path.name}")
    if path.exists():
        data = json.loads(path.read_text())
        print(json.dumps(data, indent=2)[:5000])
    else:
        print("File not found")


phase4_experiment_results.json
[
  {
    "experiment": "A4 frozen encoder - standard loss",
    "accuracy": 0.5258,
    "macro_f1": 0.4487,
    "macro_precision": 0.5607,
    "macro_recall": 0.5266,
    "trainable_parameters": 1026
  },
  {
    "experiment": "A4 frozen encoder - class-weighted loss",
    "accuracy": 0.5327,
    "macro_f1": 0.5235,
    "trainable_parameters": 1026
  },
  {
    "experiment": "A4 BitFit",
    "accuracy": 0.499,
    "macro_f1": 0.353,
    "macro_precision": 0.4995,
    "macro_recall": 0.5,
    "trainable_parameters": 84994,
    "training_loss": 0.6891,
    "training_accuracy": 0.5849
  }
]

a4_adapter_pilot_results.json
{
  "experiment": "A4 multiplicative low-rank adapters pilot",
  "checkpoint": "A4 step 22000",
  "training_examples": 2048,
  "epochs": 1,
  "batch_size": 8,
  "learning_rate": 5e-05,
  "trainable_parameters": 99330,
  "development_examples": 3505,
  "development_macro_f1_threshold_0.50": 0.4806,
  "development_macro_f1_threshold_0.55": 0

In [193]:
import json
from pathlib import Path

paths = [
    Path("/kaggle/working/phase4_experiment_results.json"),
    Path("/kaggle/working/a4_adapter_pilot_results.json"),
    Path("/kaggle/working/lorma_plus_a4/results.json"),
    Path("/kaggle/working/phase4_results/a3_distillation_results.json"),
]

for path in paths:
    if not path.exists():
        continue

    data = json.loads(path.read_text())
    records = data if isinstance(data, list) else [data]

    for record in records:
        print(
            record.get("experiment", "Unnamed experiment"),
            "|",
            record.get("checkpoint", "Architecture not specified")
        )

A4 frozen encoder - standard loss | Architecture not specified
A4 frozen encoder - class-weighted loss | Architecture not specified
A4 BitFit | Architecture not specified
A4 multiplicative low-rank adapters pilot | A4 step 22000
Corrected LoRMA+ on A4 | Architecture not specified
A3 frozen-backbone knowledge distillation from A4 teacher | Architecture not specified


In [194]:
from pathlib import Path
import torch

checkpoints = {
    "A1": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
}

for name, path in checkpoints.items():
    print(f"{name}: checkpoint_exists={Path(path).exists()}")

print("\nModel objects currently in memory:")
for name in ["model", "a2_model", "a3_student"]:
    obj = globals().get(name)
    if isinstance(obj, torch.nn.Module):
        print(f"{name}: {type(obj).__name__}")
    else:
        print(f"{name}: not loaded")

A1: checkpoint_exists=True
A2: checkpoint_exists=True
A3: checkpoint_exists=True
A4: checkpoint_exists=True

Model objects currently in memory:
model: A4EncoderDecoder
a2_model: Phase4A2Model
a3_student: A3PrefixLM


In [195]:
import torch
from pathlib import Path

a1_checkpoint_path = Path(
    "/kaggle/input/datasets/aq1de3/"
    "architecture1-sustained-step-500-pt/"
    "architecture1_sustained_step_500.pt"
)

a1_checkpoint = torch.load(
    a1_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint loaded:", a1_checkpoint_path.exists())
print("Checkpoint type:", type(a1_checkpoint).__name__)

if isinstance(a1_checkpoint, dict):
    print("Checkpoint keys:", list(a1_checkpoint.keys()))
    state = a1_checkpoint.get("model_state_dict", a1_checkpoint)
    if isinstance(state, dict):
        print("\nFirst 15 parameter names and shapes:")
        for i, (name, value) in enumerate(state.items()):
            if i == 15:
                break
            if torch.is_tensor(value):
                print(name, tuple(value.shape))

Checkpoint loaded: True
Checkpoint type: dict
Checkpoint keys: ['step', 'epoch', 'train_loss', 'val_loss', 'model_state_dict', 'optimizer_state_dict']

First 15 parameter names and shapes:
token_embedding.weight (50000, 512)
position_embedding.weight (2048, 512)
blocks.0.norm1.weight (512,)
blocks.0.norm1.bias (512,)
blocks.0.attn.in_proj_weight (1536, 512)
blocks.0.attn.in_proj_bias (1536,)
blocks.0.attn.out_proj.weight (512, 512)
blocks.0.attn.out_proj.bias (512,)
blocks.0.norm2.weight (512,)
blocks.0.norm2.bias (512,)
blocks.0.ffn.0.weight (2048, 512)
blocks.0.ffn.0.bias (2048,)
blocks.0.ffn.2.weight (512, 2048)
blocks.0.ffn.2.bias (512,)
blocks.1.norm1.weight (512,)


In [196]:
from pathlib import Path

roots = [
    Path("/kaggle/working/.virtual_documents"),
    Path("/kaggle/input/datasets/aq1de3"),
]

for root in roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file() or path.suffix not in {".py", ".ipynb"}:
            continue

        try:
            text = path.read_text(errors="ignore")
        except Exception:
            continue

        if (
            "architecture1_sustained_step_500" in text
            or "class Architecture1" in text
            or "class GPTLanguageModel" in text
        ):
            print(f"\nFile: {path}")
            for i, line in enumerate(text.splitlines()):
                if any(term in line for term in [
                    "class Architecture1",
                    "class GPTLanguageModel",
                    "class TransformerBlock",
                    "class Block",
                ]):
                    print(f"  Line {i + 1}: {line[:180]}")


File: /kaggle/working/.virtual_documents/__notebook_source__.ipynb
  Line 1676:     "class TransformerBlock",
  Line 1710:     "class TransformerBlock",

File: /kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb
  Line 1: {"metadata":{"kernelspec":{"language":"python","display_name":"Python 3","name":"python3"},"language_info":{"name":"python","version":"3.13.15","mimetype":"text/x-python","codemirr

File: /kaggle/input/datasets/aq1de3/p2p2p2/phase-2 (1).ipynb


In [197]:
from pathlib import Path

root = Path("/kaggle/input/datasets/aq1de3")

for path in root.rglob("*.ipynb"):
    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue

    matches = []
    for term in [
        "architecture1_sustained_step_500.pt",
        "architecture1_sustained",
        "class TransformerBlock",
        "class GPTLanguageModel",
    ]:
        if term in text:
            matches.append(term)

    if matches:
        print(f"\nNotebook: {path}")
        print("Matches:", ", ".join(matches))


Notebook: /kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb
Matches: architecture1_sustained_step_500.pt, architecture1_sustained, class TransformerBlock

Notebook: /kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb
Matches: architecture1_sustained

Notebook: /kaggle/input/datasets/aq1de3/p2p2p2/phase-2 (1).ipynb
Matches: architecture1_sustained_step_500.pt, architecture1_sustained


In [198]:
import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/p2p2p2/phase-2 (1).ipynb"
)

nb = json.loads(path.read_text())

terms = [
    "class TransformerBlock",
    "class GPTLanguageModel",
    "architecture1_sustained_step_500.pt",
    "architecture1_sustained",
]

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    if any(term in source for term in terms):
        print(f"\n{'=' * 60}\nCELL {i} | type={cell.get('cell_type')}")
        print(source[:7000])


CELL 13 | type=code
# ============================================================
# STEP 2.6 — INSPECT PHASE-1 CHECKPOINTS
# GPU: OFF
# ============================================================

import torch
import os

CHECKPOINTS = {
    "A1": "/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/a3check/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt",
}

print("=" * 80)
print("PHASE-1 CHECKPOINT METADATA")
print("=" * 80)

for name, path in CHECKPOINTS.items():

    print(f"\n{'='*30} {name} {'='*30}")
    print("File:", path)
    print("Size:", f"{os.path.getsize(path)/(1024**3):.3f} GB")

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Type:", type(ckpt).__name__)

    if isinstance(ckpt, dict):
        print("Keys:", list(ckpt.keys()))

        for k

In [199]:
import torch
import torch.nn as nn
from pathlib import Path

A1_PATH = Path(
    "/kaggle/input/datasets/aq1de3/"
    "architecture1-sustained-step-500-pt/"
    "architecture1_sustained_step_500.pt"
)

ckpt = torch.load(A1_PATH, map_location="cpu", weights_only=False)
state = ckpt["model_state_dict"]

vocab_size, hidden_size = state["token_embedding.weight"].shape
context_length = state["position_embedding.weight"].shape[0]
num_layers = sum(
    key.endswith(".norm1.weight") and key.startswith("blocks.")
    for key in state
)
ffn_hidden = state["blocks.0.ffn.0.weight"].shape[0]
num_heads = 8

class A1Block(nn.Module):
    def __init__(self, hidden_size, num_heads, ffn_hidden):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = nn.MultiheadAttention(
            hidden_size, num_heads, batch_first=True
        )
        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, ffn_hidden),
            nn.GELU(),
            nn.Linear(ffn_hidden, hidden_size),
        )

    def forward(self, x, causal_mask):
        h = self.norm1(x)
        attn_out, _ = self.attn(
            h, h, h, attn_mask=causal_mask, need_weights=False
        )
        x = x + attn_out
        return x + self.ffn(self.norm2(x))


class A1Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_size)
        self.position_embedding = nn.Embedding(
            context_length, hidden_size
        )
        self.blocks = nn.ModuleList([
            A1Block(hidden_size, num_heads, ffn_hidden)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device)
        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)[None, :, :]
        )
        causal_mask = torch.triu(
            torch.ones(
                seq_len, seq_len,
                device=input_ids.device,
                dtype=torch.bool,
            ),
            diagonal=1,
        )
        for block in self.blocks:
            x = block(x, causal_mask)
        return self.lm_head(self.final_norm(x))


a1_model = A1Model()
a1_model.load_state_dict(state, strict=True)
a1_model = a1_model.to("cuda" if torch.cuda.is_available() else "cpu")
a1_model.eval()

device = next(a1_model.parameters()).device
dummy_ids = torch.randint(0, vocab_size, (1, 32), device=device)

with torch.no_grad():
    logits = a1_model(dummy_ids)

print("A1 checkpoint loaded successfully.")
print("Strict load: passed")
print("Device:", device)
print("Parameters:", f"{sum(p.numel() for p in a1_model.parameters()):,}")
print("Forward-pass output:", tuple(logits.shape))
print("Finite logits:", bool(torch.isfinite(logits).all()))

A1 checkpoint loaded successfully.
Strict load: passed
Device: cuda:0
Parameters: 77,468,672
Forward-pass output: (1, 32, 50000)
Finite logits: True


In [200]:
import inspect

for name in [
    "prepare_batch",
    "classification_head",
    "evaluate_classifier",
]:
    obj = globals().get(name)

    print(f"\n--- {name} ---")

    if obj is None:
        print("Not defined")
    elif callable(obj):
        try:
            print(inspect.signature(obj))
            print(inspect.getsource(obj)[:3500])
        except (OSError, TypeError):
            print("Object exists, but source is unavailable.")
    else:
        print("Type:", type(obj).__name__)
        if hasattr(obj, "state_dict"):
            print("Parameters:", sum(p.numel() for p in obj.parameters()))


--- prepare_batch ---
(records, max_length=1024)
def prepare_batch(records, max_length=1024):
    pad_id = tokenizer.token_to_id("[PAD]")
    bos_id = tokenizer.token_to_id("[BOS]")
    eos_id = tokenizer.token_to_id("[EOS]")

    all_ids = []
    all_labels = []

    for row in records:
        ids = tokenizer.encode(row["text"]).ids
        ids = ids[:max_length - 2]
        ids = [bos_id] + ids + [eos_id]

        # Fixed-length padding
        ids += [pad_id] * (max_length - len(ids))

        all_ids.append(ids)
        all_labels.append(int(row["label"]))

    input_ids = torch.tensor(all_ids, dtype=torch.long, device=device)
    labels = torch.tensor(all_labels, dtype=torch.long, device=device)
    attention_mask = (input_ids != pad_id).long()

    return input_ids, attention_mask, labels


--- classification_head ---
(*args, **kwargs)
Object exists, but source is unavailable.

--- evaluate_classifier ---
Not defined


In [202]:
from pathlib import Path

root = Path("/kaggle/input/datasets/aq1de3")

for split in ["train", "dev", "test"]:
    print(f"\n{split.upper()} files:")
    matches = [
        p for p in root.rglob("*.json")
        if split in p.name.lower()
        and p.name.lower().endswith(".json")
    ]
    for p in matches[:10]:
        print(p)
    if not matches:
        print("No matching files found.")


TRAIN files:
/kaggle/input/datasets/aq1de3/phase3/CL_train.json
/kaggle/input/datasets/aq1de3/phase3/IT_train.json
/kaggle/input/datasets/aq1de3/trainjson/train.json

DEV files:
/kaggle/input/datasets/aq1de3/devjson/dev.json
/kaggle/input/datasets/aq1de3/phase3/CL_dev.json
/kaggle/input/datasets/aq1de3/phase3/IT_dev.json

TEST files:
/kaggle/input/datasets/aq1de3/testjson/test.json
/kaggle/input/datasets/aq1de3/phase3/IT_test.json
/kaggle/input/datasets/aq1de3/phase3/CL_test.json


In [203]:
import json
from pathlib import Path
from collections import Counter

train_path = Path("/kaggle/input/datasets/aq1de3/trainjson/train.json")
dev_path = Path("/kaggle/input/datasets/aq1de3/devjson/dev.json")

with train_path.open("r", encoding="utf-8") as f:
    train_records = json.load(f)

with dev_path.open("r", encoding="utf-8") as f:
    dev_records = json.load(f)

print("Train records:", len(train_records))
print("Dev records:", len(dev_records))

print("Train label counts:", Counter(int(r["label"]) for r in train_records))
print("Dev label counts:", Counter(int(r["label"]) for r in dev_records))

print("Train sample keys:", list(train_records[0].keys()))
print("Dev sample keys:", list(dev_records[0].keys()))
print("Sample label:", train_records[0]["label"])
print("Sample text length:", len(train_records[0]["text"]))

Train records: 37387
Dev records: 3505
Train label counts: Counter({0: 22067, 1: 15320})
Dev label counts: Counter({1: 1756, 0: 1749})
Train sample keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
Dev sample keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
Sample label: 0
Sample text length: 273042


In [204]:
print("Tokenizer type:", type(tokenizer).__name__)

for token in ["[PAD]", "[BOS]", "[EOS]"]:
    print(f"{token} ID:", tokenizer.token_to_id(token))

print("\nVocabulary size:", tokenizer.get_vocab_size())

batch_ids, batch_mask, batch_labels = prepare_batch(
    train_records[:2],
    max_length=1024
)

print("\nInput shape:", tuple(batch_ids.shape))
print("Attention mask shape:", tuple(batch_mask.shape))
print("Labels:", batch_labels.tolist())
print("Non-padding tokens:", batch_mask.sum(dim=1).tolist())
print(
    "IDs within vocabulary:",
    bool((batch_ids >= 0).all() and
         (batch_ids < tokenizer.get_vocab_size()).all())
)

Tokenizer type: Tokenizer
[PAD] ID: 0
[BOS] ID: 2
[EOS] ID: 3

Vocabulary size: 50000

Input shape: (2, 1024)
Attention mask shape: (2, 1024)
Labels: [0, 0]
Non-padding tokens: [1024, 1024]
IDs within vocabulary: True


In [205]:
import torch
import torch.nn as nn

# Freeze the pretrained A1 model
for param in a1_model.parameters():
    param.requires_grad = False

a1_model.eval()

# Small binary classification head
a1_classification_head = nn.Sequential(
    nn.Linear(512, 256),
    nn.GELU(),
    nn.Dropout(0.1),
    nn.Linear(256, 2)
).to(device)

optimizer_a1 = torch.optim.AdamW(
    a1_classification_head.parameters(),
    lr=1e-3
)

criterion_a1 = nn.CrossEntropyLoss()

print("A1 backbone frozen:",
      not any(p.requires_grad for p in a1_model.parameters()))
print("Head trainable parameters:",
      sum(p.numel() for p in a1_classification_head.parameters()))
print("Device:", device)

A1 backbone frozen: True
Head trainable parameters: 131842
Device: cuda:0


In [206]:
a1_model.eval()
a1_classification_head.eval()

with torch.no_grad():
    input_ids, attention_mask, labels = prepare_batch(
        train_records[:2],
        max_length=1024
    )

    # Extract A1 transformer representations
    x = a1_model.token_embedding(input_ids)
    positions = torch.arange(input_ids.shape[1], device=device)
    x = x + a1_model.position_embedding(positions)[None, :, :]

    causal_mask = torch.triu(
        torch.ones(
            input_ids.shape[1],
            input_ids.shape[1],
            device=device,
            dtype=torch.bool
        ),
        diagonal=1
    )

    for block in a1_model.blocks:
        x = block(x, causal_mask)

    x = a1_model.final_norm(x)

    # EOS is the final non-padding token for these examples
    last_positions = attention_mask.sum(dim=1) - 1
    features = x[
        torch.arange(input_ids.shape[0], device=device),
        last_positions
    ]

    logits = a1_classification_head(features)

print("Features:", tuple(features.shape))
print("Logits:", tuple(logits.shape))
print("Labels:", labels.tolist())
print("Features finite:", bool(torch.isfinite(features).all()))
print("Logits finite:", bool(torch.isfinite(logits).all()))

Features: (2, 512)
Logits: (2, 2)
Labels: [0, 0]
Features finite: True
Logits finite: True


In [207]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

a1_model.eval()
a1_classification_head.train()

pilot_records = train_records[:256]
batch_size = 8
losses = []

for start in range(0, len(pilot_records), batch_size):
    batch = pilot_records[start:start + batch_size]
    input_ids, attention_mask, labels = prepare_batch(
        batch, max_length=1024
    )

    # Frozen backbone: no gradient computation
    with torch.no_grad():
        x = a1_model.token_embedding(input_ids)
        positions = torch.arange(input_ids.shape[1], device=device)
        x = x + a1_model.position_embedding(positions)[None, :, :]

        causal_mask = torch.triu(
            torch.ones(
                input_ids.shape[1], input_ids.shape[1],
                device=device, dtype=torch.bool
            ),
            diagonal=1
        )

        for block in a1_model.blocks:
            x = block(x, causal_mask)

        x = a1_model.final_norm(x)
        last_positions = attention_mask.sum(dim=1) - 1
        features = x[
            torch.arange(input_ids.shape[0], device=device),
            last_positions
        ]

    logits = a1_classification_head(features)
    loss = F.cross_entropy(logits, labels)

    optimizer_a1.zero_grad(set_to_none=True)
    loss.backward()
    optimizer_a1.step()

    losses.append(loss.item())

print("Pilot examples:", len(pilot_records))
print("Batches:", len(losses))
print("Mean training loss:", round(sum(losses) / len(losses), 4))
print("Head parameters finite:", all(
    torch.isfinite(p).all().item()
    for p in a1_classification_head.parameters()
))

Pilot examples: 256
Batches: 32
Mean training loss: 0.7589
Head parameters finite: True


In [208]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

a1_model.eval()
a1_classification_head.eval()

all_predictions = []
all_labels = []
all_probabilities = []

eval_batch_size = 8

with torch.no_grad():
    for start in range(0, len(dev_records), eval_batch_size):
        batch = dev_records[start:start + eval_batch_size]

        input_ids, attention_mask, labels = prepare_batch(
            batch, max_length=1024
        )

        x = a1_model.token_embedding(input_ids)
        positions = torch.arange(input_ids.shape[1], device=device)
        x = x + a1_model.position_embedding(positions)[None, :, :]

        causal_mask = torch.triu(
            torch.ones(
                input_ids.shape[1], input_ids.shape[1],
                device=device, dtype=torch.bool
            ),
            diagonal=1
        )

        for block in a1_model.blocks:
            x = block(x, causal_mask)

        x = a1_model.final_norm(x)
        last_positions = attention_mask.sum(dim=1) - 1
        features = x[
            torch.arange(input_ids.shape[0], device=device),
            last_positions
        ]

        logits = a1_classification_head(features)
        probabilities = torch.softmax(logits, dim=1)
        predictions = logits.argmax(dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probabilities.extend(probabilities.cpu().tolist())

print("Development examples:", len(all_labels))
print("Accuracy:", round(accuracy_score(all_labels, all_predictions), 4))
print("Macro-F1:", round(f1_score(all_labels, all_predictions, average="macro", zero_division=0), 4))
print("Macro precision:", round(precision_score(all_labels, all_predictions, average="macro", zero_division=0), 4))
print("Macro recall:", round(recall_score(all_labels, all_predictions, average="macro", zero_division=0), 4))
print("Confusion matrix:\n", confusion_matrix(all_labels, all_predictions))
print("Prediction counts:", dict(zip(*np.unique(all_predictions, return_counts=True))))

Development examples: 3505
Accuracy: 0.501
Macro-F1: 0.3338
Macro precision: 0.2505
Macro recall: 0.5
Confusion matrix:
 [[   0 1749]
 [   0 1756]]
Prediction counts: {np.int64(1): np.int64(3505)}


In [209]:
import numpy as np

probs = np.asarray(all_probabilities)
preds = np.asarray(all_predictions)

print("Probability shape:", probs.shape)
print("Mean probability per class:", probs.mean(axis=0).round(5))
print("Std probability per class:", probs.std(axis=0).round(5))
print("Probability range, class 0:",
      round(float(probs[:, 0].min()), 5),
      round(float(probs[:, 0].max()), 5))
print("Probability range, class 1:",
      round(float(probs[:, 1].min()), 5),
      round(float(probs[:, 1].max()), 5))
print("Predicted class counts:", np.bincount(preds, minlength=2))
print("Actual class counts:", np.bincount(np.asarray(all_labels), minlength=2))

Probability shape: (3505, 2)
Mean probability per class: [0.48221 0.51779]
Std probability per class: [0.0012 0.0012]
Probability range, class 0: 0.47419 0.48839
Probability range, class 1: 0.51161 0.52581
Predicted class counts: [   0 3505]
Actual class counts: [1749 1756]


In [210]:
import numpy as np
import torch

a1_model.eval()

# Use examples from both classes, rather than the first 32 records only.
sample_0 = next(r for r in train_records if int(r["label"]) == 0)
sample_1 = next(r for r in train_records if int(r["label"]) == 1)
sample_records = [sample_0, sample_1]

with torch.no_grad():
    input_ids, attention_mask, labels = prepare_batch(
        sample_records, max_length=1024
    )

    x = a1_model.token_embedding(input_ids)
    positions = torch.arange(input_ids.shape[1], device=device)
    x = x + a1_model.position_embedding(positions)[None, :, :]

    causal_mask = torch.triu(
        torch.ones(
            input_ids.shape[1], input_ids.shape[1],
            device=device, dtype=torch.bool
        ),
        diagonal=1
    )

    for block in a1_model.blocks:
        x = block(x, causal_mask)

    x = a1_model.final_norm(x)
    last_positions = attention_mask.sum(dim=1) - 1
    features = x[
        torch.arange(input_ids.shape[0], device=device),
        last_positions
    ]

print("Labels:", labels.tolist())
print("Feature shape:", tuple(features.shape))
print("Per-example feature norms:",
      features.norm(dim=1).cpu().numpy().round(4))
print("Feature vector distance:",
      round(torch.norm(features[0] - features[1]).item(), 6))
print("Features finite:", bool(torch.isfinite(features).all()))

Labels: [0, 1]
Feature shape: (2, 512)
Per-example feature norms: [26.5042 26.5009]
Feature vector distance: 1.403693
Features finite: True


In [211]:
import torch

a1_classification_head.eval()

with torch.no_grad():
    input_ids, attention_mask, labels = prepare_batch(
        train_records[:16], max_length=1024
    )

    x = a1_model.token_embedding(input_ids)
    positions = torch.arange(input_ids.shape[1], device=device)
    x = x + a1_model.position_embedding(positions)[None, :, :]

    causal_mask = torch.triu(
        torch.ones(
            input_ids.shape[1], input_ids.shape[1],
            device=device, dtype=torch.bool
        ),
        diagonal=1
    )

    for block in a1_model.blocks:
        x = block(x, causal_mask)

    x = a1_model.final_norm(x)
    last_positions = attention_mask.sum(dim=1) - 1
    features = x[
        torch.arange(input_ids.shape[0], device=device),
        last_positions
    ]

    logits = a1_classification_head(features)
    probabilities = torch.softmax(logits, dim=1)

print("Labels:", labels.tolist())
print("Predictions:", logits.argmax(dim=1).tolist())
print("Mean probabilities:", probabilities.mean(dim=0).cpu().tolist())
print("Feature std across examples:", features.std(dim=0).mean().item())
print("Head output-weight norm:",
      a1_classification_head[-1].weight.norm().item())
print("Head output-bias:", a1_classification_head[-1].bias.detach().cpu().tolist())

Labels: [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0]
Predictions: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Mean probabilities: [0.48243841528892517, 0.517561674118042]
Feature std across examples: 0.0387401282787323
Head output-weight norm: 0.7415471076965332
Head output-bias: [-0.011575695127248764, -0.023629430681467056]


In [212]:
a2_model.eval()

print("A2 model:", type(a2_model).__name__)
print("Device:", next(a2_model.parameters()).device)
print("Parameters:", f"{sum(p.numel() for p in a2_model.parameters()):,}")

with torch.no_grad():
    input_ids, attention_mask, labels = prepare_batch(
        train_records[:2], max_length=128
    )
    outputs = a2_model(input_ids)

    if isinstance(outputs, tuple):
        print("Output shapes:", [
            tuple(x.shape) for x in outputs if torch.is_tensor(x)
        ])
    elif torch.is_tensor(outputs):
        print("Output shape:", tuple(outputs.shape))
    else:
        print("Output type:", type(outputs).__name__)

A2 model: Phase4A2Model
Device: cuda:0
Parameters: 51,513,680
Output shape: (2, 128, 50000)


In [213]:
print("Model attributes:")
for name in ["layers", "final_norm", "token_embedding", "position_embedding"]:
    obj = getattr(a2_model, name, None)
    if obj is None:
        print(f"{name}: not found")
    elif isinstance(obj, torch.nn.Module):
        print(f"{name}: {type(obj).__name__}")
    else:
        print(f"{name}: {type(obj).__name__}, length={len(obj) if hasattr(obj, '__len__') else 'N/A'}")

print("\nFirst layer type:", type(a2_model.layers[0]).__name__)
print("Number of layers:", len(a2_model.layers))

Model attributes:
layers: ModuleList
final_norm: LayerNorm
token_embedding: Embedding
position_embedding: Embedding

First layer type: Phase4A2Layer
Number of layers: 9


In [214]:
a2_model.eval()

with torch.no_grad():
    input_ids, attention_mask, labels = prepare_batch(
        train_records[:2], max_length=128
    )

    batch_size, seq_len = input_ids.shape
    positions = torch.arange(seq_len, device=device)

    x = (
        a2_model.token_embedding(input_ids)
        + a2_model.position_embedding(positions)[None, :, :]
    )

    # Reuse the model's own transformer layers.
    for layer in a2_model.layers:
        x = layer(x)

    x = a2_model.final_norm(x)

    # A2 is causal, so EOS can summarize preceding tokens.
    last_positions = attention_mask.sum(dim=1) - 1
    a2_features = x[
        torch.arange(batch_size, device=device),
        last_positions
    ]

print("Features shape:", tuple(a2_features.shape))
print("Labels:", labels.tolist())
print("Features finite:", bool(torch.isfinite(a2_features).all()))
print("Feature norms:", a2_features.norm(dim=1).tolist())


Features shape: (2, 512)
Labels: [0, 0]
Features finite: True
Feature norms: [14.904109001159668, 14.897618293762207]


In [215]:
import torch
import torch.nn as nn

for param in a2_model.parameters():
    param.requires_grad = False

a2_model.eval()

a2_classification_head = nn.Sequential(
    nn.Linear(512, 256),
    nn.GELU(),
    nn.Dropout(0.1),
    nn.Linear(256, 2)
).to(device)

optimizer_a2 = torch.optim.AdamW(
    a2_classification_head.parameters(),
    lr=1e-3
)

print(
    "A2 backbone frozen:",
    not any(p.requires_grad for p in a2_model.parameters())
)
print(
    "Head trainable parameters:",
    sum(p.numel() for p in a2_classification_head.parameters())
)
print("Device:", device)

A2 backbone frozen: True
Head trainable parameters: 131842
Device: cuda:0


In [216]:
import torch
import torch.nn.functional as F

a2_model.eval()
a2_classification_head.train()

pilot_records_a2 = train_records[:256]
batch_size = 8
losses_a2 = []

for start in range(0, len(pilot_records_a2), batch_size):
    batch = pilot_records_a2[start:start + batch_size]

    input_ids, attention_mask, labels = prepare_batch(
        batch, max_length=1024
    )

    with torch.no_grad():
        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=device)

        x = (
            a2_model.token_embedding(input_ids)
            + a2_model.position_embedding(positions)[None, :, :]
        )

        for layer in a2_model.layers:
            x = layer(x)

        x = a2_model.final_norm(x)

        last_positions = attention_mask.sum(dim=1) - 1
        features = x[
            torch.arange(input_ids.shape[0], device=device),
            last_positions
        ]

    logits = a2_classification_head(features)
    loss = F.cross_entropy(logits, labels)

    optimizer_a2.zero_grad(set_to_none=True)
    loss.backward()
    optimizer_a2.step()

    losses_a2.append(loss.item())

print("Pilot examples:", len(pilot_records_a2))
print("Batches:", len(losses_a2))
print("Mean training loss:", round(sum(losses_a2) / len(losses_a2), 4))
print(
    "Head parameters finite:",
    all(torch.isfinite(p).all().item()
        for p in a2_classification_head.parameters())
)

Pilot examples: 256
Batches: 32
Mean training loss: 0.7228
Head parameters finite: True


In [217]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

a2_model.eval()
a2_classification_head.eval()

preds_a2 = []
labels_a2 = []
probs_a2 = []

eval_batch_size = 8

with torch.no_grad():
    for start in range(0, len(dev_records), eval_batch_size):
        batch = dev_records[start:start + eval_batch_size]

        input_ids, attention_mask, labels = prepare_batch(
            batch, max_length=1024
        )

        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=device)

        x = (
            a2_model.token_embedding(input_ids)
            + a2_model.position_embedding(positions)[None, :, :]
        )

        for layer in a2_model.layers:
            x = layer(x)

        x = a2_model.final_norm(x)

        last_positions = attention_mask.sum(dim=1) - 1
        features = x[
            torch.arange(input_ids.shape[0], device=device),
            last_positions
        ]

        logits = a2_classification_head(features)
        probabilities = torch.softmax(logits, dim=1)

        preds_a2.extend(logits.argmax(dim=1).cpu().tolist())
        labels_a2.extend(labels.cpu().tolist())
        probs_a2.extend(probabilities.cpu().tolist())

print("Development examples:", len(labels_a2))
print("Accuracy:", round(accuracy_score(labels_a2, preds_a2), 4))
print("Macro-F1:", round(f1_score(labels_a2, preds_a2, average="macro", zero_division=0), 4))
print("Macro precision:", round(precision_score(labels_a2, preds_a2, average="macro", zero_division=0), 4))
print("Macro recall:", round(recall_score(labels_a2, preds_a2, average="macro", zero_division=0), 4))
print("Confusion matrix:\n", confusion_matrix(labels_a2, preds_a2))
print("Prediction counts:", np.bincount(preds_a2, minlength=2))

Development examples: 3505
Accuracy: 0.501
Macro-F1: 0.3338
Macro precision: 0.2505
Macro recall: 0.5
Confusion matrix:
 [[   0 1749]
 [   0 1756]]
Prediction counts: [   0 3505]


In [218]:
import json
from pathlib import Path

results_path = Path("/kaggle/working/phase4_a1_a2_pilot_results.json")

results = [
    {
        "experiment": "A1 frozen backbone classification head pilot",
        "training_examples": 256,
        "epochs": 1,
        "development_examples": 3505,
        "accuracy": 0.501,
        "macro_f1": 0.3338,
        "macro_precision": 0.2505,
        "macro_recall": 0.5,
        "confusion_matrix": [[0, 1749], [0, 1756]],
        "status": "unsuccessful: predicts class 1 for every development record"
    },
    {
        "experiment": "A2 frozen backbone classification head pilot",
        "training_examples": 256,
        "epochs": 1,
        "development_examples": 3505,
        "accuracy": 0.501,
        "macro_f1": 0.3338,
        "macro_precision": 0.2505,
        "macro_recall": 0.5,
        "confusion_matrix": [[0, 1749], [0, 1756]],
        "status": "unsuccessful: predicts class 1 for every development record"
    }
]

results_path.write_text(json.dumps(results, indent=2))
print("Saved:", results_path)
print("Experiments recorded:", len(results))

Saved: /kaggle/working/phase4_a1_a2_pilot_results.json
Experiments recorded: 2


In [219]:
import json
from pathlib import Path

result_files = [
    Path("/kaggle/working/phase4_experiment_results.json"),
    Path("/kaggle/working/a4_adapter_pilot_results.json"),
    Path("/kaggle/working/lorma_plus_a4/results.json"),
    Path("/kaggle/working/phase4_results/a3_distillation_results.json"),
    Path("/kaggle/working/phase4_a1_a2_pilot_results.json"),
]

for path in result_files:
    print(f"\n{'=' * 65}")
    print(path)

    if not path.exists():
        print("MISSING")
        continue

    data = json.loads(path.read_text())
    records = data if isinstance(data, list) else [data]

    for record in records:
        print(
            "Experiment:", record.get("experiment", "Unnamed"),
            "| Macro-F1:", record.get("macro_f1", "not recorded"),
            "| Status:", record.get("status", "review required")
        )


/kaggle/working/phase4_experiment_results.json
Experiment: A4 frozen encoder - standard loss | Macro-F1: 0.4487 | Status: review required
Experiment: A4 frozen encoder - class-weighted loss | Macro-F1: 0.5235 | Status: review required
Experiment: A4 BitFit | Macro-F1: 0.353 | Status: review required

/kaggle/working/a4_adapter_pilot_results.json
Experiment: A4 multiplicative low-rank adapters pilot | Macro-F1: not recorded | Status: review required

/kaggle/working/lorma_plus_a4/results.json
Experiment: Corrected LoRMA+ on A4 | Macro-F1: not recorded | Status: review required

/kaggle/working/phase4_results/a3_distillation_results.json
Experiment: A3 frozen-backbone knowledge distillation from A4 teacher | Macro-F1: 0.3338 | Status: unsuccessful: predicts class 1 for every development record

/kaggle/working/phase4_a1_a2_pilot_results.json
Experiment: A1 frozen backbone classification head pilot | Macro-F1: 0.3338 | Status: unsuccessful: predicts class 1 for every development record
E

In [220]:

import json
from pathlib import Path

files = [
    "/kaggle/working/phase4_experiment_results.json",
    "/kaggle/working/a4_adapter_pilot_results.json",
    "/kaggle/working/lorma_plus_a4/results.json",
    "/kaggle/working/phase4_results/a3_distillation_results.json",
    "/kaggle/working/phase4_a1_a2_pilot_results.json",
]

for filename in files:
    path = Path(filename)
    print(f"\n{'=' * 60}\n{path.name}")

    if not path.exists():
        print("FILE MISSING")
        continue

    data = json.loads(path.read_text())
    records = data if isinstance(data, list) else [data]

    for record in records:
        print(json.dumps(record, indent=2, default=str))



phase4_experiment_results.json
{
  "experiment": "A4 frozen encoder - standard loss",
  "accuracy": 0.5258,
  "macro_f1": 0.4487,
  "macro_precision": 0.5607,
  "macro_recall": 0.5266,
  "trainable_parameters": 1026
}
{
  "experiment": "A4 frozen encoder - class-weighted loss",
  "accuracy": 0.5327,
  "macro_f1": 0.5235,
  "trainable_parameters": 1026
}
{
  "experiment": "A4 BitFit",
  "accuracy": 0.499,
  "macro_f1": 0.353,
  "macro_precision": 0.4995,
  "macro_recall": 0.5,
  "trainable_parameters": 84994,
  "training_loss": 0.6891,
  "training_accuracy": 0.5849
}

a4_adapter_pilot_results.json
{
  "experiment": "A4 multiplicative low-rank adapters pilot",
  "checkpoint": "A4 step 22000",
  "training_examples": 2048,
  "epochs": 1,
  "batch_size": 8,
  "learning_rate": 5e-05,
  "trainable_parameters": 99330,
  "development_examples": 3505,
  "development_macro_f1_threshold_0.50": 0.4806,
  "development_macro_f1_threshold_0.55": 0.5314,
  "selected_threshold": 0.55,
  "threshold_sele

In [221]:

import re
from pathlib import Path

notebook_path = Path("/kaggle/working/phase4_cap.ipynb")

if not notebook_path.exists():
    print("Notebook not found at:", notebook_path)
    print("Current working notebooks/files:")
    for p in Path("/kaggle/working").glob("*.ipynb"):
        print(p)
else:
    import nbformat

    nb = nbformat.read(notebook_path, as_version=4)

    terms = [
        "BitFit", "LoRMA", "AdaLoRA", "adapter",
        "requires_grad", "distill", "freeze",
        "unfreeze", "optimizer", "chunk", "pool"
    ]

    for i, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue

        hits = [term for term in terms
                if re.search(re.escape(term), cell.source, re.IGNORECASE)]

        if hits:
            print(f"\n--- Cell {i} | matches: {', '.join(hits)} ---")
            print(cell.source[:1800])


Notebook not found at: /kaggle/working/phase4_cap.ipynb
Current working notebooks/files:


In [222]:

import re
from pathlib import Path

notebook_path = Path("/kaggle/working/phase4_cap.ipynb")

if not notebook_path.exists():
    print("Notebook not found at:", notebook_path)
    print("Current working notebooks/files:")
    for p in Path("/kaggle/working").glob("*.ipynb"):
        print(p)
else:
    import nbformat

    nb = nbformat.read(notebook_path, as_version=4)

    terms = [
        "BitFit", "LoRMA", "AdaLoRA", "adapter",
        "requires_grad", "distill", "freeze",
        "unfreeze", "optimizer", "chunk", "pool"
    ]

    for i, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue

        hits = [term for term in terms
                if re.search(re.escape(term), cell.source, re.IGNORECASE)]

        if hits:
            print(f"\n--- Cell {i} | matches: {', '.join(hits)} ---")
            print(cell.source[:1800])


Notebook not found at: /kaggle/working/phase4_cap.ipynb
Current working notebooks/files:


In [223]:

from pathlib import Path

for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    print(f"\n=== Searching {root} ===")
    count = 0

    for path in root.rglob("*.ipynb"):
        print(path)
        count += 1

        if count >= 40:
            print("Showing first 40 notebooks only.")
            break

    if count == 0:
        print("No notebooks found.")



=== Searching /kaggle/working ===
/kaggle/working/.virtual_documents/__notebook_source__.ipynb

=== Searching /kaggle/input ===
/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a4a4a4.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a2a2a2.ipynb
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
/kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb
/kaggle/input/datasets/aq1de3/p2p2p2/phase-2 (1).ipynb


In [225]:

from pathlib import Path
import re

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")

source = path.read_text(encoding="utf-8")
print("File size:", len(source), "characters")
print("First line:", repr(source.splitlines()[0] if source else ""))

terms = [
    "BitFit", "LoRMA", "AdaLoRA", "adapter",
    "distill", "requires_grad", "freeze",
    "unfreeze", "optimizer", "chunk", "pool",
    "full fine-tuning", "partial"
]

lines = source.splitlines()

for i, line in enumerate(lines):
    hits = [term for term in terms
            if re.search(re.escape(term), line, re.IGNORECASE)]

    if hits:
        print(f"Line {i + 1} [{', '.join(hits)}]:")
        print(line[:250])


File size: 185017 characters
First line: ''
Line 158 [adapter]:
    "adapter_config.json",
Line 159 [adapter]:
    "adapter_model.safetensors",
Line 801 [LoRMA]:
        "GQA", "LORMA", "encoder_layers", "decoder_layers",
Line 957 [optimizer]:
                       "optimizer_state_dict"):
Line 1032 [optimizer]:
                     "optimizer_state_dict")
Line 2558 [optimizer]:
    elif key not in ("model_state_dict", "optimizer_state_dict"):
Line 3602 [chunk]:
def chunk_count(token_count, chunk_size=1024, overlap=128):
Line 3603 [chunk]:
    if token_count <= chunk_size:
Line 3605 [chunk]:
    stride = chunk_size - overlap
Line 3606 [chunk]:
    return 1 + (token_count - chunk_size + stride - 1) // stride
Line 3614 [chunk]:
    counts = [chunk_count(n) for n in lengths]
Line 3617 [chunk]:
    print("Average chunks/document:", round(sum(counts) / len(counts), 2))
Line 3618 [chunk]:
    print("Maximum chunks/document:", max(counts))
Line 3619 [chunk]:
    print("Documents needing mult

In [226]:

from pathlib import Path

source = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
).read_text(encoding="utf-8")

lines = source.splitlines()

for i, line in enumerate(lines):
    if any(term in line.lower() for term in [
        "encoder_layers", "unfreeze", "layer-wise",
        "partial fine", "requires_grad = true"
    ]):
        print(f"\nLine {i + 1}:")
        print("\n".join(lines[max(0, i - 2):min(len(lines), i + 4)]))



Line 801:
        "class ", "def create_", "architecture", "state_dict",
        "D_MODEL", "N_LAYERS", "N_HEADS", "PREFIX_LEN",
        "GQA", "LORMA", "encoder_layers", "decoder_layers",
        "model.load_state_dict", "torch.save"
    ]


Line 1036:
    })

    for prefix in ("blocks.", "layers.", "encoder_layers.", "decoder_layers."):
        indices = sorted({
            int(k.split(".")[1])
            for k in state

Line 1314:
                "self.final_norm", "self.ln_f",
                "self.output_bias", "self.lm_head",
                "self.encoder_layers", "self.decoder_layers",
                "self.encoder_pos", "self.decoder_pos"
            ])
        ):

Line 1338:
    "d_model = 512",
    "self.layers = nn.ModuleList",
    "self.encoder_layers = nn.ModuleList",
    "self.decoder_layers = nn.ModuleList",
    "self.output_bias",
]

Line 1377:
    "N_Q_HEADS", "N_KV_HEADS", "nn.ModuleList",
    "nn.MultiheadAttention", "q_proj", "k_proj", "v_proj",
    "encoder_lay

In [227]:

import torch

print("A4 model:", type(model).__name__)
print("Encoder layers:", len(model.encoder_layers))

for i, layer in enumerate(model.encoder_layers):
    total = sum(p.numel() for p in layer.parameters())
    trainable = sum(
        p.numel() for p in layer.parameters() if p.requires_grad
    )
    print(
        f"Layer {i}: total={total:,}, "
        f"currently_trainable={trainable:,}"
    )

print(
    "Total A4 trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)
print(
    "GPU available:",
    torch.cuda.is_available()
)


A4 model: A4EncoderDecoder
Encoder layers: 6
Layer 0: total=3,168,768, currently_trainable=0
Layer 1: total=3,168,768, currently_trainable=0
Layer 2: total=3,168,768, currently_trainable=0
Layer 3: total=3,168,768, currently_trainable=0
Layer 4: total=3,168,768, currently_trainable=0
Layer 5: total=3,168,768, currently_trainable=0
Total A4 trainable parameters: 0
GPU available: True


In [228]:

import torch

print("A4 model:", type(model).__name__)
print("Device:", next(model.parameters()).device)

print("\nEncoder attention projections:")
for i, layer in enumerate(model.encoder_layers):
    print(
        f"Layer {i}:",
        "q_proj =", type(layer.attn.q_proj).__name__,
        "| k_proj =", type(layer.attn.k_proj).__name__,
        "| v_proj =", type(layer.attn.v_proj).__name__,
        "| out_proj =", type(layer.attn.out_proj).__name__,
    )

print("\nClassification head:")
print("Exists:", "classification_head" in globals())

if "classification_head" in globals():
    print("Type:", type(classification_head).__name__)
    print(
        "Device:",
        next(classification_head.parameters()).device
    )
    print(
        "Trainable parameters:",
        sum(p.numel() for p in classification_head.parameters()
            if p.requires_grad)
    )

print("\nCUDA memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("CUDA memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")


A4 model: A4EncoderDecoder
Device: cuda:0

Encoder attention projections:
Layer 0: q_proj = PaperLoRMAPlusLinear | k_proj = Linear | v_proj = PaperLoRMAPlusLinear | out_proj = Linear
Layer 1: q_proj = PaperLoRMAPlusLinear | k_proj = Linear | v_proj = PaperLoRMAPlusLinear | out_proj = Linear
Layer 2: q_proj = PaperLoRMAPlusLinear | k_proj = Linear | v_proj = PaperLoRMAPlusLinear | out_proj = Linear
Layer 3: q_proj = PaperLoRMAPlusLinear | k_proj = Linear | v_proj = PaperLoRMAPlusLinear | out_proj = Linear
Layer 4: q_proj = PaperLoRMAPlusLinear | k_proj = Linear | v_proj = PaperLoRMAPlusLinear | out_proj = Linear
Layer 5: q_proj = PaperLoRMAPlusLinear | k_proj = Linear | v_proj = PaperLoRMAPlusLinear | out_proj = Linear

Classification head:
Exists: True
Type: Linear
Device: cuda:0
Trainable parameters: 0

CUDA memory allocated: 1.08 GB
CUDA memory reserved: 3.83 GB


In [229]:

from pathlib import Path
import torch

checkpoint_path = Path(
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt"
)

print("Original A4 checkpoint exists:", checkpoint_path.exists())
if checkpoint_path.exists():
    print("Checkpoint size (MB):",
          round(checkpoint_path.stat().st_size / 1024**2, 2))

lorma_path = Path(
    "/kaggle/working/lorma_plus_a4/lorma_plus_a4_checkpoint.pt"
)

print("\nSaved LoRMA+ checkpoint exists:", lorma_path.exists())
if lorma_path.exists():
    print("LoRMA+ checkpoint size (MB):",
          round(lorma_path.stat().st_size / 1024**2, 2))
    saved = torch.load(lorma_path, map_location="cpu", weights_only=False)
    print("Saved checkpoint keys:", list(saved.keys()))


Original A4 checkpoint exists: True
Checkpoint size (MB): 367.79

Saved LoRMA+ checkpoint exists: True
LoRMA+ checkpoint size (MB): 270.52
Saved checkpoint keys: ['model_state_dict', 'classification_head_state_dict', 'adapter', 'base_checkpoint', 'rank', 'alpha', 'trainable_parameters']


In [230]:

import torch
from pathlib import Path

checkpoint_path = Path(
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Current model class:", type(model).__name__)
print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))

    state = checkpoint.get("model_state_dict", checkpoint)
    print("State dictionary type:", type(state).__name__)
    print("Number of state entries:", len(state))

    wrapped_keys = [
        key for key in state
        if ".A" in key or ".B" in key or "base.weight" in key
    ]
    print("LoRMA-style keys in original checkpoint:", len(wrapped_keys))

del checkpoint


Current model class: A4EncoderDecoder
Checkpoint type: dict
Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
State dictionary type: OrderedDict
Number of state entries: 260
LoRMA-style keys in original checkpoint: 0


In [231]:

import inspect

print("Model class:", type(model).__name__)
print("Constructor:", inspect.signature(type(model).__init__))
print("Forward method:", inspect.signature(type(model).forward))
print("Checkpoint loading method available:",
      hasattr(model, "load_state_dict"))


Model class: A4EncoderDecoder
Constructor: (self)
Forward method: (self, encoder_ids, decoder_ids)
Checkpoint loading method available: True


In [233]:

import torch
from pathlib import Path

checkpoint_path = Path(
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt"
)

# Preserve the current model's device.
device = next(model.parameters()).device

# Create a fresh model and load the original Phase 1 checkpoint.
fresh_a4 = type(model)().to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

result = fresh_a4.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

fresh_a4.eval()
model = fresh_a4

print("Fresh A4 restored successfully.")
print("Strict loading:", result)
print("Encoder layers:", len(model.encoder_layers))
print("Model device:", next(model.parameters()).device)
print("LoRMA wrappers remaining:", sum(
    isinstance(
        getattr(layer.attn, name),
        PaperLoRMAPlusLinear
    )
    for layer in model.encoder_layers
    for name in ("q_proj", "v_proj")
))
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))


Fresh A4 restored successfully.
Strict loading: <All keys matched successfully>
Encoder layers: 6
Model device: cuda:0
LoRMA wrappers remaining: 0
Trainable parameters: 70789120


In [234]:

# Freeze all original A4 parameters first.
for parameter in model.parameters():
    parameter.requires_grad = False

# Unfreeze the final two encoder layers.
for layer in model.encoder_layers[4:6]:
    for parameter in layer.parameters():
        parameter.requires_grad = True

# Keep the final encoder normalization trainable.
for parameter in model.encoder_norm.parameters():
    parameter.requires_grad = True

print("Partial fine-tuning configuration")
for i, layer in enumerate(model.encoder_layers):
    total = sum(p.numel() for p in layer.parameters())
    trainable = sum(
        p.numel() for p in layer.parameters()
        if p.requires_grad
    )
    print(f"Layer {i}: {trainable:,} / {total:,} trainable")

print("Total trainable model parameters:",
      sum(p.numel() for p in model.parameters()
          if p.requires_grad))


Partial fine-tuning configuration
Layer 0: 0 / 3,152,384 trainable
Layer 1: 0 / 3,152,384 trainable
Layer 2: 0 / 3,152,384 trainable
Layer 3: 0 / 3,152,384 trainable
Layer 4: 3,152,384 / 3,152,384 trainable
Layer 5: 3,152,384 / 3,152,384 trainable
Total trainable model parameters: 6305792


In [235]:

import torch
import torch.nn as nn

# Enable the existing classification head.
classification_head = classification_head.to(device)
for parameter in classification_head.parameters():
    parameter.requires_grad = True

# Collect trainable parameters from the encoder and classifier.
trainable_parameters = [
    p for p in model.parameters() if p.requires_grad
] + [
    p for p in classification_head.parameters() if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=1e-5,
    weight_decay=0.01
)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Trainable encoder parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Trainable classifier parameters:",
      sum(p.numel() for p in classification_head.parameters()
          if p.requires_grad))
print("Total trainable parameters:",
      sum(p.numel() for p in trainable_parameters))


Optimizer: AdamW
Learning rate: 1e-05
Trainable encoder parameters: 6305792
Trainable classifier parameters: 1026
Total trainable parameters: 6306818


In [236]:

print("Training records:", len(train_records))
print("Development records:", len(dev_records))
print("Batch function available:", callable(prepare_batch))

if train_records:
    print("Training record keys:", list(train_records[0].keys()))
    print("Training label:", train_records[0].get("label"))
    print("Text type:", type(train_records[0].get("text")).__name__)

batch = prepare_batch(train_records[:2], max_length=128)
print("\nBatch outputs:")
for item in batch:
    print(type(item).__name__, getattr(item, "shape", None))

input_ids, attention_mask, labels = batch
print("Input IDs dtype:", input_ids.dtype)
print("Attention mask dtype:", attention_mask.dtype)
print("Labels dtype:", labels.dtype)
print("Labels:", labels.tolist())


Training records: 37387
Development records: 3505
Batch function available: True
Training record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
Training label: 0
Text type: str

Batch outputs:
Tensor torch.Size([2, 128])
Tensor torch.Size([2, 128])
Tensor torch.Size([2])
Input IDs dtype: torch.int64
Attention mask dtype: torch.int64
Labels dtype: torch.int64
Labels: [0, 0]


In [237]:

model.eval()

input_ids, attention_mask, labels = prepare_batch(
    train_records[:2],
    max_length=128
)

with torch.no_grad():
    x = model.token_embedding(input_ids)
    positions = torch.arange(input_ids.size(1), device=device)
    x = x + model.encoder_pos(positions)[None, :, :]
    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    encoded = model.encoder_norm(x)

    mask = attention_mask.unsqueeze(-1).to(encoded.dtype)
    pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    logits = classification_head(pooled)

print("Encoded shape:", tuple(encoded.shape))
print("Pooled shape:", tuple(pooled.shape))
print("Logits shape:", tuple(logits.shape))
print("Labels shape:", tuple(labels.shape))
print("Logits finite:", torch.isfinite(logits).all().item())


Encoded shape: (2, 128, 512)
Pooled shape: (2, 512)
Logits shape: (2, 2)
Labels shape: (2,)
Logits finite: True


In [238]:

import torch.nn.functional as F

model.train()
classification_head.train()

input_ids, attention_mask, labels = prepare_batch(
    train_records[:8],
    max_length=128
)

optimizer.zero_grad(set_to_none=True)

x = model.token_embedding(input_ids)
positions = torch.arange(input_ids.size(1), device=device)
x = x + model.encoder_pos(positions)[None, :, :]
x = model.dropout(x)

for layer in model.encoder_layers:
    x = layer(x)

encoded = model.encoder_norm(x)

mask = attention_mask.unsqueeze(-1).to(encoded.dtype)
pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

logits = classification_head(pooled)
loss = F.cross_entropy(logits, labels)

loss.backward()
torch.nn.utils.clip_grad_norm_(trainable_parameters, 1.0)
optimizer.step()

print("One-batch training test completed.")
print("Loss:", round(loss.item(), 4))
print("Predictions:", logits.argmax(dim=1).detach().cpu().tolist())
print("Labels:", labels.detach().cpu().tolist())
print("Finite loss:", torch.isfinite(loss).item())
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


One-batch training test completed.
Loss: 0.5054
Predictions: [0, 0, 0, 1, 0, 0, 0, 0]
Labels: [0, 0, 0, 0, 0, 0, 1, 0]
Finite loss: True
GPU memory allocated: 1.17 GB


In [239]:

import torch
import torch.nn as nn
from pathlib import Path

checkpoint_path = Path(
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Restore the original Phase 1 A4 model.
model = A4EncoderDecoder().to(device)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.train()

# Freeze all parameters, then unfreeze the final two encoder layers.
for p in model.parameters():
    p.requires_grad = False

for layer in model.encoder_layers[4:6]:
    for p in layer.parameters():
        p.requires_grad = True

for p in model.encoder_norm.parameters():
    p.requires_grad = True

# Fresh classification head.
torch.manual_seed(42)
classification_head = nn.Linear(512, 2).to(device)

trainable_parameters = [
    p for p in model.parameters() if p.requires_grad
] + list(classification_head.parameters())

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=1e-5,
    weight_decay=0.01
)

print("Original checkpoint restored:", True)
print("LoRMA wrappers:", sum(
    isinstance(getattr(layer.attn, name), PaperLoRMAPlusLinear)
    for layer in model.encoder_layers
    for name in ("q_proj", "v_proj")
))
print("Trainable parameters:", sum(p.numel() for p in trainable_parameters))
print("Training examples available:", len(train_records))
print("Development examples available:", len(dev_records))


Original checkpoint restored: True
LoRMA wrappers: 0
Trainable parameters: 6306818
Training examples available: 37387
Development examples available: 3505


In [240]:

import random
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Reproducible shuffled ordering.
random.seed(42)
torch.manual_seed(42)

train_loader_partial = DataLoader(
    train_records,
    batch_size=8,
    shuffle=True,
    collate_fn=lambda records: prepare_batch(records, max_length=128),
    num_workers=0
)

model.train()
classification_head.train()

running_loss = 0.0
examples_seen = 0

for step, (input_ids, attention_mask, labels) in enumerate(
    train_loader_partial, start=1
):
    optimizer.zero_grad(set_to_none=True)

    x = model.token_embedding(input_ids)
    positions = torch.arange(input_ids.size(1), device=device)
    x = x + model.encoder_pos(positions)[None, :, :]
    x = model.dropout(x)

    for layer in model.encoder_layers:
        x = layer(x)

    encoded = model.encoder_norm(x)
    mask = attention_mask.unsqueeze(-1).to(encoded.dtype)
    pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    logits = classification_head(pooled)
    loss = F.cross_entropy(logits, labels)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_parameters, 1.0)
    optimizer.step()

    running_loss += loss.item() * labels.size(0)
    examples_seen += labels.size(0)

    if step % 100 == 0:
        print(
            f"Batch {step}/{len(train_loader_partial)} | "
            f"Examples {examples_seen} | "
            f"Running loss {running_loss / examples_seen:.4f}"
        )

print("\nPartial-fine-tuning epoch finished.")
print("Training examples processed:", examples_seen)
print("Mean training loss:", round(running_loss / examples_seen, 4))


Batch 100/4674 | Examples 800 | Running loss 0.6887
Batch 200/4674 | Examples 1600 | Running loss 0.6890
Batch 300/4674 | Examples 2400 | Running loss 0.6843
Batch 400/4674 | Examples 3200 | Running loss 0.6848
Batch 500/4674 | Examples 4000 | Running loss 0.6827
Batch 600/4674 | Examples 4800 | Running loss 0.6835
Batch 700/4674 | Examples 5600 | Running loss 0.6827
Batch 800/4674 | Examples 6400 | Running loss 0.6826
Batch 900/4674 | Examples 7200 | Running loss 0.6828
Batch 1000/4674 | Examples 8000 | Running loss 0.6821
Batch 1100/4674 | Examples 8800 | Running loss 0.6821
Batch 1200/4674 | Examples 9600 | Running loss 0.6810
Batch 1300/4674 | Examples 10400 | Running loss 0.6809
Batch 1400/4674 | Examples 11200 | Running loss 0.6811
Batch 1500/4674 | Examples 12000 | Running loss 0.6803
Batch 1600/4674 | Examples 12800 | Running loss 0.6808
Batch 1700/4674 | Examples 13600 | Running loss 0.6804
Batch 1800/4674 | Examples 14400 | Running loss 0.6807
Batch 1900/4674 | Examples 15200

In [241]:

import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

model.eval()
classification_head.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for start in range(0, len(dev_records), 8):
        batch_records = dev_records[start:start + 8]

        input_ids, attention_mask, labels = prepare_batch(
            batch_records, max_length=128
        )

        x = model.token_embedding(input_ids)
        positions = torch.arange(input_ids.size(1), device=device)
        x = x + model.encoder_pos(positions)[None, :, :]
        x = model.dropout(x)

        for layer in model.encoder_layers:
            x = layer(x)

        encoded = model.encoder_norm(x)

        mask = attention_mask.unsqueeze(-1).to(encoded.dtype)
        pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        logits = classification_head(pooled)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

print("Development examples:", len(all_labels))
print("Accuracy:", round(accuracy_score(all_labels, all_preds), 4))
print("Macro-F1:", round(f1_score(all_labels, all_preds, average="macro", zero_division=0), 4))
print("Macro precision:", round(precision_score(all_labels, all_preds, average="macro", zero_division=0), 4))
print("Macro recall:", round(recall_score(all_labels, all_preds, average="macro", zero_division=0), 4))
print("Confusion matrix:\n", confusion_matrix(all_labels, all_preds))
print("Predicted class counts:", {
    0: all_preds.count(0),
    1: all_preds.count(1)
})


Development examples: 3505
Accuracy: 0.499
Macro-F1: 0.3329
Macro precision: 0.2495
Macro recall: 0.5
Confusion matrix:
 [[1749    0]
 [1756    0]]
Predicted class counts: {0: 3505, 1: 0}


In [242]:

import os
import json
import torch
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix
)

save_dir = "/kaggle/working/phase4_results"
os.makedirs(save_dir, exist_ok=True)

checkpoint_path = os.path.join(
    save_dir, "a4_partial_finetuning_checkpoint.pt"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "classification_head_state_dict": classification_head.state_dict(),
    "experiment": "A4 partial fine-tuning",
    "max_length": 128,
    "trainable_parameters": 6306818,
    "training_examples": 37387,
    "mean_training_loss": 0.6776,
}, checkpoint_path)

metrics = {
    "experiment": "A4 partial fine-tuning",
    "training_examples": 37387,
    "development_examples": len(all_labels),
    "max_length": 128,
    "mean_training_loss": 0.6776,
    "accuracy": accuracy_score(all_labels, all_preds),
    "macro_f1": f1_score(
        all_labels, all_preds, average="macro", zero_division=0
    ),
    "macro_precision": precision_score(
        all_labels, all_preds, average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        all_labels, all_preds, average="macro", zero_division=0
    ),
    "confusion_matrix": confusion_matrix(
        all_labels, all_preds
    ).tolist(),
    "predicted_class_counts": {
        "0": all_preds.count(0),
        "1": all_preds.count(1),
    },
    "limitation": (
        "Sequence length 128; only a short prefix of each judgment "
        "is represented."
    ),
}

metrics_path = os.path.join(
    save_dir, "a4_partial_finetuning_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Checkpoint saved:", checkpoint_path)
print("Metrics saved:", metrics_path)
print("Macro-F1:", round(metrics["macro_f1"], 4))


Checkpoint saved: /kaggle/working/phase4_results/a4_partial_finetuning_checkpoint.pt
Metrics saved: /kaggle/working/phase4_results/a4_partial_finetuning_metrics.json
Macro-F1: 0.3329


In [243]:

from pathlib import Path

for folder in [
    Path("/kaggle/working/phase4_results"),
    Path("/kaggle/working/lorma_plus_a4"),
]:
    print(f"\n--- {folder} ---")
    if folder.exists():
        for path in sorted(folder.iterdir()):
            if path.is_file():
                print(f"{path.name} ({path.stat().st_size / (1024**2):.2f} MB)")
    else:
        print("Folder not found")



--- /kaggle/working/phase4_results ---
a3_distillation_experiment.pt (197.91 MB)
a3_distillation_results.json (0.00 MB)
a4_partial_finetuning_checkpoint.pt (270.14 MB)
a4_partial_finetuning_metrics.json (0.00 MB)

--- /kaggle/working/lorma_plus_a4 ---
lorma_plus_a4_checkpoint.pt (270.52 MB)
results.json (0.00 MB)


In [244]:

import json
from pathlib import Path

result_files = [
    Path("/kaggle/working/phase4_results/a3_distillation_results.json"),
    Path("/kaggle/working/phase4_results/a4_partial_finetuning_metrics.json"),
    Path("/kaggle/working/lorma_plus_a4/results.json"),
]

for path in result_files:
    print(f"\n{'=' * 60}\n{path.name}")
    if path.exists():
        with open(path, "r") as f:
            data = json.load(f)
        print(json.dumps(data, indent=2))
    else:
        print("File not found")



a3_distillation_results.json
{
  "experiment": "A3 frozen-backbone knowledge distillation from A4 teacher",
  "development_records": 3505,
  "accuracy": 0.501,
  "macro_f1": 0.3338,
  "macro_precision": 0.2505,
  "macro_recall": 0.5,
  "confusion_matrix": [
    [
      0,
      1749
    ],
    [
      0,
      1756
    ]
  ],
  "prediction_counts": {
    "class_0": 0,
    "class_1": 3505
  },
  "probability_std": 0.000553,
  "status": "unsuccessful: predicts class 1 for every development record",
  "caveat": "Frozen student representations did not yield useful class separation."
}

a4_partial_finetuning_metrics.json
{
  "experiment": "A4 partial fine-tuning",
  "training_examples": 37387,
  "development_examples": 3505,
  "max_length": 128,
  "mean_training_loss": 0.6776,
  "accuracy": 0.49900142653352353,
  "macro_f1": 0.3328892272554244,
  "macro_precision": 0.24950071326676176,
  "macro_recall": 0.5,
  "confusion_matrix": [
    [
      1749,
      0
    ],
    [
      1756,
      0

In [245]:

import json
from pathlib import Path

test_path = "/kaggle/input/datasets/aq1de3/testjson/test.json"

with open(test_path, "r") as f:
    test_data = json.load(f)

print("Loaded object type:", type(test_data).__name__)

if isinstance(test_data, list):
    print("Number of test records:", len(test_data))
    print("First record keys:", list(test_data[0].keys()))
    print("First record ID:", test_data[0].get("id"))
    print("First record text length:",
          len(str(test_data[0].get("text", ""))))
elif isinstance(test_data, dict):
    print("Top-level keys:", list(test_data.keys()))
    for key, value in test_data.items():
        if isinstance(value, list):
            print(f"{key}: {len(value)} records")


Loaded object type: list
Number of test records: 1517
First record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
First record ID: 1986_397
First record text length: 14675


In [246]:

import torch

checkpoint_path = (
    "/kaggle/working/lorma_plus_a4/lorma_plus_a4_checkpoint.pt"
)

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", list(ckpt.keys()))
print("Base checkpoint:", ckpt.get("base_checkpoint"))
print("Adapter:", ckpt.get("adapter"))
print("Rank:", ckpt.get("rank"))
print("Alpha:", ckpt.get("alpha"))
print("Trainable parameters:", ckpt.get("trainable_parameters"))

print("\nModel state entries:",
      len(ckpt["model_state_dict"]))
print("Classification head entries:",
      len(ckpt["classification_head_state_dict"]))


Checkpoint keys: ['model_state_dict', 'classification_head_state_dict', 'adapter', 'base_checkpoint', 'rank', 'alpha', 'trainable_parameters']
Base checkpoint: A4 step 22000
Adapter: LoRMA+
Rank: 8
Alpha: 16
Trainable parameters: 99330

Model state entries: 284
Classification head entries: 2


In [247]:

from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
source = source_path.read_text()

for term in [
    "class PaperLoRMAPlusLinear",
    "lorma_plus_a4_checkpoint.pt",
    "def prepare_batch",
]:
    pos = source.find(term)
    print(f"\n--- {term} ---")
    if pos == -1:
        print("Not found")
    else:
        print(source[pos:pos + 1800])



--- class PaperLoRMAPlusLinear ---
class PaperLoRMAPlusLinear(nn.Module):
    """
    LoRMA+ weight transformation:
        W_eff = (I + (alpha/r) * B @ A) @ W0

    B: (out_features, rank)
    A: (rank, out_features)
    """
    def __init__(self, base_layer, rank=8, alpha=16):
        super().__init__()
        self.base = base_layer
        self.rank = rank
        self.scale = alpha / rank

        out_features, in_features = base_layer.weight.shape
        if out_features != in_features:
            raise ValueError(
                "This identity-inflated implementation requires a square weight matrix."
            )

        self.A = nn.Parameter(
            torch.empty(rank, out_features,
                        device=base_layer.weight.device,
                        dtype=base_layer.weight.dtype)
        )
        self.B = nn.Parameter(
            torch.zeros(out_features, rank,
                        device=base_layer.weight.device,
                        dtype=base_lay

In [248]:

import torch
import torch.nn as nn
import torch.nn.functional as F

base_path = (
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt"
)
lorma_path = (
    "/kaggle/working/lorma_plus_a4/lorma_plus_a4_checkpoint.pt"
)

# Load the original A4 backbone
base_ckpt = torch.load(
    base_path, map_location="cpu", weights_only=False
)
model = A4EncoderDecoder().to(device)
model.load_state_dict(base_ckpt["model_state_dict"], strict=True)

# Recreate LoRMA+ wrappers on the same attention projections
for layer in model.encoder_layers:
    layer.attn.q_proj = PaperLoRMAPlusLinear(
        layer.attn.q_proj, rank=8, alpha=16
    )
    layer.attn.v_proj = PaperLoRMAPlusLinear(
        layer.attn.v_proj, rank=8, alpha=16
    )

# Load the trained LoRMA+ parameters
ckpt = torch.load(
    lorma_path, map_location="cpu", weights_only=False
)
result = model.load_state_dict(
    ckpt["model_state_dict"], strict=True
)

classification_head = nn.Linear(512, 2).to(device)
classification_head.load_state_dict(
    ckpt["classification_head_state_dict"], strict=True
)

model.eval()
classification_head.eval()

print("Checkpoint loaded:", result)
print("LoRMA+ wrappers:", sum(
    isinstance(layer.attn.q_proj, PaperLoRMAPlusLinear)
    + isinstance(layer.attn.v_proj, PaperLoRMAPlusLinear)
    for layer in model.encoder_layers
))
print("Model device:", next(model.parameters()).device)
print("Model and classifier ready for inference.")


Checkpoint loaded: <All keys matched successfully>
LoRMA+ wrappers: 12
Model device: cuda:0
Model and classifier ready for inference.


In [249]:

from pathlib import Path

source = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
).read_text()

needle = 'lorma_plus_a4_checkpoint.pt'
pos = source.find(needle)

if pos == -1:
    print("Checkpoint save reference not found.")
else:
    start = max(0, pos - 9000)
    end = min(len(source), pos + 2500)
    print(source[start:end])


features != in_features:
            raise ValueError(
                "This identity-inflated implementation requires a square weight matrix."
            )

        self.A = nn.Parameter(
            torch.empty(rank, out_features,
                        device=base_layer.weight.device,
                        dtype=base_layer.weight.dtype)
        )
        self.B = nn.Parameter(
            torch.zeros(out_features, rank,
                        device=base_layer.weight.device,
                        dtype=base_layer.weight.dtype)
        )

        nn.init.normal_(self.A, mean=0.0, std=0.02)

        for param in self.base.parameters():
            param.requires_grad = False

    def forward(self, x):
        # Effective W = (I + scale * B @ A) @ W0
        delta = (self.B @ self.A) @ self.base.weight
        effective_weight = self.base.weight + self.scale * delta
        return F.linear(x, effective_weight, self.base.bias)


# Test the wrapper independently on a copied linear

In [250]:

import json
import torch
from pathlib import Path
from tqdm.auto import tqdm

# Read test data
test_path = "/kaggle/input/datasets/aq1de3/testjson/test.json"
with open(test_path, "r") as f:
    test_data = json.load(f)

# Tokenizer special tokens
pad_id = tokenizer.token_to_id("[PAD]")
bos_id = tokenizer.token_to_id("[BOS]")
eos_id = tokenizer.token_to_id("[EOS]")

max_length = 1024
batch_size = 8
threshold = 0.50

all_probabilities = []
model.eval()
classification_head.eval()

with torch.no_grad():
    for start in tqdm(range(0, len(test_data), batch_size)):
        batch_records = test_data[start:start + batch_size]
        batch_ids = []

        for record in batch_records:
            token_ids = tokenizer.encode(record["text"]).ids
            token_ids = token_ids[:max_length - 2]
            token_ids = [bos_id] + token_ids + [eos_id]
            token_ids += [pad_id] * (max_length - len(token_ids))
            batch_ids.append(token_ids)

        input_ids = torch.tensor(
            batch_ids, dtype=torch.long, device=device
        )
        attention_mask = (input_ids != pad_id).long()

        positions = torch.arange(max_length, device=device)
        x = model.token_embedding(input_ids)
        x = x + model.encoder_pos(positions)[None, :, :]
        x = model.dropout(x)

        for layer in model.encoder_layers:
            x = layer(x)

        encoded = model.encoder_norm(x)
        mask = attention_mask.unsqueeze(-1).to(encoded.dtype)
        pooled = (
            (encoded * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1)
        )

        logits = classification_head(pooled)
        probabilities = torch.softmax(logits, dim=1)[:, 1]
        all_probabilities.extend(probabilities.cpu().tolist())

# Create the required JSON list
submission = [
    {
        "id": record["id"],
        "pred": int(probability >= threshold)
    }
    for record, probability in zip(test_data, all_probabilities)
]

# Validate before saving
assert len(submission) == len(test_data) == 1517
assert len({item["id"] for item in submission}) == len(submission)
assert all(set(item.keys()) == {"id", "pred"} for item in submission)
assert all(item["pred"] in (0, 1) for item in submission)

output_path = "/kaggle/working/submission.json"
with open(output_path, "w") as f:
    json.dump(submission, f, indent=2)

print("Saved:", output_path)
print("Number of predictions:", len(submission))
print("Class counts:", {
    0: sum(item["pred"] == 0 for item in submission),
    1: sum(item["pred"] == 1 for item in submission)
})
print("First 3 predictions:", submission[:3])
print("File size (bytes):", Path(output_path).stat().st_size)


  0%|          | 0/190 [00:00<?, ?it/s]

Saved: /kaggle/working/submission.json
Number of predictions: 1517
Class counts: {0: 314, 1: 1203}
First 3 predictions: [{'id': '1986_397', 'pred': 1}, {'id': '1977_183', 'pred': 1}, {'id': '1993_98', 'pred': 1}]
File size (bytes): 67830


In [251]:

import json
import zipfile
from pathlib import Path

working = Path("/kaggle/working")
source = working / "submission.json"
prediction = working / "prediction.json"
archive = working / "submission_fixed.zip"

# Confirm the existing predictions are valid JSON
with open(source, "r") as f:
    data = json.load(f)

assert len(data) == 1517
assert all(set(row) == {"id", "pred"} for row in data)
assert all(row["pred"] in (0, 1) for row in data)

# Save with the exact required filename
with open(prediction, "w") as f:
    json.dump(data, f, indent=2)

# ZIP with prediction.json directly at the archive root
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(prediction, arcname="prediction.json")

# Verify archive contents and filename
with zipfile.ZipFile(archive, "r") as z:
    print("Archive contents:", z.namelist())
    assert "prediction.json" in z.namelist()
    assert "/" not in z.namelist()[0]

print("Fixed archive:", archive)
print("Prediction count:", len(data))
print("Class counts:", {
    0: sum(row["pred"] == 0 for row in data),
    1: sum(row["pred"] == 1 for row in data)
})


Archive contents: ['prediction.json']
Fixed archive: /kaggle/working/submission_fixed.zip
Prediction count: 1517
Class counts: {0: 314, 1: 1203}


In [1]:

import os

for root, dirs, files in os.walk("/kaggle/input"):
    matched = [
        f for f in files
        if (
            "train.json" in f.lower()
            or "dev.json" in f.lower()
            or "test.json" in f.lower()
            or f.endswith(".pt")
            or "tokenizer.json" in f.lower()
        )
    ]

    for f in matched:
        print(os.path.join(root, f))


/kaggle/input/datasets/aq1de3/trainjson/train.json
/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
/kaggle/input/datasets/aq1de3/devjson/dev.json
/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt
/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt
/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
/kaggle/input/datasets/aq1de3/testjson/test.json
/kaggle/input/datasets/aq1de3/phase3/CL_dev.json
/kaggle/input/datasets/aq1de3/phase3/CL_test.json
/kaggle/input/datasets/aq1de3/phase3/CL_train.json
/kaggle/input/datasets/aq1de3/phase3/IT_dev.json
/kaggle/input/datasets/aq1de3/phase3/IT_test.json
/kaggle/input/datasets/aq1de3/phase3/IT_train.json


In [2]:

import json
import os

BASE = "/kaggle/input/datasets/aq1de3"

paths = {
    "train": f"{BASE}/trainjson/train.json",
    "dev": f"{BASE}/devjson/dev.json",
    "test": f"{BASE}/testjson/test.json",
}

for split, path in paths.items():
    print(f"\n{'=' * 20} {split.upper()} {'=' * 20}")
    
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Data type:", type(data).__name__)

    if isinstance(data, dict):
        print("Number of entries:", len(data))
        for key in list(data)[:2]:
            print("Example ID:", key)
            print("Example:", str(data[key])[:1200])

    elif isinstance(data, list):
        print("Number of entries:", len(data))
        for item in data[:2]:
            print("Example:", str(item)[:1200])



==================== TRAIN ====================
Data type: list
Number of entries: 37387
Example: {'id': '2020_1', 'text': ' Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09.12.2015 passed by the Division Bench of the High Court 1 dismissing S.T. No.192 of 2014 and other companynected matters and thereby affirming the decision of the Single Judge of the High Court passed on 12.03.2014 in Writ Petition No.20650  W  of 2013 which in turn had found Sections 8, 10, 11 and 12 of the West Bengal Madrasah Service Commission Act, 2008  the Commission Act, for short  to be ultra vires. 1 The High Court of Judicature at Calcutta Civil Appeal No.5808 of 2017 SK. MD. Rafique vs. Managing Committee, companytai Rahamania High Madrasah and Others The aforementioned Writ Petition No.20650 W  of 2013 was filed by the Managing Committee of Contai Rahmania High Madrasah challenging validity of Sections 8, 10, 11 and 12 of the Commission Act submitting, inter alia, that by 

In [4]:

import torch
import os

checkpoint_paths = [
    f"{BASE}/a4-step-22000-pt/a4_step_22000.pt",
    f"{BASE}/a2-step-9000-pt/a2_step_9000.pt",
    f"{BASE}/step-0001000-pt/step_0001000.pt",
    f"{BASE}/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
]

for path in checkpoint_paths:
    print(f"\n{'=' * 60}")
    print("File:", os.path.basename(path))
    print("Size (MB):", round(os.path.getsize(path) / (1024**2), 2))

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=True)
        print("Object type:", type(ckpt).__name__)

        if isinstance(ckpt, dict):
            print("Top-level keys:", list(ckpt.keys())[:30])

            state = ckpt.get("model_state_dict", ckpt.get("model", ckpt))
            if isinstance(state, dict):
                tensors = [
                    (k, v.shape, v.dtype)
                    for k, v in state.items()
                    if torch.is_tensor(v)
                ]
                print("Tensor count:", len(tensors))
                print("First 15 tensors:")
                for item in tensors[:15]:
                    print(item)
    except Exception as e:
        print("Inspection error:", repr(e))



File: a4_step_22000.pt
Size (MB): 367.79
Object type: dict
Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
Tensor count: 260
First 15 tensors:
('token_embedding.weight', torch.Size([50000, 512]), torch.float32)
('encoder_pos.weight', torch.Size([1024, 512]), torch.float32)
('decoder_pos.weight', torch.Size([1024, 512]), torch.float32)
('encoder_layers.0.norm1.weight', torch.Size([512]), torch.float32)
('encoder_layers.0.norm1.bias', torch.Size([512]), torch.float32)
('encoder_layers.0.attn.q_proj.weight', torch.Size([512, 512]), torch.float32)
('encoder_layers.0.attn.q_proj.bias', torch.Size([512]), torch.float32)
('encoder_layers.0.attn.k_proj.weight', torch.Size([512, 512]), torch.float32)
('encoder_layers.0.attn.k_proj.bias', torch.Size([512]), torch.float32)
('encoder_layers.0.attn.v_proj.weight', torch.Size([512, 512]), torch.float32)
('encoder_layers.0.attn.v_proj.bias', torch.Size([512]), torch.float32)
('encoder_layers.0.att

In [5]:

import json
from collections import Counter
from tokenizers import Tokenizer

BASE = "/kaggle/input/datasets/aq1de3"

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train = load_json(f"{BASE}/trainjson/train.json")
dev = load_json(f"{BASE}/devjson/dev.json")
test = load_json(f"{BASE}/testjson/test.json")

print("TRAIN KEYS:", list(train[0].keys()))
print("TRAIN SAMPLE (excluding full text):")
for k, v in train[0].items():
    print(k, repr(v)[:500] if k != "text" else f"<text length={len(v)}>")

print("\nDEV KEYS:", list(dev[0].keys()))
print("DEV SAMPLE:")
for k, v in dev[0].items():
    print(k, repr(v)[:500] if k != "text" else f"<text length={len(v)}>")

print("\nTEST KEYS:", list(test[0].keys()))
print("TEST SAMPLE:")
for k, v in test[0].items():
    print(k, repr(v)[:300] if k != "text" else f"<text length={len(v)}>")

print("\nTOKENIZER INSPECTION")
tok = Tokenizer.from_file(f"{BASE}/tokenizer-json/tokenizer.json")
print("Vocabulary size:", tok.get_vocab_size())
print("Sample encoding:", tok.encode("The court considered the relevant statute.").ids)

print("\nCHECKPOINT METADATA")
for name, path in [
    ("A4", f"{BASE}/a4-step-22000-pt/a4_step_22000.pt"),
    ("A2", f"{BASE}/a2-step-9000-pt/a2_step_9000.pt"),
    ("A1", f"{BASE}/step-0001000-pt/step_0001000.pt"),
    ("Architecture1", f"{BASE}/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt"),
]:
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    print(name, {
        k: ckpt.get(k)
        for k in ["step", "tokens_processed", "tokens_seen",
                  "architecture", "parameters", "context_length",
                  "hidden_size", "num_layers", "num_heads"]
        if k in ckpt
    })

# Try to identify likely statute-label fields without assuming a schema.
print("\nPOSSIBLE LABEL FIELDS")
for key in train[0]:
    if key != "text":
        values = [row.get(key) for row in train[:1000]]
        print(key, "types:", Counter(type(v).__name__ for v in values))
        print("Example values:", [repr(v)[:200] for v in values[:3]])


TRAIN KEYS: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
TRAIN SAMPLE (excluding full text):
id '2020_1'
text <text length=273042>
label 0
expert_1 None
expert_2 None
expert_3 None
expert_4 None
expert_5 None

DEV KEYS: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
DEV SAMPLE:
id '1989_75'
text <text length=34774>
label 0
expert_1 None
expert_2 None
expert_3 None
expert_4 None
expert_5 None

TEST KEYS: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
TEST SAMPLE:
id '1986_397'
text <text length=14675>
label 1
expert_1 None
expert_2 None
expert_3 None
expert_4 None
expert_5 None

TOKENIZER INSPECTION
Vocabulary size: 50000
Sample encoding: [550, 777, 1266, 238, 1658, 4957, 18]

CHECKPOINT METADATA
A4 {'step': 22000, 'tokens_processed': 180096704}
A2 {'step': 9000, 'tokens_processed': 73728000}
A1 {'step': 1000, 'tokens_seen': 512000, 'architecture': 'A3_prefix_lm_hybrid', 'par

In [6]:

from collections import Counter

for name, data in [("train", train), ("dev", dev), ("test", test)]:
    print(f"\n{name.upper()}")
    print("Total records:", len(data))
    print("Field names:", list(data[0].keys()))

    if name != "test":
        print("Label distribution:", Counter(
            str(row.get("label")) for row in data
        ))
        print("Records with a labels field:",
              sum("labels" in row for row in data))
        print("Records with a label field:",
              sum("label" in row for row in data))

print("\nDATASET DIRECTORIES")
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "Phase_5_LSI" in root or "phase_5" in root.lower():
        print(root, files)



TRAIN
Total records: 37387
Field names: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
Label distribution: Counter({'0': 22067, '1': 15320})
Records with a labels field: 0
Records with a label field: 37387

DEV
Total records: 3505
Field names: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
Label distribution: Counter({'1': 1756, '0': 1749})
Records with a labels field: 0
Records with a label field: 3505

TEST
Total records: 1517
Field names: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']

DATASET DIRECTORIES


In [2]:
from pathlib import Path

base = Path("/kaggle/input")

for name in ["traintr", "devdev", "testtest"]:
    print(f"\n{'='*15} {name} {'='*15}")

    matches = [
        p for p in base.rglob("*")
        if p.is_dir() and p.name.lower() == name
    ]

    if matches:
        for folder in matches:
            print("Folder:", folder)
            for file in folder.rglob("*"):
                if file.is_file():
                    print("  ", file)
    else:
        print("Folder not found by name. Searching JSON files:")
        for file in base.rglob("*.json"):
            if name in str(file).lower():
                print("  ", file)


=============== traintr ===============
Folder: /kaggle/input/datasets/aq1de3/traintr
   /kaggle/input/datasets/aq1de3/traintr/train (2).json

=============== devdev ===============
Folder: /kaggle/input/datasets/aq1de3/devdev
   /kaggle/input/datasets/aq1de3/devdev/dev (2).json

=============== testtest ===============
Folder: /kaggle/input/datasets/aq1de3/testtest
   /kaggle/input/datasets/aq1de3/testtest/test (3).json


In [3]:
import json
from pathlib import Path
from collections import Counter

paths = {
    "train": "/kaggle/input/datasets/aq1de3/traintr/train (2).json",
    "dev":   "/kaggle/input/datasets/aq1de3/devdev/dev (2).json",
    "test":  "/kaggle/input/datasets/aq1de3/testtest/test (3).json",
}

phase5_data = {}

for split, path in paths.items():
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    phase5_data[split] = data

    print(f"\n{'='*20} {split.upper()} {'='*20}")
    print("Records:", len(data))
    print("Fields:", list(data[0].keys()) if data else "EMPTY")

    if data:
        row = data[0]
        for key, value in row.items():
            if isinstance(value, str):
                print(f"{key}: {value[:250]!r}")
            else:
                print(f"{key}: {value}")

        print("\nField value counts:")
        for key in data[0]:
            if key != "text":
                print(key, Counter(str(r.get(key)) for r in data[:1000]).most_common(10))


==================== TRAIN ====================
Records: 42750
Fields: ['id', 'text', 'labels']
id: '1000008'
text: '(a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stated thus: With the growth of industry, commerceand trade in and around the city of Mumbai which generatessubstantial quantity of we'
labels: [69, 9, 3]

Field value counts:
id [('1000008', 1), ('100009671', 1), ('1000196', 1), ('100120460', 1), ('100122782', 1), ('10013032', 1), ('100130788', 1), ('100132633', 1), ('10013754', 1), ('100138595', 1)]
labels [('[5]', 19), ('[34]', 19), ('[61]', 12), ('[77]', 10), ('[39]', 10), ('[94]', 8), ('[66]', 8), ('[22]', 7), ('[54]', 7), ('[95]', 7)]

==================== DEV ====================
Records: 10181
Fields: ['id', 'text', 'labels']
id: '1000139'
text: 'Dr. <ENTITY>, J. The accused persons are the <ENTITY> andbrother respectively of the deceased. The incident purportedly took place on 3.1.1990.Though the 

In [4]:
from collections import Counter
import numpy as np

train = phase5_data["train"]
dev = phase5_data["dev"]
test = phase5_data["test"]

# Basic label statistics
all_labels = [
    int(label)
    for row in train + dev
    for label in row["labels"]
]

label_ids = sorted(set(all_labels))
label_counts = Counter(
    int(label)
    for row in train
    for label in row["labels"]
)

print("Unique labels in train + dev:", len(label_ids))
print("Minimum label ID:", min(label_ids))
print("Maximum label ID:", max(label_ids))
print("Labels per training case:")
print(Counter(len(row["labels"]) for row in train))
print("Labels per validation case:")
print(Counter(len(row["labels"]) for row in dev))

print("\nMost frequent training labels:")
print(label_counts.most_common(20))

print("\nDocument character lengths:")
for name, data in phase5_data.items():
    lengths = [len(row["text"]) for row in data]
    print(
        name,
        "median =", int(np.median(lengths)),
        "90th percentile =", int(np.percentile(lengths, 90)),
        "maximum =", max(lengths)
    )

print("\nEmpty or duplicate-label examples:")
print("Empty training label lists:",
      sum(not row["labels"] for row in train))
print("Training rows with duplicate labels:",
      sum(len(row["labels"]) != len(set(row["labels"])) for row in train))

Unique labels in train + dev: 100
Minimum label ID: 0
Maximum label ID: 99
Labels per training case:
Counter({1: 9146, 2: 7585, 3: 7045, 4: 5782, 5: 4265, 6: 3283, 7: 2131, 8: 1390, 9: 836, 10: 458, 11: 358, 12: 167, 13: 106, 14: 53, 15: 46, 16: 27, 17: 16, 18: 13, 19: 13, 20: 5, 22: 5, 26: 5, 21: 4, 24: 4, 30: 3, 27: 2, 31: 1, 23: 1})
Labels per validation case:
Counter({2: 1904, 3: 1817, 1: 1544, 4: 1536, 5: 1147, 6: 837, 7: 519, 8: 310, 9: 227, 10: 133, 11: 69, 12: 54, 14: 27, 13: 20, 15: 19, 17: 5, 23: 4, 18: 3, 16: 3, 21: 1, 28: 1, 22: 1})

Most frequent training labels:
[(5, 11731), (34, 10380), (97, 7881), (43, 7728), (39, 7121), (12, 4324), (13, 4136), (10, 4134), (77, 4046), (94, 3835), (14, 3818), (61, 3724), (44, 3660), (51, 3184), (35, 3157), (30, 3009), (45, 2618), (46, 2510), (71, 2303), (54, 2205)]

Document character lengths:
train median = 2717 90th percentile = 16789 maximum = 535964
dev median = 3418 90th percentile = 17900 maximum = 442434
test median = 3368 90th pe

In [5]:

from pathlib import Path

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\nSearching {root}")
    for p in root.rglob("*.py"):
        name = p.name.lower()
        if any(term in name for term in [
            "model", "transformer", "architecture",
            "train", "pretrain", "a4"
        ]):
            print(p)



Searching /kaggle/input

Searching /kaggle/working


In [6]:

import torch
from pathlib import Path

p = next(Path("/kaggle/input/datasets/aq1de3").rglob("a4_step_22000.pt"))
ckpt = torch.load(p, map_location="cpu", weights_only=True)
sd = ckpt["model_state_dict"]

for k, v in sd.items():
    if torch.is_tensor(v) and any(
        s in k.lower()
        for s in ["token_embedding", "encoder_pos", "decoder_pos",
                  "embedding", "encoder_layers", "decoder_layers"]
    ):
        print(k, tuple(v.shape))


token_embedding.weight (50000, 512)
encoder_pos.weight (1024, 512)
decoder_pos.weight (1024, 512)
encoder_layers.0.norm1.weight (512,)
encoder_layers.0.norm1.bias (512,)
encoder_layers.0.attn.q_proj.weight (512, 512)
encoder_layers.0.attn.q_proj.bias (512,)
encoder_layers.0.attn.k_proj.weight (512, 512)
encoder_layers.0.attn.k_proj.bias (512,)
encoder_layers.0.attn.v_proj.weight (512, 512)
encoder_layers.0.attn.v_proj.bias (512,)
encoder_layers.0.attn.out_proj.weight (512, 512)
encoder_layers.0.attn.out_proj.bias (512,)
encoder_layers.0.norm2.weight (512,)
encoder_layers.0.norm2.bias (512,)
encoder_layers.0.ffn.net.0.weight (2048, 512)
encoder_layers.0.ffn.net.0.bias (2048,)
encoder_layers.0.ffn.net.2.weight (512, 2048)
encoder_layers.0.ffn.net.2.bias (512,)
encoder_layers.1.norm1.weight (512,)
encoder_layers.1.norm1.bias (512,)
encoder_layers.1.attn.q_proj.weight (512, 512)
encoder_layers.1.attn.q_proj.bias (512,)
encoder_layers.1.attn.k_proj.weight (512, 512)
encoder_layers.1.attn.k_

In [7]:
from pathlib import Path
import torch

p = next(
    Path("/kaggle/input/datasets/aq1de3").rglob("a4_step_22000.pt")
)
ckpt = torch.load(p, map_location="cpu", weights_only=True)

print("Checkpoint metadata:")
for k, v in ckpt.items():
    if k != "model_state_dict":
        print(k, "=", v)

print("\nFinal model parameters:")
for k, v in ckpt["model_state_dict"].items():
    if not any(x in k for x in ["encoder_layers.", "decoder_layers."]):
        if torch.is_tensor(v):
            print(k, tuple(v.shape))

Checkpoint metadata:
step = 22000
tokens_processed = 180096704
train_loss = 0.1944424146786332
reason = regular_checkpoint
saved_at = 2026-10-05T15:18:59.981007

Final model parameters:
token_embedding.weight (50000, 512)
encoder_pos.weight (1024, 512)
decoder_pos.weight (1024, 512)
encoder_norm.weight (512,)
encoder_norm.bias (512,)
decoder_norm.weight (512,)
decoder_norm.bias (512,)
lm_head.weight (50000, 512)


In [8]:
from pathlib import Path

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    for pattern in ["*.ipynb", "*.py"]:
        for p in root.rglob(pattern):
            print(p)

/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a2a2a2.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a4a4a4.ipynb
/kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
/kaggle/input/datasets/aq1de3/p2p2p2/phase-2 (1).ipynb
/kaggle/working/.virtual_documents/__notebook_source__.ipynb


In [9]:

import json
from pathlib import Path

p = Path(
    "/kaggle/input/datasets/aq1de3/p1p1p1/a4a4a4.ipynb"
)

with p.open(encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    low = source.lower()

    if any(term in low for term in [
        "class encoderblock",
        "class decoderblock",
        "class encoderdecod",
        "class attention",
        "encoder_layers",
        "model_state_dict",
        "a4_step_22000"
    ]):
        print(f"\n{'='*25} CELL {i} {'='*25}")
        print(source[:12000])



========================= CELL 3 =========================
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = 50_000
D_MODEL = 512
NUM_HEADS = 8
FFN_DIM = 2048
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
ENCODER_CONTEXT = 1024
DECODER_CONTEXT = 1024
DROPOUT = 0.1


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, q, k, v, causal=False):
        B, Tq, C = q.shape
        Tk = k.shape[1]

        q = self.q_proj(q)
        k = self.k_proj(k)
        v = self.v_proj(v)

        q = q.view(B, Tq, self.num_head

In [27]:
# Cell 5: Test the classifier

classifier_model = A4StatuteClassifier(backbone).to(device)
classifier_model.eval()

with torch.no_grad():
    test_ids = torch.randint(
        0, VOCAB_SIZE, (2, 64), device=device
    )
    test_mask = torch.ones_like(test_ids, dtype=torch.bool)

    test_logits = classifier_model(test_ids, test_mask)

print("Classifier output shape:", test_logits.shape)
print("Expected shape:", (2, NUM_LABELS))
print("Output finite:", torch.isfinite(test_logits).all().item())

assert test_logits.shape == (2, 100)
assert torch.isfinite(test_logits).all()

print("Classifier test passed!")

Classifier output shape: torch.Size([2, 100])
Expected shape: (2, 100)
Output finite: True
Classifier test passed!


In [52]:
print("NEW CELL CONFIRMED")
print("Model:", type(classifier_model).__name__)
print("Optimizer:", type(optimizer).__name__)
print("Batch shape:", tuple(input_ids.shape))

NEW CELL CONFIRMED
Model: A4StatuteClassifier
Optimizer: AdamW
Batch shape: (4, 512)


In [10]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

# ------------------------------------------------------------
# Configuration from the matching A4 checkpoint implementation
# ------------------------------------------------------------
VOCAB_SIZE = 50_000
D_MODEL = 512
NUM_HEADS = 8
FFN_DIM = 2048
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
ENCODER_CONTEXT = 1024
DECODER_CONTEXT = 1024
DROPOUT = 0.1
NUM_LABELS = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Enable the Kaggle GPU first."

# ------------------------------------------------------------
# A4 modules — preserve checkpoint parameter names
# ------------------------------------------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, q, k, v, causal=False, key_padding_mask=None):
        B, Tq, C = q.shape
        Tk = k.shape[1]

        q = self.q_proj(q).view(
            B, Tq, self.num_heads, self.head_dim
        ).transpose(1, 2)
        k = self.k_proj(k).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)
        v = self.v_proj(v).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        attn_mask = None
        if key_padding_mask is not None:
            # True means a key position should be ignored.
            attn_mask = (~key_padding_mask.bool())[:, None, None, :]

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = y.transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(y)


class FeedForward(nn.Module):
    def __init__(self, d_model, ffn_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(D_MODEL)
        self.attn = MultiHeadAttention(D_MODEL, NUM_HEADS, DROPOUT)
        self.norm2 = nn.LayerNorm(D_MODEL)
        self.ffn = FeedForward(D_MODEL, FFN_DIM, DROPOUT)

    def forward(self, x, attention_mask=None):
        x = x + self.attn(
            self.norm1(x), self.norm1(x), self.norm1(x),
            key_padding_mask=attention_mask,
        )
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(D_MODEL)
        self.self_attn = MultiHeadAttention(
            D_MODEL, NUM_HEADS, DROPOUT
        )
        self.norm2 = nn.LayerNorm(D_MODEL)
        self.cross_attn = MultiHeadAttention(
            D_MODEL, NUM_HEADS, DROPOUT
        )
        self.norm3 = nn.LayerNorm(D_MODEL)
        self.ffn = FeedForward(D_MODEL, FFN_DIM, DROPOUT)

    def forward(self, x, memory):
        x = x + self.self_attn(
            self.norm1(x), self.norm1(x), self.norm1(x),
            causal=True,
        )
        x = x + self.cross_attn(
            self.norm2(x), memory, memory,
        )
        x = x + self.ffn(self.norm3(x))
        return x


class A4EncoderDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.encoder_pos = nn.Embedding(ENCODER_CONTEXT, D_MODEL)
        self.decoder_pos = nn.Embedding(DECODER_CONTEXT, D_MODEL)

        self.encoder_layers = nn.ModuleList(
            [EncoderBlock() for _ in range(NUM_ENCODER_LAYERS)]
        )
        self.decoder_layers = nn.ModuleList(
            [DecoderBlock() for _ in range(NUM_DECODER_LAYERS)]
        )

        self.encoder_norm = nn.LayerNorm(D_MODEL)
        self.decoder_norm = nn.LayerNorm(D_MODEL)
        self.lm_head = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.dropout = nn.Dropout(DROPOUT)

    def encode(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device)

        x = self.token_embedding(input_ids)
        x = x + self.encoder_pos(positions)[None, :, :]
        x = self.dropout(x)

        for layer in self.encoder_layers:
            x = layer(x, attention_mask)

        return self.encoder_norm(x)

    def decode(self, decoder_ids, memory):
        B, T = decoder_ids.shape
        positions = torch.arange(T, device=decoder_ids.device)
        x = self.token_embedding(decoder_ids)
        x = x + self.decoder_pos(positions)[None, :, :]
        x = self.dropout(x)

        for layer in self.decoder_layers:
            x = layer(x, memory)

        return self.lm_head(self.decoder_norm(x))

    def forward(self, encoder_ids, decoder_ids):
        return self.decode(decoder_ids, self.encode(encoder_ids))


# ------------------------------------------------------------
# Locate checkpoint and load with strict verification
# ------------------------------------------------------------
root = Path("/kaggle/input/datasets/aq1de3")
candidates = list(root.rglob("a4_step_22000.pt"))
assert candidates, "A4 checkpoint not found."
ckpt_path = candidates[0]

checkpoint = torch.load(
    ckpt_path, map_location="cpu", weights_only=True
)
state_dict = checkpoint["model_state_dict"]

backbone = A4EncoderDecoder()
load_result = backbone.load_state_dict(state_dict, strict=True)

print("Checkpoint:", ckpt_path)
print("Step:", checkpoint.get("step"))
print("Strict load:", load_result)
print("Total parameters:", sum(p.numel() for p in backbone.parameters()))
print("Encoder position capacity:", backbone.encoder_pos.num_embeddings)

# Smoke test the encoder with a short sequence before full-length tests.
backbone = backbone.to(device)
backbone.eval()

with torch.no_grad():
    sample_ids = torch.randint(
        0, VOCAB_SIZE, (1, 64), device=device
    )
    sample_mask = torch.ones_like(sample_ids, dtype=torch.bool)
    sample_output = backbone.encode(sample_ids, sample_mask)

print("Encoder smoke test:", tuple(sample_output.shape))
assert sample_output.shape == (1, 64, D_MODEL)

print("A4 checkpoint loaded and encoder test passed.")


Checkpoint: /kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt
Step: 22000
Strict load: <All keys matched successfully>
Total parameters: 70789120
Encoder position capacity: 1024
Encoder smoke test: (1, 64, 512)
A4 checkpoint loaded and encoder test passed.


In [28]:
import json
from pathlib import Path
from collections import Counter

DATA_ROOT = Path("/kaggle/input/datasets/aq1de3")

TRAIN_PATH = DATA_ROOT / "traintr" / "train (2).json"
DEV_PATH   = DATA_ROOT / "devdev" / "dev (2).json"
TEST_PATH  = DATA_ROOT / "testtest" / "test (3).json"

def load_json_records(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Support either a list of records or a dictionary containing records
    if isinstance(data, dict):
        for key in ("data", "records", "examples"):
            if key in data and isinstance(data[key], list):
                data = data[key]
                break

    if not isinstance(data, list):
        raise ValueError(
            f"Unexpected JSON format in {path}: {type(data)}"
        )

    return data

train_data = load_json_records(TRAIN_PATH)
dev_data   = load_json_records(DEV_PATH)
test_data  = load_json_records(TEST_PATH)

print("Train records:", len(train_data))
print("Dev records:", len(dev_data))
print("Test records:", len(test_data))

print("\nTrain sample keys:", train_data[0].keys())
print("Train sample ID:", train_data[0]["id"])
print("Train sample labels:", train_data[0]["labels"])
print("Train text preview:", train_data[0]["text"][:300])

print("\nDev sample keys:", dev_data[0].keys())
print("Test sample keys:", test_data[0].keys())

all_labels = [
    label
    for row in train_data
    for label in row["labels"]
]

print("\nUnique training labels:", len(set(all_labels)))
print("Minimum label ID:", min(all_labels))
print("Maximum label ID:", max(all_labels))
print("Training label count distribution:", Counter(map(len, [r["labels"] for r in train_data])))

Train records: 42750
Dev records: 10181
Test records: 13019

Train sample keys: dict_keys(['id', 'text', 'labels'])
Train sample ID: 1000008
Train sample labels: [69, 9, 3]
Train text preview: (a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stated thus: With the growth of industry, commerceand trade in and around the city of Mumbai which generatessubstantial quantity of wealth, there has been increase oforganised activiti

Dev sample keys: dict_keys(['id', 'text', 'labels'])
Test sample keys: dict_keys(['id', 'text'])

Unique training labels: 100
Minimum label ID: 0
Maximum label ID: 99
Training label count distribution: Counter({1: 9146, 2: 7585, 3: 7045, 4: 5782, 5: 4265, 6: 3283, 7: 2131, 8: 1390, 9: 836, 10: 458, 11: 358, 12: 167, 13: 106, 14: 53, 15: 46, 16: 27, 17: 16, 18: 13, 19: 13, 20: 5, 22: 5, 26: 5, 21: 4, 24: 4, 30: 3, 27: 2, 31: 1, 23: 1})


In [29]:
from pathlib import Path
import json

DATA_ROOT = Path("/kaggle/input/datasets/aq1de2")
if not DATA_ROOT.exists():
    DATA_ROOT = Path("/kaggle/input/datasets/aq1de3")

tokenizer_files = list(DATA_ROOT.rglob("tokenizer.json"))

print("Tokenizer files found:", len(tokenizer_files))
for p in tokenizer_files[:10]:
    print(p)

Tokenizer files found: 1
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json


In [30]:
from tokenizers import Tokenizer

assert tokenizer_files, "No tokenizer.json found. Check the dataset paths."

TOKENIZER_PATH = tokenizer_files[0]
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("Tokenizer path:", TOKENIZER_PATH)
print("Tokenizer vocabulary size:", tokenizer.get_vocab_size())
print("Special tokens:", tokenizer.get_added_tokens_decoder())

Tokenizer path: /kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
Tokenizer vocabulary size: 50000
Special tokens: {0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 2: AddedToken("[BOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 3: AddedToken("[EOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True)}


In [31]:
print("PAD ID:", tokenizer.token_to_id("[PAD]"))
print("UNK ID:", tokenizer.token_to_id("[UNK]"))
print("BOS ID:", tokenizer.token_to_id("[BOS]"))
print("EOS ID:", tokenizer.token_to_id("[EOS]"))
print("MASK ID:", tokenizer.token_to_id("[MASK]"))

assert tokenizer.get_vocab_size() == VOCAB_SIZE
assert tokenizer.token_to_id("[PAD]") is not None
assert tokenizer.token_to_id("[BOS]") is not None
assert tokenizer.token_to_id("[EOS]") is not None

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("\nSpecial-token verification passed!")

PAD ID: 0
UNK ID: 1
BOS ID: 2
EOS ID: 3
MASK ID: 4

Special-token verification passed!


In [32]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

MAX_LENGTH = 512

class StatuteDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=MAX_LENGTH,
                 labeled=True):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labeled = labeled

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records[idx]
        text = row["text"]

        # Reserve space for BOS and EOS
        ids = self.tokenizer.encode(
            text,
            add_special_tokens=False
        ).ids[:self.max_length - 2]

        input_ids = [BOS_ID] + ids + [EOS_ID]
        attention_mask = [1] * len(input_ids)

        pad_length = self.max_length - len(input_ids)
        input_ids += [PAD_ID] * pad_length
        attention_mask += [0] * pad_length

        item = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.bool),
            "id": row["id"],
        }

        if self.labeled:
            targets = torch.zeros(NUM_LABELS, dtype=torch.float32)
            targets[row["labels"]] = 1.0
            item["labels"] = targets

        return item

train_dataset = StatuteDataset(train_data, tokenizer)
dev_dataset = StatuteDataset(dev_data, tokenizer)
test_dataset = StatuteDataset(test_data, tokenizer, labeled=False)

print("Train dataset:", len(train_dataset))
print("Dev dataset:", len(dev_dataset))
print("Test dataset:", len(test_dataset))

sample = train_dataset[0]
print("\nInput shape:", sample["input_ids"].shape)
print("Mask shape:", sample["attention_mask"].shape)
print("Target shape:", sample["labels"].shape)
print("Nonzero target labels:", int(sample["labels"].sum().item()))
print("Valid tokens:", int(sample["attention_mask"].sum().item()))

assert sample["input_ids"].shape == (MAX_LENGTH,)
assert sample["attention_mask"].shape == (MAX_LENGTH,)
assert sample["labels"].shape == (NUM_LABELS,)
assert int(sample["labels"].sum().item()) == len(train_data[0]["labels"])

print("\nDataset preparation test passed!")

Train dataset: 42750
Dev dataset: 10181
Test dataset: 13019

Input shape: torch.Size([512])
Mask shape: torch.Size([512])
Target shape: torch.Size([100])
Nonzero target labels: 3
Valid tokens: 512

Dataset preparation test passed!


In [33]:
from torch.utils.data import DataLoader

BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

print("Input IDs:", input_ids.shape)
print("Attention mask:", attention_mask.shape)
print("Labels:", labels.shape)
print("Valid tokens per example:", attention_mask.sum(dim=1).tolist())
print("Label counts per example:", labels.sum(dim=1).tolist())
print("Device:", device)

assert input_ids.shape == (BATCH_SIZE, MAX_LENGTH)
assert attention_mask.shape == (BATCH_SIZE, MAX_LENGTH)
assert labels.shape == (BATCH_SIZE, NUM_LABELS)

print("Batch test passed!")

Input IDs: torch.Size([4, 512])
Attention mask: torch.Size([4, 512])
Labels: torch.Size([4, 100])
Valid tokens per example: [512, 512, 512, 512]
Label counts per example: [1.0, 2.0, 2.0, 5.0]
Device: cuda
Batch test passed!


In [34]:
import torch
import torch.nn as nn

classifier_model.train()
classifier_model.zero_grad(set_to_none=True)

criterion = nn.BCEWithLogitsLoss()

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
    enabled=(device.type == "cuda")
):
    logits = classifier_model(input_ids, attention_mask)
    loss = criterion(logits.float(), labels)

print("Logits shape:", logits.shape)
print("Loss:", loss.item())
print("Loss finite:", torch.isfinite(loss).item())

loss.backward()

grad_count = sum(
    1 for p in classifier_model.parameters()
    if p.grad is not None
)
finite_grads = all(
    torch.isfinite(p.grad).all().item()
    for p in classifier_model.parameters()
    if p.grad is not None
)

print("Parameters with gradients:", grad_count)
print("Gradients finite:", finite_grads)

assert logits.shape == (BATCH_SIZE, NUM_LABELS)
assert torch.isfinite(loss)
assert grad_count > 0
assert finite_grads

classifier_model.zero_grad(set_to_none=True)
classifier_model.eval()

print("Forward/backward smoke test passed!")

Logits shape: torch.Size([4, 100])
Loss: 0.6896378993988037
Loss finite: True
Parameters with gradients: 102
Gradients finite: True
Forward/backward smoke test passed!


In [35]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from pathlib import Path

# Training configuration
EPOCHS = 3
ENCODER_LR = 2e-5
HEAD_LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

OUTPUT_DIR = Path("/kaggle/working/phase5_exp1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Separate learning rates for pretrained and new parameters
encoder_params = list(classifier_model.encoder.parameters())
head_params = list(classifier_model.classifier.parameters())

optimizer = AdamW(
    [
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

print("Experiment: A4 full fine-tuning baseline")
print("Epochs:", EPOCHS)
print("Encoder LR:", ENCODER_LR)
print("Classifier LR:", HEAD_LR)
print("Train batches per epoch:", len(train_loader))
print("Dev batches:", len(dev_loader))
print("Checkpoint directory:", OUTPUT_DIR)
print("Trainable parameters:", sum(
    p.numel() for p in classifier_model.parameters() if p.requires_grad
))

Experiment: A4 full fine-tuning baseline
Epochs: 3
Encoder LR: 2e-05
Classifier LR: 0.0002
Train batches per epoch: 10688
Dev batches: 2546
Checkpoint directory: /kaggle/working/phase5_exp1
Trainable parameters: 70840420


In [36]:
import torch
import torch.nn as nn

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

try:
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        logits = classifier_model(input_ids, attention_mask)
        loss = nn.BCEWithLogitsLoss()(logits.float(), labels)

    scaler.scale(loss).backward()

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(
        classifier_model.parameters(), GRAD_CLIP
    )

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    if device.type == "cuda":
        torch.cuda.synchronize()
        print(
            "GPU allocated (GB):",
            round(torch.cuda.memory_allocated() / 1024**3, 2)
        )
        print(
            "GPU reserved (GB):",
            round(torch.cuda.memory_reserved() / 1024**3, 2)
        )

    print("Loss:", round(loss.item(), 5))
    print("Gradient norm:", round(float(grad_norm), 4))
    print("Optimizer update: successful")

except torch.cuda.OutOfMemoryError:
    optimizer.zero_grad(set_to_none=True)
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("GPU out of memory. Do not start full training yet.")
    raise

classifier_model.eval()

GPU allocated (GB): 0.62
GPU reserved (GB): 1.13
Loss: 0.69483
Gradient norm: 0.2484
Optimizer update: successful


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [37]:
import torch
import torch.nn as nn

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
    enabled=(device.type == "cuda")
):
    logits = classifier_model(input_ids, attention_mask)
    loss = nn.BCEWithLogitsLoss()(logits.float(), labels)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(
    classifier_model.parameters(), GRAD_CLIP
)

scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

if device.type == "cuda":
    torch.cuda.synchronize()
    print("GPU allocated (GB):",
          round(torch.cuda.memory_allocated() / 1024**3, 2))
    print("GPU reserved (GB):",
          round(torch.cuda.memory_reserved() / 1024**3, 2))

print("Loss:", round(loss.item(), 5))
print("Gradient norm:", round(float(grad_norm), 4))
print("Optimizer update: successful")

classifier_model.eval()

GPU allocated (GB): 0.62
GPU reserved (GB): 1.13
Loss: 0.68166
Gradient norm: 0.2482
Optimizer update: successful


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [38]:
print("=== PHASE 5 GPU DIAGNOSTIC ===")
print("Device:", device)
print("Model exists:", "classifier_model" in globals())
print("Optimizer exists:", "optimizer" in globals())
print("Scaler exists:", "scaler" in globals())
print("Input batch exists:", "input_ids" in globals())
print("Attention mask exists:", "attention_mask" in globals())
print("Labels exist:", "labels" in globals())

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU allocated (GB):",
          round(torch.cuda.memory_allocated() / 1024**3, 2))

print("=== DIAGNOSTIC COMPLETE ===")

=== PHASE 5 GPU DIAGNOSTIC ===
Device: cuda
Model exists: True
Optimizer exists: True
Scaler exists: True
Input batch exists: True
Attention mask exists: True
Labels exist: True
GPU: Tesla T4
GPU allocated (GB): 0.62
=== DIAGNOSTIC COMPLETE ===


In [39]:
import torch
import torch.nn as nn

print("Starting optimizer update...")

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = classifier_model(input_ids, attention_mask)
    loss = nn.BCEWithLogitsLoss()(logits.float(), labels)

print("Forward pass completed.")
print("Loss:", round(loss.item(), 5))

scaler.scale(loss).backward()
print("Backward pass completed.")

scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(
    classifier_model.parameters(), GRAD_CLIP
)

scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

torch.cuda.synchronize()

print("Gradient norm:", round(float(grad_norm), 4))
print("GPU allocated (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))
print("Optimizer update: SUCCESS")

classifier_model.eval()

Starting optimizer update...
Forward pass completed.
Loss: 0.67168
Backward pass completed.
Gradient norm: 0.2447
GPU allocated (GB): 0.62
Optimizer update: SUCCESS


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [40]:
print("PHASE 5 TRAINING STEP STARTED", flush=True)

print("Logits shape:", logits.shape if "logits" in globals() else "Not computed")
print("Current GPU:", torch.cuda.get_device_name(0))

# Run one actual training update
classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    logits = classifier_model(input_ids, attention_mask)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(), labels
    )

print("Forward pass complete. Loss:", loss.item(), flush=True)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), GRAD_CLIP)
scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

print("OPTIMIZER UPDATE SUCCESSFUL", flush=True)
classifier_model.eval()

PHASE 5 TRAINING STEP STARTED
Logits shape: torch.Size([4, 100])
Current GPU: Tesla T4
Forward pass complete. Loss: 0.6637312173843384
OPTIMIZER UPDATE SUCCESSFUL


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [41]:
print("HELLO FROM THE NEW CELL", flush=True)
print("Model:", type(classifier_model).__name__, flush=True)
print("Device:", device, flush=True)

HELLO FROM THE NEW CELL
Model: A4StatuteClassifier
Device: cuda


In [42]:
import torch
import torch.nn.functional as F

print("Starting optimizer update...", flush=True)

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    logits = classifier_model(input_ids, attention_mask)
    loss = F.binary_cross_entropy_with_logits(
        logits.float(), labels
    )

print("Forward pass complete. Loss:", round(loss.item(), 5), flush=True)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(
    classifier_model.parameters(), GRAD_CLIP
)

scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

torch.cuda.synchronize()

print("Gradient norm:", round(float(grad_norm), 4))
print("GPU memory allocated (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))
print("OPTIMIZER UPDATE SUCCESSFUL", flush=True)

classifier_model.eval()

Starting optimizer update...
Forward pass complete. Loss: 0.65076
Gradient norm: 0.244
GPU memory allocated (GB): 0.62
OPTIMIZER UPDATE SUCCESSFUL


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [43]:
print("OPTIMIZER TEST CELL STARTED", flush=True)

print("Optimizer:", type(optimizer).__name__)
print("Batch:", tuple(input_ids.shape))
print("Device:", input_ids.device)

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    logits = classifier_model(input_ids, attention_mask)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(), labels
    )

print("Forward pass done. Loss:", loss.item(), flush=True)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), 1.0)
scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

print("OPTIMIZER UPDATE SUCCESSFUL", flush=True)
classifier_model.eval()

OPTIMIZER TEST CELL STARTED
Optimizer: AdamW
Batch: (4, 512)
Device: cuda:0
Forward pass done. Loss: 0.6389752626419067
OPTIMIZER UPDATE SUCCESSFUL


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [44]:
print("CELL EXECUTED SUCCESSFULLY")
print("Current execution reached this line.")
print("Model type:", type(classifier_model).__name__)
print("Logits from previous run:", 
      tuple(logits.shape) if "logits" in globals() else "Not available")

CELL EXECUTED SUCCESSFULLY
Current execution reached this line.
Model type: A4StatuteClassifier
Logits from previous run: (4, 100)


In [45]:
import torch
import torch.nn.functional as F

print("=== STARTING OPTIMIZER UPDATE ===", flush=True)

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    logits = classifier_model(input_ids, attention_mask)
    loss = F.binary_cross_entropy_with_logits(
        logits.float(), labels
    )

print(f"Training loss: {loss.item():.5f}", flush=True)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(
    classifier_model.parameters(), GRAD_CLIP
)

scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

torch.cuda.synchronize()

print(f"Gradient norm: {float(grad_norm):.4f}", flush=True)
print("=== OPTIMIZER UPDATE COMPLETED ===", flush=True)

classifier_model.eval()

=== STARTING OPTIMIZER UPDATE ===
Training loss: 0.62780
Gradient norm: 0.2421
=== OPTIMIZER UPDATE COMPLETED ===


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [46]:
print("UNIQUE CELL 731")
print("Optimizer type:", type(optimizer).__name__)
print("Loss function ready:", callable(torch.nn.functional.binary_cross_entropy_with_logits))

UNIQUE CELL 731
Optimizer type: AdamW
Loss function ready: True


In [47]:
import torch
import torch.nn.functional as F

print("STARTING REAL TRAINING STEP", flush=True)

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    logits = classifier_model(input_ids, attention_mask)
    loss = F.binary_cross_entropy_with_logits(
        logits.float(), labels
    )

print(f"Forward pass complete | Loss: {loss.item():.5f}", flush=True)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(
    classifier_model.parameters(), GRAD_CLIP
)

scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

torch.cuda.synchronize()

print(f"Gradient norm: {float(grad_norm):.4f}", flush=True)
print("REAL TRAINING STEP COMPLETED", flush=True)

classifier_model.eval()

STARTING REAL TRAINING STEP
Forward pass complete | Loss: 0.61789
Gradient norm: 0.2482
REAL TRAINING STEP COMPLETED


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [48]:
print("RUN ID: PHASE5-UPDATE-02", flush=True)

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    out = classifier_model(input_ids, attention_mask)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        out.float(), labels
    )

print("LOSS =", float(loss.detach()), flush=True)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), 1.0)
scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

print("UPDATE FINISHED", flush=True)
classifier_model.eval()

RUN ID: PHASE5-UPDATE-02
LOSS = 0.6089350581169128
UPDATE FINISHED


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [49]:
print("NEW CELL OUTPUT TEST 456")
print("Current model:", type(classifier_model).__name__)
print("Current device:", device)

NEW CELL OUTPUT TEST 456
Current model: A4StatuteClassifier
Current device: cuda


In [50]:
import torch
import torch.nn.functional as F

print("=== PHASE5 UPDATE START ===", flush=True)

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    logits = classifier_model(input_ids, attention_mask)
    loss = F.binary_cross_entropy_with_logits(
        logits.float(), labels
    )

print("Loss:", round(loss.item(), 5), flush=True)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(
    classifier_model.parameters(), GRAD_CLIP
)

scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

torch.cuda.synchronize()

print("Gradient norm:", round(float(grad_norm), 4), flush=True)
print("GPU memory (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
), flush=True)
print("=== PHASE5 UPDATE SUCCESS ===", flush=True)

classifier_model.eval()

=== PHASE5 UPDATE START ===
Loss: 0.59586
Gradient norm: 0.2532
GPU memory (GB): 0.62
=== PHASE5 UPDATE SUCCESS ===


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [51]:
print("OPTIMIZER UPDATE: START")

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    logits = classifier_model(input_ids, attention_mask)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(), labels
    )

print("Loss:", loss.item(), flush=True)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), 1.0)
scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

print("OPTIMIZER UPDATE: SUCCESS", flush=True)
classifier_model.eval()

OPTIMIZER UPDATE: START
Loss: 0.5848292708396912
OPTIMIZER UPDATE: SUCCESS


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [53]:
import torch
import torch.nn.functional as F

print("=== TRAINING UPDATE START ===", flush=True)

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    logits = classifier_model(input_ids, attention_mask)
    loss = F.binary_cross_entropy_with_logits(
        logits.float(), labels
    )

print(f"Loss: {loss.item():.5f}", flush=True)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(
    classifier_model.parameters(), GRAD_CLIP
)

scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

torch.cuda.synchronize()

print(f"Gradient norm: {float(grad_norm):.4f}", flush=True)
print("=== TRAINING UPDATE SUCCESS ===", flush=True)

classifier_model.eval()

=== TRAINING UPDATE START ===
Loss: 0.56982
Gradient norm: 0.2634
=== TRAINING UPDATE SUCCESS ===


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [54]:
print("CELL ID: TRAINING-777")

# Run a forward pass and display only its result
classifier_model.eval()

with torch.no_grad():
    with torch.autocast("cuda", dtype=torch.float16):
        test_logits = classifier_model(input_ids, attention_mask)

print("Logits:", tuple(test_logits.shape))
print("Finite:", torch.isfinite(test_logits).all().item())

CELL ID: TRAINING-777
Logits: (4, 100)
Finite: True


In [55]:
import torch
import torch.nn.functional as F

print("TRAINING UPDATE START", flush=True)

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    logits = classifier_model(input_ids, attention_mask)
    loss = F.binary_cross_entropy_with_logits(
        logits.float(), labels
    )

print(f"Loss: {loss.item():.5f}", flush=True)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(
    classifier_model.parameters(), 1.0
)

scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

print(f"Gradient norm: {float(grad_norm):.4f}", flush=True)
print("OPTIMIZER UPDATE SUCCESS", flush=True)

classifier_model.eval()

TRAINING UPDATE START
Loss: 0.55992
Gradient norm: 0.2650
OPTIMIZER UPDATE SUCCESS


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [56]:
print("UPDATE CELL 901", flush=True)

classifier_model.train()
optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    out = classifier_model(input_ids, attention_mask)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        out.float(), labels
    )

print("Loss:", loss.item(), flush=True)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), 1.0)
scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

print("OPTIMIZER UPDATE SUCCESS", flush=True)
classifier_model.eval()

UPDATE CELL 901
Loss: 0.5485549569129944
OPTIMIZER UPDATE SUCCESS


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [57]:
print("DIAGNOSTIC 999 — NEW CELL")
print("Model:", type(classifier_model).__name__)
print("Optimizer:", type(optimizer).__name__)
print("Current batch:", tuple(input_ids.shape))
print("Ready to train:", classifier_model.training)

DIAGNOSTIC 999 — NEW CELL
Model: A4StatuteClassifier
Optimizer: AdamW
Current batch: (4, 512)
Ready to train: False


In [58]:
# PHASE 5 — FIRST REAL TRAINING UPDATE
classifier_model.train()

optimizer.zero_grad(set_to_none=True)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
    enabled=(device.type == "cuda")
):
    logits = classifier_model(input_ids, attention_mask)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        logits, labels
    )

if not torch.isfinite(loss):
    raise RuntimeError(f"Non-finite loss: {loss.item()}")

scaler.scale(loss).backward()
scaler.unscale_(optimizer)

torch.nn.utils.clip_grad_norm_(
    classifier_model.parameters(), max_norm=1.0
)

scaler.step(optimizer)
scaler.update()

print("=== FIRST TRAINING UPDATE ===")
print("Model training mode:", classifier_model.training)
print("Loss:", round(loss.item(), 5))
print("Logits shape:", tuple(logits.shape))
print("Loss finite:", torch.isfinite(loss).item())
print("Optimizer update attempted: Yes")

=== FIRST TRAINING UPDATE ===
Model training mode: True
Loss: 0.53453
Logits shape: (4, 100)
Loss finite: True
Optimizer update attempted: Yes


In [60]:
batch = next(iter(train_loader))

print("Batch type:", type(batch))

if isinstance(batch, dict):
    for key, value in batch.items():
        print(
            key,
            "| type:", type(value).__name__,
            "| shape:", getattr(value, "shape", None)
        )
elif isinstance(batch, (tuple, list)):
    for i, value in enumerate(batch):
        print(
            i,
            "| type:", type(value).__name__,
            "| shape:", getattr(value, "shape", None),
            "| sample:", str(value)[:100] if isinstance(value, str) else ""
        )
else:
    print(batch)

Batch type: <class 'dict'>
input_ids | type: Tensor | shape: torch.Size([4, 512])
attention_mask | type: Tensor | shape: torch.Size([4, 512])
id | type: list | shape: None
labels | type: Tensor | shape: torch.Size([4, 100])


In [61]:
import time
import torch
import torch.nn.functional as F
from pathlib import Path

classifier_model.train()
pilot_losses = []
start_time = time.time()
MAX_PILOT_BATCHES = 100

for step, batch in enumerate(train_loader):
    if step >= MAX_PILOT_BATCHES:
        break

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        logits = classifier_model(input_ids, attention_mask)
        loss = F.binary_cross_entropy_with_logits(logits, labels)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at batch {step}")

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        classifier_model.parameters(), max_norm=1.0
    )

    scaler.step(optimizer)
    scaler.update()

    pilot_losses.append(loss.item())

    if (step + 1) % 10 == 0:
        print(
            f"Batch {step + 1}/{MAX_PILOT_BATCHES} | "
            f"Loss: {loss.item():.5f}"
        )

print("\n=== PILOT COMPLETE ===")
print("Batches completed:", len(pilot_losses))
print("Average loss:", round(sum(pilot_losses) / len(pilot_losses), 5))
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))

save_dir = Path("/kaggle/working/phase5_exp1")
save_dir.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": classifier_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scaler_state_dict": scaler.state_dict(),
    "batches_completed": len(pilot_losses),
    "pilot_losses": pilot_losses,
}, save_dir / "pilot_checkpoint.pt")

print("Checkpoint saved:", save_dir / "pilot_checkpoint.pt")

Batch 10/100 | Loss: 0.39822
Batch 20/100 | Loss: 0.30848
Batch 30/100 | Loss: 0.19675
Batch 40/100 | Loss: 0.17339
Batch 50/100 | Loss: 0.10666
Batch 60/100 | Loss: 0.12694
Batch 70/100 | Loss: 0.09863
Batch 80/100 | Loss: 0.17180
Batch 90/100 | Loss: 0.16528
Batch 100/100 | Loss: 0.12349

=== PILOT COMPLETE ===
Batches completed: 100
Average loss: 0.20941
Elapsed minutes: 0.1
Checkpoint saved: /kaggle/working/phase5_exp1/pilot_checkpoint.pt


In [62]:
import torch
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

classifier_model.eval()

all_probs = []
all_targets = []

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = classifier_model(input_ids, attention_mask)

        all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
        all_targets.append(labels.cpu().numpy())

probs = np.concatenate(all_probs, axis=0)
targets = np.concatenate(all_targets, axis=0).astype(int)
predictions = (probs >= 0.5).astype(int)

print("=== PHASE 5 DEVELOPMENT RESULTS ===")
print("Documents evaluated:", len(targets))
print("Macro-F1:", round(f1_score(
    targets, predictions, average="macro", zero_division=0
), 5))
print("Micro-F1:", round(f1_score(
    targets, predictions, average="micro", zero_division=0
), 5))
print("Subset accuracy:", round(accuracy_score(
    targets, predictions
), 5))
print("Mean predicted labels per document:", round(
    predictions.sum(axis=1).mean(), 3
))
print("Mean actual labels per document:", round(
    targets.sum(axis=1).mean(), 3
))

classifier_model.train()

=== PHASE 5 DEVELOPMENT RESULTS ===
Documents evaluated: 10181
Macro-F1: 0.0
Micro-F1: 0.0
Subset accuracy: 0.0
Mean predicted labels per document: 0.0
Mean actual labels per document: 3.916


A4StatuteClassifier(
  (encoder): A4EncoderDecoder(
    (token_embedding): Embedding(50000, 512)
    (encoder_pos): Embedding(1024, 512)
    (decoder_pos): Embedding(1024, 512)
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inp

In [63]:
print("DEV EVALUATION CHECK")
print("Dev batches:", len(dev_loader))
print("Model mode before evaluation:", classifier_model.training)
print("Checkpoint exists:", __import__("os").path.exists(
    "/kaggle/working/phase5_exp1/pilot_checkpoint.pt"
))
print("Ready to evaluate:", True)

DEV EVALUATION CHECK
Dev batches: 2546
Model mode before evaluation: True
Checkpoint exists: True
Ready to evaluate: True


In [64]:
import torch
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

classifier_model.eval()
all_probs = []
all_targets = []

with torch.no_grad():
    for i, batch in enumerate(dev_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = classifier_model(input_ids, attention_mask)

        all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
        all_targets.append(labels.cpu().numpy())

        if (i + 1) % 500 == 0:
            print(f"Evaluated {i + 1}/{len(dev_loader)} batches")

probs = np.concatenate(all_probs, axis=0)
targets = np.concatenate(all_targets, axis=0).astype(int)
predictions = (probs >= 0.5).astype(int)

print("\n=== DEVELOPMENT METRICS ===")
print("Documents:", len(targets))
print("Macro-F1:", round(f1_score(
    targets, predictions, average="macro", zero_division=0
), 5))
print("Micro-F1:", round(f1_score(
    targets, predictions, average="micro", zero_division=0
), 5))
print("Subset accuracy:", round(accuracy_score(
    targets, predictions
), 5))
print("Mean predicted labels/document:", round(
    predictions.sum(axis=1).mean(), 3
))
print("Mean actual labels/document:", round(
    targets.sum(axis=1).mean(), 3
))

Evaluated 500/2546 batches
Evaluated 1000/2546 batches
Evaluated 1500/2546 batches
Evaluated 2000/2546 batches
Evaluated 2500/2546 batches

=== DEVELOPMENT METRICS ===
Documents: 10181
Macro-F1: 0.0
Micro-F1: 0.0
Subset accuracy: 0.0
Mean predicted labels/document: 0.0
Mean actual labels/document: 3.916


In [65]:
import numpy as np
from sklearn.metrics import f1_score

print("=== PREDICTION DIAGNOSTICS ===")
print("Probability min:", float(probs.min()))
print("Probability max:", float(probs.max()))
print("Probability mean:", float(probs.mean()))
print("Probability percentiles:")
print(np.percentile(probs, [50, 75, 90, 95, 99, 99.9]))

print("\n=== THRESHOLD CHECK ===")
for threshold in [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]:
    pred = (probs >= threshold).astype(int)
    macro = f1_score(
        targets, pred, average="macro", zero_division=0
    )
    micro = f1_score(
        targets, pred, average="micro", zero_division=0
    )
    print(
        f"Threshold={threshold:.3f} | "
        f"Predicted labels/doc={pred.sum(axis=1).mean():.3f} | "
        f"Macro-F1={macro:.5f} | Micro-F1={micro:.5f}"
    )

=== PREDICTION DIAGNOSTICS ===
Probability min: 0.00279998779296875
Probability max: 0.340576171875
Probability mean: 0.039295632392168045
Probability percentiles:
[0.02438354 0.04217529 0.08709717 0.12927856 0.27001953 0.31762695]

=== THRESHOLD CHECK ===
Threshold=0.001 | Predicted labels/doc=100.000 | Macro-F1=0.07144 | Micro-F1=0.07536
Threshold=0.005 | Predicted labels/doc=96.724 | Macro-F1=0.07149 | Micro-F1=0.07713
Threshold=0.010 | Predicted labels/doc=78.652 | Macro-F1=0.07203 | Micro-F1=0.08786
Threshold=0.020 | Predicted labels/doc=61.674 | Macro-F1=0.07043 | Micro-F1=0.09961
Threshold=0.050 | Predicted labels/doc=19.228 | Macro-F1=0.04552 | Micro-F1=0.18954
Threshold=0.100 | Predicted labels/doc=6.959 | Macro-F1=0.02467 | Micro-F1=0.23910
Threshold=0.200 | Predicted labels/doc=2.992 | Macro-F1=0.01618 | Micro-F1=0.23174
Threshold=0.500 | Predicted labels/doc=0.000 | Macro-F1=0.00000 | Micro-F1=0.00000


In [66]:
import inspect
import torch
import numpy as np

print("=== ATTENTION MASK IMPLEMENTATION ===")
print(inspect.getsource(MultiHeadAttention.forward))

print("\n=== LABEL DISTRIBUTION ===")
positive_counts = targets.sum(axis=0)
print("Labels with at least one positive:", int((positive_counts > 0).sum()))
print("Labels never positive in dev:", int((positive_counts == 0).sum()))
print("Most frequent label counts:", sorted(
    positive_counts.tolist(), reverse=True
)[:10])

print("\n=== CLASSIFIER HEAD ===")
print("Classifier bias mean:", round(
    classifier_model.classifier.bias.detach().mean().item(), 5
))
print("Classifier weight std:", round(
    classifier_model.classifier.weight.detach().std().item(), 5
))

print("\n=== CURRENT MODEL CHECK ===")
print("Checkpoint training batches:", 100)
print("Development documents:", len(targets))
print("Mask True means valid token in dataset:", True)

=== ATTENTION MASK IMPLEMENTATION ===
    def forward(self, q, k, v, causal=False, key_padding_mask=None):
        B, Tq, C = q.shape
        Tk = k.shape[1]

        q = self.q_proj(q).view(
            B, Tq, self.num_heads, self.head_dim
        ).transpose(1, 2)
        k = self.k_proj(k).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)
        v = self.v_proj(v).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        attn_mask = None
        if key_padding_mask is not None:
            # True means a key position should be ignored.
            attn_mask = (~key_padding_mask.bool())[:, None, None, :]

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = y.transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(y)


=== LABEL DISTRIBUTION ===
La

In [67]:
import torch
import numpy as np

print("=== PHASE 5 TRAINING STATUS ===")
print("Encoder LR:", optimizer.param_groups[0]["lr"])
print("Classifier LR:", optimizer.param_groups[1]["lr"])

# Check whether the pretrained encoder weights changed
# during the 100-batch pilot by comparing against the original checkpoint.
original = torch.load(
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
    map_location="cpu",
    weights_only=False
)

print("Original checkpoint type:", type(original).__name__)
print("Current probability mean:", round(float(probs.mean()), 5))
print("Current probability max:", round(float(probs.max()), 5))
print("Mean true labels/document:", round(float(targets.sum(axis=1).mean()), 3))

=== PHASE 5 TRAINING STATUS ===
Encoder LR: 2e-05
Classifier LR: 0.0002
Original checkpoint type: dict
Current probability mean: 0.0393
Current probability max: 0.34058
Mean true labels/document: 3.916


In [68]:
import torch

checkpoint = torch.load(
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
    map_location="cpu",
    weights_only=False
)

print("=== ORIGINAL CHECKPOINT STRUCTURE ===")
print("Top-level keys:", list(checkpoint.keys())[:20])

state = checkpoint
for key in ["model_state_dict", "state_dict", "model"]:
    if key in checkpoint and isinstance(checkpoint[key], dict):
        state = checkpoint[key]
        print("Weights stored under:", key)
        break

print("\n=== SAMPLE WEIGHT KEYS ===")
print(list(state.keys())[:10])

current_state = classifier_model.encoder.state_dict()
matching_key = next(
    (k for k in current_state if k in state and current_state[k].shape == state[k].shape),
    None
)

if matching_key is None:
    print("\nNo directly matching encoder key found; inspect the sample keys above.")
else:
    diff = (
        current_state[matching_key].detach().cpu().float()
        - state[matching_key].detach().cpu().float()
    ).abs().mean().item()

    print("Comparison key:", matching_key)
    print("Mean absolute weight difference:", diff)
    print("Encoder weights differ from original:", diff > 0)

=== ORIGINAL CHECKPOINT STRUCTURE ===
Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
Weights stored under: model_state_dict

=== SAMPLE WEIGHT KEYS ===
['token_embedding.weight', 'encoder_pos.weight', 'decoder_pos.weight', 'encoder_layers.0.norm1.weight', 'encoder_layers.0.norm1.bias', 'encoder_layers.0.attn.q_proj.weight', 'encoder_layers.0.attn.q_proj.bias', 'encoder_layers.0.attn.k_proj.weight', 'encoder_layers.0.attn.k_proj.bias', 'encoder_layers.0.attn.v_proj.weight']
Comparison key: token_embedding.weight
Mean absolute weight difference: 0.020751027390360832
Encoder weights differ from original: True


In [69]:
checkpoint = torch.load(
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
    map_location="cpu",
    weights_only=False
)

print("Original A4 training step:", checkpoint.get("step"))
print("Original A4 training loss:", checkpoint.get("train_loss"))
print("Current pilot batches completed:", 100)
print("Current development Macro-F1:", 0.0)
print("Current development Micro-F1:", 0.0)

Original A4 training step: 22000
Original A4 training loss: 0.1944424146786332
Current pilot batches completed: 100
Current development Macro-F1: 0.0
Current development Micro-F1: 0.0


In [70]:

import time
import torch
import torch.nn.functional as F
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/phase5_exp1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

classifier_model.train()
epoch_losses = []
start_time = time.time()

for step, batch in enumerate(train_loader, start=1):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        logits = classifier_model(input_ids, attention_mask)
        loss = F.binary_cross_entropy_with_logits(logits, labels)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at batch {step}")

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(
        classifier_model.parameters(), max_norm=1.0
    )
    scaler.step(optimizer)
    scaler.update()

    epoch_losses.append(loss.item())

    if step % 250 == 0:
        elapsed = (time.time() - start_time) / 60
        print(
            f"Batch {step}/{len(train_loader)} | "
            f"Loss: {loss.item():.5f} | "
            f"Average loss: {sum(epoch_losses)/len(epoch_losses):.5f} | "
            f"Elapsed: {elapsed:.2f} min"
        )

checkpoint_path = OUTPUT_DIR / "baseline_epoch1.pt"

torch.save({
    "model_state_dict": classifier_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scaler_state_dict": scaler.state_dict(),
    "epoch": 1,
    "batches_completed": len(epoch_losses),
    "average_train_loss": sum(epoch_losses) / len(epoch_losses),
}, checkpoint_path)

print("\n=== BASELINE EPOCH 1 COMPLETE ===")
print("Batches completed:", len(epoch_losses))
print("Average training loss:", round(
    sum(epoch_losses) / len(epoch_losses), 5
))
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))
print("Checkpoint:", checkpoint_path)


Batch 250/10688 | Loss: 0.11663 | Average loss: 0.13884 | Elapsed: 0.25 min
Batch 500/10688 | Loss: 0.16299 | Average loss: 0.13822 | Elapsed: 0.51 min
Batch 750/10688 | Loss: 0.07922 | Average loss: 0.13618 | Elapsed: 0.78 min
Batch 1000/10688 | Loss: 0.11573 | Average loss: 0.13580 | Elapsed: 1.04 min
Batch 1250/10688 | Loss: 0.28524 | Average loss: 0.13632 | Elapsed: 1.30 min
Batch 1500/10688 | Loss: 0.12729 | Average loss: 0.13658 | Elapsed: 1.56 min
Batch 1750/10688 | Loss: 0.13283 | Average loss: 0.13657 | Elapsed: 1.82 min
Batch 2000/10688 | Loss: 0.15064 | Average loss: 0.13664 | Elapsed: 2.08 min
Batch 2250/10688 | Loss: 0.11588 | Average loss: 0.13687 | Elapsed: 2.35 min
Batch 2500/10688 | Loss: 0.16827 | Average loss: 0.13704 | Elapsed: 2.61 min
Batch 2750/10688 | Loss: 0.12908 | Average loss: 0.13685 | Elapsed: 2.88 min
Batch 3000/10688 | Loss: 0.10062 | Average loss: 0.13681 | Elapsed: 3.14 min
Batch 3250/10688 | Loss: 0.11825 | Average loss: 0.13650 | Elapsed: 3.40 min
Ba

In [71]:

import torch
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

classifier_model.eval()
all_probs = []
all_targets = []

with torch.no_grad():
    for i, batch in enumerate(dev_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = classifier_model(input_ids, attention_mask)

        all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
        all_targets.append(labels.cpu().numpy())

        if (i + 1) % 500 == 0:
            print(f"Evaluated {i + 1}/{len(dev_loader)} batches")

probs = np.concatenate(all_probs, axis=0)
targets = np.concatenate(all_targets, axis=0).astype(int)
predictions = (probs >= 0.5).astype(int)

print("\n=== BASELINE EPOCH 1 RESULTS ===")
print("Documents:", len(targets))
print("Macro-F1:", round(f1_score(
    targets, predictions, average="macro", zero_division=0
), 5))
print("Micro-F1:", round(f1_score(
    targets, predictions, average="micro", zero_division=0
), 5))
print("Subset accuracy:", round(accuracy_score(
    targets, predictions
), 5))
print("Mean predicted labels/document:", round(
    predictions.sum(axis=1).mean(), 3
))
print("Mean actual labels/document:", round(
    targets.sum(axis=1).mean(), 3
))
print("Probability mean:", round(float(probs.mean()), 5))
print("Probability max:", round(float(probs.max()), 5))


Evaluated 500/2546 batches
Evaluated 1000/2546 batches
Evaluated 1500/2546 batches
Evaluated 2000/2546 batches
Evaluated 2500/2546 batches

=== BASELINE EPOCH 1 RESULTS ===
Documents: 10181
Macro-F1: 0.00765
Micro-F1: 0.08877
Subset accuracy: 0.01562
Mean predicted labels/document: 0.295
Mean actual labels/document: 3.916
Probability mean: 0.03596
Probability max: 0.96387


In [72]:

import numpy as np
from sklearn.metrics import f1_score, accuracy_score

print("=== EPOCH 1: THRESHOLD SEARCH ===")
results = []

for threshold in [0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.30, 0.50]:
    pred = (probs >= threshold).astype(int)

    macro = f1_score(
        targets, pred, average="macro", zero_division=0
    )
    micro = f1_score(
        targets, pred, average="micro", zero_division=0
    )
    subset = accuracy_score(targets, pred)
    avg_labels = pred.sum(axis=1).mean()

    results.append((threshold, macro, micro, subset, avg_labels))

    print(
        f"Threshold={threshold:.3f} | "
        f"Macro-F1={macro:.5f} | "
        f"Micro-F1={micro:.5f} | "
        f"Subset={subset:.5f} | "
        f"Labels/doc={avg_labels:.3f}"
    )

best_macro = max(results, key=lambda x: x[1])
best_micro = max(results, key=lambda x: x[2])

print("\n=== BEST THRESHOLDS ON DEV ===")
print("Best Macro-F1 threshold:", best_macro[0],
      "| Macro-F1:", round(best_macro[1], 5))
print("Best Micro-F1 threshold:", best_micro[0],
      "| Micro-F1:", round(best_micro[2], 5))



=== EPOCH 1: THRESHOLD SEARCH ===
Threshold=0.005 | Macro-F1=0.07545 | Micro-F1=0.08644 | Subset=0.00000 | Labels/doc=85.380
Threshold=0.010 | Macro-F1=0.08363 | Micro-F1=0.11035 | Subset=0.00000 | Labels/doc=62.470
Threshold=0.020 | Macro-F1=0.09166 | Micro-F1=0.15414 | Subset=0.00000 | Labels/doc=38.061
Threshold=0.030 | Macro-F1=0.08901 | Micro-F1=0.18627 | Subset=0.00000 | Labels/doc=27.283
Threshold=0.050 | Macro-F1=0.08300 | Micro-F1=0.22461 | Subset=0.00000 | Labels/doc=18.332
Threshold=0.075 | Macro-F1=0.07757 | Micro-F1=0.25727 | Subset=0.00000 | Labels/doc=12.675
Threshold=0.100 | Macro-F1=0.07606 | Micro-F1=0.27602 | Subset=0.00000 | Labels/doc=9.082
Threshold=0.150 | Macro-F1=0.06519 | Micro-F1=0.28965 | Subset=0.00157 | Labels/doc=4.977
Threshold=0.200 | Macro-F1=0.05098 | Micro-F1=0.27114 | Subset=0.00658 | Labels/doc=2.850
Threshold=0.300 | Macro-F1=0.02632 | Micro-F1=0.17332 | Subset=0.01621 | Labels/doc=0.882
Threshold=0.500 | Macro-F1=0.00765 | Micro-F1=0.08877 | Subs

In [73]:

import time
import torch
import torch.nn.functional as F
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/phase5_exp1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

classifier_model.train()
epoch2_losses = []
start_time = time.time()

for step, batch in enumerate(train_loader, start=1):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        logits = classifier_model(input_ids, attention_mask)
        loss = F.binary_cross_entropy_with_logits(logits, labels)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at batch {step}")

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(
        classifier_model.parameters(), max_norm=1.0
    )
    scaler.step(optimizer)
    scaler.update()

    epoch2_losses.append(loss.item())

    if step % 1000 == 0:
        print(
            f"Epoch 2 | Batch {step}/{len(train_loader)} | "
            f"Average loss: "
            f"{sum(epoch2_losses)/len(epoch2_losses):.5f} | "
            f"Elapsed: {(time.time()-start_time)/60:.2f} min"
        )

torch.save({
    "model_state_dict": classifier_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scaler_state_dict": scaler.state_dict(),
    "epoch": 2,
    "average_train_loss": sum(epoch2_losses)/len(epoch2_losses),
}, OUTPUT_DIR / "baseline_epoch2.pt")

print("\n=== EPOCH 2 COMPLETE ===")
print("Average loss:", round(sum(epoch2_losses)/len(epoch2_losses), 5))
print("Elapsed minutes:", round((time.time()-start_time)/60, 2))
print("Checkpoint saved:", OUTPUT_DIR / "baseline_epoch2.pt")


Epoch 2 | Batch 1000/10688 | Average loss: 0.13035 | Elapsed: 1.04 min
Epoch 2 | Batch 2000/10688 | Average loss: 0.12914 | Elapsed: 2.09 min
Epoch 2 | Batch 3000/10688 | Average loss: 0.12876 | Elapsed: 3.14 min
Epoch 2 | Batch 4000/10688 | Average loss: 0.12817 | Elapsed: 4.18 min
Epoch 2 | Batch 5000/10688 | Average loss: 0.12749 | Elapsed: 5.23 min
Epoch 2 | Batch 6000/10688 | Average loss: 0.12712 | Elapsed: 6.28 min
Epoch 2 | Batch 7000/10688 | Average loss: 0.12684 | Elapsed: 7.32 min
Epoch 2 | Batch 8000/10688 | Average loss: 0.12660 | Elapsed: 8.37 min
Epoch 2 | Batch 9000/10688 | Average loss: 0.12638 | Elapsed: 9.42 min
Epoch 2 | Batch 10000/10688 | Average loss: 0.12599 | Elapsed: 10.48 min

=== EPOCH 2 COMPLETE ===
Average loss: 0.12578
Elapsed minutes: 11.21
Checkpoint saved: /kaggle/working/phase5_exp1/baseline_epoch2.pt


In [74]:

import torch
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

classifier_model.eval()
all_probs = []
all_targets = []

with torch.no_grad():
    for i, batch in enumerate(dev_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = classifier_model(input_ids, attention_mask)

        all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
        all_targets.append(labels.cpu().numpy())

        if (i + 1) % 500 == 0:
            print(f"Evaluated {i + 1}/{len(dev_loader)} batches")

probs = np.concatenate(all_probs, axis=0)
targets = np.concatenate(all_targets, axis=0).astype(int)

print("\n=== BASELINE EPOCH 2 RESULTS ===")
for threshold in [0.05, 0.10, 0.15, 0.20, 0.50]:
    pred = (probs >= threshold).astype(int)
    print(
        f"Threshold={threshold:.2f} | "
        f"Macro-F1={f1_score(targets, pred, average='macro', zero_division=0):.5f} | "
        f"Micro-F1={f1_score(targets, pred, average='micro', zero_division=0):.5f} | "
        f"Subset accuracy={accuracy_score(targets, pred):.5f} | "
        f"Labels/doc={pred.sum(axis=1).mean():.3f}"
    )


Evaluated 500/2546 batches
Evaluated 1000/2546 batches
Evaluated 1500/2546 batches
Evaluated 2000/2546 batches
Evaluated 2500/2546 batches

=== BASELINE EPOCH 2 RESULTS ===
Threshold=0.05 | Macro-F1=0.11583 | Micro-F1=0.24942 | Subset accuracy=0.00000 | Labels/doc=16.994
Threshold=0.10 | Macro-F1=0.11044 | Micro-F1=0.31387 | Subset accuracy=0.00010 | Labels/doc=8.775
Threshold=0.15 | Macro-F1=0.10060 | Micro-F1=0.33515 | Subset accuracy=0.00226 | Labels/doc=5.345
Threshold=0.20 | Macro-F1=0.08794 | Micro-F1=0.32372 | Subset accuracy=0.00589 | Labels/doc=3.387
Threshold=0.50 | Macro-F1=0.01636 | Micro-F1=0.10333 | Subset accuracy=0.01640 | Labels/doc=0.328


In [75]:

import time
import torch
import torch.nn.functional as F
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/phase5_exp1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

classifier_model.train()
epoch3_losses = []
start_time = time.time()

for step, batch in enumerate(train_loader, start=1):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        logits = classifier_model(input_ids, attention_mask)
        loss = F.binary_cross_entropy_with_logits(logits, labels)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at batch {step}")

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(
        classifier_model.parameters(), max_norm=1.0
    )
    scaler.step(optimizer)
    scaler.update()

    epoch3_losses.append(loss.item())

    if step % 1000 == 0:
        print(
            f"Epoch 3 | Batch {step}/{len(train_loader)} | "
            f"Average loss: {sum(epoch3_losses)/len(epoch3_losses):.5f} | "
            f"Elapsed: {(time.time()-start_time)/60:.2f} min"
        )

torch.save({
    "model_state_dict": classifier_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scaler_state_dict": scaler.state_dict(),
    "epoch": 3,
    "average_train_loss": sum(epoch3_losses)/len(epoch3_losses),
}, OUTPUT_DIR / "baseline_epoch3.pt")

print("\n=== EPOCH 3 COMPLETE ===")
print("Average loss:", round(sum(epoch3_losses)/len(epoch3_losses), 5))
print("Elapsed minutes:", round((time.time()-start_time)/60, 2))
print("Checkpoint saved:", OUTPUT_DIR / "baseline_epoch3.pt")


Epoch 3 | Batch 1000/10688 | Average loss: 0.12236 | Elapsed: 1.05 min
Epoch 3 | Batch 2000/10688 | Average loss: 0.12177 | Elapsed: 2.10 min
Epoch 3 | Batch 3000/10688 | Average loss: 0.12192 | Elapsed: 3.15 min
Epoch 3 | Batch 4000/10688 | Average loss: 0.12084 | Elapsed: 4.21 min
Epoch 3 | Batch 5000/10688 | Average loss: 0.12073 | Elapsed: 5.26 min
Epoch 3 | Batch 6000/10688 | Average loss: 0.12051 | Elapsed: 6.30 min
Epoch 3 | Batch 7000/10688 | Average loss: 0.12039 | Elapsed: 7.36 min
Epoch 3 | Batch 8000/10688 | Average loss: 0.12044 | Elapsed: 8.40 min
Epoch 3 | Batch 9000/10688 | Average loss: 0.12046 | Elapsed: 9.45 min
Epoch 3 | Batch 10000/10688 | Average loss: 0.12026 | Elapsed: 10.49 min

=== EPOCH 3 COMPLETE ===
Average loss: 0.12026
Elapsed minutes: 11.22
Checkpoint saved: /kaggle/working/phase5_exp1/baseline_epoch3.pt


In [76]:

import torch
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

classifier_model.eval()
all_probs = []
all_targets = []

with torch.no_grad():
    for i, batch in enumerate(dev_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = classifier_model(input_ids, attention_mask)

        all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
        all_targets.append(labels.cpu().numpy())

        if (i + 1) % 500 == 0:
            print(f"Evaluated {i + 1}/{len(dev_loader)} batches")

probs = np.concatenate(all_probs, axis=0)
targets = np.concatenate(all_targets, axis=0).astype(int)

print("\n=== BASELINE EPOCH 3 RESULTS ===")
for threshold in [0.05, 0.10, 0.15, 0.20, 0.50]:
    pred = (probs >= threshold).astype(int)
    print(
        f"Threshold={threshold:.2f} | "
        f"Macro-F1={f1_score(targets, pred, average='macro', zero_division=0):.5f} | "
        f"Micro-F1={f1_score(targets, pred, average='micro', zero_division=0):.5f} | "
        f"Subset accuracy={accuracy_score(targets, pred):.5f} | "
        f"Labels/doc={pred.sum(axis=1).mean():.3f}"
    )


Evaluated 500/2546 batches
Evaluated 1000/2546 batches
Evaluated 1500/2546 batches
Evaluated 2000/2546 batches
Evaluated 2500/2546 batches

=== BASELINE EPOCH 3 RESULTS ===
Threshold=0.05 | Macro-F1=0.13868 | Micro-F1=0.25799 | Subset accuracy=0.00000 | Labels/doc=17.337
Threshold=0.10 | Macro-F1=0.14061 | Micro-F1=0.32781 | Subset accuracy=0.00020 | Labels/doc=9.082
Threshold=0.15 | Macro-F1=0.13128 | Micro-F1=0.35527 | Subset accuracy=0.00432 | Labels/doc=5.802
Threshold=0.20 | Macro-F1=0.12136 | Micro-F1=0.35622 | Subset accuracy=0.01159 | Labels/doc=3.942
Threshold=0.50 | Macro-F1=0.03863 | Micro-F1=0.12502 | Subset accuracy=0.01689 | Labels/doc=0.405


In [78]:

import json
from pathlib import Path

out = Path("/kaggle/working/phase5_exp1")
out.mkdir(parents=True, exist_ok=True)

baseline_results = {
    "experiment": "Experiment 1 - A4 Full Fine-Tuning Baseline",
    "best_epoch": 3,
    "checkpoint": str(out / "baseline_epoch3.pt"),
    "threshold_results": {
        "0.10": {"macro_f1": 0.14061, "micro_f1": 0.32781},
        "0.15": {"macro_f1": 0.13128, "micro_f1": 0.35527},
        "0.20": {"macro_f1": 0.12136, "micro_f1": 0.35622}
    },
    "primary_metric": "Macro-F1",
    "note": "Thresholds evaluated on development set"
}

with open(out / "baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

print("Baseline results saved:", out / "baseline_results.json")
print("Best baseline checkpoint:", out / "baseline_epoch3.pt")
print("Experiment 1 recorded successfully.")


Baseline results saved: /kaggle/working/phase5_exp1/baseline_results.json
Best baseline checkpoint: /kaggle/working/phase5_exp1/baseline_epoch3.pt
Experiment 1 recorded successfully.


In [79]:

import copy
import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.optim import AdamW

EXP2_DIR = Path("/kaggle/working/phase5_exp2_weighted_bce")
EXP2_DIR.mkdir(parents=True, exist_ok=True)

# Load original A4 weights; do not use the fine-tuned baseline weights.
original_ckpt = torch.load(
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
    map_location="cpu",
    weights_only=False
)
original_state = original_ckpt["model_state_dict"]

# Create a separate encoder so Experiment 1 remains unchanged in memory.
exp2_encoder = copy.deepcopy(classifier_model.encoder)
exp2_encoder.load_state_dict(original_state, strict=True)

exp2_model = A4StatuteClassifier(
    exp2_encoder, num_labels=100, dropout=0.2
).to(device)

# Count positive labels using training data only.
positive_counts = np.zeros(100, dtype=np.float64)

for row in train_data:
    y = np.asarray(row["labels"])
    if y.ndim == 1 and len(y) == 100 and np.isin(y, [0, 1]).all():
        positive_counts += y
    else:
        positive_counts[np.asarray(y, dtype=int)] += 1

negative_counts = len(train_data) - positive_counts

# Capped positive weights reduce the impact of extreme rare-label ratios.
pos_weights = np.sqrt(
    negative_counts / np.maximum(positive_counts, 1)
)
pos_weights = np.clip(pos_weights, 1.0, 10.0)

pos_weight_tensor = torch.tensor(
    pos_weights, dtype=torch.float32, device=device
)

exp2_optimizer = AdamW([
    {
        "params": list(exp2_model.encoder.parameters()),
        "lr": 2e-5
    },
    {
        "params": list(exp2_model.classifier.parameters()),
        "lr": 2e-4
    }
], weight_decay=0.01)

exp2_scaler = torch.amp.GradScaler(
    "cuda", enabled=(device.type == "cuda")
)

print("=== EXPERIMENT 2 SETUP ===")
print("Experiment: Weighted BCE")
print("Original A4 step:", original_ckpt.get("step"))
print("Fresh classifier:", type(exp2_model).__name__)
print("Training documents:", len(train_data))
print("Labels:", len(pos_weights))
print("Positive weight range:",
      round(float(pos_weights.min()), 3),
      "to", round(float(pos_weights.max()), 3))
print("Baseline checkpoint preserved:",
      Path("/kaggle/working/phase5_exp1/baseline_epoch3.pt").exists())
print("Experiment 2 directory:", EXP2_DIR)


=== EXPERIMENT 2 SETUP ===
Experiment: Weighted BCE
Original A4 step: 22000
Fresh classifier: A4StatuteClassifier
Training documents: 42750
Labels: 100
Positive weight range: 1.626 to 10.0
Baseline checkpoint preserved: True
Experiment 2 directory: /kaggle/working/phase5_exp2_weighted_bce


In [80]:

import time
import torch.nn.functional as F

exp2_model.train()
exp2_losses = []
start_time = time.time()

for step, batch in enumerate(train_loader, start=1):
    ids = batch["input_ids"].to(device)
    mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    exp2_optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        logits = exp2_model(ids, mask)
        loss = F.binary_cross_entropy_with_logits(
            logits, labels, pos_weight=pos_weight_tensor
        )

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at batch {step}")

    exp2_scaler.scale(loss).backward()
    exp2_scaler.unscale_(exp2_optimizer)
    torch.nn.utils.clip_grad_norm_(
        exp2_model.parameters(), max_norm=1.0
    )
    exp2_scaler.step(exp2_optimizer)
    exp2_scaler.update()

    exp2_losses.append(loss.item())

    if step % 1000 == 0:
        print(
            f"Exp 2 | Batch {step}/{len(train_loader)} | "
            f"Average loss: {np.mean(exp2_losses):.5f} | "
            f"Elapsed: {(time.time()-start_time)/60:.2f} min"
        )

exp2_checkpoint = EXP2_DIR / "weighted_bce_epoch1.pt"

torch.save({
    "model_state_dict": exp2_model.state_dict(),
    "optimizer_state_dict": exp2_optimizer.state_dict(),
    "scaler_state_dict": exp2_scaler.state_dict(),
    "epoch": 1,
    "average_train_loss": float(np.mean(exp2_losses)),
    "positive_weights": pos_weights.tolist()
}, exp2_checkpoint)

print("\n=== EXPERIMENT 2 COMPLETE ===")
print("Batches:", len(exp2_losses))
print("Average weighted training loss:", round(float(np.mean(exp2_losses)), 5))
print("Elapsed minutes:", round((time.time()-start_time)/60, 2))
print("Checkpoint:", exp2_checkpoint)


Exp 2 | Batch 1000/10688 | Average loss: 0.44562 | Elapsed: 1.05 min
Exp 2 | Batch 2000/10688 | Average loss: 0.44499 | Elapsed: 2.10 min
Exp 2 | Batch 3000/10688 | Average loss: 0.44256 | Elapsed: 3.15 min
Exp 2 | Batch 4000/10688 | Average loss: 0.43983 | Elapsed: 4.19 min
Exp 2 | Batch 5000/10688 | Average loss: 0.43707 | Elapsed: 5.24 min
Exp 2 | Batch 6000/10688 | Average loss: 0.43520 | Elapsed: 6.30 min
Exp 2 | Batch 7000/10688 | Average loss: 0.43289 | Elapsed: 7.36 min
Exp 2 | Batch 8000/10688 | Average loss: 0.43058 | Elapsed: 8.41 min
Exp 2 | Batch 9000/10688 | Average loss: 0.42896 | Elapsed: 9.45 min
Exp 2 | Batch 10000/10688 | Average loss: 0.42751 | Elapsed: 10.49 min

=== EXPERIMENT 2 COMPLETE ===
Batches: 10688
Average weighted training loss: 0.4261
Elapsed minutes: 11.23
Checkpoint: /kaggle/working/phase5_exp2_weighted_bce/weighted_bce_epoch1.pt


In [81]:

import torch
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

exp2_model.eval()
exp2_probs = []
exp2_targets = []

with torch.no_grad():
    for i, batch in enumerate(dev_loader):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = exp2_model(ids, mask)

        exp2_probs.append(torch.sigmoid(logits).float().cpu().numpy())
        exp2_targets.append(labels.cpu().numpy())

        if (i + 1) % 500 == 0:
            print(f"Evaluated {i + 1}/{len(dev_loader)} batches")

exp2_probs = np.concatenate(exp2_probs, axis=0)
exp2_targets = np.concatenate(exp2_targets, axis=0).astype(int)

print("\n=== EXPERIMENT 2: WEIGHTED BCE RESULTS ===")
for threshold in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    pred = (exp2_probs >= threshold).astype(int)
    print(
        f"Threshold={threshold:.2f} | "
        f"Macro-F1={f1_score(exp2_targets, pred, average='macro', zero_division=0):.5f} | "
        f"Micro-F1={f1_score(exp2_targets, pred, average='micro', zero_division=0):.5f} | "
        f"Subset accuracy={accuracy_score(exp2_targets, pred):.5f} | "
        f"Labels/doc={pred.sum(axis=1).mean():.3f}"
    )


Evaluated 500/2546 batches
Evaluated 1000/2546 batches
Evaluated 1500/2546 batches
Evaluated 2000/2546 batches
Evaluated 2500/2546 batches

=== EXPERIMENT 2: WEIGHTED BCE RESULTS ===
Threshold=0.05 | Macro-F1=0.07937 | Micro-F1=0.09073 | Subset accuracy=0.00000 | Labels/doc=80.981
Threshold=0.10 | Macro-F1=0.09519 | Micro-F1=0.12929 | Subset accuracy=0.00000 | Labels/doc=50.358
Threshold=0.15 | Macro-F1=0.10644 | Micro-F1=0.17256 | Subset accuracy=0.00000 | Labels/doc=31.407
Threshold=0.20 | Macro-F1=0.11186 | Micro-F1=0.21559 | Subset accuracy=0.00000 | Labels/doc=20.208
Threshold=0.30 | Macro-F1=0.10489 | Micro-F1=0.26976 | Subset accuracy=0.00079 | Labels/doc=8.431
Threshold=0.50 | Macro-F1=0.05447 | Micro-F1=0.19662 | Subset accuracy=0.01051 | Labels/doc=1.780


In [82]:

from pathlib import Path

root = Path("/kaggle/input")
matches = []

for path in root.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".pt", ".pth", ".ckpt"}:
        name = path.name.lower()
        if any(term in name for term in ["step", "checkpoint", "model", "phase1", "a1", "a2", "a3", "a4"]):
            matches.append(path)

print("=== PHASE 1 CHECKPOINT SEARCH ===")
print("Checkpoint files found:", len(matches))

for path in matches[:100]:
    print(path)


=== PHASE 1 CHECKPOINT SEARCH ===
Checkpoint files found: 4
/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt
/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt
/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt


In [83]:

import torch
from pathlib import Path

checkpoint_paths = [
    "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
    "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
]

for path in checkpoint_paths:
    print("\n" + "=" * 80)
    print("FILE:", Path(path).name)

    checkpoint = torch.load(path, map_location="cpu", weights_only=False)

    if isinstance(checkpoint, dict):
        print("Top-level keys:", list(checkpoint.keys())[:20])

        # Find the model state dictionary.
        state = None
        for key in ["model_state_dict", "state_dict", "model", "encoder", "model_state"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                state = checkpoint[key]
                print("State dictionary key:", key)
                break

        if state is None and any(torch.is_tensor(v) for v in checkpoint.values()):
            state = checkpoint

        if state is not None:
            print("Parameter tensors:", sum(torch.is_tensor(v) for v in state.values()))
            for name, tensor in list(state.items())[:12]:
                if torch.is_tensor(tensor):
                    print(f"  {name}: {tuple(tensor.shape)}")
    else:
        print("Checkpoint object type:", type(checkpoint))



FILE: step_0001000.pt
Top-level keys: ['model', 'step', 'tokens_seen', 'architecture', 'parameters', 'context_length', 'hidden_size', 'num_layers', 'num_heads', 'ffn_size', 'prefix_ratio', 'dropout', 'learning_rate', 'weight_decay']
State dictionary key: model
Parameter tensors: 100
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512)
  layers.0.ln1.weight: (512,)
  layers.0.ln1.bias: (512,)
  layers.0.attn.in_proj_weight: (1536, 512)
  layers.0.attn.in_proj_bias: (1536,)
  layers.0.attn.out_proj.weight: (512, 512)
  layers.0.attn.out_proj.bias: (512,)
  layers.0.ln2.weight: (512,)
  layers.0.ln2.bias: (512,)
  layers.0.ffn.0.weight: (2048, 512)
  layers.0.ffn.0.bias: (2048,)

FILE: architecture1_sustained_step_500.pt
Top-level keys: ['step', 'epoch', 'train_loss', 'val_loss', 'model_state_dict', 'optimizer_state_dict']
State dictionary key: model_state_dict
Parameter tensors: 101
  token_embedding.weight: (50000, 512)
  position_embedding.weight: (2048, 512

In [84]:

from pathlib import Path

root = Path("/kaggle/input")
matches = []

for path in root.rglob("*.ipynb"):
    name = path.name.lower()
    if any(term in name for term in ["a2", "architecture2", "phase1", "phase-1"]):
        matches.append(path)

print("Candidate Phase 1 notebooks:", len(matches))
for path in matches[:60]:
    print(path)


Candidate Phase 1 notebooks: 3
/kaggle/input/datasets/aq1de3/p1p1p1/a2a2a2.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb


In [85]:

import json
from pathlib import Path

path = Path("/kaggle/input/datasets/aq1de3/p1p1p1/a2a2a2.ipynb")
nb = json.loads(path.read_text())

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if any(term in source for term in [
        "class GQAAttention",
        "class A2",
        "class Transformer",
        "class Decoder",
        "D_MODEL =",
        "N_KV_HEADS =",
    ]):
        print(f"\n{'='*30} CELL {i} {'='*30}")
        print(source[:7000])



============================== CELL 0 ==============================
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = 50_000
D_MODEL = 528
NUM_LAYERS = 8
NUM_Q_HEADS = 8
NUM_KV_HEADS = 2
FFN_DIM = 2048
MAX_SEQ_LEN = 2048
DROPOUT = 0.1


class GQAAttention(nn.Module):
    def __init__(self, d_model, num_q_heads, num_kv_heads, dropout=0.1):
        super().__init__()

        self.d_model = d_model
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_q_heads

        assert d_model % num_q_heads == 0

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_p

In [86]:

import torch

path = "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)
sd = ckpt["model_state_dict"]

print("Checkpoint step:", ckpt.get("step"))
print("Training loss:", ckpt.get("train_loss"))
print("\nModel configuration clues:")

for key in [
    "token_embedding.weight",
    "position_embedding.weight",
    "layers.0.norm1.weight",
    "layers.0.attention.q_proj.weight",
    "layers.0.attention.k_proj.weight",
    "layers.0.attention.v_proj.weight",
    "layers.0.attention.out_proj.weight",
    "layers.0.ffn.fc1.weight",
    "layers.0.ffn.fc2.weight",
    "final_norm.weight",
    "output_bias",
]:
    if key in sd:
        print(f"{key}: {tuple(sd[key].shape)}")

print("\nLayer indices:")
print(sorted({
    int(k.split(".")[1])
    for k in sd
    if k.startswith("layers.") and k.split(".")[1].isdigit()
}))



Checkpoint step: 9000
Training loss: 6.801188290119171

Model configuration clues:
token_embedding.weight: (50000, 512)
position_embedding.weight: (2048, 512)
layers.0.norm1.weight: (512,)
layers.0.attention.q_proj.weight: (512, 512)
layers.0.attention.k_proj.weight: (128, 512)
layers.0.attention.v_proj.weight: (128, 512)
layers.0.attention.out_proj.weight: (512, 512)
layers.0.ffn.fc1.weight: (2048, 512)
layers.0.ffn.fc2.weight: (512, 2048)
final_norm.weight: (512,)
output_bias: (50000,)

Layer indices:
[0, 1, 2, 3, 4, 5, 6, 7, 8]


In [87]:

import torch

path = "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)
sd = ckpt["model_state_dict"]

for name, tensor in sd.items():
    if name.startswith("layers.8.") or name.startswith("final_norm") or name.startswith("output_bias"):
        print(f"{name}: {tuple(tensor.shape)}")

print("\nCheckpoint metadata:")
for key, value in ckpt.items():
    if key != "model_state_dict" and key != "optimizer_state_dict":
        print(f"{key}: {value}")


output_bias: (50000,)
layers.8.norm1.weight: (512,)
layers.8.norm1.bias: (512,)
layers.8.attention.q_proj.weight: (512, 512)
layers.8.attention.k_proj.weight: (128, 512)
layers.8.attention.v_proj.weight: (128, 512)
layers.8.attention.out_proj.weight: (512, 512)
layers.8.norm2.weight: (512,)
layers.8.norm2.bias: (512,)
layers.8.ffn.fc1.weight: (2048, 512)
layers.8.ffn.fc1.bias: (2048,)
layers.8.ffn.fc2.weight: (512, 2048)
layers.8.ffn.fc2.bias: (512,)
final_norm.weight: (512,)
final_norm.bias: (512,)

Checkpoint metadata:
step: 9000
tokens_processed: 73728000
train_loss: 6.801188290119171
reason: regular_checkpoint
saved_at: 2026-10-05T19:17:55.016904


In [88]:

import torch

path = "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)
sd = ckpt["model_state_dict"]

print("Total parameter tensors:", len(sd))
print("\nAll checkpoint keys:")
for name, tensor in sd.items():
    print(f"{name:55s} {tuple(tensor.shape)}")

print("\nMetadata:")
print("Step:", ckpt["step"])
print("Tokens processed:", ckpt["tokens_processed"])
print("Original training loss:", ckpt["train_loss"])


Total parameter tensors: 113

All checkpoint keys:
output_bias                                             (50000,)
token_embedding.weight                                  (50000, 512)
position_embedding.weight                               (2048, 512)
layers.0.norm1.weight                                   (512,)
layers.0.norm1.bias                                     (512,)
layers.0.attention.q_proj.weight                        (512, 512)
layers.0.attention.k_proj.weight                        (128, 512)
layers.0.attention.v_proj.weight                        (128, 512)
layers.0.attention.out_proj.weight                      (512, 512)
layers.0.norm2.weight                                   (512,)
layers.0.norm2.bias                                     (512,)
layers.0.ffn.fc1.weight                                 (2048, 512)
layers.0.ffn.fc1.bias                                   (2048,)
layers.0.ffn.fc2.weight                                 (512, 2048)
layers.0.ffn.fc2.bias      

In [89]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class A2GQAAttention(nn.Module):
    def __init__(self, d_model=512, num_q_heads=8, num_kv_heads=2):
        super().__init__()
        self.d_model = d_model
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_q_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, attention_mask=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, 8, 64).transpose(1, 2)
        k = self.k_proj(x).view(B, T, 2, 64).transpose(1, 2)
        v = self.v_proj(x).view(B, T, 2, 64).transpose(1, 2)

        k = k.repeat_interleave(4, dim=1)
        v = v.repeat_interleave(4, dim=1)

        mask = None
        if attention_mask is not None:
            mask = attention_mask.bool()[:, None, None, :]

        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y)


class A2GQABlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(512)
        self.attention = A2GQAAttention()
        self.norm2 = nn.LayerNorm(512)
        self.ffn = nn.Module()
        self.ffn.fc1 = nn.Linear(512, 2048)
        self.ffn.fc2 = nn.Linear(2048, 512)

    def forward(self, x, attention_mask=None):
        x = x + self.attention(self.norm1(x), attention_mask)
        x = x + self.ffn.fc2(F.gelu(self.ffn.fc1(self.norm2(x))))
        return x


class A2GQALanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(50000, 512)
        self.position_embedding = nn.Embedding(2048, 512)
        self.layers = nn.ModuleList([A2GQABlock() for _ in range(9)])
        self.final_norm = nn.LayerNorm(512)
        self.output_bias = nn.Parameter(torch.zeros(50000))

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)

        for layer in self.layers:
            x = layer(x, attention_mask)

        return self.final_norm(x)


a2_checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt"
)
a2_ckpt = torch.load(
    a2_checkpoint_path, map_location="cpu", weights_only=False
)

a2_backbone = A2GQALanguageModel()
load_result = a2_backbone.load_state_dict(
    a2_ckpt["model_state_dict"], strict=True
)

print("Strict checkpoint loading succeeded:", load_result)
print("Transformer blocks:", len(a2_backbone.layers))
print("Embedding shape:", tuple(a2_backbone.token_embedding.weight.shape))
print("A2 checkpoint step:", a2_ckpt["step"])


Strict checkpoint loading succeeded: <All keys matched successfully>
Transformer blocks: 9
Embedding shape: (50000, 512)
A2 checkpoint step: 9000


In [90]:

import torch
import torch.nn as nn

class A2StatuteClassifier(nn.Module):
    def __init__(self, backbone, num_labels=100, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(512, num_labels)

    def forward(self, input_ids, attention_mask):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device)

        x = (
            self.backbone.token_embedding(input_ids)
            + self.backbone.position_embedding(positions)
        )

        for layer in self.backbone.layers:
            x = layer(x, attention_mask)

        x = self.backbone.final_norm(x)

        mask = attention_mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * mask).sum(dim=1)
        pooled = pooled / mask.sum(dim=1).clamp(min=1.0)

        return self.classifier(self.dropout(pooled))


# Start from the verified Phase 1 A2 checkpoint.
a2_backbone = A2GQALanguageModel()
a2_backbone.load_state_dict(a2_ckpt["model_state_dict"], strict=True)

# Freeze the backbone initially.
for param in a2_backbone.parameters():
    param.requires_grad = False

# Unfreeze the top three transformer blocks and final normalization.
for layer in a2_backbone.layers[-3:]:
    for param in layer.parameters():
        param.requires_grad = True

for param in a2_backbone.final_norm.parameters():
    param.requires_grad = True

exp3_model = A2StatuteClassifier(a2_backbone).to(device)

# The classification head is trainable.
for param in exp3_model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in exp3_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in exp3_model.parameters())

print("Experiment 3 model ready.")
print(f"Trainable parameters: {trainable:,} / {total:,}")
print(f"Trainable percentage: {100 * trainable / total:.2f}%")
print("Frozen: embeddings and bottom 6 transformer blocks.")
print("Trainable: top 3 blocks, final norm, classification head.")


Experiment 3 model ready.
Trainable parameters: 8,323,684 / 51,564,980
Trainable percentage: 16.14%
Frozen: embeddings and bottom 6 transformer blocks.
Trainable: top 3 blocks, final norm, classification head.


In [91]:

import os, time, torch
import torch.nn as nn
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

exp3_dir = "/kaggle/working/phase5_exp3_a2_partial"
os.makedirs(exp3_dir, exist_ok=True)

# Separate learning rates for pretrained layers and the new classifier.
backbone_params = [
    p for p in exp3_model.backbone.parameters() if p.requires_grad
]
head_params = list(exp3_model.classifier.parameters())

exp3_optimizer = AdamW(
    [
        {"params": backbone_params, "lr": 1e-5},
        {"params": head_params, "lr": 2e-4},
    ],
    weight_decay=0.01,
)

exp3_scaler = GradScaler("cuda", enabled=(device.type == "cuda"))
exp3_criterion = nn.BCEWithLogitsLoss()

# One-epoch smoke test; save the checkpoint when complete.
exp3_model.train()
running_loss = 0.0
start_time = time.time()

for step, batch in enumerate(train_loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device).float()

    exp3_optimizer.zero_grad(set_to_none=True)

    with autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == "cuda"),
    ):
        logits = exp3_model(input_ids, attention_mask)
        loss = exp3_criterion(logits, labels)

    exp3_scaler.scale(loss).backward()
    exp3_scaler.unscale_(exp3_optimizer)
    torch.nn.utils.clip_grad_norm_(
        [p for p in exp3_model.parameters() if p.requires_grad], 1.0
    )
    exp3_scaler.step(exp3_optimizer)
    exp3_scaler.update()

    running_loss += loss.item()

    if (step + 1) % 500 == 0:
        print(
            f"Step {step+1}/{len(train_loader)} | "
            f"Mean loss: {running_loss/(step+1):.5f} | "
            f"Elapsed: {(time.time()-start_time)/60:.1f} min"
        )

    # Optional early interruption safeguard: keep progress recoverable.
    if (step + 1) % 1000 == 0:
        torch.save(
            {
                "step": step + 1,
                "model_state_dict": exp3_model.state_dict(),
                "optimizer_state_dict": exp3_optimizer.state_dict(),
                "mean_loss": running_loss / (step + 1),
            },
            f"{exp3_dir}/a2_partial_step_{step+1}.pt",
        )

exp3_train_loss = running_loss / len(train_loader)

torch.save(
    {
        "model_state_dict": exp3_model.state_dict(),
        "optimizer_state_dict": exp3_optimizer.state_dict(),
        "train_loss": exp3_train_loss,
        "experiment": "A2 partial fine-tuning",
    },
    f"{exp3_dir}/a2_partial_epoch1.pt",
)

print(f"\nExperiment 3 training loss: {exp3_train_loss:.5f}")
print(f"Elapsed: {(time.time()-start_time)/60:.1f} minutes")
print("Checkpoint:", f"{exp3_dir}/a2_partial_epoch1.pt")


Step 500/10688 | Mean loss: 0.16496 | Elapsed: 0.3 min
Step 1000/10688 | Mean loss: 0.15271 | Elapsed: 0.6 min
Step 1500/10688 | Mean loss: 0.14837 | Elapsed: 0.9 min
Step 2000/10688 | Mean loss: 0.14642 | Elapsed: 1.2 min
Step 2500/10688 | Mean loss: 0.14499 | Elapsed: 1.5 min
Step 3000/10688 | Mean loss: 0.14390 | Elapsed: 1.8 min
Step 3500/10688 | Mean loss: 0.14269 | Elapsed: 2.1 min
Step 4000/10688 | Mean loss: 0.14223 | Elapsed: 2.4 min
Step 4500/10688 | Mean loss: 0.14181 | Elapsed: 2.7 min
Step 5000/10688 | Mean loss: 0.14118 | Elapsed: 2.9 min
Step 5500/10688 | Mean loss: 0.14104 | Elapsed: 3.2 min
Step 6000/10688 | Mean loss: 0.14105 | Elapsed: 3.5 min
Step 6500/10688 | Mean loss: 0.14064 | Elapsed: 3.8 min
Step 7000/10688 | Mean loss: 0.14056 | Elapsed: 4.1 min
Step 7500/10688 | Mean loss: 0.14038 | Elapsed: 4.4 min
Step 8000/10688 | Mean loss: 0.14025 | Elapsed: 4.7 min
Step 8500/10688 | Mean loss: 0.14024 | Elapsed: 5.0 min
Step 9000/10688 | Mean loss: 0.14011 | Elapsed: 5

In [92]:

import numpy as np
import torch
from sklearn.metrics import f1_score, accuracy_score
from torch.amp import autocast

exp3_model.eval()
all_probs, all_targets = [], []

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()

        with autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            logits = exp3_model(input_ids, attention_mask)

        all_probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        all_targets.append(labels)

probs_exp3 = np.concatenate(all_probs)
targets_exp3 = np.concatenate(all_targets).astype(int)

print("Development examples:", len(targets_exp3))
print("Probability shape:", probs_exp3.shape)
print("\nThreshold | Macro-F1 | Micro-F1 | Subset Acc | Avg labels/doc")

for threshold in [0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.50]:
    preds = (probs_exp3 >= threshold).astype(int)
    macro = f1_score(targets_exp3, preds, average="macro", zero_division=0)
    micro = f1_score(targets_exp3, preds, average="micro", zero_division=0)
    subset = accuracy_score(targets_exp3, preds)
    avg_labels = preds.sum(axis=1).mean()

    print(
        f"{threshold:9.3f} | {macro:.5f} | {micro:.5f} | "
        f"{subset:.5f} | {avg_labels:.3f}"
    )


Development examples: 10181
Probability shape: (10181, 100)

Threshold | Macro-F1 | Micro-F1 | Subset Acc | Avg labels/doc
    0.050 | 0.04122 | 0.19054 | 0.00000 | 20.683
    0.075 | 0.03027 | 0.22333 | 0.00000 | 11.989
    0.100 | 0.02356 | 0.23682 | 0.00000 | 8.000
    0.125 | 0.02177 | 0.24097 | 0.00000 | 6.880
    0.150 | 0.01463 | 0.21909 | 0.00432 | 3.135
    0.200 | 0.00854 | 0.18370 | 0.00835 | 2.000
    0.300 | 0.00000 | 0.00000 | 0.00000 | 0.000
    0.500 | 0.00000 | 0.00000 | 0.00000 | 0.000


In [93]:

import numpy as np

print("=== A2 DEVELOPMENT PREDICTION DIAGNOSTICS ===")

print("\nProbability distribution:")
for q in [0, 10, 25, 50, 75, 90, 95, 99, 100]:
    print(f"{q:3d}th percentile: {np.percentile(probs_exp3, q):.6f}")

print("\nTarget positive-label rate:", targets_exp3.mean())
print("Mean predicted probability:", probs_exp3.mean())
print("Mean maximum probability per document:", probs_exp3.max(axis=1).mean())

print("\nAverage positive labels per document:")
print("Actual:", targets_exp3.sum(axis=1).mean())
for threshold in [0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30]:
    print(
        f"Threshold {threshold:.3f}: "
        f"{(probs_exp3 >= threshold).sum(axis=1).mean():.3f}"
    )


=== A2 DEVELOPMENT PREDICTION DIAGNOSTICS ===

Probability distribution:
  0th percentile: 0.003173
 10th percentile: 0.007012
 25th percentile: 0.009746
 50th percentile: 0.016153
 75th percentile: 0.041618
 90th percentile: 0.086016
 95th percentile: 0.134558
 99th percentile: 0.258700
100th percentile: 0.299720

Target positive-label rate: 0.03915725370788724
Mean predicted probability: 0.03568544
Mean maximum probability per document: 0.2874779

Average positive labels per document:
Actual: 3.9157253707887243
Threshold 0.050: 20.683
Threshold 0.075: 11.989
Threshold 0.100: 8.000
Threshold 0.125: 6.880
Threshold 0.150: 3.135
Threshold 0.200: 2.000
Threshold 0.300: 0.000


In [94]:

from sklearn.metrics import f1_score
import numpy as np

thresholds = np.round(np.arange(0.05, 0.251, 0.005), 3)
results = []

for threshold in thresholds:
    preds = (probs_exp3 >= threshold).astype(np.int8)
    results.append({
        "threshold": float(threshold),
        "macro_f1": f1_score(
            targets_exp3, preds, average="macro", zero_division=0
        ),
        "micro_f1": f1_score(
            targets_exp3, preds, average="micro", zero_division=0
        ),
        "avg_labels": float(preds.sum(axis=1).mean()),
    })

best_macro = max(results, key=lambda r: r["macro_f1"])
best_micro = max(results, key=lambda r: r["micro_f1"])

print("Best Macro-F1:", best_macro)
print("Best Micro-F1:", best_micro)

print("\nTop 10 thresholds by Macro-F1:")
for row in sorted(results, key=lambda r: r["macro_f1"], reverse=True)[:10]:
    print(row)


Best Macro-F1: {'threshold': 0.05, 'macro_f1': 0.04122315304403088, 'micro_f1': 0.19053893793648702, 'avg_labels': 20.682742363225618}
Best Micro-F1: {'threshold': 0.13, 'macro_f1': 0.01948122642354337, 'micro_f1': 0.24752938106782923, 'avg_labels': 5.118848836067184}

Top 10 thresholds by Macro-F1:
{'threshold': 0.05, 'macro_f1': 0.04122315304403088, 'micro_f1': 0.19053893793648702, 'avg_labels': 20.682742363225618}
{'threshold': 0.055, 'macro_f1': 0.03931018264213091, 'micro_f1': 0.19558527967498568, 'avg_labels': 18.907081819074747}
{'threshold': 0.06, 'macro_f1': 0.036787646862000256, 'micro_f1': 0.2041679339880888, 'avg_labels': 16.106079952853353}
{'threshold': 0.065, 'macro_f1': 0.03436948802556124, 'micro_f1': 0.21070340043511446, 'avg_labels': 15.001178666142815}
{'threshold': 0.07, 'macro_f1': 0.03290865305153697, 'micro_f1': 0.21468455241146112, 'avg_labels': 13.74540811315195}
{'threshold': 0.075, 'macro_f1': 0.03026572782851, 'micro_f1': 0.2233283721274927, 'avg_labels': 1

In [95]:

print("A4 backbone parameter names:")
for name, param in backbone.named_parameters():
    if any(term in name.lower() for term in [
        "attn", "attention", "q_proj", "k_proj",
        "v_proj", "out_proj", "ffn", "linear"
    ]):
        print(f"{name:65s} {tuple(param.shape)}")


A4 backbone parameter names:
encoder_layers.0.attn.q_proj.weight                               (512, 512)
encoder_layers.0.attn.q_proj.bias                                 (512,)
encoder_layers.0.attn.k_proj.weight                               (512, 512)
encoder_layers.0.attn.k_proj.bias                                 (512,)
encoder_layers.0.attn.v_proj.weight                               (512, 512)
encoder_layers.0.attn.v_proj.bias                                 (512,)
encoder_layers.0.attn.out_proj.weight                             (512, 512)
encoder_layers.0.attn.out_proj.bias                               (512,)
encoder_layers.0.ffn.net.0.weight                                 (2048, 512)
encoder_layers.0.ffn.net.0.bias                                   (2048,)
encoder_layers.0.ffn.net.2.weight                                 (512, 2048)
encoder_layers.0.ffn.net.2.bias                                   (512,)
encoder_layers.1.attn.q_proj.weight                               (5

In [99]:

from pathlib import Path
import torch

for p in Path("/kaggle/input").rglob("*"):
    if p.is_file() and p.suffix.lower() in {".pt", ".pth", ".ckpt"}:
        print(p)


/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt
/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt
/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt


In [100]:

import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb"
)
nb = json.loads(path.read_text())

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if any(term in source.lower() for term in [
        "torch.save", "torch.load", "state_dict",
        "checkpoint", "class a3", "class architecture"
    ]):
        print(f"\n{'='*25} CELL {i} {'='*25}")
        print(source[:3500])



========================= CELL 8 =========================
# ============================================================
# A2 — FINAL MEMORY-SAFE PRODUCTION RUN
# ============================================================
#
# 50.41M parameter GQA decoder-only LM
# Official Phase-1 data
# 200M training tokens
#
# MEMORY PROTECTION:
#   Micro-batch          = 2
#   Gradient accumulation= 8
#   Effective batch      = 16
#   Sequence length      = 512
#   Activation checkpointing = ON
#   Chunked vocabulary CE    = ON
#   FP32 master weights + FP16 AMP
#   Fused AdamW
#   NO torch.compile
#
# CHECKPOINT:
#   Every 1000 optimizer steps
#   Keeps only latest 2 checkpoints
#   Automatically resumes after kernel restart
# ============================================================

import os
import gc
import math
import json
import time
import glob
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.checkpoint import checkpoint
from kag

In [101]:

from pathlib import Path

paths_to_check = [
    Path("/kaggle/working/a3_checkpoints"),
    Path("/kaggle/working/phase1_production/architecture_3_prefix_lm/checkpoints"),
]

for folder in paths_to_check:
    print(f"\nFolder: {folder}")
    print("Exists:", folder.exists())

    if folder.exists():
        for p in sorted(folder.glob("*.pt")):
            print(f"  {p} ({p.stat().st_size / 1024**2:.1f} MB)")



Folder: /kaggle/working/a3_checkpoints
Exists: False

Folder: /kaggle/working/phase1_production/architecture_3_prefix_lm/checkpoints
Exists: False


In [103]:
from pathlib import Path

for p in Path("/kaggle/input").iterdir():
    print(p.name, "->", p)

    if "33" in p.name.lower():
        for f in p.rglob("*"):
            if f.is_file():
                print("   ", f, f"({f.stat().st_size / 1024**2:.1f} MB)")

datasets -> /kaggle/input/datasets


In [104]:
from pathlib import Path

root = Path("/kaggle/input/datasets")

matches = []
for p in root.rglob("*"):
    if p.is_file():
        name = p.name.lower()
        if any(key in name for key in ["33000", "33k", "a3", "prefix"]):
            matches.append(p)

print("Matching files:", len(matches))
for p in matches:
    print(f"{p}  ({p.stat().st_size / 1024**2:.1f} MB)")

Matching files: 2
/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb  (0.9 MB)
/kaggle/input/datasets/aq1de3/33kkkk/step_0330000.pt  (593.7 MB)


In [105]:
import torch

ckpt_path = "/kaggle/input/datasets/aq1de3/33kkkk/step_0330000.pt"

ckpt = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(ckpt))

if isinstance(ckpt, dict):
    print("\nTop-level keys:", list(ckpt.keys()))

    for key, value in ckpt.items():
        if isinstance(value, dict):
            print(f"\n{key}: dictionary with {len(value)} entries")
            print("Sample keys:", list(value.keys())[:12])
        elif isinstance(value, (int, float, str)):
            print(f"{key}: {value}")
        else:
            print(f"{key}: {type(value).__name__}")

Checkpoint type: <class 'dict'>

Top-level keys: ['step', 'tokens_seen', 'model_state_dict', 'optimizer_state_dict', 'architecture', 'vocab_size', 'context_length', 'hidden_size', 'num_layers', 'num_heads', 'ffn_size', 'prefix_ratio', 'dropout', 'learning_rate', 'weight_decay']
step: 330000
tokens_seen: 168960000

model_state_dict: dictionary with 100 entries
Sample keys: ['token_embedding.weight', 'position_embedding.weight', 'layers.0.ln1.weight', 'layers.0.ln1.bias', 'layers.0.attn.in_proj_weight', 'layers.0.attn.in_proj_bias', 'layers.0.attn.out_proj.weight', 'layers.0.attn.out_proj.bias', 'layers.0.ln2.weight', 'layers.0.ln2.bias', 'layers.0.ffn.0.weight', 'layers.0.ffn.0.bias']

optimizer_state_dict: dictionary with 2 entries
Sample keys: ['state', 'param_groups']
architecture: A3_prefix_lm_hybrid
vocab_size: 50000
context_length: 2048
hidden_size: 512
num_layers: 8
num_heads: 8
ffn_size: 2048
prefix_ratio: 0.5
dropout: 0.1
learning_rate: 0.0001
weight_decay: 0.1


In [106]:
import torch

a3_checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/33kkkk/step_0330000.pt"
)

a3_ckpt = torch.load(
    a3_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

assert a3_ckpt["architecture"] == "A3_prefix_lm_hybrid"
assert a3_ckpt["step"] == 330000
assert a3_ckpt["vocab_size"] == 50000
assert a3_ckpt["hidden_size"] == 512
assert a3_ckpt["num_layers"] == 8

print("Architecture:", a3_ckpt["architecture"])
print("Checkpoint step:", a3_ckpt["step"])
print("Model tensors:", len(a3_ckpt["model_state_dict"]))
print("Checkpoint verification passed.")

Architecture: A3_prefix_lm_hybrid
Checkpoint step: 330000
Model tensors: 100
Checkpoint verification passed.


In [107]:
print("A3PrefixLM defined:", "A3PrefixLM" in globals())
print("A3PrefixLMBlock defined:", "A3PrefixLMBlock" in globals())

if "A3PrefixLM" in globals():
    print(A3PrefixLM)
else:
    print("A3 model class is not currently defined.")

A3PrefixLM defined: False
A3PrefixLMBlock defined: False
A3 model class is not currently defined.


In [108]:
import json
from pathlib import Path

nb_path = Path(
    "/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb"
)

nb = json.loads(nb_path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    if "class A3PrefixLM" in source or "class PrefixLMBlock" in source:
        print(f"\n{'='*25} CELL {i} {'='*25}")
        print(source[:12000])


========================= CELL 17 =========================
# ============================================================
# A3 PREFIX-LM HYBRID — MODEL
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

DEVICE = "cuda"

VOCAB_SIZE = 50_000
CONTEXT_LEN = 2_048
HIDDEN_SIZE = 512
NUM_LAYERS = 8
NUM_HEADS = 8
FFN_SIZE = 2_048
PREFIX_RATIO = 0.5

class PrefixLMBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(HIDDEN_SIZE)
        self.attn = nn.MultiheadAttention(
            embed_dim=HIDDEN_SIZE,
            num_heads=NUM_HEADS,
            batch_first=True
        )

        self.ln2 = nn.LayerNorm(HIDDEN_SIZE)

        self.ffn = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, FFN_SIZE),
            nn.GELU(),
            nn.Linear(FFN_SIZE, HIDDEN_SIZE)
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)

        attn_o

In [109]:

import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = 50_000
CONTEXT_LEN = 2_048
HIDDEN_SIZE = 512
NUM_LAYERS = 8
NUM_HEADS = 8
FFN_SIZE = 2_048
PREFIX_RATIO = 0.5
DROPOUT = 0.1

class PrefixLMBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(HIDDEN_SIZE)
        self.attn = nn.MultiheadAttention(
            HIDDEN_SIZE, NUM_HEADS,
            dropout=DROPOUT, batch_first=True
        )
        self.attn_dropout = nn.Dropout(DROPOUT)
        self.ln2 = nn.LayerNorm(HIDDEN_SIZE)
        self.ffn = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, FFN_SIZE),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(FFN_SIZE, HIDDEN_SIZE),
            nn.Dropout(DROPOUT)
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)
        attn_out, _ = self.attn(
            h, h, h, attn_mask=attn_mask,
            need_weights=False
        )
        x = x + self.attn_dropout(attn_out)
        x = x + self.ffn(self.ln2(x))
        return x


class A3PrefixLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(
            VOCAB_SIZE, HIDDEN_SIZE
        )
        self.position_embedding = nn.Embedding(
            CONTEXT_LEN, HIDDEN_SIZE
        )
        self.embedding_dropout = nn.Dropout(DROPOUT)
        self.layers = nn.ModuleList([
            PrefixLMBlock() for _ in range(NUM_LAYERS)
        ])
        self.ln_f = nn.LayerNorm(HIDDEN_SIZE)
        self.register_buffer(
            "position_ids",
            torch.arange(CONTEXT_LEN).unsqueeze(0),
            persistent=False
        )

    def build_mask(self, seq_len, device):
        prefix_len = int(seq_len * PREFIX_RATIO)
        mask = torch.full(
            (seq_len, seq_len), float("-inf"),
            device=device
        )
        mask[:prefix_len, :prefix_len] = 0
        for i in range(prefix_len, seq_len):
            mask[i, :prefix_len] = 0
            mask[i, prefix_len:i+1] = 0
        return mask

    def forward(self, input_ids):
        _, seq_len = input_ids.shape
        positions = self.position_ids[:, :seq_len]
        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )
        x = self.embedding_dropout(x)
        mask = self.build_mask(seq_len, input_ids.device)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.ln_f(x)
        return F.linear(x, self.token_embedding.weight)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

a3_model = A3PrefixLM()
load_result = a3_model.load_state_dict(
    a3_ckpt["model_state_dict"], strict=True
)

a3_model = a3_model.to(device)

print("Checkpoint loaded successfully!")
print("Architecture:", a3_ckpt["architecture"])
print("Training step:", a3_ckpt["step"])
print("Device:", device)
print("Parameters:", f"{sum(p.numel() for p in a3_model.parameters()):,}")
print("Strict loading:", load_result)


Checkpoint loaded successfully!
Architecture: A3_prefix_lm_hybrid
Training step: 330000
Device: cuda
Parameters: 51,868,672
Strict loading: <All keys matched successfully>


In [110]:

import torch
import torch.nn as nn

class A3StatuteClassifier(nn.Module):
    def __init__(self, pretrained_model, num_labels=100):
        super().__init__()
        self.backbone = pretrained_model
        self.classifier = nn.Linear(512, num_labels)

    def forward(self, input_ids, attention_mask):
        # Reproduce A3's pretrained forward pass up to final normalization.
        x = (
            self.backbone.token_embedding(input_ids)
            + self.backbone.position_embedding(
                self.backbone.position_ids[:, :input_ids.shape[1]]
            )
        )
        x = self.backbone.embedding_dropout(x)

        mask = self.backbone.build_mask(
            input_ids.shape[1], input_ids.device
        )

        for layer in self.backbone.layers:
            x = layer(x, mask)

        x = self.backbone.ln_f(x)

        # Masked mean pooling: ignore padding tokens.
        valid = attention_mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1)

        return self.classifier(pooled)


a3_classifier = A3StatuteClassifier(a3_model).to(device)

print("A3 statute classifier ready.")
print("Number of labels:", a3_classifier.classifier.out_features)
print("Device:", next(a3_classifier.parameters()).device)
print("Total parameters:", f"{sum(p.numel() for p in a3_classifier.parameters()):,}")


A3 statute classifier ready.
Number of labels: 100
Device: cuda:0
Total parameters: 51,919,972


In [112]:

a3_classifier.eval()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device).float()

with torch.no_grad():
    logits = a3_classifier(input_ids, attention_mask)

print("Input shape:", tuple(input_ids.shape))
print("Logits shape:", tuple(logits.shape))
print("Labels shape:", tuple(labels.shape))
print("Logits finite:", torch.isfinite(logits).all().item())

assert logits.shape == labels.shape
assert logits.shape[-1] == 100
assert torch.isfinite(logits).all()

print("Batch verification passed.")


Input shape: (4, 512)
Logits shape: (4, 100)
Labels shape: (4, 100)
Logits finite: True
Batch verification passed.


In [113]:

import os
import time
import torch
import torch.nn as nn

exp4_dir = "/kaggle/working/phase5_exp4_a3"
os.makedirs(exp4_dir, exist_ok=True)

# Fine-tune all A3 parameters, using a smaller LR for pretrained weights.
for p in a3_classifier.backbone.parameters():
    p.requires_grad = True

optimizer_a3 = torch.optim.AdamW(
    [
        {
            "params": a3_classifier.backbone.parameters(),
            "lr": 2e-5
        },
        {
            "params": a3_classifier.classifier.parameters(),
            "lr": 2e-4
        }
    ],
    weight_decay=0.01
)

criterion_a3 = nn.BCEWithLogitsLoss()
use_amp = (device.type == "cuda")
scaler_a3 = torch.amp.GradScaler("cuda", enabled=use_amp)

a3_classifier.train()
total_loss = 0.0
steps = 0
start = time.time()

for batch in train_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device).float()

    optimizer_a3.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=use_amp
    ):
        logits = a3_classifier(input_ids, attention_mask)
        loss = criterion_a3(logits, labels)

    scaler_a3.scale(loss).backward()
    scaler_a3.unscale_(optimizer_a3)
    torch.nn.utils.clip_grad_norm_(
        a3_classifier.parameters(), max_norm=1.0
    )
    scaler_a3.step(optimizer_a3)
    scaler_a3.update()

    total_loss += loss.item()
    steps += 1

    if steps % 1000 == 0:
        print(
            f"Step {steps}/{len(train_loader)} | "
            f"Loss: {total_loss / steps:.5f} | "
            f"Elapsed: {(time.time()-start)/60:.1f} min"
        )

avg_loss = total_loss / max(steps, 1)

checkpoint_path = os.path.join(
    exp4_dir, "a3_full_finetune_epoch1.pt"
)

torch.save({
    "epoch": 1,
    "model_state_dict": a3_classifier.state_dict(),
    "optimizer_state_dict": optimizer_a3.state_dict(),
    "train_loss": avg_loss,
    "architecture": "A3_prefix_lm_hybrid",
    "pretrained_step": 330000
}, checkpoint_path)

print("\nA3 Experiment 4 — Epoch 1 complete")
print(f"Average training loss: {avg_loss:.5f}")
print(f"Time: {(time.time()-start)/60:.2f} minutes")
print("Checkpoint:", checkpoint_path)


Step 1000/10688 | Loss: 0.14134 | Elapsed: 1.4 min
Step 2000/10688 | Loss: 0.13423 | Elapsed: 2.8 min
Step 3000/10688 | Loss: 0.12883 | Elapsed: 4.2 min
Step 4000/10688 | Loss: 0.12586 | Elapsed: 5.6 min
Step 5000/10688 | Loss: 0.12354 | Elapsed: 7.0 min
Step 6000/10688 | Loss: 0.12149 | Elapsed: 8.4 min
Step 7000/10688 | Loss: 0.11965 | Elapsed: 9.8 min
Step 8000/10688 | Loss: 0.11847 | Elapsed: 11.2 min
Step 9000/10688 | Loss: 0.11758 | Elapsed: 12.5 min
Step 10000/10688 | Loss: 0.11655 | Elapsed: 13.9 min

A3 Experiment 4 — Epoch 1 complete
Average training loss: 0.11606
Time: 14.90 minutes
Checkpoint: /kaggle/working/phase5_exp4_a3/a3_full_finetune_epoch1.pt


In [114]:

import numpy as np
import torch
from sklearn.metrics import f1_score, accuracy_score

a3_classifier.eval()
all_probs = []
all_targets = []

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = a3_classifier(input_ids, attention_mask)

        probs_batch = torch.sigmoid(logits.float()).cpu().numpy()
        all_probs.append(probs_batch)
        all_targets.append(labels)

probs_a3 = np.concatenate(all_probs, axis=0)
targets_a3 = np.concatenate(all_targets, axis=0).astype(int)

print("Dev probabilities:", probs_a3.shape)
print("Dev targets:", targets_a3.shape)
print("\nThreshold | Macro-F1 | Micro-F1 | Subset Acc | Avg labels/doc")

for threshold in [0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.50]:
    preds = (probs_a3 >= threshold).astype(int)
    macro = f1_score(targets_a3, preds, average="macro", zero_division=0)
    micro = f1_score(targets_a3, preds, average="micro", zero_division=0)
    subset = accuracy_score(targets_a3, preds)
    avg_labels = preds.sum(axis=1).mean()

    print(
        f"{threshold:8.3f} | {macro:.5f} | {micro:.5f} | "
        f"{subset:.5f} | {avg_labels:.2f}"
    )

np.savez(
    "/kaggle/working/phase5_exp4_a3/a3_dev_predictions.npz",
    probs=probs_a3,
    targets=targets_a3
)
print("\nSaved A3 dev predictions.")


Dev probabilities: (10181, 100)
Dev targets: (10181, 100)

Threshold | Macro-F1 | Micro-F1 | Subset Acc | Avg labels/doc
   0.030 | 0.18817 | 0.24590 | 0.00000 | 22.22
   0.050 | 0.21522 | 0.30142 | 0.00000 | 15.40
   0.075 | 0.23050 | 0.34835 | 0.00029 | 11.28
   0.100 | 0.23491 | 0.37927 | 0.00147 | 8.93
   0.125 | 0.23397 | 0.39799 | 0.00246 | 7.39
   0.150 | 0.22766 | 0.40997 | 0.00629 | 6.23
   0.200 | 0.21212 | 0.41720 | 0.01306 | 4.65
   0.300 | 0.17676 | 0.39602 | 0.02750 | 2.84
   0.500 | 0.11201 | 0.27135 | 0.03634 | 1.05

Saved A3 dev predictions.


In [115]:

import json
from pathlib import Path

summary = {
    "experiment": "A3 full fine-tuning",
    "pretrained_checkpoint_step": 330000,
    "epochs_completed": 1,
    "training_loss": 0.11606,
    "best_macro_f1": 0.23491,
    "best_macro_threshold": 0.10,
    "best_micro_f1": 0.41720,
    "best_micro_threshold": 0.20,
    "dev_samples": int(len(targets_a3)),
    "num_labels": int(targets_a3.shape[1]),
}

out = Path("/kaggle/working/phase5_exp4_a3/a3_results.json")
out.write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))
print("Saved:", out)


{
  "experiment": "A3 full fine-tuning",
  "pretrained_checkpoint_step": 330000,
  "epochs_completed": 1,
  "training_loss": 0.11606,
  "best_macro_f1": 0.23491,
  "best_macro_threshold": 0.1,
  "best_micro_f1": 0.4172,
  "best_micro_threshold": 0.2,
  "dev_samples": 10181,
  "num_labels": 100
}
Saved: /kaggle/working/phase5_exp4_a3/a3_results.json


In [116]:

from pathlib import Path

root = Path("/kaggle/input/datasets")

for p in root.rglob("*"):
    if p.is_file() and any(
        word in p.name.lower()
        for word in ["capstone", "lorma", "reference", "paper"]
    ):
        print(p)


/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb


In [117]:

import json
from pathlib import Path

nb_path = Path(
    "/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb"
)

nb = json.loads(nb_path.read_text(encoding="utf-8"))

terms = ["lorma", "low-rank", "low rank", "knowledge distillation"]

matches = 0
for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    if any(term in source.lower() for term in terms):
        matches += 1
        print(f"\n{'='*20} CELL {i} {'='*20}")
        lines = source.splitlines()
        for j, line in enumerate(lines):
            if any(term in line.lower() for term in terms):
                start = max(0, j - 5)
                end = min(len(lines), j + 10)
                print("\n".join(lines[start:end]))
                print("---")

print("\nCells containing relevant terms:", matches)



==================== CELL 14 ====================
    print("Total notebook cells:", len(cells))

    keywords = [
        "class ", "def create_", "architecture", "state_dict",
        "D_MODEL", "N_LAYERS", "N_HEADS", "PREFIX_LEN",
        "GQA", "LORMA", "encoder_layers", "decoder_layers",
        "model.load_state_dict", "torch.save"
    ]

    for i, cell in enumerate(cells):
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)

        hits = [word for word in keywords if word.lower() in source.lower()]
---

Cells containing relevant terms: 1


In [118]:

import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb"
)

nb = json.loads(path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"]):
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    if cell.get("cell_type") == "markdown":
        if any(k in source.lower() for k in ["lorma", "low-rank", "low rank", "[12]"]):
            print(f"\n--- Markdown cell {i} ---\n{source[:6000]}")

    elif cell.get("cell_type") == "code":
        if "lorma" in source.lower():
            print(f"\n--- Code cell {i} ---\n{source[:8000]}")



--- Code cell 14 ---

import os
import json

NOTEBOOK_PATH = (
    "/kaggle/input/datasets/aq1de3/phase1-model/"
    "phase1_model_code_extract.ipynb"
)

print("Notebook exists:", os.path.exists(NOTEBOOK_PATH))

if not os.path.exists(NOTEBOOK_PATH):
    print("\nAvailable files in the Phase 1 model input:")
    for root, dirs, files in os.walk("/kaggle/input/datasets/aq1de3/phase1-model"):
        for filename in files:
            print(os.path.join(root, filename))
else:
    with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    cells = notebook.get("cells", [])
    print("Total notebook cells:", len(cells))

    keywords = [
        "class ", "def create_", "architecture", "state_dict",
        "D_MODEL", "N_LAYERS", "N_HEADS", "PREFIX_LEN",
        "GQA", "LORMA", "encoder_layers", "decoder_layers",
        "model.load_state_dict", "torch.save"
    ]

    for i, cell in enumerate(cells):
        source = cell.get("source", "")
        if isinsta

In [119]:

import os
import json

root = "/kaggle/input/datasets/aq1de3"

# Find files whose names may contain the Phase 1 model code.
for current_root, dirs, files in os.walk(root):
    for filename in files:
        if any(word in filename.lower() for word in [
            "phase1_model", "model_code_extract", "phase1-model"
        ]):
            print(os.path.join(current_root, filename))


/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb


In [120]:

import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/phase1-model/"
    "phase1_model_code_extract.ipynb"
)

nb = json.loads(path.read_text(encoding="utf-8"))
print("Notebook cells:", len(nb["cells"]))

for i, cell in enumerate(nb["cells"]):
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    terms = ["lorma", "lora", "low-rank", "low rank"]
    hits = [term for term in terms if term in source.lower()]

    if hits:
        print(f"\n{'='*60}")
        print(f"Cell {i} | Matches: {hits}")
        print(source[:5000])


Notebook cells: 13


In [121]:

import json
from pathlib import Path

path = Path(
    "/kaggle/input/datasets/aq1de3/phase1-model/"
    "phase1_model_code_extract.ipynb"
)
nb = json.loads(path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"]):
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    print(f"\n{'='*25} CELL {i} [{cell.get('cell_type')}] {'='*25}")
    print(source[:1800] if source.strip() else "[Empty cell]")




========================= CELL 0 [markdown] =========================
# Phase 1 model-code extracts (from the uploaded notebook)

These are copied source cells from the supplied notebook, not a verified runnable loader. Do not Run All. The notebook contains multiple architecture variants and benchmark/training cells.

**Compatibility warning:** checkpoint inspection showed A1 keys such as `blocks.0.norm1` (not standard Hugging Face GPT-2 keys); A2 checkpoint tensors are width 512, while the later A2 cell 122 configures width 528. Therefore these cells must be matched against the exact saved checkpoint before use. A3 and A4 also need strict state-dict verification.

This extract intentionally omits pretraining loops and corpus download/training cells.

========================= CELL 1 [markdown] =========================
## Original notebook cell 9: A1 architecture/configuration cell (GPT2-based candidate; checkpoint compatibility must be checked)

Copied verbatim from the supplied not

In [122]:

import torch
import torch.nn as nn

# Confirm the A3 classifier from Experiment 4 is still available.
assert "a3_model" in globals(), (
    "a3_model is missing. Reload the A3 checkpoint first."
)

print("A3 model:", type(a3_model).__name__)
print("\nLinear layers suitable for inspecting LoRMA targets:")

for name, module in a3_model.named_modules():
    if isinstance(module, nn.Linear):
        print(
            f"{name:55s} "
            f"weight_shape={tuple(module.weight.shape)} "
            f"bias={module.bias is not None}"
        )

print("\nDevice:", next(a3_model.parameters()).device)


A3 model: A3PrefixLM

Linear layers suitable for inspecting LoRMA targets:
layers.0.attn.out_proj                                  weight_shape=(512, 512) bias=True
layers.0.ffn.0                                          weight_shape=(2048, 512) bias=True
layers.0.ffn.3                                          weight_shape=(512, 2048) bias=True
layers.1.attn.out_proj                                  weight_shape=(512, 512) bias=True
layers.1.ffn.0                                          weight_shape=(2048, 512) bias=True
layers.1.ffn.3                                          weight_shape=(512, 2048) bias=True
layers.2.attn.out_proj                                  weight_shape=(512, 512) bias=True
layers.2.ffn.0                                          weight_shape=(2048, 512) bias=True
layers.2.ffn.3                                          weight_shape=(512, 2048) bias=True
layers.3.attn.out_proj                                  weight_shape=(512, 512) bias=True
layers.3.ffn.0     

In [123]:

import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

# Isolated copy of one compatible A3 feed-forward projection.
base_linear = copy.deepcopy(a3_model.layers[0].ffn[3])
base_linear.eval()

for p in base_linear.parameters():
    p.requires_grad_(False)

class LoRMAPlusLinear(nn.Module):
    """LoRMA+ multiplicative update for a linear layer."""
    def __init__(self, base, rank=8, alpha=8):
        super().__init__()
        self.base = base
        self.rank = rank
        self.scale = alpha / rank

        # Output-space transform: (out_features, out_features)
        d = base.out_features
        self.A = nn.Parameter(torch.empty(rank, d))
        self.B = nn.Parameter(torch.zeros(d, rank))
        nn.init.normal_(self.A, mean=0.0, std=0.02)

        for p in self.base.parameters():
            p.requires_grad_(False)

    def forward(self, x):
        # W_eff = (I + scale * B @ A) @ W0
        # Equivalent to W0x + scale * B @ (A @ (W0x)).
        y = self.base(x)
        delta = F.linear(F.linear(y, self.A), self.B)
        return y + self.scale * delta

test_layer = LoRMAPlusLinear(base_linear, rank=8, alpha=8).to(device)

# Test only: do not start training yet.
x = torch.randn(2, 16, base_linear.in_features, device=device)
with torch.no_grad():
    original_y = base_linear(x)
    adapted_y = test_layer(x)

print("Input shape:", tuple(x.shape))
print("Output shape:", tuple(adapted_y.shape))
print("Finite output:", bool(torch.isfinite(adapted_y).all()))
print(
    "Initial output matches frozen base:",
    torch.allclose(original_y, adapted_y, atol=1e-5, rtol=1e-5)
)

# Verify gradients reach the adaptation parameters.
test_layer.zero_grad(set_to_none=True)
test_layer(x).square().mean().backward()

print("A gradient finite:", test_layer.A.grad is not None and
      bool(torch.isfinite(test_layer.A.grad).all()))
print("B gradient finite:", test_layer.B.grad is not None and
      bool(torch.isfinite(test_layer.B.grad).all()))
print("Base weights frozen:", all(
    p.grad is None for p in test_layer.base.parameters()
))


Input shape: (2, 16, 2048)
Output shape: (2, 16, 512)
Finite output: True
Initial output matches frozen base: True
A gradient finite: True
B gradient finite: True
Base weights frozen: True


In [124]:

from pathlib import Path

for root in [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]:
    print(f"\nSearching {root}")
    for path in root.rglob("*lorma*"):
        print(path)



Searching /kaggle/input

Searching /kaggle/working


In [125]:

import subprocess
from pathlib import Path

repo_dir = Path("/kaggle/working/LoRMA")

if not repo_dir.exists():
    result = subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/Exploration-Lab/LoRMA.git",
            str(repo_dir),
        ],
        capture_output=True,
        text=True,
        timeout=120,
    )
    print("Clone return code:", result.returncode)
    print(result.stdout[-2000:])
    print(result.stderr[-2000:])
else:
    print("Repository already exists:", repo_dir)

if repo_dir.exists():
    print("\nRelevant implementation files:")
    for p in repo_dir.rglob("*.py"):
        name = p.name.lower()
        if "lorma" in name or "lora" in name:
            print(p)


Clone return code: 0

Cloning into '/kaggle/working/LoRMA'...


Relevant implementation files:
/kaggle/working/LoRMA/NLU/run_glue_lorma_pi.py
/kaggle/working/LoRMA/NLU/run_glue_lorma_plus.py
/kaggle/working/LoRMA/NLG/MetaMathQA/train_math_lorma.py
/kaggle/working/LoRMA/NLG/Language-generation/src/gpt2_ft_lorma_plus.py
/kaggle/working/LoRMA/NLG/Language-generation/src/model_lorma_plus.py
/kaggle/working/LoRMA/NLG/Language-generation/src/gpt2_ft_lorma_pi.py
/kaggle/working/LoRMA/NLG/Language-generation/src/model_lorma_pi.py
/kaggle/working/LoRMA/NLG/Language-generation/src/gpt2_beam_lorma_plus.py
/kaggle/working/LoRMA/NLG/Language-generation/src/gpt2_beam_lorma_pi.py


In [126]:

from pathlib import Path

repo = Path("/kaggle/working/LoRMA")

files_to_inspect = [
    repo / "NLG/Language-generation/src/model_lorma_plus.py",
    repo / "NLG/Language-generation/src/gpt2_ft_lorma_plus.py",
    repo / "NLU/run_glue_lorma_plus.py",
]

for path in files_to_inspect:
    print("\n" + "=" * 80)
    print("FILE:", path)
    print("=" * 80)

    if not path.exists():
        print("File not found")
        continue

    lines = path.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()

    keywords = (
        "lorma", "rank", "inflation", "multiplicative",
        "class ", "def forward", "def __init__"
    )

    matches = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    shown = set()
    for i in matches:
        for j in range(max(0, i - 3), min(len(lines), i + 7)):
            if j not in shown:
                print(f"{j + 1:4}: {lines[j]}")
                shown.add(j)
        print("----")



FILE: /kaggle/working/LoRMA/NLG/Language-generation/src/model_lorma_plus.py
  49:     return x * 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))
  50: 
  51: 
  52: class LayerNorm(nn.Module):
  53:     def __init__(self, hidden_size, eps=1e-12):
  54:         """Construct a layernorm module in the TF style (epsilon inside the square root)."""
  55:         super(LayerNorm, self).__init__()
  56:         self.weight = nn.Parameter(torch.ones(hidden_size))
  57:         self.bias = nn.Parameter(torch.zeros(hidden_size))
  58:         self.variance_epsilon = eps
----
  59: 
----
  60:     def forward(self, x):
  61:         u = x.mean(-1, keepdim=True)
  62:         s = (x - u).pow(2).mean(-1, keepdim=True)
  63:         x = (x - u) / torch.sqrt(s + self.variance_epsilon)
  64:         return self.weight * x + self.bias
  65: 
  66: 
----
  67: class Conv1D(nn.Module):
  68:     def __init__(self, nf, nx):
  69:         super(Conv1D, self).__init__()
  70:         self.nf = nf
  71:        

In [127]:

from pathlib import Path

path = Path(
    "/kaggle/working/LoRMA/NLU/run_glue_lorma_plus.py"
)
lines = path.read_text(encoding="utf-8").splitlines()

# Print the official implementation and its trainable-parameter helper.
for start, end in [(177, 240), (285, 310)]:
    print(f"\n--- Lines {start + 1}-{end} ---")
    for i in range(start, min(end, len(lines))):
        print(f"{i + 1:4}: {lines[i]}")



--- Lines 178-240 ---
 178: class LormaLinear_plus(nn.Module):
 179: 
 180:     def __init__(self, original_linear: nn.Linear, lorma_r: int, lorma_alpha: int, mode: str, do_rri: bool, lorma_dropout: float = 0.0):
 181:         super().__init__()
 182:         
 183:         assert mode in ["pre", "post"], "mode must be either pre, post"
 184:         self.mode = mode
 185:         self.original_layer = original_linear
 186:         self.in_features = original_linear.in_features
 187:         self.out_features = original_linear.out_features
 188:         self.lorma_r = lorma_r
 189:         self.scaling = lorma_alpha / lorma_r
 190:         
 191:         self.do_rri = do_rri 
 192:         if lorma_dropout > 0.:
 193:             self.lora_dropout = nn.Dropout(p=lorma_dropout)
 194:         else:
 195:             self.lora_dropout = lambda x: x
 196:         
 197:         # initialize lora_A and lora_B params
 198:         if self.mode == "pre":
 199:             self.lora_B = nn.Pa

In [128]:

import torch
import torch.nn as nn

target = a3_model.layers[0].ffn[3]

assert isinstance(target, nn.Linear)
assert target.in_features == 2048
assert target.out_features == 512

rank = 8
alpha = 8

# Official LoRMA+ pre-mode factor shapes
pre_B_shape = (target.out_features, rank)
pre_A_shape = (rank, target.out_features)

# Official LoRMA+ post-mode factor shapes
post_B_shape = (target.in_features, rank)
post_A_shape = (rank, target.in_features)

print("Target:", "a3_model.layers[0].ffn[3]")
print("Base weight:", tuple(target.weight.shape))
print("Pre factors:", pre_B_shape, pre_A_shape)
print("Post factors:", post_B_shape, post_A_shape)
print("Scaling:", alpha / rank)
print("Original model device:", target.weight.device)


Target: a3_model.layers[0].ffn[3]
Base weight: (512, 2048)
Pre factors: (512, 8) (8, 512)
Post factors: (2048, 8) (8, 2048)
Scaling: 1.0
Original model device: cuda:0


In [129]:

import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class OfficialLoRMAPlusPre(nn.Module):
    def __init__(self, original_linear, rank=8, alpha=8, do_rri=True):
        super().__init__()
        self.original_layer = copy.deepcopy(original_linear)
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.rank = rank
        self.scaling = alpha / rank
        self.do_rri = do_rri

        for p in self.original_layer.parameters():
            p.requires_grad_(False)

        self.lora_B = nn.Parameter(
            original_linear.weight.new_zeros(self.out_features, rank)
        )
        self.lora_A = nn.Parameter(
            original_linear.weight.new_empty(rank, self.out_features)
        )

        if do_rri:
            nn.init.zeros_(self.lora_B)
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        else:
            nn.init.kaiming_uniform_(self.lora_B, a=math.sqrt(5))
            with torch.no_grad():
                self.lora_A.copy_(torch.linalg.pinv(self.lora_B))
                self.lora_A.data /= math.sqrt(self.scaling)
                self.lora_B.data /= math.sqrt(self.scaling)

    def forward(self, x):
        W = self.original_layer.weight
        if self.do_rri:
            W_eff = (self.lora_B @ (self.lora_A @ W)) * self.scaling + W
        else:
            W_eff = (self.lora_B @ (self.lora_A @ W)) * self.scaling
        return F.linear(x, W_eff, self.original_layer.bias)

# Test on an isolated copy of the selected A3 projection.
test_lorma = OfficialLoRMAPlusPre(
    a3_model.layers[0].ffn[3], rank=8, alpha=8, do_rri=True
).to(device)

x = torch.randn(2, 16, 2048, device=device)
with torch.no_grad():
    y_base = test_lorma.original_layer(x)
    y_lorma = test_lorma(x)

print("Output shape:", tuple(y_lorma.shape))
print("Finite:", bool(torch.isfinite(y_lorma).all()))
print("Identity at initialization:",
      torch.allclose(y_base, y_lorma, atol=1e-5, rtol=1e-5))

test_lorma.zero_grad(set_to_none=True)
test_lorma(x).square().mean().backward()

print("B gradient finite:",
      test_lorma.lora_B.grad is not None and
      bool(torch.isfinite(test_lorma.lora_B.grad).all()))
print("A gradient finite:",
      test_lorma.lora_A.grad is not None and
      bool(torch.isfinite(test_lorma.lora_A.grad).all()))
print("Base weight frozen:",
      not test_lorma.original_layer.weight.requires_grad)


Output shape: (2, 16, 512)
Finite: True
Identity at initialization: True
B gradient finite: True
A gradient finite: True
Base weight frozen: True


In [130]:
import os
import torch

checkpoint_path = "/kaggle/input/datasets/aq1de3/33kkkk/step_0330000.pt"

print("Checkpoint exists:", os.path.exists(checkpoint_path))
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("A3 model class:", type(a3_model).__name__)

Checkpoint exists: True
Device: Tesla T4
A3 model class: A3PrefixLM


In [132]:
import inspect

print(inspect.signature(A3PrefixLM.__init__))
print(inspect.getsource(A3PrefixLM.__init__))

(self)
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(
            VOCAB_SIZE, HIDDEN_SIZE
        )
        self.position_embedding = nn.Embedding(
            CONTEXT_LEN, HIDDEN_SIZE
        )
        self.embedding_dropout = nn.Dropout(DROPOUT)
        self.layers = nn.ModuleList([
            PrefixLMBlock() for _ in range(NUM_LAYERS)
        ])
        self.ln_f = nn.LayerNorm(HIDDEN_SIZE)
        self.register_buffer(
            "position_ids",
            torch.arange(CONTEXT_LEN).unsqueeze(0),
            persistent=False
        )



In [133]:
a3_lorma_backbone = A3PrefixLM().to(device)

print("Fresh A3 backbone created.")
print("Parameters:", sum(p.numel() for p in a3_lorma_backbone.parameters()))

Fresh A3 backbone created.
Parameters: 51868672


In [134]:
print(type(checkpoint))

if isinstance(checkpoint, dict):
    print(list(checkpoint.keys())[:25])

<class 'dict'>
['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']


In [137]:
import torch

a3_path = "/kaggle/input/datasets/aq1de3/33kkkk/step_0330000.pt"

a3_checkpoint = torch.load(
    a3_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", list(a3_checkpoint.keys()))

state = a3_checkpoint.get("model_state_dict", a3_checkpoint)

print("\nFirst 20 parameter names:")
for name in list(state.keys())[:20]:
    print(name)

print("\nContains A3 token embedding:",
      "token_embedding.weight" in state)
print("Contains A4 encoder layers:",
      any(name.startswith("encoder_layers.") for name in state))

Checkpoint keys: ['step', 'tokens_seen', 'model_state_dict', 'optimizer_state_dict', 'architecture', 'vocab_size', 'context_length', 'hidden_size', 'num_layers', 'num_heads', 'ffn_size', 'prefix_ratio', 'dropout', 'learning_rate', 'weight_decay']

First 20 parameter names:
token_embedding.weight
position_embedding.weight
layers.0.ln1.weight
layers.0.ln1.bias
layers.0.attn.in_proj_weight
layers.0.attn.in_proj_bias
layers.0.attn.out_proj.weight
layers.0.attn.out_proj.bias
layers.0.ln2.weight
layers.0.ln2.bias
layers.0.ffn.0.weight
layers.0.ffn.0.bias
layers.0.ffn.3.weight
layers.0.ffn.3.bias
layers.1.ln1.weight
layers.1.ln1.bias
layers.1.attn.in_proj_weight
layers.1.attn.in_proj_bias
layers.1.attn.out_proj.weight
layers.1.attn.out_proj.bias

Contains A3 token embedding: True
Contains A4 encoder layers: False


In [138]:
a3_lorma_backbone = A3PrefixLM().to(device)

a3_lorma_backbone.load_state_dict(state, strict=True)

print("A3 pretrained checkpoint loaded successfully.")

A3 pretrained checkpoint loaded successfully.


In [139]:
a3_lorma_classifier = A3StatuteClassifier(
    a3_lorma_backbone
).to(device)

print("Classifier created.")
print("Backbone:", type(a3_lorma_classifier.backbone).__name__)
print("Target layer:", a3_lorma_classifier.backbone.layers[0].ffn[3])

Classifier created.
Backbone: A3PrefixLM
Target layer: Linear(in_features=2048, out_features=512, bias=True)


In [140]:
for name, module in a3_lorma_classifier.named_children():
    print(name, type(module).__name__)

print("\nClassifier parameter names:")
for name, param in a3_lorma_classifier.named_parameters():
    if "classifier" in name.lower() or "head" in name.lower():
        print(name, tuple(param.shape))

backbone A3PrefixLM
classifier Linear

Classifier parameter names:
classifier.weight (100, 512)
classifier.bias (100,)


In [141]:
import torch
import torch.nn as nn

# Freeze the entire classifier first.
for param in a3_lorma_classifier.parameters():
    param.requires_grad = False

# Replace the selected feed-forward projections.
# OfficialLoRMAPlusPre is the wrapper tested in your earlier cell.
target_layers = [0, 7]

for layer_idx in target_layers:
    original_linear = a3_lorma_classifier.backbone.layers[layer_idx].ffn[3]

    lorma_layer = OfficialLoRMAPlusPre(
        original_linear,
        rank=8,
        alpha=8,
        do_rri=True
    ).to(device)

    a3_lorma_classifier.backbone.layers[layer_idx].ffn[3] = lorma_layer

# Enable gradients for LORMA factors.
for name, param in a3_lorma_classifier.named_parameters():
    if "lora_A" in name or "lora_B" in name:
        param.requires_grad = True

# Enable gradients for the 100-label classification head.
for param in a3_lorma_classifier.classifier.parameters():
    param.requires_grad = True

trainable = sum(
    p.numel() for p in a3_lorma_classifier.parameters()
    if p.requires_grad
)
total = sum(p.numel() for p in a3_lorma_classifier.parameters())

print("Adapted layers:", target_layers)
print(f"Trainable parameters: {trainable:,} / {total:,}")
print(f"Trainable percentage: {100 * trainable / total:.3f}%")

print("\nTrainable parameter groups:")
for name, param in a3_lorma_classifier.named_parameters():
    if param.requires_grad:
        print(name, tuple(param.shape))

Adapted layers: [0, 7]
Trainable parameters: 67,684 / 51,936,356
Trainable percentage: 0.130%

Trainable parameter groups:
backbone.layers.0.ffn.3.lora_B (512, 8)
backbone.layers.0.ffn.3.lora_A (8, 512)
backbone.layers.7.ffn.3.lora_B (512, 8)
backbone.layers.7.ffn.3.lora_A (8, 512)
classifier.weight (100, 512)
classifier.bias (100,)


In [142]:
a3_lorma_classifier.train()
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device).float()

a3_lorma_classifier.zero_grad(set_to_none=True)

logits = a3_lorma_classifier(input_ids, attention_mask)

print("Logits shape:", tuple(logits.shape))
print("Labels shape:", tuple(labels.shape))
print("Logits finite:", bool(torch.isfinite(logits).all()))

loss = torch.nn.BCEWithLogitsLoss()(logits, labels)
loss.backward()

print("Smoke-test loss:", loss.item())

for name, param in a3_lorma_classifier.named_parameters():
    if param.requires_grad:
        grad_ok = (
            param.grad is not None
            and bool(torch.isfinite(param.grad).all())
        )
        print(name, "gradient finite:", grad_ok)

Logits shape: (4, 100)
Labels shape: (4, 100)
Logits finite: True
Smoke-test loss: 0.7490156292915344
backbone.layers.0.ffn.3.lora_B gradient finite: True
backbone.layers.0.ffn.3.lora_A gradient finite: True
backbone.layers.7.ffn.3.lora_B gradient finite: True
backbone.layers.7.ffn.3.lora_A gradient finite: True
classifier.weight gradient finite: True
classifier.bias gradient finite: True


In [143]:
import os
import torch

save_dir = "/kaggle/working/phase5_exp5_lorma"
os.makedirs(save_dir, exist_ok=True)

torch.save(
    {
        "model_state_dict": a3_lorma_classifier.state_dict(),
        "adapted_layers": [0, 7],
        "rank": 8,
        "alpha": 8,
        "do_rri": True,
        "base_checkpoint": "/kaggle/input/datasets/aq1de3/33kkkk/step_0330000.pt",
    },
    os.path.join(save_dir, "lorma_initialization.pt")
)

print("Saved:", os.path.join(save_dir, "lorma_initialization.pt"))

Saved: /kaggle/working/phase5_exp5_lorma/lorma_initialization.pt


In [144]:
import os
import torch

checkpoint_path = "/kaggle/working/phase5_exp5_lorma/lorma_initialization.pt"

saved = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint exists:", os.path.exists(checkpoint_path))
print("Adapted layers:", saved["adapted_layers"])
print("Rank:", saved["rank"])
print("Alpha:", saved["alpha"])
print("RRI:", saved["do_rri"])
print("Saved state entries:", len(saved["model_state_dict"]))

Checkpoint exists: True
Adapted layers: [0, 7]
Rank: 8
Alpha: 8
RRI: True
Saved state entries: 106


In [145]:
import torch

trainable_params = [
    p for p in a3_lorma_classifier.parameters()
    if p.requires_grad
]

assert trainable_params, "No trainable parameters found."

optimizer_lorma = torch.optim.AdamW(
    trainable_params,
    lr=1e-4,
    weight_decay=0.01
)

print("Trainable tensors:", len(trainable_params))
print("Optimizer parameters:",
      sum(p.numel() for p in trainable_params))

Trainable tensors: 6
Optimizer parameters: 67684


In [146]:
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print("Training batches:", len(train_loader))
print("Development batches:", len(dev_loader))

GPU allocated: 3.03 GB
GPU reserved:  3.47 GB
Training batches: 10688
Development batches: 2546


In [147]:
import os
import time
import torch
import torch.nn as nn

save_dir = "/kaggle/working/phase5_exp5_lorma"
os.makedirs(save_dir, exist_ok=True)

criterion_lorma = nn.BCEWithLogitsLoss()
num_epochs = 1

a3_lorma_classifier.train()
epoch_loss = 0.0
num_batches = 0
start_time = time.time()

for batch_idx, batch in enumerate(train_loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device).float()

    optimizer_lorma.zero_grad(set_to_none=True)

    logits = a3_lorma_classifier(input_ids, attention_mask)
    loss = criterion_lorma(logits, labels)

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Non-finite loss at batch {batch_idx}: {loss.item()}"
        )

    loss.backward()
    optimizer_lorma.step()

    epoch_loss += loss.item()
    num_batches += 1

    if (batch_idx + 1) % 500 == 0:
        elapsed = (time.time() - start_time) / 60
        print(
            f"Batch {batch_idx + 1}/{len(train_loader)} | "
            f"Mean loss: {epoch_loss / num_batches:.5f} | "
            f"Elapsed: {elapsed:.1f} min"
        )

mean_train_loss = epoch_loss / num_batches
elapsed_minutes = (time.time() - start_time) / 60

print("\nTraining complete")
print(f"Mean training loss: {mean_train_loss:.5f}")
print(f"Training time: {elapsed_minutes:.2f} minutes")

torch.save(
    {
        "model_state_dict": a3_lorma_classifier.state_dict(),
        "optimizer_state_dict": optimizer_lorma.state_dict(),
        "epoch": 1,
        "train_loss": mean_train_loss,
        "adapted_layers": [0, 7],
        "rank": 8,
        "alpha": 8,
        "do_rri": True,
    },
    os.path.join(save_dir, "lorma_trained.pt")
)

print("Saved trained checkpoint:", os.path.join(save_dir, "lorma_trained.pt"))

Batch 500/10688 | Mean loss: 0.18545 | Elapsed: 1.5 min
Batch 1000/10688 | Mean loss: 0.16175 | Elapsed: 3.1 min
Batch 1500/10688 | Mean loss: 0.15367 | Elapsed: 4.6 min
Batch 2000/10688 | Mean loss: 0.14981 | Elapsed: 6.1 min
Batch 2500/10688 | Mean loss: 0.14654 | Elapsed: 7.7 min
Batch 3000/10688 | Mean loss: 0.14426 | Elapsed: 9.2 min
Batch 3500/10688 | Mean loss: 0.14267 | Elapsed: 10.8 min
Batch 4000/10688 | Mean loss: 0.14106 | Elapsed: 12.3 min
Batch 4500/10688 | Mean loss: 0.13990 | Elapsed: 13.9 min
Batch 5000/10688 | Mean loss: 0.13882 | Elapsed: 15.4 min
Batch 5500/10688 | Mean loss: 0.13810 | Elapsed: 16.9 min
Batch 6000/10688 | Mean loss: 0.13741 | Elapsed: 18.5 min
Batch 6500/10688 | Mean loss: 0.13680 | Elapsed: 20.0 min
Batch 7000/10688 | Mean loss: 0.13630 | Elapsed: 21.5 min
Batch 7500/10688 | Mean loss: 0.13603 | Elapsed: 23.1 min
Batch 8000/10688 | Mean loss: 0.13557 | Elapsed: 24.6 min
Batch 8500/10688 | Mean loss: 0.13503 | Elapsed: 26.2 min
Batch 9000/10688 | Me

In [148]:
import os
import torch

lorma_checkpoint_path = (
    "/kaggle/working/phase5_exp5_lorma/lorma_trained.pt"
)

assert os.path.exists(lorma_checkpoint_path), "Trained checkpoint not found."

saved = torch.load(
    lorma_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

a3_lorma_classifier.load_state_dict(
    saved["model_state_dict"],
    strict=True
)

a3_lorma_classifier = a3_lorma_classifier.to(device)
print("LORMA+ trained checkpoint loaded successfully.")
print("Epoch:", saved.get("epoch"))
print("Training loss:", saved.get("train_loss"))

LORMA+ trained checkpoint loaded successfully.
Epoch: 1
Training loss: 0.1335378156805517


In [149]:
import numpy as np
import torch

a3_lorma_classifier.eval()

all_logits = []
all_targets = []

with torch.no_grad():
    for batch_idx, batch in enumerate(dev_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device).float()

        logits = a3_lorma_classifier(input_ids, attention_mask)

        all_logits.append(logits.float().cpu().numpy())
        all_targets.append(labels.cpu().numpy())

        if (batch_idx + 1) % 500 == 0:
            print(f"Development batch {batch_idx + 1}/{len(dev_loader)}")

probs_lorma = 1 / (
    1 + np.exp(-np.concatenate(all_logits, axis=0))
)
targets_lorma = np.concatenate(all_targets, axis=0).astype(np.int32)

print("Probabilities shape:", probs_lorma.shape)
print("Targets shape:", targets_lorma.shape)

np.savez_compressed(
    "/kaggle/working/phase5_exp5_lorma/lorma_dev_predictions.npz",
    probs=probs_lorma,
    targets=targets_lorma
)

print("Development predictions saved.")

Development batch 500/2546
Development batch 1000/2546
Development batch 1500/2546
Development batch 2000/2546
Development batch 2500/2546
Probabilities shape: (10181, 100)
Targets shape: (10181, 100)
Development predictions saved.


In [150]:
import json
from sklearn.metrics import f1_score, accuracy_score

thresholds = [0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.50]
lorma_results = []

for threshold in thresholds:
    predictions = (probs_lorma >= threshold).astype(np.int32)

    row = {
        "threshold": threshold,
        "macro_f1": f1_score(
            targets_lorma, predictions,
            average="macro", zero_division=0
        ),
        "micro_f1": f1_score(
            targets_lorma, predictions,
            average="micro", zero_division=0
        ),
        "subset_accuracy": accuracy_score(
            targets_lorma, predictions
        ),
        "avg_predicted_labels": float(predictions.sum(axis=1).mean())
    }
    lorma_results.append(row)

    print(
        f"t={threshold:.3f} | "
        f"Macro-F1={row['macro_f1']:.5f} | "
        f"Micro-F1={row['micro_f1']:.5f} | "
        f"Subset={row['subset_accuracy']:.5f} | "
        f"Avg labels={row['avg_predicted_labels']:.2f}"
    )

with open(
    "/kaggle/working/phase5_exp5_lorma/lorma_results.json",
    "w"
) as f:
    json.dump(lorma_results, f, indent=2)

best_macro = max(lorma_results, key=lambda row: row["macro_f1"])
print("\nBest Macro-F1 threshold:", best_macro["threshold"])
print("Best Macro-F1:", best_macro["macro_f1"])

t=0.030 | Macro-F1=0.10591 | Micro-F1=0.18025 | Subset=0.00000 | Avg labels=30.54
t=0.050 | Macro-F1=0.10952 | Micro-F1=0.22323 | Subset=0.00000 | Avg labels=20.24
t=0.075 | Macro-F1=0.09979 | Micro-F1=0.25976 | Subset=0.00000 | Avg labels=13.78
t=0.100 | Macro-F1=0.09345 | Micro-F1=0.28429 | Subset=0.00010 | Avg labels=9.94
t=0.125 | Macro-F1=0.08523 | Micro-F1=0.29970 | Subset=0.00020 | Avg labels=7.38
t=0.150 | Macro-F1=0.07825 | Micro-F1=0.30641 | Subset=0.00108 | Avg labels=5.64
t=0.200 | Macro-F1=0.06654 | Micro-F1=0.29388 | Subset=0.00746 | Avg labels=3.32
t=0.300 | Macro-F1=0.03714 | Micro-F1=0.20090 | Subset=0.01424 | Avg labels=1.17
t=0.500 | Macro-F1=0.01036 | Micro-F1=0.09345 | Subset=0.01552 | Avg labels=0.32

Best Macro-F1 threshold: 0.05
Best Macro-F1: 0.10951620013224335


In [151]:
import torch
import importlib.util

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())
print("GPU memory allocated:",
      f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB")

print("bitsandbytes installed:",
      importlib.util.find_spec("bitsandbytes") is not None)

print("PEFT installed:",
      importlib.util.find_spec("peft") is not None)

GPU: Tesla T4
CUDA available: True
GPU memory allocated: 3.03 GB
bitsandbytes installed: False
PEFT installed: True


In [152]:
import torch.nn as nn

linear_layers = []

for name, module in a3_lorma_backbone.named_modules():
    if isinstance(module, nn.Linear):
        linear_layers.append(
            (name, module.in_features, module.out_features)
        )

print("Number of linear layers:", len(linear_layers))

for item in linear_layers[:15]:
    print(item)

Number of linear layers: 24
('layers.0.attn.out_proj', 512, 512)
('layers.0.ffn.0', 512, 2048)
('layers.0.ffn.3.original_layer', 2048, 512)
('layers.1.attn.out_proj', 512, 512)
('layers.1.ffn.0', 512, 2048)
('layers.1.ffn.3', 2048, 512)
('layers.2.attn.out_proj', 512, 512)
('layers.2.ffn.0', 512, 2048)
('layers.2.ffn.3', 2048, 512)
('layers.3.attn.out_proj', 512, 512)
('layers.3.ffn.0', 512, 2048)
('layers.3.ffn.3', 2048, 512)
('layers.4.attn.out_proj', 512, 512)
('layers.4.ffn.0', 512, 2048)
('layers.4.ffn.3', 2048, 512)


In [153]:
%pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.6 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [154]:
import torch
import bitsandbytes as bnb

print("bitsandbytes version:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

test_layer = bnb.nn.Linear4bit(
    16, 8,
    bias=False,
    compute_dtype=torch.float16,
    quant_type="nf4"
).cuda()

print("4-bit layer created:", type(test_layer).__name__)

bitsandbytes version: 0.50.2
CUDA available: True
GPU: Tesla T4
4-bit layer created: Linear4bit


In [155]:
import torch

a3_checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/33kkkk/step_0330000.pt"
)

checkpoint_a3 = torch.load(
    a3_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

state_a3 = checkpoint_a3.get(
    "model_state_dict", checkpoint_a3
)

print("Checkpoint step:", checkpoint_a3.get("step"))
print("State dictionary entries:", len(state_a3))
print("Contains token embedding:",
      "token_embedding.weight" in state_a3)

Checkpoint step: 330000
State dictionary entries: 100
Contains token embedding: True


In [156]:
import inspect

print("=== A3PrefixLM.forward ===")
print(inspect.getsource(A3PrefixLM.forward))

print("\n=== PrefixLMBlock.forward ===")
print(inspect.getsource(PrefixLMBlock.forward))

=== A3PrefixLM.forward ===
    def forward(self, input_ids):
        _, seq_len = input_ids.shape
        positions = self.position_ids[:, :seq_len]
        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )
        x = self.embedding_dropout(x)
        mask = self.build_mask(seq_len, input_ids.device)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.ln_f(x)
        return F.linear(x, self.token_embedding.weight)


=== PrefixLMBlock.forward ===
    def forward(self, x, attn_mask):
        h = self.ln1(x)
        attn_out, _ = self.attn(
            h, h, h, attn_mask=attn_mask,
            need_weights=False
        )
        x = x + self.attn_dropout(attn_out)
        x = x + self.ffn(self.ln2(x))
        return x



In [157]:
import torch
import torch.nn as nn

a3_qlora_backbone = A3PrefixLM()
a3_qlora_backbone.load_state_dict(state_a3, strict=True)
a3_qlora_backbone = a3_qlora_backbone.to(device)

print("Original A3 checkpoint loaded.")

for name, module in a3_qlora_backbone.named_modules():
    if isinstance(module, nn.Linear):
        print(
            f"{name:35s}",
            f"{module.in_features:5d} -> {module.out_features:5d}"
        )

Original A3 checkpoint loaded.
layers.0.attn.out_proj                512 ->   512
layers.0.ffn.0                        512 ->  2048
layers.0.ffn.3                       2048 ->   512
layers.1.attn.out_proj                512 ->   512
layers.1.ffn.0                        512 ->  2048
layers.1.ffn.3                       2048 ->   512
layers.2.attn.out_proj                512 ->   512
layers.2.ffn.0                        512 ->  2048
layers.2.ffn.3                       2048 ->   512
layers.3.attn.out_proj                512 ->   512
layers.3.ffn.0                        512 ->  2048
layers.3.ffn.3                       2048 ->   512
layers.4.attn.out_proj                512 ->   512
layers.4.ffn.0                        512 ->  2048
layers.4.ffn.3                       2048 ->   512
layers.5.attn.out_proj                512 ->   512
layers.5.ffn.0                        512 ->  2048
layers.5.ffn.3                       2048 ->   512
layers.6.attn.out_proj                512 ->   512


In [158]:
import copy
import torch
import torch.nn as nn
import bitsandbytes as bnb

# Keep the original A3 model intact.
a3_qlora_test = copy.deepcopy(a3_qlora_backbone).cpu()

# Select the first FFN expansion projection.
original_linear = a3_qlora_test.layers[0].ffn[0]

print("Original layer:", type(original_linear).__name__)
print("Weight shape:", tuple(original_linear.weight.shape))
print("Weight dtype:", original_linear.weight.dtype)

Original layer: Linear
Weight shape: (2048, 512)
Weight dtype: torch.float32


In [159]:
import torch
import bitsandbytes as bnb

original_linear = a3_qlora_test.layers[0].ffn[0]

# Copy the pretrained weights to a 4-bit layer.
quant_linear = bnb.nn.Linear4bit(
    original_linear.in_features,
    original_linear.out_features,
    bias=(original_linear.bias is not None),
    compute_dtype=torch.float16,
    compress_statistics=True,
    quant_type="nf4"
)

quant_linear.weight = bnb.nn.Params4bit(
    original_linear.weight.detach().clone(),
    requires_grad=False,
    compress_statistics=True,
    quant_type="nf4",
    quant_storage=torch.uint8
)

if original_linear.bias is not None:
    quant_linear.bias = torch.nn.Parameter(
        original_linear.bias.detach().clone(),
        requires_grad=False
    )

quant_linear = quant_linear.to("cuda")

print("Quantized layer:", type(quant_linear).__name__)
print("Weight parameter type:", type(quant_linear.weight).__name__)
print("Weight dtype:", quant_linear.weight.dtype)
print("Quantization state available:", quant_linear.weight.quant_state is not None)

Quantized layer: Linear4bit
Weight parameter type: Params4bit
Weight dtype: torch.uint8
Quantization state available: True


In [160]:
quant_linear.eval()

x = torch.randn(2, 16, 512, device="cuda", dtype=torch.float16)

with torch.no_grad():
    y = quant_linear(x)

print("Output shape:", tuple(y.shape))
print("Output dtype:", y.dtype)
print("Finite output:", bool(torch.isfinite(y).all()))

Output shape: (2, 16, 2048)
Output dtype: torch.float16
Finite output: True


In [161]:
original_linear = a3_qlora_backbone.layers[0].ffn[0].to("cuda").eval()
quant_linear.eval()

x32 = torch.randn(2, 16, 512, device="cuda", dtype=torch.float32)

with torch.no_grad():
    y_original = original_linear(x32).float()
    y_quantized = quant_linear(x32.half()).float()

difference = y_original - y_quantized

print("Original output shape:", tuple(y_original.shape))
print("Quantized output shape:", tuple(y_quantized.shape))
print("Mean absolute error:", difference.abs().mean().item())
print("Maximum absolute error:", difference.abs().max().item())

Original output shape: (2, 16, 2048)
Quantized output shape: (2, 16, 2048)
Mean absolute error: 0.050353698432445526
Maximum absolute error: 0.3936476707458496


In [162]:
import torch
import torch.nn as nn

print("GPU:", torch.cuda.get_device_name(0))
print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")

# Verify the fresh A3 backbone is still FP32 and unmodified.
print("FFN weight dtype:",
      a3_qlora_backbone.layers[0].ffn[0].weight.dtype)

print("FFN projection shape:",
      tuple(a3_qlora_backbone.layers[0].ffn[0].weight.shape))

GPU: Tesla T4
Allocated: 3.22 GB
Reserved:  3.66 GB
FFN weight dtype: torch.float32
FFN projection shape: (2048, 512)


In [164]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import bitsandbytes as bnb
import math

class QLoRALinear(nn.Module):
    def __init__(self, original_linear, rank=8, alpha=16):
        super().__init__()

        in_features = original_linear.in_features
        out_features = original_linear.out_features

        # Frozen 4-bit base weights
        self.base = bnb.nn.Linear4bit(
            in_features,
            out_features,
            bias=original_linear.bias is not None,
            compute_dtype=torch.float16,
            compress_statistics=True,
            quant_type="nf4",
        )

        self.base.weight = bnb.nn.Params4bit(
            original_linear.weight.detach().clone(),
            requires_grad=False,
            compress_statistics=True,
            quant_type="nf4",
            quant_storage=torch.uint8,
        )

        if original_linear.bias is not None:
            self.base.bias = nn.Parameter(
                original_linear.bias.detach().clone(),
                requires_grad=False,
            )

        self.base = self.base.to("cuda")
        self.base.eval()

        # Trainable LoRA adapter
        self.A = nn.Parameter(
            torch.empty(rank, in_features, device="cuda")
        )
        self.B = nn.Parameter(
            torch.zeros(out_features, rank, device="cuda")
        )

        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scaling = alpha / rank

    def forward(self, x):
        base_output = self.base(x.to(torch.float16)).to(x.dtype)

        adapter_output = F.linear(
            F.linear(x, self.A),
            self.B,
        ) * self.scaling

        return base_output + adapter_output

In [165]:
original_linear = a3_qlora_backbone.layers[0].ffn[0]

qlora_test = QLoRALinear(
    original_linear,
    rank=8,
    alpha=16,
)

x = torch.randn(2, 16, 512, device="cuda")

y = qlora_test(x)
loss = y.float().square().mean()
loss.backward()

print("Output shape:", tuple(y.shape))
print("Output finite:", torch.isfinite(y).all().item())
print("A gradient:", qlora_test.A.grad is not None)
print("B gradient:", qlora_test.B.grad is not None)
print("Base weight trainable:", qlora_test.base.weight.requires_grad)
print("GPU allocated:", f"{torch.cuda.memory_allocated()/1024**3:.2f} GB")

Output shape: (2, 16, 2048)
Output finite: True
A gradient: True
B gradient: True
Base weight trainable: False
GPU allocated: 3.22 GB


In [166]:
print("Backbone:", type(a3_qlora_backbone).__name__)
print("Classifier:", type(a3_lorma_classifier).__name__)

print(
    "A3 layer count:",
    len(a3_qlora_backbone.layers)
)

print(
    "First FFN:",
    a3_qlora_backbone.layers[0].ffn
)

Backbone: A3PrefixLM
Classifier: A3StatuteClassifier
A3 layer count: 8
First FFN: Sequential(
  (0): Linear(in_features=512, out_features=2048, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=2048, out_features=512, bias=True)
  (4): Dropout(p=0.1, inplace=False)
)


In [167]:
import copy
import torch

# Work on a separate copy so the original checkpoint stays intact.
a3_qlora_integrated = copy.deepcopy(a3_qlora_backbone).cpu()

# Replace only layer 0's first FFN projection.
original = a3_qlora_integrated.layers[0].ffn[0]
a3_qlora_integrated.layers[0].ffn[0] = QLoRALinear(
    original,
    rank=8,
    alpha=16,
)

a3_qlora_integrated = a3_qlora_integrated.to("cuda")
a3_qlora_integrated.eval()

print("Replacement:", type(a3_qlora_integrated.layers[0].ffn[0]).__name__)
print("Layers:", len(a3_qlora_integrated.layers))
print("Quantized base:",
      type(a3_qlora_integrated.layers[0].ffn[0].base).__name__)

Replacement: QLoRALinear
Layers: 8
Quantized base: Linear4bit


In [168]:
a3_qlora_integrated.eval()

input_ids = torch.randint(
    low=0,
    high=50000,
    size=(2, 16),
    device="cuda",
)

with torch.no_grad():
    logits = a3_qlora_integrated(input_ids)

print("Output shape:", tuple(logits.shape))
print("Output finite:", torch.isfinite(logits).all().item())
print("Output dtype:", logits.dtype)

Output shape: (2, 16, 50000)
Output finite: True
Output dtype: torch.float32


In [169]:
import inspect

print("Classifier forward:")
print(inspect.getsource(A3StatuteClassifier.forward))

print("\nClassifier initialization:")
print(inspect.getsource(A3StatuteClassifier.__init__))

Classifier forward:
    def forward(self, input_ids, attention_mask):
        # Reproduce A3's pretrained forward pass up to final normalization.
        x = (
            self.backbone.token_embedding(input_ids)
            + self.backbone.position_embedding(
                self.backbone.position_ids[:, :input_ids.shape[1]]
            )
        )
        x = self.backbone.embedding_dropout(x)

        mask = self.backbone.build_mask(
            input_ids.shape[1], input_ids.device
        )

        for layer in self.backbone.layers:
            x = layer(x, mask)

        x = self.backbone.ln_f(x)

        # Masked mean pooling: ignore padding tokens.
        valid = attention_mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1)

        return self.classifier(pooled)


Classifier initialization:
    def __init__(self, pretrained_model, num_labels=100):
        super().__init__()
        self.backbone = pretrained_model
        sel

In [170]:
import torch
import torch.nn as nn
import copy

# Start from the integrated QLoRA backbone.
qlora_classifier = A3StatuteClassifier(
    pretrained_model=a3_qlora_integrated,
    num_labels=100,
).to("cuda")

# Freeze all model parameters first.
for p in qlora_classifier.parameters():
    p.requires_grad = False

# Unfreeze the adapter parameters in the quantized projection.
qlora_layer = qlora_classifier.backbone.layers[0].ffn[0]
qlora_layer.A.requires_grad = True
qlora_layer.B.requires_grad = True

# Train the classification head.
for p in qlora_classifier.classifier.parameters():
    p.requires_grad = True

trainable = sum(
    p.numel() for p in qlora_classifier.parameters()
    if p.requires_grad
)
total = sum(p.numel() for p in qlora_classifier.parameters())

print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters: {total:,}")
print(f"Trainable percentage: {100 * trainable / total:.4f}%")

Trainable parameters: 71,780
Total parameters: 51,416,164
Trainable percentage: 0.1396%


In [171]:
batch = next(iter(dev_loader))

input_ids = batch["input_ids"].to("cuda")
attention_mask = batch["attention_mask"].to("cuda")

qlora_classifier.eval()

with torch.no_grad():
    logits = qlora_classifier(input_ids, attention_mask)

print("Logits shape:", tuple(logits.shape))
print("All finite:", torch.isfinite(logits).all().item())
print("Expected shape:", (input_ids.shape[0], 100))

Logits shape: (4, 100)
All finite: True
Expected shape: (4, 100)


In [172]:
import torch
import torch.nn.functional as F

batch = next(iter(train_loader))
input_ids = batch["input_ids"].to("cuda")
attention_mask = batch["attention_mask"].to("cuda")
labels = batch["labels"].to("cuda").float()

qlora_classifier.train()
qlora_classifier.zero_grad(set_to_none=True)

logits = qlora_classifier(input_ids, attention_mask)
loss = F.binary_cross_entropy_with_logits(logits, labels)
loss.backward()

layer = qlora_classifier.backbone.layers[0].ffn[0]

print("Loss:", loss.item())
print("A gradient exists:", layer.A.grad is not None)
print("B gradient exists:", layer.B.grad is not None)
print("Classifier gradient exists:",
      qlora_classifier.classifier.weight.grad is not None)
print("Quantized base frozen:",
      not layer.base.weight.requires_grad)
print("Gradients finite:",
      torch.isfinite(layer.A.grad).all().item()
      and torch.isfinite(layer.B.grad).all().item()
      and torch.isfinite(
          qlora_classifier.classifier.weight.grad
      ).all().item())

Loss: 0.7080209851264954
A gradient exists: True
B gradient exists: True
Classifier gradient exists: True
Quantized base frozen: True
Gradients finite: True


In [173]:
import time
import torch
import torch.nn.functional as F

qlora_classifier.train()

optimizer = torch.optim.AdamW(
    [p for p in qlora_classifier.parameters() if p.requires_grad],
    lr=2e-4,
    weight_decay=0.01,
)

pilot_losses = []
start_time = time.time()

for step, batch in enumerate(train_loader):
    if step >= 100:
        break

    input_ids = batch["input_ids"].to("cuda")
    attention_mask = batch["attention_mask"].to("cuda")
    labels = batch["labels"].to("cuda").float()

    optimizer.zero_grad(set_to_none=True)

    logits = qlora_classifier(input_ids, attention_mask)
    loss = F.binary_cross_entropy_with_logits(logits, labels)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}")

    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in qlora_classifier.parameters() if p.requires_grad],
        max_norm=1.0,
    )
    optimizer.step()

    pilot_losses.append(loss.item())

    if (step + 1) % 20 == 0:
        print(
            f"Step {step + 1}/100 | "
            f"Loss: {sum(pilot_losses[-20:])/len(pilot_losses[-20:]):.4f}"
        )

print(f"Elapsed: {(time.time()-start_time)/60:.2f} minutes")
print(f"First 20-step mean: {sum(pilot_losses[:20])/len(pilot_losses[:20]):.4f}")
print(f"Last 20-step mean: {sum(pilot_losses[-20:])/len(pilot_losses[-20:]):.4f}")

Step 20/100 | Loss: 0.5458
Step 40/100 | Loss: 0.2965
Step 60/100 | Loss: 0.1857
Step 80/100 | Loss: 0.1499
Step 100/100 | Loss: 0.1338
Elapsed: 0.29 minutes
First 20-step mean: 0.5458
Last 20-step mean: 0.1338


In [174]:
import os
import torch
import json
import time

save_dir = "/kaggle/working/phase5_exp6_qlora"
os.makedirs(save_dir, exist_ok=True)

checkpoint_path = os.path.join(save_dir, "qlora_pilot_100_steps.pt")

torch.save({
    "model_state_dict": qlora_classifier.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "pilot_steps": len(pilot_losses),
    "pilot_losses": pilot_losses,
    "experiment": "A3 one-projection QLoRA-style pilot",
}, checkpoint_path)

print("Saved:", checkpoint_path)
print("File size: {:.2f} MB".format(
    os.path.getsize(checkpoint_path) / (1024**2)
))

Saved: /kaggle/working/phase5_exp6_qlora/qlora_pilot_100_steps.pt
File size: 195.24 MB


In [175]:
import numpy as np
from sklearn.metrics import f1_score
import torch

qlora_classifier.eval()

all_probs = []
all_labels = []

start_time = time.time()

with torch.no_grad():
    for i, batch in enumerate(dev_loader):
        input_ids = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")
        labels = batch["labels"].cpu().numpy()

        logits = qlora_classifier(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_labels.append(labels)

        if (i + 1) % 500 == 0:
            print(f"Evaluated {i + 1}/{len(dev_loader)} batches")

all_probs = np.concatenate(all_probs, axis=0)
all_labels = np.concatenate(all_labels, axis=0).astype(np.int32)

print("Predictions:", all_probs.shape)
print("Labels:", all_labels.shape)
print("Elapsed: {:.2f} minutes".format(
    (time.time() - start_time) / 60
))

np.savez_compressed(
    os.path.join(save_dir, "qlora_pilot_dev_predictions.npz"),
    probabilities=all_probs,
    labels=all_labels,
)

thresholds = [0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.50]
results = []

for threshold in thresholds:
    preds = (all_probs >= threshold).astype(np.int32)

    macro = f1_score(
        all_labels, preds, average="macro", zero_division=0
    )
    micro = f1_score(
        all_labels, preds, average="micro", zero_division=0
    )
    subset = np.all(preds == all_labels, axis=1).mean()
    avg_labels = preds.sum(axis=1).mean()

    results.append({
        "threshold": threshold,
        "macro_f1": float(macro),
        "micro_f1": float(micro),
        "subset_accuracy": float(subset),
        "avg_predicted_labels": float(avg_labels),
    })

print("\nThreshold evaluation")
print("Threshold | Macro-F1 | Micro-F1 | Subset Acc | Avg labels")
for r in results:
    print(
        f'{r["threshold"]:9.3f} | '
        f'{r["macro_f1"]:.5f} | '
        f'{r["micro_f1"]:.5f} | '
        f'{r["subset_accuracy"]:.5f} | '
        f'{r["avg_predicted_labels"]:.2f}'
    )

with open(os.path.join(save_dir, "qlora_pilot_dev_results.json"), "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved predictions and metrics to:", save_dir)

Evaluated 500/2546 batches
Evaluated 1000/2546 batches
Evaluated 1500/2546 batches
Evaluated 2000/2546 batches
Evaluated 2500/2546 batches
Predictions: (10181, 100)
Labels: (10181, 100)
Elapsed: 4.39 minutes

Threshold evaluation
Threshold | Macro-F1 | Micro-F1 | Subset Acc | Avg labels
    0.030 | 0.06605 | 0.13917 | 0.00000 | 35.69
    0.050 | 0.04902 | 0.19602 | 0.00000 | 15.46
    0.075 | 0.03121 | 0.23275 | 0.00000 | 8.02
    0.100 | 0.02347 | 0.23624 | 0.00000 | 6.42
    0.125 | 0.02053 | 0.23582 | 0.00000 | 4.99
    0.150 | 0.01691 | 0.23099 | 0.00069 | 3.84
    0.200 | 0.01153 | 0.18944 | 0.00678 | 2.07
    0.300 | 0.00242 | 0.02096 | 0.00432 | 0.10
    0.500 | 0.00000 | 0.00000 | 0.00000 | 0.00

Saved predictions and metrics to: /kaggle/working/phase5_exp6_qlora


In [178]:
import os

print("Available dataset files:\n")

for root, dirs, files in os.walk("/kaggle/input"):
    for name in files:
        if name.lower().endswith(".json") and (
            "train" in name.lower() or "dev" in name.lower()
        ):
            print(os.path.join(root, name))

Available dataset files:

/kaggle/input/datasets/aq1de3/trainjson/train.json
/kaggle/input/datasets/aq1de3/phase3/CL_dev.json
/kaggle/input/datasets/aq1de3/phase3/CL_train.json
/kaggle/input/datasets/aq1de3/phase3/IT_dev.json
/kaggle/input/datasets/aq1de3/phase3/IT_train.json
/kaggle/input/datasets/aq1de3/devjson/dev.json
/kaggle/input/datasets/aq1de3/devdev/dev (2).json
/kaggle/input/datasets/aq1de3/traintr/train (2).json


In [179]:
import json
import numpy as np

train_path = "/kaggle/input/datasets/aq1de3/traintr/train (2).json"
dev_path = "/kaggle/input/datasets/aq1de3/devdev/dev (2).json"

def load_records(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        for key in ["data", "records", "examples"]:
            if key in data and isinstance(data[key], list):
                data = data[key]
                break

    return data

train_records = load_records(train_path)
dev_records = load_records(dev_path)

train_labels = np.asarray(
    [r["labels"] for r in train_records], dtype=np.float32
)
dev_labels = np.asarray(
    [r["labels"] for r in dev_records], dtype=np.float32
)

print("Train shape:", train_labels.shape)
print("Dev shape:", dev_labels.shape)
print("Train positives/document:", train_labels.sum(axis=1).mean())
print("Dev positives/document:", dev_labels.sum(axis=1).mean())
print("Train labels never positive:", (train_labels.sum(axis=0) == 0).sum())
print("Dev labels never positive:", (dev_labels.sum(axis=0) == 0).sum())

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (42750,) + inhomogeneous part.

In [180]:
print("Train record type:", type(train_records))
print("Number of train records:", len(train_records))

for i in range(3):
    r = train_records[i]
    print(f"\nRecord {i}:")
    print("Keys:", r.keys())
    print("ID:", r.get("id"))
    print("Labels type:", type(r.get("labels")))
    print("Labels preview:", repr(r.get("labels"))[:500])

Train record type: <class 'list'>
Number of train records: 42750

Record 0:
Keys: dict_keys(['id', 'text', 'labels'])
ID: 1000008
Labels type: <class 'list'>
Labels preview: [69, 9, 3]

Record 1:
Keys: dict_keys(['id', 'text', 'labels'])
ID: 100009671
Labels type: <class 'list'>
Labels preview: [81, 5, 94]

Record 2:
Keys: dict_keys(['id', 'text', 'labels'])
ID: 1000196
Labels type: <class 'list'>
Labels preview: [10, 17, 88, 7]


In [181]:
from collections import Counter

label_lengths = Counter()
label_types = Counter()

for r in train_records:
    labels = r.get("labels")
    label_types[type(labels).__name__] += 1

    if isinstance(labels, (list, tuple)):
        label_lengths[len(labels)] += 1

print("Label value types:", label_types)
print("Label sequence lengths:", label_lengths.most_common(15))

Label value types: Counter({'list': 42750})
Label sequence lengths: [(1, 9146), (2, 7585), (3, 7045), (4, 5782), (5, 4265), (6, 3283), (7, 2131), (8, 1390), (9, 836), (10, 458), (11, 358), (12, 167), (13, 106), (14, 53), (15, 46)]


In [182]:
import numpy as np

NUM_LABELS = 100

def to_multihot(records, num_labels=NUM_LABELS):
    result = np.zeros((len(records), num_labels), dtype=np.int32)

    for i, record in enumerate(records):
        ids = record["labels"]

        for label_id in ids:
            if not isinstance(label_id, (int, np.integer)):
                raise TypeError(f"Unexpected label ID: {label_id!r}")
            if not 0 <= label_id < num_labels:
                raise ValueError(f"Label ID out of range: {label_id}")
            result[i, label_id] = 1

    return result

train_labels = to_multihot(train_records)
dev_labels = to_multihot(dev_records)

print("Train shape:", train_labels.shape)
print("Dev shape:", dev_labels.shape)
print("Train positives/document:", train_labels.sum(axis=1).mean())
print("Dev positives/document:", dev_labels.sum(axis=1).mean())
print("Train labels never positive:", (train_labels.sum(axis=0) == 0).sum())
print("Dev labels never positive:", (dev_labels.sum(axis=0) == 0).sum())

Train shape: (42750, 100)
Dev shape: (10181, 100)
Train positives/document: 3.704982456140351
Dev positives/document: 3.9157253707887243
Train labels never positive: 0
Dev labels never positive: 0


In [183]:
import numpy as np

path = "/kaggle/working/phase5_exp6_qlora/qlora_pilot_dev_predictions.npz"
data = np.load(path)

probs = data["probabilities"]
saved_labels = data["labels"]

print("Probabilities:", probs.shape)
print("Saved labels:", saved_labels.shape)
print("Actual positive rate:", saved_labels.mean())
print("Dataset positive rate:", dev_labels.mean())

print("\nProbability percentiles:")
for q in [0, 25, 50, 75, 90, 95, 99, 100]:
    print(f"{q:3d}%: {np.percentile(probs, q):.5f}")

Probabilities: (10181, 100)
Saved labels: (10181, 100)
Actual positive rate: 0.03915725370788724
Dataset positive rate: 0.03915725370788724

Probability percentiles:
  0%: 0.00128
 25%: 0.01574
 50%: 0.02376
 75%: 0.03731
 90%: 0.06424
 95%: 0.12477
 99%: 0.23792
100%: 0.44939


In [185]:
import numpy as np

qlora_path = (
    "/kaggle/working/phase5_exp6_qlora/"
    "qlora_pilot_dev_predictions.npz"
)
a3_path = (
    "/kaggle/working/phase5_exp4_a3/"
    "a3_dev_predictions.npz"
)

for name, path in [("QLoRA", qlora_path), ("A3", a3_path)]:
    data = np.load(path)
    print(f"\n{name} file keys:", data.files)

    for key in data.files:
        arr = data[key]
        print(
            f"  {key}: shape={arr.shape}, dtype={arr.dtype}"
        )


QLoRA file keys: ['probabilities', 'labels']
  probabilities: shape=(10181, 100), dtype=float32
  labels: shape=(10181, 100), dtype=int32

A3 file keys: ['probs', 'targets']
  probs: shape=(10181, 100), dtype=float32
  targets: shape=(10181, 100), dtype=int64


In [186]:
import numpy as np
from sklearn.metrics import f1_score

qlora_data = np.load(
    "/kaggle/working/phase5_exp6_qlora/qlora_pilot_dev_predictions.npz"
)
a3_data = np.load(
    "/kaggle/working/phase5_exp4_a3/a3_dev_predictions.npz"
)

y_true = qlora_data["labels"].astype(np.int32)
qlora_probs = qlora_data["probabilities"]
a3_probs = a3_data["probs"]
a3_targets = a3_data["targets"].astype(np.int32)

assert qlora_probs.shape == a3_probs.shape == y_true.shape
assert np.array_equal(y_true, a3_targets), "Dev labels do not match!"

for name, probs in [
    ("QLoRA pilot (100 steps)", qlora_probs),
    ("A3 full fine-tuning", a3_probs),
]:
    print(f"\n{name}")
    for threshold in [0.05, 0.10, 0.15, 0.20]:
        pred = (probs >= threshold).astype(np.int32)
        macro = f1_score(y_true, pred, average="macro", zero_division=0)
        micro = f1_score(y_true, pred, average="micro", zero_division=0)
        subset = np.all(pred == y_true, axis=1).mean()

        print(
            f"Threshold={threshold:.2f} | "
            f"Macro-F1={macro:.5f} | "
            f"Micro-F1={micro:.5f} | "
            f"Subset accuracy={subset:.5f} | "
            f"Avg labels={pred.sum(axis=1).mean():.2f}"
        )


QLoRA pilot (100 steps)
Threshold=0.05 | Macro-F1=0.04902 | Micro-F1=0.19602 | Subset accuracy=0.00000 | Avg labels=15.46
Threshold=0.10 | Macro-F1=0.02347 | Micro-F1=0.23624 | Subset accuracy=0.00000 | Avg labels=6.42
Threshold=0.15 | Macro-F1=0.01691 | Micro-F1=0.23099 | Subset accuracy=0.00069 | Avg labels=3.84
Threshold=0.20 | Macro-F1=0.01153 | Micro-F1=0.18944 | Subset accuracy=0.00678 | Avg labels=2.07

A3 full fine-tuning
Threshold=0.05 | Macro-F1=0.21522 | Micro-F1=0.30142 | Subset accuracy=0.00000 | Avg labels=15.40
Threshold=0.10 | Macro-F1=0.23491 | Micro-F1=0.37927 | Subset accuracy=0.00147 | Avg labels=8.93
Threshold=0.15 | Macro-F1=0.22766 | Micro-F1=0.40997 | Subset accuracy=0.00629 | Avg labels=6.23
Threshold=0.20 | Macro-F1=0.21212 | Micro-F1=0.41720 | Subset accuracy=0.01306 | Avg labels=4.65


In [187]:
import os
import torch

checkpoints = {
    "A3 pretrained": "/kaggle/input/datasets/aq1de3/33kkkk/step_0330000.pt",
    "A4 pretrained": "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
    "A3 full fine-tuned": "/kaggle/working/phase5_exp4_a3/a3_full_finetune_epoch1.pt",
}

for name, path in checkpoints.items():
    print(f"\n{name}")
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        obj = torch.load(path, map_location="cpu", weights_only=False)
        print("Object type:", type(obj).__name__)

        if isinstance(obj, dict):
            print("Top-level keys:", list(obj.keys())[:15])


A3 pretrained
Exists: True
Object type: dict
Top-level keys: ['step', 'tokens_seen', 'model_state_dict', 'optimizer_state_dict', 'architecture', 'vocab_size', 'context_length', 'hidden_size', 'num_layers', 'num_heads', 'ffn_size', 'prefix_ratio', 'dropout', 'learning_rate', 'weight_decay']

A4 pretrained
Exists: True
Object type: dict
Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']

A3 full fine-tuned
Exists: True
Object type: dict
Top-level keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'train_loss', 'architecture', 'pretrained_step']


In [188]:
import torch

path = "/kaggle/working/phase5_exp4_a3/a3_full_finetune_epoch1.pt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)

print("Architecture:", ckpt.get("architecture"))
print("Epoch:", ckpt.get("epoch"))
print("Pretrained step:", ckpt.get("pretrained_step"))

state = ckpt["model_state_dict"]

print("\nNumber of state-dict entries:", len(state))

for name, tensor in list(state.items())[:20]:
    print(name, tuple(tensor.shape))

Architecture: A3_prefix_lm_hybrid
Epoch: 1
Pretrained step: 330000

Number of state-dict entries: 102
backbone.token_embedding.weight (50000, 512)
backbone.position_embedding.weight (2048, 512)
backbone.layers.0.ln1.weight (512,)
backbone.layers.0.ln1.bias (512,)
backbone.layers.0.attn.in_proj_weight (1536, 512)
backbone.layers.0.attn.in_proj_bias (1536,)
backbone.layers.0.attn.out_proj.weight (512, 512)
backbone.layers.0.attn.out_proj.bias (512,)
backbone.layers.0.ln2.weight (512,)
backbone.layers.0.ln2.bias (512,)
backbone.layers.0.ffn.0.weight (2048, 512)
backbone.layers.0.ffn.0.bias (2048,)
backbone.layers.0.ffn.3.weight (512, 2048)
backbone.layers.0.ffn.3.bias (512,)
backbone.layers.1.ln1.weight (512,)
backbone.layers.1.ln1.bias (512,)
backbone.layers.1.attn.in_proj_weight (1536, 512)
backbone.layers.1.attn.in_proj_bias (1536,)
backbone.layers.1.attn.out_proj.weight (512, 512)
backbone.layers.1.attn.out_proj.bias (512,)


In [189]:
path = "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt"
ckpt_a4 = torch.load(path, map_location="cpu", weights_only=False)

print("A4 step:", ckpt_a4.get("step"))
print("A4 training loss:", ckpt_a4.get("train_loss"))

state_a4 = ckpt_a4["model_state_dict"]

print("\nNumber of A4 state-dict entries:", len(state_a4))

for name, tensor in list(state_a4.items())[:20]:
    print(name, tuple(tensor.shape))

A4 step: 22000
A4 training loss: 0.1944424146786332

Number of A4 state-dict entries: 260
token_embedding.weight (50000, 512)
encoder_pos.weight (1024, 512)
decoder_pos.weight (1024, 512)
encoder_layers.0.norm1.weight (512,)
encoder_layers.0.norm1.bias (512,)
encoder_layers.0.attn.q_proj.weight (512, 512)
encoder_layers.0.attn.q_proj.bias (512,)
encoder_layers.0.attn.k_proj.weight (512, 512)
encoder_layers.0.attn.k_proj.bias (512,)
encoder_layers.0.attn.v_proj.weight (512, 512)
encoder_layers.0.attn.v_proj.bias (512,)
encoder_layers.0.attn.out_proj.weight (512, 512)
encoder_layers.0.attn.out_proj.bias (512,)
encoder_layers.0.norm2.weight (512,)
encoder_layers.0.norm2.bias (512,)
encoder_layers.0.ffn.net.0.weight (2048, 512)
encoder_layers.0.ffn.net.0.bias (2048,)
encoder_layers.0.ffn.net.2.weight (512, 2048)
encoder_layers.0.ffn.net.2.bias (512,)
encoder_layers.1.norm1.weight (512,)


In [190]:
import torch

path = "/kaggle/working/phase5_exp4_a3/a3_full_finetune_epoch1.pt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)

print("Architecture:", ckpt.get("architecture"))
print("Epoch:", ckpt.get("epoch"))
print("Pretrained step:", ckpt.get("pretrained_step"))

state = ckpt["model_state_dict"]
print("\nNumber of entries:", len(state))

for name, tensor in state.items():
    if "classifier" in name.lower() or "backbone" not in name.lower():
        print(name, tuple(tensor.shape))

Architecture: A3_prefix_lm_hybrid
Epoch: 1
Pretrained step: 330000

Number of entries: 102
classifier.weight (100, 512)
classifier.bias (100,)


In [191]:
for name, tensor in ckpt["model_state_dict"].items():
    print(name, tuple(tensor.shape))
    

backbone.token_embedding.weight (50000, 512)
backbone.position_embedding.weight (2048, 512)
backbone.layers.0.ln1.weight (512,)
backbone.layers.0.ln1.bias (512,)
backbone.layers.0.attn.in_proj_weight (1536, 512)
backbone.layers.0.attn.in_proj_bias (1536,)
backbone.layers.0.attn.out_proj.weight (512, 512)
backbone.layers.0.attn.out_proj.bias (512,)
backbone.layers.0.ln2.weight (512,)
backbone.layers.0.ln2.bias (512,)
backbone.layers.0.ffn.0.weight (2048, 512)
backbone.layers.0.ffn.0.bias (2048,)
backbone.layers.0.ffn.3.weight (512, 2048)
backbone.layers.0.ffn.3.bias (512,)
backbone.layers.1.ln1.weight (512,)
backbone.layers.1.ln1.bias (512,)
backbone.layers.1.attn.in_proj_weight (1536, 512)
backbone.layers.1.attn.in_proj_bias (1536,)
backbone.layers.1.attn.out_proj.weight (512, 512)
backbone.layers.1.attn.out_proj.bias (512,)
backbone.layers.1.ln2.weight (512,)
backbone.layers.1.ln2.bias (512,)
backbone.layers.1.ffn.0.weight (2048, 512)
backbone.layers.1.ffn.0.bias (2048,)
backbone.laye

In [192]:
import inspect

print("Available A4 model classes:")

for name, obj in globals().copy().items():
    if (
        isinstance(obj, type)
        and isinstance(obj, type(torch.nn.Module))
        and ("A4" in name or "Encoder" in name)
    ):
        print("\nClass:", name)
        try:
            print(inspect.getsource(obj.forward))
        except (OSError, TypeError):
            print("Source unavailable")

Available A4 model classes:

Class: EncoderBlock
    def forward(self, x, attention_mask=None):
        x = x + self.attn(
            self.norm1(x), self.norm1(x), self.norm1(x),
            key_padding_mask=attention_mask,
        )
        x = x + self.ffn(self.norm2(x))
        return x


Class: A4EncoderDecoder
    def forward(self, encoder_ids, decoder_ids):
        return self.decode(decoder_ids, self.encode(encoder_ids))


Class: A4StatuteClassifier
    def forward(self, input_ids, attention_mask):
        # Encoder output: [batch_size, seq_len, 512]
        hidden = self.encoder.encode(input_ids, attention_mask)

        # Mean-pool only valid tokens
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1)
        pooled = pooled / mask.sum(dim=1).clamp(min=1.0)

        logits = self.classifier(self.dropout(pooled))
        return logits



In [193]:
import inspect

print(inspect.getsource(A4StatuteClassifier.__init__))

    def __init__(self, encoder, num_labels=NUM_LABELS, dropout=0.2):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(D_MODEL, num_labels)



In [194]:
print(inspect.getsource(A4EncoderDecoder.__init__))
print(inspect.getsource(A4EncoderDecoder.encode))

    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.encoder_pos = nn.Embedding(ENCODER_CONTEXT, D_MODEL)
        self.decoder_pos = nn.Embedding(DECODER_CONTEXT, D_MODEL)

        self.encoder_layers = nn.ModuleList(
            [EncoderBlock() for _ in range(NUM_ENCODER_LAYERS)]
        )
        self.decoder_layers = nn.ModuleList(
            [DecoderBlock() for _ in range(NUM_DECODER_LAYERS)]
        )

        self.encoder_norm = nn.LayerNorm(D_MODEL)
        self.decoder_norm = nn.LayerNorm(D_MODEL)
        self.lm_head = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.dropout = nn.Dropout(DROPOUT)

    def encode(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device)

        x = self.token_embedding(input_ids)
        x = x + self.encoder_pos(positions)[None, :, 

In [195]:
import torch

a4_checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt"
)

a4_checkpoint = torch.load(
    a4_checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

a4_encoder_decoder = A4EncoderDecoder()

load_result = a4_encoder_decoder.load_state_dict(
    a4_checkpoint["model_state_dict"],
    strict=True,
)

print("A4 checkpoint loaded successfully:", load_result)
print("Encoder layers:", len(a4_encoder_decoder.encoder_layers))
print("Decoder layers:", len(a4_encoder_decoder.decoder_layers))
print("Embedding shape:", tuple(a4_encoder_decoder.token_embedding.weight.shape))

A4 checkpoint loaded successfully: <All keys matched successfully>
Encoder layers: 6
Decoder layers: 6
Embedding shape: (50000, 512)


In [196]:
batch = next(iter(dev_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

a4_encoder_decoder = a4_encoder_decoder.to(device).eval()

with torch.no_grad():
    hidden = a4_encoder_decoder.encode(input_ids, attention_mask)

print("Hidden shape:", tuple(hidden.shape))
print("Finite:", torch.isfinite(hidden).all().item())

Hidden shape: (4, 512, 512)
Finite: True


In [197]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Teacher: trained A3 statute classifier
teacher_ckpt = torch.load(
    "/kaggle/working/phase5_exp4_a3/a3_full_finetune_epoch1.pt",
    map_location="cpu",
    weights_only=False,
)

teacher = A3StatuteClassifier(
    pretrained_model=A3PrefixLM(),
    num_labels=100,
)
teacher.load_state_dict(teacher_ckpt["model_state_dict"], strict=True)
teacher = teacher.to(device).eval()

for p in teacher.parameters():
    p.requires_grad = False

# Student: pretrained A4 encoder with a new classification head
student_encoder = A4EncoderDecoder()
student_encoder.load_state_dict(
    a4_checkpoint["model_state_dict"], strict=True
)

student = A4StatuteClassifier(
    encoder=student_encoder,
    num_labels=100,
).to(device)

print("Teacher parameters:", sum(p.numel() for p in teacher.parameters()))
print("Student parameters:", sum(p.numel() for p in student.parameters()))
print("Teacher frozen:", not any(p.requires_grad for p in teacher.parameters()))

Teacher parameters: 51919972
Student parameters: 70840420
Teacher frozen: True


In [198]:
batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

teacher.eval()
student.eval()

with torch.no_grad():
    teacher_logits = teacher(input_ids, attention_mask)
    student_logits = student(input_ids, attention_mask)

print("Teacher logits:", tuple(teacher_logits.shape))
print("Student logits:", tuple(student_logits.shape))
print("Teacher finite:", torch.isfinite(teacher_logits).all().item())
print("Student finite:", torch.isfinite(student_logits).all().item())

Teacher logits: (4, 100)
Student logits: (4, 100)
Teacher finite: True
Student finite: True


In [199]:
import torch
import torch.nn.functional as F

batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device).float()

teacher.eval()
student.train()

with torch.no_grad():
    teacher_logits = teacher(input_ids, attention_mask)

student_logits = student(input_ids, attention_mask)

temperature = 2.0
alpha = 0.5

supervised_loss = F.binary_cross_entropy_with_logits(
    student_logits, labels
)

teacher_probs = torch.sigmoid(teacher_logits / temperature)
student_log_probs = F.logsigmoid(student_logits / temperature)
student_log_not_probs = F.logsigmoid(-student_logits / temperature)

distillation_loss = -(
    teacher_probs * student_log_probs
    + (1 - teacher_probs) * student_log_not_probs
).mean() * (temperature ** 2)

total_loss = (
    (1 - alpha) * supervised_loss
    + alpha * distillation_loss
)

print("Supervised BCE:", supervised_loss.item())
print("Distillation loss:", distillation_loss.item())
print("Combined loss:", total_loss.item())
print("Finite:", torch.isfinite(total_loss).item())

Supervised BCE: 0.71101975440979
Distillation loss: 2.8005998134613037
Combined loss: 1.7558097839355469
Finite: True


In [200]:
import torch
import torch.nn.functional as F
import time
import os

teacher.eval()
student.train()

optimizer_kd = torch.optim.AdamW(
    [p for p in student.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01,
)

temperature = 2.0
alpha = 0.5
pilot_kd_losses = []

start_time = time.time()

for step, batch in enumerate(train_loader):
    if step >= 100:
        break

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device).float()

    with torch.no_grad():
        teacher_logits = teacher(input_ids, attention_mask)

    optimizer_kd.zero_grad(set_to_none=True)
    student_logits = student(input_ids, attention_mask)

    supervised_loss = F.binary_cross_entropy_with_logits(
        student_logits, labels
    )

    teacher_probs = torch.sigmoid(teacher_logits / temperature)
    student_log_probs = F.logsigmoid(student_logits / temperature)
    student_log_not_probs = F.logsigmoid(-student_logits / temperature)

    kd_loss = -(
        teacher_probs * student_log_probs
        + (1 - teacher_probs) * student_log_not_probs
    ).mean() * temperature**2

    loss = (1 - alpha) * supervised_loss + alpha * kd_loss

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}")

    loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    optimizer_kd.step()

    pilot_kd_losses.append(loss.item())

    if (step + 1) % 20 == 0:
        recent = pilot_kd_losses[-20:]
        print(f"Step {step+1}/100 | Mean loss: {sum(recent)/len(recent):.4f}")

print(f"Elapsed: {(time.time()-start_time)/60:.2f} minutes")
print("First 20 mean:", sum(pilot_kd_losses[:20])/20)
print("Last 20 mean:", sum(pilot_kd_losses[-20:])/20)

Step 20/100 | Mean loss: 1.4545
Step 40/100 | Mean loss: 1.0008
Step 60/100 | Mean loss: 0.8293
Step 80/100 | Mean loss: 0.7771
Step 100/100 | Mean loss: 0.7497
Elapsed: 0.37 minutes
First 20 mean: 1.4544631123542786
Last 20 mean: 0.7497457325458526


In [201]:

import torch
import numpy as np
import time
from sklearn.metrics import f1_score, accuracy_score

student.eval()
all_probs = []
all_targets = []

start_time = time.time()

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = student(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_targets.append(batch["labels"].cpu().numpy())

all_probs = np.concatenate(all_probs, axis=0)
all_targets = np.concatenate(all_targets, axis=0).astype(int)

print("Evaluation complete")
print("Predictions shape:", all_probs.shape)
print("Targets shape:", all_targets.shape)
print(f"Elapsed: {(time.time()-start_time)/60:.2f} minutes")

for threshold in [0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.50]:
    preds = (all_probs >= threshold).astype(int)

    macro = f1_score(
        all_targets, preds, average="macro", zero_division=0
    )
    micro = f1_score(
        all_targets, preds, average="micro", zero_division=0
    )
    subset = accuracy_score(all_targets, preds)
    avg_labels = preds.sum(axis=1).mean()

    print(
        f"Threshold={threshold:.3f} | "
        f"Macro-F1={macro:.5f} | "
        f"Micro-F1={micro:.5f} | "
        f"Subset Accuracy={subset:.5f} | "
        f"Avg labels/doc={avg_labels:.2f}"
    )

np.savez_compressed(
    "/kaggle/working/phase5_exp7_kd_dev_predictions.npz",
    probabilities=all_probs,
    labels=all_targets
)
print("Saved development predictions.")


Evaluation complete
Predictions shape: (10181, 100)
Targets shape: (10181, 100)
Elapsed: 1.87 minutes
Threshold=0.030 | Macro-F1=0.04653 | Micro-F1=0.16731 | Subset Accuracy=0.00000 | Avg labels/doc=27.07
Threshold=0.050 | Macro-F1=0.03702 | Micro-F1=0.21971 | Subset Accuracy=0.00000 | Avg labels/doc=14.07
Threshold=0.075 | Macro-F1=0.02165 | Micro-F1=0.24404 | Subset Accuracy=0.00000 | Avg labels/doc=6.00
Threshold=0.100 | Macro-F1=0.01909 | Micro-F1=0.25129 | Subset Accuracy=0.00000 | Avg labels/doc=4.61
Threshold=0.125 | Macro-F1=0.01909 | Micro-F1=0.25129 | Subset Accuracy=0.00000 | Avg labels/doc=4.61
Threshold=0.150 | Macro-F1=0.01612 | Micro-F1=0.23824 | Subset Accuracy=0.00010 | Avg labels/doc=3.61
Threshold=0.200 | Macro-F1=0.01095 | Micro-F1=0.18261 | Subset Accuracy=0.00049 | Avg labels/doc=2.39
Threshold=0.300 | Macro-F1=0.00000 | Micro-F1=0.00000 | Subset Accuracy=0.00000 | Avg labels/doc=0.00
Threshold=0.500 | Macro-F1=0.00000 | Micro-F1=0.00000 | Subset Accuracy=0.00000 

In [202]:

import os
import time
import torch
import torch.nn.functional as F

teacher.eval()
student.train()

kd_losses = []
start_time = time.time()

for step, batch in enumerate(train_loader):
    if step < 100:
        continue  # The first 100 batches were already trained.

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device).float()

    with torch.no_grad():
        teacher_logits = teacher(input_ids, attention_mask)

    optimizer_kd.zero_grad(set_to_none=True)
    student_logits = student(input_ids, attention_mask)

    supervised_loss = F.binary_cross_entropy_with_logits(
        student_logits, labels
    )

    temperature = 2.0
    alpha = 0.5
    teacher_probs = torch.sigmoid(teacher_logits / temperature)

    distillation_loss = F.binary_cross_entropy_with_logits(
        student_logits / temperature,
        teacher_probs
    ) * temperature**2

    loss = (
        (1 - alpha) * supervised_loss
        + alpha * distillation_loss
    )

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at batch {step}")

    loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    optimizer_kd.step()

    kd_losses.append(loss.item())

    if (step + 1) % 1000 == 0:
        print(
            f"Batch {step+1}/{len(train_loader)} | "
            f"Recent mean loss: "
            f"{sum(kd_losses[-100:]) / len(kd_losses[-100:]):.4f}"
        )

os.makedirs("/kaggle/working/phase5_exp7_kd", exist_ok=True)

torch.save(
    {
        "model_state_dict": student.state_dict(),
        "optimizer_state_dict": optimizer_kd.state_dict(),
        "epoch": 1,
        "batches_completed": len(train_loader),
        "temperature": temperature,
        "alpha": alpha,
    },
    "/kaggle/working/phase5_exp7_kd/kd_epoch1.pt"
)

print(f"Remaining batches trained: {len(kd_losses)}")
print(f"Mean remaining loss: {sum(kd_losses)/len(kd_losses):.4f}")
print(f"Elapsed: {(time.time()-start_time)/60:.2f} minutes")
print("Saved KD epoch checkpoint.")


Batch 1000/10688 | Recent mean loss: 0.7369
Batch 2000/10688 | Recent mean loss: 0.7222
Batch 3000/10688 | Recent mean loss: 0.7188
Batch 4000/10688 | Recent mean loss: 0.7144
Batch 5000/10688 | Recent mean loss: 0.7177
Batch 6000/10688 | Recent mean loss: 0.7149
Batch 7000/10688 | Recent mean loss: 0.7151
Batch 8000/10688 | Recent mean loss: 0.7060
Batch 9000/10688 | Recent mean loss: 0.7090
Batch 10000/10688 | Recent mean loss: 0.7004
Remaining batches trained: 10588
Mean remaining loss: 0.7140
Elapsed: 40.83 minutes
Saved KD epoch checkpoint.


In [203]:

import os
import time
import numpy as np
import torch
from sklearn.metrics import f1_score, accuracy_score

# Evaluate the trained student model
student.eval()
all_probs = []
all_targets = []

start_time = time.time()

with torch.no_grad():
    for i, batch in enumerate(dev_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = student(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_targets.append(batch["labels"].cpu().numpy())

        if (i + 1) % 500 == 0:
            print(f"Evaluated {i+1}/{len(dev_loader)} batches")

all_probs = np.concatenate(all_probs, axis=0)
all_targets = np.concatenate(all_targets, axis=0).astype(int)

print("\nEvaluation complete")
print("Predictions shape:", all_probs.shape)
print("Targets shape:", all_targets.shape)
print(f"Elapsed: {(time.time()-start_time)/60:.2f} minutes")

results = []

for threshold in [0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.50]:
    preds = (all_probs >= threshold).astype(int)

    macro = f1_score(
        all_targets, preds, average="macro", zero_division=0
    )
    micro = f1_score(
        all_targets, preds, average="micro", zero_division=0
    )
    subset = accuracy_score(all_targets, preds)
    avg_labels = preds.sum(axis=1).mean()

    results.append({
        "threshold": threshold,
        "macro_f1": macro,
        "micro_f1": micro,
        "subset_accuracy": subset,
        "avg_labels_per_doc": avg_labels
    })

    print(
        f"Threshold={threshold:.3f} | "
        f"Macro-F1={macro:.5f} | "
        f"Micro-F1={micro:.5f} | "
        f"Subset Accuracy={subset:.5f} | "
        f"Avg labels/doc={avg_labels:.2f}"
    )

output_dir = "/kaggle/working/phase5_exp7_kd"
os.makedirs(output_dir, exist_ok=True)

np.savez_compressed(
    os.path.join(output_dir, "kd_epoch1_dev_predictions.npz"),
    probabilities=all_probs,
    labels=all_targets
)

import json

with open(os.path.join(output_dir, "kd_epoch1_dev_results.json"), "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved KD development predictions and metrics.")


Evaluated 500/2546 batches
Evaluated 1000/2546 batches
Evaluated 1500/2546 batches
Evaluated 2000/2546 batches
Evaluated 2500/2546 batches

Evaluation complete
Predictions shape: (10181, 100)
Targets shape: (10181, 100)
Elapsed: 1.85 minutes
Threshold=0.030 | Macro-F1=0.11902 | Micro-F1=0.21930 | Subset Accuracy=0.00000 | Avg labels/doc=21.76
Threshold=0.050 | Macro-F1=0.10711 | Micro-F1=0.26367 | Subset Accuracy=0.00000 | Avg labels/doc=14.21
Threshold=0.075 | Macro-F1=0.09777 | Micro-F1=0.29396 | Subset Accuracy=0.00000 | Avg labels/doc=9.96
Threshold=0.100 | Macro-F1=0.09346 | Micro-F1=0.31095 | Subset Accuracy=0.00000 | Avg labels/doc=7.62
Threshold=0.125 | Macro-F1=0.08872 | Micro-F1=0.31989 | Subset Accuracy=0.00029 | Avg labels/doc=6.06
Threshold=0.150 | Macro-F1=0.08274 | Micro-F1=0.32238 | Subset Accuracy=0.00147 | Avg labels/doc=4.95
Threshold=0.200 | Macro-F1=0.06986 | Micro-F1=0.31188 | Subset Accuracy=0.00619 | Avg labels/doc=3.31
Threshold=0.300 | Macro-F1=0.04189 | Micro

In [204]:

import torch
import torch.nn as nn

print("Device:", device)
print("A3 layers:", len(a3_backbone.layers) if "a3_backbone" in globals() else "a3_backbone not defined")

# Inspect the current A3 model structure.
for name, module in A3PrefixLM().named_modules():
    if isinstance(module, nn.MultiheadAttention):
        print(
            name,
            "| embed_dim:", module.embed_dim,
            "| num_heads:", module.num_heads,
            "| batch_first:", module.batch_first
        )


Device: cuda
A3 layers: a3_backbone not defined
layers.0.attn | embed_dim: 512 | num_heads: 8 | batch_first: True
layers.1.attn | embed_dim: 512 | num_heads: 8 | batch_first: True
layers.2.attn | embed_dim: 512 | num_heads: 8 | batch_first: True
layers.3.attn | embed_dim: 512 | num_heads: 8 | batch_first: True
layers.4.attn | embed_dim: 512 | num_heads: 8 | batch_first: True
layers.5.attn | embed_dim: 512 | num_heads: 8 | batch_first: True
layers.6.attn | embed_dim: 512 | num_heads: 8 | batch_first: True
layers.7.attn | embed_dim: 512 | num_heads: 8 | batch_first: True


In [205]:

import os
import json
import inspect
import torch

test_path = "/kaggle/input/datasets/aq1de3/testtest/test (3).json"
checkpoint_path = "/kaggle/working/phase5_exp4_a3/a3_full_finetune_epoch1.pt"

print("Test file exists:", os.path.exists(test_path))
print("A3 checkpoint exists:", os.path.exists(checkpoint_path))

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Test data type:", type(test_data).__name__)
print("Test examples:", len(test_data))
print("First test record:", test_data[0] if isinstance(test_data, list) else next(iter(test_data.items())))
print("StatuteDataset constructor:", inspect.signature(StatuteDataset.__init__))


Test file exists: True
A3 checkpoint exists: True
Test data type: list
Test examples: 13019
First test record: {'id': '100002997', 'text': "2.Prosecution case in brief was that <ENTITY>, daughter of informant <ENTITY> (PW-1) was married to <ENTITY> 5 years ago. After marriage <ENTITY> was treated with cruelty by her husband <ENTITY>, <ENTITY>-in-law <ENTITY> and mother-in-law <ENTITY> <ENTITY> demand of dowry. For this reason these three accused had committed murder of <ENTITY> on 8.5.2010 by inflicting injuries on her body at their house situate in village Chaubepur, police station- Mardah, district Ghazipur and tried to dispose of her dead body. After receiving knowledge of this incident, victim's <ENTITY> <ENTITY> (informant) had given a written report (Ex-Ka-1) to the police on the basis of which case crime no. 689 of 2010 was registered. 3.In the aforesaid case inquest of the dead body was performed on 08.05.2010 and postmortem of the deceased was conducted the <ENTITY> evening at

In [206]:

import os
import json
import zipfile
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Paths
test_path = "/kaggle/input/datasets/aq1de3/testtest/test (3).json"
checkpoint_path = "/kaggle/working/phase5_exp4_a3/a3_full_finetune_epoch1.pt"
out_dir = "/kaggle/working/phase5_submission_a3"
os.makedirs(out_dir, exist_ok=True)

# Load test data
with open(test_path, "r", encoding="utf-8") as f:
    test_records = json.load(f)

# Build test dataset and loader using the existing tokenizer
test_dataset = StatuteDataset(
    records=test_records,
    tokenizer=tokenizer,
    max_length=512,
    labeled=False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

# Load the A3 Phase 1 architecture and fine-tuned classifier
a3_backbone = A3PrefixLM()
a3_pretrained = torch.load(
    "/kaggle/input/datasets/aq1de3/33kkkk/step_0330000.pt",
    map_location="cpu",
    weights_only=False
)
a3_backbone.load_state_dict(
    a3_pretrained.get("model_state_dict", a3_pretrained),
    strict=True
)

a3_submission_model = A3StatuteClassifier(
    pretrained_model=a3_backbone,
    num_labels=100
)

a3_ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)
a3_submission_model.load_state_dict(
    a3_ckpt["model_state_dict"],
    strict=True
)
a3_submission_model = a3_submission_model.to(device).eval()

# Use the threshold that gave the best Macro-F1 on development data
threshold = 0.10

predictions = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting test documents"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = a3_submission_model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()

        ids = batch["id"]
        for case_id, row in zip(ids, probs):
            pred = np.flatnonzero(row >= threshold).astype(int).tolist()

            # Ensure every document receives at least one prediction.
            if not pred:
                pred = [int(np.argmax(row))]

            predictions.append({
                "id": str(case_id),
                "pred": pred
            })

# Validate required output format
test_ids = [str(r["id"]) for r in test_records]
pred_ids = [p["id"] for p in predictions]

assert len(predictions) == len(test_records), "Wrong number of predictions"
assert len(set(pred_ids)) == len(pred_ids), "Duplicate document IDs"
assert set(pred_ids) == set(test_ids), "Test/prediction IDs do not match"

valid_labels = set(range(100))
for p in predictions:
    assert isinstance(p["pred"], list)
    assert all(isinstance(x, int) and x in valid_labels for x in p["pred"])
    assert len(p["pred"]) == len(set(p["pred"]))

# Save JSON and ZIP
json_path = os.path.join(out_dir, "prediction.json")
zip_path = os.path.join(out_dir, "submission.zip")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname="prediction.json")

print("Validation passed.")
print("Test documents:", len(test_records))
print("Predictions:", len(predictions))
print("Mean predicted statutes:", np.mean([len(p["pred"]) for p in predictions]))
print("Sample prediction:", predictions[0])
print("JSON:", json_path)
print("ZIP:", zip_path)
print("ZIP contents:", zipfile.ZipFile(zip_path).namelist())


Predicting test documents:   0%|          | 0/3255 [00:00<?, ?it/s]

Validation passed.
Test documents: 13019
Predictions: 13019
Mean predicted statutes: 8.842230586066519
Sample prediction: {'id': '100002997', 'pred': [5, 34, 35, 37, 38, 94]}
JSON: /kaggle/working/phase5_submission_a3/prediction.json
ZIP: /kaggle/working/phase5_submission_a3/submission.zip
ZIP contents: ['prediction.json']


In [1]:

from pathlib import Path
import torch

ROOT = Path("/kaggle/input/datasets/aq1de3")

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

print("\n========== PHASE 1 CHECKPOINTS ==========")

checkpoint_paths = {
    "A1": ROOT / "architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2": ROOT / "a2-step-9000-pt/a2_step_9000.pt",
    "A3": ROOT / "step-0001000-pt/step_0001000.pt",
    "A4": ROOT / "a4-step-22000-pt/a4_step_22000.pt",
}

for name, path in checkpoint_paths.items():
    print(name, "FOUND" if path.is_file() else "MISSING", path)

print("\n========== TOKENIZER ==========")

tokenizer_path = ROOT / "tokenizer-json/tokenizer.json"
print("Tokenizer:", "FOUND" if tokenizer_path.is_file() else "MISSING")

print("\n========== PHASE 6 DATASET ==========")

for path in ROOT.rglob("*"):
    if path.is_file() and "phase_6" in str(path).lower():
        print(path)


GPU available: True
GPU: Tesla T4
GPU memory (GB): 15.64

========== PHASE 1 CHECKPOINTS ==========
A1 FOUND /kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
A2 FOUND /kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt
A3 FOUND /kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt
A4 FOUND /kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt

========== TOKENIZER ==========
Tokenizer: FOUND

========== PHASE 6 DATASET ==========


In [2]:

import json
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/aq1de3")
PHASE6 = ROOT / "phase6"

print("PHASE 6 FILES")

for f in PHASE6.iterdir():
    print(f.name, f.stat().st_size, "bytes")

train_path = PHASE6 / "train (3).json"
test_path = PHASE6 / "test (4).json"

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("\nTRAIN TYPE:", type(train_data))
print("TEST TYPE:", type(test_data))

print("\nTRAIN COUNT:", len(train_data))
print("TEST COUNT:", len(test_data))

print("\nTRAIN SAMPLE:")
print(json.dumps(
    train_data[0] if isinstance(train_data, list)
    else next(iter(train_data.values())),
    indent=2,
    ensure_ascii=False
)[:4000])

print("\nTEST SAMPLE:")
print(json.dumps(
    test_data[0] if isinstance(test_data, list)
    else next(iter(test_data.values())),
    indent=2,
    ensure_ascii=False
)[:2000])


PHASE 6 FILES
test (4).json 2734113 bytes
train (3).json 210600429 bytes

TRAIN TYPE: <class 'list'>
TEST TYPE: <class 'list'>

TRAIN COUNT: 7030
TEST COUNT: 100

TRAIN SAMPLE:
{
  "id": "6712",
  "document": "IVIL Appeal No.768 (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh High Court in M.C.C. No.144 of 1966. V.A. Bobde, B.R.Aggarwala and U.A.Rana for the Appellant. S.V.Deshpande and S.K.Agnihotri for the Respondents. The Judgment of the Court was delivered by RANGANATH MISHRA, CJ. These are appeals by special leave 252 and are directed against the separate decisions of the Madhya Pradesh High Court in references under the Madhya Pradesh Sales Tax Act. Civil Appeal No.768/77 relates to the assessment period 1951 52, Civil Appeal 539/78 relates to 1950 51 and Civil Appeal 1038/78 to 1952 53. The appellant is a manufacturer of cement in the factory located at Kymore in Madhya Pradesh. Several cement manufacturing companies as also the appellant had ente

In [3]:

import json
import numpy as np
from collections import Counter

# =====================================================
# PHASE 6 DATASET QUALITY INSPECTION
# =====================================================

print("========== DATASET STRUCTURE ==========")

print("Train keys:", train_data[0].keys())
print("Test keys:", test_data[0].keys())

# Validate required fields
required = ["id", "document", "summary"]

missing = {
    key: sum(key not in x for x in train_data)
    for key in required
}

print("\nMissing fields:", missing)

empty_documents = sum(
    not str(x.get("document", "")).strip()
    for x in train_data
)

empty_summaries = sum(
    not str(x.get("summary", "")).strip()
    for x in train_data
)

print("Empty documents:", empty_documents)
print("Empty summaries:", empty_summaries)

# Duplicate IDs
ids = [x["id"] for x in train_data]
print("Duplicate train IDs:", len(ids) - len(set(ids)))

test_ids = [x["id"] for x in test_data]
print("Duplicate test IDs:", len(test_ids) - len(set(test_ids)))

# Length statistics using words
doc_lengths = [
    len(str(x["document"]).split())
    for x in train_data
]

summary_lengths = [
    len(str(x["summary"]).split())
    for x in train_data
]

def report(name, values):
    print(f"\n{name}")
    for p in [0, 25, 50, 75, 90, 95, 99, 100]:
        print(f"P{p}: {np.percentile(values, p):.0f}")

report("DOCUMENT LENGTH (WORDS)", doc_lengths)
report("SUMMARY LENGTH (WORDS)", summary_lengths)

# Compression ratio
ratios = [
    s / max(d, 1)
    for s, d in zip(summary_lengths, doc_lengths)
]

report("SUMMARY / DOCUMENT RATIO", ratios)

print("\n========== SAMPLE SUMMARY ==========")
print(train_data[0]["summary"][:2000])

print("\n========== INSPECTION COMPLETE ==========")


========== DATASET STRUCTURE ==========
Train keys: dict_keys(['id', 'document', 'summary'])
Test keys: dict_keys(['id', 'document'])

Missing fields: {'id': 0, 'document': 0, 'summary': 0}
Empty documents: 0
Empty summaries: 2
Duplicate train IDs: 0
Duplicate test IDs: 0

DOCUMENT LENGTH (WORDS)
P0: 93
P25: 2017
P50: 3128
P75: 4972
P90: 7851
P95: 11327
P99: 24245
P100: 139645

SUMMARY LENGTH (WORDS)
P0: 0
P25: 421
P50: 638
P75: 963
P90: 1501
P95: 2011
P99: 4251
P100: 27768

SUMMARY / DOCUMENT RATIO
P0: 0
P25: 0
P50: 0
P75: 0
P90: 0
P95: 0
P99: 1
P100: 9

========== SAMPLE SUMMARY ==========
The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell its cement only through the marketing company, and certain sums would be paid for the cement supplied by the marketing company, which had the discretion to fix the sale price. For the 1950 51, 1951 52 and 1952 5

In [4]:

import json
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer

# =====================================================
# PHASE 6 - TOKENIZER ANALYSIS + VALIDATION SPLIT
# =====================================================

# Find Phase 1 tokenizer
tokenizer_files = list(ROOT.rglob("tokenizer.json"))
assert tokenizer_files, "Tokenizer JSON not found!"

TOKENIZER_PATH = tokenizer_files[0]
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("Tokenizer:", TOKENIZER_PATH)

# Remove empty summaries
clean_data = [
    x for x in train_data
    if str(x.get("summary", "")).strip()
]

print("\nOriginal records:", len(train_data))
print("Clean records:", len(clean_data))
print("Removed records:", len(train_data) - len(clean_data))

# Actual tokenizer lengths
doc_tokens = []
summary_tokens = []

for x in clean_data:
    d = len(tokenizer.encode(x["document"]).ids)
    s = len(tokenizer.encode(x["summary"]).ids)

    doc_tokens.append(d)
    summary_tokens.append(s)

doc_tokens = np.array(doc_tokens)
summary_tokens = np.array(summary_tokens)

def token_report(name, values):
    print(f"\n{name}")
    for p in [25, 50, 75, 90, 95, 99, 100]:
        print(f"P{p}: {np.percentile(values, p):.1f}")

token_report("DOCUMENT TOKEN LENGTH", doc_tokens)
token_report("SUMMARY TOKEN LENGTH", summary_tokens)

ratios = summary_tokens / np.maximum(doc_tokens, 1)

print("\nSUMMARY/DOCUMENT TOKEN RATIO")
for p in [25, 50, 75, 90, 95, 99, 100]:
    print(f"P{p}: {np.percentile(ratios, p):.4f}")

# Find extreme examples
print("\nLONGEST DOCUMENTS")
for i in np.argsort(doc_tokens)[-5:][::-1]:
    print(clean_data[i]["id"], doc_tokens[i])

print("\nLONGEST SUMMARIES")
for i in np.argsort(summary_tokens)[-5:][::-1]:
    print(clean_data[i]["id"], summary_tokens[i])

# Reproducible 90:10 split
train_indices, dev_indices = train_test_split(
    np.arange(len(clean_data)),
    test_size=0.10,
    random_state=42,
    shuffle=True
)

phase6_train = [clean_data[i] for i in train_indices]
phase6_dev = [clean_data[i] for i in dev_indices]

print("\n========== VALIDATION SPLIT ==========")
print("Training:", len(phase6_train))
print("Validation:", len(phase6_dev))
print("Test:", len(test_data))

print("\nPhase 6 preprocessing ready.")


Tokenizer: /kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json

Original records: 7030
Clean records: 7028
Removed records: 2

DOCUMENT TOKEN LENGTH
P25: 2337.8
P50: 3641.5
P75: 5749.5
P90: 9062.8
P95: 13080.2
P99: 28054.8
P100: 161283.0

SUMMARY TOKEN LENGTH
P25: 493.0
P50: 746.5
P75: 1135.0
P90: 1763.3
P95: 2384.3
P99: 5048.0
P100: 32183.0

SUMMARY/DOCUMENT TOKEN RATIO
P25: 0.1440
P50: 0.2193
P75: 0.3136
P90: 0.3989
P95: 0.4472
P99: 0.5540
P100: 5.0251

LONGEST DOCUMENTS
3498 161283
34 104544
67 97704
4656 96887
2189 95013

LONGEST SUMMARIES
3498 32183
4267 20181
6581 16967
4740 16370
5712 15740

========== VALIDATION SPLIT ==========
Training: 6325
Validation: 703
Test: 100

Phase 6 preprocessing ready.


In [5]:

import torch
from pathlib import Path

A4_PATH = Path(
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-22000-pt/a4_step_22000.pt"
)

print("========== A4 CHECKPOINT INSPECTION ==========")

checkpoint = torch.load(
    A4_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nTop-level keys:")
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(key, "-> dictionary,", len(value), "entries")
        elif isinstance(value, torch.Tensor):
            print(key, "->", tuple(value.shape))
        else:
            print(key, "->", str(value)[:200])

    state = checkpoint.get(
        "model_state_dict",
        checkpoint.get("model", checkpoint)
    )

    print("\n========== MODEL PARAMETERS ==========")

    if isinstance(state, dict):
        for key, value in list(state.items())[:30]:
            if isinstance(value, torch.Tensor):
                print(key, tuple(value.shape))

print("\n========== EXISTING MODEL CLASSES ==========")

for name, obj in globals().copy().items():
    if isinstance(obj, type) and (
        "model" in name.lower()
        or "transformer" in name.lower()
        or "encoder" in name.lower()
    ):
        print(name)


========== A4 CHECKPOINT INSPECTION ==========
Checkpoint type: <class 'dict'>

Top-level keys:
model_state_dict -> dictionary, 260 entries
step -> 22000
tokens_processed -> 180096704
train_loss -> 0.1944424146786332
reason -> regular_checkpoint
saved_at -> 2026-10-05T15:18:59.981007

========== MODEL PARAMETERS ==========
token_embedding.weight (50000, 512)
encoder_pos.weight (1024, 512)
decoder_pos.weight (1024, 512)
encoder_layers.0.norm1.weight (512,)
encoder_layers.0.norm1.bias (512,)
encoder_layers.0.attn.q_proj.weight (512, 512)
encoder_layers.0.attn.q_proj.bias (512,)
encoder_layers.0.attn.k_proj.weight (512, 512)
encoder_layers.0.attn.k_proj.bias (512,)
encoder_layers.0.attn.v_proj.weight (512, 512)
encoder_layers.0.attn.v_proj.bias (512,)
encoder_layers.0.attn.out_proj.weight (512, 512)
encoder_layers.0.attn.out_proj.bias (512,)
encoder_layers.0.norm2.weight (512,)
encoder_layers.0.norm2.bias (512,)
encoder_layers.0.ffn.net.0.weight (2048, 512)
encoder_layers.0.ffn.net.0.bias

In [6]:

# Search previously executed notebook cells
print("========== SEARCHING NOTEBOOK HISTORY ==========")

keywords = [
    "class A4",
    "class Encoder",
    "class Decoder",
    "class MultiHead",
    "class CrossAttention",
    "def generate",
    "def forward",
    "encoder_layers",
    "A4StatuteClassifier"
]

for i, cell in enumerate(In):
    if not isinstance(cell, str):
        continue

    matches = [k for k in keywords if k.lower() in cell.lower()]

    if matches:
        print(f"\n--- CELL {i} | Matches: {matches} ---")
        print(cell[:5000])

print("\n========== MODEL DEFINITION SEARCH COMPLETE ==========")


========== SEARCHING NOTEBOOK HISTORY ==========

--- CELL 6 | Matches: ['class A4', 'class Encoder', 'class Decoder', 'class MultiHead', 'class CrossAttention', 'def generate', 'def forward', 'encoder_layers', 'A4StatuteClassifier'] ---
# Search previously executed notebook cells
print("========== SEARCHING NOTEBOOK HISTORY ==========")

keywords = [
    "class A4",
    "class Encoder",
    "class Decoder",
    "class MultiHead",
    "class CrossAttention",
    "def generate",
    "def forward",
    "encoder_layers",
    "A4StatuteClassifier"
]

for i, cell in enumerate(In):
    if not isinstance(cell, str):
        continue

    matches = [k for k in keywords if k.lower() in cell.lower()]

    if matches:
        print(f"\n--- CELL {i} | Matches: {matches} ---")
        print(cell[:5000])

print("\n========== MODEL DEFINITION SEARCH COMPLETE ==========")

========== MODEL DEFINITION SEARCH COMPLETE ==========


In [8]:
from pathlib import Path
import json

candidates = [
    Path("/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb"),
    Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb"),
]

found = False

for path in candidates:
    print(f"\nChecking: {path}")
    print("Exists:", path.exists())

    if not path.exists():
        continue

    try:
        raw = path.read_text(encoding="utf-8").strip()

        if not raw:
            print("Skipped: file is empty.")
            continue

        nb = json.loads(raw)

        for i, cell in enumerate(nb.get("cells", [])):
            src = "".join(cell.get("source", []))

            if any(name in src for name in [
                "class A4Model",
                "class A4EncoderLayer",
                "class A4DecoderLayer",
                "class A4FFNCheckpoint",
                "class A4Attention",
            ]):
                found = True
                print(f"\n========== CELL {i} ==========")
                print(src)

    except (json.JSONDecodeError, UnicodeDecodeError) as e:
        print("Skipped: not a valid notebook JSON file.")
        print("Reason:", e)

if not found:
    print("\nNo A4 model class definitions found in these files.")


Checking: /kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
Exists: True

Checking: /kaggle/working/.virtual_documents/__notebook_source__.ipynb
Exists: True
Skipped: not a valid notebook JSON file.
Reason: Expecting value: line 1 column 1 (char 0)

No A4 model class definitions found in these files.


In [9]:
from pathlib import Path
import json

path = Path(
    "/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb"
)

print("File size:", path.stat().st_size, "bytes")

nb = json.loads(path.read_text(encoding="utf-8"))

print("Notebook format:", nb.get("nbformat"))
print("Total cells:", len(nb.get("cells", [])))

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))

    if source.strip():
        print(f"\n{'='*25} CELL {i} | {cell.get('cell_type')} {'='*25}")
        print(source[:3000])

File size: 57142 bytes
Notebook format: 4
Total cells: 13

========================= CELL 0 | markdown =========================
# Phase 1 model-code extracts (from the uploaded notebook)

These are copied source cells from the supplied notebook, not a verified runnable loader. Do not Run All. The notebook contains multiple architecture variants and benchmark/training cells.

**Compatibility warning:** checkpoint inspection showed A1 keys such as `blocks.0.norm1` (not standard Hugging Face GPT-2 keys); A2 checkpoint tensors are width 512, while the later A2 cell 122 configures width 528. Therefore these cells must be matched against the exact saved checkpoint before use. A3 and A4 also need strict state-dict verification.

This extract intentionally omits pretraining loops and corpus download/training cells.

========================= CELL 1 | markdown =========================
## Original notebook cell 9: A1 architecture/configuration cell (GPT2-based candidate; checkpoint compatibili

In [10]:
keywords = [
    "A4",
    "Attention",
    "EncoderLayer",
    "DecoderLayer",
    "FFN",
    "Transformer",
    "token_embedding",
    "encoder_pos",
    "decoder_pos",
]

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    matches = [word for word in keywords if word.lower() in source.lower()]

    if matches:
        print(f"\nCell {i} | Matches: {matches}")
        print(source[:5000])


Cell 0 | Matches: ['A4']
# Phase 1 model-code extracts (from the uploaded notebook)

These are copied source cells from the supplied notebook, not a verified runnable loader. Do not Run All. The notebook contains multiple architecture variants and benchmark/training cells.

**Compatibility warning:** checkpoint inspection showed A1 keys such as `blocks.0.norm1` (not standard Hugging Face GPT-2 keys); A2 checkpoint tensors are width 512, while the later A2 cell 122 configures width 528. Therefore these cells must be matched against the exact saved checkpoint before use. A3 and A4 also need strict state-dict verification.

This extract intentionally omits pretraining loops and corpus download/training cells.

Cell 2 | Matches: ['Attention', 'Transformer']
# ============================================================
# PHASE 1 — ARCHITECTURE 1
# DECODER-ONLY TRANSFORMER
# MODEL CONSTRUCTION & RESOURCE CHECK
# ============================================================

import torch
fro

In [11]:
import torch

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-22000-pt/a4_step_22000.pt"
)

ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model_state_dict"]

print("Checkpoint metadata:")
for key in ["step", "tokens_processed", "train_loss", "saved_at"]:
    print(f"{key}: {ckpt.get(key)}")

print("\nTotal state-dict entries:", len(state))

print("\nALL parameter names and shapes:")
for name, tensor in state.items():
    print(f"{name:65s} {tuple(tensor.shape)}")

Checkpoint metadata:
step: 22000
tokens_processed: 180096704
train_loss: 0.1944424146786332
saved_at: 2026-10-05T15:18:59.981007

Total state-dict entries: 260

ALL parameter names and shapes:
token_embedding.weight                                            (50000, 512)
encoder_pos.weight                                                (1024, 512)
decoder_pos.weight                                                (1024, 512)
encoder_layers.0.norm1.weight                                     (512,)
encoder_layers.0.norm1.bias                                       (512,)
encoder_layers.0.attn.q_proj.weight                               (512, 512)
encoder_layers.0.attn.q_proj.bias                                 (512,)
encoder_layers.0.attn.k_proj.weight                               (512, 512)
encoder_layers.0.attn.k_proj.bias                                 (512,)
encoder_layers.0.attn.v_proj.weight                               (512, 512)
encoder_layers.0.attn.v_proj.bias                

In [12]:
import re

print("VOCAB SIZE:", state["token_embedding.weight"].shape[0])
print("HIDDEN SIZE:", state["token_embedding.weight"].shape[1])

print("ENCODER CONTEXT:", state["encoder_pos.weight"].shape[0])
print("DECODER CONTEXT:", state["decoder_pos.weight"].shape[0])

encoder_layers = set()
decoder_layers = set()

for key in state:
    match = re.match(r"encoder_layers\.(\d+)\.", key)
    if match:
        encoder_layers.add(int(match.group(1)))

    match = re.match(r"decoder_layers\.(\d+)\.", key)
    if match:
        decoder_layers.add(int(match.group(1)))

print("ENCODER LAYERS:", len(encoder_layers))
print("DECODER LAYERS:", len(decoder_layers))

VOCAB SIZE: 50000
HIDDEN SIZE: 512
ENCODER CONTEXT: 1024
DECODER CONTEXT: 1024
ENCODER LAYERS: 6
DECODER LAYERS: 6


In [13]:
print("A4 ENCODER LAYER KEYS")
for key in state:
    if key.startswith("encoder_layers.0."):
        print(key, tuple(state[key].shape))

print("\nA4 DECODER LAYER KEYS")
for key in state:
    if key.startswith("decoder_layers.0."):
        print(key, tuple(state[key].shape))

print("\nOUTPUT HEAD")
for key in state:
    if key.startswith("lm_head."):
        print(key, tuple(state[key].shape))

A4 ENCODER LAYER KEYS
encoder_layers.0.norm1.weight (512,)
encoder_layers.0.norm1.bias (512,)
encoder_layers.0.attn.q_proj.weight (512, 512)
encoder_layers.0.attn.q_proj.bias (512,)
encoder_layers.0.attn.k_proj.weight (512, 512)
encoder_layers.0.attn.k_proj.bias (512,)
encoder_layers.0.attn.v_proj.weight (512, 512)
encoder_layers.0.attn.v_proj.bias (512,)
encoder_layers.0.attn.out_proj.weight (512, 512)
encoder_layers.0.attn.out_proj.bias (512,)
encoder_layers.0.norm2.weight (512,)
encoder_layers.0.norm2.bias (512,)
encoder_layers.0.ffn.net.0.weight (2048, 512)
encoder_layers.0.ffn.net.0.bias (2048,)
encoder_layers.0.ffn.net.2.weight (512, 2048)
encoder_layers.0.ffn.net.2.bias (512,)

A4 DECODER LAYER KEYS
decoder_layers.0.norm1.weight (512,)
decoder_layers.0.norm1.bias (512,)
decoder_layers.0.self_attn.q_proj.weight (512, 512)
decoder_layers.0.self_attn.q_proj.bias (512,)
decoder_layers.0.self_attn.k_proj.weight (512, 512)
decoder_layers.0.self_attn.k_proj.bias (512,)
decoder_layers.0

In [15]:
# ============================================================
# PHASE 6 - A4 MODEL CHECKPOINT COMPATIBILITY
# CS 781 LEGAL SUMMARIZATION
# ============================================================

import os
import re
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-22000-pt/a4_step_22000.pt"
)

TOKENIZER_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "tokenizer-json/tokenizer.json"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_HEADS = 8
FFN_DIM = 2048
DROPOUT = 0.0

print("Device:", DEVICE)
print("Checkpoint exists:", os.path.exists(CHECKPOINT_PATH))
print("Tokenizer exists:", os.path.exists(TOKENIZER_PATH))

assert os.path.exists(CHECKPOINT_PATH), "Checkpoint not found!"
assert os.path.exists(TOKENIZER_PATH), "Tokenizer not found!"

# ------------------------------------------------------------
# 2. LOAD CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

state = checkpoint["model_state_dict"]

print("\nCheckpoint metadata:")
for key in ["step", "tokens_processed", "train_loss", "saved_at"]:
    if key in checkpoint:
        print(f"{key}: {checkpoint[key]}")

# ------------------------------------------------------------
# 3. EXTRACT MODEL DIMENSIONS
# ------------------------------------------------------------

vocab_size, hidden_size = state["token_embedding.weight"].shape

# IMPORTANT:
# The actual checkpoint keys contain ".weight".
encoder_context = state["encoder_pos.weight"].shape[0]
decoder_context = state["decoder_pos.weight"].shape[0]

encoder_layers = len(set(
    int(re.search(r"encoder_layers\.(\d+)\.", key).group(1))
    for key in state
    if re.match(r"encoder_layers\.\d+\.", key)
))

decoder_layers = len(set(
    int(re.search(r"decoder_layers\.(\d+)\.", key).group(1))
    for key in state
    if re.match(r"decoder_layers\.\d+\.", key)
))

print("\nModel configuration:")
print("Vocabulary size:", vocab_size)
print("Hidden size:", hidden_size)
print("Encoder context:", encoder_context)
print("Decoder context:", decoder_context)
print("Encoder layers:", encoder_layers)
print("Decoder layers:", decoder_layers)
print("Attention heads:", NUM_HEADS)
print("FFN dimension:", FFN_DIM)

assert hidden_size % NUM_HEADS == 0
assert state["lm_head.weight"].shape == (vocab_size, hidden_size)

# ------------------------------------------------------------
# 4. ATTENTION
# ------------------------------------------------------------

class A4Attention(nn.Module):

    def __init__(self, hidden_size, num_heads):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, query, key, value, mask=None):

        batch_size = query.size(0)

        Q = self.q_proj(query)
        K = self.k_proj(key)
        V = self.v_proj(value)

        Q = Q.view(
            batch_size, -1, self.num_heads, self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size, -1, self.num_heads, self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size, -1, self.num_heads, self.head_dim
        ).transpose(1, 2)

        scores = torch.matmul(
            Q, K.transpose(-2, -1)
        ) / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        attention = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)

        output = output.transpose(1, 2).contiguous()

        output = output.view(
            batch_size, -1, self.hidden_size
        )

        return self.out_proj(output)


# ------------------------------------------------------------
# 5. FEED-FORWARD NETWORK
# ------------------------------------------------------------

class A4FFNCheckpoint(nn.Module):

    def __init__(self, hidden_size, ffn_dim):
        super().__init__()

        # Keep the .net naming to match checkpoint keys:
        # ffn.net.0.weight
        # ffn.net.2.weight

        self.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, hidden_size)
        )

    def forward(self, x):
        return self.net(x)


# ------------------------------------------------------------
# 6. ENCODER LAYER
# ------------------------------------------------------------

class A4EncoderLayer(nn.Module):

    def __init__(self, hidden_size, num_heads, ffn_dim):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = A4Attention(hidden_size, num_heads)

        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn = A4FFNCheckpoint(hidden_size, ffn_dim)

    def forward(self, x, mask=None):

        h = self.norm1(x)
        x = x + self.attn(h, h, h, mask)

        x = x + self.ffn(self.norm2(x))

        return x


# ------------------------------------------------------------
# 7. DECODER LAYER
# ------------------------------------------------------------

class A4DecoderLayer(nn.Module):

    def __init__(self, hidden_size, num_heads, ffn_dim):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)
        self.self_attn = A4Attention(hidden_size, num_heads)

        self.norm2 = nn.LayerNorm(hidden_size)
        self.cross_attn = A4Attention(hidden_size, num_heads)

        self.norm3 = nn.LayerNorm(hidden_size)
        self.ffn = A4FFNCheckpoint(hidden_size, ffn_dim)

    def forward(
        self,
        x,
        memory,
        self_mask=None,
        cross_mask=None
    ):

        h = self.norm1(x)
        x = x + self.self_attn(h, h, h, self_mask)

        h = self.norm2(x)
        x = x + self.cross_attn(
            h, memory, memory, cross_mask
        )

        x = x + self.ffn(self.norm3(x))

        return x


# ------------------------------------------------------------
# 8. COMPLETE A4 MODEL
# ------------------------------------------------------------

class A4Model(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_size,
        encoder_context,
        decoder_context,
        encoder_layers,
        decoder_layers,
        num_heads,
        ffn_dim
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.encoder_context = encoder_context
        self.decoder_context = decoder_context

        self.token_embedding = nn.Embedding(
            vocab_size, hidden_size
        )

        # IMPORTANT: these must be Embedding modules.
        # Their state-dict keys must end in ".weight".

        self.encoder_pos = nn.Embedding(
            encoder_context, hidden_size
        )

        self.decoder_pos = nn.Embedding(
            decoder_context, hidden_size
        )

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(hidden_size, num_heads, ffn_dim)
            for _ in range(encoder_layers)
        ])

        self.encoder_norm = nn.LayerNorm(hidden_size)

        self.decoder_layers = nn.ModuleList([
            A4DecoderLayer(hidden_size, num_heads, ffn_dim)
            for _ in range(decoder_layers)
        ])

        self.decoder_norm = nn.LayerNorm(hidden_size)

        self.lm_head = nn.Linear(
            hidden_size, vocab_size, bias=False
        )

    def forward(
        self,
        encoder_ids,
        decoder_ids,
        encoder_mask=None,
        decoder_mask=None
    ):

        batch_size, src_len = encoder_ids.shape
        _, tgt_len = decoder_ids.shape

        if src_len > self.encoder_context:
            raise ValueError("Encoder sequence exceeds context length.")

        if tgt_len > self.decoder_context:
            raise ValueError("Decoder sequence exceeds context length.")

        # Encoder embeddings
        src_positions = torch.arange(
            src_len, device=encoder_ids.device
        )

        encoder_output = (
            self.token_embedding(encoder_ids)
            + self.encoder_pos(src_positions).unsqueeze(0)
        )

        for layer in self.encoder_layers:
            encoder_output = layer(
                encoder_output,
                encoder_mask
            )

        memory = self.encoder_norm(encoder_output)

        # Decoder embeddings
        tgt_positions = torch.arange(
            tgt_len, device=decoder_ids.device
        )

        decoder_output = (
            self.token_embedding(decoder_ids)
            + self.decoder_pos(tgt_positions).unsqueeze(0)
        )

        # Causal mask: decoder cannot see future tokens.
        causal_mask = torch.triu(
            torch.ones(
                tgt_len,
                tgt_len,
                dtype=torch.bool,
                device=decoder_ids.device
            ),
            diagonal=1
        )[None, None, :, :]

        if decoder_mask is not None:
            causal_mask = causal_mask | decoder_mask

        for layer in self.decoder_layers:
            decoder_output = layer(
                decoder_output,
                memory,
                self_mask=causal_mask,
                cross_mask=encoder_mask
            )

        decoder_output = self.decoder_norm(decoder_output)

        logits = self.lm_head(decoder_output)

        return logits


# ------------------------------------------------------------
# 9. INITIALIZE MODEL
# ------------------------------------------------------------

model = A4Model(
    vocab_size=vocab_size,
    hidden_size=hidden_size,
    encoder_context=encoder_context,
    decoder_context=decoder_context,
    encoder_layers=encoder_layers,
    decoder_layers=decoder_layers,
    num_heads=NUM_HEADS,
    ffn_dim=FFN_DIM
)

# ------------------------------------------------------------
# 10. STRICT CHECKPOINT LOADING
# ------------------------------------------------------------

print("\nLoading A4 checkpoint...")

load_result = model.load_state_dict(
    state,
    strict=True
)

print("Checkpoint loaded successfully!")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

assert len(load_result.missing_keys) == 0
assert len(load_result.unexpected_keys) == 0

model = model.to(DEVICE)
model.eval()

# ------------------------------------------------------------
# 11. PARAMETER COUNT
# ------------------------------------------------------------

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("\nTotal parameters:", f"{total_parameters:,}")

# ------------------------------------------------------------
# 12. FORWARD PASS SMOKE TEST
# ------------------------------------------------------------

print("\nRunning forward-pass smoke test...")

BATCH_SIZE = 1
SRC_LEN = 16
TGT_LEN = 8

dummy_encoder_ids = torch.randint(
    low=0,
    high=vocab_size,
    size=(BATCH_SIZE, SRC_LEN),
    device=DEVICE
)

dummy_decoder_ids = torch.randint(
    low=0,
    high=vocab_size,
    size=(BATCH_SIZE, TGT_LEN),
    device=DEVICE
)

with torch.no_grad():

    dummy_logits = model(
        dummy_encoder_ids,
        dummy_decoder_ids
    )

print("Encoder input shape:", tuple(dummy_encoder_ids.shape))
print("Decoder input shape:", tuple(dummy_decoder_ids.shape))
print("Output logits shape:", tuple(dummy_logits.shape))

assert dummy_logits.shape == (
    BATCH_SIZE,
    TGT_LEN,
    vocab_size
)

assert torch.isfinite(dummy_logits).all()

print("\n" + "=" * 60)
print("A4 CHECKPOINT COMPATIBILITY TEST COMPLETED")
print("=" * 60)
print("Strict loading: PASSED")
print("Forward pass: PASSED")
print("Output shape: VERIFIED")
print("Device:", DEVICE)

Device: cuda
Checkpoint exists: True
Tokenizer exists: True

Checkpoint metadata:
step: 22000
tokens_processed: 180096704
train_loss: 0.1944424146786332
saved_at: 2026-10-05T15:18:59.981007

Model configuration:
Vocabulary size: 50000
Hidden size: 512
Encoder context: 1024
Decoder context: 1024
Encoder layers: 6
Decoder layers: 6
Attention heads: 8
FFN dimension: 2048

Loading A4 checkpoint...
Checkpoint loaded successfully!
Missing keys: []
Unexpected keys: []

Total parameters: 96,389,120

Running forward-pass smoke test...
Encoder input shape: (1, 16)
Decoder input shape: (1, 8)
Output logits shape: (1, 8, 50000)

A4 CHECKPOINT COMPATIBILITY TEST COMPLETED
Strict loading: PASSED
Forward pass: PASSED
Output shape: VERIFIED
Device: cuda


In [16]:
# ============================================================
# PHASE 6 - TOKENIZER VERIFICATION
# ============================================================

from tokenizers import Tokenizer
import os

TOKENIZER_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "tokenizer-json/tokenizer.json"
)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer loaded successfully")
print("Vocabulary size:", tokenizer.get_vocab_size())

# Check common special tokens
special_tokens = [
    "[PAD]", "[BOS]", "[EOS]", "[UNK]",
    "<pad>", "<bos>", "<eos>", "<unk>",
    "<|pad|>", "<|bos|>", "<|eos|>"
]

print("\nSpecial token verification:")

for token in special_tokens:
    token_id = tokenizer.token_to_id(token)
    if token_id is not None:
        print(f"{token}: {token_id}")

# Test encoding
sample_text = (
    "The court hereby orders the defendant "
    "to submit the required documents."
)

encoded = tokenizer.encode(sample_text)

print("\nEncoding test:")
print("Input:", sample_text)
print("Token IDs:", encoded.ids)
print("Number of tokens:", len(encoded.ids))
print("Decoded:", tokenizer.decode(encoded.ids))

Tokenizer loaded successfully
Vocabulary size: 50000

Special token verification:
[PAD]: 0
[BOS]: 2
[EOS]: 3
[UNK]: 1

Encoding test:
Input: The court hereby orders the defendant to submit the required documents.
Token IDs: [550, 777, 3081, 1283, 238, 2297, 271, 1199, 238, 1396, 1698, 18]
Number of tokens: 12
Decoded: The Ġcourt Ġhereby Ġorders Ġthe Ġdefendant Ġto Ġsubmit Ġthe Ġrequired Ġdocuments .


In [17]:
# ============================================================
# PHASE 6 - TOKENIZER CONFIGURATION INSPECTION
# ============================================================

import json

with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
    tokenizer_config = json.load(f)

print("TOKENIZER COMPONENTS")
print("=" * 60)

for component in [
    "model",
    "normalizer",
    "pre_tokenizer",
    "post_processor",
    "decoder"
]:
    print(f"\n{component.upper()}:")
    print(tokenizer_config.get(component))

print("\n" + "=" * 60)
print("ADDED TOKENS")
print("=" * 60)

for token in tokenizer_config.get("added_tokens", []):
    print(token)

print("\n" + "=" * 60)
print("SPECIAL TOKEN BEHAVIOR")
print("=" * 60)

sample = "[BOS] The court hereby orders the defendant. [EOS]"

for add_special_tokens in [False, True]:
    result = tokenizer.encode(
        sample,
        add_special_tokens=add_special_tokens
    )

    print("\nadd_special_tokens:", add_special_tokens)
    print("IDs:", result.ids)
    print("Decoded:", tokenizer.decode(result.ids))

print("\n" + "=" * 60)
print("ROUND-TRIP TEST")
print("=" * 60)

test_sentences = [
    "The court issued an order.",
    "The defendant shall submit the documents.",
    "This agreement shall remain effective."
]

for sentence in test_sentences:
    ids = tokenizer.encode(sentence).ids
    decoded = tokenizer.decode(ids)

    print("\nOriginal:", sentence)
    print("Decoded:", decoded)
    print("Exact match:", sentence == decoded)

TOKENIZER COMPONENTS

MODEL:
{'type': 'BPE', 'dropout': None, 'unk_token': '[UNK]', 'continuing_subword_prefix': None, 'end_of_word_suffix': None, 'fuse_unk': False, 'byte_fallback': False, 'ignore_merges': False, 'vocab': {'[PAD]': 0, '[UNK]': 1, '[BOS]': 2, '[EOS]': 3, '[MASK]': 4, '!': 5, '"': 6, '#': 7, '$': 8, '%': 9, '&': 10, "'": 11, '(': 12, ')': 13, '*': 14, '+': 15, ',': 16, '-': 17, '.': 18, '/': 19, '0': 20, '1': 21, '2': 22, '3': 23, '4': 24, '5': 25, '6': 26, '7': 27, '8': 28, '9': 29, ':': 30, ';': 31, '<': 32, '=': 33, '>': 34, '?': 35, '@': 36, 'A': 37, 'B': 38, 'C': 39, 'D': 40, 'E': 41, 'F': 42, 'G': 43, 'H': 44, 'I': 45, 'J': 46, 'K': 47, 'L': 48, 'M': 49, 'N': 50, 'O': 51, 'P': 52, 'Q': 53, 'R': 54, 'S': 55, 'T': 56, 'U': 57, 'V': 58, 'W': 59, 'X': 60, 'Y': 61, 'Z': 62, '[': 63, '\\': 64, ']': 65, '^': 66, '_': 67, '`': 68, 'a': 69, 'b': 70, 'c': 71, 'd': 72, 'e': 73, 'f': 74, 'g': 75, 'h': 76, 'i': 77, 'j': 78, 'k': 79, 'l': 80, 'm': 81, 'n': 82, 'o': 83, 'p': 84,

In [18]:
# ============================================================
# PHASE 6 - BYTELEVEL TOKENIZER DECODER FIX
# ============================================================

from tokenizers import Tokenizer, decoders

TOKENIZER_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "tokenizer-json/tokenizer.json"
)

# Load original tokenizer
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

# Configure the matching ByteLevel decoder
tokenizer.decoder = decoders.ByteLevel()

print("Decoder:", tokenizer.decoder)

# Verify round-trip behavior
test_sentences = [
    "The court issued an order.",
    "The defendant shall submit the documents.",
    "This agreement shall remain effective."
]

print("\nROUND-TRIP VERIFICATION")
print("=" * 60)

for sentence in test_sentences:

    ids = tokenizer.encode(sentence).ids
    decoded = tokenizer.decode(ids)

    print("\nOriginal:", sentence)
    print("Decoded: ", decoded)
    print("Exact match:", sentence == decoded)

# Verify special tokens
print("\nSPECIAL TOKENS")
for token in ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]:
    print(token, ":", tokenizer.token_to_id(token))

Decoder: ByteLevel(add_prefix_space=True, trim_offsets=True, use_regex=True)

ROUND-TRIP VERIFICATION

Original: The court issued an order.
Decoded:  The court issued an order.
Exact match: True

Original: The defendant shall submit the documents.
Decoded:  The defendant shall submit the documents.
Exact match: True

Original: This agreement shall remain effective.
Decoded:  This agreement shall remain effective.
Exact match: True

SPECIAL TOKENS
[PAD] : 0
[UNK] : 1
[BOS] : 2
[EOS] : 3
[MASK] : 4


In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
from tokenizers import Tokenizer, decoders

# ============================================================
# 1. PATHS AND CONFIGURATION
# ============================================================

BASE_PATH = "/kaggle/input/datasets/aq1de3"

CHECKPOINT_PATH = (
    BASE_PATH
    + "/a4-step-22000-pt/a4_step_22000.pt"
)

TOKENIZER_PATH = (
    BASE_PATH
    + "/tokenizer-json/tokenizer.json"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = 50000
D_MODEL = 512
NUM_HEADS = 8
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
FFN_DIM = 2048
MAX_POSITION = 1024

PAD_ID = 0
BOS_ID = 2
EOS_ID = 3

print("Device:", DEVICE)

# ============================================================
# 2. LOAD TOKENIZER
# ============================================================

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

# Restore the verified decoder in this Kaggle session
tokenizer.decoder = decoders.ByteLevel()

print("Tokenizer vocabulary:", tokenizer.get_vocab_size())
print("Special tokens:", PAD_ID, BOS_ID, EOS_ID)

# ============================================================
# 3. A4 ATTENTION
# ============================================================

class A4Attention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):

        B, Tq, C = query.shape
        Tk = key.shape[1]

        q = self.q_proj(query)
        k = self.k_proj(key)
        v = self.v_proj(value)

        q = q.view(B, Tq, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, Tk, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, Tk, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        attention = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention, v)

        output = output.transpose(1, 2).contiguous()
        output = output.view(B, Tq, C)

        return self.out_proj(output)


# ============================================================
# 4. FFN WRAPPER
# Important: preserves checkpoint keys ffn.net.0 and ffn.net.2
# ============================================================

class A4FFNCheckpoint(nn.Module):

    def __init__(self, d_model, ffn_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model)
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# 5. ENCODER LAYER
# ============================================================

class A4EncoderLayer(nn.Module):

    def __init__(self):
        super().__init__()

        self.norm1 = nn.LayerNorm(D_MODEL)
        self.attn = A4Attention(D_MODEL, NUM_HEADS)

        self.norm2 = nn.LayerNorm(D_MODEL)
        self.ffn = A4FFNCheckpoint(D_MODEL, FFN_DIM)

    def forward(self, x, mask=None):

        h = self.norm1(x)
        x = x + self.attn(h, h, h, mask)

        x = x + self.ffn(self.norm2(x))

        return x


# ============================================================
# 6. DECODER LAYER
# ============================================================

class A4DecoderLayer(nn.Module):

    def __init__(self):
        super().__init__()

        self.norm1 = nn.LayerNorm(D_MODEL)
        self.self_attn = A4Attention(D_MODEL, NUM_HEADS)

        self.norm2 = nn.LayerNorm(D_MODEL)
        self.cross_attn = A4Attention(D_MODEL, NUM_HEADS)

        self.norm3 = nn.LayerNorm(D_MODEL)
        self.ffn = A4FFNCheckpoint(D_MODEL, FFN_DIM)

    def forward(
        self,
        x,
        memory,
        self_mask=None,
        cross_mask=None
    ):

        h = self.norm1(x)
        x = x + self.self_attn(h, h, h, self_mask)

        h = self.norm2(x)
        x = x + self.cross_attn(h, memory, memory, cross_mask)

        x = x + self.ffn(self.norm3(x))

        return x


# ============================================================
# 7. A4 MODEL
# ============================================================

class A4Model(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)

        # Embeddings are used here because checkpoint keys are
        # encoder_pos.weight and decoder_pos.weight.
        self.encoder_pos = nn.Embedding(MAX_POSITION, D_MODEL)
        self.decoder_pos = nn.Embedding(MAX_POSITION, D_MODEL)

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer()
            for _ in range(NUM_ENCODER_LAYERS)
        ])

        self.encoder_norm = nn.LayerNorm(D_MODEL)

        self.decoder_layers = nn.ModuleList([
            A4DecoderLayer()
            for _ in range(NUM_DECODER_LAYERS)
        ])

        self.decoder_norm = nn.LayerNorm(D_MODEL)

        self.lm_head = nn.Linear(
            D_MODEL,
            VOCAB_SIZE,
            bias=False
        )

    def forward(self, encoder_ids, decoder_ids):

        B, Te = encoder_ids.shape
        Td = decoder_ids.shape[1]

        if Te > MAX_POSITION or Td > MAX_POSITION:
            raise ValueError("Sequence exceeds A4 positional capacity.")

        device = encoder_ids.device

        # Encoder
        enc_positions = torch.arange(Te, device=device)
        encoder = (
            self.token_embedding(encoder_ids)
            + self.encoder_pos(enc_positions)[None, :, :]
        )

        encoder_padding_mask = (
            encoder_ids == PAD_ID
        )[:, None, None, :]

        for layer in self.encoder_layers:
            encoder = layer(encoder, encoder_padding_mask)

        memory = self.encoder_norm(encoder)

        # Decoder
        dec_positions = torch.arange(Td, device=device)
        decoder = (
            self.token_embedding(decoder_ids)
            + self.decoder_pos(dec_positions)[None, :, :]
        )

        causal_mask = torch.triu(
            torch.ones(Td, Td, device=device, dtype=torch.bool),
            diagonal=1
        )[None, None, :, :]

        decoder_padding_mask = (
            decoder_ids == PAD_ID
        )[:, None, None, :]

        self_mask = causal_mask | decoder_padding_mask

        for layer in self.decoder_layers:
            decoder = layer(
                decoder,
                memory,
                self_mask,
                encoder_padding_mask
            )

        decoder = self.decoder_norm(decoder)

        return self.lm_head(decoder)


# ============================================================
# 8. LOAD CHECKPOINT STRICTLY
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

state_dict = checkpoint["model_state_dict"]

model = A4Model()

load_result = model.load_state_dict(
    state_dict,
    strict=True
)

model = model.to(DEVICE)
model.eval()

print("\nCHECKPOINT LOAD SUCCESSFUL")
print("Strict loading:", load_result)
print("Checkpoint step:", checkpoint["step"])
print("Training loss:", checkpoint["train_loss"])

# ============================================================
# 9. FORWARD PASS SMOKE TEST
# ============================================================

dummy_encoder = torch.tensor(
    [[BOS_ID, 100, 200, EOS_ID]],
    dtype=torch.long,
    device=DEVICE
)

dummy_decoder = torch.tensor(
    [[BOS_ID, 300, 400]],
    dtype=torch.long,
    device=DEVICE
)

with torch.no_grad():

    logits = model(
        dummy_encoder,
        dummy_decoder
    )

print("\nFORWARD PASS SUCCESSFUL")
print("Logits shape:", tuple(logits.shape))
print("Expected shape:", (1, 3, VOCAB_SIZE))
print("All logits finite:", torch.isfinite(logits).all().item())

print("\nA4 VERIFICATION COMPLETE")

Device: cuda
Tokenizer vocabulary: 50000
Special tokens: 0 2 3

CHECKPOINT LOAD SUCCESSFUL
Strict loading: <All keys matched successfully>
Checkpoint step: 22000
Training loss: 0.1944424146786332

FORWARD PASS SUCCESSFUL
Logits shape: (1, 3, 50000)
Expected shape: (1, 3, 50000)
All logits finite: True

A4 VERIFICATION COMPLETE


In [20]:
import json
import torch

# ============================================================
# 1. LOAD A REAL TRAINING DOCUMENT
# ============================================================

BASE_PATH = "/kaggle/input/datasets/aq1de3"
TRAIN_PATH = BASE_PATH + "/phase6/train (3).json"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

sample = next(
    item for item in train_data
    if item.get("document", "").strip()
)

document = sample["document"]

print("Sample ID:", sample["id"])
print("Original document characters:", len(document))

# ============================================================
# 2. TOKENIZE SOURCE DOCUMENT
# ============================================================

encoded = tokenizer.encode(document)

source_ids = encoded.ids[:512]

# Add BOS and EOS
source_ids = [BOS_ID] + source_ids + [EOS_ID]

encoder_input = torch.tensor(
    [source_ids],
    dtype=torch.long,
    device=DEVICE
)

print("Encoder input shape:", tuple(encoder_input.shape))
print("Source tokens:", len(source_ids))

# ============================================================
# 3. GREEDY AUTOREGRESSIVE GENERATION
# ============================================================

MAX_NEW_TOKENS = 64

generated_ids = [BOS_ID]

model.eval()

with torch.no_grad():

    for step in range(MAX_NEW_TOKENS):

        decoder_input = torch.tensor(
            [generated_ids],
            dtype=torch.long,
            device=DEVICE
        )

        logits = model(
            encoder_input,
            decoder_input
        )

        next_token_logits = logits[0, -1, :]

        # Greedy decoding
        next_token = torch.argmax(
            next_token_logits
        ).item()

        generated_ids.append(next_token)

        if next_token == EOS_ID:
            break

# ============================================================
# 4. DECODE OUTPUT
# ============================================================

output_ids = [
    token_id
    for token_id in generated_ids
    if token_id not in [BOS_ID, EOS_ID, PAD_ID]
]

generated_text = tokenizer.decode(
    output_ids,
    skip_special_tokens=True
)

print("\n" + "=" * 60)
print("GENERATED TEXT")
print("=" * 60)

print(generated_text)

print("\n" + "=" * 60)
print("GENERATION STATISTICS")
print("=" * 60)

print("Generated token count:", len(output_ids))
print("Stopped at EOS:", generated_ids[-1] == EOS_ID)
print("Output characters:", len(generated_text))
print("Output non-empty:", bool(generated_text.strip()))

Sample ID: 6712
Original document characters: 8091
Encoder input shape: (1, 514)
Source tokens: 514

GENERATED TEXT
 exercises Appeal No. marriage (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh Pradesh1975 in M.C.C. No.144 of 1966. V.A. Bobde, B.R.Aggar U and U.A.Rana for the Appellant

GENERATION STATISTICS
Generated token count: 64
Stopped at EOS: False
Output characters: 203
Output non-empty: True


In [21]:
import numpy as np

# Reuse train_data, tokenizer and sample from the previous cells

document = sample["document"]
reference_summary = sample["summary"]

doc_ids = tokenizer.encode(document).ids
summary_ids = tokenizer.encode(reference_summary).ids

print("=" * 65)
print("DOCUMENT–SUMMARY INSPECTION")
print("=" * 65)

print("Sample ID:", sample["id"])
print("Document characters:", len(document))
print("Summary characters:", len(reference_summary))
print("Document tokens:", len(doc_ids))
print("Summary tokens:", len(summary_ids))

print("\nSummary / document token ratio:",
      round(len(summary_ids) / max(len(doc_ids), 1), 4))

print("\n" + "=" * 65)
print("REFERENCE SUMMARY")
print("=" * 65)

print(reference_summary[:3000])

print("\n" + "=" * 65)
print("DOCUMENT PREVIEW")
print("=" * 65)

print(document[:2500])

# Inspect chunks using non-overlapping 900-token windows.
# Reserve room for BOS and EOS within the 1024-token limit.

CHUNK_SIZE = 900
OVERLAP = 100

starts = list(range(
    0,
    len(doc_ids),
    CHUNK_SIZE - OVERLAP
))

chunks = [
    doc_ids[start:start + CHUNK_SIZE]
    for start in starts
]

print("\n" + "=" * 65)
print("CHUNKING INSPECTION")
print("=" * 65)

print("Number of chunks:", len(chunks))
print("Chunk token lengths:", [len(c) for c in chunks])
print("Maximum input length with BOS/EOS:",
      max(len(c) + 2 for c in chunks))

print("\nChunking respects A4 context:",
      all(len(c) + 2 <= 1024 for c in chunks))

DOCUMENT–SUMMARY INSPECTION
Sample ID: 6712
Document characters: 8091
Summary characters: 3039
Document tokens: 1544
Summary tokens: 565

Summary / document token ratio: 0.3659

REFERENCE SUMMARY
The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell its cement only through the marketing company, and certain sums would be paid for the cement supplied by the marketing company, which had the discretion to fix the sale price. For the 1950 51, 1951 52 and 1952 53 periods when the appellant was assessed to sales tax for the supply of cement, it maintained at the assessment stage that the transactions were not exigible to sales tax as they were covered under the Explanation to Article 286(1)(a). The first Appellate Authority and the Board of Revenue did not accept the stand of the appellant. The Board of Revenue held that cement being a controlled commodity, 

In [22]:
import re
import numpy as np

def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]

# Reuse the previously generated chunks and tokenizer
summary_sentences = split_sentences(reference_summary)

chunk_texts = [
    tokenizer.decode(chunk)
    for chunk in chunks
]

print("REFERENCE SUMMARY SENTENCES:", len(summary_sentences))

for i, chunk_text in enumerate(chunk_texts):

    print("\n" + "=" * 65)
    print(f"CHUNK {i + 1}")
    print("=" * 65)

    print("Tokens:", len(chunks[i]))
    print("Preview:")
    print(chunk_text[:1500])

print("\n" + "=" * 65)
print("REFERENCE SUMMARY SENTENCES")
print("=" * 65)

for i, sentence in enumerate(summary_sentences):
    print(f"{i + 1}. {sentence}")

REFERENCE SUMMARY SENTENCES: 16

CHUNK 1
Tokens: 900
Preview:
IVIL Appeal No.768 (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh High Court in M.C.C. No.144 of 1966. V.A. Bobde, B.R.Aggarwala and U.A.Rana for the Appellant. S.V.Deshpande and S.K.Agnihotri for the Respondents. The Judgment of the Court was delivered by RANGANATH MISHRA, CJ. These are appeals by special leave 252 and are directed against the separate decisions of the Madhya Pradesh High Court in references under the Madhya Pradesh Sales Tax Act. Civil Appeal No.768/77 relates to the assessment period 1951 52, Civil Appeal 539/78 relates to 1950 51 and Civil Appeal 1038/78 to 1952 53. The appellant is a manufacturer of cement in the factory located at Kymore in Madhya Pradesh. Several cement manufacturing companies as also the appellant had entered into arrangement with the Cement Manufacturing Company of India Limited whereunder the Marketing Company was appointed as the sole and exclusive

In [23]:
# Inspect the end of the original legal document
print("=" * 65)
print("DOCUMENT ENDING")
print("=" * 65)

print(document[-3500:])

# Inspect the final part of the second chunk
print("\n" + "=" * 65)
print("FINAL CHUNK ENDING")
print("=" * 65)

print(tokenizer.decode(chunks[-1][-350:]))

# Inspect reference summary ending
print("\n" + "=" * 65)
print("REFERENCE SUMMARY ENDING")
print("=" * 65)

print(reference_summary[-1200:])

DOCUMENT ENDING
f principal and agent. " This Court in Rohtas Industries case on a detailed analysis of the terms of the contract came to hold that there was a sale between the manufacturer and the Marketing Company. It is not in dispute that the agreement between the appellant and the Marketing Company in this case has the same terms as this Court considered in Rohtas Industries case. It follows, therefore, that it must be held that there was a sale between the appellant and the Marketing Company. The Marketing Company had its establishment at Nagpur within the State of Madhya Pradesh at that time. There was, therefore, a 254 preceding local sale prior to the sales between the Marketing Company and the allottee of cement by the regulating authority. This Court in Rohtas Industries further found that the transaction between the manufacturer and the Marketing Company had nothing to do with the Marketing Company 's sales to third parties. There was no privity between the manufacturer and

In [24]:
import json
import numpy as np
from collections import Counter

BASE_PATH = "/kaggle/input/datasets/aq1de3"
TRAIN_PATH = BASE_PATH + "/phase6/train (3).json"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

# Remove empty summaries
clean_data = [
    x for x in train_data
    if x["document"].strip() and x["summary"].strip()
]

CHUNK_SIZE = 900
OVERLAP = 100
STRIDE = CHUNK_SIZE - OVERLAP

document_lengths = []
summary_lengths = []
chunk_counts = []

for item in clean_data:

    doc_len = len(tokenizer.encode(item["document"]).ids)
    sum_len = len(tokenizer.encode(item["summary"]).ids)

    n_chunks = max(
        1,
        int(np.ceil(max(0, doc_len - CHUNK_SIZE) / STRIDE)) + 1
    )

    document_lengths.append(doc_len)
    summary_lengths.append(sum_len)
    chunk_counts.append(n_chunks)

print("CLEAN TRAIN RECORDS:", len(clean_data))

print("\nDOCUMENT TOKEN STATISTICS")
for p in [50, 75, 90, 95, 99, 100]:
    print(f"P{p}: {np.percentile(document_lengths, p):.0f}")

print("\nSUMMARY TOKEN STATISTICS")
for p in [50, 75, 90, 95, 99, 100]:
    print(f"P{p}: {np.percentile(summary_lengths, p):.0f}")

print("\nCHUNK STATISTICS")
print("Total chunks:", sum(chunk_counts))
print("Average chunks/document:", round(np.mean(chunk_counts), 2))
print("Maximum chunks/document:", max(chunk_counts))

print("\nCHUNKS PER DOCUMENT")
print(dict(sorted(Counter(chunk_counts).items())))

print("\nDOCUMENTS WITH LONG SUMMARIES")
print("Above 1022 tokens:", sum(x > 1022 for x in summary_lengths))
print("Above 900 tokens:", sum(x > 900 for x in summary_lengths))

CLEAN TRAIN RECORDS: 7028

DOCUMENT TOKEN STATISTICS
P50: 3642
P75: 5750
P90: 9063
P95: 13080
P99: 28055
P100: 161283

SUMMARY TOKEN STATISTICS
P50: 746
P75: 1135
P90: 1763
P95: 2384
P99: 5048
P100: 32183

CHUNK STATISTICS
Total chunks: 47114
Average chunks/document: 6.7
Maximum chunks/document: 202

CHUNKS PER DOCUMENT
{1: 166, 2: 704, 3: 1124, 4: 1124, 5: 915, 6: 669, 7: 542, 8: 381, 9: 290, 10: 197, 11: 173, 12: 133, 13: 85, 14: 68, 15: 52, 16: 44, 17: 39, 18: 30, 19: 34, 20: 25, 21: 20, 22: 24, 23: 15, 24: 14, 25: 15, 26: 13, 27: 8, 28: 8, 29: 5, 30: 5, 31: 6, 32: 6, 33: 6, 34: 9, 35: 10, 36: 2, 37: 2, 38: 8, 39: 3, 40: 2, 41: 1, 42: 3, 43: 1, 44: 4, 45: 4, 46: 5, 47: 1, 48: 2, 49: 2, 52: 4, 53: 1, 54: 1, 57: 1, 59: 1, 60: 1, 61: 1, 62: 1, 65: 2, 66: 2, 67: 1, 69: 1, 71: 1, 73: 1, 77: 1, 80: 1, 85: 2, 100: 1, 119: 1, 121: 1, 123: 1, 131: 1, 202: 1}

DOCUMENTS WITH LONG SUMMARIES
Above 1022 tokens: 2119
Above 900 tokens: 2652


In [25]:
import numpy as np

# Reuse clean_data and tokenizer

records = []

for item in clean_data:
    doc_len = len(tokenizer.encode(item["document"]).ids)
    sum_len = len(tokenizer.encode(item["summary"]).ids)

    records.append({
        "id": item["id"],
        "document_tokens": doc_len,
        "summary_tokens": sum_len,
        "ratio": sum_len / max(doc_len, 1)
    })

# Longest documents
print("=" * 70)
print("TOP 10 LONGEST DOCUMENTS")
print("=" * 70)

for r in sorted(
    records,
    key=lambda x: x["document_tokens"],
    reverse=True
)[:10]:
    print(r)

# Longest summaries
print("\n" + "=" * 70)
print("TOP 10 LONGEST SUMMARIES")
print("=" * 70)

for r in sorted(
    records,
    key=lambda x: x["summary_tokens"],
    reverse=True
)[:10]:
    print(r)

# Highest summary/document ratios
print("\n" + "=" * 70)
print("TOP 10 SUMMARY/DOCUMENT RATIOS")
print("=" * 70)

for r in sorted(
    records,
    key=lambda x: x["ratio"],
    reverse=True
)[:10]:
    print(r)

# Inspect the most extreme document-summary pair
extreme = max(records, key=lambda x: x["summary_tokens"])

item = next(
    x for x in clean_data
    if x["id"] == extreme["id"]
)

print("\n" + "=" * 70)
print("EXTREME SUMMARY SAMPLE")
print("=" * 70)

print("ID:", item["id"])
print("Document preview:\n", item["document"][:1500])
print("\nSummary preview:\n", item["summary"][:2500])

TOP 10 LONGEST DOCUMENTS
{'id': '3498', 'document_tokens': 161283, 'summary_tokens': 32183, 'ratio': 0.1995436592821314}
{'id': '34', 'document_tokens': 104544, 'summary_tokens': 2612, 'ratio': 0.024984695439240895}
{'id': '67', 'document_tokens': 97704, 'summary_tokens': 3001, 'ratio': 0.030715221485302546}
{'id': '4656', 'document_tokens': 96887, 'summary_tokens': 10425, 'ratio': 0.10759957476235202}
{'id': '2189', 'document_tokens': 95013, 'summary_tokens': 10597, 'ratio': 0.11153210613284498}
{'id': '6581', 'document_tokens': 79892, 'summary_tokens': 16967, 'ratio': 0.21237420517698893}
{'id': '568', 'document_tokens': 67864, 'summary_tokens': 2094, 'ratio': 0.030855829305670164}
{'id': '5712', 'document_tokens': 67549, 'summary_tokens': 15740, 'ratio': 0.23301603280581504}
{'id': '1861', 'document_tokens': 63649, 'summary_tokens': 7324, 'ratio': 0.11506857923926535}
{'id': '6310', 'document_tokens': 61108, 'summary_tokens': 11644, 'ratio': 0.19054788243765136}

TOP 10 LONGEST SUMM

In [26]:
# Inspect selected outliers using the original dataset

INSPECT_IDS = ["5493", "5676", "3498"]

for target_id in INSPECT_IDS:

    item = next(
        (x for x in clean_data if str(x["id"]) == target_id),
        None
    )

    if item is None:
        print("Missing ID:", target_id)
        continue

    document = item["document"]
    summary = item["summary"]

    doc_tokens = len(tokenizer.encode(document).ids)
    sum_tokens = len(tokenizer.encode(summary).ids)

    print("\n" + "=" * 80)
    print("RECORD ID:", target_id)
    print("=" * 80)

    print("Document tokens:", doc_tokens)
    print("Summary tokens:", sum_tokens)
    print("Summary/document ratio:", round(sum_tokens / max(doc_tokens, 1), 3))

    print("\n--- DOCUMENT BEGINNING ---")
    print(document[:2500])

    print("\n--- DOCUMENT ENDING ---")
    print(document[-1500:])

    print("\n--- SUMMARY BEGINNING ---")
    print(summary[:3000])

    print("\n--- SUMMARY ENDING ---")
    print(summary[-1500:])


RECORD ID: 5493
Document tokens: 199
Summary tokens: 1000
Summary/document ratio: 5.025

--- DOCUMENT BEGINNING ---
Special Leave Petition No. 1900 of 1981 etc. From the Judgment and order dated 3.2.1981 of the Allahabad High Court in C.M.W.P. No. 1924 of 1981. P.P. Rao, Ambrish Kumar, Mrs. Rani Chhabra, M. Qamaruddin, Mrs. Qamaruddin, A.K. Srivastava, B.B. Tawakley, Mrs. Subhadra, S.N. Singh, C.K. Ratnaparkhi, S.K. Gupta, Uma Dutt, C.P. Lal, M.K. Garg, and Lokesh Kumar for the Petitioners. Anil Dev Singh, O.P. Rana, B.P. Maheshwari, Mrs. section Dikshit, P.K. Pillai, R. Ramachandran, A.K. Srivastava, S.C. Birla, section Wasim, A. Qadri, N.N. Sharma, Shakeel Ahmad and K.K. Gupta for the Respondents.

--- DOCUMENT ENDING ---
Special Leave Petition No. 1900 of 1981 etc. From the Judgment and order dated 3.2.1981 of the Allahabad High Court in C.M.W.P. No. 1924 of 1981. P.P. Rao, Ambrish Kumar, Mrs. Rani Chhabra, M. Qamaruddin, Mrs. Qamaruddin, A.K. Srivastava, B.B. Tawakley, Mrs. Subhad

In [27]:
# Reuse clean_data and tokenizer

suspicious_records = []

for item in clean_data:

    doc_len = len(tokenizer.encode(item["document"]).ids)
    sum_len = len(tokenizer.encode(item["summary"]).ids)

    if doc_len < 700 and sum_len > 900:
        suspicious_records.append({
            "id": item["id"],
            "document_tokens": doc_len,
            "summary_tokens": sum_len,
            "ratio": round(sum_len / max(doc_len, 1), 3)
        })

print("SUSPICIOUS RECORDS:", len(suspicious_records))

for record in sorted(
    suspicious_records,
    key=lambda x: x["ratio"],
    reverse=True
)[:50]:
    print(record)

print("\nTotal records flagged:", len(suspicious_records))

SUSPICIOUS RECORDS: 2
{'id': '5493', 'document_tokens': 199, 'summary_tokens': 1000, 'ratio': 5.025}
{'id': '5676', 'document_tokens': 571, 'summary_tokens': 2071, 'ratio': 3.627}

Total records flagged: 2


In [28]:
import numpy as np

MAX_LENGTH = 1022
SUSPICIOUS_IDS = {"5493", "5676"}

eligible_records = []
long_document_only = 0
long_summary_only = 0
both_too_long = 0

for item in clean_data:

    if str(item["id"]) in SUSPICIOUS_IDS:
        continue

    doc_len = len(tokenizer.encode(item["document"]).ids)
    sum_len = len(tokenizer.encode(item["summary"]).ids)

    doc_fits = doc_len <= MAX_LENGTH
    summary_fits = sum_len <= MAX_LENGTH

    if doc_fits and summary_fits:
        eligible_records.append(item)

    elif not doc_fits and summary_fits:
        long_document_only += 1

    elif doc_fits and not summary_fits:
        long_summary_only += 1

    else:
        both_too_long += 1

print("=" * 60)
print("BASELINE ELIGIBILITY")
print("=" * 60)

print("Provisional records:", len(clean_data) - 2)
print("Both document and summary fit:", len(eligible_records))
print("Only document exceeds limit:", long_document_only)
print("Only summary exceeds limit:", long_summary_only)
print("Both exceed limit:", both_too_long)

print(
    "Direct baseline eligibility:",
    round(
        len(eligible_records) / (len(clean_data) - 2) * 100,
        2
    ),
    "%"
)

print("\nExample eligible IDs:")
print([x["id"] for x in eligible_records[:10]])

BASELINE ELIGIBILITY
Provisional records: 7026
Both document and summary fit: 226
Only document exceeds limit: 4682
Only summary exceeds limit: 0
Both exceed limit: 2118
Direct baseline eligibility: 3.22 %

Example eligible IDs:
['3453', '4161', '7112', '7089', '5307', '5255', '2855', '2546', '4031', '5462']


In [29]:
import numpy as np
from sklearn.model_selection import train_test_split

# eligible_records was created in the previous cell

indices = np.arange(len(eligible_records))

train_idx, dev_idx = train_test_split(
    indices,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

baseline_train = [
    eligible_records[i] for i in train_idx
]

baseline_dev = [
    eligible_records[i] for i in dev_idx
]

print("SHORT-DOCUMENT BASELINE SPLIT")
print("=" * 50)

print("Total:", len(eligible_records))
print("Training:", len(baseline_train))
print("Validation:", len(baseline_dev))

print("\nTRAIN EXAMPLE IDS:")
print([x["id"] for x in baseline_train[:10]])

print("\nVALIDATION EXAMPLE IDS:")
print([x["id"] for x in baseline_dev[:10]])

# Verify no overlap
train_ids = {str(x["id"]) for x in baseline_train}
dev_ids = {str(x["id"]) for x in baseline_dev}

print("\nNo train/dev overlap:", train_ids.isdisjoint(dev_ids))

# Verify positional limits
assert all(
    len(tokenizer.encode(x["document"]).ids) <= 1022
    and len(tokenizer.encode(x["summary"]).ids) <= 1022
    for x in baseline_train + baseline_dev
)

print("All examples fit A4 positional limits: True")

SHORT-DOCUMENT BASELINE SPLIT
Total: 226
Training: 203
Validation: 23

TRAIN EXAMPLE IDS:
['4709', '4329', '5664', '6102', '4380', '7055', '3246', '127', '4555', '7034']

VALIDATION EXAMPLE IDS:
['5462', '6244', '4381', '6916', '4037', '3721', '6358', '4834', '4187', '3852']

No train/dev overlap: True
All examples fit A4 positional limits: True


In [30]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. CONFIGURATION
# ============================================================

BATCH_SIZE = 1
MAX_LENGTH = 1024

# ============================================================
# 2. DATASET
# ============================================================

class LegalSummarizationDataset(Dataset):

    def __init__(self, records, tokenizer):
        self.records = records
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):

        item = self.records[idx]

        source_ids = self.tokenizer.encode(
            item["document"]
        ).ids

        target_ids = self.tokenizer.encode(
            item["summary"]
        ).ids

        # Encoder: BOS + source + EOS
        source_ids = (
            [BOS_ID] + source_ids + [EOS_ID]
        )

        # Decoder input: BOS + summary
        decoder_ids = [BOS_ID] + target_ids

        # Labels: summary + EOS
        labels = target_ids + [EOS_ID]

        assert len(source_ids) <= MAX_LENGTH
        assert len(decoder_ids) <= MAX_LENGTH
        assert len(labels) <= MAX_LENGTH

        return {
            "encoder_ids": source_ids,
            "decoder_ids": decoder_ids,
            "labels": labels
        }


# ============================================================
# 3. PADDING COLLATOR
# ============================================================

def collate_fn(batch):

    max_src = max(
        len(x["encoder_ids"]) for x in batch
    )

    max_tgt = max(
        len(x["decoder_ids"]) for x in batch
    )

    encoder_ids = []
    decoder_ids = []
    labels = []

    for item in batch:

        src = item["encoder_ids"]
        dec = item["decoder_ids"]
        tgt = item["labels"]

        encoder_ids.append(
            src + [PAD_ID] * (max_src - len(src))
        )

        decoder_ids.append(
            dec + [PAD_ID] * (max_tgt - len(dec))
        )

        labels.append(
            tgt + [-100] * (max_tgt - len(tgt))
        )

    return {
        "encoder_ids": torch.tensor(
            encoder_ids, dtype=torch.long
        ),
        "decoder_ids": torch.tensor(
            decoder_ids, dtype=torch.long
        ),
        "labels": torch.tensor(
            labels, dtype=torch.long
        )
    }


# ============================================================
# 4. CREATE DATASET AND DATALOADER
# ============================================================

train_dataset = LegalSummarizationDataset(
    baseline_train,
    tokenizer
)

dev_dataset = LegalSummarizationDataset(
    baseline_dev,
    tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

batch = next(iter(train_loader))

encoder_ids = batch["encoder_ids"].to(DEVICE)
decoder_ids = batch["decoder_ids"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

print("ENCODER SHAPE:", tuple(encoder_ids.shape))
print("DECODER SHAPE:", tuple(decoder_ids.shape))
print("LABEL SHAPE:", tuple(labels.shape))

print("Encoder tokens:", (encoder_ids != PAD_ID).sum().item())
print("Decoder tokens:", (decoder_ids != PAD_ID).sum().item())
print("Valid target tokens:", (labels != -100).sum().item())


# ============================================================
# 5. ONE-BATCH FORWARD AND LOSS TEST
# ============================================================

model.train()
model.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

try:

    logits = model(
        encoder_ids,
        decoder_ids
    )

    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        labels.reshape(-1),
        ignore_index=-100
    )

    print("\nFORWARD PASS SUCCESS")
    print("Logits:", tuple(logits.shape))
    print("Loss:", loss.item())
    print("Loss finite:", torch.isfinite(loss).item())

    loss.backward()

    print("\nBACKWARD PASS SUCCESS")

    grad_exists = any(
        p.grad is not None
        for p in model.parameters()
        if p.requires_grad
    )

    print("Gradients present:", grad_exists)

    if torch.cuda.is_available():
        peak_memory = torch.cuda.max_memory_allocated() / 1024**3
        print("Peak GPU memory (GB):", round(peak_memory, 3))

except torch.cuda.OutOfMemoryError:
    print("\nCUDA OUT OF MEMORY")
    model.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()

finally:
    model.zero_grad(set_to_none=True)
    model.eval()

ENCODER SHAPE: (1, 369)
DECODER SHAPE: (1, 162)
LABEL SHAPE: (1, 162)
Encoder tokens: 369
Decoder tokens: 162
Valid target tokens: 162

FORWARD PASS SUCCESS
Logits: (1, 162, 50000)
Loss: 25.24332618713379
Loss finite: True

BACKWARD PASS SUCCESS
Gradients present: True
Peak GPU memory (GB): 0.88


In [31]:
import torch
import torch.nn.functional as F

LEARNING_RATE = 2e-5

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

model.train()
optimizer.zero_grad(set_to_none=True)

# Forward
logits = model(encoder_ids, decoder_ids)

loss = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    labels.reshape(-1),
    ignore_index=-100
)

# Backward
loss.backward()

# Prevent unusually large gradient updates
torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

# Optimizer update
optimizer.step()

print("OPTIMIZER UPDATE SUCCESSFUL")
print("Learning rate:", LEARNING_RATE)
print("Loss before update:", round(loss.item(), 6))

# Verify parameters remain finite
all_finite = all(
    torch.isfinite(p).all().item()
    for p in model.parameters()
)

print("All model parameters finite:", all_finite)

model.eval()

OPTIMIZER UPDATE SUCCESSFUL
Learning rate: 2e-05
Loss before update: 25.243326
All model parameters finite: True


A4Model(
  (token_embedding): Embedding(50000, 512)
  (encoder_pos): Embedding(1024, 512)
  (decoder_pos): Embedding(1024, 512)
  (encoder_layers): ModuleList(
    (0-5): 6 x A4EncoderLayer(
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): A4Attention(
        (q_proj): Linear(in_features=512, out_features=512, bias=True)
        (k_proj): Linear(in_features=512, out_features=512, bias=True)
        (v_proj): Linear(in_features=512, out_features=512, bias=True)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
      )
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ffn): A4FFNCheckpoint(
        (net): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=2048, out_features=512, bias=True)
        )
      )
    )
  )
  (encoder_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (decoder_laye

In [32]:
# Verify the optimizer update and model parameters

print("OPTIMIZER:", type(optimizer).__name__)
print("LEARNING RATE:", optimizer.param_groups[0]["lr"])

print("MODEL PARAMETERS FINITE:",
      all(torch.isfinite(p).all().item()
          for p in model.parameters()))

print("OPTIMIZER STATE INITIALIZED:",
      len(optimizer.state) > 0)

model.eval()

OPTIMIZER: AdamW
LEARNING RATE: 2e-05
MODEL PARAMETERS FINITE: True
OPTIMIZER STATE INITIALIZED: True


A4Model(
  (token_embedding): Embedding(50000, 512)
  (encoder_pos): Embedding(1024, 512)
  (decoder_pos): Embedding(1024, 512)
  (encoder_layers): ModuleList(
    (0-5): 6 x A4EncoderLayer(
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): A4Attention(
        (q_proj): Linear(in_features=512, out_features=512, bias=True)
        (k_proj): Linear(in_features=512, out_features=512, bias=True)
        (v_proj): Linear(in_features=512, out_features=512, bias=True)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
      )
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ffn): A4FFNCheckpoint(
        (net): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=2048, out_features=512, bias=True)
        )
      )
    )
  )
  (encoder_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (decoder_laye

In [33]:
import torch
import torch.nn.functional as F

# Use the existing batch
model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

# Save a parameter before the update
param_before = model.lm_head.weight.detach().clone()

optimizer.zero_grad(set_to_none=True)

# Forward pass
logits = model(encoder_ids, decoder_ids)

# Calculate loss
loss = F.cross_entropy(
    logits.reshape(-1, logits.size(-1)),
    labels.reshape(-1),
    ignore_index=-100
)

# Backward pass
loss.backward()

# Gradient clipping
grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(), max_norm=1.0
)

# Optimizer update
optimizer.step()

# Measure actual parameter change
param_change = (
    model.lm_head.weight.detach() - param_before
).abs().max().item()

print("LOSS:", loss.item())
print("GRADIENT NORM:", grad_norm.item())
print("MAX PARAMETER CHANGE:", param_change)
print("OPTIMIZER STATE SIZE:", len(optimizer.state))
print(
    "PARAMETERS FINITE:",
    all(torch.isfinite(p).all().item() for p in model.parameters())
)

model.eval()

LOSS: 20.043529510498047
GRADIENT NORM: 61.84125518798828
MAX PARAMETER CHANGE: 2.0742416381835938e-05
OPTIMIZER STATE SIZE: 260
PARAMETERS FINITE: True


A4Model(
  (token_embedding): Embedding(50000, 512)
  (encoder_pos): Embedding(1024, 512)
  (decoder_pos): Embedding(1024, 512)
  (encoder_layers): ModuleList(
    (0-5): 6 x A4EncoderLayer(
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): A4Attention(
        (q_proj): Linear(in_features=512, out_features=512, bias=True)
        (k_proj): Linear(in_features=512, out_features=512, bias=True)
        (v_proj): Linear(in_features=512, out_features=512, bias=True)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
      )
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ffn): A4FFNCheckpoint(
        (net): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=2048, out_features=512, bias=True)
        )
      )
    )
  )
  (encoder_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (decoder_laye

In [34]:
import torch
import torch.nn.functional as F

# 1. Prepare model and optimizer
model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

# 2. Save weights before update
weight_before = model.lm_head.weight.detach().clone()

# 3. Forward pass
optimizer.zero_grad(set_to_none=True)

logits = model(encoder_ids, decoder_ids)

loss = F.cross_entropy(
    logits.reshape(-1, logits.size(-1)),
    labels.reshape(-1),
    ignore_index=-100
)

# 4. Backpropagation
loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

# 5. Update weights
optimizer.step()

# 6. Verify actual update
weight_change = (
    model.lm_head.weight.detach() - weight_before
).abs().max().item()

print("========== A4 TRAINING VERIFICATION ==========")
print(f"Loss:                 {loss.item():.6f}")
print(f"Gradient norm:        {grad_norm.item():.6f}")
print(f"Maximum weight change:{weight_change:.10f}")
print(f"Optimizer states:     {len(optimizer.state)}")
print(
    "Parameters finite:   ",
    all(torch.isfinite(p).all().item() for p in model.parameters())
)

print(
    "UPDATE SUCCESSFUL:",
    weight_change > 0 and len(optimizer.state) > 0
)

model.eval()

========== A4 TRAINING VERIFICATION ==========
Loss:                 16.304697
Gradient norm:        46.940491
Maximum weight change:0.0000207424
Optimizer states:     260
Parameters finite:    True
UPDATE SUCCESSFUL: True


A4Model(
  (token_embedding): Embedding(50000, 512)
  (encoder_pos): Embedding(1024, 512)
  (decoder_pos): Embedding(1024, 512)
  (encoder_layers): ModuleList(
    (0-5): 6 x A4EncoderLayer(
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): A4Attention(
        (q_proj): Linear(in_features=512, out_features=512, bias=True)
        (k_proj): Linear(in_features=512, out_features=512, bias=True)
        (v_proj): Linear(in_features=512, out_features=512, bias=True)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
      )
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ffn): A4FFNCheckpoint(
        (net): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=2048, out_features=512, bias=True)
        )
      )
    )
  )
  (encoder_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (decoder_laye

In [35]:
print("OPTIMIZER EXISTS:", "optimizer" in globals())
print("BATCH EXISTS:", "batch" in globals())
print("ENCODER IDS:", "encoder_ids" in globals())
print("DECODER IDS:", "decoder_ids" in globals())
print("LABELS:", "labels" in globals())

if "optimizer" in globals():
    print("OPTIMIZER TYPE:", type(optimizer).__name__)
    print("OPTIMIZER STATE:", len(optimizer.state))

if "loss" in globals():
    print("LAST LOSS:", loss.item() if torch.is_tensor(loss) else loss)

OPTIMIZER EXISTS: True
BATCH EXISTS: True
ENCODER IDS: True
DECODER IDS: True
LABELS: True
OPTIMIZER TYPE: AdamW
OPTIMIZER STATE: 260
LAST LOSS: 16.304697036743164


In [36]:
param = model.lm_head.weight

print("WEIGHT FINITE:", torch.isfinite(param).all().item())
print("GRADIENT EXISTS:", param.grad is not None)

if param.grad is not None:
    print("GRADIENT FINITE:", torch.isfinite(param.grad).all().item())
    print("GRADIENT MAX:", param.grad.abs().max().item())

print("OPTIMIZER STATE:", len(optimizer.state))
print("LAST LOSS:", float(loss))

WEIGHT FINITE: True
GRADIENT EXISTS: True
GRADIENT FINITE: True
GRADIENT MAX: 0.002576785394921899
OPTIMIZER STATE: 260
LAST LOSS: 16.304697036743164


/tmp/ipykernel_49/1156982562.py:11: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("LAST LOSS:", float(loss))


In [37]:
print("WEIGHT FINITE:", torch.isfinite(model.lm_head.weight).all().item())
print("GRADIENT EXISTS:", model.lm_head.weight.grad is not None)
print("GRADIENT FINITE:", torch.isfinite(model.lm_head.weight.grad).all().item())
print("GRADIENT MAX:", model.lm_head.weight.grad.abs().max().item())
print("OPTIMIZER STATES:", len(optimizer.state))
print("LOSS:", loss.detach().item())

WEIGHT FINITE: True
GRADIENT EXISTS: True
GRADIENT FINITE: True
GRADIENT MAX: 0.002576785394921899
OPTIMIZER STATES: 260
LOSS: 16.304697036743164


In [38]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

assert "train_loader" in globals(), "train_loader not found"
assert "dev_loader" in globals(), "dev_loader not found"

model.train()

# Reinitialize optimizer for a clean, controlled run
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

def run_epoch(loader, training=True):
    model.train() if training else model.eval()

    total_loss = 0.0
    total_batches = 0

    for batch in tqdm(loader):
        encoder_ids = batch["encoder_ids"].to(DEVICE)
        decoder_ids = batch["decoder_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.set_grad_enabled(training):
            logits = model(encoder_ids, decoder_ids)

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100
            )

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=1.0
                )

                optimizer.step()

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss detected!")

        total_loss += loss.detach().item()
        total_batches += 1

    return total_loss / total_batches

# One training epoch
train_loss = run_epoch(train_loader, training=True)

# Validation without gradient updates
with torch.no_grad():
    val_loss = run_epoch(dev_loader, training=False)

model.eval()

print("\n========== A4 BASELINE RESULTS ==========")
print(f"Training loss:   {train_loss:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Optimizer steps: {len(optimizer.state)}")
print("Training completed successfully.")

  0%|          | 0/203 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]


========== A4 BASELINE RESULTS ==========
Training loss:   6.8404
Validation loss: 6.2475
Optimizer steps: 260
Training completed successfully.


In [39]:
import os
import torch

SAVE_DIR = "/kaggle/working/phase6_a4"
os.makedirs(SAVE_DIR, exist_ok=True)

checkpoint_path = os.path.join(
    SAVE_DIR, "a4_smoke_epoch1.pt"
)

torch.save({
    "epoch": 1,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "train_loss": train_loss,
    "val_loss": val_loss,
    "learning_rate": 2e-5,
    "architecture": "A4",
    "train_records": len(train_loader.dataset),
    "validation_records": len(dev_loader.dataset),
}, checkpoint_path)

print("CHECKPOINT SAVED:", checkpoint_path)
print("FILE SIZE (MB):", round(os.path.getsize(checkpoint_path) / 1e6, 2))
print("TRAINING BATCHES:", len(train_loader))
print("VALIDATION BATCHES:", len(dev_loader))

CHECKPOINT SAVED: /kaggle/working/phase6_a4/a4_smoke_epoch1.pt
FILE SIZE (MB): 1156.99
TRAINING BATCHES: 203
VALIDATION BATCHES: 23


In [40]:
import torch

print("CUDA AVAILABLE:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

print("\nPOSSIBLE DATASET VARIABLES:")

for name, value in globals().copy().items():
    if (
        isinstance(value, list)
        and len(value) > 0
        and isinstance(value[0], dict)
        and "document" in value[0]
        and "summary" in value[0]
    ):
        print(
            f"{name}: {len(value)} records | "
            f"fields: {list(value[0].keys())}"
        )

CUDA AVAILABLE: True
GPU: Tesla T4

POSSIBLE DATASET VARIABLES:
train_data: 7030 records | fields: ['id', 'document', 'summary']
clean_data: 7028 records | fields: ['id', 'document', 'summary']
phase6_train: 6325 records | fields: ['id', 'document', 'summary']
phase6_dev: 703 records | fields: ['id', 'document', 'summary']
eligible_records: 226 records | fields: ['id', 'document', 'summary']
baseline_train: 203 records | fields: ['id', 'document', 'summary']
baseline_dev: 23 records | fields: ['id', 'document', 'summary']


In [41]:
import pandas as pd
from tqdm.auto import tqdm

assert "tokenizer" in globals(), "Tokenizer not loaded"
assert "phase6_train" in globals()
assert "phase6_dev" in globals()

CHUNK_SIZE = 900
OVERLAP = 100
STRIDE = CHUNK_SIZE - OVERLAP

def get_chunk_starts(n):
    if n <= CHUNK_SIZE:
        return [0]

    starts = list(range(0, n - CHUNK_SIZE + 1, STRIDE))

    # Ensure the final document tokens are covered
    final_start = n - CHUNK_SIZE

    if starts[-1] != final_start:
        starts.append(final_start)

    return starts

manifest = []
record_stats = []

for split_name, records in [
    ("train", phase6_train),
    ("dev", phase6_dev)
]:
    for item in tqdm(records, desc=f"Processing {split_name}"):

        doc_ids = tokenizer.encode(item["document"]).ids
        summary_ids = tokenizer.encode(item["summary"]).ids

        starts = get_chunk_starts(len(doc_ids))

        record_stats.append({
            "id": str(item["id"]),
            "split": split_name,
            "document_tokens": len(doc_ids),
            "summary_tokens": len(summary_ids),
            "chunk_count": len(starts),
            "review_required": str(item["id"]) in {"5493", "5676"}
        })

        for chunk_idx, start in enumerate(starts):
            end = min(start + CHUNK_SIZE, len(doc_ids))

            manifest.append({
                "id": str(item["id"]),
                "split": split_name,
                "chunk_index": chunk_idx,
                "start_token": start,
                "end_token": end,
                "chunk_tokens": end - start,
                "document_tokens": len(doc_ids)
            })

manifest_df = pd.DataFrame(manifest)
stats_df = pd.DataFrame(record_stats)

print("\n========== CHUNK MANIFEST ==========")
print("Training records:", (stats_df.split == "train").sum())
print("Validation records:", (stats_df.split == "dev").sum())
print("Total records:", len(stats_df))
print("Total chunks:", len(manifest_df))
print("Average chunks/document:", round(stats_df.chunk_count.mean(), 2))
print("Maximum chunks/document:", stats_df.chunk_count.max())
print("Maximum chunk length:", manifest_df.chunk_tokens.max())
print("Documents flagged for review:", stats_df.review_required.sum())

print("\nCHUNKS BY SPLIT:")
print(manifest_df.groupby("split").size())

print("\nCHUNK LENGTHS:")
print(manifest_df.chunk_tokens.describe())

print("\nEXAMPLE CHUNKS:")
print(manifest_df.head(10).to_string(index=False))

# Save manifests for reproducibility
manifest_df.to_csv(
    "/kaggle/working/phase6_chunk_manifest.csv",
    index=False
)

stats_df.to_csv(
    "/kaggle/working/phase6_record_statistics.csv",
    index=False
)

print("\nMANIFESTS SAVED SUCCESSFULLY")

Processing train:   0%|          | 0/6325 [00:00<?, ?it/s]

Processing dev:   0%|          | 0/703 [00:00<?, ?it/s]


========== CHUNK MANIFEST ==========
Training records: 6325
Validation records: 703
Total records: 7028
Total chunks: 47114
Average chunks/document: 6.7
Maximum chunks/document: 202
Maximum chunk length: 900
Documents flagged for review: 2

CHUNKS BY SPLIT:
split
dev       4894
train    42220
dtype: int64

CHUNK LENGTHS:
count    47114.000000
mean       899.190283
std         17.434123
min        199.000000
25%        900.000000
50%        900.000000
75%        900.000000
max        900.000000
Name: chunk_tokens, dtype: float64

EXAMPLE CHUNKS:
  id split  chunk_index  start_token  end_token  chunk_tokens  document_tokens
5006 train            0            0        900           900             5622
5006 train            1          800       1700           900             5622
5006 train            2         1600       2500           900             5622
5006 train            3         2400       3300           900             5622
5006 train            4         3200       4100      

In [42]:
# Verify train/validation split integrity
train_ids = set(
    str(x["id"]) for x in phase6_train
)

dev_ids = set(
    str(x["id"]) for x in phase6_dev
)

print("========== SPLIT VALIDATION ==========")
print("Training IDs:", len(train_ids))
print("Validation IDs:", len(dev_ids))
print("Overlap:", len(train_ids & dev_ids))
print("Split valid:", len(train_ids & dev_ids) == 0)

# Verify chunk coverage for every document
coverage_errors = []

for record in stats_df.to_dict("records"):
    rid = record["id"]

    chunks = manifest_df[
        manifest_df["id"] == rid
    ].sort_values("start_token")

    expected_length = record["document_tokens"]

    if chunks.empty:
        coverage_errors.append((rid, "No chunks"))
        continue

    if chunks.iloc[0]["start_token"] != 0:
        coverage_errors.append((rid, "Missing beginning"))

    if chunks.iloc[-1]["end_token"] != expected_length:
        coverage_errors.append((rid, "Missing ending"))

    for i in range(1, len(chunks)):
        previous_end = chunks.iloc[i-1]["end_token"]
        current_start = chunks.iloc[i]["start_token"]

        if current_start > previous_end:
            coverage_errors.append((rid, "Token gap"))

print("\n========== COVERAGE VALIDATION ==========")
print("Coverage errors:", len(coverage_errors))
print("First errors:", coverage_errors[:10])

# Inspect flagged records
print("\n========== FLAGGED RECORDS ==========")

for rid in ["5493", "5676"]:
    rows = [x for x in clean_data if str(x["id"]) == rid]

    if rows:
        item = rows[0]
        print(
            rid,
            "| Document tokens:", len(tokenizer.encode(item["document"]).ids),
            "| Summary tokens:", len(tokenizer.encode(item["summary"]).ids)
        )

========== SPLIT VALIDATION ==========
Training IDs: 6325
Validation IDs: 703
Overlap: 0
Split valid: True

========== COVERAGE VALIDATION ==========
Coverage errors: 0
First errors: []

========== FLAGGED RECORDS ==========
5493 | Document tokens: 199 | Summary tokens: 1000
5676 | Document tokens: 571 | Summary tokens: 2071


In [43]:
import numpy as np

summary_lengths = [
    len(tokenizer.encode(x["summary"]).ids)
    for x in phase6_train
    if str(x["id"]) not in {"5493", "5676"}
]

print("========== SUMMARY LENGTH ANALYSIS ==========")
print("Records:", len(summary_lengths))

for p in [25, 50, 75, 90, 95, 99, 100]:
    print(
        f"P{p}:",
        round(np.percentile(summary_lengths, p))
    )

print("\nSummaries fitting 900 tokens:",
      sum(x <= 900 for x in summary_lengths))

print("Summaries fitting 1022 tokens:",
      sum(x <= 1022 for x in summary_lengths))

print("Summaries exceeding 1022:",
      sum(x > 1022 for x in summary_lengths))

========== SUMMARY LENGTH ANALYSIS ==========
Records: 6323
P25: 494
P50: 746
P75: 1126
P90: 1756
P95: 2401
P99: 5080
P100: 32183

Summaries fitting 900 tokens: 3950
Summaries fitting 1022 tokens: 4432
Summaries exceeding 1022: 1891


In [44]:
def summary_length(record):
    return len(tokenizer.encode(record["summary"]).ids)

MAX_TARGET_LENGTH = 1022

train_eligible = [
    x for x in phase6_train
    if str(x["id"]) not in {"5493", "5676"}
    and summary_length(x) <= MAX_TARGET_LENGTH
]

dev_eligible = [
    x for x in phase6_dev
    if str(x["id"]) not in {"5493", "5676"}
    and summary_length(x) <= MAX_TARGET_LENGTH
]

print("========== BASELINE ELIGIBILITY ==========")
print("Original train:", len(phase6_train))
print("Eligible train:", len(train_eligible))
print("Excluded train:", len(phase6_train) - len(train_eligible))

print("\nOriginal dev:", len(phase6_dev))
print("Eligible dev:", len(dev_eligible))
print("Excluded dev:", len(phase6_dev) - len(dev_eligible))

print("\nTotal eligible:", len(train_eligible) + len(dev_eligible))

# Verify split integrity
train_ids = {str(x["id"]) for x in train_eligible}
dev_ids = {str(x["id"]) for x in dev_eligible}

print("Train/dev overlap:", len(train_ids & dev_ids))

========== BASELINE ELIGIBILITY ==========
Original train: 6325
Eligible train: 4432
Excluded train: 1893

Original dev: 703
Eligible dev: 476
Excluded dev: 227

Total eligible: 4908
Train/dev overlap: 0


In [45]:
import inspect
import torch

print("========== A4 MODEL INTERFACE ==========")

print("Forward signature:")
print(inspect.signature(model.forward))

print("\nEncoder layer signature:")
print(inspect.signature(model.encoder_layers[0].forward))

print("\nDecoder layer signature:")
print(inspect.signature(model.decoder_layers[0].forward))

print("\nModel forward implementation:")
print(inspect.getsource(type(model).forward))

print("\nEncoder layer implementation:")
print(inspect.getsource(type(model.encoder_layers[0]).forward))

print("\nGPU memory allocated (GB):",
      round(torch.cuda.memory_allocated() / 1e9, 3))

========== A4 MODEL INTERFACE ==========
Forward signature:
(encoder_ids, decoder_ids)

Encoder layer signature:
(x, mask=None)

Decoder layer signature:
(x, memory, self_mask=None, cross_mask=None)

Model forward implementation:
    def forward(self, encoder_ids, decoder_ids):

        B, Te = encoder_ids.shape
        Td = decoder_ids.shape[1]

        if Te > MAX_POSITION or Td > MAX_POSITION:
            raise ValueError("Sequence exceeds A4 positional capacity.")

        device = encoder_ids.device

        # Encoder
        enc_positions = torch.arange(Te, device=device)
        encoder = (
            self.token_embedding(encoder_ids)
            + self.encoder_pos(enc_positions)[None, :, :]
        )

        encoder_padding_mask = (
            encoder_ids == PAD_ID
        )[:, None, None, :]

        for layer in self.encoder_layers:
            encoder = layer(encoder, encoder_padding_mask)

        memory = self.encoder_norm(encoder)

        # Decoder
        dec_positio

In [46]:
import torch

def a4_encode(model, encoder_ids):
    B, Te = encoder_ids.shape

    if Te > MAX_POSITION:
        raise ValueError("Encoder sequence exceeds positional capacity")

    device = encoder_ids.device

    positions = torch.arange(Te, device=device)

    encoder = (
        model.token_embedding(encoder_ids)
        + model.encoder_pos(positions)[None, :, :]
    )

    padding_mask = (
        encoder_ids == PAD_ID
    )[:, None, None, :]

    for layer in model.encoder_layers:
        encoder = layer(encoder, padding_mask)

    return model.encoder_norm(encoder), padding_mask


def a4_decode(model, decoder_ids, memory, memory_padding_mask=None):
    B, Td = decoder_ids.shape

    if Td > MAX_POSITION:
        raise ValueError("Decoder sequence exceeds positional capacity")

    device = decoder_ids.device

    positions = torch.arange(Td, device=device)

    decoder = (
        model.token_embedding(decoder_ids)
        + model.decoder_pos(positions)[None, :, :]
    )

    causal_mask = torch.triu(
        torch.ones(Td, Td, device=device, dtype=torch.bool),
        diagonal=1
    )[None, None, :, :]

    decoder_padding_mask = (
        decoder_ids == PAD_ID
    )[:, None, None, :]

    self_mask = causal_mask | decoder_padding_mask

    for layer in model.decoder_layers:
        decoder = layer(
            decoder,
            memory,
            self_mask,
            memory_padding_mask
        )

    decoder = model.decoder_norm(decoder)

    return model.lm_head(decoder)


# Verify equivalence with original forward
model.eval()

with torch.no_grad():
    original_logits = model(encoder_ids, decoder_ids)

    memory, memory_mask = a4_encode(model, encoder_ids)

    refactored_logits = a4_decode(
        model,
        decoder_ids,
        memory,
        memory_mask
    )

max_difference = (
    original_logits - refactored_logits
).abs().max().item()

print("========== A4 REFACTOR VERIFICATION ==========")
print("Original logits:", tuple(original_logits.shape))
print("Refactored logits:", tuple(refactored_logits.shape))
print("Maximum difference:", max_difference)
print("Equivalent:", torch.allclose(
    original_logits,
    refactored_logits,
    atol=1e-5,
    rtol=1e-5
))
print("GPU memory allocated (GB):",
      round(torch.cuda.memory_allocated() / 1e9, 3))

========== A4 REFACTOR VERIFICATION ==========
Original logits: (1, 162, 50000)
Refactored logits: (1, 162, 50000)
Maximum difference: 0.0
Equivalent: True
GPU memory allocated (GB): 1.885


In [47]:
import torch
import torch.nn as nn

class HierarchicalAggregator(nn.Module):
    def __init__(
        self,
        d_model=512,
        num_heads=8,
        chunk_queries=16,
        document_queries=128
    ):
        super().__init__()

        self.chunk_queries = nn.Parameter(
            torch.randn(1, chunk_queries, d_model) * 0.02
        )

        self.document_queries = nn.Parameter(
            torch.randn(1, document_queries, d_model) * 0.02
        )

        self.chunk_attention = nn.MultiheadAttention(
            d_model, num_heads, batch_first=True
        )

        self.document_attention = nn.MultiheadAttention(
            d_model, num_heads, batch_first=True
        )

        self.chunk_norm = nn.LayerNorm(d_model)
        self.document_norm = nn.LayerNorm(d_model)

    def compress_chunk(self, chunk_memory):
        queries = self.chunk_queries.expand(
            chunk_memory.size(0), -1, -1
        )

        output, _ = self.chunk_attention(
            queries, chunk_memory, chunk_memory,
            need_weights=False
        )

        return self.chunk_norm(output + queries)

    def aggregate_document(self, chunk_vectors):
        queries = self.document_queries.expand(
            chunk_vectors.size(0), -1, -1
        )

        output, _ = self.document_attention(
            queries, chunk_vectors, chunk_vectors,
            need_weights=False
        )

        return self.document_norm(output + queries)

    def forward(self, chunk_memories):
        compressed_chunks = [
            self.compress_chunk(memory)
            for memory in chunk_memories
        ]

        combined = torch.cat(compressed_chunks, dim=1)

        return self.aggregate_document(combined)


aggregator = HierarchicalAggregator().to(DEVICE)

print("Aggregator initialized")
print("Chunk queries:", aggregator.chunk_queries.shape)
print("Document queries:", aggregator.document_queries.shape)
print(
    "Trainable aggregator parameters:",
    sum(p.numel() for p in aggregator.parameters())
)

Aggregator initialized
Chunk queries: torch.Size([1, 16, 512])
Document queries: torch.Size([1, 128, 512])
Trainable aggregator parameters: 2177024


In [48]:
# Locate sample
sample = next(
    x for x in clean_data if str(x["id"]) == "6712"
)

token_ids = tokenizer.encode(sample["document"]).ids

# Create chunks using the validated chunking function
starts = get_chunk_starts(len(token_ids))

chunk_memories = []

model.eval()
aggregator.eval()

with torch.no_grad():
    for start in starts:
        chunk = token_ids[start:start + CHUNK_SIZE]

        input_ids = torch.tensor(
            [[BOS_ID] + chunk + [EOS_ID]],
            dtype=torch.long,
            device=DEVICE
        )

        memory, _ = a4_encode(model, input_ids)

        chunk_memories.append(memory)

    document_memory = aggregator(chunk_memories)

print("========== HIERARCHICAL TEST ==========")
print("Document tokens:", len(token_ids))
print("Number of chunks:", len(starts))
print("Chunk memory vectors:", len(starts) * 16)
print("Final document memory:", tuple(document_memory.shape))
print("Memory finite:", torch.isfinite(document_memory).all().item())
print(
    "GPU memory allocated (GB):",
    round(torch.cuda.memory_allocated() / 1e9, 3)
)

========== HIERARCHICAL TEST ==========
Document tokens: 1544
Number of chunks: 2
Chunk memory vectors: 32
Final document memory: (1, 128, 512)
Memory finite: True
GPU memory allocated (GB): 1.899


In [49]:
# Greedy generation using hierarchical memory

MAX_NEW_TOKENS = 64

model.eval()
aggregator.eval()

generated_ids = [BOS_ID]

with torch.no_grad():
    for _ in range(MAX_NEW_TOKENS):

        decoder_input = torch.tensor(
            [generated_ids],
            dtype=torch.long,
            device=DEVICE
        )

        logits = a4_decode(
            model,
            decoder_input,
            document_memory,
            memory_padding_mask=None
        )

        next_token = logits[:, -1, :].argmax(dim=-1).item()

        if next_token == EOS_ID:
            break

        generated_ids.append(next_token)

generated_summary = tokenizer.decode(
    generated_ids[1:],
    skip_special_tokens=True
)

print("========== HIERARCHICAL GENERATION ==========")
print("Generated tokens:", len(generated_ids) - 1)
print("Generated summary:")
print(generated_summary)
print("\nReference summary:")
print(sample["summary"][:1500])

========== HIERARCHICAL GENERATION ==========
Generated tokens: 64
Generated summary:
 right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right right

Reference summary:
The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell its cement only through the marketing company, and certain sums would be paid for the cement supplied by the marketing company, which had the discretion to fix the sale price. For the 1950 51, 1951 52 and 1952 53 periods when the appellant was assessed to sales tax for the supply of cement, it maintained at the assess

In [50]:
import torch
import torch.nn.functional as F

model.train()
aggregator.train()

# Use the existing sample (ID 6712)
doc_ids = tokenizer.encode(sample["document"]).ids
target_ids = tokenizer.encode(sample["summary"]).ids

# Teacher-forcing inputs
decoder_ids_test = torch.tensor(
    [[BOS_ID] + target_ids],
    dtype=torch.long,
    device=DEVICE
)

labels_test = torch.tensor(
    [target_ids + [EOS_ID]],
    dtype=torch.long,
    device=DEVICE
)

# Build chunk memories WITH gradients
chunk_memories_test = []

for start in get_chunk_starts(len(doc_ids)):
    chunk = doc_ids[start:start + CHUNK_SIZE]

    input_ids = torch.tensor(
        [[BOS_ID] + chunk + [EOS_ID]],
        dtype=torch.long,
        device=DEVICE
    )

    memory, _ = a4_encode(model, input_ids)
    chunk_memories_test.append(memory)

# Hierarchical aggregation
document_memory_test = aggregator(chunk_memories_test)

# Decoder
logits = a4_decode(
    model,
    decoder_ids_test,
    document_memory_test
)

loss = F.cross_entropy(
    logits.reshape(-1, logits.size(-1)),
    labels_test.reshape(-1)
)

# Backpropagation
model.zero_grad(set_to_none=True)
aggregator.zero_grad(set_to_none=True)

loss.backward()

print("========== HIERARCHICAL GRADIENT TEST ==========")
print("Loss:", loss.detach().item())

print(
    "A4 gradient exists:",
    model.encoder_layers[0].attn.q_proj.weight.grad is not None
)

print(
    "Aggregator gradient exists:",
    aggregator.chunk_queries.grad is not None
)

print(
    "Document query gradient exists:",
    aggregator.document_queries.grad is not None
)

print(
    "A4 gradient finite:",
    torch.isfinite(
        model.encoder_layers[0].attn.q_proj.weight.grad
    ).all().item()
)

print(
    "Aggregator gradient finite:",
    torch.isfinite(aggregator.chunk_queries.grad).all().item()
)

========== HIERARCHICAL GRADIENT TEST ==========
Loss: 9.027968406677246
A4 gradient exists: True
Aggregator gradient exists: True
Document query gradient exists: True
A4 gradient finite: True
Aggregator gradient finite: True


In [51]:
import torch
import torch.nn.functional as F

# Freeze the complete pretrained A4 model first
for p in model.parameters():
    p.requires_grad = False

# Unfreeze decoder components
for p in model.decoder_layers.parameters():
    p.requires_grad = True

for p in model.decoder_norm.parameters():
    p.requires_grad = True

for p in model.decoder_pos.parameters():
    p.requires_grad = True

for p in model.lm_head.parameters():
    p.requires_grad = True

# Train the hierarchical aggregator
for p in aggregator.parameters():
    p.requires_grad = True

trainable_params = [
    p for p in list(model.parameters()) + list(aggregator.parameters())
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=2e-5,
    weight_decay=0.01
)

print("Trainable parameters:",
      sum(p.numel() for p in trainable_params))

print("Frozen encoder:",
      not any(p.requires_grad for p in model.encoder_layers.parameters()))

Trainable parameters: 53526528
Frozen encoder: True


In [52]:
def hierarchical_training_step(record):

    model.train()
    aggregator.train()

    doc_ids = tokenizer.encode(record["document"]).ids
    target_ids = tokenizer.encode(record["summary"]).ids

    assert len(target_ids) <= 1022

    chunk_memories = []

    # Frozen encoder: process each chunk without storing encoder gradients
    with torch.no_grad():
        for start in get_chunk_starts(len(doc_ids)):
            chunk = doc_ids[start:start + CHUNK_SIZE]

            input_ids = torch.tensor(
                [[BOS_ID] + chunk + [EOS_ID]],
                dtype=torch.long,
                device=DEVICE
            )

            memory, _ = a4_encode(model, input_ids)
            chunk_memories.append(memory)

    # Trainable hierarchical aggregation
    document_memory = aggregator(chunk_memories)

    decoder_ids = torch.tensor(
        [[BOS_ID] + target_ids],
        dtype=torch.long,
        device=DEVICE
    )

    labels = torch.tensor(
        [target_ids + [EOS_ID]],
        dtype=torch.long,
        device=DEVICE
    )

    optimizer.zero_grad(set_to_none=True)

    logits = a4_decode(
        model,
        decoder_ids,
        document_memory
    )

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1)
    )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_params,
        max_norm=1.0
    )

    optimizer.step()

    return {
        "loss": loss.detach().item(),
        "gradient_norm": grad_norm.item(),
        "chunks": len(chunk_memories),
        "target_tokens": len(target_ids)
    }


# Use the first eligible training record
first_result = hierarchical_training_step(train_eligible[0])

print("========== FIRST HIERARCHICAL TRAINING UPDATE ==========")
print("Record ID:", train_eligible[0]["id"])
print("Chunks:", first_result["chunks"])
print("Target tokens:", first_result["target_tokens"])
print("Loss:", first_result["loss"])
print("Gradient norm:", first_result["gradient_norm"])
print("Optimizer states:", len(optimizer.state))
print("UPDATE COMPLETED:", len(optimizer.state) > 0)

========== FIRST HIERARCHICAL TRAINING UPDATE ==========
Record ID: 1434
Chunks: 2
Target tokens: 189
Loss: 9.91405963897705
Gradient norm: 111.25236511230469
Optimizer states: 174
UPDATE COMPLETED: True


In [53]:
import numpy as np
from tqdm.auto import tqdm

PILOT_SIZE = 100

pilot_records = train_eligible[:PILOT_SIZE]

pilot_losses = []
pilot_grad_norms = []

for i, record in enumerate(tqdm(pilot_records)):
    result = hierarchical_training_step(record)

    pilot_losses.append(result["loss"])
    pilot_grad_norms.append(result["gradient_norm"])

    if (i + 1) % 10 == 0:
        print(
            f"Step {i+1}/{PILOT_SIZE} | "
            f"Loss: {np.mean(pilot_losses[-10:]):.4f} | "
            f"Gradient norm: {np.mean(pilot_grad_norms[-10:]):.4f}"
        )

print("\n========== HIERARCHICAL PILOT RESULTS ==========")
print("Training examples:", len(pilot_losses))
print("Initial loss:", round(pilot_losses[0], 4))
print("Final loss:", round(pilot_losses[-1], 4))
print("Mean loss:", round(float(np.mean(pilot_losses)), 4))
print("Mean gradient norm:", round(float(np.mean(pilot_grad_norms)), 4))
print("Finite losses:", bool(np.isfinite(pilot_losses).all()))

  0%|          | 0/100 [00:00<?, ?it/s]

Step 10/100 | Loss: 6.9798 | Gradient norm: 19.8323
Step 20/100 | Loss: 6.6243 | Gradient norm: 9.0299
Step 30/100 | Loss: 6.5095 | Gradient norm: 8.4584
Step 40/100 | Loss: 6.2093 | Gradient norm: 9.6714
Step 50/100 | Loss: 6.3129 | Gradient norm: 8.7791
Step 60/100 | Loss: 5.9237 | Gradient norm: 9.1542
Step 70/100 | Loss: 6.4110 | Gradient norm: 10.0555
Step 80/100 | Loss: 6.1697 | Gradient norm: 8.3269
Step 90/100 | Loss: 6.0805 | Gradient norm: 9.9959
Step 100/100 | Loss: 5.9151 | Gradient norm: 9.0198

========== HIERARCHICAL PILOT RESULTS ==========
Training examples: 100
Initial loss: 8.6014
Final loss: 5.6999
Mean loss: 6.3136
Mean gradient norm: 10.2323
Finite losses: True


In [54]:
import os
import torch

SAVE_DIR = "/kaggle/working/phase6_hierarchical"
os.makedirs(SAVE_DIR, exist_ok=True)

checkpoint_path = os.path.join(
    SAVE_DIR,
    "a4_hierarchical_pilot_100.pt"
)

torch.save({
    "epoch": 0,
    "pilot_steps": 100,
    "model_state_dict": model.state_dict(),
    "aggregator_state_dict": aggregator.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "pilot_initial_loss": pilot_losses[0],
    "pilot_final_loss": pilot_losses[-1],
    "pilot_mean_loss": float(np.mean(pilot_losses)),
    "pilot_gradient_norm": float(np.mean(pilot_grad_norms)),
    "architecture": "A4_Hierarchical",
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": OVERLAP,
    "chunk_queries": 16,
    "document_queries": 128,
    "encoder_frozen": True
}, checkpoint_path)

print("CHECKPOINT:", checkpoint_path)
print("SIZE (MB):", round(os.path.getsize(checkpoint_path) / 1e6, 2))
print("EXISTS:", os.path.exists(checkpoint_path))

CHECKPOINT: /kaggle/working/phase6_hierarchical/a4_hierarchical_pilot_100.pt
SIZE (MB): 822.74
EXISTS: True


In [55]:
import numpy as np
from tqdm.auto import tqdm

validation_records = dev_eligible[:50]
validation_losses = []

model.eval()
aggregator.eval()

with torch.no_grad():
    for record in tqdm(validation_records, desc="Validation"):

        doc_ids = tokenizer.encode(record["document"]).ids
        target_ids = tokenizer.encode(record["summary"]).ids

        chunk_memories = []

        for start in get_chunk_starts(len(doc_ids)):
            chunk = doc_ids[start:start + CHUNK_SIZE]

            input_ids = torch.tensor(
                [[BOS_ID] + chunk + [EOS_ID]],
                dtype=torch.long,
                device=DEVICE
            )

            memory, _ = a4_encode(model, input_ids)
            chunk_memories.append(memory)

        document_memory = aggregator(chunk_memories)

        decoder_ids = torch.tensor(
            [[BOS_ID] + target_ids],
            dtype=torch.long,
            device=DEVICE
        )

        labels = torch.tensor(
            [target_ids + [EOS_ID]],
            dtype=torch.long,
            device=DEVICE
        )

        logits = a4_decode(
            model,
            decoder_ids,
            document_memory
        )

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        )

        validation_losses.append(loss.item())

print("\n========== HIERARCHICAL VALIDATION ==========")
print("Validation records:", len(validation_losses))
print("Mean validation loss:", round(float(np.mean(validation_losses)), 4))
print("Median validation loss:", round(float(np.median(validation_losses)), 4))
print("Finite losses:", bool(np.isfinite(validation_losses).all()))

Validation:   0%|          | 0/50 [00:00<?, ?it/s]


========== HIERARCHICAL VALIDATION ==========
Validation records: 50
Mean validation loss: 6.1495
Median validation loss: 6.054
Finite losses: True


In [56]:
import os
import torch

SAVE_DIR = "/kaggle/working/phase6_hierarchical"
os.makedirs(SAVE_DIR, exist_ok=True)

checkpoint_path = os.path.join(
    SAVE_DIR, "a4_hierarchical_pilot_100.pt"
)

torch.save({
    "pilot_steps": 100,
    "model_state_dict": model.state_dict(),
    "aggregator_state_dict": aggregator.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "train_loss": float(np.mean(pilot_losses)),
    "validation_loss": float(np.mean(validation_losses)),
    "chunk_size": CHUNK_SIZE,
    "overlap": OVERLAP,
    "encoder_frozen": True
}, checkpoint_path)

print("Saved:", os.path.exists(checkpoint_path))
print("Checkpoint size (MB):",
      round(os.path.getsize(checkpoint_path) / 1e6, 2))

Saved: True
Checkpoint size (MB): 822.74


In [57]:
model.eval()
aggregator.eval()

test_record = dev_eligible[0]
doc_ids = tokenizer.encode(test_record["document"]).ids

chunk_memories = []

with torch.no_grad():
    for start in get_chunk_starts(len(doc_ids)):
        chunk = doc_ids[start:start + CHUNK_SIZE]

        input_ids = torch.tensor(
            [[BOS_ID] + chunk + [EOS_ID]],
            dtype=torch.long,
            device=DEVICE
        )

        memory, _ = a4_encode(model, input_ids)
        chunk_memories.append(memory)

    document_memory = aggregator(chunk_memories)

    generated_ids = [BOS_ID]

    for _ in range(128):
        decoder_input = torch.tensor(
            [generated_ids],
            dtype=torch.long,
            device=DEVICE
        )

        logits = a4_decode(
            model,
            decoder_input,
            document_memory
        )

        next_token = logits[:, -1, :].argmax(dim=-1).item()

        if next_token == EOS_ID:
            break

        generated_ids.append(next_token)

generated_text = tokenizer.decode(
    generated_ids[1:],
    skip_special_tokens=True
)

print("========== POST-TRAINING GENERATION ==========")
print("Validation ID:", test_record["id"])
print("Document tokens:", len(doc_ids))
print("Generated tokens:", len(generated_ids) - 1)
print("\nGENERATED SUMMARY:\n", generated_text)
print("\nREFERENCE SUMMARY:\n", test_record["summary"][:2000])

========== POST-TRAINING GENERATION ==========
Validation ID: 6827
Document tokens: 1077
Generated tokens: 128

GENERATED SUMMARY:
 The appellant was a section of the Act, the appellant was not been and the appellant was not later than the State of the appellant was a section of the appellant was a Act and the appellant was not. The appellant was not been found to the Act and the Act, the appeal by the appellant was not been rejected by the appellant of the appellant was not a Act. The High Court. The appellant was not a Act. The appellant was not been in the appellant was not been found to be the appeal. The High Court of the appellant was not. The appellant was a as the appellant of the appellant was not been a suit by the

REFERENCE SUMMARY:
 The appellant filed an application before the Central Administrative Tribunal, Patna Bench, for a declaration that the second proviso to rule 4 of Civil Services Examination Rules was violative of Articles 14 and 16 of the Constitu tion of Indi

In [58]:
import torch

def generate_controlled(
    document_memory,
    max_new_tokens=200,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3
):
    model.eval()

    generated = [BOS_ID]

    with torch.no_grad():
        for _ in range(max_new_tokens):

            decoder_input = torch.tensor(
                [generated],
                dtype=torch.long,
                device=DEVICE
            )

            logits = a4_decode(
                model,
                decoder_input,
                document_memory
            )

            next_logits = logits[0, -1, :].clone()

            # Repetition penalty
            for token_id in set(generated):
                if next_logits[token_id] > 0:
                    next_logits[token_id] /= repetition_penalty
                else:
                    next_logits[token_id] *= repetition_penalty

            # Prevent repeated n-grams
            if len(generated) >= no_repeat_ngram_size - 1:
                prefix = tuple(generated[-(no_repeat_ngram_size - 1):])

                for i in range(
                    len(generated) - no_repeat_ngram_size + 1
                ):
                    ngram = tuple(
                        generated[i:i + no_repeat_ngram_size]
                    )

                    if ngram[:-1] == prefix:
                        next_logits[ngram[-1]] = -float("inf")

            next_token = next_logits.argmax().item()

            if next_token == EOS_ID:
                break

            generated.append(next_token)

    return tokenizer.decode(
        generated[1:],
        skip_special_tokens=True
    )


controlled_summary = generate_controlled(
    document_memory,
    max_new_tokens=200
)

print("========== CONTROLLED GENERATION ==========")
print(controlled_summary)

========== CONTROLLED GENERATION ==========
The appellant was a section of the Act, and that the Court had not later than the appellant was in the State of the respondent. The2 of the appellant were an appeal to be followed by the appellant for the appellant or as to the High Court under section 3 of the time of the amount of the law. The [D] (2) of the fact of the High Courts Act, 1954, the suit was not been completed of the order passed by the Act. The Criminal Procedure, accused accused was not to the Act and the Act was not a Commissioner of the terms of the record of the Court and the account of the decision of the appeal against the appellant of the court and that Government of the same was not be rejected by the petition was made by the Highness of the period of the said the appellant had not be taken into the appellant. The Court was not below the appellant by the Court of the jurisdiction of the decree of the charge of the: 1. The appellant was not


In [59]:
import torch

record = next(
    x for x in dev_eligible if str(x["id"]) == "6827"
)

doc_ids = tokenizer.encode(record["document"]).ids
starts = get_chunk_starts(len(doc_ids))

model.eval()
aggregator.eval()

with torch.no_grad():
    # Original A4: first 1022 source tokens
    original_ids = torch.tensor(
        [[BOS_ID] + doc_ids[:1022] + [EOS_ID]],
        dtype=torch.long,
        device=DEVICE
    )
    original_memory, original_mask = a4_encode(model, original_ids)

    # Hierarchical memory: all document chunks
    chunk_memories = []

    for start in starts:
        chunk = doc_ids[start:start + CHUNK_SIZE]

        ids = torch.tensor(
            [[BOS_ID] + chunk + [EOS_ID]],
            dtype=torch.long,
            device=DEVICE
        )

        memory, _ = a4_encode(model, ids)
        chunk_memories.append(memory)

    hierarchical_memory = aggregator(chunk_memories)

print("Original memory:", tuple(original_memory.shape))
print("Hierarchical memory:", tuple(hierarchical_memory.shape))
print("Reference tokens:", len(tokenizer.encode(record["summary"]).ids))

Original memory: (1, 1024, 512)
Hierarchical memory: (1, 128, 512)
Reference tokens: 581


In [60]:
def generate_from_memory(memory, max_tokens=150):
    generated = [BOS_ID]

    with torch.no_grad():
        for _ in range(max_tokens):
            ids = torch.tensor(
                [generated], dtype=torch.long, device=DEVICE
            )

            logits = a4_decode(model, ids, memory)
            next_id = logits[0, -1].argmax().item()

            if next_id == EOS_ID:
                break

            generated.append(next_id)

    return tokenizer.decode(
        generated[1:],
        skip_special_tokens=True
    )


print("========== ORIGINAL A4 ==========")
print(generate_from_memory(original_memory))

print("\n========== HIERARCHICAL A4 ==========")
print(generate_from_memory(hierarchical_memory))

print("\n========== REFERENCE ==========")
print(record["summary"][:1500])

========== ORIGINAL A4 ==========
The appellant was the appellant was not the appellant was not the appellant was not the appellant was not the appellant was not in the appellant was not been and the appellant was not the appellant of the appellant of the appellant was not the appellant and the respondent. The appellant was not the appellant was not the respondent and the respondent of the High Court and the appellant was not the appellant was not the appellant of the respondent. The respondent of the appellant was not a the appellant was not the appellant of the High Court of the High Court, the appellant was not the respondent and the appellant was not the High Court and the appellant was not the High Court of the appellant of the High Court of the appellant. The High Court, the appellant of the appellant was not be

========== HIERARCHICAL A4 ==========
The appellant was a section of the Act, the appellant was not been and the appellant was not later than the State of the appellant 

In [61]:
import torch
import torch.nn.functional as F

model.eval()
aggregator.eval()

target_ids = tokenizer.encode(record["summary"]).ids

decoder_ids = torch.tensor(
    [[BOS_ID] + target_ids],
    dtype=torch.long,
    device=DEVICE
)

labels = torch.tensor(
    [target_ids + [EOS_ID]],
    dtype=torch.long,
    device=DEVICE
)

def calculate_loss(memory, memory_mask=None):
    with torch.no_grad():
        logits = a4_decode(
            model,
            decoder_ids,
            memory,
            memory_mask
        )

        return F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        ).item()

# 1. Actual hierarchical memory
loss_actual = calculate_loss(hierarchical_memory)

# 2. Zeroed hierarchical memory
loss_zero = calculate_loss(
    torch.zeros_like(hierarchical_memory)
)

# 3. Shuffled hierarchical memory positions
perm = torch.randperm(
    hierarchical_memory.size(1),
    device=DEVICE
)

shuffled_memory = hierarchical_memory[:, perm, :]
loss_shuffled = calculate_loss(shuffled_memory)

# 4. Original A4 encoder memory
loss_original = calculate_loss(
    original_memory,
    original_mask
)

print("Teacher-forced validation losses")
print("--------------------------------")
print(f"Hierarchical actual memory : {loss_actual:.4f}")
print(f"Hierarchical zero memory   : {loss_zero:.4f}")
print(f"Hierarchical shuffled      : {loss_shuffled:.4f}")
print(f"Original encoder memory    : {loss_original:.4f}")

Teacher-forced validation losses
--------------------------------
Hierarchical actual memory : 6.1339
Hierarchical zero memory   : 10.2220
Hierarchical shuffled      : 6.1339
Original encoder memory    : 6.2008


In [62]:
# Find a different validation document
other_record = next(
    x for x in dev_eligible
    if str(x["id"]) != "6827"
)

other_doc_ids = tokenizer.encode(
    other_record["document"]
).ids

other_starts = get_chunk_starts(len(other_doc_ids))

# Build memory from the other document
other_chunk_memories = []

with torch.no_grad():
    for start in other_starts:
        chunk = other_doc_ids[start:start + CHUNK_SIZE]

        ids = torch.tensor(
            [[BOS_ID] + chunk + [EOS_ID]],
            dtype=torch.long,
            device=DEVICE
        )

        memory, _ = a4_encode(model, ids)
        other_chunk_memories.append(memory)

    other_memory = aggregator(other_chunk_memories)

# Compare teacher-forced losses
loss_wrong_document = calculate_loss(other_memory)

print("Correct document memory :", round(loss_actual, 4))
print("Wrong document memory   :", round(loss_wrong_document, 4))
print("Zero document memory    :", round(loss_zero, 4))
print("Other document ID       :", other_record["id"])

Correct document memory : 6.1339
Wrong document memory   : 6.0994
Zero document memory    : 10.222
Other document ID       : 1034


In [63]:
import random
import torch
import torch.nn.functional as F
import numpy as np

model.eval()
aggregator.eval()

# Use 20 validation examples
diagnostic_records = dev_eligible[:20]

def build_document_memory(document):
    ids = tokenizer.encode(document).ids
    starts = get_chunk_starts(len(ids))
    memories = []

    with torch.no_grad():
        for start in starts:
            chunk = ids[start:start + CHUNK_SIZE]

            input_ids = torch.tensor(
                [[BOS_ID] + chunk + [EOS_ID]],
                dtype=torch.long,
                device=DEVICE
            )

            memory, _ = a4_encode(model, input_ids)
            memories.append(memory)

        return aggregator(memories)

def summary_loss(memory, summary):
    target = tokenizer.encode(summary).ids

    decoder_ids = torch.tensor(
        [[BOS_ID] + target],
        dtype=torch.long,
        device=DEVICE
    )

    labels = torch.tensor(
        [target + [EOS_ID]],
        dtype=torch.long,
        device=DEVICE
    )

    with torch.no_grad():
        logits = a4_decode(model, decoder_ids, memory)

        return F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        ).item()

# Build all document memories
memories = [
    build_document_memory(x["document"])
    for x in diagnostic_records
]

results = []

for i, item in enumerate(diagnostic_records):
    wrong_index = (i + 1) % len(diagnostic_records)

    own_loss = summary_loss(
        memories[i], item["summary"]
    )

    wrong_loss = summary_loss(
        memories[wrong_index], item["summary"]
    )

    zero_loss = summary_loss(
        torch.zeros_like(memories[i]), item["summary"]
    )

    results.append({
        "id": item["id"],
        "correct": own_loss,
        "wrong": wrong_loss,
        "zero": zero_loss,
        "wrong_minus_correct": wrong_loss - own_loss
    })

print("Mean correct loss:", np.mean([x["correct"] for x in results]))
print("Mean wrong loss:", np.mean([x["wrong"] for x in results]))
print("Mean zero loss:", np.mean([x["zero"] for x in results]))
print(
    "Mean wrong-correct difference:",
    np.mean([x["wrong_minus_correct"] for x in results])
)
print(
    "Cases where correct memory is better:",
    sum(x["wrong_minus_correct"] > 0 for x in results),
    "/", len(results)
)

Mean correct loss: 6.1585403203964235
Mean wrong loss: 6.157782196998596
Mean zero loss: 9.53111526966095
Mean wrong-correct difference: -0.0007581233978271485
Cases where correct memory is better: 9 / 20


In [64]:
import torch

A4_CHECKPOINT = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-22000-pt/a4_step_22000.pt"
)

checkpoint = torch.load(
    A4_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key, value in checkpoint.items():
        if isinstance(value, torch.Tensor):
            print(key, tuple(value.shape))
        elif isinstance(value, dict):
            print(key, "dictionary; entries:", len(value))
        else:
            print(key, type(value).__name__, str(value)[:150])

Checkpoint type: <class 'dict'>

Checkpoint keys:
model_state_dict dictionary; entries: 260
step int 22000
tokens_processed int 180096704
train_loss float 0.1944424146786332
reason str regular_checkpoint
saved_at str 2026-10-05T15:18:59.981007


In [65]:
import copy
import torch

# Create a separate model using the same architecture
original_model = copy.deepcopy(model).to(DEVICE)

# Load original pretrained weights
load_result = original_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

original_model.eval()

print("Original checkpoint loaded successfully.")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

Original checkpoint loaded successfully.
Missing keys: []
Unexpected keys: []


In [66]:
def generate_with_model(model_to_use, source_ids, max_tokens=150):
    model_to_use.eval()

    input_ids = torch.tensor(
        [source_ids],
        dtype=torch.long,
        device=DEVICE
    )

    with torch.no_grad():
        memory, memory_mask = a4_encode(
            model_to_use, input_ids
        )

        generated = [BOS_ID]

        for _ in range(max_tokens):
            decoder_ids = torch.tensor(
                [generated],
                dtype=torch.long,
                device=DEVICE
            )

            logits = a4_decode(
                model_to_use,
                decoder_ids,
                memory,
                memory_mask
            )

            next_token = logits[0, -1].argmax().item()

            if next_token == EOS_ID:
                break

            generated.append(next_token)

    return tokenizer.decode(
        generated[1:],
        skip_special_tokens=True
    )


# Same 1022 document tokens used in the earlier original A4 test
source_ids = [BOS_ID] + doc_ids[:1022] + [EOS_ID]

original_output = generate_with_model(
    original_model,
    source_ids,
    max_tokens=150
)

print("========== ORIGINAL PRETRAINED A4 ==========")
print(original_output)

print("\n========== REFERENCE SUMMARY ==========")
print(record["summary"][:1500])

========== ORIGINAL PRETRAINED A4 ==========
erred Case No. 2 of 1991. (Under Article Article A(1) of the Constitution of India) examinationologist, Madhan Pani which, Mrs. 1860 1860 and143 Singh for the Appellant.Up Sibal and Arun documentsley, Additional Place Generals, Ms. K K line line above.V.S. Rao for the Respond ents. The Judgment of the Court was delivered by section77EL Penc, J. The above case has been regisis ( in pursuance of our order No..11.90 in Transfer Petition (State) No. No/90 transferring O.leglegNo.191 of 1990 1990 1990 1990 (A) of the Constitution of India from the 87 of the information information information,

========== REFERENCE SUMMARY ==========
The appellant filed an application before the Central Administrative Tribunal, Patna Bench, for a declaration that the second proviso to rule 4 of Civil Services Examination Rules was violative of Articles 14 and 16 of the Constitu tion of India. By an interim order the Tribunal allowed the appellant to appear at the

In [67]:
import torch
import os

CHECKPOINTS = {
    "A1": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt",
}

for name, path in CHECKPOINTS.items():
    print(f"\n{'='*20} {name} {'='*20}")

    if not os.path.exists(path):
        print("CHECKPOINT NOT FOUND:", path)
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("File size (MB):", round(os.path.getsize(path)/1024**2, 2))

    if isinstance(ckpt, dict):
        for key, value in ckpt.items():
            if isinstance(value, dict):
                print(key, "entries:", len(value))
            elif isinstance(value, torch.Tensor):
                print(key, "shape:", tuple(value.shape))
            else:
                print(key, ":", str(value)[:200])
    else:
        print("Checkpoint type:", type(ckpt))


==================== A1 ====================
File size (MB): 593.71
step : 500
epoch : 0
train_loss : 5.366012096405029
val_loss : None
model_state_dict entries: 101
optimizer_state_dict entries: 2

==================== A2 ====================
File size (MB): 196.55
model_state_dict entries: 113
step : 9000
tokens_processed : 73728000
train_loss : 6.801188290119171
reason : regular_checkpoint
saved_at : 2026-10-05T19:17:55.016904

==================== A3 ====================
File size (MB): 197.9
model entries: 100
step : 1000
tokens_seen : 512000
architecture : A3_prefix_lm_hybrid
parameters : 51868672
context_length : 2048
hidden_size : 512
num_layers : 8
num_heads : 8
ffn_size : 2048
prefix_ratio : 0.5
dropout : 0.1
learning_rate : 0.0001
weight_decay : 0.1


In [68]:
import torch

for name, path in CHECKPOINTS.items():

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    state = ckpt.get(
        "model_state_dict",
        ckpt.get("model", {})
    )

    print(f"\n{'='*25} {name} {'='*25}")

    for key, value in list(state.items())[:35]:
        if isinstance(value, torch.Tensor):
            print(f"{key:65s} {tuple(value.shape)}")
        else:
            print(key, type(value))


========================= A1 =========================
token_embedding.weight                                            (50000, 512)
position_embedding.weight                                         (2048, 512)
blocks.0.norm1.weight                                             (512,)
blocks.0.norm1.bias                                               (512,)
blocks.0.attn.in_proj_weight                                      (1536, 512)
blocks.0.attn.in_proj_bias                                        (1536,)
blocks.0.attn.out_proj.weight                                     (512, 512)
blocks.0.attn.out_proj.bias                                       (512,)
blocks.0.norm2.weight                                             (512,)
blocks.0.norm2.bias                                               (512,)
blocks.0.ffn.0.weight                                             (2048, 512)
blocks.0.ffn.0.bias                                               (2048,)
blocks.0.ffn.2.weight                    

In [69]:
import torch

for name, path in CHECKPOINTS.items():

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    state = ckpt.get(
        "model_state_dict",
        ckpt.get("model", {})
    )

    print(f"\n{'='*25} {name} {'='*25}")

    # Count transformer layers
    prefixes = set()

    for key in state:
        parts = key.split(".")
        if len(parts) > 1 and parts[0] in [
            "blocks", "layers"
        ]:
            prefixes.add(parts[0])

    for prefix in prefixes:
        indices = [
            int(key.split(".")[1])
            for key in state
            if key.startswith(prefix + ".")
            and key.split(".")[1].isdigit()
        ]

        print("Layer prefix:", prefix)
        print("Number of layers:", max(indices) + 1 if indices else 0)

    # Important final/output parameters
    print("\nOutput and normalization parameters:")

    for key, value in state.items():
        if any(x in key.lower() for x in [
            "norm", "ln_f", "lm_head",
            "output", "position_embedding",
            "final"
        ]):
            if isinstance(value, torch.Tensor):
                print(key, tuple(value.shape))

    total_params = sum(
        v.numel() for v in state.values()
        if isinstance(v, torch.Tensor)
    )

    print("\nTotal parameters:", f"{total_params:,}")


========================= A1 =========================
Layer prefix: blocks
Number of layers: 8

Output and normalization parameters:
position_embedding.weight (2048, 512)
blocks.0.norm1.weight (512,)
blocks.0.norm1.bias (512,)
blocks.0.norm2.weight (512,)
blocks.0.norm2.bias (512,)
blocks.1.norm1.weight (512,)
blocks.1.norm1.bias (512,)
blocks.1.norm2.weight (512,)
blocks.1.norm2.bias (512,)
blocks.2.norm1.weight (512,)
blocks.2.norm1.bias (512,)
blocks.2.norm2.weight (512,)
blocks.2.norm2.bias (512,)
blocks.3.norm1.weight (512,)
blocks.3.norm1.bias (512,)
blocks.3.norm2.weight (512,)
blocks.3.norm2.bias (512,)
blocks.4.norm1.weight (512,)
blocks.4.norm1.bias (512,)
blocks.4.norm2.weight (512,)
blocks.4.norm2.bias (512,)
blocks.5.norm1.weight (512,)
blocks.5.norm1.bias (512,)
blocks.5.norm2.weight (512,)
blocks.5.norm2.bias (512,)
blocks.6.norm1.weight (512,)
blocks.6.norm1.bias (512,)
blocks.6.norm2.weight (512,)
blocks.6.norm2.bias (512,)
blocks.7.norm1.weight (512,)
blocks.7.norm1

In [70]:
import torch

for name, path in CHECKPOINTS.items():
    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    state = ckpt.get(
        "model_state_dict",
        ckpt.get("model", {})
    )

    print(f"\n========== {name} ==========")

    for key, value in state.items():
        if isinstance(value, torch.Tensor):
            if (
                "lm_head" in key
                or "output" in key
                or "bias" in key
                and value.numel() == 50000
            ):
                print(key, tuple(value.shape))

    total = sum(
        value.numel()
        for value in state.values()
        if isinstance(value, torch.Tensor)
    )

    print(f"Total parameters: {total:,}")


========== A1 ==========
lm_head.weight (50000, 512)
Total parameters: 77,468,672

========== A2 ==========
output_bias (50000,)
Total parameters: 51,513,680

========== A3 ==========
Total parameters: 51,868,672


In [71]:
for name, path in CHECKPOINTS.items():
    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    state = ckpt.get(
        "model_state_dict",
        ckpt.get("model", {})
    )

    print(f"\n========== {name}: FINAL KEYS ==========")

    for key in state:
        if not any(
            key.startswith(prefix)
            for prefix in [
                "blocks.",
                "layers.",
                "token_embedding.",
                "position_embedding."
            ]
        ):
            print(key)


========== A1: FINAL KEYS ==========
final_norm.weight
final_norm.bias
lm_head.weight

========== A2: FINAL KEYS ==========
output_bias
final_norm.weight
final_norm.bias

========== A3: FINAL KEYS ==========
ln_f.weight
ln_f.bias


In [72]:
import torch

for name, path in CHECKPOINTS.items():
    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    state = ckpt.get(
        "model_state_dict",
        ckpt.get("model", {})
    )

    embedding = state["token_embedding.weight"]

    print(f"\n{name}")

    if "lm_head.weight" in state:
        head = state["lm_head.weight"]
        print(
            "LM head equals token embedding:",
            torch.equal(head, embedding)
        )

    else:
        print("No separate LM-head weight in checkpoint.")
        print("Weight tying must be verified from forward code.")


A1
LM head equals token embedding: True

A2
No separate LM-head weight in checkpoint.
Weight tying must be verified from forward code.

A3
No separate LM-head weight in checkpoint.
Weight tying must be verified from forward code.


In [73]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/aq1de3"):
    for file in files:
        if file.endswith((".py", ".ipynb")):
            print(os.path.join(root, file))

/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a2a2a2.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a4a4a4.ipynb
/kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb
/kaggle/input/datasets/aq1de3/p2p2p2/phase-2 (1).ipynb


In [74]:
import json
import os
import re

NOTEBOOKS = {
    "A1": "/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb",
    "A2": "/kaggle/input/datasets/aq1de3/p1p1p1/a2a2a2.ipynb",
    "A3": "/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb",
    "A4": "/kaggle/input/datasets/aq1de3/p1p1p1/a4a4a4.ipynb",
}

PATTERNS = [
    r"class .*Model",
    r"class .*Transformer",
    r"class .*Attention",
    r"def forward",
    r"lm_head",
    r"output_bias",
    r"token_embedding",
    r"causal_mask",
    r"prefix.*mask",
    r"attn_mask",
    r"generate"
]

for name, path in NOTEBOOKS.items():
    print(f"\n{'='*30} {name} {'='*30}")

    if not os.path.exists(path):
        print("Notebook not found:", path)
        continue

    with open(path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    for i, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        if any(re.search(p, source, re.I) for p in PATTERNS):
            print(f"\n--- CELL {i} ---")
            print(source[:10000])


============================== A1 ==============================

--- CELL 4 ---
# ============================================================
# PHASE 1 — ARCHITECTURE 2
# GQA — UPDATED MODEL WIDTH
# PARAMETER + GPU CHECK
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

print("=" * 70)
print("ARCHITECTURE 2 — GQA UPDATED RESOURCE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

VOCAB_SIZE = Tokenizer.from_file(
    TOKENIZER_PATH
).get_vocab_size()

SEQ_LEN = 2048

D_MODEL = 528
N_LAYERS = 8
N_Q_HEADS = 8
N_KV_HEADS = 2
FFN_DIM = 2048

HEAD_DIM = D_MODEL // N_Q_HEADS
GROUP_SIZE = N_Q_HEADS // N_KV_HEADS

print(f"\nVocabulary       : {VOCAB_SIZE:,}")
print(f"Context

In [75]:
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = 50_000
CONTEXT_LENGTH = 2048
HIDDEN_SIZE = 512
NUM_LAYERS = 8
NUM_HEADS = 8
FFN_SIZE = 2048
PREFIX_RATIO = 0.5

class A3ExactBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(HIDDEN_SIZE)

        self.attn = nn.MultiheadAttention(
            embed_dim=HIDDEN_SIZE,
            num_heads=NUM_HEADS,
            dropout=0.0,
            batch_first=True
        )

        self.ln2 = nn.LayerNorm(HIDDEN_SIZE)

        self.ffn = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, FFN_SIZE),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(FFN_SIZE, HIDDEN_SIZE)
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)

        attn_out, _ = self.attn(
            h, h, h,
            attn_mask=attn_mask,
            need_weights=False
        )

        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x


class A3Exact(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE, HIDDEN_SIZE
        )

        self.position_embedding = nn.Embedding(
            CONTEXT_LENGTH, HIDDEN_SIZE
        )

        self.layers = nn.ModuleList([
            A3ExactBlock()
            for _ in range(NUM_LAYERS)
        ])

        self.ln_f = nn.LayerNorm(HIDDEN_SIZE)

    def forward(self, input_ids):
        B, T = input_ids.shape

        if T > CONTEXT_LENGTH:
            raise ValueError("Input exceeds A3 context length")

        positions = torch.arange(
            T, device=input_ids.device
        ).unsqueeze(0)

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        prefix_len = int(T * PREFIX_RATIO)

        mask = torch.full(
            (T, T),
            float("-inf"),
            device=input_ids.device
        )

        mask[:prefix_len, :prefix_len] = 0

        for i in range(prefix_len, T):
            mask[i, :i + 1] = 0

        for layer in self.layers:
            x = layer(x, mask)

        x = self.ln_f(x)

        return F.linear(x, self.token_embedding.weight)


# Load the untouched A3 checkpoint
A3_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "step-0001000-pt/step_0001000.pt"
)

a3_checkpoint = torch.load(
    A3_PATH,
    map_location="cpu",
    weights_only=False
)

a3_original = A3Exact().to(DEVICE)

result = a3_original.load_state_dict(
    a3_checkpoint["model"],
    strict=True
)

a3_original.eval()

print("A3 strict loading successful")
print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)
print("Parameters:", sum(p.numel() for p in a3_original.parameters()))

# Forward smoke test
test_ids = torch.randint(
    0, VOCAB_SIZE, (1, 128), device=DEVICE
)

with torch.no_grad():
    test_logits = a3_original(test_ids)

print("Output shape:", tuple(test_logits.shape))
print("All finite:", torch.isfinite(test_logits).all().item())

A3 strict loading successful
Missing keys: []
Unexpected keys: []
Parameters: 51868672
Output shape: (1, 128, 50000)
All finite: True


In [76]:
# ============================================================
# PHASE 6 - A3 DATASET ELIGIBILITY CHECK
# No training or data modification in this cell
# ============================================================

MAX_CONTEXT = 2048
SPECIAL_TOKENS = 2  # BOS and document EOS

def get_a3_lengths(record):
    doc_len = len(tokenizer.encode(record["document"]).ids)
    summary_len = len(tokenizer.encode(record["summary"]).ids)

    # BOS + document + document EOS + summary + final EOS
    total_len = SPECIAL_TOKENS + doc_len + summary_len + 1

    return doc_len, summary_len, total_len

a3_train_lengths = [
    get_a3_lengths(r) for r in phase6_train
    if r.get("summary") and r["id"] not in [5493, 5676]
]

a3_dev_lengths = [
    get_a3_lengths(r) for r in phase6_dev
    if r.get("summary") and r["id"] not in [5493, 5676]
]

def report_a3_eligibility(lengths, name):
    eligible = [x for x in lengths if x[2] <= MAX_CONTEXT]

    print(f"\n{name}")
    print("-" * 45)
    print("Total records:", len(lengths))
    print("Eligible without truncation:", len(eligible))
    print("Excluded:", len(lengths) - len(eligible))
    print("Eligibility percentage:",
          round(100 * len(eligible) / max(1, len(lengths)), 2), "%")

    if eligible:
        print("Maximum document tokens:", max(x[0] for x in eligible))
        print("Maximum summary tokens:", max(x[1] for x in eligible))
        print("Maximum combined tokens:", max(x[2] for x in eligible))

report_a3_eligibility(a3_train_lengths, "A3 TRAIN")
report_a3_eligibility(a3_dev_lengths, "A3 VALIDATION")


A3 TRAIN
---------------------------------------------
Total records: 6325
Eligible without truncation: 626
Excluded: 5699
Eligibility percentage: 9.9 %
Maximum document tokens: 1830
Maximum summary tokens: 1000
Maximum combined tokens: 2048

A3 VALIDATION
---------------------------------------------
Total records: 703
Eligible without truncation: 63
Excluded: 640
Eligibility percentage: 8.96 %
Maximum document tokens: 1701
Maximum summary tokens: 618
Maximum combined tokens: 2021


In [77]:
# ============================================================
# PHASE 6 - A3 CONTEXT BUDGET COMPARISON
# Preserve complete summaries; measure document truncation
# ============================================================

CONTEXT = 2048
SPECIAL_TOKENS = 3  # BOS + document EOS + final EOS

def compare_budget(lengths, split_name):
    print(f"\n{split_name}")
    print("=" * 75)
    print(
        f"{'Summary cap':>12} | {'Doc budget':>11} | "
        f"{'Eligible':>9} | {'Coverage':>9} | {'Docs cut':>9}"
    )

    for summary_cap in [256, 384, 512, 640, 768, 900]:
        doc_budget = CONTEXT - SPECIAL_TOKENS - summary_cap

        eligible = [
            x for x in lengths
            if x[1] <= summary_cap
        ]

        docs_cut = sum(1 for x in eligible if x[0] > doc_budget)

        coverage = 100 * len(eligible) / max(1, len(lengths))

        print(
            f"{summary_cap:12d} | {doc_budget:11d} | "
            f"{len(eligible):9d} | {coverage:8.2f}% | "
            f"{docs_cut:9d}"
        )

compare_budget(a3_train_lengths, "TRAIN")
compare_budget(a3_dev_lengths, "VALIDATION")


TRAIN
 Summary cap |  Doc budget |  Eligible |  Coverage |  Docs cut
         256 |        1789 |       258 |     4.08% |       120
         384 |        1661 |       891 |    14.09% |       553
         512 |        1533 |      1707 |    26.99% |      1246
         640 |        1405 |      2538 |    40.13% |      2089
         768 |        1277 |      3289 |    52.00% |      2920
         900 |        1145 |      3950 |    62.45% |      3681

VALIDATION
 Summary cap |  Doc budget |  Eligible |  Coverage |  Docs cut
         256 |        1789 |        31 |     4.41% |        18
         384 |        1661 |        98 |    13.94% |        64
         512 |        1533 |       192 |    27.31% |       145
         640 |        1405 |       288 |    40.97% |       238
         768 |        1277 |       361 |    51.35% |       326
         900 |        1145 |       426 |    60.60% |       400


In [78]:
# ============================================================
# PHASE 6 - A3 PREFIX-LM DATA PREPARATION
# Summary cap: 900
# Document cap: 1145
# ============================================================

A3_MAX_DOC = 1145
A3_MAX_SUMMARY = 900

BOS_ID = 2
EOS_ID = 3

def select_head_tail(token_ids, max_length=A3_MAX_DOC):
    if len(token_ids) <= max_length:
        return token_ids

    head_len = 687
    tail_len = max_length - head_len

    return token_ids[:head_len] + token_ids[-tail_len:]


def prepare_a3_record(record):
    doc_ids = tokenizer.encode(record["document"]).ids
    summary_ids = tokenizer.encode(record["summary"]).ids

    # Preserve the complete summary; exclude summaries above the cap.
    if len(summary_ids) > A3_MAX_SUMMARY:
        return None

    doc_ids = select_head_tail(doc_ids)

    # Prefix: BOS + document + separator EOS
    prefix_ids = [BOS_ID] + doc_ids + [EOS_ID]
    prefix_len = len(prefix_ids)

    # Full sequence ends with summary EOS.
    full_ids = prefix_ids + summary_ids + [EOS_ID]

    # Next-token prediction.
    input_ids = full_ids[:-1]
    labels = full_ids[1:]

    # Do not calculate loss on document-prefix tokens.
    labels[:prefix_len - 1] = [-100] * (prefix_len - 1)

    assert len(input_ids) == len(labels)
    assert len(input_ids) <= 2047
    assert prefix_len <= len(input_ids)

    return {
        "input_ids": input_ids,
        "labels": labels,
        "prefix_len": prefix_len,
        "document_tokens": len(doc_ids),
        "summary_tokens": len(summary_ids)
    }


# Prepare train and validation datasets.
a3_train_records = []
for record in phase6_train:
    if record["id"] in [5493, 5676] or not record.get("summary"):
        continue

    item = prepare_a3_record(record)
    if item is not None:
        a3_train_records.append(item)


a3_dev_records = []
for record in phase6_dev:
    if record["id"] in [5493, 5676] or not record.get("summary"):
        continue

    item = prepare_a3_record(record)
    if item is not None:
        a3_dev_records.append(item)


print("A3 TRAIN examples:", len(a3_train_records))
print("A3 VALIDATION examples:", len(a3_dev_records))

print("\nFirst training example:")
sample = a3_train_records[0]

print("Input length:", len(sample["input_ids"]))
print("Prefix length:", sample["prefix_len"])
print("Document tokens:", sample["document_tokens"])
print("Summary tokens:", sample["summary_tokens"])
print("Supervised labels:", sum(x != -100 for x in sample["labels"]))

assert all(len(x["input_ids"]) <= 2047 for x in a3_train_records)
assert all(len(x["input_ids"]) == len(x["labels"]) for x in a3_train_records)

print("\nA3 data preparation checks passed.")

A3 TRAIN examples: 3950
A3 VALIDATION examples: 426

First training example:
Input length: 1336
Prefix length: 1147
Document tokens: 1145
Summary tokens: 189
Supervised labels: 190

A3 data preparation checks passed.


In [79]:
# ============================================================
# PHASE 6 - A3 DYNAMIC PREFIX MASK + SMOKE TEST
# No optimizer update in this cell
# ============================================================

import torch
import torch.nn.functional as F

A3_NUM_HEADS = 8
A3_VOCAB_SIZE = 50_000

def build_a3_prefix_mask(prefix_lens, lengths, max_len, num_heads, device):
    """
    True = masked, False = allowed.

    Prefix queries attend bidirectionally within the document prefix.
    Summary queries attend to the full prefix and previous/current summary.
    Padding positions are excluded as keys.
    """
    batch_size = len(prefix_lens)

    mask = torch.ones(
        batch_size, max_len, max_len,
        dtype=torch.bool,
        device=device
    )

    for b in range(batch_size):
        prefix_len = int(prefix_lens[b])
        valid_len = int(lengths[b])

        assert 0 < prefix_len <= valid_len <= max_len

        # Document prefix: bidirectional attention within prefix.
        mask[b, :prefix_len, :prefix_len] = False

        # Summary suffix: prefix + causal summary attention.
        for q in range(prefix_len, valid_len):
            mask[b, q, :q + 1] = False

        # Padded query positions are unused by the loss.
        # Allow them to attend to key 0 to avoid all-masked rows.
        for q in range(valid_len, max_len):
            mask[b, q, 0] = False

    # MultiheadAttention expects (batch_size * num_heads, T, T).
    mask = mask.repeat_interleave(num_heads, dim=0)

    return mask


def a3_forward_dynamic(model, input_ids, prefix_lens, lengths):
    B, T = input_ids.shape

    if T > CONTEXT_LENGTH:
        raise ValueError(f"Sequence length {T} exceeds {CONTEXT_LENGTH}")

    positions = torch.arange(T, device=input_ids.device).unsqueeze(0)

    x = (
        model.token_embedding(input_ids)
        + model.position_embedding(positions)
    )

    attention_mask = build_a3_prefix_mask(
        prefix_lens=prefix_lens,
        lengths=lengths,
        max_len=T,
        num_heads=A3_NUM_HEADS,
        device=input_ids.device
    )

    for layer in model.layers:
        x = layer(x, attention_mask)

    x = model.ln_f(x)

    return F.linear(x, model.token_embedding.weight)


# ------------------------------------------------------------
# Single-example batch for initial memory-safe testing
# ------------------------------------------------------------

sample = a3_train_records[0]

input_ids = torch.tensor(
    [sample["input_ids"]],
    dtype=torch.long,
    device=DEVICE
)

labels = torch.tensor(
    [sample["labels"]],
    dtype=torch.long,
    device=DEVICE
)

prefix_lens = [sample["prefix_len"]]
lengths = [len(sample["input_ids"])]

a3_original.train()
a3_original.zero_grad(set_to_none=True)

logits = a3_forward_dynamic(
    a3_original,
    input_ids,
    prefix_lens,
    lengths
)

loss = F.cross_entropy(
    logits.reshape(-1, A3_VOCAB_SIZE),
    labels.reshape(-1),
    ignore_index=-100
)

print("Input shape:", tuple(input_ids.shape))
print("Logits shape:", tuple(logits.shape))
print("Prefix length:", prefix_lens[0])
print("Sequence length:", lengths[0])
print("Supervised tokens:", (labels != -100).sum().item())
print("Loss:", loss.item())
print("Logits finite:", torch.isfinite(logits).all().item())
print("Loss finite:", torch.isfinite(loss).item())

# Verify gradients without updating model weights.
loss.backward()

grad_exists = any(
    p.grad is not None
    for p in a3_original.parameters()
    if p.requires_grad
)

grad_finite = all(
    torch.isfinite(p.grad).all().item()
    for p in a3_original.parameters()
    if p.grad is not None
)

print("Gradient exists:", grad_exists)
print("Gradients finite:", grad_finite)

a3_original.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    print(
        "GPU allocated (GB):",
        round(torch.cuda.memory_allocated() / 1024**3, 3)
    )

assert logits.shape == (1, len(sample["input_ids"]), A3_VOCAB_SIZE)
assert torch.isfinite(loss)
assert grad_exists and grad_finite

print("\nA3 dynamic-prefix smoke test passed.")

Input shape: (1, 1336)
Logits shape: (1, 1336, 50000)
Prefix length: 1147
Sequence length: 1336
Supervised tokens: 190
Loss: 16.90991973876953
Logits finite: True
Loss finite: True
Gradient exists: True
Gradients finite: True
GPU allocated (GB): 2.278

A3 dynamic-prefix smoke test passed.


In [80]:
# ============================================================
# PHASE 6 - A3 CONTROLLED TRAINING PILOT
# 100 train / 50 validation / 1 epoch
# ============================================================

import copy
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import AdamW

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# 1. Select deterministic subsets
# ------------------------------------------------------------

train_indices = list(range(len(a3_train_records)))
random.Random(SEED).shuffle(train_indices)

pilot_train = [a3_train_records[i] for i in train_indices[:100]]
pilot_dev = a3_dev_records[:50]

print("Pilot train:", len(pilot_train))
print("Pilot validation:", len(pilot_dev))

# ------------------------------------------------------------
# 2. Create an independent model copy
# ------------------------------------------------------------

a3_pilot = copy.deepcopy(a3_original).to(DEVICE)
a3_pilot.train()

optimizer_a3 = AdamW(
    a3_pilot.parameters(),
    lr=1e-5,
    weight_decay=0.01
)

# ------------------------------------------------------------
# 3. Single-record tensor preparation
# ------------------------------------------------------------

def make_a3_tensors(record):
    input_ids = torch.tensor(
        [record["input_ids"]],
        dtype=torch.long,
        device=DEVICE
    )

    labels = torch.tensor(
        [record["labels"]],
        dtype=torch.long,
        device=DEVICE
    )

    return input_ids, labels

# ------------------------------------------------------------
# 4. Training
# ------------------------------------------------------------

a3_train_losses = []
a3_grad_norms = []

for step, record in enumerate(pilot_train, start=1):

    a3_pilot.train()
    optimizer_a3.zero_grad(set_to_none=True)

    input_ids, labels = make_a3_tensors(record)

    prefix_lens = [record["prefix_len"]]
    lengths = [len(record["input_ids"])]

    logits = a3_forward_dynamic(
        a3_pilot,
        input_ids,
        prefix_lens,
        lengths
    )

    loss = F.cross_entropy(
        logits.reshape(-1, A3_VOCAB_SIZE),
        labels.reshape(-1),
        ignore_index=-100
    )

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite training loss at step {step}")

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        a3_pilot.parameters(),
        max_norm=1.0
    )

    if not torch.isfinite(grad_norm):
        raise RuntimeError(f"Non-finite gradient at step {step}")

    optimizer_a3.step()

    a3_train_losses.append(loss.item())
    a3_grad_norms.append(float(grad_norm))

    if step % 10 == 0:
        window_loss = np.mean(a3_train_losses[-10:])
        print(
            f"Step {step:03d}/100 | "
            f"Loss: {loss.item():.4f} | "
            f"Last 10 mean: {window_loss:.4f} | "
            f"Grad norm: {float(grad_norm):.4f}"
        )

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

a3_pilot.eval()
a3_validation_losses = []

with torch.no_grad():
    for record in pilot_dev:

        input_ids, labels = make_a3_tensors(record)

        logits = a3_forward_dynamic(
            a3_pilot,
            input_ids,
            [record["prefix_len"]],
            [len(record["input_ids"])]
        )

        loss = F.cross_entropy(
            logits.reshape(-1, A3_VOCAB_SIZE),
            labels.reshape(-1),
            ignore_index=-100
        )

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite validation loss")

        a3_validation_losses.append(loss.item())

# ------------------------------------------------------------
# 6. Results
# ------------------------------------------------------------

print("\n========== A3 PILOT RESULTS ==========")
print("Training updates:", len(a3_train_losses))
print("Training initial loss:", round(a3_train_losses[0], 4))
print("Training final loss:", round(a3_train_losses[-1], 4))
print("Training mean loss:", round(np.mean(a3_train_losses), 4))
print("Validation mean loss:", round(np.mean(a3_validation_losses), 4))
print("Validation median loss:", round(np.median(a3_validation_losses), 4))
print("All training losses finite:", np.isfinite(a3_train_losses).all())
print("All validation losses finite:", np.isfinite(a3_validation_losses).all())

# ------------------------------------------------------------
# 7. Save separate pilot checkpoint
# ------------------------------------------------------------

import os

A3_SAVE_DIR = "/kaggle/working/phase6_a3"
os.makedirs(A3_SAVE_DIR, exist_ok=True)

A3_PILOT_PATH = os.path.join(
    A3_SAVE_DIR,
    "a3_prefix_summarization_pilot_100.pt"
)

torch.save({
    "model_state_dict": a3_pilot.state_dict(),
    "optimizer_state_dict": optimizer_a3.state_dict(),
    "train_losses": a3_train_losses,
    "validation_losses": a3_validation_losses,
    "train_examples": len(pilot_train),
    "validation_examples": len(pilot_dev),
    "learning_rate": 1e-5,
    "seed": SEED,
    "document_budget": A3_MAX_DOC,
    "summary_budget": A3_MAX_SUMMARY,
    "training_method": "Full fine-tuning",
}, A3_PILOT_PATH)

print("\nSaved checkpoint:", A3_PILOT_PATH)
print(
    "Checkpoint size (MB):",
    round(os.path.getsize(A3_PILOT_PATH) / 1024**2, 2)
)

if torch.cuda.is_available():
    print(
        "Peak GPU memory (GB):",
        round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    )

print("\nA3 pilot completed.")

Pilot train: 100
Pilot validation: 50
Step 010/100 | Loss: 14.8028 | Last 10 mean: 14.5824 | Grad norm: 5.2657
Step 020/100 | Loss: 12.2488 | Last 10 mean: 14.4798 | Grad norm: 4.4579
Step 030/100 | Loss: 13.7392 | Last 10 mean: 14.2349 | Grad norm: 5.9365
Step 040/100 | Loss: 14.9003 | Last 10 mean: 14.0593 | Grad norm: 4.7643
Step 050/100 | Loss: 10.4463 | Last 10 mean: 13.0781 | Grad norm: 4.7653
Step 060/100 | Loss: 13.0019 | Last 10 mean: 13.6217 | Grad norm: 4.5574
Step 070/100 | Loss: 11.6037 | Last 10 mean: 13.2464 | Grad norm: 4.5374
Step 080/100 | Loss: 14.1397 | Last 10 mean: 13.3278 | Grad norm: 4.0896
Step 090/100 | Loss: 14.9317 | Last 10 mean: 14.1849 | Grad norm: 3.6548
Step 100/100 | Loss: 13.3011 | Last 10 mean: 13.8385 | Grad norm: 4.6803

========== A3 PILOT RESULTS ==========
Training updates: 100
Training initial loss: 14.804
Training final loss: 13.3011
Training mean loss: 13.8654
Validation mean loss: 13.7396
Validation median loss: 13.7453
All training losses f

In [81]:
# ============================================================
# PHASE 6 - A3 GENERATION QUALITY CHECK
# Three held-out validation examples
# ============================================================

import torch

a3_pilot.eval()

def generate_a3_summary(record, max_new_tokens=160):
    # Extract document prefix only; exclude reference summary.
    prefix = record["input_ids"][:record["prefix_len"]]

    generated = torch.tensor(
        [prefix],
        dtype=torch.long,
        device=DEVICE
    )

    prefix_len = len(prefix)

    with torch.no_grad():
        for _ in range(max_new_tokens):

            logits = a3_forward_dynamic(
                a3_pilot,
                generated,
                [prefix_len],
                [generated.shape[1]]
            )

            next_token_logits = logits[0, -1, :].clone()

            next_token = torch.argmax(
                next_token_logits
            ).item()

            if next_token == EOS_ID:
                break

            generated = torch.cat(
                [
                    generated,
                    torch.tensor(
                        [[next_token]],
                        dtype=torch.long,
                        device=DEVICE
                    )
                ],
                dim=1
            )

            if generated.shape[1] >= CONTEXT_LENGTH:
                break

    generated_ids = generated[0, prefix_len:].tolist()

    return tokenizer.decode(generated_ids)


# Use three held-out validation records.
for i, record in enumerate(a3_dev_records[:3], start=1):

    prediction = generate_a3_summary(
        record,
        max_new_tokens=160
    )

    reference_ids = record["labels"]
    reference_ids = [
        token for token in reference_ids
        if token != -100 and token != EOS_ID
    ]

    reference = tokenizer.decode(reference_ids)

    print("\n" + "=" * 80)
    print(f"VALIDATION EXAMPLE {i}")
    print("=" * 80)

    print("\nGENERATED SUMMARY:\n")
    print(prediction)

    print("\nREFERENCE SUMMARY:\n")
    print(reference)


VALIDATION EXAMPLE 1

GENERATED SUMMARY:

 of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court. The The State Government of the case case case case of the Court of the Court of the Court of the Court Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court the Court of the Court Court of the Court of the Court of the Court Court Court of the Court of the Court of the Court. The State Government of the Court of the Court. The The Court of the Court. The State Government. The not not not not not not not not not not not not not not not not not not not not not

REFERENCE SUMMARY:

The appellant filed an application before the Central Administrative Tribunal, Patna Bench, for a declaration that the second proviso to rule 4 of Civil Services Examination Rules was violative of Articles 14 and 16 of the Constitu 

In [82]:
# ============================================================
# PHASE 6 - ORIGINAL A3 vs FINE-TUNED A3
# Same validation inputs, same greedy decoding
# ============================================================

def generate_a3_with_model(model, record, max_new_tokens=120):

    prefix = record["input_ids"][:record["prefix_len"]]

    generated = torch.tensor(
        [prefix],
        dtype=torch.long,
        device=DEVICE
    )

    prefix_len = len(prefix)

    model.eval()

    with torch.no_grad():
        for _ in range(max_new_tokens):

            logits = a3_forward_dynamic(
                model,
                generated,
                [prefix_len],
                [generated.shape[1]]
            )

            next_token = logits[0, -1].argmax().item()

            if next_token == EOS_ID:
                break

            generated = torch.cat(
                [
                    generated,
                    torch.tensor(
                        [[next_token]],
                        device=DEVICE,
                        dtype=torch.long
                    )
                ],
                dim=1
            )

            if generated.shape[1] >= CONTEXT_LENGTH:
                break

    return tokenizer.decode(
        generated[0, prefix_len:].tolist()
    )


for i, record in enumerate(a3_dev_records[:3], start=1):

    original_output = generate_a3_with_model(
        a3_original, record
    )

    finetuned_output = generate_a3_with_model(
        a3_pilot, record
    )

    print("\n" + "=" * 90)
    print(f"EXAMPLE {i}")
    print("=" * 90)

    print("\nORIGINAL A3 CHECKPOINT:")
    print(original_output)

    print("\nFINE-TUNED A3 CHECKPOINT:")
    print(finetuned_output)

print("\nComparison completed.")


EXAMPLE 1

ORIGINAL A3 CHECKPOINT:
 a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a Board of the Board Board of the Board of the Board a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a Board a a a

FINE-TUNED A3 CHECKPOINT:
 of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court. The The State Government of the case case case case of the Court of the Court of the Court of the Court Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court the Court of the Court Court of the Court of the Court of the Court Court Court of the Court of the Court of the Court. The State Government

EXAMPLE 2

ORIGINAL A3 CHECKPOINT:
 a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a

In [83]:
# ============================================================
# PHASE 6 - CRITICAL A3 TOKENIZER COMPATIBILITY CHECK
# ============================================================

import os
import hashlib
from tokenizers import Tokenizer

A3_TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "mathimohankowshik/tokenizer1/tokenizer.json"
)

PHASE6_TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "aq1de3/tokenizer-json/tokenizer.json"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path in [A3_TOKENIZER_PATH, PHASE6_TOKENIZER_PATH]:
    print("\nPath:", path)
    print("Exists:", os.path.exists(path))
    if os.path.exists(path):
        print("SHA256:", sha256_file(path))

assert os.path.exists(A3_TOKENIZER_PATH), "Original A3 tokenizer missing"
assert os.path.exists(PHASE6_TOKENIZER_PATH), "Phase 6 tokenizer missing"

tok_a3 = Tokenizer.from_file(A3_TOKENIZER_PATH)
tok_phase6 = Tokenizer.from_file(PHASE6_TOKENIZER_PATH)

vocab_a3 = tok_a3.get_vocab()
vocab_phase6 = tok_phase6.get_vocab()

print("\n========== VOCABULARY COMPARISON ==========")
print("A3 vocabulary size:", len(vocab_a3))
print("Phase 6 vocabulary size:", len(vocab_phase6))
print("Exact vocabulary mapping match:", vocab_a3 == vocab_phase6)

if vocab_a3 != vocab_phase6:
    only_a3 = set(vocab_a3) - set(vocab_phase6)
    only_phase6 = set(vocab_phase6) - set(vocab_a3)

    different_ids = [
        (token, vocab_a3[token], vocab_phase6[token])
        for token in set(vocab_a3) & set(vocab_phase6)
        if vocab_a3[token] != vocab_phase6[token]
    ]

    print("Tokens only in A3:", len(only_a3))
    print("Tokens only in Phase 6:", len(only_phase6))
    print("Shared tokens with different IDs:", len(different_ids))
    print("First 10 ID mismatches:", different_ids[:10])

print("\n========== ENCODING COMPARISON ==========")

test_texts = [
    "The Supreme Court dismissed the appeal.",
    "The appellant challenged the validity of Article 14.",
    "The State Government filed a petition before the High Court."
]

for text in test_texts:
    ids_a3 = tok_a3.encode(text).ids
    ids_phase6 = tok_phase6.encode(text).ids

    print("\nText:", text)
    print("A3 IDs:", ids_a3[:20])
    print("Phase 6 IDs:", ids_phase6[:20])
    print("Exact token IDs match:", ids_a3 == ids_phase6)


Path: /kaggle/input/datasets/mathimohankowshik/tokenizer1/tokenizer.json
Exists: False

Path: /kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
Exists: True
SHA256: 61465ad788ec819ea768b54fe70de1be02bc948b035de6065fd5433ae238e09e


AssertionError: Original A3 tokenizer missing

In [84]:
import os

SEARCH_ROOTS = [
    "/kaggle/input",
    "/kaggle/working"
]

matches = []

for root in SEARCH_ROOTS:
    for current, dirs, files in os.walk(root):
        # Avoid scanning irrelevant hidden folders
        dirs[:] = [d for d in dirs if not d.startswith(".")]

        for filename in files:
            if "tokenizer" in filename.lower() or filename.lower().endswith(
                (".json", ".model", ".vocab")
            ):
                path = os.path.join(current, filename)
                matches.append(path)

print("Possible tokenizer/config files:")
for path in matches:
    print(path)

print("\nCurrent Phase 6 tokenizer exists:",
      os.path.exists(
          "/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json"
      ))

Possible tokenizer/config files:
/kaggle/input/datasets/aq1de3/trainjson/train.json
/kaggle/input/datasets/aq1de3/testtest/test (3).json
/kaggle/input/datasets/aq1de3/traintr/train (2).json
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
/kaggle/input/datasets/aq1de3/phase3/CL_dev.json
/kaggle/input/datasets/aq1de3/phase3/CL_test.json
/kaggle/input/datasets/aq1de3/phase3/CL_train.json
/kaggle/input/datasets/aq1de3/phase3/IT_dev.json
/kaggle/input/datasets/aq1de3/phase3/IT_test.json
/kaggle/input/datasets/aq1de3/phase3/IT_train.json
/kaggle/input/datasets/aq1de3/phase3/label_vocab.json
/kaggle/input/datasets/aq1de3/devjson/dev.json
/kaggle/input/datasets/aq1de3/phase6/test (4).json
/kaggle/input/datasets/aq1de3/phase6/train (3).json
/kaggle/input/datasets/aq1de3/devdev/dev (2).json
/kaggle/input/datasets/aq1de3/testjson/test.json

Current Phase 6 tokenizer exists: True


In [85]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if filename.lower() == "tokenizer.json":
            print(os.path.join(root, filename))

/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json


In [86]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if filename.lower() in ["tokenizer.json", "tokenizer_config.json"]:
            print(os.path.join(root, filename))

/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json


In [87]:
import os
from tokenizers import Tokenizer

TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer loaded successfully")
print("Vocabulary size:", tokenizer.get_vocab_size())
print("Special tokens:")
for token in ["<pad>", "<unk>", "<bos>", "<eos>", "<mask>"]:
    print(token, "->", tokenizer.token_to_id(token))

Tokenizer loaded successfully
Vocabulary size: 50000
Special tokens:
<pad> -> None
<unk> -> None
<bos> -> None
<eos> -> None
<mask> -> None


In [88]:
from tokenizers import Tokenizer
import json

TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer configuration:")
print(tokenizer.to_str()[:5000])

print("\nSpecial token candidates:")
vocab = tokenizer.get_vocab()

for token, token_id in sorted(vocab.items(), key=lambda x: x[1]):
    if any(word in token.lower() for word in ["pad", "bos", "eos", "unk", "mask", "start", "end"]):
        print(repr(token), "->", token_id)

Tokenizer configuration:
{"version":"1.0","truncation":null,"padding":null,"added_tokens":[{"id":0,"content":"[PAD]","single_word":false,"lstrip":false,"rstrip":false,"normalized":false,"special":true},{"id":1,"content":"[UNK]","single_word":false,"lstrip":false,"rstrip":false,"normalized":false,"special":true},{"id":2,"content":"[BOS]","single_word":false,"lstrip":false,"rstrip":false,"normalized":false,"special":true},{"id":3,"content":"[EOS]","single_word":false,"lstrip":false,"rstrip":false,"normalized":false,"special":true},{"id":4,"content":"[MASK]","single_word":false,"lstrip":false,"rstrip":false,"normalized":false,"special":true}],"normalizer":null,"pre_tokenizer":{"type":"ByteLevel","add_prefix_space":false,"trim_offsets":true,"use_regex":true},"post_processor":null,"decoder":null,"model":{"type":"BPE","dropout":null,"unk_token":"[UNK]","continuing_subword_prefix":null,"end_of_word_suffix":null,"fuse_unk":false,"byte_fallback":false,"ignore_merges":false,"vocab":{"[PAD]":0,"[

In [89]:
from tokenizers import Tokenizer, decoders

TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer.decoder = decoders.ByteLevel()

PAD_ID = tokenizer.token_to_id("[PAD]")
UNK_ID = tokenizer.token_to_id("[UNK]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")
MASK_ID = tokenizer.token_to_id("[MASK]")

print("Vocabulary:", tokenizer.get_vocab_size())
print("PAD:", PAD_ID)
print("UNK:", UNK_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)
print("MASK:", MASK_ID)

sample = "[BOS] The Court considered the appeal. [EOS]"
encoded = tokenizer.encode(sample)

print("\nEncoded IDs:", encoded.ids)
print("Decoded:", tokenizer.decode(encoded.ids))

Vocabulary: 50000
PAD: 0
UNK: 1
BOS: 2
EOS: 3
MASK: 4

Encoded IDs: [2, 344, 430, 1266, 238, 668, 18, 200, 3]
Decoded:  The Court considered the appeal. 


In [90]:
import os

SEARCH_ROOTS = [
    "/kaggle/input",
    "/kaggle/working"
]

print("Searching for A3 tokenizer files...\n")

for root in SEARCH_ROOTS:
    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if not d.startswith(".")]

        for filename in files:
            lower = filename.lower()

            if (
                "tokenizer" in lower
                or "vocab" in lower
                or "merges" in lower
            ):
                print(os.path.join(current, filename))

Searching for A3 tokenizer files...

/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
/kaggle/input/datasets/aq1de3/phase3/label_vocab.json


In [91]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)

    if depth <= 3:
        print(f"\nFOLDER: {root}")
        for filename in files:
            print("  FILE:", filename)


FOLDER: /kaggle/input

FOLDER: /kaggle/input/datasets

FOLDER: /kaggle/input/datasets/aq1de3

FOLDER: /kaggle/input/datasets/aq1de3/a2-step-9000-pt
  FILE: a2_step_9000.pt

FOLDER: /kaggle/input/datasets/aq1de3/step-0001000-pt
  FILE: step_0001000.pt

FOLDER: /kaggle/input/datasets/aq1de3/trainjson
  FILE: train.json

FOLDER: /kaggle/input/datasets/aq1de3/33kkkk
  FILE: step_0330000.pt

FOLDER: /kaggle/input/datasets/aq1de3/phase1-model
  FILE: phase1_model_code_extract.ipynb

FOLDER: /kaggle/input/datasets/aq1de3/testtest
  FILE: test (3).json

FOLDER: /kaggle/input/datasets/aq1de3/traintr
  FILE: train (2).json

FOLDER: /kaggle/input/datasets/aq1de3/p3p3p3p3
  FILE: phase3-capstone.ipynb

FOLDER: /kaggle/input/datasets/aq1de3/tokenizer-json
  FILE: tokenizer.json

FOLDER: /kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt
  FILE: architecture1_sustained_step_500.pt

FOLDER: /kaggle/input/datasets/aq1de3/phase3
  FILE: CL_dev.json
  FILE: CL_test.json
  FILE: CL_train.

In [92]:
import json

NOTEBOOK_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "p1p1p1/a3-phase1 (1).ipynb"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Searching original A3 notebook...\n")

for i, cell in enumerate(notebook.get("cells", [])):
    source = "".join(cell.get("source", []))

    if any(word in source.lower() for word in [
        "tokenizer",
        "tokenizer.json",
        "token_to_id",
        "train_new_from_iterator",
        "bpe"
    ]):
        print(f"\n{'='*60}")
        print(f"CELL {i} | TYPE: {cell.get('cell_type')}")
        print(source[:5000])

Searching original A3 notebook...


CELL 0 | TYPE: code
# ============================================================
# A2 — FRESH NOTEBOOK SETUP
# ============================================================

import os
import gc
import time
import math
import json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from kaggle_secrets import UserSecretsClient
from tokenizers import Tokenizer
from datasets import load_dataset

# ------------------------------------------------------------
# Basic environment
# ------------------------------------------------------------

assert torch.cuda.is_available(), "❌ CUDA GPU not available"

device = torch.device("cuda")

# ------------------------------------------------------------
# Hugging Face token
# ------------------------------------------------------------

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# -----------------------------------------------------------

In [93]:
TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "mathimohankowshik/tokenizer1/tokenizer.json"
)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

Exception: No such file or directory (os error 2)

In [94]:
TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "aq1de3/tokenizer-json/tokenizer.json"
)

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer loaded:", tokenizer.get_vocab_size())

Tokenizer loaded: 50000


In [95]:
from tokenizers import Tokenizer, decoders

TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer.decoder = decoders.ByteLevel()

PAD_ID = tokenizer.token_to_id("[PAD]")
UNK_ID = tokenizer.token_to_id("[UNK]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")
MASK_ID = tokenizer.token_to_id("[MASK]")

assert tokenizer.get_vocab_size() == 50000
assert (PAD_ID, UNK_ID, BOS_ID, EOS_ID, MASK_ID) == (0, 1, 2, 3, 4)

sample = "The Supreme Court considered the legal appeal."
encoded = tokenizer.encode(sample)

print("Vocabulary:", tokenizer.get_vocab_size())
print("Special IDs:", PAD_ID, UNK_ID, BOS_ID, EOS_ID, MASK_ID)
print("Encoded:", encoded.ids)
print("Decoded:", tokenizer.decode(encoded.ids))

Vocabulary: 50000
Special IDs: 0 1 2 3 4
Encoded: [550, 1487, 430, 1266, 238, 1999, 668, 18]
Decoded: The Supreme Court considered the legal appeal.


In [96]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def a3_forward_fixed_prefix(model, input_ids, prefix_len):
    B, T = input_ids.shape

    assert 0 < prefix_len <= T
    assert T <= 2048

    positions = torch.arange(T, device=input_ids.device).unsqueeze(0)

    x = (
        model.token_embedding(input_ids)
        + model.position_embedding(positions)
    )

    # Fixed prefix: bidirectional attention.
    # Generated suffix: causal attention, with access to the prefix.
    mask = torch.full(
        (T, T), float("-inf"), device=input_ids.device
    )

    mask[:prefix_len, :prefix_len] = 0

    for i in range(prefix_len, T):
        mask[i, :prefix_len] = 0
        mask[i, prefix_len:i+1] = 0

    for layer in model.layers:
        x = layer(x, mask)

    x = model.ln_f(x)
    return F.linear(x, model.token_embedding.weight)


@torch.no_grad()
def generate_a3_fixed_prefix(model, tokenizer, prompt, max_new_tokens=100):
    model.eval()

    ids = tokenizer.encode(prompt).ids
    ids = ids[:2048 - max_new_tokens]

    prefix_len = len(ids)
    generated = ids.copy()

    for _ in range(max_new_tokens):
        input_ids = torch.tensor(
            [generated], dtype=torch.long, device=next(model.parameters()).device
        )

        logits = a3_forward_fixed_prefix(model, input_ids, prefix_len)
        next_id = logits[0, -1].argmax().item()
        generated.append(next_id)

    return tokenizer.decode(generated[prefix_len:])


prompt = (
    "The appellant challenged the decision of the High Court. "
    "The Supreme Court considered the facts and legal arguments."
)

output = generate_a3_fixed_prefix(
    model, tokenizer, prompt, max_new_tokens=100
)

print("A3 fixed-prefix generation:")
print(output)

AttributeError: 'A4Model' object has no attribute 'position_embedding'

In [97]:
import torch

print("Available model variables:")

for name, obj in list(globals().items()):
    if isinstance(obj, torch.nn.Module):
        print(
            f"{name}: {obj.__class__.__name__} | "
            f"Parameters: {sum(p.numel() for p in obj.parameters()):,}"
        )

Available model variables:
__: A4Model | Parameters: 96,389,120
___: A4Model | Parameters: 96,389,120
model: A4Model | Parameters: 96,389,120
_31: A4Model | Parameters: 96,389,120
_32: A4Model | Parameters: 96,389,120
_33: A4Model | Parameters: 96,389,120
_34: A4Model | Parameters: 96,389,120
aggregator: HierarchicalAggregator | Parameters: 2,177,024
original_model: A4Model | Parameters: 96,389,120
a3_original: A3Exact | Parameters: 51,868,672
a3_pilot: A3Exact | Parameters: 51,868,672


In [98]:
Available model variables:
__: A4Model | Parameters: 96,389,120
___: A4Model | Parameters: 96,389,120
model: A4Model | Parameters: 96,389,120
_31: A4Model | Parameters: 96,389,120
_32: A4Model | Parameters: 96,389,120
_33: A4Model | Parameters: 96,389,120
_34: A4Model | Parameters: 96,389,120
aggregator: HierarchicalAggregator | Parameters: 2,177,024
original_model: A4Model | Parameters: 96,389,120
a3_original: A3Exact | Parameters: 51,868,672
a3_pilot: A3Exact | Parameters: 51,868,672

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (2653928747.py, line 9)

In [99]:
prompt = (
    "The appellant challenged the decision of the High Court. "
    "The Supreme Court considered the facts and legal arguments."
)

print("=" * 70)
print("ORIGINAL A3")
print("=" * 70)

original_output = generate_a3_fixed_prefix(
    a3_original,
    tokenizer,
    prompt,
    max_new_tokens=100
)

print(original_output)


print("\n" + "=" * 70)
print("FINE-TUNED A3")
print("=" * 70)

pilot_output = generate_a3_fixed_prefix(
    a3_pilot,
    tokenizer,
    prompt,
    max_new_tokens=100
)

print(pilot_output)

ORIGINAL A3

(b) a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a

FINE-TUNED A3
 The of the Court of the Court of the Court of the Court of the Court of the Court of the Court Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court Court of the Court Court of the Court of the Court of the Court of the Court of the Court of the Court of the Court Court of the Court Court of the Court Court Court of the Court of the Court of the Court of the Court of the Court of the


In [100]:
import os
from tokenizers import Tokenizer

phase6_path = (
    "/kaggle/input/datasets/aq1de3/"
    "tokenizer-json/tokenizer.json"
)

original_path = (
    "/kaggle/input/datasets/mathimohankowshik/"
    "tokenizer1/tokenizer.json"
)

print("Phase 6 tokenizer:", os.path.exists(phase6_path))
print("Original A3 tokenizer:", os.path.exists(original_path))

if os.path.exists(original_path):
    t1 = Tokenizer.from_file(phase6_path)
    t2 = Tokenizer.from_file(original_path)

    vocab1 = t1.get_vocab()
    vocab2 = t2.get_vocab()

    print("Vocab sizes:", len(vocab1), len(vocab2))
    print("Identical mappings:", vocab1 == vocab2)
else:
    print("Original A3 tokenizer is still unavailable.")

Phase 6 tokenizer: True
Original A3 tokenizer: False
Original A3 tokenizer is still unavailable.


In [101]:
import os
import torch

checkpoint_path = (
    "/kaggle/working/phase6_hierarchical/"
    "a4_hierarchical_pilot_100.pt"
)

print("Checkpoint exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    print(
        "Checkpoint size:",
        round(os.path.getsize(checkpoint_path) / 1024**2, 2),
        "MB"
    )
    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )
    print("Checkpoint keys:", list(ckpt.keys()))
else:
    print("Pilot checkpoint not found. We will inspect the current model state.")

Checkpoint exists: True
Checkpoint size: 784.62 MB
Checkpoint keys: ['pilot_steps', 'model_state_dict', 'aggregator_state_dict', 'optimizer_state_dict', 'train_loss', 'validation_loss', 'chunk_size', 'overlap', 'encoder_frozen']


In [102]:
import torch

checkpoint_path = (
    "/kaggle/working/phase6_hierarchical/"
    "a4_hierarchical_pilot_100.pt"
)

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Pilot steps:", ckpt["pilot_steps"])
print("Training loss:", ckpt["train_loss"])
print("Validation loss:", ckpt["validation_loss"])
print("Chunk size:", ckpt["chunk_size"])
print("Overlap:", ckpt["overlap"])
print("Encoder frozen:", ckpt["encoder_frozen"])

print("\nModel parameter tensors:", len(ckpt["model_state_dict"]))
print("Aggregator parameter tensors:", len(ckpt["aggregator_state_dict"]))
print("Optimizer state entries:", len(ckpt["optimizer_state_dict"]["state"]))


Pilot steps: 100
Training loss: 6.313575329780579
Validation loss: 6.149533729553223
Chunk size: 900
Overlap: 100
Encoder frozen: True

Model parameter tensors: 260
Aggregator parameter tensors: 14
Optimizer state entries: 174


In [103]:
import torch

checkpoint_path = (
    "/kaggle/working/phase6_hierarchical/"
    "a4_hierarchical_pilot_100.pt"
)

# Load checkpoint
ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Restore model and aggregator
model.load_state_dict(
    ckpt["model_state_dict"],
    strict=True
)

aggregator.load_state_dict(
    ckpt["aggregator_state_dict"],
    strict=True
)

# Restore optimizer
optimizer.load_state_dict(
    ckpt["optimizer_state_dict"]
)

# Move models to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
aggregator.to(device)

# Verify restored weights
model_key = next(iter(ckpt["model_state_dict"]))
aggregator_key = next(iter(ckpt["aggregator_state_dict"]))

model_match = torch.equal(
    model.state_dict()[model_key].cpu(),
    ckpt["model_state_dict"][model_key]
)

aggregator_match = torch.equal(
    aggregator.state_dict()[aggregator_key].cpu(),
    ckpt["aggregator_state_dict"][aggregator_key]
)

print("Checkpoint restoration completed.")
print("Pilot steps:", ckpt["pilot_steps"])
print("Model weights match:", model_match)
print("Aggregator weights match:", aggregator_match)
print("Device:", device)
print("Optimizer state entries:", len(optimizer.state))
print("Training loss:", ckpt["train_loss"])
print("Validation loss:", ckpt["validation_loss"])

assert model_match and aggregator_match

Checkpoint restoration completed.
Pilot steps: 100
Model weights match: True
Aggregator weights match: True
Device: cuda
Optimizer state entries: 174
Training loss: 6.313575329780579
Validation loss: 6.149533729553223


In [104]:
# Verify restored training configuration

print("Device:", next(model.parameters()).device)
print("Optimizer:", type(optimizer).__name__)
print("Learning rates:")

for i, group in enumerate(optimizer.param_groups):
    print(f"Group {i}: LR={group['lr']}, weight_decay={group['weight_decay']}")

trainable_model = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

trainable_aggregator = sum(
    p.numel() for p in aggregator.parameters() if p.requires_grad
)

print("\nTrainable model parameters:", trainable_model)
print("Trainable aggregator parameters:", trainable_aggregator)
print("Total trainable parameters:", trainable_model + trainable_aggregator)

print("\nEncoder frozen:", ckpt["encoder_frozen"])
print("Pilot updates completed:", ckpt["pilot_steps"])

Device: cuda:0
Optimizer: AdamW
Learning rates:
Group 0: LR=2e-05, weight_decay=0.01

Trainable model parameters: 51349504
Trainable aggregator parameters: 2177024
Total trainable parameters: 53526528

Encoder frozen: True
Pilot updates completed: 100


In [105]:
print("train_loader:", "train_loader" in globals())
print("dev_loader:", "dev_loader" in globals())
print("train_eligible:", "train_eligible" in globals())
print("dev_eligible:", "dev_eligible" in globals())

train_loader: True
dev_loader: True
train_eligible: True
dev_eligible: True


In [107]:
import inspect

print("TRAINING FUNCTIONS")
for name in [
    "train_hierarchical_step",
    "train_one_epoch",
    "validate_hierarchical",
    "evaluate_hierarchical",
    "process_document",
    "get_chunk_starts",
    "a4_encode",
    "a4_decode",
]:
    obj = globals().get(name)
    if callable(obj):
        print(f"\n{name}:")
        try:
            print(inspect.signature(obj))
        except Exception:
            print("Signature unavailable")

print("\nDATASET SIZES")
print("Train eligible:", len(train_eligible))
print("Dev eligible:", len(dev_eligible))
print("Train loader batches:", len(train_loader))
print("Dev loader batches:", len(dev_loader))

print("\nGPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

TRAINING FUNCTIONS

get_chunk_starts:
(n)

a4_encode:
(model, encoder_ids)

a4_decode:
(model, decoder_ids, memory, memory_padding_mask=None)

DATASET SIZES
Train eligible: 4432
Dev eligible: 476
Train loader batches: 203
Dev loader batches: 23

GPU memory allocated: 3.1 GB


In [108]:
print("TRAIN LOADER DATASET")
print("Type:", type(train_loader.dataset))
print("Length:", len(train_loader.dataset))

print("\nVALIDATION LOADER DATASET")
print("Type:", type(dev_loader.dataset))
print("Length:", len(dev_loader.dataset))

print("\nCOLLATE FUNCTION")
print("Train:", train_loader.collate_fn)

print("\nBATCH SIZE")
print("Train:", train_loader.batch_size)
print("Validation:", dev_loader.batch_size)

print("\nELIGIBLE DATA SAMPLE")
print("Train keys:", train_eligible[0].keys())
print("Document characters:", len(train_eligible[0]["document"]))
print("Summary characters:", len(train_eligible[0]["summary"]))


TRAIN LOADER DATASET
Type: <class '__main__.LegalSummarizationDataset'>
Length: 203

VALIDATION LOADER DATASET
Type: <class '__main__.LegalSummarizationDataset'>
Length: 23

COLLATE FUNCTION
Train: <function collate_fn at 0x7942ccae6fc0>

BATCH SIZE
Train: 1
Validation: 1

ELIGIBLE DATA SAMPLE
Train keys: dict_keys(['id', 'document', 'summary'])
Document characters: 6596
Summary characters: 883


In [109]:
import inspect

print("=" * 60)
print("DATASET CLASS")
print("=" * 60)
print(inspect.getsource(LegalSummarizationDataset))

print("\n" + "=" * 60)
print("COLLATE FUNCTION")
print("=" * 60)
print(inspect.getsource(collate_fn))

print("\n" + "=" * 60)
print("CHUNKING CONFIGURATION")
print("=" * 60)
print("CHUNK_SIZE:", CHUNK_SIZE)
print("OVERLAP:", OVERLAP)
print("STRIDE:", STRIDE)

print("\n" + "=" * 60)
print("DOCUMENT LENGTH CHECK")
print("=" * 60)

for record in train_eligible[:5]:
    token_count = len(tokenizer.encode(record["document"]).ids)
    print(
        "ID:", record["id"],
        "| Document tokens:", token_count,
        "| Summary tokens:", len(tokenizer.encode(record["summary"]).ids)
    )

DATASET CLASS


OSError: source code not available

In [110]:
print("=" * 55)
print("DATASET CONFIGURATION")
print("=" * 55)

print("Dataset class:", LegalSummarizationDataset)
print("Dataset length:", len(train_loader.dataset))
print("Dataset attributes:", vars(train_loader.dataset).keys())

print("\n" + "=" * 55)
print("COLLATOR CONFIGURATION")
print("=" * 55)

print("Collator:", collate_fn)
print("Collator module:", collate_fn.__module__)

print("\n" + "=" * 55)
print("CHUNKING CONFIGURATION")
print("=" * 55)

print("CHUNK_SIZE:", CHUNK_SIZE)
print("OVERLAP:", OVERLAP)
print("STRIDE:", STRIDE)

print("\n" + "=" * 55)
print("DOCUMENT LENGTH CHECK")
print("=" * 55)

for record in train_eligible[:5]:
    doc_tokens = len(tokenizer.encode(record["document"]).ids)
    summary_tokens = len(tokenizer.encode(record["summary"]).ids)

    print(
        f"ID: {record['id']} | "
        f"Document: {doc_tokens} tokens | "
        f"Summary: {summary_tokens} tokens"
    )

DATASET CONFIGURATION
Dataset class: <class '__main__.LegalSummarizationDataset'>
Dataset length: 203
Dataset attributes: dict_keys(['records', 'tokenizer'])

COLLATOR CONFIGURATION
Collator: <function collate_fn at 0x7942ccae6fc0>
Collator module: __main__

CHUNKING CONFIGURATION
CHUNK_SIZE: 900
OVERLAP: 100
STRIDE: 800

DOCUMENT LENGTH CHECK
ID: 1434 | Document: 1368 tokens | Summary: 189 tokens
ID: 25 | Document: 3645 tokens | Summary: 264 tokens
ID: 1116 | Document: 3939 tokens | Summary: 523 tokens
ID: 2805 | Document: 4187 tokens | Summary: 662 tokens
ID: 1566 | Document: 5489 tokens | Summary: 515 tokens


In [111]:
from torch.utils.data import Dataset, DataLoader

class HierarchicalLegalDataset(Dataset):
    def __init__(self, records, tokenizer):
        self.records = records
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]

        # Tokenize complete document and summary
        doc_ids = self.tokenizer.encode(item["document"]).ids
        summary_ids = self.tokenizer.encode(item["summary"]).ids

        # Split document into overlapping chunks
        starts = get_chunk_starts(len(doc_ids))

        chunks = []
        for start in starts:
            chunk = doc_ids[start:start + CHUNK_SIZE]
            chunk = [BOS_ID] + chunk + [EOS_ID]

            assert len(chunk) <= 1024
            chunks.append(chunk)

        # One summary target per complete document
        decoder_ids = [BOS_ID] + summary_ids
        labels = summary_ids + [EOS_ID]

        assert len(decoder_ids) <= 1024
        assert len(labels) <= 1024

        return {
            "id": item["id"],
            "chunks": chunks,
            "decoder_ids": decoder_ids,
            "labels": labels,
        }


def hierarchical_collate_fn(batch):
    assert len(batch) == 1, "Use batch size 1 for hierarchical training."
    return batch[0]


# Build full-data datasets
hier_train_dataset = HierarchicalLegalDataset(
    train_eligible, tokenizer
)

hier_dev_dataset = HierarchicalLegalDataset(
    dev_eligible, tokenizer
)

# Build loaders
hier_train_loader = DataLoader(
    hier_train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=hierarchical_collate_fn
)

hier_dev_loader = DataLoader(
    hier_dev_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=hierarchical_collate_fn
)

print("Full hierarchical loaders created.")
print("Training documents:", len(hier_train_dataset))
print("Validation documents:", len(hier_dev_dataset))
print("Training batches:", len(hier_train_loader))
print("Validation batches:", len(hier_dev_loader))

Full hierarchical loaders created.
Training documents: 4432
Validation documents: 476
Training batches: 4432
Validation batches: 476


In [112]:
sample = hier_train_dataset[0]

print("Document ID:", sample["id"])
print("Number of chunks:", len(sample["chunks"]))
print("Chunk lengths:", [len(x) for x in sample["chunks"]])
print("Maximum chunk length:", max(map(len, sample["chunks"])))
print("Decoder input length:", len(sample["decoder_ids"]))
print("Label length:", len(sample["labels"]))

# Verify the first batch
batch = next(iter(hier_train_loader))

print("\nFirst batch verification:")
print("ID:", batch["id"])
print("Chunks:", len(batch["chunks"]))
print("All chunks within A4 limit:",
      all(len(x) <= 1024 for x in batch["chunks"]))
print("Target within A4 limit:",
      len(batch["decoder_ids"]) <= 1024)

Document ID: 1434
Number of chunks: 2
Chunk lengths: [902, 902]
Maximum chunk length: 902
Decoder input length: 190
Label length: 190

First batch verification:
ID: 885
Chunks: 3
All chunks within A4 limit: True
Target within A4 limit: True


In [113]:
# Verify optimizer parameter references

optimizer_param_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
}

model_trainable = [
    p for p in model.parameters() if p.requires_grad
]

aggregator_trainable = [
    p for p in aggregator.parameters() if p.requires_grad
]

all_trainable = model_trainable + aggregator_trainable

missing_params = [
    p for p in all_trainable
    if id(p) not in optimizer_param_ids
]

print("Trainable model parameters:", sum(p.numel() for p in model_trainable))
print("Trainable aggregator parameters:", sum(p.numel() for p in aggregator_trainable))
print("Missing from optimizer:", len(missing_params))

print("Model device:", next(model.parameters()).device)
print("Aggregator device:", next(aggregator.parameters()).device)
print("Optimizer learning rate:", optimizer.param_groups[0]["lr"])

assert len(missing_params) == 0, "Some trainable parameters are missing from optimizer!"
assert next(model.parameters()).is_cuda
assert next(aggregator.parameters()).is_cuda

print("\nPreflight passed.")

Trainable model parameters: 51349504
Trainable aggregator parameters: 2177024
Missing from optimizer: 0
Model device: cuda:0
Aggregator device: cuda:0
Optimizer learning rate: 2e-05

Preflight passed.


In [116]:
import os
import torch
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

DEVICE = torch.device("cuda")
MAX_UPDATES = 50
SAVE_EVERY = 10
VALIDATION_COUNT = 30

SAVE_DIR = "/kaggle/working/phase6_hierarchical"
os.makedirs(SAVE_DIR, exist_ok=True)

base_steps = int(ckpt["pilot_steps"])

# --------------------------------------------------
# DOCUMENT-LEVEL FORWARD PASS
# --------------------------------------------------

def hierarchical_document_forward(batch):
    compressed_chunks = []

    # Encoder is frozen: no encoder gradient storage.
    for chunk in batch["chunks"]:
        chunk_ids = torch.tensor(
            [chunk],
            dtype=torch.long,
            device=DEVICE
        )

        with torch.no_grad():
            memory, _ = a4_encode(model, chunk_ids)

        # Compress immediately to 16 vectors.
        compressed = aggregator.compress_chunk(memory)
        compressed_chunks.append(compressed)

        del memory, chunk_ids

    # Combine all chunk representations.
    combined = torch.cat(compressed_chunks, dim=1)

    # Compress the complete document to 128 vectors.
    document_memory = aggregator.aggregate_document(combined)

    decoder_ids = torch.tensor(
        [batch["decoder_ids"]],
        dtype=torch.long,
        device=DEVICE
    )

    labels = torch.tensor(
        [batch["labels"]],
        dtype=torch.long,
        device=DEVICE
    )

    logits = a4_decode(
        model,
        decoder_ids,
        document_memory,
        memory_padding_mask=None
    )

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1)
    )

    return loss


# --------------------------------------------------
# CHECKPOINT FUNCTION
# --------------------------------------------------

def save_hierarchical_checkpoint(step, train_loss, validation_loss):
    path = os.path.join(
        SAVE_DIR,
        f"a4_hierarchical_fulltrain_step_{step}.pt"
    )

    state = {
        "pilot_steps": step,
        "model_state_dict": model.state_dict(),
        "aggregator_state_dict": aggregator.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "chunk_size": CHUNK_SIZE,
        "overlap": OVERLAP,
        "encoder_frozen": True
    }

    torch.save(state, path)
    print(f"Checkpoint saved: {path}")


# --------------------------------------------------
# TRAINING
# --------------------------------------------------

model.train()
aggregator.train()

# Keep the frozen encoder in evaluation mode.
model.encoder_layers.eval()

train_losses = []
grad_norms = []

print("Starting hierarchical training...")
print("Starting from pilot update:", base_steps)

for update, batch in enumerate(hier_train_loader, start=1):

    if update > MAX_UPDATES:
        break

    optimizer.zero_grad(set_to_none=True)

    loss = hierarchical_document_forward(batch)

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Non-finite training loss at update {update}"
        )

    loss.backward()

    grad_norm = clip_grad_norm_(
        [p for p in all_trainable if p.grad is not None],
        max_norm=1.0
    )

    optimizer.step()

    loss_value = loss.item()
    train_losses.append(loss_value)
    grad_norms.append(float(grad_norm))

    del loss

    if update % 5 == 0:
        recent = sum(train_losses[-5:]) / 5

        print(
            f"Update {update}/{MAX_UPDATES} | "
            f"Loss: {loss_value:.4f} | "
            f"5-update mean: {recent:.4f} | "
            f"Grad norm: {float(grad_norm):.4f} | "
            f"GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB"
        )

    if update % SAVE_EVERY == 0:
        save_hierarchical_checkpoint(
            base_steps + update,
            sum(train_losses) / len(train_losses),
            ckpt["validation_loss"]
        )

# --------------------------------------------------
# VALIDATION ON 30 DOCUMENTS
# --------------------------------------------------

model.eval()
aggregator.eval()

validation_losses = []

with torch.no_grad():
    for i, batch in enumerate(hier_dev_loader):
        if i >= VALIDATION_COUNT:
            break

        loss = hierarchical_document_forward(batch)
        validation_losses.append(loss.item())

new_validation_loss = (
    sum(validation_losses) / len(validation_losses)
)

final_train_loss = sum(train_losses) / len(train_losses)
completed_steps = base_steps + len(train_losses)

# Save final state after validation.
save_hierarchical_checkpoint(
    completed_steps,
    final_train_loss,
    new_validation_loss
)

print("\n" + "=" * 55)
print("TRAINING BLOCK COMPLETED")
print("=" * 55)
print("Updates completed:", len(train_losses))
print("Total updates:", completed_steps)
print("Mean training loss:", round(final_train_loss, 4))
print("Mean gradient norm:", round(sum(grad_norms)/len(grad_norms), 4))
print("Validation documents:", len(validation_losses))
print("Validation loss:", round(new_validation_loss, 4))
print("GPU memory:", round(torch.cuda.memory_allocated()/1024**3, 2), "GB")

Starting hierarchical training...
Starting from pilot update: 100
Update 5/50 | Loss: 6.6244 | 5-update mean: 6.2539 | Grad norm: 7.7617 | GPU: 3.10 GB
Update 10/50 | Loss: 6.5139 | 5-update mean: 6.1001 | Grad norm: 13.3297 | GPU: 3.10 GB
Checkpoint saved: /kaggle/working/phase6_hierarchical/a4_hierarchical_fulltrain_step_110.pt
Update 15/50 | Loss: 6.4665 | 5-update mean: 6.3270 | Grad norm: 9.0868 | GPU: 3.10 GB
Update 20/50 | Loss: 4.9452 | 5-update mean: 5.7335 | Grad norm: 9.7604 | GPU: 3.10 GB
Checkpoint saved: /kaggle/working/phase6_hierarchical/a4_hierarchical_fulltrain_step_120.pt
Update 25/50 | Loss: 6.0368 | 5-update mean: 6.2998 | Grad norm: 6.9801 | GPU: 3.10 GB
Update 30/50 | Loss: 5.7685 | 5-update mean: 6.0249 | Grad norm: 8.2420 | GPU: 3.10 GB
Checkpoint saved: /kaggle/working/phase6_hierarchical/a4_hierarchical_fulltrain_step_130.pt
Update 35/50 | Loss: 5.8593 | 5-update mean: 6.1850 | Grad norm: 7.8713 | GPU: 3.10 GB
Update 40/50 | Loss: 6.2529 | 5-update mean: 6.01

In [115]:
from torch.nn.utils import clip_grad_norm_

print("Gradient clipping imported successfully.")

Gradient clipping imported successfully.


In [117]:
import torch

@torch.inference_mode()
def generate_hierarchical_summary(record, max_new_tokens=128):

    model.eval()
    aggregator.eval()

    # Tokenize and split the complete document
    doc_ids = tokenizer.encode(record["document"]).ids
    starts = get_chunk_starts(len(doc_ids))

    compressed_chunks = []

    for start in starts:
        chunk = doc_ids[start:start + CHUNK_SIZE]
        chunk = [BOS_ID] + chunk + [EOS_ID]

        chunk_tensor = torch.tensor(
            [chunk], dtype=torch.long, device=DEVICE
        )

        memory, _ = a4_encode(model, chunk_tensor)
        compressed = aggregator.compress_chunk(memory)
        compressed_chunks.append(compressed)

        del memory, chunk_tensor

    combined = torch.cat(compressed_chunks, dim=1)
    document_memory = aggregator.aggregate_document(combined)

    # Autoregressive generation
    generated = [BOS_ID]

    for _ in range(max_new_tokens):
        decoder_ids = torch.tensor(
            [generated], dtype=torch.long, device=DEVICE
        )

        logits = a4_decode(
            model,
            decoder_ids,
            document_memory,
            memory_padding_mask=None
        )

        next_token = logits[0, -1].argmax().item()

        if next_token == EOS_ID:
            break

        generated.append(next_token)

    return tokenizer.decode(
        generated[1:],
        skip_special_tokens=True
    )


# Evaluate three unseen validation examples
for i in range(3):
    record = dev_eligible[i]

    prediction = generate_hierarchical_summary(record)

    print("\n" + "=" * 70)
    print("DOCUMENT ID:", record["id"])
    print("\nREFERENCE SUMMARY:")
    print(record["summary"][:1500])

    print("\nGENERATED SUMMARY:")
    print(prediction)

    print("\nGenerated tokens:", len(tokenizer.encode(prediction).ids))


DOCUMENT ID: 6827

REFERENCE SUMMARY:
The appellant filed an application before the Central Administrative Tribunal, Patna Bench, for a declaration that the second proviso to rule 4 of Civil Services Examination Rules was violative of Articles 14 and 16 of the Constitu tion of India. By an interim order the Tribunal allowed the appellant to appear at the Civil Services (Main) Examina tion, 1990, subject to the result of the final orders in the original application. The said application was transferred to this Court. In a bunch of similar cases, the Central Administrative Tribunal, Delhi upheld the validity of Rule 4 of Civil Services Examination Rules. In appeal to this Court (Civil Appeal Nos. 5439 52/90)** by an interim order dated 7.12.1990, the appellants therein were allowed to appear in Civil Services (Main) Examination, 1990; and while finally disposing of the appeals, the judgment of CAT, Delhi was affirmed. Dismissing the case of the appellant in view of the judgment in C.As 

In [118]:
import torch
import torch.nn.functional as F

model.eval()
aggregator.eval()

@torch.no_grad()
def get_document_memory(record):
    doc_ids = tokenizer.encode(record["document"]).ids
    starts = get_chunk_starts(len(doc_ids))

    compressed = []

    for start in starts:
        chunk = doc_ids[start:start + CHUNK_SIZE]
        chunk = [BOS_ID] + chunk + [EOS_ID]

        ids = torch.tensor(
            [chunk], dtype=torch.long, device=DEVICE
        )

        memory, _ = a4_encode(model, ids)
        compressed.append(aggregator.compress_chunk(memory))

    return aggregator.aggregate_document(
        torch.cat(compressed, dim=1)
    )


@torch.no_grad()
def get_teacher_forced_logits(record, memory):
    target = tokenizer.encode(record["summary"]).ids

    decoder_ids = torch.tensor(
        [[BOS_ID] + target[:-1]],
        dtype=torch.long,
        device=DEVICE
    )

    labels = torch.tensor(
        [target],
        dtype=torch.long,
        device=DEVICE
    )

    logits = a4_decode(
        model,
        decoder_ids,
        memory,
        memory_padding_mask=None
    )

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1)
    )

    accuracy = (
        logits.argmax(-1) == labels
    ).float().mean().item()

    return loss.item(), accuracy, logits


# Use two different validation documents
record_a = dev_eligible[0]
record_b = dev_eligible[1]

memory_a = get_document_memory(record_a)
memory_b = get_document_memory(record_b)

# Correct memory
loss_correct, acc_correct, logits_correct = (
    get_teacher_forced_logits(record_a, memory_a)
)

# Zero memory
loss_zero, acc_zero, logits_zero = (
    get_teacher_forced_logits(record_a, torch.zeros_like(memory_a))
)

# Wrong document memory
loss_wrong, acc_wrong, logits_wrong = (
    get_teacher_forced_logits(record_a, memory_b)
)

print("DOCUMENT A:", record_a["id"])
print("\nCORRECT MEMORY")
print("Loss:", round(loss_correct, 4))
print("Accuracy:", round(acc_correct, 4))

print("\nZERO MEMORY")
print("Loss:", round(loss_zero, 4))
print("Accuracy:", round(acc_zero, 4))

print("\nWRONG DOCUMENT MEMORY")
print("Loss:", round(loss_wrong, 4))
print("Accuracy:", round(acc_wrong, 4))

print("\nLOGIT DIFFERENCES")
print(
    "Correct vs zero:",
    (logits_correct - logits_zero).abs().mean().item()
)
print(
    "Correct vs wrong:",
    (logits_correct - logits_wrong).abs().mean().item()
)

DOCUMENT A: 6827

CORRECT MEMORY
Loss: 5.9647
Accuracy: 0.2238

ZERO MEMORY
Loss: 9.873
Accuracy: 0.1411

WRONG DOCUMENT MEMORY
Loss: 5.9446
Accuracy: 0.2186

LOGIT DIFFERENCES
Correct vs zero: 2.064467430114746
Correct vs wrong: 0.1397436112165451


In [119]:
import torch
import torch.nn.functional as F

model.eval()
aggregator.eval()

records = dev_eligible[:4]

# Generate memory for each document
memories = [
    get_document_memory(record)
    for record in records
]

# Evaluate every summary against every memory
loss_matrix = torch.zeros(4, 4)

with torch.no_grad():
    for i, record in enumerate(records):

        target = tokenizer.encode(record["summary"]).ids

        decoder_ids = torch.tensor(
            [[BOS_ID] + target[:-1]],
            dtype=torch.long,
            device=DEVICE
        )

        labels = torch.tensor(
            [target],
            dtype=torch.long,
            device=DEVICE
        )

        for j, memory in enumerate(memories):

            logits = a4_decode(
                model,
                decoder_ids,
                memory,
                memory_padding_mask=None
            )

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )

            loss_matrix[i, j] = loss.item()

print("Document IDs:", [r["id"] for r in records])
print("\nRows = reference summaries")
print("Columns = supplied document memories")
print("\nLoss matrix:")
print(loss_matrix)

diagonal = loss_matrix.diag().mean().item()

off_diagonal = (
    loss_matrix.sum() - loss_matrix.diag().sum()
).item() / 12

print("\nCorrect memory mean loss:", round(diagonal, 4))
print("Wrong memory mean loss:", round(off_diagonal, 4))
print("Difference:", round(off_diagonal - diagonal, 4))

Document IDs: ['6827', '1034', '1761', '695']

Rows = reference summaries
Columns = supplied document memories

Loss matrix:
tensor([[5.9647, 5.9446, 5.9659, 5.9607],
        [6.1375, 6.1192, 6.1158, 6.1070],
        [5.8121, 5.8006, 5.8172, 5.8135],
        [6.4527, 6.4329, 6.4481, 6.4348]])

Correct memory mean loss: 6.084
Wrong memory mean loss: 6.0826
Difference: -0.0013


In [120]:
import torch
import torch.nn.functional as F

records = dev_eligible[:4]

memories = [get_document_memory(record) for record in records]

# Each memory: [1, document_queries, d_model]
memory_vectors = [
    memory.squeeze(0).mean(dim=0)
    for memory in memories
]

memory_matrix = torch.stack(memory_vectors)

# Cosine similarity between document memories
normalized = F.normalize(memory_matrix, p=2, dim=1)
similarity = normalized @ normalized.T

print("Document IDs:", [r["id"] for r in records])
print("\nMemory cosine similarity:")
print(similarity.cpu())

print("\nMemory norms:")
for record, memory in zip(records, memories):
    print(record["id"], memory.norm().item())

print("\nMean off-diagonal cosine similarity:")
off_diag = similarity[~torch.eye(4, dtype=torch.bool, device=similarity.device)]
print(off_diag.mean().item())

Document IDs: ['6827', '1034', '1761', '695']

Memory cosine similarity:
tensor([[1.0000, 0.9947, 0.9932, 0.9931],
        [0.9947, 1.0000, 0.9976, 0.9972],
        [0.9932, 0.9976, 1.0000, 0.9970],
        [0.9931, 0.9972, 0.9970, 1.0000]])

Memory norms:
6827 255.9388427734375
1034 255.93905639648438
1761 255.93919372558594
695 255.9385986328125

Mean off-diagonal cosine similarity:
0.9954490661621094


In [121]:
import torch
import torch.nn.functional as F

records = dev_eligible[:4]

encoder_vectors = []
chunk_vectors = []
document_vectors = []

@torch.no_grad()
def inspect_document(record):
    doc_ids = tokenizer.encode(record["document"]).ids
    starts = get_chunk_starts(len(doc_ids))

    first_chunk = doc_ids[starts[0]:starts[0] + CHUNK_SIZE]
    first_chunk = [BOS_ID] + first_chunk + [EOS_ID]

    ids = torch.tensor(
        [first_chunk], dtype=torch.long, device=DEVICE
    )

    encoded, _ = a4_encode(model, ids)
    compressed = aggregator.compress_chunk(encoded)

    all_compressed = []
    for start in starts:
        chunk = doc_ids[start:start + CHUNK_SIZE]
        chunk = [BOS_ID] + chunk + [EOS_ID]

        chunk_ids = torch.tensor(
            [chunk], dtype=torch.long, device=DEVICE
        )
        memory, _ = a4_encode(model, chunk_ids)
        all_compressed.append(aggregator.compress_chunk(memory))

    document_memory = aggregator.aggregate_document(
        torch.cat(all_compressed, dim=1)
    )

    return (
        encoded.mean(dim=1).squeeze(0),
        compressed.mean(dim=1).squeeze(0),
        document_memory.mean(dim=1).squeeze(0)
    )

for record in records:
    enc, chunk, doc = inspect_document(record)
    encoder_vectors.append(enc)
    chunk_vectors.append(chunk)
    document_vectors.append(doc)

def similarity_matrix(vectors):
    x = torch.stack(vectors)
    x = F.normalize(x, dim=1)
    return (x @ x.T).cpu()

print("Encoder output similarity:")
print(similarity_matrix(encoder_vectors))

print("\nChunk-compressed similarity:")
print(similarity_matrix(chunk_vectors))

print("\nFinal document-memory similarity:")
print(similarity_matrix(document_vectors))

Encoder output similarity:
tensor([[1.0000, 0.9878, 0.9889, 0.9828],
        [0.9878, 1.0000, 0.9908, 0.9834],
        [0.9889, 0.9908, 1.0000, 0.9892],
        [0.9828, 0.9834, 0.9892, 1.0000]])

Chunk-compressed similarity:
tensor([[1.0000, 0.9917, 0.9917, 0.9880],
        [0.9917, 1.0000, 0.9933, 0.9893],
        [0.9917, 0.9933, 1.0000, 0.9928],
        [0.9880, 0.9893, 0.9928, 1.0000]])

Final document-memory similarity:
tensor([[1.0000, 0.9947, 0.9932, 0.9931],
        [0.9947, 1.0000, 0.9976, 0.9972],
        [0.9932, 0.9976, 1.0000, 0.9970],
        [0.9931, 0.9972, 0.9970, 1.0000]])


In [122]:
import torch
import torch.nn.functional as F

def compare_similarity(vectors, name):
    x = torch.stack(vectors).float()

    # Ordinary cosine similarity
    normal = F.normalize(x, dim=1)
    ordinary = normal @ normal.T

    # Center representations to remove their shared mean direction
    centered = x - x.mean(dim=0, keepdim=True)
    centered_normal = F.normalize(centered, dim=1)
    centered_sim = centered_normal @ centered_normal.T

    mask = ~torch.eye(len(x), dtype=torch.bool, device=x.device)

    print(f"\n{name}")
    print("Ordinary mean off-diagonal:",
          ordinary[mask].mean().item())
    print("Centered mean off-diagonal:",
          centered_sim[mask].mean().item())
    print("Centered similarity matrix:")
    print(centered_sim.cpu())

compare_similarity(encoder_vectors, "A4 encoder")
compare_similarity(chunk_vectors, "Chunk compression")
compare_similarity(document_vectors, "Document aggregation")


A4 encoder
Ordinary mean off-diagonal: 0.9871624708175659
Centered mean off-diagonal: -0.326970636844635
Centered similarity matrix:
tensor([[ 1.0000, -0.2350, -0.3799, -0.4580],
        [-0.2350,  1.0000, -0.1956, -0.5179],
        [-0.3799, -0.1956,  1.0000, -0.1754],
        [-0.4580, -0.5179, -0.1754,  1.0000]])

Chunk compression
Ordinary mean off-diagonal: 0.9911450147628784
Centered mean off-diagonal: -0.328063040971756
Centered similarity matrix:
tensor([[ 1.0000, -0.1985, -0.3822, -0.5086],
        [-0.1985,  1.0000, -0.2617, -0.4776],
        [-0.3822, -0.2617,  1.0000, -0.1398],
        [-0.5086, -0.4776, -0.1398,  1.0000]])

Document aggregation
Ordinary mean off-diagonal: 0.9954490661621094
Centered mean off-diagonal: -0.3058294951915741
Centered similarity matrix:
tensor([[ 1.0000, -0.3913, -0.5948, -0.5593],
        [-0.3913,  1.0000, -0.0543, -0.1778],
        [-0.5948, -0.0543,  1.0000, -0.0575],
        [-0.5593, -0.1778, -0.0575,  1.0000]])


In [123]:
import torch
import torch.nn.functional as F
import random

N = 32
records = dev_eligible[:N]

# Generate document memories
memories = [get_document_memory(r) for r in records]

# Prepare reference summaries
targets = []
for r in records:
    ids = tokenizer.encode(r["summary"]).ids
    targets.append(ids)

# Evaluate each summary against each supplied memory
losses = torch.zeros(N, N)

model.eval()
aggregator.eval()

with torch.no_grad():
    for i in range(N):
        target = targets[i]

        decoder_ids = torch.tensor(
            [[BOS_ID] + target[:-1]],
            dtype=torch.long,
            device=DEVICE
        )
        labels = torch.tensor(
            [target],
            dtype=torch.long,
            device=DEVICE
        )

        for j in range(N):
            logits = a4_decode(
                model,
                decoder_ids,
                memories[j],
                memory_padding_mask=None
            )

            losses[i, j] = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            ).item()

diagonal = losses.diag().mean().item()

mask = ~torch.eye(N, dtype=torch.bool)
off_diagonal = losses[mask].mean().item()

print("Documents:", N)
print("Correct-memory loss:", diagonal)
print("Wrong-memory loss:", off_diagonal)
print("Wrong - correct:", off_diagonal - diagonal)

Documents: 32
Correct-memory loss: 6.024778366088867
Wrong-memory loss: 6.0283589363098145
Wrong - correct: 0.0035805702209472656


In [124]:
import numpy as np
import torch

N = losses.shape[0]
mask = ~torch.eye(N, dtype=torch.bool, device=losses.device)

# For each summary, compare its correct memory against
# the average of all 31 incorrect memories.
wrong_per_document = losses.masked_fill(
    ~mask, float("nan")
).nanmean(dim=1)

correct_per_document = losses.diag()

improvement = wrong_per_document - correct_per_document

print("Mean improvement:", improvement.mean().item())
print("Median improvement:", improvement.median().item())
print("Documents helped:", (improvement > 0).sum().item(), "/", N)
print("Documents hurt:", (improvement < 0).sum().item(), "/", N)
print("Per-document improvements:")
print(improvement.cpu())

Mean improvement: 0.003580048680305481
Median improvement: 0.004231929779052734
Documents helped: 24 / 32
Documents hurt: 8 / 32
Per-document improvements:
tensor([-0.0029,  0.0004, -0.0086,  0.0126,  0.0168,  0.0091,  0.0080, -0.0013,
         0.0062, -0.0107, -0.0070,  0.0003,  0.0006,  0.0059,  0.0054,  0.0146,
        -0.0008,  0.0019,  0.0031,  0.0042,  0.0114,  0.0043,  0.0047, -0.0014,
         0.0031,  0.0086,  0.0045,  0.0130, -0.0021,  0.0059,  0.0043,  0.0006])


In [125]:
import os
import torch

CHECKPOINT_PATH = (
    "/kaggle/working/phase6_hierarchical/"
    "a4_hierarchical_fulltrain_step_150.pt"
)

assert os.path.exists(CHECKPOINT_PATH), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
aggregator.load_state_dict(checkpoint["aggregator_state_dict"])

model.eval()
aggregator.eval()

print("Checkpoint restored successfully")
print("Training steps:", checkpoint.get("pilot_steps"))
print("Training loss:", checkpoint.get("train_loss"))
print("Validation loss:", checkpoint.get("validation_loss"))
print("Device:", DEVICE)

Checkpoint restored successfully
Training steps: 150
Training loss: 6.112812805175781
Validation loss: 6.037646802266439
Device: cuda


In [126]:
import json
from tqdm.auto import tqdm

MAX_NEW_TOKENS = 128
N_EVAL = 32

eval_records = dev_eligible[:N_EVAL]

@torch.no_grad()
def generate_summary(record, max_new_tokens=128):
    memory = get_document_memory(record)

    generated = [BOS_ID]

    for _ in range(max_new_tokens):
        decoder_ids = torch.tensor(
            [generated],
            dtype=torch.long,
            device=DEVICE
        )

        logits = a4_decode(
            model,
            decoder_ids,
            memory,
            memory_padding_mask=None
        )

        next_logits = logits[0, -1, :].clone()

        # Prevent generation of special tokens that should not appear
        next_logits[PAD_ID] = float("-inf")
        next_logits[BOS_ID] = float("-inf")
        next_logits[MASK_ID] = float("-inf")

        next_token = next_logits.argmax().item()
        generated.append(next_token)

        if next_token == EOS_ID:
            break

    summary = tokenizer.decode(
        generated[1:],
        skip_special_tokens=True
    )

    return summary.strip()


predictions = []

for record in tqdm(eval_records, desc="Generating A4 summaries"):
    predicted_summary = generate_summary(
        record,
        MAX_NEW_TOKENS
    )

    predictions.append({
        "id": str(record["id"]),
        "reference": record["summary"],
        "prediction": predicted_summary
    })

print("Generated summaries:", len(predictions))

Generating A4 summaries:   0%|          | 0/32 [00:00<?, ?it/s]

Generated summaries: 32


In [127]:
for item in predictions[:5]:
    print("=" * 80)
    print("ID:", item["id"])
    print("\nREFERENCE:")
    print(item["reference"][:1000])
    print("\nPREDICTION:")
    print(item["prediction"][:1000])
    print()

ID: 6827

REFERENCE:
The appellant filed an application before the Central Administrative Tribunal, Patna Bench, for a declaration that the second proviso to rule 4 of Civil Services Examination Rules was violative of Articles 14 and 16 of the Constitu tion of India. By an interim order the Tribunal allowed the appellant to appear at the Civil Services (Main) Examina tion, 1990, subject to the result of the final orders in the original application. The said application was transferred to this Court. In a bunch of similar cases, the Central Administrative Tribunal, Delhi upheld the validity of Rule 4 of Civil Services Examination Rules. In appeal to this Court (Civil Appeal Nos. 5439 52/90)** by an interim order dated 7.12.1990, the appellants therein were allowed to appear in Civil Services (Main) Examination, 1990; and while finally disposing of the appeals, the judgment of CAT, Delhi was affirmed. Dismissing the case of the appellant in view of the judgment in C.As Nos. 5439 52/90,**

In [128]:
OUTPUT_PATH = "/kaggle/working/a4_hierarchical_32_predictions.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_PATH)
print("Number of predictions:", len(predictions))

Saved: /kaggle/working/a4_hierarchical_32_predictions.json
Number of predictions: 32


In [129]:
# Install only if the package is missing
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("rouge_score") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "rouge-score"
    ])

from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

rouge1_scores = []
rougeL_scores = []

for item in predictions:
    reference = item["reference"]
    prediction = item["prediction"]

    scores = scorer.score(reference, prediction)

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

print("A4 HIERARCHICAL — 32 DOCUMENT EVALUATION")
print("-" * 55)
print("Documents:", len(predictions))
print("ROUGE-1 F1:", np.mean(rouge1_scores))
print("ROUGE-L F1:", np.mean(rougeL_scores))

A4 HIERARCHICAL — 32 DOCUMENT EVALUATION
-------------------------------------------------------
Documents: 32
ROUGE-1 F1: 0.20841236970674965
ROUGE-L F1: 0.16494600418184407


In [130]:
import importlib.util

print("bert_score installed:",
      importlib.util.find_spec("bert_score") is not None)

bert_score installed: False


In [131]:
from bert_score import score

references = [x["reference"] for x in predictions]
candidates = [x["prediction"] for x in predictions]

P, R, F1 = score(
    candidates,
    references,
    lang="en",
    verbose=True,
    batch_size=4,
    device="cuda"
)

print("BERTScore Precision:", P.mean().item())
print("BERTScore Recall:", R.mean().item())
print("BERTScore F1:", F1.mean().item())

ModuleNotFoundError: No module named 'bert_score'

In [132]:
import json
import numpy as np

a4_results = {
    "experiment": "A4_Hierarchical",
    "checkpoint_step": 150,
    "evaluation_documents": 32,
    "generation_method": "Greedy",
    "max_new_tokens": 128,
    "rouge1_f1": float(np.mean(rouge1_scores)),
    "rougeL_f1": float(np.mean(rougeL_scores)),
    "bert_score_f1": None,
    "bert_score_status": (
        "Not calculated: approved pretrained evaluation "
        "model not available"
    ),
    "generation_quality": "Poor - repetitive output"
}

path = "/kaggle/working/a4_hierarchical_metrics.json"

with open(path, "w", encoding="utf-8") as f:
    json.dump(a4_results, f, indent=2)

print(json.dumps(a4_results, indent=2))
print("\nSaved:", path)

{
  "experiment": "A4_Hierarchical",
  "checkpoint_step": 150,
  "evaluation_documents": 32,
  "generation_method": "Greedy",
  "max_new_tokens": 128,
  "rouge1_f1": 0.20841236970674965,
  "rougeL_f1": 0.16494600418184407,
  "bert_score_f1": null,
  "bert_score_status": "Not calculated: approved pretrained evaluation model not available",
  "generation_quality": "Poor - repetitive output"
}

Saved: /kaggle/working/a4_hierarchical_metrics.json


In [133]:
import os
import torch

A1_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "architecture1-sustained-step-500-pt/"
    "architecture1_sustained_step_500.pt"
)

assert os.path.exists(A1_PATH), "A1 checkpoint not found"

a1_checkpoint = torch.load(
    A1_PATH,
    map_location="cpu",
    weights_only=False
)

print("CHECKPOINT TYPE:", type(a1_checkpoint))

if isinstance(a1_checkpoint, dict):
    print("\nCHECKPOINT KEYS:")
    for key, value in a1_checkpoint.items():
        if isinstance(value, torch.Tensor):
            print(key, value.shape, value.dtype)
        elif isinstance(value, dict):
            print(key, "Dictionary:", len(value), "entries")
        else:
            print(key, str(value)[:200])

    print("\nMODEL STATE DICTIONARY CANDIDATES:")
    for key, value in a1_checkpoint.items():
        if isinstance(value, dict):
            tensor_count = sum(
                isinstance(v, torch.Tensor)
                for v in value.values()
            )
            if tensor_count:
                print(key, "Tensor entries:", tensor_count)
                for name, tensor in list(value.items())[:12]:
                    if isinstance(tensor, torch.Tensor):
                        print(" ", name, tuple(tensor.shape))

CHECKPOINT TYPE: <class 'dict'>

CHECKPOINT KEYS:
step 500
epoch 0
train_loss 5.366012096405029
val_loss None
model_state_dict Dictionary: 101 entries
optimizer_state_dict Dictionary: 2 entries

MODEL STATE DICTIONARY CANDIDATES:
model_state_dict Tensor entries: 101
  token_embedding.weight (50000, 512)
  position_embedding.weight (2048, 512)
  blocks.0.norm1.weight (512,)
  blocks.0.norm1.bias (512,)
  blocks.0.attn.in_proj_weight (1536, 512)
  blocks.0.attn.in_proj_bias (1536,)
  blocks.0.attn.out_proj.weight (512, 512)
  blocks.0.attn.out_proj.bias (512,)
  blocks.0.norm2.weight (512,)
  blocks.0.norm2.bias (512,)
  blocks.0.ffn.0.weight (2048, 512)
  blocks.0.ffn.0.bias (2048,)


In [134]:
import torch

state = a1_checkpoint["model_state_dict"]

print("ALL A1 MODEL PARAMETERS")
print("=" * 75)

total_params = 0

for name, tensor in state.items():
    count = tensor.numel()
    total_params += count
    print(f"{name:55s} {str(tuple(tensor.shape)):20s} {count:,}")

print("\nTOTAL PARAMETERS:", f"{total_params:,}")
print("TOTAL PARAMETERS (M):", round(total_params / 1e6, 3))

print("\nTRANSFORMER BLOCKS:")
blocks = sorted({
    int(name.split(".")[1])
    for name in state
    if name.startswith("blocks.")
})
print(blocks)
print("Number of blocks:", len(blocks))

print("\nEMBEDDING / OUTPUT WEIGHTS:")
for name in state:
    if any(x in name.lower() for x in [
        "embedding", "lm_head", "output", "final", "norm"
    ]):
        print(name, tuple(state[name].shape))

ALL A1 MODEL PARAMETERS
token_embedding.weight                                  (50000, 512)         25,600,000
position_embedding.weight                               (2048, 512)          1,048,576
blocks.0.norm1.weight                                   (512,)               512
blocks.0.norm1.bias                                     (512,)               512
blocks.0.attn.in_proj_weight                            (1536, 512)          786,432
blocks.0.attn.in_proj_bias                              (1536,)              1,536
blocks.0.attn.out_proj.weight                           (512, 512)           262,144
blocks.0.attn.out_proj.bias                             (512,)               512
blocks.0.norm2.weight                                   (512,)               512
blocks.0.norm2.bias                                     (512,)               512
blocks.0.ffn.0.weight                                   (2048, 512)          1,048,576
blocks.0.ffn.0.bias                                     

In [135]:
import os

ROOT = "/kaggle/input/datasets/aq1de3"

print("TOKENIZER AND CONFIGURATION FILES")
print("=" * 65)

for root, dirs, files in os.walk(ROOT):
    for filename in files:
        lower = filename.lower()

        if any(key in lower for key in [
            "tokenizer", "vocab", "merges",
            "config", "architecture1", "a1"
        ]):
            path = os.path.join(root, filename)
            print(path)

TOKENIZER AND CONFIGURATION FILES
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
/kaggle/input/datasets/aq1de3/phase3/label_vocab.json


In [136]:
state = a1_checkpoint["model_state_dict"]

print(
    "Embedding and LM head identical:",
    torch.equal(
        state["token_embedding.weight"],
        state["lm_head.weight"]
    )
)

print("Embedding shape:", state["token_embedding.weight"].shape)
print("LM head shape:", state["lm_head.weight"].shape)

Embedding and LM head identical: True
Embedding shape: torch.Size([50000, 512])
LM head shape: torch.Size([50000, 512])


In [137]:
from tokenizers import Tokenizer
import json
import os

TOKENIZER_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "tokenizer-json/tokenizer.json"
)

tok = Tokenizer.from_file(TOKENIZER_PATH)

print("TOKENIZER INSPECTION")
print("=" * 60)

print("Vocabulary size:", tok.get_vocab_size())
print("Tokenizer model:", tok.model)
print("Pre-tokenizer:", tok.pre_tokenizer)
print("Decoder:", tok.decoder)

print("\nSPECIAL TOKEN IDS")
for token in ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]:
    print(token, "=>", tok.token_to_id(token))

print("\nSAMPLE TOKENIZATION")
samples = [
    "The court held that the appellant was liable.",
    "This agreement shall be governed by law.",
    "Supreme Court of India"
]

for sentence in samples:
    encoded = tok.encode(sentence)
    print("\nText:", sentence)
    print("Token IDs:", encoded.ids[:30])
    print("Tokens:", encoded.tokens[:30])

print("\nCHECKPOINT WEIGHT TIE")
print(
    "Weights identical:",
    torch.equal(
        state["token_embedding.weight"],
        state["lm_head.weight"]
    )
)
print(
    "Same underlying storage:",
    state["token_embedding.weight"].data_ptr()
    == state["lm_head.weight"].data_ptr()
)

TOKENIZER INSPECTION
Vocabulary size: 50000
Tokenizer model: BPE(dropout=None, unk_token="[UNK]", continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"[PAD]":0, "[UNK]":1, "[BOS]":2, "[EOS]":3, "[MASK]":4, ...}, merges=[("Ġ", "t"), ("h", "e"), ("Ġt", "he"), ("Ġ", "a"), ("o", "n"), ...])
Pre-tokenizer: ByteLevel(add_prefix_space=False, trim_offsets=True, use_regex=True)
Decoder: None

SPECIAL TOKEN IDS
[PAD] => 0
[UNK] => 1
[BOS] => 2
[EOS] => 3
[MASK] => 4

SAMPLE TOKENIZATION

Text: The court held that the appellant was liable.
Token IDs: [550, 777, 956, 302, 238, 553, 343, 2182, 18]
Tokens: ['The', 'Ġcourt', 'Ġheld', 'Ġthat', 'Ġthe', 'Ġappellant', 'Ġwas', 'Ġliable', '.']

Text: This agreement shall be governed by law.
Token IDs: [1909, 2108, 544, 284, 6704, 309, 835, 18]
Tokens: ['This', 'Ġagreement', 'Ġshall', 'Ġbe', 'Ġgoverned', 'Ġby', 'Ġlaw', '.']

Text: Supreme Court of India
Token IDs: [27951, 430, 247, 787]
T

In [138]:
from tokenizers import Tokenizer, decoders
import torch

# Configure the decoder
tok.decoder = decoders.ByteLevel()

# Verify round-trip behavior
samples = [
    "The court held that the appellant was liable.",
    "This agreement shall be governed by law.",
    "Supreme Court of India"
]

print("TOKENIZER ROUND-TRIP TEST")
for sentence in samples:
    ids = tok.encode(sentence).ids
    decoded = tok.decode(ids)
    print("\nOriginal:", sentence)
    print("Decoded :", decoded)
    print("Match   :", sentence == decoded)

# Check model architecture dimensions
state = a1_checkpoint["model_state_dict"]

print("\nA1 CHECKPOINT VALIDATION")
print("Blocks:", len(blocks))
print("Hidden dimension:", state["token_embedding.weight"].shape[1])
print("Vocabulary:", state["token_embedding.weight"].shape[0])
print("Context length:", state["position_embedding.weight"].shape[0])
print("Embedding tied:", torch.equal(
    state["token_embedding.weight"],
    state["lm_head.weight"]
))

# Check whether all parameters contain finite values
all_finite = all(torch.isfinite(v).all().item() for v in state.values())
print("All checkpoint parameters finite:", all_finite)

TOKENIZER ROUND-TRIP TEST

Original: The court held that the appellant was liable.
Decoded : The court held that the appellant was liable.
Match   : True

Original: This agreement shall be governed by law.
Decoded : This agreement shall be governed by law.
Match   : True

Original: Supreme Court of India
Decoded : Supreme Court of India
Match   : True

A1 CHECKPOINT VALIDATION
Blocks: 8
Hidden dimension: 512
Vocabulary: 50000
Context length: 2048
Embedding tied: True
All checkpoint parameters finite: True


In [139]:
import os

SEARCH_ROOTS = [
    "/kaggle/input/datasets/aq1de3",
    "/kaggle/working"
]

extensions = (".py", ".ipynb", ".json", ".yaml", ".yml", ".txt")

keywords = [
    "Architecture1",
    "architecture1",
    "class TransformerBlock",
    "class CausalSelfAttention",
    "position_embedding",
    "sustained"
]

print("SEARCHING FOR A1 SOURCE CODE")
print("=" * 65)

for base in SEARCH_ROOTS:
    for root, dirs, files in os.walk(base):
        # Avoid unnecessary large directory traversal
        dirs[:] = [d for d in dirs if d not in [
            ".git", "__pycache__"
        ]]

        for filename in files:
            if filename.endswith(extensions):
                path = os.path.join(root, filename)
                print(path)

SEARCHING FOR A1 SOURCE CODE
/kaggle/input/datasets/aq1de3/trainjson/train.json
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
/kaggle/input/datasets/aq1de3/testtest/test (3).json
/kaggle/input/datasets/aq1de3/traintr/train (2).json
/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
/kaggle/input/datasets/aq1de3/phase3/CL_dev.json
/kaggle/input/datasets/aq1de3/phase3/CL_test.json
/kaggle/input/datasets/aq1de3/phase3/CL_train.json
/kaggle/input/datasets/aq1de3/phase3/IT_dev.json
/kaggle/input/datasets/aq1de3/phase3/IT_test.json
/kaggle/input/datasets/aq1de3/phase3/IT_train.json
/kaggle/input/datasets/aq1de3/phase3/label_vocab.json
/kaggle/input/datasets/aq1de3/p1p1p1/a2a2a2.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a4a4a4.ipynb
/kaggle/input/datasets/aq1de3/devjson/dev.json
/kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (

In [140]:
import json
import os

NOTEBOOK_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "phase1-model/phase1_model_code_extract.ipynb"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("TOTAL CELLS:", len(nb["cells"]))

keywords = [
    "Architecture 1",
    "Architecture1",
    "architecture1",
    "class ",
    "position_embedding",
    "MultiheadAttention",
    "tokenizer",
    "num_heads",
    "n_heads",
    "causal"
]

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    if any(k.lower() in source.lower() for k in keywords):
        print("\n" + "=" * 80)
        print("CELL:", i, "| TYPE:", cell["cell_type"])
        print("=" * 80)
        print(source[:12000])

TOTAL CELLS: 13

CELL: 2 | TYPE: code
# ============================================================
# PHASE 1 — ARCHITECTURE 1
# DECODER-ONLY TRANSFORMER
# MODEL CONSTRUCTION & RESOURCE CHECK
# ============================================================

import torch
from tokenizers import Tokenizer
from transformers import GPT2Config, GPT2LMHeadModel

print("=" * 70)
print("PHASE 1 — ARCHITECTURE 1: DECODER-ONLY TRANSFORMER")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

VOCAB_SIZE = tokenizer.get_vocab_size()

print(f"\nVocabulary size : {VOCAB_SIZE:,}")

# --------------------------------------------------------

In [141]:
import os
import json

ROOT = "/kaggle/input/datasets/aq1de3"

notebooks = [
    os.path.join(ROOT, "notebook1/notebook1d0fcbb0df (3).ipynb"),
    os.path.join(ROOT, "phase1-model/phase1_model_code_extract.ipynb"),
]

keywords = [
    "class TransformerBlock",
    "class GPT",
    "class LanguageModel",
    "class Decoder",
    "class Causal",
    "token_embedding",
    "position_embedding",
    "architecture 1",
    "architecture1",
    "architecture 1 — decoder"
]

for path in notebooks:
    print("\n" + "=" * 90)
    print("NOTEBOOK:", path)
    print("=" * 90)

    with open(path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    for i, cell in enumerate(nb["cells"]):
        source = "".join(cell.get("source", []))

        if any(k.lower() in source.lower() for k in keywords):
            print(f"\n--- CELL {i} ({cell['cell_type']}) ---")
            print(source[:10000])


NOTEBOOK: /kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb

--- CELL 9 (code) ---
# ============================================================
# PHASE 1 — ARCHITECTURE 1
# DECODER-ONLY TRANSFORMER
# MODEL CONSTRUCTION & RESOURCE CHECK
# ============================================================

import torch
from tokenizers import Tokenizer
from transformers import GPT2Config, GPT2LMHeadModel

print("=" * 70)
print("PHASE 1 — ARCHITECTURE 1: DECODER-ONLY TRANSFORMER")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

VOCAB_SIZE = tokenizer.get_vocab_size()

print(f"\nVocabulary size : {VOCAB_SIZE

In [142]:
import json

path = (
    "/kaggle/input/datasets/aq1de3/"
    "notebook1/notebook1d0fcbb0df (3).ipynb"
)

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "class ",
    "nn.MultiheadAttention",
    "scaled_dot_product_attention",
    "causal_mask",
    "attn.in_proj",
    "GELU",
    "GPT2LMHeadModel"
]

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    if any(k.lower() in source.lower() for k in keywords):
        print("\n" + "=" * 75)
        print("CELL:", i)
        print("=" * 75)
        print(source[:8000])


CELL: 5
# ============================================================
# PHASE 1 — GPU SMOKE TEST
# ============================================================

import torch
import time
from transformers import GPT2Config, GPT2LMHeadModel

print("=" * 70)
print("PHASE 1 — GPU SMOKE TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"Device          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

if device.type != "cuda":
    raise RuntimeError("GPU is not available. Stop here.")

# ------------------------------------------------------------
# 2. Small test model
# ------------------------------------------------------------
config = GPT2Config(
    vocab_size=50_000,

In [143]:
import os

paths = [
    "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",

    "/kaggle/working/phase1_production/architecture_1_decoder_only/checkpoints/step_00080000.pt",

    "/kaggle/input/datasets/aq1de3/phase1-model/architecture_1_decoder_only/checkpoints/step_00080000.pt",
]

print("A1 CHECKPOINT AVAILABILITY")
print("=" * 65)

for path in paths:
    print(("FOUND  " if os.path.exists(path) else "MISSING"), path)

A1 CHECKPOINT AVAILABILITY
FOUND   /kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
MISSING /kaggle/working/phase1_production/architecture_1_decoder_only/checkpoints/step_00080000.pt
MISSING /kaggle/input/datasets/aq1de3/phase1-model/architecture_1_decoder_only/checkpoints/step_00080000.pt


In [144]:
import torch
from collections import OrderedDict

A1_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "architecture1-sustained-step-500-pt/"
    "architecture1_sustained_step_500.pt"
)

checkpoint = torch.load(
    A1_PATH,
    map_location="cpu",
    weights_only=False
)

print("CHECKPOINT TYPE:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCHECKPOINT KEYS:")
    for key in checkpoint.keys():
        print(" ", key)

    print("\nMETADATA:")
    for key, value in checkpoint.items():
        if key != "model_state_dict" and not isinstance(value, (dict, OrderedDict)):
            print(f"{key}: {value}")

    state_dict = checkpoint.get("model_state_dict", checkpoint)

    print("\nMODEL STATE DICT:")
    print("Number of tensors:", len(state_dict))

    total_params = 0
    for name, tensor in state_dict.items():
        if torch.is_tensor(tensor):
            count = tensor.numel()
            total_params += count
            print(f"{name:55s} {str(tuple(tensor.shape)):25s} {count:,}")

    print("\nTOTAL PARAMETERS:", f"{total_params:,}")

CHECKPOINT TYPE: <class 'dict'>

CHECKPOINT KEYS:
  step
  epoch
  train_loss
  val_loss
  model_state_dict
  optimizer_state_dict

METADATA:
step: 500
epoch: 0
train_loss: 5.366012096405029
val_loss: None

MODEL STATE DICT:
Number of tensors: 101
token_embedding.weight                                  (50000, 512)              25,600,000
position_embedding.weight                               (2048, 512)               1,048,576
blocks.0.norm1.weight                                   (512,)                    512
blocks.0.norm1.bias                                     (512,)                    512
blocks.0.attn.in_proj_weight                            (1536, 512)               786,432
blocks.0.attn.in_proj_bias                              (1536,)                   1,536
blocks.0.attn.out_proj.weight                           (512, 512)                262,144
blocks.0.attn.out_proj.bias                             (512,)                    512
blocks.0.norm2.weight                    

In [145]:
import os
import json

ROOTS = [
    "/kaggle/working",
    "/kaggle/input/datasets/aq1de3"
]

KEYWORDS = [
    "class A1",
    "class TransformerBlock",
    "class DecoderBlock",
    "position_embedding",
    "in_proj_weight",
    "token_embedding"
]

matches = []

for root in ROOTS:
    if not os.path.exists(root):
        continue

    for current, dirs, files in os.walk(root):
        # Avoid scanning irrelevant hidden/cache directories
        dirs[:] = [
            d for d in dirs
            if not d.startswith(".") and d not in ["__pycache__"]
        ]

        for filename in files:
            path = os.path.join(current, filename)

            if filename.endswith(".py"):
                try:
                    with open(path, "r", encoding="utf-8", errors="ignore") as f:
                        content = f.read()
                except Exception:
                    continue

                found = [k for k in KEYWORDS if k in content]
                if found:
                    matches.append((path, found))

            elif filename.endswith(".ipynb"):
                try:
                    with open(path, "r", encoding="utf-8") as f:
                        notebook = json.load(f)

                    for i, cell in enumerate(notebook.get("cells", [])):
                        source = "".join(cell.get("source", []))
                        found = [k for k in KEYWORDS if k in source]

                        if found:
                            matches.append(
                                (f"{path} [cell {i}]", found)
                            )
                except Exception:
                    continue

print("POSSIBLE A1 IMPLEMENTATIONS")
print("=" * 80)

for path, found in matches[:100]:
    print(path)
    print("  Matches:", ", ".join(found))

print("\nTotal matching files/cells:", len(matches))

POSSIBLE A1 IMPLEMENTATIONS
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb [cell 4]
  Matches: position_embedding, token_embedding
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb [cell 6]
  Matches: position_embedding, token_embedding
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb [cell 8]
  Matches: class DecoderBlock, token_embedding
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb [cell 12]
  Matches: position_embedding, token_embedding
/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb [cell 6]
  Matches: position_embedding, token_embedding
/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb [cell 11]
  Matches: position_embedding, token_embedding
/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb [cell 16]
  Matches: class DecoderBlock
/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb [cell 39]
  Matches: in_proj_weight
/kaggle/input/

In [146]:
import json

NOTEBOOK_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "p2p2p2/phase-2 (1).ipynb"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

cells_to_inspect = [25, 26, 28, 29, 30, 139, 141]

for idx in cells_to_inspect:
    cell = nb["cells"][idx]
    source = "".join(cell.get("source", []))

    print("\n" + "=" * 90)
    print(f"CELL {idx} | {cell.get('cell_type')}")
    print("=" * 90)
    print(source)


CELL 25 | code
# ============================================================
# PHASE 2 — A1 CHECKPOINT RECONSTRUCTION
# CPU ONLY
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

A1_PATH = "/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt"

ckpt = torch.load(
    A1_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model_state_dict"]

print("=" * 70)
print("A1 CHECKPOINT")
print("=" * 70)

# ------------------------------------------------------------
# Infer architecture directly from checkpoint
# ------------------------------------------------------------

vocab_size = state["token_embedding.weight"].shape[0]
hidden_size = state["token_embedding.weight"].shape[1]
context_length = state["position_embedding.weight"].shape[0]

layer_ids = sorted({
    int(k.split(".")[1])
    for k in state
    if k.startswith("blocks.") and k.split(".")[1].isdigit()
})

num_

In [147]:
import torch
import torch.nn as nn

A1_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "architecture1-sustained-step-500-pt/"
    "architecture1_sustained_step_500.pt"
)

ckpt = torch.load(
    A1_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model_state_dict"]

# Infer configuration from checkpoint
VOCAB_SIZE = state["token_embedding.weight"].shape[0]
D_MODEL = state["token_embedding.weight"].shape[1]
MAX_SEQ_LEN = state["position_embedding.weight"].shape[0]
N_LAYERS = len([
    k for k in state
    if k.startswith("blocks.") and k.endswith(".norm1.weight")
])
FFN_DIM = state["blocks.0.ffn.0.weight"].shape[0]
N_HEADS = 8

class A1Block(nn.Module):
    def __init__(self):
        super().__init__()

        self.norm1 = nn.LayerNorm(D_MODEL)
        self.attn = nn.MultiheadAttention(
            D_MODEL, N_HEADS, batch_first=True
        )

        self.norm2 = nn.LayerNorm(D_MODEL)
        self.ffn = nn.Sequential(
            nn.Linear(D_MODEL, FFN_DIM),
            nn.GELU(),
            nn.Linear(FFN_DIM, D_MODEL)
        )

    def forward(self, x, causal_mask):
        h = self.norm1(x)

        attn_out, _ = self.attn(
            h, h, h,
            attn_mask=causal_mask,
            need_weights=False
        )

        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x


class A1Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.position_embedding = nn.Embedding(MAX_SEQ_LEN, D_MODEL)

        self.blocks = nn.ModuleList([
            A1Block() for _ in range(N_LAYERS)
        ])

        self.final_norm = nn.LayerNorm(D_MODEL)

        self.lm_head = nn.Linear(
            D_MODEL, VOCAB_SIZE, bias=False
        )

    def forward(self, input_ids):
        B, T = input_ids.shape

        if T > MAX_SEQ_LEN:
            raise ValueError(f"Sequence length {T} exceeds {MAX_SEQ_LEN}")

        positions = torch.arange(T, device=input_ids.device)

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)[None, :, :]
        )

        causal_mask = torch.triu(
            torch.ones(
                T, T,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        for block in self.blocks:
            x = block(x, causal_mask)

        x = self.final_norm(x)
        return self.lm_head(x)


# Build and load
a1_model = A1Model()

print("Model parameters:",
      f"{sum(p.numel() for p in a1_model.parameters()):,}")

result = a1_model.load_state_dict(state, strict=True)

print("Strict loading:", result)

# Forward test
a1_model.eval()

test_ids = torch.randint(
    0, VOCAB_SIZE, (1, 128)
)

with torch.no_grad():
    logits = a1_model(test_ids)

print("Input shape :", tuple(test_ids.shape))
print("Output shape:", tuple(logits.shape))
print("Finite logits:", torch.isfinite(logits).all().item())

assert logits.shape == (1, 128, VOCAB_SIZE)
assert torch.isfinite(logits).all()

print("\nA1 CHECKPOINT LOAD AND FORWARD TEST PASSED")

Model parameters: 77,468,672
Strict loading: <All keys matched successfully>
Input shape : (1, 128)
Output shape: (1, 128, 50000)
Finite logits: True

A1 CHECKPOINT LOAD AND FORWARD TEST PASSED


In [148]:
import os
import hashlib
from tokenizers import Tokenizer

CANDIDATE_PATHS = {
    "A1_original_working": (
        "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
    ),
    "A1_old_input": (
        "/kaggle/input/datasets/mathimohankowshik/"
        "tokenizer/tokenizer.json"
    ),
    "Phase6": (
        "/kaggle/input/datasets/aq1de3/"
        "tokenizer-json/tokenizer.json"
    ),
}

loaded_tokenizers = {}

for name, path in CANDIDATE_PATHS.items():
    print(f"\n{name}: {path}")

    if not os.path.isfile(path):
        print("  MISSING")
        continue

    with open(path, "rb") as f:
        sha256 = hashlib.sha256(f.read()).hexdigest()

    tok = Tokenizer.from_file(path)
    loaded_tokenizers[name] = tok

    print("  FOUND")
    print("  Vocabulary:", tok.get_vocab_size())
    print("  SHA256:", sha256)

    for token in ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]:
        print(f"  {token}: {tok.token_to_id(token)}")

# Compare complete token-to-ID mappings
if "A1_original_working" in loaded_tokenizers:
    reference_name = "A1_original_working"
elif "A1_old_input" in loaded_tokenizers:
    reference_name = "A1_old_input"
else:
    reference_name = None

if reference_name:
    reference_vocab = loaded_tokenizers[reference_name].get_vocab()

    for name, tok in loaded_tokenizers.items():
        vocab = tok.get_vocab()

        print(f"\nComparison: {reference_name} vs {name}")
        print("Same vocabulary mapping:", reference_vocab == vocab)
else:
    print("\nOriginal A1 tokenizer not found.")
    print("Cannot establish token-ID compatibility yet.")


A1_original_working: /kaggle/working/legal_bpe_tokenizer/tokenizer.json
  MISSING

A1_old_input: /kaggle/input/datasets/mathimohankowshik/tokenizer/tokenizer.json
  MISSING

Phase6: /kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
  FOUND
  Vocabulary: 50000
  SHA256: 61465ad788ec819ea768b54fe70de1be02bc948b035de6065fd5433ae238e09e
  [PAD]: 0
  [UNK]: 1
  [BOS]: 2
  [EOS]: 3
  [MASK]: 4

Original A1 tokenizer not found.
Cannot establish token-ID compatibility yet.


In [149]:
import os
from tokenizers import Tokenizer

ROOT = "/kaggle/input/datasets/aq1de3"

print("SEARCHING FOR TOKENIZER FILES")
print("=" * 65)

candidates = []

for current, dirs, files in os.walk(ROOT):
    if "tokenizer.json" in files:
        path = os.path.join(current, "tokenizer.json")
        candidates.append(path)
        print(path)

print("\nTotal tokenizer files:", len(candidates))

# Compare each candidate with the Phase 6 tokenizer
phase6_path = (
    "/kaggle/input/datasets/aq1de3/"
    "tokenizer-json/tokenizer.json"
)

phase6_tok = Tokenizer.from_file(phase6_path)
phase6_vocab = phase6_tok.get_vocab()

print("\nTOKENIZER COMPATIBILITY")
print("=" * 65)

for path in candidates:
    try:
        tok = Tokenizer.from_file(path)
        vocab = tok.get_vocab()

        print("\nCandidate:", path)
        print("Vocabulary size:", tok.get_vocab_size())
        print("Same token-to-ID mapping:", vocab == phase6_vocab)

    except Exception as e:
        print("ERROR:", path, str(e))

SEARCHING FOR TOKENIZER FILES
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json

Total tokenizer files: 1

TOKENIZER COMPATIBILITY

Candidate: /kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
Vocabulary size: 50000
Same token-to-ID mapping: True


In [150]:
import os
import gc
import json
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert torch.cuda.is_available(), "Enable the Kaggle GPU."
assert "a1_model" in globals(), "Run the A1 reconstruction cell first."
assert "tokenizer" in globals(), "Load the Phase 6 tokenizer first."
assert "train_eligible" in globals(), "Training data not found."
assert "dev_eligible" in globals(), "Validation data not found."

PAD_ID = 0
BOS_ID = 2
EOS_ID = 3

VOCAB_SIZE = 50000
MAX_SEQ_LEN = 2048

# Keep prompt and summary within the model's context.
MAX_DOC_TOKENS = 1200
MAX_SUMMARY_TOKENS = 800

OUTPUT_DIR = "/kaggle/working/phase6_a1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("Training documents:", len(train_eligible))
print("Validation documents:", len(dev_eligible))
print("Tokenizer vocabulary:", tokenizer.get_vocab_size())
print("A1 parameters:", sum(p.numel() for p in a1_model.parameters()))

assert tokenizer.get_vocab_size() == VOCAB_SIZE
assert a1_model.token_embedding.weight.shape[0] == VOCAB_SIZE

print("\nA1 Phase 6 setup ready.")

Device: cuda
GPU: Tesla T4
Training documents: 4432
Validation documents: 476
Tokenizer vocabulary: 50000
A1 parameters: 77468672

A1 Phase 6 setup ready.


In [151]:
class A1SummarizationDataset(Dataset):
    def __init__(self, records):
        self.examples = []

        for item in records:
            doc = tokenizer.encode(item["document"]).ids
            summary = tokenizer.encode(item["summary"]).ids

            doc = doc[:MAX_DOC_TOKENS]
            summary = summary[:MAX_SUMMARY_TOKENS]

            if not doc or not summary:
                continue

            prompt = [BOS_ID] + doc + [EOS_ID]
            full_ids = prompt + summary + [EOS_ID]

            labels = (
                [-100] * len(prompt)
                + summary
                + [EOS_ID]
            )

            assert len(full_ids) == len(labels)
            assert len(full_ids) <= MAX_SEQ_LEN

            self.examples.append({
                "input_ids": full_ids,
                "labels": labels
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def a1_collate(batch):
    max_len = max(len(x["input_ids"]) for x in batch)

    input_ids = []
    labels = []

    for item in batch:
        pad_len = max_len - len(item["input_ids"])

        input_ids.append(
            item["input_ids"] + [PAD_ID] * pad_len
        )

        labels.append(
            item["labels"] + [-100] * pad_len
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }


# Initial controlled experiment subset
a1_train_records = train_eligible[:512]
a1_dev_records = dev_eligible[:64]

a1_train_dataset = A1SummarizationDataset(a1_train_records)
a1_dev_dataset = A1SummarizationDataset(a1_dev_records)

a1_train_loader = DataLoader(
    a1_train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=a1_collate,
    num_workers=0
)

a1_dev_loader = DataLoader(
    a1_dev_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=a1_collate,
    num_workers=0
)

print("A1 train examples:", len(a1_train_dataset))
print("A1 validation examples:", len(a1_dev_dataset))
print("Train batches:", len(a1_train_loader))
print("Validation batches:", len(a1_dev_loader))

sample = a1_train_dataset[0]
print("Sample sequence length:", len(sample["input_ids"]))
print("Supervised summary tokens:",
      sum(x != -100 for x in sample["labels"]))

assert len(a1_train_dataset) > 0
assert len(a1_dev_dataset) > 0

A1 train examples: 512
A1 validation examples: 64
Train batches: 512
Validation batches: 64
Sample sequence length: 1392
Supervised summary tokens: 190


In [152]:
# Move model to GPU
a1_model = a1_model.to(DEVICE)

# Full fine-tuning
for p in a1_model.parameters():
    p.requires_grad = True

optimizer_a1 = torch.optim.AdamW(
    a1_model.parameters(),
    lr=1e-5,
    weight_decay=0.01
)

scaler_a1 = torch.cuda.amp.GradScaler()

TRAIN_STEPS = 200
LOG_EVERY = 20

train_iterator = iter(a1_train_loader)
training_losses = []
start_time = time.time()

a1_model.train()

for step in range(1, TRAIN_STEPS + 1):

    try:
        batch = next(train_iterator)
    except StopIteration:
        train_iterator = iter(a1_train_loader)
        batch = next(train_iterator)

    input_ids = batch["input_ids"].to(DEVICE)
    labels = batch["labels"].to(DEVICE)

    optimizer_a1.zero_grad(set_to_none=True)

    with torch.cuda.amp.autocast(dtype=torch.float16):
        logits = a1_model(input_ids)

        loss = F.cross_entropy(
            logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
            labels[:, 1:].reshape(-1),
            ignore_index=-100
        )

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}")

    scaler_a1.scale(loss).backward()

    scaler_a1.unscale_(optimizer_a1)
    torch.nn.utils.clip_grad_norm_(
        a1_model.parameters(),
        max_norm=1.0
    )

    scaler_a1.step(optimizer_a1)
    scaler_a1.update()

    training_losses.append(float(loss.item()))

    if step % LOG_EVERY == 0:
        mean_loss = np.mean(training_losses[-LOG_EVERY:])
        elapsed = time.time() - start_time

        print(
            f"Step {step:03d}/{TRAIN_STEPS} | "
            f"Train loss: {mean_loss:.4f} | "
            f"Elapsed: {elapsed/60:.1f} min"
        )

    if step % 50 == 0:
        checkpoint_path = os.path.join(
            OUTPUT_DIR,
            f"a1_full_finetune_step_{step}.pt"
        )

        torch.save({
            "experiment": "A1_Full_Finetuning",
            "step": step,
            "model_state_dict": a1_model.state_dict(),
            "optimizer_state_dict": optimizer_a1.state_dict(),
            "train_loss": float(np.mean(training_losses[-20:])),
            "tokenizer_path": (
                "/kaggle/input/datasets/aq1de3/"
                "tokenizer-json/tokenizer.json"
            ),
            "tokenizer_compatibility": "UNVERIFIED"
        }, checkpoint_path)

        print("Saved:", checkpoint_path)

print("\nA1 training finished.")
print("Mean final 20-step loss:",
      float(np.mean(training_losses[-20:])))

/tmp/ipykernel_49/2744834390.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_a1 = torch.cuda.amp.GradScaler()
/tmp/ipykernel_49/2744834390.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


Step 020/200 | Train loss: 6.4378 | Elapsed: 0.0 min
Step 040/200 | Train loss: 6.5548 | Elapsed: 0.1 min
Saved: /kaggle/working/phase6_a1/a1_full_finetune_step_50.pt
Step 060/200 | Train loss: 6.5358 | Elapsed: 0.1 min
Step 080/200 | Train loss: 6.4791 | Elapsed: 0.2 min
Step 100/200 | Train loss: 6.4176 | Elapsed: 0.2 min
Saved: /kaggle/working/phase6_a1/a1_full_finetune_step_100.pt
Step 120/200 | Train loss: 6.3958 | Elapsed: 0.3 min
Step 140/200 | Train loss: 6.3375 | Elapsed: 0.3 min
Saved: /kaggle/working/phase6_a1/a1_full_finetune_step_150.pt
Step 160/200 | Train loss: 6.3106 | Elapsed: 0.4 min
Step 180/200 | Train loss: 6.2392 | Elapsed: 0.4 min
Step 200/200 | Train loss: 6.2999 | Elapsed: 0.4 min
Saved: /kaggle/working/phase6_a1/a1_full_finetune_step_200.pt

A1 training finished.
Mean final 20-step loss: 6.299882936477661


In [153]:
a1_model.eval()
validation_losses = []

with torch.no_grad():
    for batch in a1_dev_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.cuda.amp.autocast(dtype=torch.float16):
            logits = a1_model(input_ids)

            loss = F.cross_entropy(
                logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
                labels[:, 1:].reshape(-1),
                ignore_index=-100
            )

        validation_losses.append(float(loss.item()))

a1_validation_loss = float(np.mean(validation_losses))

print("A1 validation loss:", a1_validation_loss)
print("A1 training loss:", float(np.mean(training_losses[-20:])))

/tmp/ipykernel_49/2501056756.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


A1 validation loss: 6.239953763782978
A1 training loss: 6.299882936477661


In [154]:
a1_results = {
    "experiment": "A1_Full_Finetuning",
    "architecture": "A1 decoder-only Transformer",
    "checkpoint_source": A1_PATH,
    "checkpoint_initial_step": 500,
    "training_updates": TRAIN_STEPS,
    "training_records": len(a1_train_dataset),
    "validation_records": len(a1_dev_dataset),
    "max_document_tokens": MAX_DOC_TOKENS,
    "max_summary_tokens": MAX_SUMMARY_TOKENS,
    "learning_rate": 1e-5,
    "final_train_loss": float(np.mean(training_losses[-20:])),
    "validation_loss": a1_validation_loss,
    "tokenizer_compatibility": "UNVERIFIED",
    "status": "COMPLETED"
}

results_path = os.path.join(
    OUTPUT_DIR, "a1_full_finetune_results.json"
)

with open(results_path, "w") as f:
    json.dump(a1_results, f, indent=4)

print(json.dumps(a1_results, indent=4))
print("\nResults saved:", results_path)

{
    "experiment": "A1_Full_Finetuning",
    "architecture": "A1 decoder-only Transformer",
    "checkpoint_source": "/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt",
    "checkpoint_initial_step": 500,
    "training_updates": 200,
    "training_records": 512,
    "validation_records": 64,
    "max_document_tokens": 1200,
    "max_summary_tokens": 800,
    "learning_rate": 1e-05,
    "final_train_loss": 6.299882936477661,
    "validation_loss": 6.239953763782978,
    "tokenizer_compatibility": "UNVERIFIED",
    "status": "COMPLETED"
}

Results saved: /kaggle/working/phase6_a1/a1_full_finetune_results.json


In [155]:
import torch
import json
import os

a1_model.eval()

def generate_a1_summary(document, max_new_tokens=160):
    doc_ids = tokenizer.encode(document).ids[:1200]

    input_ids = torch.tensor(
        [[BOS_ID] + doc_ids + [EOS_ID]],
        dtype=torch.long,
        device=DEVICE
    )

    generated = []

    with torch.no_grad():
        for _ in range(max_new_tokens):

            logits = a1_model(input_ids)

            next_token = torch.argmax(
                logits[:, -1, :],
                dim=-1
            )

            token_id = next_token.item()

            if token_id == EOS_ID:
                break

            generated.append(token_id)

            input_ids = torch.cat(
                [input_ids, next_token.unsqueeze(0)],
                dim=1
            )

            if input_ids.size(1) >= MAX_SEQ_LEN:
                break

    return tokenizer.decode(generated)


generation_results = []

for i, item in enumerate(a1_dev_records[:5]):

    prediction = generate_a1_summary(item["document"])

    result = {
        "id": str(item["id"]),
        "reference": item["summary"],
        "prediction": prediction
    }

    generation_results.append(result)

    print("\n" + "=" * 80)
    print("DOCUMENT:", item["id"])
    print("\nREFERENCE:\n", item["summary"][:1000])
    print("\nGENERATED:\n", prediction)
    print("=" * 80)


generation_path = os.path.join(
    OUTPUT_DIR,
    "a1_generation_samples.json"
)

with open(generation_path, "w") as f:
    json.dump(generation_results, f, indent=2)

print("\nSaved:", generation_path)


DOCUMENT: 6827

REFERENCE:
 The appellant filed an application before the Central Administrative Tribunal, Patna Bench, for a declaration that the second proviso to rule 4 of Civil Services Examination Rules was violative of Articles 14 and 16 of the Constitu tion of India. By an interim order the Tribunal allowed the appellant to appear at the Civil Services (Main) Examina tion, 1990, subject to the result of the final orders in the original application. The said application was transferred to this Court. In a bunch of similar cases, the Central Administrative Tribunal, Delhi upheld the validity of Rule 4 of Civil Services Examination Rules. In appeal to this Court (Civil Appeal Nos. 5439 52/90)** by an interim order dated 7.12.1990, the appellants therein were allowed to appear in Civil Services (Main) Examination, 1990; and while finally disposing of the appeals, the judgment of CAT, Delhi was affirmed. Dismissing the case of the appellant in view of the judgment in C.As Nos. 5439 

In [156]:
import copy
import torch

# Save current fine-tuned model
finetuned_model = a1_model

# Reload original Phase 1 checkpoint
baseline_model = A1Model()

original_ckpt = torch.load(
    A1_PATH,
    map_location="cpu",
    weights_only=False
)

baseline_model.load_state_dict(
    original_ckpt["model_state_dict"],
    strict=True
)

baseline_model = baseline_model.to(DEVICE)
baseline_model.eval()
finetuned_model.eval()

def compare_generation(model, document, max_new_tokens=80):
    doc_ids = tokenizer.encode(document).ids[:800]

    ids = torch.tensor(
        [[BOS_ID] + doc_ids + [EOS_ID]],
        dtype=torch.long,
        device=DEVICE
    )

    output = []

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(ids)
            next_id = logits[:, -1, :].argmax(dim=-1)

            token = next_id.item()
            if token == EOS_ID:
                break

            output.append(token)
            ids = torch.cat([ids, next_id.unsqueeze(0)], dim=1)

    return tokenizer.decode(output)


document = a1_dev_records[0]["document"]

print("ORIGINAL CHECKPOINT:")
print(compare_generation(baseline_model, document))

print("\nFINE-TUNED CHECKPOINT:")
print(compare_generation(finetuned_model, document))

ORIGINAL CHECKPOINT:
.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1.
(1

FINE-TUNED CHECKPOINT:
. The State of the State of the State of the State of the State of the Court of the State of the Court of the State of the Court of the State of the Court of the Court of the State of the Court of the State of the State of the Court of the State of the Court of the Court of the State of the State of the Court of the Court of the State of the


In [157]:
import os

ROOT = "/kaggle/input/datasets/aq1de3/phase6"

print("PHASE 6 DATASET FILES")
print("=" * 65)

for current, dirs, files in os.walk(ROOT):
    for filename in files:
        print(os.path.join(current, filename))

PHASE 6 DATASET FILES
/kaggle/input/datasets/aq1de3/phase6/test (4).json
/kaggle/input/datasets/aq1de3/phase6/train (3).json


In [158]:
import os
import json

ROOT = "/kaggle/input/datasets/aq1de3/phase6"

for filename in os.listdir(ROOT):
    path = os.path.join(ROOT, filename)

    if filename.endswith(".json"):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print("\nFILE:", filename)
        print("Type:", type(data).__name__)

        if isinstance(data, list) and data:
            print("Number of records:", len(data))
            print("First record keys:", list(data[0].keys()))
            print("First record:", str(data[0])[:1500])

        elif isinstance(data, dict):
            print("Keys:", list(data.keys()))
            print("Preview:", str(data)[:1500])


FILE: test (4).json
Type: list
Number of records: 100
First record keys: ['id', 'document']
First record: {'id': '4963', 'document': 'Tax Reference Cases Nos. 3 to 5 of 1975 Tax Reference under Section 257 of the Income tax Act, 1961 made by the Income tax Appellate Tribunal, Cuttack Bench, Cuttack P.A. Francis, Champat Rai and Miss A. Subhashini for the Appellant. C.S.S. Rao for the Respondent. The Judgment of the Court was delivered by 308 PATHAK, J. : These references under s.27(1) of the have been made by the Income Tax Appellate tribunal, Cuttack Bench at the instance of the Commissioner of Wealth Tax, Orissa for the opinion of this Court on the following question of law: "Whether on the facts and in the circumstances of the case, the Wealth Tax Officer was in law justified in including in the net wealth of the assessee interest due on accrual basis (though not realised) on the outstandings of the money lending business, the accounts of the assessee being maintained on cash basis

In [159]:
import os

paths = [
    "/kaggle/input/datasets/aq1de3",
    "/kaggle/working/phase6_hierarchical",
    "/kaggle/working/phase6_a1",
]

for path in paths:
    print(f"\n{'='*70}\n{path}")
    if os.path.exists(path):
        for root, dirs, files in os.walk(path):
            for f in files:
                if f.endswith((".pt", ".pth", ".json", ".py", ".ipynb")):
                    print(os.path.join(root, f))


/kaggle/input/datasets/aq1de3
/kaggle/input/datasets/aq1de3/a2-step-9000-pt/a2_step_9000.pt
/kaggle/input/datasets/aq1de3/step-0001000-pt/step_0001000.pt
/kaggle/input/datasets/aq1de3/trainjson/train.json
/kaggle/input/datasets/aq1de3/33kkkk/step_0330000.pt
/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
/kaggle/input/datasets/aq1de3/testtest/test (3).json
/kaggle/input/datasets/aq1de3/traintr/train (2).json
/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
/kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
/kaggle/input/datasets/aq1de3/phase3/CL_dev.json
/kaggle/input/datasets/aq1de3/phase3/CL_test.json
/kaggle/input/datasets/aq1de3/phase3/CL_train.json
/kaggle/input/datasets/aq1de3/phase3/IT_dev.json
/kaggle/input/datasets/aq1de3/phase3/IT_test.json
/kaggle/input/datasets/aq1de3/phase3/IT_train.json
/kaggle/input/datasets/aq1de3/phase3/

In [160]:
import torch

A4_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-22000-pt/a4_step_22000.pt"
)

ckpt = torch.load(
    A4_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(ckpt))

if isinstance(ckpt, dict):
    print("Top-level keys:", list(ckpt.keys()))

    state = ckpt.get(
        "model_state_dict",
        ckpt.get("model", ckpt)
    )

    if isinstance(state, dict):
        print("\nTotal tensors:", len(state))

        print("\nAttention and linear layer parameters:")
        for name, tensor in state.items():
            if torch.is_tensor(tensor) and (
                "attn" in name.lower()
                or "proj" in name.lower()
                or "linear" in name.lower()
            ):
                print(f"{name:65s} {tuple(tensor.shape)}")

Checkpoint type: <class 'dict'>
Top-level keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']

Total tensors: 260

Attention and linear layer parameters:
encoder_layers.0.attn.q_proj.weight                               (512, 512)
encoder_layers.0.attn.q_proj.bias                                 (512,)
encoder_layers.0.attn.k_proj.weight                               (512, 512)
encoder_layers.0.attn.k_proj.bias                                 (512,)
encoder_layers.0.attn.v_proj.weight                               (512, 512)
encoder_layers.0.attn.v_proj.bias                                 (512,)
encoder_layers.0.attn.out_proj.weight                             (512, 512)
encoder_layers.0.attn.out_proj.bias                               (512,)
encoder_layers.1.attn.q_proj.weight                               (512, 512)
encoder_layers.1.attn.q_proj.bias                                 (512,)
encoder_layers.1.attn.k_proj.weight                      

In [166]:

# ============================================================
# LORMA+ INITIALIZATION TEST
# ============================================================

device = next(model.parameters()).device

tested = 0
max_difference = 0.0

for module in model.modules():
    if isinstance(module, LoRMALinear):
        base = module.base

        test_input = torch.randn(
            2, 7, base.in_features,
            device=base.weight.device,
            dtype=base.weight.dtype
        )

        with torch.no_grad():
            original_output = base(test_input)
            adapted_output = module(test_input)

        difference = (
            original_output - adapted_output
        ).abs().max().item()

        max_difference = max(max_difference, difference)
        tested += 1

print("Layers tested:", tested)
print("Maximum output difference:", max_difference)

assert tested == 36
assert max_difference < 1e-5, (
    "Identity initialization failed. Do not start training."
)

print("\nSUCCESS: LoRMA+ starts as the original A4 model.")


Layers tested: 0
Maximum output difference: 0.0


AssertionError: 

In [163]:
import torch
import torch.nn as nn
import math

class LoRMALinear(nn.Module):
    def __init__(self, base_layer, rank=8, alpha=16):
        super().__init__()

        if not isinstance(base_layer, nn.Linear):
            raise TypeError("Expected nn.Linear")

        if base_layer.in_features != base_layer.out_features:
            raise ValueError("LoRMA currently expects square Linear layers")

        self.base = base_layer
        self.rank = rank
        self.alpha = alpha

        # Freeze original pretrained weights
        for param in self.base.parameters():
            param.requires_grad = False

        dim = base_layer.in_features

        # LoRMA+ parameters
        self.A = nn.Parameter(
            torch.empty(rank, dim,
                        device=base_layer.weight.device,
                        dtype=base_layer.weight.dtype)
        )

        self.B = nn.Parameter(
            torch.zeros(dim, rank,
                        device=base_layer.weight.device,
                        dtype=base_layer.weight.dtype)
        )

        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        base_output = self.base(x)

        update = (base_output @ self.A.T) @ self.B.T

        return base_output + (self.alpha / self.rank) * update

In [164]:
print("LoRMALinear defined:", "LoRMALinear" in globals())

LoRMALinear defined: True


In [165]:
for module in model.modules():
    if isinstance(module, LoRMALinear):

_IncompleteInputError: incomplete input (855041997.py, line 2)

In [167]:
# Count LoRMA modules
lorma_modules = [
    (name, module)
    for name, module in model.named_modules()
    if isinstance(module, LoRMALinear)
]

print("LoRMA modules found:", len(lorma_modules))

# Count remaining original q/v projections
remaining = []

for name, module in model.named_modules():
    if name.endswith(("q_proj", "v_proj")):
        if isinstance(module, nn.Linear):
            remaining.append(name)

print("Remaining original q/v Linear layers:", len(remaining))

# Show injected module names
print("\nInjected modules:")
for name, module in lorma_modules:
    print(name, module.base.weight.shape)

# Check trainable parameters
trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("\nTrainable parameters:", trainable)

LoRMA modules found: 0
Remaining original q/v Linear layers: 36

Injected modules:

Trainable parameters: 51349504


In [168]:
import torch.nn as nn

def inject_lorma(model, rank=8, alpha=16):
    replaced = []

    for parent_name, parent in list(model.named_modules()):
        for child_name, child in list(parent.named_children()):

            if child_name in ("q_proj", "v_proj") and isinstance(child, nn.Linear):

                adapter = LoRMALinear(
                    base_layer=child,
                    rank=rank,
                    alpha=alpha
                )

                setattr(parent, child_name, adapter)
                replaced.append(f"{parent_name}.{child_name}")

    return replaced


replaced = inject_lorma(model)

print("Adapters injected:", len(replaced))

for name in replaced:
    print(name)

Adapters injected: 36
encoder_layers.0.attn.q_proj
encoder_layers.0.attn.v_proj
encoder_layers.1.attn.q_proj
encoder_layers.1.attn.v_proj
encoder_layers.2.attn.q_proj
encoder_layers.2.attn.v_proj
encoder_layers.3.attn.q_proj
encoder_layers.3.attn.v_proj
encoder_layers.4.attn.q_proj
encoder_layers.4.attn.v_proj
encoder_layers.5.attn.q_proj
encoder_layers.5.attn.v_proj
decoder_layers.0.self_attn.q_proj
decoder_layers.0.self_attn.v_proj
decoder_layers.0.cross_attn.q_proj
decoder_layers.0.cross_attn.v_proj
decoder_layers.1.self_attn.q_proj
decoder_layers.1.self_attn.v_proj
decoder_layers.1.cross_attn.q_proj
decoder_layers.1.cross_attn.v_proj
decoder_layers.2.self_attn.q_proj
decoder_layers.2.self_attn.v_proj
decoder_layers.2.cross_attn.q_proj
decoder_layers.2.cross_attn.v_proj
decoder_layers.3.self_attn.q_proj
decoder_layers.3.self_attn.v_proj
decoder_layers.3.cross_attn.q_proj
decoder_layers.3.cross_attn.v_proj
decoder_layers.4.self_attn.q_proj
decoder_layers.4.self_attn.v_proj
decoder_la

In [169]:
lorma_count = sum(
    isinstance(m, LoRMALinear)
    for m in model.modules()
)

print("LoRMA modules:", lorma_count)

assert lorma_count == 36, (
    f"Expected 36 adapters, found {lorma_count}"
)

LoRMA modules: 36


In [170]:
for param in model.parameters():
    param.requires_grad = False

for module in model.modules():
    if isinstance(module, LoRMALinear):
        module.A.requires_grad = True
        module.B.requires_grad = True

trainable = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable)
print("Trainable percentage:",
      round(100 * trainable / sum(p.numel() for p in model.parameters()), 3))

Trainable parameters: 294912
Trainable percentage: 0.305


In [171]:
import torch

model.eval()

max_difference = 0.0
tested = 0

with torch.no_grad():
    for name, module in model.named_modules():

        if isinstance(module, LoRMALinear):

            base = module.base

            x = torch.randn(
                2, 7, base.in_features,
                device=base.weight.device,
                dtype=base.weight.dtype
            )

            original_output = base(x)
            lorma_output = module(x)

            difference = (
                original_output - lorma_output
            ).abs().max().item()

            max_difference = max(max_difference, difference)
            tested += 1

print("Layers tested:", tested)
print("Maximum output difference:", max_difference)

assert tested == 36
assert max_difference < 1e-5

print("SUCCESS: LoRMA+ identity initialization verified.")

Layers tested: 36
Maximum output difference: 0.0
SUCCESS: LoRMA+ identity initialization verified.


In [172]:
import torch

model.eval()

device = next(model.parameters()).device

# Synthetic input and decoder sequences
encoder_ids = torch.randint(
    5, 50000, (1, 32), device=device
)

decoder_ids = torch.randint(
    5, 50000, (1, 16), device=device
)

with torch.no_grad():
    # Encoder
    memory, memory_padding_mask = a4_encode(
        model, encoder_ids
    )

    # Decoder
    logits = a4_decode(
        model,
        decoder_ids,
        memory,
        memory_padding_mask
    )

print("Encoder memory shape:", tuple(memory.shape))
print("Decoder logits shape:", tuple(logits.shape))
print("All logits finite:", torch.isfinite(logits).all().item())
print("Model device:", device)

assert memory.shape == (1, 32, 512)
assert logits.shape == (1, 16, 50000)
assert torch.isfinite(logits).all()

print("SUCCESS: Complete A4 forward pass works with LoRMA+.")

Encoder memory shape: (1, 32, 512)
Decoder logits shape: (1, 16, 50000)
All logits finite: True
Model device: cuda:0
SUCCESS: Complete A4 forward pass works with LoRMA+.


In [173]:
batch = next(iter(train_loader))

print("Batch type:", type(batch))

if isinstance(batch, dict):
    print("Batch keys:", batch.keys())

    for key, value in batch.items():
        if torch.is_tensor(value):
            print(key, value.shape, value.dtype)
        else:
            print(key, type(value))

elif isinstance(batch, (tuple, list)):
    print("Number of elements:", len(batch))

    for i, value in enumerate(batch):
        if torch.is_tensor(value):
            print(f"Element {i}: shape={value.shape}, dtype={value.dtype}")
        else:
            print(f"Element {i}: type={type(value)}")

Batch type: <class 'dict'>
Batch keys: dict_keys(['encoder_ids', 'decoder_ids', 'labels'])
encoder_ids torch.Size([1, 498]) torch.int64
decoder_ids torch.Size([1, 201]) torch.int64
labels torch.Size([1, 201]) torch.int64


In [174]:
{
    "encoder_ids": encoder_ids,
    "decoder_ids": decoder_ids,
    "labels": labels
}

{'encoder_ids': tensor([[  618,  7210, 49731, 48860, 47777, 49314, 24338, 17035,  6444, 19046,
           3958, 17149, 21135,  3471, 18840, 46808, 41900,  8693, 28327, 23741,
          23291, 27407, 15601, 37885,  7643, 13740, 38816, 43837,  4076, 29780,
          47178, 44116]], device='cuda:0'),
 'decoder_ids': tensor([[ 8223, 30147, 11470, 30669, 37924,  7063,  8954, 45567, 11682, 44759,
          18308,  5678,  7636,  7423, 24409, 14791]], device='cuda:0'),
 'labels': tensor([[-100, -100, -100,  ...,  349, 4251,    3]], device='cuda:0')}

In [175]:
batch = next(iter(train_loader))

labels = batch["labels"]

print("PAD_ID:", PAD_ID)
print("Unique label IDs:", torch.unique(labels).tolist())
print("Minimum label ID:", labels.min().item())
print("Maximum label ID:", labels.max().item())

print("Padding count:", (labels == PAD_ID).sum().item())
print("Ignored label count:", (labels == -100).sum().item())
print("Total labels:", labels.numel())

PAD_ID: 0
Unique label IDs: [3, 16, 18, 30, 40, 65, 238, 239, 247, 256, 265, 271, 278, 291, 294, 302, 306, 308, 309, 320, 343, 344, 353, 355, 362, 365, 403, 408, 421, 427, 430, 448, 449, 451, 458, 463, 466, 468, 520, 534, 540, 550, 553, 574, 593, 668, 721, 731, 737, 774, 779, 809, 821, 838, 870, 878, 940, 985, 1022, 1048, 1152, 1286, 1291, 1323, 1366, 1478, 1511, 1642, 1818, 2036, 2088, 2115, 2204, 2286, 2504, 2505, 2570, 2637, 3317, 3738, 3946, 4007, 4119, 4928, 5847, 6296, 6365, 6414, 6944, 7726, 8685, 9889, 11323, 11494, 11756, 12252, 13783, 14706, 15043, 15466, 18528, 19475, 21291, 40090, 43240, 43751, 44382, 44491]
Minimum label ID: 3
Maximum label ID: 44491
Padding count: 0
Ignored label count: 0
Total labels: 209


In [176]:
batch = next(iter(train_loader))

encoder_ids = batch["encoder_ids"]
decoder_ids = batch["decoder_ids"]
labels = batch["labels"]

print("Encoder shape:", encoder_ids.shape)
print("Decoder shape:", decoder_ids.shape)
print("Labels shape:", labels.shape)

print("\nPAD_ID:", PAD_ID)
print("Ignore index:", -100)

print("\nUnique label IDs:", torch.unique(labels).tolist())
print("PAD count:", (labels == PAD_ID).sum().item())
print("-100 count:", (labels == -100).sum().item())

print("\nDecoder and label shapes match:",
      decoder_ids.shape == labels.shape)

assert decoder_ids.shape == labels.shape
assert labels.min().item() >= -100
assert labels.max().item() < 50000

print("\nSUCCESS: Batch structure verified.")

Encoder shape: torch.Size([1, 961])
Decoder shape: torch.Size([1, 370])
Labels shape: torch.Size([1, 370])

PAD_ID: 0
Ignore index: -100

Unique label IDs: [3, 6, 12, 13, 16, 18, 27, 31, 39, 40, 65, 238, 239, 240, 247, 265, 271, 277, 278, 284, 291, 301, 302, 306, 309, 318, 320, 343, 344, 345, 348, 349, 353, 355, 358, 372, 387, 403, 408, 427, 430, 448, 449, 451, 458, 463, 466, 468, 485, 520, 540, 544, 550, 647, 655, 664, 668, 670, 702, 708, 737, 764, 788, 824, 827, 828, 848, 898, 913, 940, 942, 956, 985, 1000, 1002, 1104, 1145, 1152, 1183, 1319, 1366, 1454, 1557, 1598, 1642, 1658, 1703, 1715, 1785, 1796, 1818, 1949, 2064, 2115, 2320, 2440, 2580, 2708, 2772, 3075, 3159, 3199, 3214, 3234, 3547, 3664, 3682, 3929, 3946, 3947, 3967, 3996, 4053, 4088, 4161, 4163, 4492, 4499, 4576, 4719, 4766, 4952, 5155, 5211, 5256, 5307, 5523, 5958, 6032, 6276, 6541, 6588, 6696, 7327, 7427, 8068, 8140, 8544, 8701, 9064, 9117, 9231, 10058, 10146, 11213, 11622, 11974, 12232, 13070, 13629, 18446, 22766, 24514, 

In [177]:
import torch
import torch.nn.functional as F

batch = next(iter(train_loader))

encoder_ids = batch["encoder_ids"].to(device)
decoder_ids = batch["decoder_ids"].to(device)
labels = batch["labels"].to(device)

model.eval()

with torch.no_grad():
    memory, memory_padding_mask = a4_encode(
        model, encoder_ids
    )

    logits = a4_decode(
        model,
        decoder_ids,
        memory,
        memory_padding_mask
    )

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        ignore_index=-100
    )

print("Logits shape:", logits.shape)
print("Labels shape:", labels.shape)
print("Initial LoRMA training loss:", loss.item())
print("Loss finite:", torch.isfinite(loss).item())

assert logits.shape[:2] == labels.shape
assert torch.isfinite(loss)

print("SUCCESS: Real-batch loss calculation verified.")

Logits shape: torch.Size([1, 125, 50000])
Labels shape: torch.Size([1, 125])
Initial LoRMA training loss: 7.604836940765381
Loss finite: True
SUCCESS: Real-batch loss calculation verified.


In [178]:
import os
import torch
import torch.nn.functional as F

# --------------------------------------------------
# 1. Verify LoRMA adapters
# --------------------------------------------------

lorma_modules = [
    (name, module)
    for name, module in model.named_modules()
    if isinstance(module, LoRMALinear)
]

assert len(lorma_modules) == 36, (
    f"Expected 36 adapters, found {len(lorma_modules)}"
)

# Freeze everything except LoRMA A and B
for param in model.parameters():
    param.requires_grad = False

for _, module in lorma_modules:
    module.A.requires_grad = True
    module.B.requires_grad = True

trainable_params = [
    p for p in model.parameters() if p.requires_grad
]

print("Trainable parameters:",
      sum(p.numel() for p in trainable_params))

# --------------------------------------------------
# 2. Create a fresh optimizer
# --------------------------------------------------

lorma_optimizer = torch.optim.AdamW(
    trainable_params,
    lr=1e-4,
    weight_decay=0.01
)

# --------------------------------------------------
# 3. Training function
# --------------------------------------------------

def calculate_batch_loss(batch):

    encoder_ids = batch["encoder_ids"].to(device)
    decoder_ids = batch["decoder_ids"].to(device)
    labels = batch["labels"].to(device)

    memory, memory_padding_mask = a4_encode(
        model, encoder_ids
    )

    logits = a4_decode(
        model,
        decoder_ids,
        memory,
        memory_padding_mask
    )

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        ignore_index=-100
    )

    return loss


# --------------------------------------------------
# 4. Run 30 pilot updates
# --------------------------------------------------

PILOT_STEPS = 30

train_iterator = iter(train_loader)
training_losses = []

model.train()

for step in range(1, PILOT_STEPS + 1):

    try:
        batch = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        batch = next(train_iterator)

    lorma_optimizer.zero_grad(set_to_none=True)

    loss = calculate_batch_loss(batch)

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Non-finite loss at update {step}"
        )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_params,
        max_norm=1.0
    )

    lorma_optimizer.step()

    training_losses.append(loss.item())

    if step == 1 or step % 5 == 0:
        print(
            f"Step {step:02d}/{PILOT_STEPS} | "
            f"Loss: {loss.item():.4f} | "
            f"Grad norm: {grad_norm.item():.4f}"
        )

# --------------------------------------------------
# 5. Validation on up to 20 batches
# --------------------------------------------------

model.eval()

validation_losses = []

with torch.no_grad():

    for i, batch in enumerate(dev_loader):

        if i >= 20:
            break

        loss = calculate_batch_loss(batch)

        if torch.isfinite(loss):
            validation_losses.append(loss.item())

assert validation_losses, "No validation batches evaluated."

train_mean = sum(training_losses) / len(training_losses)
val_mean = sum(validation_losses) / len(validation_losses)

print("\n========== LORMA PILOT RESULTS ==========")
print("Training updates:", len(training_losses))
print("Mean training loss:", round(train_mean, 4))
print("Mean validation loss:", round(val_mean, 4))
print("Validation batches:", len(validation_losses))

# --------------------------------------------------
# 6. Save adapter-only checkpoint
# --------------------------------------------------

save_dir = "/kaggle/working/phase6_lorma"
os.makedirs(save_dir, exist_ok=True)

adapter_state = {}

for name, module in lorma_modules:
    adapter_state[f"{name}.A"] = module.A.detach().cpu()
    adapter_state[f"{name}.B"] = module.B.detach().cpu()

checkpoint_path = os.path.join(
    save_dir,
    "a4_lorma_plus_pilot_step_30.pt"
)

torch.save(
    {
        "adapter_state_dict": adapter_state,
        "step": PILOT_STEPS,
        "rank": 8,
        "alpha": 16,
        "train_loss": train_mean,
        "validation_loss": val_mean,
        "base_checkpoint": (
            "/kaggle/input/datasets/aq1de3/"
            "a4-step-22000-pt/a4_step_22000.pt"
        )
    },
    checkpoint_path
)

print("\nSaved:", checkpoint_path)
print("SUCCESS: LoRMA+ pilot completed.")

Trainable parameters: 294912
Step 01/30 | Loss: 5.7434 | Grad norm: 0.3036
Step 05/30 | Loss: 6.0385 | Grad norm: 0.3034
Step 10/30 | Loss: 6.4589 | Grad norm: 0.4440
Step 15/30 | Loss: 5.8498 | Grad norm: 0.5508
Step 20/30 | Loss: 5.7986 | Grad norm: 0.4652
Step 25/30 | Loss: 6.1358 | Grad norm: 0.5132
Step 30/30 | Loss: 6.0345 | Grad norm: 0.6703

========== LORMA PILOT RESULTS ==========
Training updates: 30
Mean training loss: 5.8878
Mean validation loss: 6.0767
Validation batches: 20

Saved: /kaggle/working/phase6_lorma/a4_lorma_plus_pilot_step_30.pt
SUCCESS: LoRMA+ pilot completed.


In [179]:
import torch
import json
import os

BOS_ID = 2
EOS_ID = 3
MAX_NEW_TOKENS = 128
NUM_SAMPLES = 5

model.eval()
generation_results = []

def generate_summary(encoder_ids):

    with torch.no_grad():
        memory, memory_padding_mask = a4_encode(
            model, encoder_ids
        )

        generated = torch.tensor(
            [[BOS_ID]],
            device=encoder_ids.device,
            dtype=torch.long
        )

        for _ in range(MAX_NEW_TOKENS):

            logits = a4_decode(
                model,
                generated,
                memory,
                memory_padding_mask
            )

            next_token = logits[:, -1, :].argmax(
                dim=-1, keepdim=True
            )

            generated = torch.cat(
                [generated, next_token], dim=1
            )

            if next_token.item() == EOS_ID:
                break

    return generated[0, 1:].tolist()


# Generate summaries
for batch_idx, batch in enumerate(dev_loader):

    if batch_idx >= NUM_SAMPLES:
        break

    encoder_ids = batch["encoder_ids"].to(device)
    labels = batch["labels"][0].tolist()

    generated_ids = generate_summary(encoder_ids)

    reference_ids = [
        token for token in labels
        if token != -100
    ]

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    reference_text = tokenizer.decode(
        reference_ids,
        skip_special_tokens=True
    )

    generation_results.append({
        "sample": batch_idx + 1,
        "generated_summary": generated_text,
        "reference_summary": reference_text
    })

    print(f"\n{'='*70}")
    print(f"SAMPLE {batch_idx + 1}")
    print("\nGENERATED SUMMARY:\n", generated_text)
    print("\nREFERENCE SUMMARY:\n", reference_text)


# Save results
output_path = "/kaggle/working/phase6_lorma/lorma_generation_samples.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        generation_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved generation results:", output_path)


SAMPLE 1

GENERATED SUMMARY:
 The appellant, the appellant was not been made of the appellant was not an order of the appellant and the High Court. The appellant, the appellant was made in the appellant was made by the appellant was made the appellant was not the appellant. The High Court, the Court. In the High Court of the appellant, the High Court of the High Court, the appellant was not been the appellant was not the High Court, the appellant was taken in the appellant was not been made by the appeal to the appeal to the appellant and the appellant. The High Court, the appellant was not the High Court of the appellant. In the High Court,

REFERENCE SUMMARY:
 The respondent filed a first appeal in the High Court against a decree and an application for stay of its operation. Before any order was made in the stay application, the appellant, who was the decree holder, levied execution, pursuant to which the respondent deposited the decretal amount in the executing court. The responden

In [180]:
batch = next(iter(train_loader))

encoder_ids = batch["encoder_ids"][0].tolist()
decoder_ids = batch["decoder_ids"][0].tolist()
labels = batch["labels"][0].tolist()

print("Special IDs:")
print("PAD:", PAD_ID, "BOS:", BOS_ID, "EOS:", EOS_ID)

print("\nDecoder first 15 IDs:")
print(decoder_ids[:15])

print("\nLabel first 15 IDs:")
print(labels[:15])

print("\nDecoder first tokens decoded:")
print(tokenizer.decode(decoder_ids[:30]))

valid_labels = [x for x in labels if x != -100]

print("\nValid label first tokens:")
print(valid_labels[:15])

print("\nReference decoded:")
print(tokenizer.decode(valid_labels, skip_special_tokens=True))

print("\nBOS in decoder:", decoder_ids.count(BOS_ID))
print("EOS in decoder:", decoder_ids.count(EOS_ID))
print("BOS in labels:", labels.count(BOS_ID))
print("EOS in labels:", labels.count(EOS_ID))

Special IDs:
PAD: 0 BOS: 2 EOS: 3

Decoder first 15 IDs:
[2, 26349, 655, 14195, 238, 537, 18, 52, 18, 940, 430, 321, 48738, 247, 10306]

Label first 15 IDs:
[26349, 655, 14195, 238, 537, 18, 52, 18, 940, 430, 321, 48738, 247, 10306, 10517]

Decoder first tokens decoded:
HELD : Neither the U.P. High Court (Abolition of Letters Patent Appeals) Act, 1962 nor the U.P. High Court (

Valid label first tokens:
[26349, 655, 14195, 238, 537, 18, 52, 18, 940, 430, 321, 48738, 247, 10306, 10517]

Reference decoded:
HELD : Neither the U.P. High Court (Abolition of Letters Patent Appeals) Act, 1962 nor the U.P. High Court (Abolition of Letters Patent Appeals) (Amendment) Act, 1962, is unconstitutional. [1208C] State of Bombay vs Narothamdas, ; ; Union of India vs Mohindra Supply Co. ; and Ram Adhar Singh vs Ramroop Singh & Ors., ; ; followed. [The Court under its inherent powers condoned the delay in filing of S.L.P. challenging the decision of the High Court on merits and allowed the petitioner to

In [181]:
import torch
import torch.nn.functional as F

model.eval()

batch = next(iter(dev_loader))

encoder_ids = batch["encoder_ids"].to(device)
decoder_ids = batch["decoder_ids"].to(device)
labels = batch["labels"].to(device)

with torch.no_grad():
    memory, memory_padding_mask = a4_encode(
        model, encoder_ids
    )

    logits = a4_decode(
        model,
        decoder_ids,
        memory,
        memory_padding_mask
    )

    predictions = logits.argmax(dim=-1)

    valid_mask = labels != -100

    correct = (
        (predictions == labels) & valid_mask
    ).sum().item()

    total = valid_mask.sum().item()

    accuracy = correct / max(total, 1)

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        ignore_index=-100
    )

print("Teacher-forced loss:", round(loss.item(), 4))
print("Correct predictions:", correct)
print("Total valid tokens:", total)
print("Teacher-forced token accuracy:", round(accuracy * 100, 2), "%")

print("\nFirst 15 target tokens:")
print(labels[0, :15].tolist())

print("\nFirst 15 predicted tokens:")
print(predictions[0, :15].tolist())

print("\nPredicted text:")
print(tokenizer.decode(
    predictions[0, :30].tolist(),
    skip_special_tokens=True
))

print("\nActual text:")
print(tokenizer.decode(
    labels[0][valid_mask[0]][:30].tolist(),
    skip_special_tokens=True
))

Teacher-forced loss: 5.2646
Correct predictions: 67
Total valid tokens: 265
Teacher-forced token accuracy: 25.28 %

First 15 target tokens:
[550, 463, 593, 239, 1291, 668, 265, 238, 940, 430, 697, 239, 2613, 278, 349]

First 15 predicted tokens:
[550, 553, 16, 238, 553, 247, 16, 238, 553, 430, 16, 238, 430, 247, 238]

Predicted text:
The appellant, the appellant of, the appellant Court, the Court of the order of the of the Court of The the Court of not in the appellant

Actual text:
The respondent filed a first appeal in the High Court against a decree and an application for stay of its operation. Before any order was made in the stay


In [182]:
import torch
import torch.nn.functional as F

model.eval()

# Use the same validation batch for both measurements
batch = next(iter(dev_loader))

encoder_ids = batch["encoder_ids"].to(device)
decoder_ids = batch["decoder_ids"].to(device)
labels = batch["labels"].to(device)

lorma_modules = [
    module for module in model.modules()
    if isinstance(module, LoRMALinear)
]

# Preserve trained B matrices
saved_B = [
    module.B.detach().clone()
    for module in lorma_modules
]

def evaluate_current_model():

    with torch.no_grad():
        memory, memory_padding_mask = a4_encode(
            model, encoder_ids
        )

        logits = a4_decode(
            model,
            decoder_ids,
            memory,
            memory_padding_mask
        )

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100
        )

        predictions = logits.argmax(dim=-1)
        valid = labels != -100

        accuracy = (
            ((predictions == labels) & valid).sum().item()
            / valid.sum().item()
        )

    return loss.item(), accuracy * 100


# A4 + trained LoRMA
lorma_loss, lorma_accuracy = evaluate_current_model()

# Temporarily disable LoRMA updates
with torch.no_grad():
    for module in lorma_modules:
        module.B.zero_()

try:
    base_loss, base_accuracy = evaluate_current_model()

finally:
    # Restore trained LoRMA parameters
    with torch.no_grad():
        for module, saved in zip(lorma_modules, saved_B):
            module.B.copy_(saved)

print("========== COMPARISON ==========")
print(f"A4 base loss:       {base_loss:.4f}")
print(f"A4 + LoRMA loss:    {lorma_loss:.4f}")

print(f"\nA4 base accuracy:    {base_accuracy:.2f}%")
print(f"A4 + LoRMA accuracy: {lorma_accuracy:.2f}%")

print("\nLoss improvement (base - LoRMA):",
      round(base_loss - lorma_loss, 4))

print("LoRMA B matrices restored:",
      all(torch.equal(m.B, b) for m, b in zip(lorma_modules, saved_B)))

========== COMPARISON ==========
A4 base loss:       5.3603
A4 + LoRMA loss:    5.2646

A4 base accuracy:    25.28%
A4 + LoRMA accuracy: 25.28%

Loss improvement (base - LoRMA): 0.0958
LoRMA B matrices restored: True


In [183]:
import torch
import torch.nn.functional as F

model.eval()

lorma_modules = [
    m for m in model.modules()
    if isinstance(m, LoRMALinear)
]

saved_B = [m.B.detach().clone() for m in lorma_modules]

def evaluate_20_batches():

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    batches = 0

    with torch.no_grad():
        for i, batch in enumerate(dev_loader):

            if i >= 20:
                break

            enc = batch["encoder_ids"].to(device)
            dec = batch["decoder_ids"].to(device)
            labels = batch["labels"].to(device)

            memory, mask = a4_encode(model, enc)
            logits = a4_decode(model, dec, memory, mask)

            valid = labels != -100

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
                reduction="sum"
            )

            predictions = logits.argmax(dim=-1)

            total_loss += loss.item()
            total_correct += (
                ((predictions == labels) & valid).sum().item()
            )
            total_tokens += valid.sum().item()
            batches += 1

    return {
        "loss": total_loss / total_tokens,
        "accuracy": 100 * total_correct / total_tokens,
        "tokens": total_tokens,
        "batches": batches
    }


# A4 + LoRMA
lorma_results = evaluate_20_batches()

# A4 base: temporarily disable LoRMA updates
try:
    with torch.no_grad():
        for m in lorma_modules:
            m.B.zero_()

    base_results = evaluate_20_batches()

finally:
    with torch.no_grad():
        for m, saved in zip(lorma_modules, saved_B):
            m.B.copy_(saved)


print("A4 BASE:", base_results)
print("A4 + LORMA:", lorma_results)

print("\nLoss improvement:",
      round(base_results["loss"] - lorma_results["loss"], 5))

print("Accuracy improvement:",
      round(lorma_results["accuracy"] - base_results["accuracy"], 3),
      "percentage points")

assert all(
    torch.equal(m.B, saved)
    for m, saved in zip(lorma_modules, saved_B)
)

print("\nOriginal adapter weights restored.")

A4 BASE: {'loss': 6.235568633470075, 'accuracy': 18.808675946908384, 'tokens': 6178, 'batches': 20}
A4 + LORMA: {'loss': 6.092978890253217, 'accuracy': 19.100032372936226, 'tokens': 6178, 'batches': 20}

Loss improvement: 0.14259
Accuracy improvement: 0.291 percentage points

Original adapter weights restored.


In [184]:
print("Model class:", type(model))
print("Model device:", next(model.parameters()).device)

print("\nEncoder layers:", len(model.encoder_layers))
print("Decoder layers:", len(model.decoder_layers))

print("\nEncoder layer type:")
print(type(model.encoder_layers[0]))

print("\nDecoder layer type:")
print(type(model.decoder_layers[0]))

print("\nEncoder layer state keys:")
for key in model.encoder_layers[0].state_dict().keys():
    print(key)

print("\nDecoder layer state keys:")
for key in model.decoder_layers[0].state_dict().keys():
    print(key)

print("\nModel components:")
for name, module in model.named_children():
    print(name, type(module))

Model class: <class '__main__.A4Model'>
Model device: cuda:0

Encoder layers: 6
Decoder layers: 6

Encoder layer type:
<class '__main__.A4EncoderLayer'>

Decoder layer type:
<class '__main__.A4DecoderLayer'>

Encoder layer state keys:
norm1.weight
norm1.bias
attn.q_proj.A
attn.q_proj.B
attn.q_proj.base.weight
attn.q_proj.base.bias
attn.k_proj.weight
attn.k_proj.bias
attn.v_proj.A
attn.v_proj.B
attn.v_proj.base.weight
attn.v_proj.base.bias
attn.out_proj.weight
attn.out_proj.bias
norm2.weight
norm2.bias
ffn.net.0.weight
ffn.net.0.bias
ffn.net.2.weight
ffn.net.2.bias

Decoder layer state keys:
norm1.weight
norm1.bias
self_attn.q_proj.A
self_attn.q_proj.B
self_attn.q_proj.base.weight
self_attn.q_proj.base.bias
self_attn.k_proj.weight
self_attn.k_proj.bias
self_attn.v_proj.A
self_attn.v_proj.B
self_attn.v_proj.base.weight
self_attn.v_proj.base.bias
self_attn.out_proj.weight
self_attn.out_proj.bias
norm2.weight
norm2.bias
cross_attn.q_proj.A
cross_attn.q_proj.B
cross_attn.q_proj.base.weight


In [185]:
print(
    "LoRMA adapters:",
    sum(isinstance(m, LoRMALinear) for m in model.modules())
)

print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

LoRMA adapters: 36
Trainable parameters: 294912


In [186]:
import os
import torch

KD_DIR = "/kaggle/working/phase6_kd"
os.makedirs(KD_DIR, exist_ok=True)

teacher_path = os.path.join(
    KD_DIR,
    "a4_lorma_teacher_snapshot.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "step": 30,
        "architecture": "A4-6E6D-LoRMA+",
        "rank": 8,
        "alpha": 16,
        "train_loss": 5.8878,
        "validation_loss": 6.0767
    },
    teacher_path
)

print("Teacher saved:", teacher_path)
print("Teacher parameters:",
      sum(p.numel() for p in model.parameters()))

Teacher saved: /kaggle/working/phase6_kd/a4_lorma_teacher_snapshot.pt
Teacher parameters: 96684032


In [187]:
A4_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-22000-pt/a4_step_22000.pt"
)

checkpoint = torch.load(
    A4_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", checkpoint.keys())

base_state = checkpoint["model_state_dict"]

print("\nTotal tensors:", len(base_state))

print("\nEncoder layers:")
print(sorted(set(
    key.split(".")[1]
    for key in base_state
    if key.startswith("encoder_layers.")
)))

print("\nDecoder layers:")
print(sorted(set(
    key.split(".")[1]
    for key in base_state
    if key.startswith("decoder_layers.")
)))

print("\nEmbedding shape:",
      base_state["token_embedding.weight"].shape)

print("Encoder position shape:",
      base_state["encoder_pos.weight"].shape)

print("Decoder position shape:",
      base_state["decoder_pos.weight"].shape)

Checkpoint keys: dict_keys(['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at'])

Total tensors: 260

Encoder layers:
['0', '1', '2', '3', '4', '5']

Decoder layers:
['0', '1', '2', '3', '4', '5']

Embedding shape: torch.Size([50000, 512])
Encoder position shape: torch.Size([1024, 512])
Decoder position shape: torch.Size([1024, 512])


In [188]:
import inspect

print("A4Model constructor:")
print(inspect.signature(A4Model))

print("\nA4Model initialization code:")
print(inspect.getsource(A4Model.__init__))

print("\nOriginal checkpoint attention keys:")
for key in base_state:
    if "encoder_layers.0.attn.q_proj" in key:
        print(key)

print("\nCurrent teacher attention keys:")
for key in model.state_dict():
    if "encoder_layers.0.attn.q_proj" in key:
        print(key)

A4Model constructor:
()

A4Model initialization code:
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)

        # Embeddings are used here because checkpoint keys are
        # encoder_pos.weight and decoder_pos.weight.
        self.encoder_pos = nn.Embedding(MAX_POSITION, D_MODEL)
        self.decoder_pos = nn.Embedding(MAX_POSITION, D_MODEL)

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer()
            for _ in range(NUM_ENCODER_LAYERS)
        ])

        self.encoder_norm = nn.LayerNorm(D_MODEL)

        self.decoder_layers = nn.ModuleList([
            A4DecoderLayer()
            for _ in range(NUM_DECODER_LAYERS)
        ])

        self.decoder_norm = nn.LayerNorm(D_MODEL)

        self.lm_head = nn.Linear(
            D_MODEL,
            VOCAB_SIZE,
            bias=False
        )


Original checkpoint attention keys:
encoder_layers.0.attn.q_proj.weight
encoder_layers.0.attn.q_proj.

In [189]:
import torch
import torch.nn as nn
import copy

# ==================================================
# 1. PRESERVE THE TEACHER
# ==================================================

teacher = model

teacher.eval()

for param in teacher.parameters():
    param.requires_grad = False

teacher_adapters = sum(
    isinstance(m, LoRMALinear)
    for m in teacher.modules()
)

assert teacher_adapters == 36

# ==================================================
# 2. CREATE CLEAN A4 STUDENT FROM PHASE 1
# ==================================================

student = A4Model()

# Load original Phase 1 A4 checkpoint
student.load_state_dict(base_state, strict=True)

print("Original Phase 1 weights loaded strictly.")

# ==================================================
# 3. REDUCE STUDENT TO 3 ENCODER + 3 DECODER
# ==================================================

selected_layers = [0, 2, 4]

student.encoder_layers = nn.ModuleList([
    copy.deepcopy(student.encoder_layers[i])
    for i in selected_layers
])

student.decoder_layers = nn.ModuleList([
    copy.deepcopy(student.decoder_layers[i])
    for i in selected_layers
])

student = student.to(device)

# ==================================================
# 4. VERIFY TEACHER AND STUDENT
# ==================================================

print("\n========== KD MODEL CONFIGURATION ==========")

print("Teacher encoder layers:", len(teacher.encoder_layers))
print("Teacher decoder layers:", len(teacher.decoder_layers))

print("Student encoder layers:", len(student.encoder_layers))
print("Student decoder layers:", len(student.decoder_layers))

print("Teacher LoRMA adapters:",
      sum(isinstance(m, LoRMALinear) for m in teacher.modules()))

print("Student LoRMA adapters:",
      sum(isinstance(m, LoRMALinear) for m in student.modules()))

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

print("\nTeacher parameters:", f"{teacher_params:,}")
print("Student parameters:", f"{student_params:,}")
print("Student parameter reduction:",
      round(100 * (1 - student_params / teacher_params), 2), "%")

assert len(teacher.encoder_layers) == 6
assert len(teacher.decoder_layers) == 6
assert len(student.encoder_layers) == 3
assert len(student.decoder_layers) == 3

assert sum(
    isinstance(m, LoRMALinear)
    for m in student.modules()
) == 0

print("\nSUCCESS: KD teacher and student constructed.")

Original Phase 1 weights loaded strictly.

========== KD MODEL CONFIGURATION ==========
Teacher encoder layers: 6
Teacher decoder layers: 6
Student encoder layers: 3
Student decoder layers: 3
Teacher LoRMA adapters: 36
Student LoRMA adapters: 0

Teacher parameters: 96,684,032
Student parameters: 74,319,872
Student parameter reduction: 23.13 %

SUCCESS: KD teacher and student constructed.


In [190]:
teacher.eval()
student.eval()

batch = next(iter(train_loader))

encoder_ids = batch["encoder_ids"].to(device)
decoder_ids = batch["decoder_ids"].to(device)

with torch.no_grad():

    teacher_memory, teacher_mask = a4_encode(
        teacher, encoder_ids
    )

    teacher_logits = a4_decode(
        teacher,
        decoder_ids,
        teacher_memory,
        teacher_mask
    )

    student_memory, student_mask = a4_encode(
        student, encoder_ids
    )

    student_logits = a4_decode(
        student,
        decoder_ids,
        student_memory,
        student_mask
    )

print("Teacher logits:", teacher_logits.shape)
print("Student logits:", student_logits.shape)

print("Teacher finite:",
      torch.isfinite(teacher_logits).all().item())

print("Student finite:",
      torch.isfinite(student_logits).all().item())

assert teacher_logits.shape == student_logits.shape
assert torch.isfinite(teacher_logits).all()
assert torch.isfinite(student_logits).all()

print("\nSUCCESS: Both KD models passed the smoke test.")

Teacher logits: torch.Size([1, 143, 50000])
Student logits: torch.Size([1, 143, 50000])
Teacher finite: True
Student finite: True

SUCCESS: Both KD models passed the smoke test.


In [191]:
import os
import torch
import torch.nn.functional as F

# ================================================
# 1. CONFIGURATION
# ================================================

KD_TEMPERATURE = 2.0
CE_WEIGHT = 0.7
KD_WEIGHT = 0.3
KD_STEPS = 30
KD_LR = 1e-4

# ================================================
# 2. VERIFY MODELS
# ================================================

teacher.eval()

for p in teacher.parameters():
    p.requires_grad = False

student.train()

student_optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=KD_LR,
    weight_decay=0.01
)

student_trainable = sum(
    p.numel() for p in student.parameters()
    if p.requires_grad
)

print("Student trainable parameters:", student_trainable)

# ================================================
# 3. KD LOSS FUNCTION
# ================================================

def calculate_kd_losses(batch):

    encoder_ids = batch["encoder_ids"].to(device)
    decoder_ids = batch["decoder_ids"].to(device)
    labels = batch["labels"].to(device)

    valid_mask = labels != -100

    # Teacher forward pass: no gradients
    with torch.no_grad():

        teacher_memory, teacher_mask = a4_encode(
            teacher, encoder_ids
        )

        teacher_logits = a4_decode(
            teacher,
            decoder_ids,
            teacher_memory,
            teacher_mask
        )

    # Student forward pass
    student_memory, student_mask = a4_encode(
        student, encoder_ids
    )

    student_logits = a4_decode(
        student,
        decoder_ids,
        student_memory,
        student_mask
    )

    assert teacher_logits.shape == student_logits.shape
    assert student_logits.shape[:2] == labels.shape

    # Supervised cross-entropy
    ce_loss = F.cross_entropy(
        student_logits.reshape(-1, student_logits.size(-1)),
        labels.reshape(-1),
        ignore_index=-100
    )

    # Temperature-scaled distillation
    T = KD_TEMPERATURE

    teacher_probs = F.softmax(
        teacher_logits.float() / T,
        dim=-1
    )

    student_log_probs = F.log_softmax(
        student_logits.float() / T,
        dim=-1
    )

    # KL divergence per token
    token_kl = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="none"
    ).sum(dim=-1)

    kd_loss = (
        token_kl * valid_mask.float()
    ).sum() / valid_mask.sum().clamp_min(1)

    kd_loss = kd_loss * (T ** 2)

    total_loss = (
        CE_WEIGHT * ce_loss
        + KD_WEIGHT * kd_loss
    )

    return total_loss, ce_loss.detach(), kd_loss.detach()


# ================================================
# 4. TRAINING PILOT
# ================================================

train_iterator = iter(train_loader)

kd_total_losses = []
kd_ce_losses = []
kd_distillation_losses = []

for step in range(1, KD_STEPS + 1):

    try:
        batch = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        batch = next(train_iterator)

    student_optimizer.zero_grad(set_to_none=True)

    total_loss, ce_loss, kd_loss = calculate_kd_losses(batch)

    if not torch.isfinite(total_loss):
        raise RuntimeError(f"Non-finite KD loss at step {step}")

    total_loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        student.parameters(),
        max_norm=1.0
    )

    student_optimizer.step()

    kd_total_losses.append(total_loss.item())
    kd_ce_losses.append(ce_loss.item())
    kd_distillation_losses.append(kd_loss.item())

    if step == 1 or step % 5 == 0:
        print(
            f"Step {step:02d}/{KD_STEPS} | "
            f"Total: {total_loss.item():.4f} | "
            f"CE: {ce_loss.item():.4f} | "
            f"KD: {kd_loss.item():.4f} | "
            f"Grad: {grad_norm.item():.4f}"
        )


# ================================================
# 5. VALIDATION
# ================================================

student.eval()

val_ce_losses = []
val_kd_losses = []
val_total_losses = []

with torch.no_grad():

    for i, batch in enumerate(dev_loader):

        if i >= 20:
            break

        total, ce, kd = calculate_kd_losses(batch)

        val_total_losses.append(total.item())
        val_ce_losses.append(ce.item())
        val_kd_losses.append(kd.item())

assert val_total_losses, "No validation batches evaluated."

print("\n========== KD PILOT RESULTS ==========")

print("Training updates:", len(kd_total_losses))

print("Mean training total loss:",
      round(sum(kd_total_losses) / len(kd_total_losses), 4))

print("Mean training CE loss:",
      round(sum(kd_ce_losses) / len(kd_ce_losses), 4))

print("Mean training KD loss:",
      round(sum(kd_distillation_losses) / len(kd_distillation_losses), 4))

print("Mean validation total loss:",
      round(sum(val_total_losses) / len(val_total_losses), 4))

print("Mean validation CE loss:",
      round(sum(val_ce_losses) / len(val_ce_losses), 4))

print("Mean validation KD loss:",
      round(sum(val_kd_losses) / len(val_kd_losses), 4))


# ================================================
# 6. SAVE STUDENT CHECKPOINT
# ================================================

KD_DIR = "/kaggle/working/phase6_kd"
os.makedirs(KD_DIR, exist_ok=True)

kd_checkpoint_path = os.path.join(
    KD_DIR,
    "a4_kd_student_step_30.pt"
)

torch.save(
    {
        "student_state_dict": student.state_dict(),
        "step": KD_STEPS,
        "encoder_layers": 3,
        "decoder_layers": 3,
        "temperature": KD_TEMPERATURE,
        "ce_weight": CE_WEIGHT,
        "kd_weight": KD_WEIGHT,
        "learning_rate": KD_LR,
        "train_total_loss": sum(kd_total_losses) / len(kd_total_losses),
        "train_ce_loss": sum(kd_ce_losses) / len(kd_ce_losses),
        "train_kd_loss": sum(kd_distillation_losses) / len(kd_distillation_losses),
        "validation_total_loss": sum(val_total_losses) / len(val_total_losses),
        "validation_ce_loss": sum(val_ce_losses) / len(val_ce_losses),
        "validation_kd_loss": sum(val_kd_losses) / len(val_kd_losses),
        "teacher": "A4-6E6D-LoRMA+",
        "student": "A4-3E3D"
    },
    kd_checkpoint_path
)

print("\nSaved KD student:", kd_checkpoint_path)
print("SUCCESS: KD pilot completed.")

Student trainable parameters: 74319872
Step 01/30 | Total: 23.4378 | CE: 22.9536 | KD: 24.5675 | Grad: 159.6020
Step 05/30 | Total: 5.9654 | CE: 7.6445 | KD: 2.0474 | Grad: 5.1192
Step 10/30 | Total: 5.6153 | CE: 7.5453 | KD: 1.1119 | Grad: 3.8268
Step 15/30 | Total: 5.0670 | CE: 6.8002 | KD: 1.0227 | Grad: 3.1900
Step 20/30 | Total: 4.5031 | CE: 6.0176 | KD: 0.9692 | Grad: 3.3197
Step 25/30 | Total: 4.8557 | CE: 6.5909 | KD: 0.8068 | Grad: 4.1822
Step 30/30 | Total: 4.5866 | CE: 6.2459 | KD: 0.7151 | Grad: 3.2195

========== KD PILOT RESULTS ==========
Training updates: 30
Mean training total loss: 6.0923
Mean training CE loss: 7.8186
Mean training KD loss: 2.0643
Mean validation total loss: 4.8691
Mean validation CE loss: 6.668
Mean validation KD loss: 0.6716

Saved KD student: /kaggle/working/phase6_kd/a4_kd_student_step_30.pt
SUCCESS: KD pilot completed.


In [192]:
import torch
import json
import os

BOS_ID = 2
EOS_ID = 3
MAX_NEW_TOKENS = 128
NUM_SAMPLES = 5

def generate_with_model(current_model, encoder_ids):

    current_model.eval()

    with torch.no_grad():

        memory, memory_mask = a4_encode(
            current_model, encoder_ids
        )

        generated = torch.tensor(
            [[BOS_ID]],
            device=encoder_ids.device,
            dtype=torch.long
        )

        for _ in range(MAX_NEW_TOKENS):

            logits = a4_decode(
                current_model,
                generated,
                memory,
                memory_mask
            )

            next_token = logits[:, -1, :].argmax(
                dim=-1, keepdim=True
            )

            generated = torch.cat(
                [generated, next_token], dim=1
            )

            if next_token.item() == EOS_ID:
                break

    return generated[0, 1:].tolist()


kd_generation_results = []

for batch_idx, batch in enumerate(dev_loader):

    if batch_idx >= NUM_SAMPLES:
        break

    encoder_ids = batch["encoder_ids"].to(device)
    labels = batch["labels"][0].tolist()

    teacher_ids = generate_with_model(
        teacher, encoder_ids
    )

    student_ids = generate_with_model(
        student, encoder_ids
    )

    reference_ids = [
        token for token in labels if token != -100
    ]

    teacher_text = tokenizer.decode(
        teacher_ids, skip_special_tokens=True
    )

    student_text = tokenizer.decode(
        student_ids, skip_special_tokens=True
    )

    reference_text = tokenizer.decode(
        reference_ids, skip_special_tokens=True
    )

    kd_generation_results.append({
        "sample": batch_idx + 1,
        "teacher_summary": teacher_text,
        "student_summary": student_text,
        "reference_summary": reference_text
    })

    print("\n" + "=" * 75)
    print(f"SAMPLE {batch_idx + 1}")

    print("\nTEACHER:")
    print(teacher_text)

    print("\nSTUDENT:")
    print(student_text)

    print("\nREFERENCE:")
    print(reference_text)


output_path = (
    "/kaggle/working/phase6_kd/"
    "kd_teacher_student_generation.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        kd_generation_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved:", output_path)


SAMPLE 1

TEACHER:
The appellant, the appellant was not been made of the appellant was not an order of the appellant and the High Court. The appellant, the appellant was made in the appellant was made by the appellant was made the appellant was not the appellant. The High Court, the Court. In the High Court of the appellant, the High Court of the High Court, the appellant was not been the appellant was not the High Court, the appellant was taken in the appellant was not been made by the appeal to the appeal to the appellant and the appellant. The High Court, the appellant was not the High Court of the appellant. In the High Court,

STUDENT:
The a the appeal, the case of the Court, the Court, the appeal. The was a the appeal, the case of the case of the case of the appeal. The. The the appeal, the case of the case of the as a, the Court, the case of the Court. The. The a as to the appeal. The. The High Court. The The a the Court, the case of the case of the case of the case of the case

In [193]:
import json
import os

kd_record = {
    "experiment": "A4 Knowledge Distillation",
    "teacher": "A4 6E6D LoRMA+",
    "student": "A4 3E3D",
    "teacher_parameters": 96684032,
    "student_parameters": 74319872,
    "parameter_reduction_percent": 23.13,
    "training_steps": 30,
    "temperature": 2.0,
    "ce_weight": 0.7,
    "kd_weight": 0.3,
    "train_total_loss": 6.0923,
    "train_ce_loss": 7.8186,
    "train_kd_loss": 2.0643,
    "validation_total_loss": 4.8691,
    "validation_ce_loss": 6.6680,
    "validation_kd_loss": 0.6716,
    "generation_quality": "Poor; repetitive outputs",
    "conclusion": (
        "KD optimization completed, but generated summaries "
        "were repetitive and did not preserve reference facts."
    )
}

record_path = "/kaggle/working/phase6_kd/kd_experiment_record.json"

with open(record_path, "w") as f:
    json.dump(kd_record, f, indent=2)

print("Saved:", record_path)

Saved: /kaggle/working/phase6_kd/kd_experiment_record.json


In [194]:
import copy
import torch
import torch.nn as nn

# Create independent student from original Phase 1 weights
student_ce = A4Model()

student_ce.load_state_dict(base_state, strict=True)

selected_layers = [0, 2, 4]

student_ce.encoder_layers = nn.ModuleList([
    copy.deepcopy(student_ce.encoder_layers[i])
    for i in selected_layers
])

student_ce.decoder_layers = nn.ModuleList([
    copy.deepcopy(student_ce.decoder_layers[i])
    for i in selected_layers
])

student_ce = student_ce.to(device)

# Verify architecture
assert len(student_ce.encoder_layers) == 3
assert len(student_ce.decoder_layers) == 3

assert not any(
    isinstance(m, LoRMALinear)
    for m in student_ce.modules()
)

print("Fresh supervised student created.")
print("Encoder layers:", len(student_ce.encoder_layers))
print("Decoder layers:", len(student_ce.decoder_layers))
print("Parameters:", sum(p.numel() for p in student_ce.parameters()))

Fresh supervised student created.
Encoder layers: 3
Decoder layers: 3
Parameters: 74319872


In [195]:
import torch.nn.functional as F
import os

student_ce_optimizer = torch.optim.AdamW(
    student_ce.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

student_ce.train()

ce_train_losses = []
train_iterator = iter(train_loader)

for step in range(1, 31):

    try:
        batch = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        batch = next(train_iterator)

    encoder_ids = batch["encoder_ids"].to(device)
    decoder_ids = batch["decoder_ids"].to(device)
    labels = batch["labels"].to(device)

    student_ce_optimizer.zero_grad(set_to_none=True)

    memory, memory_mask = a4_encode(
        student_ce, encoder_ids
    )

    logits = a4_decode(
        student_ce, decoder_ids, memory, memory_mask
    )

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        ignore_index=-100
    )

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}")

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        student_ce.parameters(), 1.0
    )

    student_ce_optimizer.step()

    ce_train_losses.append(loss.item())

    if step == 1 or step % 5 == 0:
        print(f"Step {step:02d}/30 | CE loss: {loss.item():.4f}")


# Validation
student_ce.eval()

val_loss_sum = 0.0
val_token_count = 0

with torch.no_grad():
    for i, batch in enumerate(dev_loader):

        if i >= 20:
            break

        enc = batch["encoder_ids"].to(device)
        dec = batch["decoder_ids"].to(device)
        labels = batch["labels"].to(device)

        memory, mask = a4_encode(student_ce, enc)
        logits = a4_decode(student_ce, dec, memory, mask)

        loss_sum = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        valid_tokens = (labels != -100).sum().item()

        val_loss_sum += loss_sum.item()
        val_token_count += valid_tokens


ce_train_mean = sum(ce_train_losses) / len(ce_train_losses)
ce_val_mean = val_loss_sum / val_token_count

print("\n========== SUPERVISED BASELINE ==========")
print("Mean training CE:", round(ce_train_mean, 4))
print("Validation CE:", round(ce_val_mean, 4))
print("Validation tokens:", val_token_count)

# Save
ce_path = "/kaggle/working/phase6_kd/a4_student_ce_step_30.pt"

torch.save({
    "student_state_dict": student_ce.state_dict(),
    "step": 30,
    "encoder_layers": 3,
    "decoder_layers": 3,
    "train_ce": ce_train_mean,
    "validation_ce": ce_val_mean,
    "initialization": "Phase 1 A4 layers [0,2,4]"
}, ce_path)

print("Saved:", ce_path)

Step 01/30 | CE loss: 26.0896
Step 05/30 | CE loss: 8.1229
Step 10/30 | CE loss: 7.0512
Step 15/30 | CE loss: 6.6393
Step 20/30 | CE loss: 5.7831
Step 25/30 | CE loss: 6.8655
Step 30/30 | CE loss: 7.4860

========== SUPERVISED BASELINE ==========
Mean training CE: 8.1739
Validation CE: 6.792
Validation tokens: 6178
Saved: /kaggle/working/phase6_kd/a4_student_ce_step_30.pt


In [196]:
import torch
import torch.nn.functional as F

def evaluate_student_ce(current_model, loader, max_batches=20):

    current_model.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    with torch.no_grad():

        for i, batch in enumerate(loader):

            if i >= max_batches:
                break

            enc = batch["encoder_ids"].to(device)
            dec = batch["decoder_ids"].to(device)
            labels = batch["labels"].to(device)

            memory, mask = a4_encode(current_model, enc)

            logits = a4_decode(
                current_model, dec, memory, mask
            )

            valid = labels != -100

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
                reduction="sum"
            )

            predictions = logits.argmax(dim=-1)

            total_loss += loss.item()
            total_correct += (
                ((predictions == labels) & valid).sum().item()
            )
            total_tokens += valid.sum().item()

    return {
        "loss": total_loss / total_tokens,
        "accuracy": 100 * total_correct / total_tokens,
        "tokens": total_tokens
    }


kd_student_results = evaluate_student_ce(student)
supervised_results = evaluate_student_ce(student_ce)

print("KD STUDENT:", kd_student_results)
print("SUPERVISED STUDENT:", supervised_results)

print("\nValidation CE improvement:",
      round(
          supervised_results["loss"] - kd_student_results["loss"],
          5
      ))

print("Token accuracy difference:",
      round(
          kd_student_results["accuracy"] -
          supervised_results["accuracy"],
          3
      ), "percentage points")

TypeError: evaluate_student_ce() missing 1 required positional argument: 'loader'

In [197]:
kd_student_results = evaluate_student_ce(student, dev_loader)
supervised_results = evaluate_student_ce(student_ce, dev_loader)

print("KD STUDENT:", kd_student_results)
print("SUPERVISED STUDENT:", supervised_results)

print("\nValidation CE improvement:",
      round(
          supervised_results["loss"] - kd_student_results["loss"],
          5
      ))

print("Token accuracy difference:",
      round(
          kd_student_results["accuracy"] -
          supervised_results["accuracy"],
          3
      ), "percentage points")

KD STUDENT: {'loss': 6.685495739038726, 'accuracy': 16.186468112657817, 'tokens': 6178}
SUPERVISED STUDENT: {'loss': 6.792009575307543, 'accuracy': 15.992230495305924, 'tokens': 6178}

Validation CE improvement: 0.10651
Token accuracy difference: 0.194 percentage points


In [198]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import json
import os
import gc

# Load a fresh original A4 model
prompt_model = A4Model()
prompt_model.load_state_dict(base_state, strict=True)
prompt_model = prompt_model.to(device)

# Freeze all original model parameters
for param in prompt_model.parameters():
    param.requires_grad = False

# Create 16 learnable soft prompt tokens
PROMPT_LENGTH = 16
D_MODEL = 512

soft_prompt = nn.Parameter(
    torch.randn(1, PROMPT_LENGTH, D_MODEL, device=device) * 0.02
)

print("A4 model loaded from original Phase 1 checkpoint.")
print("Total model parameters:",
      sum(p.numel() for p in prompt_model.parameters()))

print("Trainable prompt parameters:", soft_prompt.numel())

A4 model loaded from original Phase 1 checkpoint.
Total model parameters: 96389120
Trainable prompt parameters: 8192


In [200]:
def a4_encode_soft_prompt(current_model, encoder_ids, prompt):
    B, Te = encoder_ids.shape

    if Te > MAX_POSITION:
        raise ValueError(f"Input length {Te} exceeds {MAX_POSITION}")

    positions = torch.arange(Te, device=encoder_ids.device)

    # Original token and positional embeddings
    encoder = (
        current_model.token_embedding(encoder_ids)
        + current_model.encoder_pos(positions)[None, :, :]
    )

    # Expand the learnable prompt for the batch
    prompt_batch = prompt.expand(B, -1, -1)

    # Prepend soft prompt tokens
    encoder = torch.cat([prompt_batch, encoder], dim=1)

    # Original document padding mask
    document_padding_mask = (
        encoder_ids == PAD_ID
    )[:, None, None, :]

    # Prompt tokens are never padding
    prompt_padding_mask = torch.zeros(
        B, 1, 1, PROMPT_LENGTH,
        dtype=torch.bool,
        device=encoder_ids.device
    )

    memory_padding_mask = torch.cat(
        [prompt_padding_mask, document_padding_mask],
        dim=-1
    )

    for layer in current_model.encoder_layers:
        encoder = layer(encoder, memory_padding_mask)

    return current_model.encoder_norm(encoder), memory_padding_mask

In [203]:
V = 50000

print("Vocabulary size:", V)
print("Logits shape should end with:", V)

Vocabulary size: 50000
Logits shape should end with: 50000


In [204]:
prompt_optimizer = torch.optim.AdamW(
    [soft_prompt],
    lr=1e-3,
    weight_decay=0.01
)

prompt_model.eval()

prompt_train_losses = []
prompt_model.zero_grad(set_to_none=True)

train_iterator = iter(train_loader)

for step in range(1, 31):

    try:
        batch = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        batch = next(train_iterator)

    encoder_ids = batch["encoder_ids"].to(device)
    decoder_ids = batch["decoder_ids"].to(device)
    labels = batch["labels"].to(device)

    prompt_optimizer.zero_grad(set_to_none=True)

    memory, memory_mask = a4_encode_soft_prompt(
        prompt_model,
        encoder_ids,
        soft_prompt
    )

    logits = a4_decode(
        prompt_model,
        decoder_ids,
        memory,
        memory_mask
    )

    loss = F.cross_entropy(
        logits.reshape(-1, V),
        labels.reshape(-1),
        ignore_index=-100
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_([soft_prompt], 1.0)

    prompt_optimizer.step()

    prompt_train_losses.append(loss.item())

    if step == 1 or step % 5 == 0:
        print(
            f"Step {step:02d}/30 | "
            f"Loss: {loss.item():.4f}"
        )

print("\nMean training loss:",
      sum(prompt_train_losses) / len(prompt_train_losses))

Step 01/30 | Loss: 27.3463
Step 05/30 | Loss: 18.4199
Step 10/30 | Loss: 17.0286
Step 15/30 | Loss: 18.0911
Step 20/30 | Loss: 17.9715
Step 25/30 | Loss: 17.5726
Step 30/30 | Loss: 17.2249

Mean training loss: 18.30567544301351


In [205]:
def evaluate_soft_prompt(max_batches=20):
    prompt_model.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    with torch.no_grad():
        for i, batch in enumerate(dev_loader):

            if i >= max_batches:
                break

            encoder_ids = batch["encoder_ids"].to(device)
            decoder_ids = batch["decoder_ids"].to(device)
            labels = batch["labels"].to(device)

            memory, memory_mask = a4_encode_soft_prompt(
                prompt_model,
                encoder_ids,
                soft_prompt
            )

            logits = a4_decode(
                prompt_model,
                decoder_ids,
                memory,
                memory_mask
            )

            valid = labels != -100

            loss = F.cross_entropy(
                logits.reshape(-1, V),
                labels.reshape(-1),
                ignore_index=-100,
                reduction="sum"
            )

            predictions = logits.argmax(dim=-1)

            total_loss += loss.item()
            total_correct += (
                ((predictions == labels) & valid).sum().item()
            )
            total_tokens += valid.sum().item()

    return {
        "loss": total_loss / total_tokens,
        "accuracy": 100 * total_correct / total_tokens,
        "tokens": total_tokens
    }


prompt_results = evaluate_soft_prompt()

print("\n========== A4 SOFT PROMPT RESULTS ==========")
print(prompt_results)


========== A4 SOFT PROMPT RESULTS ==========
{'loss': 17.28407935194693, 'accuracy': 3.1725477500809323, 'tokens': 6178}


In [206]:
# Evaluate the original frozen A4 model without soft prompts
base_prompt_results = evaluate_student_ce(
    prompt_model,
    dev_loader
)

print("ORIGINAL A4:", base_prompt_results)
print("SOFT PROMPT:", prompt_results)

print("\nSoft prompt parameter norm:",
      soft_prompt.detach().norm().item())

print("Soft prompt mean:",
      soft_prompt.detach().mean().item())

print("Soft prompt standard deviation:",
      soft_prompt.detach().std().item())

ORIGINAL A4: {'loss': 19.37780860513111, 'accuracy': 2.865004855940434, 'tokens': 6178}
SOFT PROMPT: {'loss': 17.28407935194693, 'accuracy': 3.1725477500809323, 'tokens': 6178}

Soft prompt parameter norm: 1.9259394407272339
Soft prompt mean: -0.0001614212669664994
Soft prompt standard deviation: 0.021279511973261833


In [207]:
import os
import json

os.makedirs("/kaggle/working/phase6_prompt", exist_ok=True)

prompt_record = {
    "experiment": "A4 Soft Prompt Tuning",
    "base_checkpoint": "A4 Phase 1 step 22000",
    "prompt_length": PROMPT_LENGTH,
    "trainable_parameters": soft_prompt.numel(),
    "training_updates": 30,
    "learning_rate": 1e-3,
    "validation_tokens": prompt_results["tokens"],
    "original_a4_validation": base_prompt_results,
    "soft_prompt_validation": prompt_results,
    "loss_improvement": (
        base_prompt_results["loss"] - prompt_results["loss"]
    ),
    "accuracy_improvement_percentage_points": (
        prompt_results["accuracy"] - base_prompt_results["accuracy"]
    ),
    "generation_quality": "Not yet evaluated",
    "conclusion": (
        "Small validation improvement over frozen A4, "
        "but absolute validation performance is poor."
    )
}

with open(
    "/kaggle/working/phase6_prompt/a4_soft_prompt_experiment.json",
    "w"
) as f:
    json.dump(prompt_record, f, indent=2)

print("Experiment record saved.")

Experiment record saved.


In [208]:
import inspect

print("ENCODER LAYER")
print(inspect.getsource(type(prompt_model.encoder_layers[0]).forward))

print("\nDECODER LAYER")
print(inspect.getsource(type(prompt_model.decoder_layers[0]).forward))

ENCODER LAYER
    def forward(self, x, mask=None):

        h = self.norm1(x)
        x = x + self.attn(h, h, h, mask)

        x = x + self.ffn(self.norm2(x))

        return x


DECODER LAYER
    def forward(
        self,
        x,
        memory,
        self_mask=None,
        cross_mask=None
    ):

        h = self.norm1(x)
        x = x + self.self_attn(h, h, h, self_mask)

        h = self.norm2(x)
        x = x + self.cross_attn(h, memory, memory, cross_mask)

        x = x + self.ffn(self.norm3(x))

        return x



In [209]:
import inspect

print("========== ENCODER ATTENTION ==========")
print("Class:", type(prompt_model.encoder_layers[0].attn))
print(inspect.getsource(type(prompt_model.encoder_layers[0].attn).forward))

print("\n========== DECODER SELF ATTENTION ==========")
print("Class:", type(prompt_model.decoder_layers[0].self_attn))
print(inspect.getsource(type(prompt_model.decoder_layers[0].self_attn).forward))

print("\n========== DECODER CROSS ATTENTION ==========")
print("Class:", type(prompt_model.decoder_layers[0].cross_attn))
print(inspect.getsource(type(prompt_model.decoder_layers[0].cross_attn).forward))

========== ENCODER ATTENTION ==========
Class: <class '__main__.A4Attention'>
    def forward(self, query, key, value, mask=None):

        B, Tq, C = query.shape
        Tk = key.shape[1]

        q = self.q_proj(query)
        k = self.k_proj(key)
        v = self.v_proj(value)

        q = q.view(B, Tq, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, Tk, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, Tk, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        attention = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention, v)

        output = output.transpose(1, 2).contiguous()
        output = output.view(B, Tq, C)

        return self.out_proj(output)


========== DECODER SELF ATTENTION ==========
Class: <class '__main__.A4Atte

In [210]:
import inspect

attn = prompt_model.encoder_layers[0].attn

print("========== ATTENTION INITIALIZATION ==========")
print(inspect.getsource(type(attn).__init__))

print("\n========== ATTENTION ATTRIBUTES ==========")
for name, param in attn.named_parameters():
    print(name, tuple(param.shape))

print("\n========== ATTENTION CONFIGURATION ==========")
print("Number of heads:", attn.num_heads)
print("Head dimension:", attn.head_dim)
print("Model dimension:", attn.q_proj.in_features)

========== ATTENTION INITIALIZATION ==========
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)


========== ATTENTION ATTRIBUTES ==========
q_proj.weight (512, 512)
q_proj.bias (512,)
k_proj.weight (512, 512)
k_proj.bias (512,)
v_proj.weight (512, 512)
v_proj.bias (512,)
out_proj.weight (512, 512)
out_proj.bias (512,)

========== ATTENTION CONFIGURATION ==========
Number of heads: 8
Head dimension: 64
Model dimension: 512


In [211]:
attention_modules = []

for name, module in prompt_model.named_modules():
    if isinstance(module, A4Attention):
        attention_modules.append((name, module))

print("Total attention modules:", len(attention_modules))

for name, module in attention_modules:
    print(name)

assert len(attention_modules) == 36, (
    f"Expected 36 attention modules, found {len(attention_modules)}"
)

Total attention modules: 18
encoder_layers.0.attn
encoder_layers.1.attn
encoder_layers.2.attn
encoder_layers.3.attn
encoder_layers.4.attn
encoder_layers.5.attn
decoder_layers.0.self_attn
decoder_layers.0.cross_attn
decoder_layers.1.self_attn
decoder_layers.1.cross_attn
decoder_layers.2.self_attn
decoder_layers.2.cross_attn
decoder_layers.3.self_attn
decoder_layers.3.cross_attn
decoder_layers.4.self_attn
decoder_layers.4.cross_attn
decoder_layers.5.self_attn
decoder_layers.5.cross_attn


AssertionError: Expected 36 attention modules, found 18

In [212]:
print("Total attention modules:", len(attention_modules))

for name, module in attention_modules:
    print(name)

assert len(attention_modules) == 18

Total attention modules: 18
encoder_layers.0.attn
encoder_layers.1.attn
encoder_layers.2.attn
encoder_layers.3.attn
encoder_layers.4.attn
encoder_layers.5.attn
decoder_layers.0.self_attn
decoder_layers.0.cross_attn
decoder_layers.1.self_attn
decoder_layers.1.cross_attn
decoder_layers.2.self_attn
decoder_layers.2.cross_attn
decoder_layers.3.self_attn
decoder_layers.3.cross_attn
decoder_layers.4.self_attn
decoder_layers.4.cross_attn
decoder_layers.5.self_attn
decoder_layers.5.cross_attn


In [213]:
prefix_count = inject_prefixes(prefix_model, prefix_length=8)

prefix_parameters = [
    p for p in prefix_model.parameters()
    if p.requires_grad
]

print("Prefix modules:", prefix_count)
print("Trainable parameters:",
      sum(p.numel() for p in prefix_parameters))

assert prefix_count == 18
assert len(prefix_parameters) == 36

NameError: name 'inject_prefixes' is not defined

In [218]:
def inject_prefixes(model, prefix_length=8):
    count = 0

    # Encoder self-attention: 6 modules
    for layer in model.encoder_layers:
        layer.attn = PrefixAttention(
            layer.attn,
            prefix_length=prefix_length
        )
        count += 1

    # Decoder self-attention + cross-attention: 12 modules
    for layer in model.decoder_layers:
        layer.self_attn = PrefixAttention(
            layer.self_attn,
            prefix_length=prefix_length
        )

        layer.cross_attn = PrefixAttention(
            layer.cross_attn,
            prefix_length=prefix_length
        )

        count += 2

    return count

In [219]:
prefix_count = inject_prefixes(
    prefix_model,
    prefix_length=8
)

prefix_parameters = [
    p for p in prefix_model.parameters()
    if p.requires_grad
]

trainable_count = sum(
    p.numel() for p in prefix_parameters
)

print("Prefix modules:", prefix_count)
print("Trainable parameters:", trainable_count)

assert prefix_count == 18
assert trainable_count == 147456

print("Prefix injection successful.")

NameError: name 'prefix_model' is not defined

In [220]:
import torch
import torch.nn as nn

prefix_model = A4Model()
prefix_model.load_state_dict(base_state, strict=True)
prefix_model = prefix_model.to(device)

# Freeze all original A4 parameters
for p in prefix_model.parameters():
    p.requires_grad = False

print("Fresh original A4 model loaded.")
print("Parameters:", sum(p.numel() for p in prefix_model.parameters()))

Fresh original A4 model loaded.
Parameters: 96389120


In [222]:
class PrefixAttention(nn.Module):
    def __init__(self, base_attention, prefix_length=8):
        super().__init__()

        self.base = base_attention
        self.prefix_length = prefix_length

        self.num_heads = base_attention.num_heads
        self.head_dim = base_attention.head_dim
        self.d_model = base_attention.q_proj.in_features

        # Freeze original attention weights
        for p in self.base.parameters():
            p.requires_grad = False

        # Trainable prefix keys and values
        self.prefix_key = nn.Parameter(
            torch.randn(
                1, self.num_heads, prefix_length, self.head_dim
            ) * 0.02
        )

        self.prefix_value = nn.Parameter(
            torch.randn(
                1, self.num_heads, prefix_length, self.head_dim
            ) * 0.02
        )

    def forward(self, query, key, value, mask=None):
        B, Tq, C = query.shape
        Tk = key.shape[1]

        q = self.base.q_proj(query)
        k = self.base.k_proj(key)
        v = self.base.v_proj(value)

        q = q.view(
            B, Tq, self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        pk = self.prefix_key.expand(B, -1, -1, -1)
        pv = self.prefix_value.expand(B, -1, -1, -1)

        k = torch.cat([pk, k], dim=2)
        v = torch.cat([pv, v], dim=2)

        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / (self.head_dim ** 0.5)

        if mask is not None:
            prefix_mask = torch.zeros(
                B, 1, Tq, self.prefix_length,
                dtype=torch.bool,
                device=query.device
            )

            mask = torch.cat([prefix_mask, mask], dim=-1)
            scores = scores.masked_fill(mask, float("-inf"))

        attention = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention, v)
        output = output.transpose(1, 2).contiguous()
        output = output.view(B, Tq, C)

        return self.base.out_proj(output)


print("PrefixAttention class defined successfully!")

PrefixAttention class defined successfully!


In [223]:
print(PrefixAttention)

<class '__main__.PrefixAttention'>


In [224]:
def inject_prefixes(model, prefix_length=8):
    count = 0

    for layer in model.encoder_layers:
        layer.attn = PrefixAttention(layer.attn, prefix_length)
        count += 1

    for layer in model.decoder_layers:
        layer.self_attn = PrefixAttention(
            layer.self_attn, prefix_length
        )
        layer.cross_attn = PrefixAttention(
            layer.cross_attn, prefix_length
        )
        count += 2

    return count


prefix_count = inject_prefixes(prefix_model, prefix_length=8)

prefix_parameters = [
    p for p in prefix_model.parameters()
    if p.requires_grad
]

trainable_count = sum(p.numel() for p in prefix_parameters)

print("Attention modules:", prefix_count)
print("Trainable parameters:", trainable_count)

assert prefix_count == 18
assert trainable_count == 147456

print("Prefix injection successful!")

Attention modules: 18
Trainable parameters: 147456
Prefix injection successful!


In [225]:
prefix_model.eval()

batch = next(iter(dev_loader))

encoder_ids = batch["encoder_ids"].to(device)
decoder_ids = batch["decoder_ids"].to(device)

with torch.no_grad():
    memory, memory_mask = a4_encode(
        prefix_model,
        encoder_ids
    )

    logits = a4_decode(
        prefix_model,
        decoder_ids,
        memory,
        memory_mask
    )

print("Encoder input:", encoder_ids.shape)
print("Decoder input:", decoder_ids.shape)
print("Memory:", memory.shape)
print("Memory mask:", memory_mask.shape)
print("Logits:", logits.shape)
print("All logits finite:", torch.isfinite(logits).all().item())

RuntimeError: Expected all tensors to be on the same device, but got tensors is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_cat)

In [226]:
prefix_model = prefix_model.to(device)

# Verify devices
print("Model device:", next(prefix_model.parameters()).device)

for name, param in prefix_model.named_parameters():
    if param.requires_grad:
        print("Trainable:", name, param.device)
        break

Model device: cuda:0
Trainable: encoder_layers.0.attn.prefix_key cuda:0


In [227]:
device_errors = []

for name, module in prefix_model.named_modules():
    if isinstance(module, PrefixAttention):
        if module.prefix_key.device != device:
            device_errors.append(name + " prefix_key")
        if module.prefix_value.device != device:
            device_errors.append(name + " prefix_value")
        if next(module.base.parameters()).device != device:
            device_errors.append(name + " base")

print("Device errors:", device_errors)

Device errors: []


In [228]:
prefix_model.eval()

batch = next(iter(dev_loader))

encoder_ids = batch["encoder_ids"].to(device)
decoder_ids = batch["decoder_ids"].to(device)

with torch.no_grad():
    memory, memory_mask = a4_encode(
        prefix_model, encoder_ids
    )

    logits = a4_decode(
        prefix_model, decoder_ids, memory, memory_mask
    )

print("Memory:", memory.shape)
print("Memory mask:", memory_mask.shape)
print("Logits:", logits.shape)
print("Finite logits:", torch.isfinite(logits).all().item())

RuntimeError: Sizes of tensors must match except in dimension 3. Expected size 1020 but got size 1 for tensor number 1 in the list.

In [229]:
class PrefixAttention(nn.Module):
    def __init__(self, base_attention, prefix_length=8):
        super().__init__()

        self.base = base_attention
        self.prefix_length = prefix_length

        self.num_heads = base_attention.num_heads
        self.head_dim = base_attention.head_dim
        self.d_model = base_attention.q_proj.in_features

        for p in self.base.parameters():
            p.requires_grad = False

        self.prefix_key = nn.Parameter(
            torch.randn(
                1, self.num_heads, prefix_length, self.head_dim
            ) * 0.02
        )

        self.prefix_value = nn.Parameter(
            torch.randn(
                1, self.num_heads, prefix_length, self.head_dim
            ) * 0.02
        )

    def forward(self, query, key, value, mask=None):

        B, Tq, C = query.shape
        Tk = key.shape[1]

        q = self.base.q_proj(query)
        k = self.base.k_proj(key)
        v = self.base.v_proj(value)

        q = q.view(
            B, Tq, self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        pk = self.prefix_key.expand(B, -1, -1, -1)
        pv = self.prefix_value.expand(B, -1, -1, -1)

        k = torch.cat([pk, k], dim=2)
        v = torch.cat([pv, v], dim=2)

        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / (self.head_dim ** 0.5)

        if mask is not None:

            # Expand original mask to all query positions
            mask = mask.expand(B, 1, Tq, Tk)

            # Prefix positions must remain unmasked
            prefix_mask = torch.zeros(
                B, 1, Tq, self.prefix_length,
                dtype=torch.bool,
                device=query.device
            )

            mask = torch.cat(
                [prefix_mask, mask],
                dim=-1
            )

            scores = scores.masked_fill(mask, float("-inf"))

        attention = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention, v)

        output = output.transpose(1, 2).contiguous()
        output = output.view(B, Tq, C)

        return self.base.out_proj(output)


print("Corrected PrefixAttention defined.")

Corrected PrefixAttention defined.


In [230]:
prefix_model = A4Model()
prefix_model.load_state_dict(base_state, strict=True)
prefix_model = prefix_model.to(device)

for p in prefix_model.parameters():
    p.requires_grad = False

prefix_count = inject_prefixes(prefix_model, prefix_length=8)

trainable_count = sum(
    p.numel()
    for p in prefix_model.parameters()
    if p.requires_grad
)

print("Attention modules:", prefix_count)
print("Trainable parameters:", trainable_count)

assert prefix_count == 18
assert trainable_count == 147456

Attention modules: 18
Trainable parameters: 147456


In [231]:
prefix_model.eval()

batch = next(iter(dev_loader))

encoder_ids = batch["encoder_ids"].to(device)
decoder_ids = batch["decoder_ids"].to(device)

with torch.no_grad():
    memory, memory_mask = a4_encode(
        prefix_model, encoder_ids
    )

    logits = a4_decode(
        prefix_model,
        decoder_ids,
        memory,
        memory_mask
    )

print("Memory:", memory.shape)
print("Memory mask:", memory_mask.shape)
print("Logits:", logits.shape)
print("All logits finite:", torch.isfinite(logits).all().item())

RuntimeError: Expected all tensors to be on the same device, but got tensors is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_cat)

In [232]:
# Move the complete model AFTER prefix injection
prefix_model = prefix_model.to(device)

# Verify every prefix module
for name, module in prefix_model.named_modules():
    if isinstance(module, PrefixAttention):
        print(
            name,
            "| key:", module.prefix_key.device,
            "| value:", module.prefix_value.device,
            "| base:", next(module.base.parameters()).device
        )

encoder_layers.0.attn | key: cuda:0 | value: cuda:0 | base: cuda:0
encoder_layers.1.attn | key: cuda:0 | value: cuda:0 | base: cuda:0
encoder_layers.2.attn | key: cuda:0 | value: cuda:0 | base: cuda:0
encoder_layers.3.attn | key: cuda:0 | value: cuda:0 | base: cuda:0
encoder_layers.4.attn | key: cuda:0 | value: cuda:0 | base: cuda:0
encoder_layers.5.attn | key: cuda:0 | value: cuda:0 | base: cuda:0
decoder_layers.0.self_attn | key: cuda:0 | value: cuda:0 | base: cuda:0
decoder_layers.0.cross_attn | key: cuda:0 | value: cuda:0 | base: cuda:0
decoder_layers.1.self_attn | key: cuda:0 | value: cuda:0 | base: cuda:0
decoder_layers.1.cross_attn | key: cuda:0 | value: cuda:0 | base: cuda:0
decoder_layers.2.self_attn | key: cuda:0 | value: cuda:0 | base: cuda:0
decoder_layers.2.cross_attn | key: cuda:0 | value: cuda:0 | base: cuda:0
decoder_layers.3.self_attn | key: cuda:0 | value: cuda:0 | base: cuda:0
decoder_layers.3.cross_attn | key: cuda:0 | value: cuda:0 | base: cuda:0
decoder_layers.4.s

In [233]:
wrong_devices = []

for name, param in prefix_model.named_parameters():
    if param.device != device:
        wrong_devices.append((name, str(param.device)))

print("Parameters on incorrect devices:", wrong_devices)
assert len(wrong_devices) == 0

print("All parameters are on", device)

Parameters on incorrect devices: []
All parameters are on cuda:0


In [234]:
prefix_model.eval()

batch = next(iter(dev_loader))
encoder_ids = batch["encoder_ids"].to(device)
decoder_ids = batch["decoder_ids"].to(device)

with torch.no_grad():
    memory, memory_mask = a4_encode(
        prefix_model, encoder_ids
    )

    logits = a4_decode(
        prefix_model, decoder_ids, memory, memory_mask
    )

print("Memory:", memory.shape)
print("Memory mask:", memory_mask.shape)
print("Logits:", logits.shape)
print("Finite logits:", torch.isfinite(logits).all().item())

Memory: torch.Size([1, 1020, 512])
Memory mask: torch.Size([1, 1, 1, 1020])
Logits: torch.Size([1, 265, 50000])
Finite logits: True


In [235]:
import torch.nn.functional as F

prefix_model.train()

prefix_optimizer = torch.optim.AdamW(
    [p for p in prefix_model.parameters() if p.requires_grad],
    lr=1e-3,
    weight_decay=0.01
)

prefix_train_losses = []

print("Trainable parameters:",
      sum(p.numel() for p in prefix_model.parameters()
          if p.requires_grad))

Trainable parameters: 147456


In [236]:
train_iterator = iter(train_loader)

for step in range(1, 31):

    try:
        batch = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        batch = next(train_iterator)

    encoder_ids = batch["encoder_ids"].to(device)
    decoder_ids = batch["decoder_ids"].to(device)
    labels = batch["labels"].to(device)

    prefix_optimizer.zero_grad(set_to_none=True)

    memory, memory_mask = a4_encode(
        prefix_model, encoder_ids
    )

    logits = a4_decode(
        prefix_model, decoder_ids, memory, memory_mask
    )

    loss = F.cross_entropy(
        logits.reshape(-1, V),
        labels.reshape(-1),
        ignore_index=-100
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        [p for p in prefix_model.parameters() if p.requires_grad],
        1.0
    )

    prefix_optimizer.step()

    prefix_train_losses.append(loss.item())

    if step == 1 or step % 5 == 0:
        print(
            f"Step {step:02d}/30 | "
            f"Training loss: {loss.item():.4f}"
        )

print("\nMean training loss:",
      sum(prefix_train_losses) / len(prefix_train_losses))

Step 01/30 | Training loss: 19.5695
Step 05/30 | Training loss: 17.5916
Step 10/30 | Training loss: 18.1823
Step 15/30 | Training loss: 20.8563
Step 20/30 | Training loss: 16.2444
Step 25/30 | Training loss: 18.6975
Step 30/30 | Training loss: 16.1958

Mean training loss: 19.336652565002442


In [237]:
def evaluate_prefix_model(max_batches=20):
    prefix_model.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    with torch.no_grad():
        for i, batch in enumerate(dev_loader):

            if i >= max_batches:
                break

            encoder_ids = batch["encoder_ids"].to(device)
            decoder_ids = batch["decoder_ids"].to(device)
            labels = batch["labels"].to(device)

            memory, memory_mask = a4_encode(
                prefix_model, encoder_ids
            )

            logits = a4_decode(
                prefix_model, decoder_ids, memory, memory_mask
            )

            valid = labels != -100

            loss = F.cross_entropy(
                logits.reshape(-1, V),
                labels.reshape(-1),
                ignore_index=-100,
                reduction="sum"
            )

            predictions = logits.argmax(dim=-1)

            total_loss += loss.item()
            total_correct += (
                ((predictions == labels) & valid).sum().item()
            )
            total_tokens += valid.sum().item()

    return {
        "loss": total_loss / total_tokens,
        "accuracy": 100 * total_correct / total_tokens,
        "tokens": total_tokens
    }


prefix_results = evaluate_prefix_model()

print("PREFIX TUNING RESULTS")
print(prefix_results)

PREFIX TUNING RESULTS
{'loss': 17.535338861496946, 'accuracy': 3.204920686306248, 'tokens': 6178}


In [238]:
import os
import json

os.makedirs("/kaggle/working/phase6_prefix", exist_ok=True)

prefix_record = {
    "experiment": "A4 Prefix Tuning",
    "base_checkpoint": "A4 Phase 1 step 22000",
    "prefix_length": 8,
    "attention_modules": 18,
    "trainable_parameters": 147456,
    "training_updates": len(prefix_train_losses),
    "mean_training_loss": sum(prefix_train_losses) / len(prefix_train_losses),
    "validation_results": prefix_results,
    "original_a4_validation": base_prompt_results,
    "soft_prompt_validation": prompt_results,
    "generation_quality": "Not yet evaluated",
    "conclusion": "Weak pilot; modest improvement over original A4, "
                  "but worse validation CE than soft prompting."
}

with open(
    "/kaggle/working/phase6_prefix/a4_prefix_experiment.json",
    "w"
) as f:
    json.dump(prefix_record, f, indent=2)

torch.save(
    {
        "prefix_state": {
            name: param.detach().cpu()
            for name, param in prefix_model.named_parameters()
            if param.requires_grad
        },
        "prefix_length": 8,
        "validation_results": prefix_results
    },
    "/kaggle/working/phase6_prefix/a4_prefix_step_30.pt"
)

print("Prefix Tuning experiment saved.")

Prefix Tuning experiment saved.


In [245]:
A2_CHECKPOINT = (
    "/kaggle/input/datasets/aq1de3/"
    "a2-step-9000-pt/a2_step_9000.pt"
)

In [246]:
import os
import torch

print("Checkpoint exists:", os.path.exists(A2_CHECKPOINT))

assert os.path.exists(A2_CHECKPOINT), "A2 checkpoint not found!"

checkpoint = torch.load(
    A2_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} entries")
        elif torch.is_tensor(value):
            print(f"{key}: Tensor {tuple(value.shape)}")
        else:
            print(f"{key}: {value}")

Checkpoint exists: True

Checkpoint type: <class 'dict'>

Checkpoint keys:
model_state_dict: dictionary with 113 entries
step: 9000
tokens_processed: 73728000
train_loss: 6.801188290119171
reason: regular_checkpoint
saved_at: 2026-10-05T19:17:55.016904


In [247]:
state = checkpoint.get(
    "model_state_dict",
    checkpoint.get("model", checkpoint)
)

print("Total state entries:", len(state))

for i, (name, tensor) in enumerate(state.items()):
    if torch.is_tensor(tensor):
        print(name, tuple(tensor.shape))

    if i >= 39:
        print("... remaining parameters omitted")
        break

Total state entries: 113
output_bias (50000,)
token_embedding.weight (50000, 512)
position_embedding.weight (2048, 512)
layers.0.norm1.weight (512,)
layers.0.norm1.bias (512,)
layers.0.attention.q_proj.weight (512, 512)
layers.0.attention.k_proj.weight (128, 512)
layers.0.attention.v_proj.weight (128, 512)
layers.0.attention.out_proj.weight (512, 512)
layers.0.norm2.weight (512,)
layers.0.norm2.bias (512,)
layers.0.ffn.fc1.weight (2048, 512)
layers.0.ffn.fc1.bias (2048,)
layers.0.ffn.fc2.weight (512, 2048)
layers.0.ffn.fc2.bias (512,)
layers.1.norm1.weight (512,)
layers.1.norm1.bias (512,)
layers.1.attention.q_proj.weight (512, 512)
layers.1.attention.k_proj.weight (128, 512)
layers.1.attention.v_proj.weight (128, 512)
layers.1.attention.out_proj.weight (512, 512)
layers.1.norm2.weight (512,)
layers.1.norm2.bias (512,)
layers.1.ffn.fc1.weight (2048, 512)
layers.1.ffn.fc1.bias (2048,)
layers.1.ffn.fc2.weight (512, 2048)
layers.1.ffn.fc2.bias (512,)
layers.2.norm1.weight (512,)
layers.2.

In [248]:
import os

TOKENIZER_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "tokenizer-json/tokenizer.json"
)

print("Phase 6 tokenizer exists:",
      os.path.exists(TOKENIZER_PATH))

print("Tokenizer vocabulary:", tokenizer.get_vocab_size())

print("A2 embedding shape:",
      [
          tuple(v.shape)
          for k, v in state.items()
          if "embedding" in k.lower()
          and torch.is_tensor(v)
      ])

Phase 6 tokenizer exists: True
Tokenizer vocabulary: 50000
A2 embedding shape: [(50000, 512), (2048, 512)]


In [249]:
# Inspect all checkpoint keys
state = checkpoint["model_state_dict"]

print("========== A2 CHECKPOINT STRUCTURE ==========")

# Identify transformer layers
layer_ids = set()

for key in state:
    if key.startswith("layers."):
        layer_id = int(key.split(".")[1])
        layer_ids.add(layer_id)

print("Number of transformer layers:", len(layer_ids))
print("Layer IDs:", sorted(layer_ids))

print("\n========== NON-LAYER PARAMETERS ==========")

for key, value in state.items():
    if not key.startswith("layers."):
        print(key, tuple(value.shape))

print("\n========== ATTENTION PARAMETERS ==========")

for key, value in state.items():
    if ".attention." in key and key.startswith("layers.0."):
        print(key, tuple(value.shape))

print("\n========== LAYER 0 COMPLETE STRUCTURE ==========")

for key, value in state.items():
    if key.startswith("layers.0."):
        print(key, tuple(value.shape))

========== A2 CHECKPOINT STRUCTURE ==========
Number of transformer layers: 9
Layer IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8]

========== NON-LAYER PARAMETERS ==========
output_bias (50000,)
token_embedding.weight (50000, 512)
position_embedding.weight (2048, 512)
final_norm.weight (512,)
final_norm.bias (512,)

========== ATTENTION PARAMETERS ==========
layers.0.attention.q_proj.weight (512, 512)
layers.0.attention.k_proj.weight (128, 512)
layers.0.attention.v_proj.weight (128, 512)
layers.0.attention.out_proj.weight (512, 512)

========== LAYER 0 COMPLETE STRUCTURE ==========
layers.0.norm1.weight (512,)
layers.0.norm1.bias (512,)
layers.0.attention.q_proj.weight (512, 512)
layers.0.attention.k_proj.weight (128, 512)
layers.0.attention.v_proj.weight (128, 512)
layers.0.attention.out_proj.weight (512, 512)
layers.0.norm2.weight (512,)
layers.0.norm2.bias (512,)
layers.0.ffn.fc1.weight (2048, 512)
layers.0.ffn.fc1.bias (2048,)
layers.0.ffn.fc2.weight (512, 2048)
layers.0.ffn.fc2.bias (512,)


In [250]:
import json

with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
    tokenizer_json = json.load(f)

print("========== TOKENIZER METADATA ==========")

print("Model type:",
      tokenizer_json.get("model", {}).get("type"))

print("Vocabulary entries:",
      len(tokenizer_json.get("model", {}).get("vocab", {})))

print("Special tokens:")
print(tokenizer_json.get("added_tokens", [])[:10])

print("Pre-tokenizer:")
print(tokenizer_json.get("pre_tokenizer"))

========== TOKENIZER METADATA ==========
Model type: BPE
Vocabulary entries: 50000
Special tokens:
[{'id': 0, 'content': '[PAD]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 1, 'content': '[UNK]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 2, 'content': '[BOS]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 3, 'content': '[EOS]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 4, 'content': '[MASK]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}]
Pre-tokenizer:
{'type': 'ByteLevel', 'add_prefix_space': False, 'trim_offsets': True, 'use_regex': True}


In [251]:
state = checkpoint["model_state_dict"]

layer_ids = sorted({
    int(key.split(".")[1])
    for key in state
    if key.startswith("layers.")
})

print("Number of layers:", len(layer_ids))
print("Layer IDs:", layer_ids)

print("\n========== NON-LAYER PARAMETERS ==========")
for key, value in state.items():
    if not key.startswith("layers."):
        print(key, tuple(value.shape))

print("\n========== LAYER 0 ==========")
for key, value in state.items():
    if key.startswith("layers.0."):
        print(key, tuple(value.shape))

print("\n========== ALL ATTENTION PROJECTIONS ==========")
for key, value in state.items():
    if ".attention." in key:
        print(key, tuple(value.shape))

Number of layers: 9
Layer IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8]

========== NON-LAYER PARAMETERS ==========
output_bias (50000,)
token_embedding.weight (50000, 512)
position_embedding.weight (2048, 512)
final_norm.weight (512,)
final_norm.bias (512,)

========== LAYER 0 ==========
layers.0.norm1.weight (512,)
layers.0.norm1.bias (512,)
layers.0.attention.q_proj.weight (512, 512)
layers.0.attention.k_proj.weight (128, 512)
layers.0.attention.v_proj.weight (128, 512)
layers.0.attention.out_proj.weight (512, 512)
layers.0.norm2.weight (512,)
layers.0.norm2.bias (512,)
layers.0.ffn.fc1.weight (2048, 512)
layers.0.ffn.fc1.bias (2048,)
layers.0.ffn.fc2.weight (512, 2048)
layers.0.ffn.fc2.bias (512,)

========== ALL ATTENTION PROJECTIONS ==========
layers.0.attention.q_proj.weight (512, 512)
layers.0.attention.k_proj.weight (128, 512)
layers.0.attention.v_proj.weight (128, 512)
layers.0.attention.out_proj.weight (512, 512)
layers.1.attention.q_proj.weight (512, 512)
layers.1.attention.k_proj.weigh

In [252]:
import os

search_roots = [
    "/kaggle/input/datasets/aq1de3",
    "/kaggle/input/datasets"
]

matches = []

for root in search_roots:
    for current_root, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in [".git", "__pycache__"]
        ]

        for filename in files:
            if filename.lower() in [
                "tokenizer.json",
                "vocab.json",
                "merges.txt",
                "tokenizer.model"
            ]:
                matches.append(os.path.join(current_root, filename))

print("Tokenizer-related files:")
for path in sorted(set(matches)):
    print(path)

Tokenizer-related files:
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json


In [253]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/aq1de3"):
    for filename in files:
        if filename.endswith(".ipynb"):
            print(os.path.join(root, filename))

/kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
/kaggle/input/datasets/aq1de3/p3p3p3p3/phase3-capstone.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a2a2a2.ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a3-phase1 (1).ipynb
/kaggle/input/datasets/aq1de3/p1p1p1/a4a4a4.ipynb
/kaggle/input/datasets/aq1de3/notebook1/notebook1d0fcbb0df (3).ipynb
/kaggle/input/datasets/aq1de3/p2p2p2/phase-2 (1).ipynb


In [254]:
import json

notebook_path = (
    "/kaggle/input/datasets/aq1de3/"
    "p1p1p1/a2a2a2.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("Total notebook cells:", len(nb["cells"]))

keywords = [
    "class ",
    "grouped",
    "gqa",
    "num_kv_heads",
    "num_heads",
    "kv_heads",
    "repeat_interleave",
    "position_embedding",
    "output_bias"
]

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    if cell["cell_type"] == "code" and any(
        word.lower() in source.lower()
        for word in keywords
    ):
        print("\n" + "=" * 70)
        print("CELL:", i)
        print("=" * 70)
        print(source[:15000])

Total notebook cells: 11

CELL: 0
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = 50_000
D_MODEL = 528
NUM_LAYERS = 8
NUM_Q_HEADS = 8
NUM_KV_HEADS = 2
FFN_DIM = 2048
MAX_SEQ_LEN = 2048
DROPOUT = 0.1


class GQAAttention(nn.Module):
    def __init__(self, d_model, num_q_heads, num_kv_heads, dropout=0.1):
        super().__init__()

        self.d_model = d_model
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_q_heads

        assert d_model % num_q_heads == 0

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
  

In [255]:
path = (
    "/kaggle/input/datasets/aq1de3/"
    "phase1-model/phase1_model_code_extract.ipynb"
)

with open(path, "r", encoding="utf-8") as f:
    nb2 = json.load(f)

print("Total cells:", len(nb2["cells"]))

for i, cell in enumerate(nb2["cells"]):
    source = "".join(cell.get("source", []))

    if any(x in source.lower() for x in [
        "a2",
        "gqa",
        "grouped query",
        "num_kv_heads"
    ]):
        print("\n========== CELL", i, "==========")
        print(source[:15000])

Total cells: 13

========== CELL 0 ==========
# Phase 1 model-code extracts (from the uploaded notebook)

These are copied source cells from the supplied notebook, not a verified runnable loader. Do not Run All. The notebook contains multiple architecture variants and benchmark/training cells.

**Compatibility warning:** checkpoint inspection showed A1 keys such as `blocks.0.norm1` (not standard Hugging Face GPT-2 keys); A2 checkpoint tensors are width 512, while the later A2 cell 122 configures width 528. Therefore these cells must be matched against the exact saved checkpoint before use. A3 and A4 also need strict state-dict verification.

This extract intentionally omits pretraining loops and corpus download/training cells.

========== CELL 3 ==========
## Original notebook cell 13: A2 GQA model definition (earlier variant; compare against checkpoint keys)

Copied verbatim from the supplied notebook. Review paths/configuration before execution.


========== CELL 4 ==========
# =====

In [256]:
import os
import json
import re

ROOT = "/kaggle/input/datasets/aq1de3"

patterns = [
    r"D_MODEL\s*=\s*512",
    r"NUM_LAYERS\s*=\s*9",
    r"N_LAYERS\s*=\s*9",
    r"output_bias",
    r"class\s+GQAAttention",
    r"self\.layers\s*=",
    r"self\.attention\s*=",
]

for root, dirs, files in os.walk(ROOT):
    for filename in files:

        if not filename.endswith(".ipynb"):
            continue

        path = os.path.join(root, filename)

        try:
            with open(path, "r", encoding="utf-8") as f:
                notebook = json.load(f)

            for i, cell in enumerate(notebook.get("cells", [])):

                source = "".join(cell.get("source", []))

                hits = [
                    pattern for pattern in patterns
                    if re.search(pattern, source, re.IGNORECASE)
                ]

                if hits:
                    print("\n" + "=" * 80)
                    print("NOTEBOOK:", path)
                    print("CELL:", i)
                    print("MATCHES:", hits)
                    print("=" * 80)
                    print(source[:10000])

        except Exception as e:
            print("Could not inspect:", path, str(e))


NOTEBOOK: /kaggle/input/datasets/aq1de3/phase1-model/phase1_model_code_extract.ipynb
CELL: 4
MATCHES: ['class\\s+GQAAttention', 'self\\.attention\\s*=']
# ============================================================
# PHASE 1 — ARCHITECTURE 2
# GQA — UPDATED MODEL WIDTH
# PARAMETER + GPU CHECK
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

print("=" * 70)
print("ARCHITECTURE 2 — GQA UPDATED RESOURCE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/working/legal_bpe_tokenizer/tokenizer.json"
)

VOCAB_SIZE = Tokenizer.from_file(
    TOKENIZER_PATH
).get_vocab_size()

SEQ_LEN = 2048

D_MODEL = 528
N_LAYERS = 8
N_Q_HEADS = 8
N_KV_HEADS = 2
FFN_DIM = 2048

HEAD_DIM = D_MODEL // N_Q_HEADS
GROUP_SIZE = N_Q_HEADS // N

In [257]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# A2 — CHECKPOINT-COMPATIBLE RECONSTRUCTION
# ============================================================

A2_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "a2-step-9000-pt/a2_step_9000.pt"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    A2_PATH,
    map_location="cpu",
    weights_only=False
)

state = checkpoint["model_state_dict"]

# Infer dimensions from checkpoint
vocab_size = state["token_embedding.weight"].shape[0]
hidden_size = state["token_embedding.weight"].shape[1]
context_length = state["position_embedding.weight"].shape[0]

num_layers = len({
    int(k.split(".")[1])
    for k in state
    if k.startswith("layers.")
    and k.split(".")[1].isdigit()
})

ffn_hidden = state["layers.0.ffn.fc1.weight"].shape[0]

num_heads = 8
num_kv_heads = 2

# ------------------------------------------------------------
# GQA Attention
# ------------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, num_kv_heads):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(
            hidden_size, hidden_size, bias=False
        )
        self.k_proj = nn.Linear(
            hidden_size, num_kv_heads * self.head_dim, bias=False
        )
        self.v_proj = nn.Linear(
            hidden_size, num_kv_heads * self.head_dim, bias=False
        )
        self.out_proj = nn.Linear(
            hidden_size, hidden_size, bias=False
        )

    def forward(self, x, causal_mask=None):
        B, T, C = x.shape

        q = self.q_proj(x).view(
            B, T, self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = self.k_proj(x).view(
            B, T, self.num_kv_heads, self.head_dim
        ).transpose(1, 2)

        v = self.v_proj(x).view(
            B, T, self.num_kv_heads, self.head_dim
        ).transpose(1, 2)

        repeat_factor = self.num_heads // self.num_kv_heads
        k = k.repeat_interleave(repeat_factor, dim=1)
        v = v.repeat_interleave(repeat_factor, dim=1)

        scores = torch.matmul(
            q, k.transpose(-2, -1)
        ) / (self.head_dim ** 0.5)

        if causal_mask is not None:
            scores = scores.masked_fill(
                causal_mask, float("-inf")
            )

        attention = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention, v)

        output = output.transpose(
            1, 2
        ).contiguous().view(B, T, C)

        return self.out_proj(output)


# ------------------------------------------------------------
# A2 Layer — checkpoint-compatible FFN names
# ------------------------------------------------------------

class A2Layer(nn.Module):
    def __init__(
        self, hidden_size, num_heads,
        num_kv_heads, ffn_hidden
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)

        self.attention = GQAAttention(
            hidden_size, num_heads, num_kv_heads
        )

        self.norm2 = nn.LayerNorm(hidden_size)

        self.ffn = nn.ModuleDict({
            "fc1": nn.Linear(hidden_size, ffn_hidden),
            "fc2": nn.Linear(ffn_hidden, hidden_size)
        })

    def forward(self, x, causal_mask):
        x = x + self.attention(
            self.norm1(x), causal_mask
        )

        y = self.ffn["fc1"](self.norm2(x))
        y = F.gelu(y)
        y = self.ffn["fc2"](y)

        return x + y


# ------------------------------------------------------------
# A2 Model — tied output projection, no lm_head module
# ------------------------------------------------------------

class A2Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size, hidden_size
        )

        self.position_embedding = nn.Embedding(
            context_length, hidden_size
        )

        self.layers = nn.ModuleList([
            A2Layer(
                hidden_size,
                num_heads,
                num_kv_heads,
                ffn_hidden
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_size)

        self.output_bias = nn.Parameter(
            torch.zeros(vocab_size)
        )

    def forward(self, input_ids):
        B, T = input_ids.shape

        if T > context_length:
            raise ValueError("Sequence exceeds A2 context length")

        positions = torch.arange(
            T, device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        causal_mask = torch.triu(
            torch.ones(
                T, T,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )[None, None, :, :]

        for layer in self.layers:
            x = layer(x, causal_mask)

        x = self.final_norm(x)

        logits = F.linear(
            x, self.token_embedding.weight
        )

        return logits + self.output_bias


# ============================================================
# STRICT LOAD + FORWARD TEST
# ============================================================

model_a2 = A2Model()

print("Checkpoint step:", checkpoint.get("step"))
print("Checkpoint tensors:", len(state))
print("Model parameters:", sum(
    p.numel() for p in model_a2.parameters()
))

# This must succeed without missing or unexpected keys.
model_a2.load_state_dict(state, strict=True)

model_a2 = model_a2.to(device).eval()

test_ids = torch.randint(
    0, vocab_size, (1, 128), device=device
)

with torch.no_grad():
    logits = model_a2(test_ids)

print("\nSTRICT LOAD: PASS")
print("Input shape:", tuple(test_ids.shape))
print("Output shape:", tuple(logits.shape))
print("Finite logits:", bool(torch.isfinite(logits).all()))
print("A2 reconstruction validation complete.")

Checkpoint step: 9000
Checkpoint tensors: 113
Model parameters: 51513680

STRICT LOAD: PASS
Input shape: (1, 128)
Output shape: (1, 128, 50000)
Finite logits: True
A2 reconstruction validation complete.


In [258]:
# ============================================================
# A2 — BASE CHECKPOINT GENERATION SMOKE TEST
# Diagnostic only: Phase 6 tokenizer compatibility unverified
# ============================================================

import torch

model_a2.eval()

# Use a short validation document
sample = phase6_dev[0]
document = sample["document"]

prompt = (
    "Summarize the following legal document:\n\n"
    + document
    + "\n\nSummary:"
)

# Encode with the currently available Phase 6 tokenizer
encoded = tokenizer.encode(prompt)
input_ids = encoded.ids[:1400]

input_ids = torch.tensor(
    [input_ids],
    dtype=torch.long,
    device=device
)

# Greedy autoregressive generation
generated = input_ids.clone()

MAX_NEW_TOKENS = 128
EOS_ID = 3

with torch.no_grad():
    for _ in range(MAX_NEW_TOKENS):

        # Keep within A2 context limit
        context = generated[:, -2048:]

        logits = model_a2(context)
        next_token = logits[:, -1, :].argmax(
            dim=-1, keepdim=True
        )

        generated = torch.cat(
            [generated, next_token], dim=1
        )

        if next_token.item() == EOS_ID:
            break

# Decode only newly generated tokens
new_tokens = generated[0, input_ids.shape[1]:].tolist()
generated_text = tokenizer.decode(new_tokens)

print("Document ID:", sample["id"])
print("\nGENERATED SUMMARY:\n")
print(generated_text)

print("\nGenerated token count:", len(new_tokens))
print("EOS generated:", EOS_ID in new_tokens)

Document ID: 4494

GENERATED SUMMARY:

 the State Government of the date of the case may be, the date of the date of the State Government may be, the case may be made under the case of the State Government may be used in the date of the case may be, the date of the State Government may be made in the date of the Committee, the case of the date of the case may be, the case of the juvenile of the juvenile of the juvenile of the case of the case may be, the case of the State Government of the case of the Committee.
(1) of the Committee, the date of the case of the Act, the Committee,

Generated token count: 128
EOS generated: False


In [259]:
import os
import json

ROOT = "/kaggle/input/datasets/aq1de3"

print("=" * 75)
print("TOKENIZER-RELATED FILES")
print("=" * 75)

matches = []

for root, dirs, files in os.walk(ROOT):
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for filename in files:
        lower = filename.lower()

        if any(term in lower for term in [
            "tokenizer", "vocab", "merges", "bpe"
        ]):
            path = os.path.join(root, filename)
            matches.append(path)
            print(path)

print("\nTotal matching files:", len(matches))

print("\n" + "=" * 75)
print("A2 NOTEBOOK TOKENIZER REFERENCES")
print("=" * 75)

notebook_path = (
    ROOT + "/p1p1p1/a3-phase1 (1).ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))

    relevant_lines = [
        line.strip()
        for line in source.splitlines()
        if any(term in line.lower() for term in [
            "tokenizer", "tokenizer_path",
            "tokenizer.json", "from_file",
            "save_model", "save("
        ])
    ]

    if relevant_lines:
        print(f"\nCell {i}:")
        for line in relevant_lines:
            print(" ", line[:250])

TOKENIZER-RELATED FILES
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json
/kaggle/input/datasets/aq1de3/phase3/label_vocab.json

Total matching files: 2

A2 NOTEBOOK TOKENIZER REFERENCES

Cell 0:
  from tokenizers import Tokenizer
  TOKENIZER_PATH = "/kaggle/input/datasets/mathimohankowshik/tokenizer1/tokenizer.json"
  print("Tokenizer    :", os.path.exists(TOKENIZER_PATH))
  assert os.path.exists(TOKENIZER_PATH), "❌ tokenizer.json not found"

Cell 1:
  # A2 — LOCATE EXISTING TOKENIZER
  if file == "tokenizer.json":
  print("TOKENIZER SEARCH")
  print("❌ tokenizer.json not found under /kaggle/input")

Cell 2:
  # A2 — TOKENIZER + MODEL CONFIG
  # Load the Phase-1 tokenizer
  tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
  VOCAB_SIZE = tokenizer.get_vocab_size()
  print("✅ TOKENIZER + CONFIG READY")

Cell 7:
  ids = tokenizer.encode(text).ids

Cell 8:
  from tokenizers import Tokenizer
  TOKENIZER_PATH = (
  "mathimohankowshik/tokenizer1/tokenizer.json"
  assert os.path.exis

In [260]:
import os
import json

ROOT = "/kaggle/working"

print("=" * 75)
print("PHASE 6 EXPERIMENT ARTIFACT INVENTORY")
print("=" * 75)

for folder in [
    "phase6_a1",
    "phase6_hierarchical",
    "phase6_lorma",
    "phase6_kd",
    "phase6_prompt",
    "phase6_prefix"
]:
    path = os.path.join(ROOT, folder)

    print(f"\n{'=' * 60}")
    print("FOLDER:", folder)

    if not os.path.exists(path):
        print("Not found")
        continue

    for filename in sorted(os.listdir(path)):
        full_path = os.path.join(path, filename)
        size_mb = os.path.getsize(full_path) / (1024 ** 2)

        print(f"\nFile: {filename} ({size_mb:.2f} MB)")

        if filename.endswith(".json"):
            try:
                with open(full_path, "r") as f:
                    data = json.load(f)

                if isinstance(data, dict):
                    for key, value in data.items():
                        if isinstance(value, (str, int, float, bool)):
                            print(f"  {key}: {value}")
                        elif isinstance(value, list):
                            print(f"  {key}: list with {len(value)} items")
                        elif isinstance(value, dict):
                            print(f"  {key}: dictionary")
            except Exception as e:
                print("  Could not read JSON:", e)

PHASE 6 EXPERIMENT ARTIFACT INVENTORY

FOLDER: phase6_a1

File: a1_full_finetune_results.json (0.00 MB)
  experiment: A1_Full_Finetuning
  architecture: A1 decoder-only Transformer
  checkpoint_source: /kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
  checkpoint_initial_step: 500
  training_updates: 200
  training_records: 512
  validation_records: 64
  max_document_tokens: 1200
  max_summary_tokens: 800
  learning_rate: 1e-05
  final_train_loss: 6.299882936477661
  validation_loss: 6.239953763782978
  tokenizer_compatibility: UNVERIFIED
  status: COMPLETED

File: a1_full_finetune_step_100.pt (886.68 MB)

File: a1_full_finetune_step_150.pt (886.68 MB)

File: a1_full_finetune_step_200.pt (886.68 MB)

File: a1_full_finetune_step_50.pt (886.68 MB)

File: a1_generation_samples.json (0.02 MB)

FOLDER: phase6_hierarchical

File: a4_hierarchical_fulltrain_step_110.pt (784.63 MB)

File: a4_hierarchical_fulltrain_step_120.pt (784.63 MB)

Fil

In [261]:
import json

path = (
    "/kaggle/working/phase6_prompt/"
    "a4_soft_prompt_experiment.json"
)

with open(path, "r") as f:
    record = json.load(f)

print(json.dumps(record, indent=2))

{
  "experiment": "A4 Soft Prompt Tuning",
  "base_checkpoint": "A4 Phase 1 step 22000",
  "prompt_length": 16,
  "trainable_parameters": 8192,
  "training_updates": 30,
  "learning_rate": 0.001,
  "validation_tokens": 6178,
  "original_a4_validation": {
    "loss": 19.37780860513111,
    "accuracy": 2.865004855940434,
    "tokens": 6178
  },
  "soft_prompt_validation": {
    "loss": 17.28407935194693,
    "accuracy": 3.1725477500809323,
    "tokens": 6178
  },
  "loss_improvement": 2.0937292531841827,
  "accuracy_improvement_percentage_points": 0.3075428941404983,
  "generation_quality": "Not yet evaluated",
  "conclusion": "Small validation improvement over frozen A4, but absolute validation performance is poor."
}


In [262]:
# ============================================================
# SOFT PROMPT — CHECKPOINT AND RUNTIME DIAGNOSTIC
# ============================================================

print("Checking soft prompt experiment...")

for name in ["prompt_model", "soft_prompt", "prompt", "model"]:
    obj = globals().get(name)

    if obj is not None:
        print(f"\n{name}:")
        print("Type:", type(obj))

        if hasattr(obj, "state_dict"):
            params = {
                k: tuple(v.shape)
                for k, v in obj.state_dict().items()
                if "prompt" in k.lower()
            }
            print("Prompt-related parameters:", params)

print("\nPrompt experiment directory:")
import os

folder = "/kaggle/working/phase6_prompt"

for filename in os.listdir(folder):
    print(filename)

Checking soft prompt experiment...

prompt_model:
Type: <class '__main__.A4Model'>
Prompt-related parameters: {}

soft_prompt:
Type: <class 'torch.nn.parameter.Parameter'>

prompt:
Type: <class 'str'>

model:
Type: <class '__main__.A4Model'>
Prompt-related parameters: {}

Prompt experiment directory:
a4_soft_prompt_experiment.json


In [263]:
# ============================================================
# INSPECT TRAINED SOFT PROMPT
# ============================================================

import torch

print("=" * 60)
print("SOFT PROMPT DIAGNOSTICS")
print("=" * 60)

print("Type:", type(soft_prompt))
print("Shape:", tuple(soft_prompt.shape))
print("Device:", soft_prompt.device)
print("Requires gradient:", soft_prompt.requires_grad)
print("Mean:", soft_prompt.detach().float().mean().item())
print("Std:", soft_prompt.detach().float().std().item())
print("Norm:", soft_prompt.detach().float().norm().item())

print("\nPrompt text:")
print(prompt)

print("\nAvailable generation-related functions:")

for name, obj in list(globals().items()):
    if callable(obj) and any(
        word in name.lower()
        for word in ["generate", "decode", "prompt"]
    ):
        print(name)

SOFT PROMPT DIAGNOSTICS
Type: <class 'torch.nn.parameter.Parameter'>
Shape: (1, 16, 512)
Device: cuda:0
Requires gradient: True
Mean: -0.0001614212669664994
Std: 0.021279511973261833
Norm: 1.9259394407272339

Prompt text:
Summarize the following legal document:

ition Nos. 5670 and 6216 of 1980. (Under Article 32 of the Constitution) Mrs. K. Hingorani, Mr. Hingorani, Mukul Mudgal and Damodar Prakash for the Petitioners. K G. Bhagat and D. Goburdhan for the Respondent. Miss A. Subhashini for the Union of India. The Order of the Court was delivered by, BHAGWATI, J. The question which arises before us for consideration is whether certain documents called for by the Court by its order dated 16th February, 1981 are liable to be produced by the State or their production is barred under some provision of law. The documents called for are set out in the order dated 16th February, 1981 and they are as follows: 1. the CID report submitted by L.V. Singh, DIG, CID Anti Dacoity) on December 9, 1980

In [264]:
import inspect

functions_to_inspect = [
    "a4_encode_soft_prompt",
    "evaluate_soft_prompt",
    "generate_summary"
]

for name in functions_to_inspect:
    fn = globals().get(name)

    print("\n" + "=" * 75)
    print("FUNCTION:", name)
    print("=" * 75)

    if fn is None:
        print("Not found")
        continue

    print("Signature:", inspect.signature(fn))
    print(inspect.getsource(fn))


FUNCTION: a4_encode_soft_prompt
Signature: (current_model, encoder_ids, prompt)
def a4_encode_soft_prompt(current_model, encoder_ids, prompt):
    B, Te = encoder_ids.shape

    if Te > MAX_POSITION:
        raise ValueError(f"Input length {Te} exceeds {MAX_POSITION}")

    positions = torch.arange(Te, device=encoder_ids.device)

    # Original token and positional embeddings
    encoder = (
        current_model.token_embedding(encoder_ids)
        + current_model.encoder_pos(positions)[None, :, :]
    )

    # Expand the learnable prompt for the batch
    prompt_batch = prompt.expand(B, -1, -1)

    # Prepend soft prompt tokens
    encoder = torch.cat([prompt_batch, encoder], dim=1)

    # Original document padding mask
    document_padding_mask = (
        encoder_ids == PAD_ID
    )[:, None, None, :]

    # Prompt tokens are never padding
    prompt_padding_mask = torch.zeros(
        B, 1, 1, PROMPT_LENGTH,
        dtype=torch.bool,
        device=encoder_ids.device
    )

    me

In [265]:
# ============================================================
# A4 SOFT PROMPT — GENERATION EVALUATION
# ============================================================

import torch
import json

prompt_model.eval()

MAX_NEW_TOKENS = 128
NUM_SAMPLES = 5

def generate_soft_prompt_summary(document):

    # Tokenize document using the available tokenizer
    ids = tokenizer.encode(document).ids[:MAX_POSITION]

    encoder_ids = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        # Encode using trained soft prompt
        memory, memory_mask = a4_encode_soft_prompt(
            prompt_model,
            encoder_ids,
            soft_prompt
        )

        # Start decoder with BOS
        generated = torch.tensor(
            [[BOS_ID]],
            dtype=torch.long,
            device=device
        )

        for _ in range(MAX_NEW_TOKENS):

            logits = a4_decode(
                prompt_model,
                generated,
                memory,
                memory_mask
            )

            next_token = logits[:, -1, :].argmax(
                dim=-1,
                keepdim=True
            )

            generated = torch.cat(
                [generated, next_token],
                dim=1
            )

            if next_token.item() == EOS_ID:
                break

    return tokenizer.decode(
        generated[0, 1:].tolist()
    )


# ------------------------------------------------------------
# Generate on validation documents
# ------------------------------------------------------------

generation_results = []

for i, sample in enumerate(phase6_dev):

    if i >= NUM_SAMPLES:
        break

    document = sample["document"]
    reference = sample["summary"]

    prediction = generate_soft_prompt_summary(document)

    result = {
        "id": sample["id"],
        "prediction": prediction,
        "reference": reference
    }

    generation_results.append(result)

    print("\n" + "=" * 80)
    print("DOCUMENT ID:", sample["id"])
    print("=" * 80)

    print("\nGENERATED SUMMARY:\n")
    print(prediction)

    print("\nREFERENCE SUMMARY:\n")
    print(reference)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

output_path = (
    "/kaggle/working/phase6_prompt/"
    "a4_soft_prompt_generation.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        generation_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved:", output_path)
print("Documents evaluated:", len(generation_results))


DOCUMENT ID: 4494

GENERATED SUMMARY:

 Nos. 5670 and 6216 of 1980. (Under Article 32 of the Constitution) Mrs. K fund,orani, Mr Unit Unit Unit Unit, RPCPort and total number for the Power. K G personickel and D.ingurd officer for the Respondent for nine years. industries test for in Union of India.. Order of the Court was delivered by333 J J J question.2003. us ( consideration is whether certain documents called for by the Court by its order dated 16th February February, are statement to be produced by the State or their production is thumb under some provision of law. The documents called for

REFERENCE SUMMARY:

The petitioners are certain under trials in the State of Bihar. In the Writ Petitions filed by them under Article 32 of the Constitution they complained that after their arrest, whilst under police custody they were blinded by the members of the police force, acting not in their private capacity but as police officials and their fundamental right to life guaranteed under Ar

In [266]:
import os
import torch
import json

save_dir = "/kaggle/working/phase6_prompt"
os.makedirs(save_dir, exist_ok=True)

checkpoint = {
    "soft_prompt": soft_prompt.detach().cpu(),
    "prompt_length": PROMPT_LENGTH,
    "hidden_size": soft_prompt.shape[-1],
    "architecture": "A4",
    "method": "Soft Prompt Tuning",
    "trainable_parameters": soft_prompt.numel(),
    "validation_results": evaluate_soft_prompt(max_batches=20),
    "generation_file": (
        "/kaggle/working/phase6_prompt/"
        "a4_soft_prompt_generation.json"
    )
}

checkpoint_path = os.path.join(
    save_dir,
    "a4_soft_prompt_step_30.pt"
)

torch.save(checkpoint, checkpoint_path)

print("Saved:", checkpoint_path)
print("Prompt shape:", tuple(soft_prompt.shape))
print("Trainable parameters:", soft_prompt.numel())
print("Validation:", checkpoint["validation_results"])

Saved: /kaggle/working/phase6_prompt/a4_soft_prompt_step_30.pt
Prompt shape: (1, 16, 512)
Trainable parameters: 8192
Validation: {'loss': 17.28407935194693, 'accuracy': 3.1725477500809323, 'tokens': 6178}


In [267]:
import torch

prefix_path = (
    "/kaggle/working/phase6_prefix/"
    "a4_prefix_step_30.pt"
)

prefix_checkpoint = torch.load(
    prefix_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(prefix_checkpoint))

if isinstance(prefix_checkpoint, dict):
    print("\nCheckpoint keys:")
    for key, value in prefix_checkpoint.items():
        if isinstance(value, torch.Tensor):
            print(key, tuple(value.shape), value.dtype)
        elif isinstance(value, dict):
            print(key, "DICT:", list(value.keys())[:15])
        else:
            print(key, type(value), str(value)[:150])

Checkpoint type: <class 'dict'>

Checkpoint keys:
prefix_state DICT: ['encoder_layers.0.attn.prefix_key', 'encoder_layers.0.attn.prefix_value', 'encoder_layers.1.attn.prefix_key', 'encoder_layers.1.attn.prefix_value', 'encoder_layers.2.attn.prefix_key', 'encoder_layers.2.attn.prefix_value', 'encoder_layers.3.attn.prefix_key', 'encoder_layers.3.attn.prefix_value', 'encoder_layers.4.attn.prefix_key', 'encoder_layers.4.attn.prefix_value', 'encoder_layers.5.attn.prefix_key', 'encoder_layers.5.attn.prefix_value', 'decoder_layers.0.self_attn.prefix_key', 'decoder_layers.0.self_attn.prefix_value', 'decoder_layers.0.cross_attn.prefix_key']
prefix_length <class 'int'> 8
validation_results DICT: ['loss', 'accuracy', 'tokens']


In [268]:
import torch
import torch.nn as nn

# --------------------------------------------------
# 1. Load original A4 checkpoint
# --------------------------------------------------

base_state = torch.load(
    "/kaggle/input/datasets/aq1de3/a4-step-22000-pt/a4_step_22000.pt",
    map_location="cpu",
    weights_only=False
)

# Handle checkpoint wrapper if present
if "model_state_dict" in base_state:
    base_state = base_state["model_state_dict"]

# --------------------------------------------------
# 2. Recreate a fresh A4 model
# --------------------------------------------------

prefix_eval_model = A4Model().to(device)

prefix_eval_model.load_state_dict(
    base_state,
    strict=True
)

# Freeze original A4 parameters
for parameter in prefix_eval_model.parameters():
    parameter.requires_grad = False

# --------------------------------------------------
# 3. Inject PrefixAttention wrappers
# --------------------------------------------------

inject_prefix_attention(
    prefix_eval_model,
    prefix_length=prefix_checkpoint["prefix_length"]
)

prefix_eval_model = prefix_eval_model.to(device)

# --------------------------------------------------
# 4. Restore trained prefix parameters
# --------------------------------------------------

prefix_eval_model.load_state_dict(
    prefix_checkpoint["prefix_state"],
    strict=False
)

# --------------------------------------------------
# 5. Verify restoration
# --------------------------------------------------

prefix_parameters = [
    (name, parameter)
    for name, parameter in prefix_eval_model.named_parameters()
    if "prefix_" in name
]

print("Prefix parameters:", len(prefix_parameters))

print(
    "Total trainable prefix parameters:",
    sum(parameter.numel() for _, parameter in prefix_parameters)
)

print(
    "All prefix parameters finite:",
    all(torch.isfinite(p).all().item() for _, p in prefix_parameters)
)

print("Prefix length:", prefix_checkpoint["prefix_length"])
print("Validation results:", prefix_checkpoint["validation_results"])

NameError: name 'inject_prefix_attention' is not defined

In [269]:
import inspect

print("PREFIX-RELATED FUNCTIONS")
print("=" * 70)

for name, obj in globals().copy().items():
    if callable(obj) and (
        "prefix" in name.lower()
        or "inject" in name.lower()
    ):
        try:
            print(f"\nFunction: {name}")
            print("Signature:", inspect.signature(obj))
            print(inspect.getsource(obj)[:5000])
        except Exception as e:
            print("Could not inspect:", e)


print("\nPREFIX-RELATED CLASSES")
print("=" * 70)

for name, obj in globals().copy().items():
    if isinstance(obj, type) and "prefix" in name.lower():
        print("\nClass:", name)
        try:
            print(inspect.getsource(obj)[:7000])
        except Exception as e:
            print("Could not inspect:", e)

PREFIX-RELATED FUNCTIONS

Function: build_a3_prefix_mask
Signature: (prefix_lens, lengths, max_len, num_heads, device)
def build_a3_prefix_mask(prefix_lens, lengths, max_len, num_heads, device):
    """
    True = masked, False = allowed.

    Prefix queries attend bidirectionally within the document prefix.
    Summary queries attend to the full prefix and previous/current summary.
    Padding positions are excluded as keys.
    """
    batch_size = len(prefix_lens)

    mask = torch.ones(
        batch_size, max_len, max_len,
        dtype=torch.bool,
        device=device
    )

    for b in range(batch_size):
        prefix_len = int(prefix_lens[b])
        valid_len = int(lengths[b])

        assert 0 < prefix_len <= valid_len <= max_len

        # Document prefix: bidirectional attention within prefix.
        mask[b, :prefix_len, :prefix_len] = False

        # Summary suffix: prefix + causal summary attention.
        for q in range(prefix_len, valid_len):
            mask[b, q

In [270]:
import torch

# --------------------------------------------------
# 1. Create fresh A4 model
# --------------------------------------------------

prefix_eval_model = A4Model().to(device)

prefix_eval_model.load_state_dict(
    base_state,
    strict=True
)

# Freeze original A4 parameters
for p in prefix_eval_model.parameters():
    p.requires_grad = False

# --------------------------------------------------
# 2. Inject Prefix Attention
# --------------------------------------------------

prefix_count = inject_prefixes(
    prefix_eval_model,
    prefix_length=prefix_checkpoint["prefix_length"]
)

prefix_eval_model = prefix_eval_model.to(device)

print("Attention modules modified:", prefix_count)

# --------------------------------------------------
# 3. Restore trained prefix parameters
# --------------------------------------------------

missing, unexpected = prefix_eval_model.load_state_dict(
    prefix_checkpoint["prefix_state"],
    strict=False
)

print("\nMissing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

# --------------------------------------------------
# 4. Verify restored parameters
# --------------------------------------------------

prefix_parameters = [
    (name, p)
    for name, p in prefix_eval_model.named_parameters()
    if "prefix_" in name
]

print("\nPrefix parameter tensors:", len(prefix_parameters))

print(
    "Trainable prefix parameter count:",
    sum(p.numel() for _, p in prefix_parameters)
)

print(
    "All prefix parameters finite:",
    all(torch.isfinite(p).all().item() for _, p in prefix_parameters)
)

print("\nExpected: 36 attention modules")
print("Expected trainable parameters: 147456")

Attention modules modified: 18

Missing keys: 260
Unexpected keys: 0

Prefix parameter tensors: 36
Trainable prefix parameter count: 147456
All prefix parameters finite: True

Expected: 36 attention modules
Expected trainable parameters: 147456


In [271]:
# ============================================================
# A4 PREFIX TUNING — GENERATION EVALUATION
# ============================================================

import torch
import json
import os

prefix_eval_model.eval()

MAX_NEW_TOKENS = 128
NUM_SAMPLES = 5

def generate_prefix_summary(document):

    ids = tokenizer.encode(document).ids[:MAX_POSITION]

    encoder_ids = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        # Encode using the restored Prefix Tuning model
        memory, memory_mask = a4_encode(
            prefix_eval_model,
            encoder_ids
        )

        generated = torch.tensor(
            [[BOS_ID]],
            dtype=torch.long,
            device=device
        )

        for _ in range(MAX_NEW_TOKENS):

            logits = a4_decode(
                prefix_eval_model,
                generated,
                memory,
                memory_mask
            )

            next_token = logits[:, -1, :].argmax(
                dim=-1,
                keepdim=True
            )

            generated = torch.cat(
                [generated, next_token],
                dim=1
            )

            if next_token.item() == EOS_ID:
                break

    return tokenizer.decode(
        generated[0, 1:].tolist()
    )


# ------------------------------------------------------------
# Generate summaries on five validation documents
# ------------------------------------------------------------

prefix_generation_results = []

for i, sample in enumerate(phase6_dev):

    if i >= NUM_SAMPLES:
        break

    prediction = generate_prefix_summary(
        sample["document"]
    )

    result = {
        "id": sample["id"],
        "prediction": prediction,
        "reference": sample["summary"]
    }

    prefix_generation_results.append(result)

    print("\n" + "=" * 80)
    print("DOCUMENT ID:", sample["id"])
    print("=" * 80)

    print("\nGENERATED SUMMARY:\n")
    print(prediction)

    print("\nREFERENCE SUMMARY:\n")
    print(sample["summary"])


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

output_path = (
    "/kaggle/working/phase6_prefix/"
    "a4_prefix_generation.json"
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        prefix_generation_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved:", output_path)
print("Documents evaluated:", len(prefix_generation_results))


DOCUMENT ID: 4494

GENERATED SUMMARY:

 Nos the Investment. Mand Mand of of, Constitution ofV.or ofPort To. some us K K Hing,ani, 2016. Hing byani certain JPort of.12 whether the.). K question question is February is. Brokers J of for the Respondent for Miss before. are law for to Union of us. The Order of the Court was delivered by, dated, J J by question which arises before us February February is whether certain documents called for by the Court by its order, 16th February, 1981. 1981 to be produced by the State or their production is thumb under some provision of law. The documents called for

REFERENCE SUMMARY:

The petitioners are certain under trials in the State of Bihar. In the Writ Petitions filed by them under Article 32 of the Constitution they complained that after their arrest, whilst under police custody they were blinded by the members of the police force, acting not in their private capacity but as police officials and their fundamental right to life guaranteed under 

In [272]:
import os
import json

folders = [
    "/kaggle/working/phase6_a1",
    "/kaggle/working/phase6_hierarchical",
    "/kaggle/working/phase6_lorma",
    "/kaggle/working/phase6_kd",
    "/kaggle/working/phase6_prompt",
    "/kaggle/working/phase6_prefix"
]

print("=" * 90)
print("PHASE 6 EXPERIMENT RECORD INVENTORY")
print("=" * 90)

for folder in folders:

    print("\nFOLDER:", os.path.basename(folder))
    print("-" * 90)

    if not os.path.exists(folder):
        print("Folder missing")
        continue

    for filename in sorted(os.listdir(folder)):

        path = os.path.join(folder, filename)

        if filename.endswith(".json"):

            print("\nJSON:", filename)

            try:
                with open(path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                if isinstance(data, dict):
                    print("Keys:", list(data.keys()))

                    for key, value in data.items():
                        if isinstance(value, (str, int, float, bool)):
                            print(f"  {key}: {value}")

                elif isinstance(data, list):
                    print("Records:", len(data))
                    if data:
                        print("First record keys:",
                              list(data[0].keys())
                              if isinstance(data[0], dict)
                              else type(data[0]))

            except Exception as e:
                print("Read error:", str(e))

        elif filename.endswith(".pt"):
            size_mb = os.path.getsize(path) / (1024 ** 2)
            print(f"CHECKPOINT: {filename} ({size_mb:.2f} MB)")

PHASE 6 EXPERIMENT RECORD INVENTORY

FOLDER: phase6_a1
------------------------------------------------------------------------------------------

JSON: a1_full_finetune_results.json
Keys: ['experiment', 'architecture', 'checkpoint_source', 'checkpoint_initial_step', 'training_updates', 'training_records', 'validation_records', 'max_document_tokens', 'max_summary_tokens', 'learning_rate', 'final_train_loss', 'validation_loss', 'tokenizer_compatibility', 'status']
  experiment: A1_Full_Finetuning
  architecture: A1 decoder-only Transformer
  checkpoint_source: /kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
  checkpoint_initial_step: 500
  training_updates: 200
  training_records: 512
  validation_records: 64
  max_document_tokens: 1200
  max_summary_tokens: 800
  learning_rate: 1e-05
  final_train_loss: 6.299882936477661
  validation_loss: 6.239953763782978
  tokenizer_compatibility: UNVERIFIED
  status: COMPLETED
CHECKPOINT: a1_ful

In [273]:
import os
import json

# --------------------------------------------------
# 1. Update Soft Prompt experiment record
# --------------------------------------------------

prompt_path = (
    "/kaggle/working/phase6_prompt/"
    "a4_soft_prompt_experiment.json"
)

with open(prompt_path, "r", encoding="utf-8") as f:
    prompt_record = json.load(f)

prompt_record["generation_quality"] = (
    "Poor: five validation summaries evaluated; "
    "outputs were incoherent and failed to preserve reference facts."
)

prompt_record["generation_file"] = (
    "/kaggle/working/phase6_prompt/"
    "a4_soft_prompt_generation.json"
)

prompt_record["conclusion"] = (
    "Small teacher-forced validation improvement, "
    "but autoregressive generation was unusable."
)

with open(prompt_path, "w", encoding="utf-8") as f:
    json.dump(prompt_record, f, indent=2)


# --------------------------------------------------
# 2. Update Prefix Tuning experiment record
# --------------------------------------------------

prefix_path = (
    "/kaggle/working/phase6_prefix/"
    "a4_prefix_experiment.json"
)

with open(prefix_path, "r", encoding="utf-8") as f:
    prefix_record = json.load(f)

prefix_record["generation_quality"] = (
    "Poor: five validation summaries evaluated; "
    "outputs were incoherent and failed to preserve reference facts."
)

prefix_record["generation_file"] = (
    "/kaggle/working/phase6_prefix/"
    "a4_prefix_generation.json"
)

prefix_record["conclusion"] = (
    "Modest teacher-forced validation improvement, "
    "but autoregressive generation was unusable."
)

with open(prefix_path, "w", encoding="utf-8") as f:
    json.dump(prefix_record, f, indent=2)


# --------------------------------------------------
# 3. Locate hierarchical metrics and records
# --------------------------------------------------

print("HIERARCHICAL EXPERIMENT FILES")
print("=" * 70)

search_roots = [
    "/kaggle/working",
    "/kaggle/working/phase6_hierarchical"
]

seen = set()

for root in search_roots:
    if not os.path.exists(root):
        continue

    for current_root, dirs, files in os.walk(root):

        # Avoid unnecessary traversal of unrelated folders
        dirs[:] = [
            d for d in dirs
            if d.startswith("phase6") or current_root == "/kaggle/working"
        ]

        for filename in files:
            if (
                "hierarchical" in filename.lower()
                or "a4_hierarchical" in filename.lower()
            ):
                path = os.path.join(current_root, filename)

                if path in seen:
                    continue

                seen.add(path)

                print(path)

print("\nUpdated Soft Prompt and Prefix experiment records.")

HIERARCHICAL EXPERIMENT FILES
/kaggle/working/a4_hierarchical_metrics.json
/kaggle/working/a4_hierarchical_32_predictions.json
/kaggle/working/phase6_hierarchical/a4_hierarchical_fulltrain_step_150.pt
/kaggle/working/phase6_hierarchical/a4_hierarchical_fulltrain_step_140.pt
/kaggle/working/phase6_hierarchical/a4_hierarchical_fulltrain_step_130.pt
/kaggle/working/phase6_hierarchical/a4_hierarchical_pilot_100.pt
/kaggle/working/phase6_hierarchical/a4_hierarchical_fulltrain_step_110.pt
/kaggle/working/phase6_hierarchical/a4_hierarchical_fulltrain_step_120.pt

Updated Soft Prompt and Prefix experiment records.


In [274]:
import json
import os

files_to_inspect = {
    "metrics": "/kaggle/working/a4_hierarchical_metrics.json",
    "predictions": "/kaggle/working/a4_hierarchical_32_predictions.json"
}

for name, path in files_to_inspect.items():

    print("\n" + "=" * 85)
    print(name.upper())
    print("=" * 85)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        print(json.dumps(data, indent=2, ensure_ascii=False))

    elif isinstance(data, list):
        print("Total records:", len(data))

        for record in data[:3]:
            print("\n", json.dumps(
                record,
                indent=2,
                ensure_ascii=False
            ))


METRICS
{
  "experiment": "A4_Hierarchical",
  "checkpoint_step": 150,
  "evaluation_documents": 32,
  "generation_method": "Greedy",
  "max_new_tokens": 128,
  "rouge1_f1": 0.20841236970674965,
  "rougeL_f1": 0.16494600418184407,
  "bert_score_f1": null,
  "bert_score_status": "Not calculated: approved pretrained evaluation model not available",
  "generation_quality": "Poor - repetitive output"
}

PREDICTIONS
Total records: 32

 {
  "id": "6827",
  "reference": "The appellant filed an application before the Central Administrative Tribunal, Patna Bench, for a declaration that the second proviso to rule 4 of Civil Services Examination Rules was violative of Articles 14 and 16 of the Constitu tion of India. By an interim order the Tribunal allowed the appellant to appear at the Civil Services (Main) Examina tion, 1990, subject to the result of the final orders in the original application. The said application was transferred to this Court. In a bunch of similar cases, the Central Admin

In [275]:
import os
import json

record = {
    "experiment": "A4_Hierarchical_Summarization",
    "architecture": "A4 Encoder-Decoder Transformer",
    "checkpoint_source": (
        "/kaggle/input/datasets/aq1de3/"
        "a4-step-22000-pt/a4_step_22000.pt"
    ),
    "checkpoint_step": 150,

    "method": "Hierarchical Chunking and Fine-Tuning",

    "chunking": {
        "chunk_size": 900,
        "overlap": 100,
        "stride": 800,
        "chunk_representation_vectors": 16,
        "document_representation_vectors": 128,
        "total_chunks": 47114
    },

    "training": {
        "initial_pilot_updates": 100,
        "additional_updates": 50,
        "final_step": 150,
        "training_records": 4432,
        "validation_records": 476,
        "final_training_loss": 6.1128,
        "validation_loss": 6.0376
    },

    "generation": {
        "evaluation_documents": 32,
        "method": "Greedy",
        "max_new_tokens": 128
    },

    "metrics": {
        "rouge1_f1": 0.20841236970674965,
        "rougeL_f1": 0.16494600418184407,
        "bert_score_f1": None,
        "bert_score_status": (
            "Not calculated: approved pretrained "
            "evaluation model not available"
        )
    },

    "generation_quality": "Poor - repetitive output",

    "conclusion": (
        "Hierarchical chunking and A4 fine-tuning completed. "
        "Generated summaries were repetitive and did not "
        "preserve document-specific legal facts reliably."
    ),

    "metrics_file": (
        "/kaggle/working/a4_hierarchical_metrics.json"
    ),

    "predictions_file": (
        "/kaggle/working/a4_hierarchical_32_predictions.json"
    ),

    "checkpoint_file": (
        "/kaggle/working/phase6_hierarchical/"
        "a4_hierarchical_fulltrain_step_150.pt"
    )
}

output_path = (
    "/kaggle/working/phase6_hierarchical/"
    "a4_hierarchical_experiment_record.json"
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(record, f, indent=2)

print("Experiment record saved:", output_path)
print("ROUGE-1:", record["metrics"]["rouge1_f1"])
print("ROUGE-L:", record["metrics"]["rougeL_f1"])
print("BERTScore:", record["metrics"]["bert_score_f1"])
print("Status: DOCUMENTED")

Experiment record saved: /kaggle/working/phase6_hierarchical/a4_hierarchical_experiment_record.json
ROUGE-1: 0.20841236970674965
ROUGE-L: 0.16494600418184407
BERTScore: None
Status: DOCUMENTED


In [276]:
import os

search_roots = [
    "/kaggle/input",
    "/kaggle/working"
]

matches = []

for root in search_roots:
    for current_root, dirs, files in os.walk(root):

        # Skip unnecessary cache directories
        dirs[:] = [
            d for d in dirs
            if d not in [".cache", "__pycache__"]
        ]

        for filename in files:
            lower = filename.lower()

            if (
                "tokenizer" in lower
                or lower in ["vocab.json", "merges.txt"]
            ):
                path = os.path.join(current_root, filename)
                matches.append(path)

print("=" * 85)
print("TOKENIZER FILE SEARCH")
print("=" * 85)

for path in matches:
    print(path)

print("\nTotal matching files:", len(matches))

TOKENIZER FILE SEARCH
/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json

Total matching files: 1


In [277]:
from tokenizers import Tokenizer
import json
import os

TOKENIZER_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "tokenizer-json/tokenizer.json"
)

tokenizer_check = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer verification")
print("=" * 60)
print("File exists:", os.path.exists(TOKENIZER_PATH))
print("Vocabulary size:", tokenizer_check.get_vocab_size())
print("Special token IDs:")

for token in ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]:
    print(f"{token}: {tokenizer_check.token_to_id(token)}")

print("\nTokenizer model:")
print(json.dumps(tokenizer_check.to_str()[:1000], indent=2))

Tokenizer verification
File exists: True
Vocabulary size: 50000
Special token IDs:
[PAD]: 0
[UNK]: 1
[BOS]: 2
[EOS]: 3
[MASK]: 4

Tokenizer model:
"{\"version\":\"1.0\",\"truncation\":null,\"padding\":null,\"added_tokens\":[{\"id\":0,\"content\":\"[PAD]\",\"single_word\":false,\"lstrip\":false,\"rstrip\":false,\"normalized\":false,\"special\":true},{\"id\":1,\"content\":\"[UNK]\",\"single_word\":false,\"lstrip\":false,\"rstrip\":false,\"normalized\":false,\"special\":true},{\"id\":2,\"content\":\"[BOS]\",\"single_word\":false,\"lstrip\":false,\"rstrip\":false,\"normalized\":false,\"special\":true},{\"id\":3,\"content\":\"[EOS]\",\"single_word\":false,\"lstrip\":false,\"rstrip\":false,\"normalized\":false,\"special\":true},{\"id\":4,\"content\":\"[MASK]\",\"single_word\":false,\"lstrip\":false,\"rstrip\":false,\"normalized\":false,\"special\":true}],\"normalizer\":null,\"pre_tokenizer\":{\"type\":\"ByteLevel\",\"add_prefix_space\":false,\"trim_offsets\":true,\"use_regex\":true},\"post_p

In [278]:
import os
import json
from datetime import datetime

audit = {
    "phase": "Phase 6 - Abstractive Summarization",
    "tokenizer_path": TOKENIZER_PATH,
    "vocabulary_size": tokenizer_check.get_vocab_size(),
    "algorithm": "BPE",
    "pre_tokenizer": "ByteLevel",
    "normalizer": None,
    "special_tokens": {
        "[PAD]": 0,
        "[UNK]": 1,
        "[BOS]": 2,
        "[EOS]": 3,
        "[MASK]": 4
    },
    "architecture_compatibility": {
        "A1": "Unverified - original tokenizer unavailable",
        "A2": "Unverified - original tokenizer unavailable",
        "A3": "Unverified - original tokenizer unavailable",
        "A4": "Phase 6 tokenizer used; checkpoint compatibility verified"
    },
    "conclusion": (
        "Only one tokenizer file was found. Matching vocabulary size "
        "and special token IDs do not establish token ID compatibility."
    ),
    "audit_date": datetime.now().isoformat()
}

output_dir = "/kaggle/working/phase6_audit"
os.makedirs(output_dir, exist_ok=True)

audit_path = os.path.join(output_dir, "tokenizer_compatibility_audit.json")

with open(audit_path, "w") as f:
    json.dump(audit, f, indent=4)

print("Tokenizer audit saved:", audit_path)
print(json.dumps(audit, indent=2))

Tokenizer audit saved: /kaggle/working/phase6_audit/tokenizer_compatibility_audit.json
{
  "phase": "Phase 6 - Abstractive Summarization",
  "tokenizer_path": "/kaggle/input/datasets/aq1de3/tokenizer-json/tokenizer.json",
  "vocabulary_size": 50000,
  "algorithm": "BPE",
  "pre_tokenizer": "ByteLevel",
  "normalizer": null,
  "special_tokens": {
    "[PAD]": 0,
    "[UNK]": 1,
    "[BOS]": 2,
    "[EOS]": 3,
    "[MASK]": 4
  },
  "architecture_compatibility": {
    "A1": "Unverified - original tokenizer unavailable",
    "A2": "Unverified - original tokenizer unavailable",
    "A3": "Unverified - original tokenizer unavailable",
    "A4": "Phase 6 tokenizer used; checkpoint compatibility verified"
  },
  "conclusion": "Only one tokenizer file was found. Matching vocabulary size and special token IDs do not establish token ID compatibility.",
  "audit_date": "2026-11-01T13:05:24.416201"
}


In [279]:
import os
import json

ROOT = "/kaggle/working"

folders = [
    "phase6_a1",
    "phase6_hierarchical",
    "phase6_lorma",
    "phase6_kd",
    "phase6_prompt",
    "phase6_prefix",
    "phase6_audit"
]

print("=" * 90)
print("PHASE 6 EXPERIMENT RECORD INVENTORY")
print("=" * 90)

for folder in folders:
    folder_path = os.path.join(ROOT, folder)

    print(f"\n{'#' * 90}")
    print("FOLDER:", folder)

    if not os.path.isdir(folder_path):
        print("Folder not found")
        continue

    json_files = [
        f for f in os.listdir(folder_path)
        if f.lower().endswith(".json")
    ]

    if not json_files:
        print("No JSON experiment records found")
        continue

    for filename in sorted(json_files):
        path = os.path.join(folder_path, filename)

        print(f"\nFILE: {filename}")

        try:
            with open(path, "r") as f:
                data = json.load(f)

            if isinstance(data, dict):
                for key, value in data.items():
                    if isinstance(value, (dict, list)):
                        value = json.dumps(value)[:700]
                    print(f"  {key}: {value}")
            else:
                print("  Record type:", type(data).__name__)
                print("  Number of entries:", len(data))

        except Exception as e:
            print("  Could not read:", str(e))

PHASE 6 EXPERIMENT RECORD INVENTORY

##########################################################################################
FOLDER: phase6_a1

FILE: a1_full_finetune_results.json
  experiment: A1_Full_Finetuning
  architecture: A1 decoder-only Transformer
  checkpoint_source: /kaggle/input/datasets/aq1de3/architecture1-sustained-step-500-pt/architecture1_sustained_step_500.pt
  checkpoint_initial_step: 500
  training_updates: 200
  training_records: 512
  validation_records: 64
  max_document_tokens: 1200
  max_summary_tokens: 800
  learning_rate: 1e-05
  final_train_loss: 6.299882936477661
  validation_loss: 6.239953763782978
  tokenizer_compatibility: UNVERIFIED
  status: COMPLETED

FILE: a1_generation_samples.json
  Record type: list
  Number of entries: 5

##########################################################################################
FOLDER: phase6_hierarchical

FILE: a4_hierarchical_experiment_record.json
  experiment: A4_Hierarchical_Summarization
  architecture: 

In [280]:
import os
import json
from datetime import datetime

output_dir = "/kaggle/working/phase6_lorma"
os.makedirs(output_dir, exist_ok=True)

lorma_record = {
    "experiment": "A4_LoRMA_Plus",
    "architecture": "A4 Encoder-Decoder Transformer",
    "checkpoint_source": (
        "/kaggle/input/datasets/aq1de3/"
        "a4-step-22000-pt/a4_step_22000.pt"
    ),
    "method": "LoRMA+ multiplicative low-rank adaptation",
    "paper": "LoRMA: Low-Rank Multiplicative Adaptation for LLMs (ACL 2025)",
    "configuration": {
        "variant": "LoRMA+",
        "rank": 8,
        "alpha": 16,
        "target_modules": ["q_proj", "v_proj"],
        "adapted_modules": 36,
        "trainable_parameters": 294912,
        "training_updates": 30,
        "learning_rate": 0.0001
    },
    "training": {
        "mean_training_loss": 5.8878,
        "validation_loss_20_batches": 6.092978890253217,
        "validation_accuracy_percent": 19.100032372936226,
        "validation_tokens": 6178
    },
    "base_model_comparison": {
        "base_validation_loss": 6.235568633470075,
        "lorma_validation_loss": 6.092978890253217,
        "base_accuracy_percent": 18.808675946908384,
        "lorma_accuracy_percent": 19.100032372936226
    },
    "generation": {
        "evaluation_documents": 5,
        "generation_quality": "Poor - repetitive and unusable summaries",
        "generation_file": (
            "/kaggle/working/phase6_lorma/"
            "lorma_generation_samples.json"
        )
    },
    "bert_score_f1": None,
    "conclusion": (
        "LoRMA+ implementation passed identity and forward tests. "
        "Teacher-forced validation showed a small improvement over "
        "the base model, but generated summaries were repetitive "
        "and did not reliably preserve document-specific facts."
    ),
    "status": "COMPLETED_PILOT_DOCUMENTED",
    "record_date": datetime.now().isoformat()
}

record_path = os.path.join(
    output_dir,
    "a4_lorma_experiment_record.json"
)

with open(record_path, "w") as f:
    json.dump(lorma_record, f, indent=4)

print("LoRMA experiment record saved:", record_path)
print("Status:", lorma_record["status"])
print("Trainable parameters:", lorma_record["configuration"]["trainable_parameters"])
print("Validation loss:", lorma_record["training"]["validation_loss_20_batches"])
print("Generation quality:", lorma_record["generation"]["generation_quality"])

LoRMA experiment record saved: /kaggle/working/phase6_lorma/a4_lorma_experiment_record.json
Status: COMPLETED_PILOT_DOCUMENTED
Trainable parameters: 294912
Validation loss: 6.092978890253217
Generation quality: Poor - repetitive and unusable summaries


In [281]:
import os
import json
from datetime import datetime

output_dir = "/kaggle/working/phase6_audit"
os.makedirs(output_dir, exist_ok=True)

diagnostics = {
    "A2": {
        "architecture": "A2 Decoder-only Transformer",
        "checkpoint": (
            "/kaggle/input/datasets/aq1de3/"
            "a2-step-9000-pt/a2_step_9000.pt"
        ),
        "checkpoint_step": 9000,
        "checkpoint_train_loss": 6.801188290119171,
        "checkpoint_reconstruction": "Strict state-dictionary loading passed",
        "model_parameters": 51513680,
        "forward_test": "Passed; finite logits",
        "tokenizer": "Original tokenizer unavailable",
        "phase6_tokenizer_compatibility": "Unverified",
        "generation_diagnostic": (
            "Repetitive legal text; EOS not generated in diagnostic"
        ),
        "summarization_training": "Not performed",
        "status": "DIAGNOSTIC_ONLY"
    },
    "A3": {
        "architecture": "A3 Prefix-LM Transformer",
        "checkpoint": (
            "/kaggle/input/datasets/aq1de3/"
            "step-0001000-pt/step_0001000.pt"
        ),
        "checkpoint_reconstruction": "Checkpoint loaded successfully",
        "tokenizer": "Original tokenizer unavailable",
        "phase6_tokenizer_compatibility": "Unverified",
        "generation_diagnostic": (
            "Repetitive output; fixed-prefix diagnostic performed"
        ),
        "summarization_training": "Not performed",
        "status": "DIAGNOSTIC_ONLY"
    }
}

record = {
    "phase": "Phase 6 - Abstractive Summarization",
    "record_type": "Architecture compatibility diagnostics",
    "diagnostics": diagnostics,
    "limitation": (
        "A2 and A3 checkpoint loading and forward/generation diagnostics "
        "do not establish valid summarization performance. Their original "
        "tokenizer files were unavailable, so Phase 6 token-ID compatibility "
        "could not be verified."
    ),
    "record_date": datetime.now().isoformat()
}

path = os.path.join(
    output_dir,
    "a2_a3_architecture_diagnostics.json"
)

with open(path, "w") as f:
    json.dump(record, f, indent=4)

print("Diagnostic record saved:", path)

for name, item in diagnostics.items():
    print(
        name,
        "|", item["status"],
        "| Tokenizer:", item["phase6_tokenizer_compatibility"]
    )

Diagnostic record saved: /kaggle/working/phase6_audit/a2_a3_architecture_diagnostics.json
A2 | DIAGNOSTIC_ONLY | Tokenizer: Unverified
A3 | DIAGNOSTIC_ONLY | Tokenizer: Unverified


In [282]:
import os
import json
from datetime import datetime

ROOT = "/kaggle/working"

record_paths = {
    "A1_Full_Finetuning":
        "phase6_a1/a1_full_finetune_results.json",

    "A4_Hierarchical":
        "phase6_hierarchical/a4_hierarchical_experiment_record.json",

    "A4_LoRMA_Plus":
        "phase6_lorma/a4_lorma_experiment_record.json",

    "A4_Knowledge_Distillation":
        "phase6_kd/kd_experiment_record.json",

    "A4_Soft_Prompt":
        "phase6_prompt/a4_soft_prompt_experiment.json",

    "A4_Prefix_Tuning":
        "phase6_prefix/a4_prefix_experiment.json",

    "Tokenizer_Audit":
        "phase6_audit/tokenizer_compatibility_audit.json",

    "A2_A3_Diagnostics":
        "phase6_audit/a2_a3_architecture_diagnostics.json"
}

experiments = {}
missing_files = []

for name, relative_path in record_paths.items():
    full_path = os.path.join(ROOT, relative_path)

    if not os.path.exists(full_path):
        missing_files.append(relative_path)
        continue

    with open(full_path, "r") as f:
        experiments[name] = json.load(f)

master_report = {
    "project": "CS 781 - Phase 6 Abstractive Summarization",
    "generated_at": datetime.now().isoformat(),

    "experiment_inventory": {
        "documented_experiment_tracks": 6,
        "architecture_distribution": {
            "A1": 1,
            "A2": 0,
            "A3": 0,
            "A4": 5
        },
        "mandatory_lorma": "Pilot completed and documented",
        "mandatory_knowledge_distillation": "Separate pilot completed and documented",
        "generation_quality": (
            "Poor or unusable in the evaluated generation samples; "
            "do not interpret teacher-forced loss improvements as "
            "successful abstractive summarization."
        )
    },

    "requirements_assessment": {
        "six_experiment_tracks": "Six tracks documented; compliance needs instructor interpretation because A1 tokenizer compatibility is unverified.",
        "all_four_architectures": "Not demonstrated with validated summarization experiments.",
        "five_adaptation_techniques": "Five method categories represented: full fine-tuning, LoRMA+, knowledge distillation, soft prompt tuning, and prefix tuning.",
        "lorma": "Pilot completed and documented.",
        "separate_knowledge_distillation": "Pilot completed and documented.",
        "rouge_metrics": "Available for A4 hierarchical evaluation; other experiment evaluations are incomplete.",
        "bert_score": "Pending approved evaluation model or instructor guidance.",
        "official_test_submission": "Not yet generated or validated."
    },

    "experiment_records": experiments,

    "missing_record_files": missing_files,

    "overall_conclusion": (
        "Multiple A4 adaptation and summarization experiments were "
        "implemented and documented. A1 was investigated, but its "
        "tokenizer compatibility remains unverified. A2 and A3 were "
        "diagnostic-only. Generation quality was poor in the evaluated "
        "samples. Further work is required before claiming full Phase 6 "
        "compliance or successful summarization."
    ),

    "next_steps": [
        "Review the master experiment matrix against the official rubric.",
        "Confirm the required interpretation of architecture diversity.",
        "Resolve BERTScore evaluation compliance with the instructor.",
        "Prepare exactly one summary for each of the 100 test documents.",
        "Validate the final submission JSON."
    ]
}

output_dir = os.path.join(ROOT, "phase6_final")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "phase6_master_experiment_report.json"
)

with open(output_path, "w") as f:
    json.dump(master_report, f, indent=4)

print("MASTER REPORT SAVED")
print(output_path)
print("\nRecords included:", len(experiments))
print("Missing record files:", missing_files)
print("\nExperiment tracks:", master_report["experiment_inventory"]["documented_experiment_tracks"])
print("Architecture diversity:", master_report["requirements_assessment"]["all_four_architectures"])
print("BERTScore:", master_report["requirements_assessment"]["bert_score"])
print("Test submission:", master_report["requirements_assessment"]["official_test_submission"])

MASTER REPORT SAVED
/kaggle/working/phase6_final/phase6_master_experiment_report.json

Records included: 8
Missing record files: []

Experiment tracks: 6
Architecture diversity: Not demonstrated with validated summarization experiments.
BERTScore: Pending approved evaluation model or instructor guidance.
Test submission: Not yet generated or validated.


In [283]:
import json
import os
import numpy as np

TEST_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "phase6/test (4).json"
)

with open(TEST_PATH, "r") as f:
    test_data = json.load(f)

print("=" * 75)
print("PHASE 6 TEST DATA AUDIT")
print("=" * 75)

print("Total test documents:", len(test_data))
print("Required submission summaries:", len(test_data))

ids = [str(x["id"]) for x in test_data]
print("Unique IDs:", len(set(ids)))
print("Duplicate IDs:", len(ids) - len(set(ids)))

missing_id = sum("id" not in x for x in test_data)
missing_doc = sum(
    "document" not in x or not str(x.get("document", "")).strip()
    for x in test_data
)

print("Missing IDs:", missing_id)
print("Missing/empty documents:", missing_doc)

lengths = [
    len(str(x["document"]).split())
    for x in test_data
    if str(x.get("document", "")).strip()
]

print("\nDocument length statistics (words)")
print("Minimum:", min(lengths))
print("Median:", int(np.median(lengths)))
print("Mean:", round(np.mean(lengths), 2))
print("Maximum:", max(lengths))

print("\nFirst 3 document previews:")
for item in test_data[:3]:
    print("\nID:", item["id"])
    print("Words:", len(str(item["document"]).split()))
    print("Preview:", str(item["document"])[:500])

print("\nTest audit completed.")

PHASE 6 TEST DATA AUDIT
Total test documents: 100
Required submission summaries: 100
Unique IDs: 100
Duplicate IDs: 0
Missing IDs: 0
Missing/empty documents: 0

Document length statistics (words)
Minimum: 511
Median: 3171
Mean: 4782.71
Maximum: 39695

First 3 document previews:

ID: 4963
Words: 1325
Preview: Tax Reference Cases Nos. 3 to 5 of 1975 Tax Reference under Section 257 of the Income tax Act, 1961 made by the Income tax Appellate Tribunal, Cuttack Bench, Cuttack P.A. Francis, Champat Rai and Miss A. Subhashini for the Appellant. C.S.S. Rao for the Respondent. The Judgment of the Court was delivered by 308 PATHAK, J. : These references under s.27(1) of the have been made by the Income Tax Appellate tribunal, Cuttack Bench at the instance of the Commissioner of Wealth Tax, Orissa for the opin

ID: 5994
Words: 6160
Preview: Civil Appeal No. 1709 of 1988. From the Judgment and Order dated 9.10.1987 of t he Central Administrative Tribunal Chandigarh in Appln. T. 1055 of 1986. WITH 

In [284]:
import torch
import os

CHECKPOINT_PATH = (
    "/kaggle/working/phase6_hierarchical/"
    "a4_hierarchical_fulltrain_step_150.pt"
)

print("Checkpoint exists:", os.path.exists(CHECKPOINT_PATH))
print("Checkpoint size (MB):", round(
    os.path.getsize(CHECKPOINT_PATH) / (1024**2), 2
))

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nTop-level keys:")
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: dictionary ({len(value)} entries)")
        elif torch.is_tensor(value):
            print(f"{key}: tensor {tuple(value.shape)}")
        else:
            print(f"{key}: {str(value)[:250]}")

print("\nCheckpoint inspection completed.")

Checkpoint exists: True
Checkpoint size (MB): 784.63

Checkpoint type: <class 'dict'>

Top-level keys:
pilot_steps: 150
model_state_dict: dictionary (260 entries)
aggregator_state_dict: dictionary (14 entries)
optimizer_state_dict: dictionary (2 entries)
train_loss: 6.112812805175781
validation_loss: 6.037646802266439
chunk_size: 900
overlap: 100
encoder_frozen: True

Checkpoint inspection completed.


In [285]:
import torch
import os

CHECKPOINT_PATH = (
    "/kaggle/working/phase6_hierarchical/"
    "a4_hierarchical_fulltrain_step_150.pt"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

# Restore A4 model
model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# Restore hierarchical aggregator
aggregator.load_state_dict(
    checkpoint["aggregator_state_dict"],
    strict=True
)

model = model.to(device)
aggregator = aggregator.to(device)

model.eval()
aggregator.eval()

for parameter in model.parameters():
    parameter.requires_grad = False

for parameter in aggregator.parameters():
    parameter.requires_grad = False

print("=" * 65)
print("A4 HIERARCHICAL CHECKPOINT RESTORED")
print("=" * 65)

print("Device:", device)
print("Training step:", checkpoint["pilot_steps"])
print("Training loss:", checkpoint["train_loss"])
print("Validation loss:", checkpoint["validation_loss"])
print("Chunk size:", checkpoint["chunk_size"])
print("Overlap:", checkpoint["overlap"])

print("Model parameters:", sum(
    p.numel() for p in model.parameters()
))

print("Aggregator parameters:", sum(
    p.numel() for p in aggregator.parameters()
))

print("Model training mode:", model.training)
print("Aggregator training mode:", aggregator.training)

print("\nRestoration completed.")

RuntimeError: Error(s) in loading state_dict for A4Model:
	Missing key(s) in state_dict: "encoder_layers.0.attn.q_proj.A", "encoder_layers.0.attn.q_proj.B", "encoder_layers.0.attn.q_proj.base.weight", "encoder_layers.0.attn.q_proj.base.bias", "encoder_layers.0.attn.v_proj.A", "encoder_layers.0.attn.v_proj.B", "encoder_layers.0.attn.v_proj.base.weight", "encoder_layers.0.attn.v_proj.base.bias", "encoder_layers.1.attn.q_proj.A", "encoder_layers.1.attn.q_proj.B", "encoder_layers.1.attn.q_proj.base.weight", "encoder_layers.1.attn.q_proj.base.bias", "encoder_layers.1.attn.v_proj.A", "encoder_layers.1.attn.v_proj.B", "encoder_layers.1.attn.v_proj.base.weight", "encoder_layers.1.attn.v_proj.base.bias", "encoder_layers.2.attn.q_proj.A", "encoder_layers.2.attn.q_proj.B", "encoder_layers.2.attn.q_proj.base.weight", "encoder_layers.2.attn.q_proj.base.bias", "encoder_layers.2.attn.v_proj.A", "encoder_layers.2.attn.v_proj.B", "encoder_layers.2.attn.v_proj.base.weight", "encoder_layers.2.attn.v_proj

In [286]:
import torch
import torch.nn as nn

print("Current model architecture inspection")
print("=" * 60)

for name, module in model.named_modules():
    if "q_proj" in name or "v_proj" in name:
        if hasattr(module, "A") and hasattr(module, "B"):
            print(name, "-> LoRMA-wrapped")
        elif isinstance(module, nn.Linear):
            print(name, "-> Original Linear")

Current model architecture inspection
encoder_layers.0.attn.q_proj -> LoRMA-wrapped
encoder_layers.0.attn.q_proj.base -> Original Linear
encoder_layers.0.attn.v_proj -> LoRMA-wrapped
encoder_layers.0.attn.v_proj.base -> Original Linear
encoder_layers.1.attn.q_proj -> LoRMA-wrapped
encoder_layers.1.attn.q_proj.base -> Original Linear
encoder_layers.1.attn.v_proj -> LoRMA-wrapped
encoder_layers.1.attn.v_proj.base -> Original Linear
encoder_layers.2.attn.q_proj -> LoRMA-wrapped
encoder_layers.2.attn.q_proj.base -> Original Linear
encoder_layers.2.attn.v_proj -> LoRMA-wrapped
encoder_layers.2.attn.v_proj.base -> Original Linear
encoder_layers.3.attn.q_proj -> LoRMA-wrapped
encoder_layers.3.attn.q_proj.base -> Original Linear
encoder_layers.3.attn.v_proj -> LoRMA-wrapped
encoder_layers.3.attn.v_proj.base -> Original Linear
encoder_layers.4.attn.q_proj -> LoRMA-wrapped
encoder_layers.4.attn.q_proj.base -> Original Linear
encoder_layers.4.attn.v_proj -> LoRMA-wrapped
encoder_layers.4.attn.v_p

In [287]:
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_CHECKPOINT = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-22000-pt/a4_step_22000.pt"
)

HIERARCHICAL_CHECKPOINT = (
    "/kaggle/working/phase6_hierarchical/"
    "a4_hierarchical_fulltrain_step_150.pt"
)

# Create a completely fresh A4 model
clean_model = A4Model().to(device)

# Load original Phase 1 A4 checkpoint
base_checkpoint = torch.load(
    BASE_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

base_state = base_checkpoint.get(
    "model_state_dict",
    base_checkpoint
)

clean_model.load_state_dict(base_state, strict=True)

# Load hierarchical checkpoint
hier_checkpoint = torch.load(
    HIERARCHICAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

# Restore hierarchical fine-tuned weights
load_result = clean_model.load_state_dict(
    hier_checkpoint["model_state_dict"],
    strict=True
)

clean_model.eval()

print("=" * 65)
print("CLEAN A4 MODEL RESTORATION")
print("=" * 65)
print("Base checkpoint loaded: YES")
print("Hierarchical checkpoint loaded: YES")
print("Strict loading: PASSED")
print("Missing keys:", len(load_result.missing_keys))
print("Unexpected keys:", len(load_result.unexpected_keys))
print("Device:", device)
print("Training step:", hier_checkpoint["pilot_steps"])

CLEAN A4 MODEL RESTORATION
Base checkpoint loaded: YES
Hierarchical checkpoint loaded: YES
Strict loading: PASSED
Missing keys: 0
Unexpected keys: 0
Device: cuda
Training step: 150


In [288]:
# Restore the aggregator from the same hierarchical checkpoint
aggregator.load_state_dict(
    hier_checkpoint["aggregator_state_dict"],
    strict=True
)

aggregator = aggregator.to(device)
aggregator.eval()

for parameter in aggregator.parameters():
    parameter.requires_grad = False

print("=" * 65)
print("HIERARCHICAL AGGREGATOR RESTORATION")
print("=" * 65)

print("Strict loading: PASSED")
print("Aggregator parameters:", sum(
    p.numel() for p in aggregator.parameters()
))
print("Device:", next(aggregator.parameters()).device)
print("Training mode:", aggregator.training)

print("\nA4 model and aggregator are ready.")

HIERARCHICAL AGGREGATOR RESTORATION
Strict loading: PASSED
Aggregator parameters: 2177024
Device: cuda:0
Training mode: False

A4 model and aggregator are ready.


In [289]:
import inspect

keywords = [
    "chunk", "aggregate", "hierarch",
    "generate", "summar", "encode",
    "decode", "token"
]

print("=" * 75)
print("AVAILABLE PHASE 6 INFERENCE FUNCTIONS")
print("=" * 75)

for name, obj in sorted(globals().items()):
    if any(word in name.lower() for word in keywords):
        if inspect.isfunction(obj):
            try:
                print(f"\nFUNCTION: {name}")
                print("SIGNATURE:", inspect.signature(obj))
            except Exception:
                print("FUNCTION:", name)

        elif inspect.isclass(obj):
            print(f"\nCLASS: {name}")

print("\nInspection completed.")

AVAILABLE PHASE 6 INFERENCE FUNCTIONS

CLASS: A1SummarizationDataset

CLASS: A4DecoderLayer

CLASS: A4EncoderLayer

CLASS: HierarchicalAggregator

CLASS: HierarchicalLegalDataset

CLASS: LegalSummarizationDataset

CLASS: Tokenizer

FUNCTION: a4_decode
SIGNATURE: (model, decoder_ids, memory, memory_padding_mask=None)

FUNCTION: a4_encode
SIGNATURE: (model, encoder_ids)

FUNCTION: a4_encode_soft_prompt
SIGNATURE: (current_model, encoder_ids, prompt)

FUNCTION: generate_a1_summary
SIGNATURE: (document, max_new_tokens=160)

FUNCTION: generate_a3_fixed_prefix
SIGNATURE: (model, tokenizer, prompt, max_new_tokens=100)

FUNCTION: generate_a3_summary
SIGNATURE: (record, max_new_tokens=160)

FUNCTION: generate_a3_with_model
SIGNATURE: (model, record, max_new_tokens=120)

FUNCTION: generate_controlled
SIGNATURE: (document_memory, max_new_tokens=200, repetition_penalty=1.2, no_repeat_ngram_size=3)

FUNCTION: generate_from_memory
SIGNATURE: (memory, max_tokens=150)

FUNCTION: generate_hierarchical_

In [290]:
import inspect

functions_to_inspect = [
    "generate_hierarchical_summary",
    "get_chunk_starts",
    "a4_encode",
    "a4_decode",
    "HierarchicalAggregator"
]

for name in functions_to_inspect:
    obj = globals().get(name)

    print("\n" + "=" * 80)
    print("SOURCE:", name)
    print("=" * 80)

    if obj is None:
        print("Not found")
        continue

    try:
        print(inspect.getsource(obj))
    except Exception as e:
        print("Could not inspect:", e)


SOURCE: generate_hierarchical_summary
@torch.inference_mode()
def generate_hierarchical_summary(record, max_new_tokens=128):

    model.eval()
    aggregator.eval()

    # Tokenize and split the complete document
    doc_ids = tokenizer.encode(record["document"]).ids
    starts = get_chunk_starts(len(doc_ids))

    compressed_chunks = []

    for start in starts:
        chunk = doc_ids[start:start + CHUNK_SIZE]
        chunk = [BOS_ID] + chunk + [EOS_ID]

        chunk_tensor = torch.tensor(
            [chunk], dtype=torch.long, device=DEVICE
        )

        memory, _ = a4_encode(model, chunk_tensor)
        compressed = aggregator.compress_chunk(memory)
        compressed_chunks.append(compressed)

        del memory, chunk_tensor

    combined = torch.cat(compressed_chunks, dim=1)
    document_memory = aggregator.aggregate_document(combined)

    # Autoregressive generation
    generated = [BOS_ID]

    for _ in range(max_new_tokens):
        decoder_ids = torch.tensor(
       

In [291]:
import torch

@torch.inference_mode()
def generate_hierarchical_summary_clean(record, max_new_tokens=128):

    clean_model.eval()
    aggregator.eval()

    # Tokenize complete document
    doc_ids = tokenizer.encode(record["document"]).ids
    starts = get_chunk_starts(len(doc_ids))

    compressed_chunks = []

    for start in starts:
        chunk = doc_ids[start:start + CHUNK_SIZE]
        chunk = [BOS_ID] + chunk + [EOS_ID]

        chunk_tensor = torch.tensor(
            [chunk], dtype=torch.long, device=device
        )

        memory, _ = a4_encode(clean_model, chunk_tensor)
        compressed = aggregator.compress_chunk(memory)
        compressed_chunks.append(compressed)

        del memory, chunk_tensor

    combined = torch.cat(compressed_chunks, dim=1)

    document_memory = aggregator.aggregate_document(combined)

    # Autoregressive greedy generation
    generated = [BOS_ID]

    for _ in range(max_new_tokens):
        decoder_ids = torch.tensor(
            [generated], dtype=torch.long, device=device
        )

        logits = a4_decode(
            clean_model,
            decoder_ids,
            document_memory,
            memory_padding_mask=None
        )

        next_token = logits[0, -1].argmax().item()

        if next_token == EOS_ID:
            break

        generated.append(next_token)

    summary = tokenizer.decode(
        generated[1:],
        skip_special_tokens=True
    )

    return summary, len(doc_ids), len(starts)


# Test on the first document
test_record = next(
    item for item in test_data
    if str(item["id"]) == "4963"
)

summary, token_count, chunk_count = generate_hierarchical_summary_clean(
    test_record,
    max_new_tokens=128
)

print("=" * 75)
print("ONE-DOCUMENT GENERATION TEST")
print("=" * 75)
print("Document ID:", test_record["id"])
print("Input tokens:", token_count)
print("Chunks processed:", chunk_count)
print("Generated summary tokens:", len(tokenizer.encode(summary).ids))

print("\nGENERATED SUMMARY:")
print(summary)

print("\nGeneration test completed.")

ONE-DOCUMENT GENERATION TEST
Document ID: 4963
Input tokens: 1567
Chunks processed: 2
Generated summary tokens: 128

GENERATED SUMMARY:
The appellant was a appellant was not later than the appellant was not been a appellant was not been section. The appellant was not, the appellant was not been a the appellant was a 's. The appellant was not, the appellant was in the appellant was the appellant was not been in the appellant was not been found to the appellant of the appellant was not be rejected by the appellant was not the appellant was not been the appellant was not been the appellant was not been rejected by the appellant. The appellant was not, the appellant was not been rejected the appellant. The appellant was in the appellant. The appellant. The High Court, the appellant

Generation test completed.


In [292]:
import torch

selected_ids = ["4963", "5994"]

for doc_id in selected_ids:
    record = next(
        item for item in test_data
        if str(item["id"]) == doc_id
    )

    summary, token_count, chunk_count = (
        generate_hierarchical_summary_clean(
            record,
            max_new_tokens=64
        )
    )

    print("\n" + "=" * 80)
    print("DOCUMENT ID:", doc_id)
    print("Input tokens:", token_count)
    print("Chunks:", chunk_count)
    print("Generated summary:")
    print(summary)


DOCUMENT ID: 4963
Input tokens: 1567
Chunks: 2
Generated summary:
The appellant was a appellant was not later than the appellant was not been a appellant was not been section. The appellant was not, the appellant was not been a the appellant was a 's. The appellant was not, the appellant was in the appellant was the appellant was not been in the appellant was not been found to

DOCUMENT ID: 5994
Input tokens: 7036
Chunks: 9
Generated summary:
The appellant was a appellant was not later. The appellant was not in the appellant was not been a appellant was the appellant was not later than the appellant was not the appellant was an. The appellant was in the appellant was not the appellant was not the appellant was the appellant was not been the appellant was not been found to


In [293]:
[
  {
    "id": "4963",
    "summary": "Generated summary text..."
  }
]

[{'id': '4963', 'summary': 'Generated summary text...'}]

In [294]:
[
  {
    "id": "4963",
    "summary": "Generated summary for document 4963."
  },
  {
    "id": "5994",
    "summary": "Generated summary for document 5994."
  }
]

[{'id': '4963', 'summary': 'Generated summary for document 4963.'},
 {'id': '5994', 'summary': 'Generated summary for document 5994.'}]

In [295]:
import os
import json
import torch
from tqdm.auto import tqdm

OUTPUT_DIR = "/kaggle/working/phase6_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "phase6_test_predictions.json"
)

predictions = []

clean_model.eval()
aggregator.eval()

for record in tqdm(test_data, desc="Generating test summaries"):

    summary, input_tokens, chunks = (
        generate_hierarchical_summary_clean(
            record,
            max_new_tokens=128
        )
    )

    predictions.append({
        "id": str(record["id"]),
        "summary": summary.strip()
    })

    # Save progress after each document
    with open(OUTPUT_PATH, "w") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=False)

print("\nGeneration completed.")
print("Predictions:", len(predictions))
print("Saved:", OUTPUT_PATH)

Generating test summaries:   0%|          | 0/100 [00:00<?, ?it/s]


Generation completed.
Predictions: 100
Saved: /kaggle/working/phase6_final/phase6_test_predictions.json


In [296]:
import json
import os

TEST_PATH = "/kaggle/input/datasets/aq1de3/phase6/test (4).json"

SUBMISSION_PATH = (
    "/kaggle/working/phase6_final/"
    "phase6_test_predictions.json"
)

with open(TEST_PATH, "r") as f:
    original_test = json.load(f)

with open(SUBMISSION_PATH, "r") as f:
    submission = json.load(f)

expected_ids = [str(x["id"]) for x in original_test]
submitted_ids = [str(x["id"]) for x in submission]

checks = {
    "Correct number of predictions": len(submission) == 100,
    "All IDs unique": len(set(submitted_ids)) == len(submitted_ids),
    "All original IDs present": set(expected_ids) == set(submitted_ids),
    "No missing summaries": all(
        isinstance(x.get("summary"), str)
        and bool(x["summary"].strip())
        for x in submission
    ),
    "Correct dictionary keys": all(
        set(x.keys()) == {"id", "summary"}
        for x in submission
    ),
    "Correct JSON list structure": isinstance(submission, list)
}

print("=" * 65)
print("FINAL SUBMISSION VALIDATION")
print("=" * 65)

for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", name)

print("\nOverall:", "VALID" if all(checks.values()) else "INVALID")
print("File size (KB):", round(os.path.getsize(SUBMISSION_PATH) / 1024, 2))
print("Submission:", SUBMISSION_PATH)

FINAL SUBMISSION VALIDATION
PASS - Correct number of predictions
PASS - All IDs unique
PASS - All original IDs present
PASS - No missing summaries
PASS - Correct dictionary keys
PASS - Correct JSON list structure

Overall: VALID
File size (KB): 67.2
Submission: /kaggle/working/phase6_final/phase6_test_predictions.json


In [297]:
import os
import json
import shutil
import zipfile
from datetime import datetime

ROOT = "/kaggle/working"
FINAL_DIR = os.path.join(ROOT, "phase6_final")
PACKAGE_DIR = os.path.join(FINAL_DIR, "Phase6_Submission_Package")

os.makedirs(PACKAGE_DIR, exist_ok=True)

# Source files to collect
files_to_copy = {
    "01_Master_Report":
        "phase6_final/phase6_master_experiment_report.json",

    "02_Test_Predictions":
        "phase6_final/phase6_test_predictions.json",

    "03_A1_Experiment":
        "phase6_a1/a1_full_finetune_results.json",

    "04_A4_Hierarchical":
        "phase6_hierarchical/a4_hierarchical_experiment_record.json",

    "05_A4_LoRMA":
        "phase6_lorma/a4_lorma_experiment_record.json",

    "06_A4_Knowledge_Distillation":
        "phase6_kd/kd_experiment_record.json",

    "07_A4_Soft_Prompt":
        "phase6_prompt/a4_soft_prompt_experiment.json",

    "08_A4_Prefix_Tuning":
        "phase6_prefix/a4_prefix_experiment.json",

    "09_Tokenizer_Audit":
        "phase6_audit/tokenizer_compatibility_audit.json",

    "10_A2_A3_Diagnostics":
        "phase6_audit/a2_a3_architecture_diagnostics.json",

    "11_Hierarchical_Metrics":
        "a4_hierarchical_metrics.json",

    "12_Hierarchical_Predictions":
        "a4_hierarchical_32_predictions.json",

    "13_LoRMA_Generation":
        "phase6_lorma/lorma_generation_samples.json",

    "14_KD_Generation":
        "phase6_kd/kd_teacher_student_generation.json",

    "15_Soft_Prompt_Generation":
        "phase6_prompt/a4_soft_prompt_generation.json",

    "16_Prefix_Generation":
        "phase6_prefix/a4_prefix_generation.json"
}

copied = []
missing = []

for destination_name, relative_path in files_to_copy.items():
    source = os.path.join(ROOT, relative_path)

    if os.path.isfile(source):
        destination = os.path.join(
            PACKAGE_DIR,
            destination_name + os.path.splitext(source)[1]
        )
        shutil.copy2(source, destination)
        copied.append(destination_name)
    else:
        missing.append(relative_path)

# Create README
readme = """
CS 781 - PHASE 6
ABSTRACTIVE SUMMARIZATION
FINAL EXPERIMENT PACKAGE

Contents:
- Master experiment report
- Six experiment records
- A2/A3 architecture diagnostic report
- Tokenizer compatibility audit
- Hierarchical ROUGE metrics and predictions
- Generation samples
- Final 100-document test predictions

Important limitations:
- Generation quality was poor and repetitive.
- A1/A2/A3 tokenizer compatibility is unverified.
- BERTScore has not been calculated.
- Validated summarization across all four architectures
  has not been demonstrated.
- Test prediction JSON passed structural validation.

Large model checkpoints are intentionally excluded.
They remain in their original Kaggle working directories.

Generated: """ + datetime.now().isoformat()

with open(os.path.join(PACKAGE_DIR, "README.txt"), "w") as f:
    f.write(readme)

# Create ZIP
ZIP_PATH = os.path.join(
    FINAL_DIR,
    "Phase6_Submission_Package.zip"
)

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as archive:

    for filename in os.listdir(PACKAGE_DIR):
        full_path = os.path.join(PACKAGE_DIR, filename)
        archive.write(full_path, arcname=filename)

print("=" * 70)
print("PHASE 6 FINAL PACKAGE")
print("=" * 70)

print("Files copied:", len(copied))
print("Missing files:", missing)
print("Package folder:", PACKAGE_DIR)
print("ZIP archive:", ZIP_PATH)
print("ZIP size (MB):", round(os.path.getsize(ZIP_PATH) / (1024**2), 2))

print("\nIncluded files:")
for name in copied:
    print("  PASS:", name)

print("\nPackaging completed.")

PHASE 6 FINAL PACKAGE
Files copied: 16
Missing files: []
Package folder: /kaggle/working/phase6_final/Phase6_Submission_Package
ZIP archive: /kaggle/working/phase6_final/Phase6_Submission_Package.zip
ZIP size (MB): 0.08

Included files:
  PASS: 01_Master_Report
  PASS: 02_Test_Predictions
  PASS: 03_A1_Experiment
  PASS: 04_A4_Hierarchical
  PASS: 05_A4_LoRMA
  PASS: 06_A4_Knowledge_Distillation
  PASS: 07_A4_Soft_Prompt
  PASS: 08_A4_Prefix_Tuning
  PASS: 09_Tokenizer_Audit
  PASS: 10_A2_A3_Diagnostics
  PASS: 11_Hierarchical_Metrics
  PASS: 12_Hierarchical_Predictions
  PASS: 13_LoRMA_Generation
  PASS: 14_KD_Generation
  PASS: 15_Soft_Prompt_Generation
  PASS: 16_Prefix_Generation

Packaging completed.


In [298]:
import zipfile
import os
import json

ZIP_PATH = (
    "/kaggle/working/phase6_final/"
    "Phase6_Submission_Package.zip"
)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = z.namelist()

    bad_file = z.testzip()

    print("=" * 65)
    print("FINAL PACKAGE INTEGRITY CHECK")
    print("=" * 65)

    print("ZIP readable:", True)
    print("Corrupted file:", bad_file)
    print("Files inside ZIP:", len(files))

    print("\nFile list:")
    for filename in files:
        print("PASS:", filename)

    # Validate prediction JSON inside ZIP
    prediction_file = "02_Test_Predictions.json"

    predictions = json.loads(
        z.read(prediction_file).decode("utf-8")
    )

    print("\nSubmission verification:")
    print("Prediction count:", len(predictions))
    print("All summaries non-empty:", all(
        isinstance(x["summary"], str) and x["summary"].strip()
        for x in predictions
    ))

    print("\nFINAL RESULT:",
          "PASS" if bad_file is None and len(files) == 17
          and len(predictions) == 100 else "CHECK REQUIRED")

FINAL PACKAGE INTEGRITY CHECK
ZIP readable: True
Corrupted file: None
Files inside ZIP: 17

File list:
PASS: 14_KD_Generation.json
PASS: 11_Hierarchical_Metrics.json
PASS: 10_A2_A3_Diagnostics.json
PASS: 08_A4_Prefix_Tuning.json
PASS: 13_LoRMA_Generation.json
PASS: 07_A4_Soft_Prompt.json
PASS: README.txt
PASS: 12_Hierarchical_Predictions.json
PASS: 15_Soft_Prompt_Generation.json
PASS: 09_Tokenizer_Audit.json
PASS: 05_A4_LoRMA.json
PASS: 04_A4_Hierarchical.json
PASS: 02_Test_Predictions.json
PASS: 06_A4_Knowledge_Distillation.json
PASS: 16_Prefix_Generation.json
PASS: 03_A1_Experiment.json
PASS: 01_Master_Report.json

Submission verification:
Prediction count: 100
All summaries non-empty: True

FINAL RESULT: PASS


In [1]:

from pathlib import Path
import json

# Locate the Phase 7 query file
matches = list(
    Path("/kaggle/input").rglob("train.json")
)

phase7_root = None

for path in matches:
    if path.parent.name.lower() == "train":
        candidate = path.parent.parent
        if (candidate / "Test" / "test.json").exists():
            phase7_root = candidate
            break

if phase7_root is None:
    print("Phase 7 dataset not found.")
    print("Attach the official dataset through Kaggle Add Input.")
else:
    print("Phase 7 root:", phase7_root)

    required_files = [
        "Train/train.json",
        "Train/train_candidates.json",
        "Dev/dev.json",
        "Dev/dev_candidates.json",
        "Test/test.json",
        "Test/test_candidates.json"
    ]

    for name in required_files:
        p = phase7_root / name
        print(("OK" if p.exists() else "MISSING"), name)

    if all((phase7_root / x).exists() for x in required_files):
        for name in required_files:
            with open(phase7_root / name, encoding="utf-8") as f:
                data = json.load(f)
            print(
                f"{name}: {len(data)} records | "
                f"sample keys: {list(data)[:3]}"
            )


Phase 7 dataset not found.
Attach the official dataset through Kaggle Add Input.


In [2]:
from pathlib import Path
import json

# Find all JSON files under Kaggle inputs
all_json = list(Path("/kaggle/input").rglob("*.json"))

# Find a folder containing all six Phase 7 files
phase7_root = None

for folder in set(p.parent for p in all_json):
    files = {p.name.lower() for p in folder.glob("*.json")}

    required = [
        "train_candidates.json",
        "dev_candidates.json",
        "test_candidates.json"
    ]

    has_train = any(
        name.startswith("train") and "candidate" not in name
        for name in files
    )
    has_dev = any(
        name.startswith("dev") and "candidate" not in name
        for name in files
    )
    has_test = any(
        name.startswith("test") and "candidate" not in name
        for name in files
    )

    if all(x in files for x in required) and has_train and has_dev and has_test:
        phase7_root = folder
        break

if phase7_root is None:
    raise FileNotFoundError("Could not locate all six Phase 7 JSON files.")

print("Phase 7 folder:", phase7_root)

def find_file(prefix, candidate=False):
    matches = [
        p for p in phase7_root.glob("*.json")
        if p.name.lower().startswith(prefix)
        and (("candidate" in p.name.lower()) == candidate)
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {prefix} file (candidate={candidate}), "
            f"found: {matches}"
        )
    return matches[0]

paths = {
    "train": find_file("train"),
    "train_candidates": find_file("train", True),
    "dev": find_file("dev"),
    "dev_candidates": find_file("dev", True),
    "test": find_file("test"),
    "test_candidates": find_file("test", True),
}

data = {}

for name, path in paths.items():
    with open(path, "r", encoding="utf-8") as f:
        data[name] = json.load(f)

    print(f"{name:18s}: {len(data[name]):,} records | {path.name}")

print("\nAll six files loaded successfully.")

Phase 7 folder: /kaggle/input/datasets/aq1de3/phase-7
train             : 827 records | train (4).json
train_candidates  : 4,320 records | train_candidates.json
dev               : 118 records | dev (3).json
dev_candidates    : 1,023 records | dev_candidates.json
test              : 237 records | test (5).json
test_candidates   : 1,727 records | test_candidates.json

All six files loaded successfully.


In [3]:
for name in ["train", "dev", "test"]:
    query_data = data[name]
    candidate_data = data[f"{name}_candidates"]

    print(f"\n{'='*50}")
    print(name.upper())
    print("="*50)

    query_id = next(iter(query_data))
    candidate_id = next(iter(candidate_data))

    print("Sample query ID:", query_id)
    print("Query fields:", list(query_data[query_id].keys()))
    print("Sample candidate ID:", candidate_id)
    print("Candidate fields:", list(candidate_data[candidate_id].keys()))

    print("Query text type:", type(query_data[query_id].get("text")))
    print("Candidate text type:", type(candidate_data[candidate_id].get("text")))

    if "relevant_candidates" in query_data[query_id]:
        print(
            "Relevant candidates:",
            len(query_data[query_id]["relevant_candidates"])
        )


TRAIN
Sample query ID: 0000000599
Query fields: ['text', 'relevant_candidates']
Sample candidate ID: 0000000262
Candidate fields: ['text']
Query text type: <class 'list'>
Candidate text type: <class 'list'>
Relevant candidates: 2

DEV
Sample query ID: 0000012637
Query fields: ['text', 'relevant_candidates']
Sample candidate ID: 0000003060
Candidate fields: ['text']
Query text type: <class 'list'>
Candidate text type: <class 'list'>
Relevant candidates: 11

TEST
Sample query ID: 0000021652
Query fields: ['text']
Sample candidate ID: 0000000987
Candidate fields: ['text']
Query text type: <class 'list'>
Candidate text type: <class 'list'>


In [4]:
from collections import Counter
import statistics

def text_stats(records):
    sentence_counts = []
    char_counts = []

    for case_id, item in records.items():
        text = item.get("text")

        if not isinstance(text, list):
            raise TypeError(f"{case_id}: text is not a list")

        sentence_counts.append(len(text))
        char_counts.append(sum(len(str(s)) for s in text))

    return {
        "cases": len(records),
        "empty_texts": sum(
            not item.get("text") for item in records.values()
        ),
        "avg_sentences": round(statistics.mean(sentence_counts), 2),
        "max_sentences": max(sentence_counts, default=0),
        "avg_characters": round(statistics.mean(char_counts), 2),
        "max_characters": max(char_counts, default=0),
    }


for split in ["train", "dev", "test"]:
    queries = data[split]
    candidates = data[f"{split}_candidates"]

    query_ids = set(queries)
    candidate_ids = set(candidates)

    print(f"\n{'='*60}")
    print(f"{split.upper()} DATA AUDIT")
    print("="*60)

    print("Queries:", len(query_ids))
    print("Candidates:", len(candidate_ids))
    print("Query/candidate ID overlap:", len(query_ids & candidate_ids))

    print("\nQuery text statistics:")
    print(text_stats(queries))

    print("\nCandidate text statistics:")
    print(text_stats(candidates))

    if split in ["train", "dev"]:
        missing_relevant = []
        relevant_counts = []

        for query_id, item in queries.items():
            relevant = item.get("relevant_candidates", [])

            if not isinstance(relevant, list):
                raise TypeError(
                    f"{query_id}: relevant_candidates is not a list"
                )

            relevant_counts.append(len(relevant))

            for candidate_id in relevant:
                if candidate_id not in candidate_ids:
                    missing_relevant.append(
                        (query_id, candidate_id)
                    )

        print("\nRelevance statistics:")
        print("Queries with relevance labels:", len(relevant_counts))
        print("Average relevant candidates:",
              round(statistics.mean(relevant_counts), 2))
        print("Minimum relevant candidates:",
              min(relevant_counts, default=0))
        print("Maximum relevant candidates:",
              max(relevant_counts, default=0))
        print("Missing relevant candidate IDs:", len(missing_relevant))

        if missing_relevant:
            print("First missing examples:", missing_relevant[:5])

    print("\nCandidate text fields valid:",
          all(isinstance(x.get("text"), list)
              for x in candidates.values()))


TRAIN DATA AUDIT
Queries: 827
Candidates: 4320
Query/candidate ID overlap: 827

Query text statistics:
{'cases': 827, 'empty_texts': 0, 'avg_sentences': 59.42, 'max_sentences': 766, 'avg_characters': 39286.19, 'max_characters': 716495}

Candidate text statistics:
{'cases': 4320, 'empty_texts': 0, 'avg_sentences': 53.17, 'max_sentences': 2981, 'avg_characters': 36369.54, 'max_characters': 2423006}

Relevance statistics:
Queries with relevance labels: 827
Average relevant candidates: 6.42
Minimum relevant candidates: 0
Maximum relevant candidates: 52
Missing relevant candidate IDs: 0

Candidate text fields valid: True

DEV DATA AUDIT
Queries: 118
Candidates: 1023
Query/candidate ID overlap: 118

Query text statistics:
{'cases': 118, 'empty_texts': 0, 'avg_sentences': 73.12, 'max_sentences': 1111, 'avg_characters': 48667.47, 'max_characters': 618469}

Candidate text statistics:
{'cases': 1023, 'empty_texts': 0, 'avg_sentences': 71.36, 'max_sentences': 2981, 'avg_characters': 49906.97, 'm

In [5]:
from tokenizers import Tokenizer
from collections import Counter
import numpy as np

TOKENIZER_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "tokenizer-json/tokenizer.json"
)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

# Match the ByteLevel decoder configuration from your notebook
from tokenizers import decoders
tokenizer.decoder = decoders.ByteLevel()

print("Tokenizer vocabulary:", tokenizer.get_vocab_size())
print("Special token IDs:")
for token in ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]:
    print(token, tokenizer.token_to_id(token))


# ------------------------------------------------
# 1. SELF-RETRIEVAL AUDIT
# ------------------------------------------------

for split in ["train", "dev", "test"]:

    queries = data[split]
    candidates = data[f"{split}_candidates"]

    self_in_pool = [
        qid for qid in queries
        if qid in candidates
    ]

    print(f"\n{split.upper()} SELF-RETRIEVAL")
    print("Query IDs:", len(queries))
    print("Query IDs present in candidate dictionary:",
          len(self_in_pool))

    if split in ["train", "dev"]:
        self_relevant = [
            qid for qid, item in queries.items()
            if qid in item["relevant_candidates"]
        ]
        print("Queries with themselves as relevant:",
              len(self_relevant))


# ------------------------------------------------
# 2. TOKEN LENGTH AUDIT
# ------------------------------------------------

def measure_token_lengths(records, label):

    lengths = []

    for case_id, item in records.items():

        full_text = "\n".join(
            str(sentence) for sentence in item["text"]
        )

        token_count = len(tokenizer.encode(full_text).ids)
        lengths.append(token_count)

    lengths = np.array(lengths)

    print(f"\n{label}")
    print("-" * 50)
    print("Documents:", len(lengths))
    print("Minimum tokens:", int(lengths.min()))
    print("Median tokens:", int(np.median(lengths)))
    print("Mean tokens:", round(float(lengths.mean()), 2))
    print("90th percentile:", int(np.percentile(lengths, 90)))
    print("95th percentile:", int(np.percentile(lengths, 95)))
    print("99th percentile:", int(np.percentile(lengths, 99)))
    print("Maximum tokens:", int(lengths.max()))

    for limit in [512, 1024, 2048, 4096, 8192]:
        count = int(np.sum(lengths > limit))
        print(
            f"Documents exceeding {limit:5d} tokens: "
            f"{count:5d} ({count / len(lengths) * 100:.2f}%)"
        )

    return lengths


token_lengths = {}

for split in ["train", "dev", "test"]:

    token_lengths[f"{split}_queries"] = measure_token_lengths(
        data[split],
        f"{split.upper()} QUERIES"
    )

    token_lengths[f"{split}_candidates"] = measure_token_lengths(
        data[f"{split}_candidates"],
        f"{split.upper()} CANDIDATES"
    )

print("\nTOKEN LENGTH AUDIT COMPLETED")

Tokenizer vocabulary: 50000
Special token IDs:
[PAD] 0
[UNK] 1
[BOS] 2
[EOS] 3
[MASK] 4

TRAIN SELF-RETRIEVAL
Query IDs: 827
Query IDs present in candidate dictionary: 827
Queries with themselves as relevant: 0

DEV SELF-RETRIEVAL
Query IDs: 118
Query IDs present in candidate dictionary: 118
Queries with themselves as relevant: 0

TEST SELF-RETRIEVAL
Query IDs: 237
Query IDs present in candidate dictionary: 237

TRAIN QUERIES
--------------------------------------------------
Documents: 827
Minimum tokens: 177
Median tokens: 5701
Mean tokens: 8607.54
90th percentile: 15812
95th percentile: 25760
99th percentile: 55915
Maximum tokens: 149015
Documents exceeding   512 tokens:   821 (99.27%)
Documents exceeding  1024 tokens:   797 (96.37%)
Documents exceeding  2048 tokens:   738 (89.24%)
Documents exceeding  4096 tokens:   552 (66.75%)
Documents exceeding  8192 tokens:   273 (33.01%)

TRAIN CANDIDATES
--------------------------------------------------
Documents: 4320
Minimum tokens: 8
Med

In [6]:
import math
import numpy as np

CONTEXT_LIMIT = 1024
SPECIAL_TOKENS = 2
CONTENT_TOKENS = CONTEXT_LIMIT - SPECIAL_TOKENS

print("A4 context limit:", CONTEXT_LIMIT)
print("Content tokens per chunk:", CONTENT_TOKENS)

for name, lengths in token_lengths.items():

    chunk_counts = [
        math.ceil(int(length) / CONTENT_TOKENS)
        for length in lengths
    ]

    print(f"\n{name.upper()}")
    print("Documents:", len(chunk_counts))
    print("Total chunks:", sum(chunk_counts))
    print("Average chunks/document:", round(np.mean(chunk_counts), 2))
    print("Maximum chunks/document:", max(chunk_counts))
    print("Documents needing multiple chunks:",
          sum(n > 1 for n in chunk_counts))

A4 context limit: 1024
Content tokens per chunk: 1022

TRAIN_QUERIES
Documents: 827
Total chunks: 7370
Average chunks/document: 8.91
Maximum chunks/document: 146
Documents needing multiple chunks: 798

TRAIN_CANDIDATES
Documents: 4320
Total chunks: 35776
Average chunks/document: 8.28
Maximum chunks/document: 505
Documents needing multiple chunks: 3980

DEV_QUERIES
Documents: 118
Total chunks: 1281
Average chunks/document: 10.86
Maximum chunks/document: 130
Documents needing multiple chunks: 116

DEV_CANDIDATES
Documents: 1023
Total chunks: 11366
Average chunks/document: 11.11
Maximum chunks/document: 505
Documents needing multiple chunks: 941

TEST_QUERIES
Documents: 237
Total chunks: 2087
Average chunks/document: 8.81
Maximum chunks/document: 112
Documents needing multiple chunks: 224

TEST_CANDIDATES
Documents: 1727
Total chunks: 15765
Average chunks/document: 9.13
Maximum chunks/document: 505
Documents needing multiple chunks: 1588


In [11]:
# A4 checkpoint diagnostic

import torch

print("CUDA available:", torch.cuda.is_available())

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

state = checkpoint.get("model_state_dict", checkpoint)

print("Checkpoint loaded")
print("Checkpoint keys:", list(state.keys())[:15])

print("\nEmbedding dimensions:")
print("Token embedding:", state["token_embedding.weight"].shape)
print("Encoder positions:", state["encoder_pos.weight"].shape)
print("Decoder positions:", state["decoder_pos.weight"].shape)

print("\nA4 checkpoint diagnostic completed.")

CUDA available: True
Checkpoint loaded
Checkpoint keys: ['token_embedding.weight', 'encoder_pos.weight', 'decoder_pos.weight', 'encoder_layers.0.norm1.weight', 'encoder_layers.0.norm1.bias', 'encoder_layers.0.attn.q_proj.weight', 'encoder_layers.0.attn.q_proj.bias', 'encoder_layers.0.attn.k_proj.weight', 'encoder_layers.0.attn.k_proj.bias', 'encoder_layers.0.attn.v_proj.weight', 'encoder_layers.0.attn.v_proj.bias', 'encoder_layers.0.attn.out_proj.weight', 'encoder_layers.0.attn.out_proj.bias', 'encoder_layers.0.norm2.weight', 'encoder_layers.0.norm2.bias']

Embedding dimensions:
Token embedding: torch.Size([50000, 512])
Encoder positions: torch.Size([1024, 512])
Decoder positions: torch.Size([1024, 512])

A4 checkpoint diagnostic completed.


In [12]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = encoder.to(device)
encoder.eval()

# Test different sequence lengths
for seq_len in [64, 512, 1024]:

    dummy_ids = torch.randint(
        low=0,
        high=50000,
        size=(1, seq_len),
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():
        output = encoder(dummy_ids)

    print(
        f"Input: {tuple(dummy_ids.shape)} | "
        f"Output: {tuple(output.shape)} | "
        f"Finite: {torch.isfinite(output).all().item()}"
    )

print("A4 encoder forward test completed.")
print("GPU:", torch.cuda.get_device_name(0))

NameError: name 'encoder' is not defined

In [13]:
print("Available model variables:")

for name, obj in globals().copy().items():
    if isinstance(obj, torch.nn.Module):
        print(name, type(obj).__name__)

Available model variables:


In [14]:
import torch
import torch.nn as nn

# A4 configuration from Phase 1
VOCAB_SIZE = 50000
HIDDEN_SIZE = 512
MAX_POSITION = 1024
NUM_LAYERS = 6
NUM_HEADS = 8
FFN_SIZE = 2048


class A4Attention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

    def forward(self, query, key, value, mask=None):
        B, Tq, C = query.shape
        Tk = key.shape[1]

        q = self.q_proj(query).view(
            B, Tq, self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = self.k_proj(key).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        v = self.v_proj(value).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        attn = torch.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, Tq, C)

        return self.out_proj(out)


class A4FFNCheckpoint(nn.Module):
    def __init__(self, hidden_size, ffn_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_size),
            nn.GELU(),
            nn.Linear(ffn_size, hidden_size)
        )

    def forward(self, x):
        return self.net(x)


class A4EncoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, ffn_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = A4Attention(hidden_size, num_heads)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn = A4FFNCheckpoint(hidden_size, ffn_size)

    def forward(self, x, mask=None):
        h = self.norm1(x)
        x = x + self.attn(h, h, h, mask)
        x = x + self.ffn(self.norm2(x))
        return x


class A4EncoderForNER(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE, HIDDEN_SIZE
        )
        self.encoder_pos = nn.Embedding(
            MAX_POSITION, HIDDEN_SIZE
        )

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(
                HIDDEN_SIZE, NUM_HEADS, FFN_SIZE
            )
            for _ in range(NUM_LAYERS)
        ])

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape

        if T > MAX_POSITION:
            raise ValueError("Sequence length exceeds 1024")

        positions = torch.arange(T, device=input_ids.device)

        x = self.token_embedding(input_ids)
        x = x + self.encoder_pos(positions).unsqueeze(0)

        for layer in self.encoder_layers:
            x = layer(x, mask=None)

        return x


encoder = A4EncoderForNER()

print("A4 encoder created.")
print("Parameters:", f"{sum(p.numel() for p in encoder.parameters()):,}")

A4 encoder created.
Parameters: 45,038,592


In [15]:
encoder_state = {
    key: value
    for key, value in state.items()
    if key.startswith((
        "token_embedding.",
        "encoder_pos.",
        "encoder_layers."
    ))
}

load_result = encoder.load_state_dict(
    encoder_state,
    strict=True
)

print("Strict loading:", load_result)

Strict loading: <All keys matched successfully>


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = encoder.to(device)
encoder.eval()

for seq_len in [64, 512, 1024]:

    dummy_ids = torch.randint(
        0, VOCAB_SIZE,
        (1, seq_len),
        device=device
    )

    with torch.no_grad():
        output = encoder(dummy_ids)

    print(
        "Input:", tuple(dummy_ids.shape),
        "| Output:", tuple(output.shape),
        "| Finite:", torch.isfinite(output).all().item()
    )

print("GPU:", torch.cuda.get_device_name(0))

Input: (1, 64) | Output: (1, 64, 512) | Finite: True
Input: (1, 512) | Output: (1, 512, 512) | Finite: True
Input: (1, 1024) | Output: (1, 1024, 512) | Finite: True
GPU: Tesla T4


In [17]:
print("Tokenizer vocabulary:", tokenizer.get_vocab_size())

sample = "This is a legal case retrieval test."

encoded = tokenizer.encode(sample)

print("Token IDs:", encoded.ids)
print("Decoded:", tokenizer.decode(encoded.ids))
print("Token count:", len(encoded.ids))

print("BOS:", tokenizer.token_to_id("[BOS]"))
print("EOS:", tokenizer.token_to_id("[EOS]"))
print("PAD:", tokenizer.token_to_id("[PAD]"))

Tokenizer vocabulary: 50000
Token IDs: [1909, 291, 239, 1999, 451, 42181, 262, 2949, 18]
Decoded: This is a legal case retrieval test.
Token count: 9
BOS: 2
EOS: 3
PAD: 0
